<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/SUBA_V8_PURE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚙️ Ячейка 0/10: Окружение SUBA RUN V8 (Java 17 + Flutter + Android SDK/NDK 28)
# ============================================================================
# SUBA RUN V8 — Subaru EJ20X · A2TB100B · SSM2-over-CAN Tuning Logger
# Clean-room сборка: только Subaru, без Nissan-наследия.
# Запускай один раз за сессию Colab. Идемпотентна (можно перезапускать).
# ============================================================================
import os

print('=' * 64)
print('  SUBA RUN V8 // A2TB100B  — подготовка окружения (~10 мин)')
print('=' * 64)

print('\n[1/4] Системные пакеты + Java 17 ...')
!apt-get update -qq
!apt-get install -y -qq curl git unzip xz-utils zip libglu1-mesa openjdk-17-jdk-headless ninja-build cmake > /dev/null
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 2>&1 | tail -1

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
!java -version 2>&1 | head -1

print('\n[2/4] Flutter SDK (stable) ...')
if not os.path.exists('/content/flutter'):
    !git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter 2>/dev/null
else:
    print('  Flutter уже скачан — пропуск.')

os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'
!/content/flutter/bin/flutter config --no-analytics --no-cli-animations 2>/dev/null
!/content/flutter/bin/flutter --disable-telemetry 2>/dev/null

print('\n[3/4] Android SDK 36 + Build-tools 36 + NDK 28 ...')
if not os.path.exists('/content/android-sdk/cmdline-tools/latest'):
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
    !mkdir -p /content/android-sdk/cmdline-tools
    !unzip -q /tmp/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
    !mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest 2>/dev/null || true

os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']

!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" "ndk;28.2.13676358" > /dev/null 2>&1

!/content/flutter/bin/flutter config --android-sdk /content/android-sdk 2>/dev/null
!/content/flutter/bin/flutter precache --android 2>/dev/null

print('\n[4/4] Проверка ...')
!/content/flutter/bin/flutter --version | head -2

print()
print('=' * 64)
print('  ГОТОВО: Java 17 / Flutter / SDK 36 / NDK 28.2.13676358')
print('  Далее -> ячейка 1/10 (проект + конфиги)')
print('=' * 64)


  SUBA RUN V8 // A2TB100B  — подготовка окружения (~10 мин)

[1/4] Системные пакеты + Java 17 ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
openjdk version "17.0.20" 2026-07-21

[2/4] Flutter SDK (stable) ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  227M  100  227M    0     0   189M      0  0:00:01  0:00:01 --:--:--  189M
Analytics reporting disabled.
Setting "cli-animations" value to "false".

You may need to restart any open editors for them to read new settings.

[3/4] Android SDK 36 + Build-tools 36 + NDK 28 ...
Setting "android-sdk" value to "/content/android-sdk".

You may need to restart any open ed

In [ ]:
# @title 🏗️ Ячейка 1/10: Проект SUBA RUN V8 + конфиги Android
# ============================================================================
# Создаёт flutter-проект suba_run_v8 и всю Android-обвязку.
# Проверенный набор из V6/V7: Gradle 8.10 · AGP 8.6 · SDK 36 · NDK 28.
# ============================================================================
import os

os.chdir('/content')
!rm -rf /content/suba_run_v8
!/content/flutter/bin/flutter create --org com.subarun --project-name suba_run_v8 suba_run_v8

os.chdir('/content/suba_run_v8')

for folder in ['models', 'services', 'screens', 'widgets', 'ssm', 'generated']:
    os.makedirs(f'lib/{folder}', exist_ok=True)

# ============ pubspec.yaml ============
with open('pubspec.yaml', 'w') as f:
    f.write('''name: suba_run_v8
description: Subaru EJ20X A2TB100B SSM2-over-CAN Tuning Logger V8
version: 8.0.0+1
publish_to: none

environment:
  sdk: ">=3.0.0 <4.0.0"

dependencies:
  flutter:
    sdk: flutter
  cupertino_icons: ^1.0.6
  fl_chart: 0.68.0
  flutter_bluetooth_serial: 0.4.0
  permission_handler: 11.3.1
  path_provider: 2.1.4
  path: ^1.9.0
  csv: 6.0.0
  shared_preferences: 2.3.2
  file_picker: 8.1.2
  share_plus: 10.0.2
  intl: ^0.19.0
  xml: ^6.5.0

dev_dependencies:
  flutter_test:
    sdk: flutter
  flutter_lints: ^4.0.0

flutter:
  uses-material-design: true
''')
print('OK  pubspec.yaml')

# ============ AndroidManifest.xml ============
with open('android/app/src/main/AndroidManifest.xml', 'w') as f:
    f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.INTERNET"/>
    <uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE" android:maxSdkVersion="28"/>
    <application
        android:label="SUBA RUN V8"
        android:name="${applicationName}"
        android:icon="@mipmap/ic_launcher">
        <activity
            android:name=".MainActivity"
            android:exported="true"
            android:launchMode="singleTop"
            android:theme="@style/LaunchTheme"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:hardwareAccelerated="true"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme"/>
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2"/>
    </application>
</manifest>
''')
print('OK  AndroidManifest.xml')

# ============ MainActivity.kt ============
main_dir = 'android/app/src/main/kotlin/com/subarun/suba_run_v8'
os.makedirs(main_dir, exist_ok=True)
!rm -rf android/app/src/main/java

with open(f'{main_dir}/MainActivity.kt', 'w') as f:
    f.write('''package com.subarun.suba_run_v8
import android.os.Bundle
import android.view.WindowManager
import io.flutter.embedding.android.FlutterActivity
class MainActivity : FlutterActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        window.addFlags(WindowManager.LayoutParams.FLAG_KEEP_SCREEN_ON)
    }
}
''')
print('OK  MainActivity.kt')

# ============ Gradle ============
with open('android/gradle/wrapper/gradle-wrapper.properties', 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip
''')

with open('android/settings.gradle.kts', 'w') as f:
    f.write('''pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        val flutterSdkPath = properties.getProperty("flutter.sdk")
        require(flutterSdkPath != null) { "flutter.sdk not set in local.properties" }
        flutterSdkPath
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.13.0" apply false
    id("org.jetbrains.kotlin.android") version "2.1.20" apply false
}
include(":app")
''')

with open('android/build.gradle.kts', 'w') as f:
    f.write('''allprojects {
    repositories { google(); mavenCentral() }
    configurations.all {
        resolutionStrategy {
            force("androidx.core:core:1.13.1")
            force("androidx.core:core-ktx:1.13.1")
            force("androidx.appcompat:appcompat:1.7.0")
            force("androidx.annotation:annotation:1.8.2")
        }
    }
}
val newBuildDir: Directory = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    val newSubprojectBuildDir: Directory = newBuildDir.dir(project.name)
    project.layout.buildDirectory.value(newSubprojectBuildDir)
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register("clean") { delete(rootProject.layout.buildDirectory) }
''')

with open('android/app/build.gradle.kts', 'w') as f:
    f.write('''plugins {
    id("com.android.application")
    id("kotlin-android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.subarun.suba_run_v8"
    compileSdk = 36
    ndkVersion = "28.2.13676358"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = JavaVersion.VERSION_17.toString() }
    defaultConfig {
        applicationId = "com.subarun.suba_run_v8"
        minSdk = 21
        targetSdk = 34
        versionCode = 8
        versionName = "8.0.0"
        multiDexEnabled = true
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
dependencies {
    implementation("androidx.core:core:1.13.1")
    implementation("androidx.core:core-ktx:1.13.1")
    implementation("androidx.appcompat:appcompat:1.7.0")
    implementation("androidx.multidex:multidex:2.0.1")
}
flutter { source = "../.." }
''')

with open('android/gradle.properties', 'w') as f:
    f.write('''org.gradle.jvmargs=-Xmx4G -XX:+UseParallelGC -XX:MaxMetaspaceSize=2G
android.useAndroidX=true
android.enableJetifier=true
android.nonTransitiveRClass=false
kotlin.code.style=official
org.gradle.parallel=true
org.gradle.caching=false
org.gradle.configuration-cache=false
kotlin.jvm.target.validation.mode=warning
android.suppressUnsupportedCompileSdk=36
''')
print('OK  Gradle-конфиги')
print()
print('=' * 64)
print('  Проект suba_run_v8 создан. Далее -> ячейка 2/10 (модели данных)')
print('=' * 64)


Creating project suba_run_v8...
Resolving dependencies in `suba_run_v8`...
Got dependencies in `suba_run_v8`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd suba_run_v8
  $ flutter run

Your application code is in suba_run_v8/lib/main.dart.

OK  pubspec.yaml
OK  AndroidManifest.xml
OK  MainActivity.kt
OK  Gradle-конфиги

  Проект suba_run_v8 создан. Далее -> ячейка 2/10 (модели данных)


In [ ]:
# @title 📦 Ячейка 2/10: Модели данных (constants / snapshot / ROM-таблицы)
# ============================================================================
# constants.dart        — профиль A2TB100B + пороги турбо-мониторинга (EJ20X)
# models/live_snapshot  — единый live-снимок по каноническим ключам (rpm, boost,
#                         iam, fbkc, fkl, kca...) — опрос больше НЕ зависит от имён PID
# models/rom_table      — определения карт + чтение значений из .bin ROM
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/constants.dart ============
with open('lib/constants.dart', 'w') as f:
    f.write('''class AppConstants {
  static const String appVersion = '8.0.0';
  static const String appName = 'SUBA RUN V8';
  static const String calId = 'A2TB100B';
  static const String ecuIdExpected = '5204584007';
  static const String engine = 'EJ20X (2.0 Turbo, AVCS)';
  static const double gasolineDensity = 745.0; // г/л
  static const double stoich = 14.7;

  // ── Пороги турбо-мониторинга (Subaru-специфика, не Nissan!) ──
  // FBKC/FKL — отрицательные градусы коррекции. Чем "глубже" минус, тем хуже.
  static const double fbkcWarn = -1.0;
  static const double fbkcDanger = -2.8;
  static const double fklWarn = -1.0;
  static const double fklDanger = -2.8;
  // IAM — множитель опережения 0..1 (float). Падение ниже 1.0 = детонация была.
  static const double iamWarn = 0.999;
  static const double iamDanger = 0.5;
  // Рассогласование буста (факт - цель), бар
  static const double boostErrWarn = 0.15;
  static const double boostErrDanger = 0.25;
  static const double overboostDanger = 1.35; // бар абсолютного избытка
  static const int ectWarn = 100;
  static const int ectDanger = 108;
  // Под бустом смесь беднее этого порога — опасно для поршней/колец
  static const double afrBoostLean = 12.0;
  static const double boostActive = 0.05; // бар — считаем что "под бустом"

  // Автолог
  static const int autoLogRpm = 1500;
  static const int autoLogIdleSec = 25;

  // Опрос (базовые значения, меняются в настройках)
  static const int elmInitTimeoutMs = 2500;
  static const int elmCmdTimeoutMs = 700;
  static const int maxBlockBytes = 0x50; // дефолтный потолок одного A8-запроса
}
''')
print('OK  lib/constants.dart')

# ============ lib/models/live_snapshot.dart ============
with open('lib/models/live_snapshot.dart', 'w') as f:
    f.write('''import '../constants.dart';

/// Единый live-снимок ECU.
/// Значения хранятся по КАНОНИЧЕСКИМ ключам (rpm, boost, iam, fbkc...),
/// которые генератор PID-библиотеки вычисляет из адресов/имён logger.xml.
/// Благодаря этому UI не зависит от того, какой именно вариант PID
/// (1-byte / 2-byte / 4-byte float) дал значение.
class LiveSnapshot {
  final DateTime ts;
  final Map<String, double> c;    // canon -> value
  final Map<String, double> byId; // pid id -> value (для CSV/отладки)

  LiveSnapshot(this.ts, this.c, [this.byId = const {}]);

  double g(String k, [double def = 0]) => c[k] ?? def;
  bool has(String k) => c[k] != null;

  double get rpm    => g('rpm');
  double get speed  => g('speed');
  double get ect    => g('ect');
  double get iat    => g('iat');
  double get boost  => g('boost');           // bar относительное
  double get tps    => g('tps');             // %
  double get kca    => g('kca');             // Knock Correction Advance = итоговый УОЗ
  double get fbkc   => g('fbkc');            // Feedback Knock Correction (обычно <= 0)
  double get fkl    => g('fkl');             // Fine Learning Knock Correction
  double get iam    => g('iam', 1.0);        // IAM 0..1
  double get afr    => g('afr', AppConstants.stoich);
  double get maf    => g('maf');             // g/s
  double get mafV   => g('mafV');
  double get tboost => g('tboost');          // bar целевой (relative)
  double get wgd    => g('wgd');             // % wastegate duty
  double get injMs  => g('injms');
  double get stft   => g('stft');            // A/F Correction #1
  double get ltft   => g('ltft');            // A/F Learning #1
  double get avcs   => g('avcs');            // град
  int    get knockSum => g('knocksum').round();
  double get batt   => g('batt');

  double get boostErr => has('tboost') ? (boost - tboost) : 0.0;
  bool   get underBoost => boost >= AppConstants.boostActive;

  /// Расход л/ч на основе MAF и AFR (как в V6, но MAF Subaru — г/с напрямую)
  double get fuelLph {
    if (maf <= 0 || afr <= 0) return 0;
    final gps = maf / afr;                    // г/с топлива
    return gps * 3600.0 / AppConstants.gasolineDensity;
  }

  String get mode {
    if (rpm < 1100 && speed < 3) return 'IDLE';
    if (tps > 85) return 'WOT';
    if (underBoost) return 'BOOST';
    if (rpm > 1500 && tps < 12 && speed > 40) return 'COAST';
    return 'CRUISE';
  }

  LiveSnapshot merge(LiveSnapshot newer) {
    final m = Map<String, double>.from(c)..addAll(newer.c);
    final i = Map<String, double>.from(byId)..addAll(newer.byId);
    return LiveSnapshot(newer.ts, m, i);
  }
}
''')
print('OK  lib/models/live_snapshot.dart')

# ============ lib/models/rom_table.dart ============
with open('lib/models/rom_table.dart', 'w') as f:
    f.write('''import 'dart:math' as math;
import 'dart:typed_data';

/// Колонка данных ROM (ось или Z-данные): где лежит, как трактовать байты.
/// [to] — raw -> физ. величина (toexpr из ECUFlash-дефинишна)
/// [fr] — физ. величина -> raw (frexpr), может быть null => read-only
class RomCol {
  final int address;
  final int count;
  final String storage; // float | uint8 | int8 | uint16 | int16
  final String endian;  // big | little
  final double Function(double num)? to;
  final double Function(double num)? fr;

  const RomCol({
    required this.address,
    required this.count,
    this.storage = 'float',
    this.endian = 'big',
    this.to,
    this.fr,
  });

  int get sizeOf {
    switch (storage) {
      case 'float':
      case 'uint32':
      case 'int32':
        return 4;
      case 'uint16':
      case 'int16':
        return 2;
      default:
        return 1;
    }
  }

  int get byteLen => sizeOf * count;

  int readRaw(List<int> rom, ByteData bd, int index) {
    final off = address + index * sizeOf;
    switch (storage) {
      case 'uint16':
        return endian == 'little' ? bd.getUint16(off, Endian.little) : bd.getUint16(off, Endian.big);
      case 'int16':
        return endian == 'little' ? bd.getInt16(off, Endian.little) : bd.getInt16(off, Endian.big);
      case 'int8':
        return bd.getInt8(off);
      case 'uint8':
      default:
        return bd.getUint8(off);
    }
  }

  double readFloat(List<int> rom, ByteData bd, int index) {
    final off = address + index * sizeOf;
    if (storage == 'float') {
      return endian == 'little' ? bd.getFloat32(off, Endian.little) : bd.getFloat32(off, Endian.big);
    }
    return readRaw(rom, bd, index).toDouble();
  }

  double value(List<int> rom, ByteData bd, int index) {
    final raw = readFloat(rom, bd, index);
    return to == null ? raw : to!(raw);
  }
}

/// Описание карты из ECUFlash-дефинишна A2TB100B (генерируется ячейкой 4/8)
class RomTableDef {
  final String name;
  final String category;
  final int address;
  final int rows;
  final int cols;
  final bool swapxy;
  final String units;
  final RomCol data;
  final RomCol? xAxis;
  final RomCol? yAxis;
  final double minHint;
  final double maxHint;

  const RomTableDef({
    required this.name,
    required this.category,
    required this.address,
    required this.rows,
    required this.cols,
    required this.data,
    this.swapxy = false,
    this.units = '',
    this.xAxis,
    this.yAxis,
    this.minHint = double.nan,
    this.maxHint = double.nan,
  });

  bool get is3d => rows > 1 || (yAxis != null && cols > 1);
  bool get editable => data.fr != null;
  String get addrHex => '0x${address.toRadixString(16).toUpperCase()}';
}

/// Загруженная из .bin карта со значениями
class RomTable {
  final RomTableDef def;
  final List<double> xValues; // физические, длина = cols
  final List<double> yValues; // физические, длина = rows
  final List<List<double>> z; // [rows][cols]

  RomTable({required this.def, required this.xValues, required this.yValues, required this.z});

  double get minV => z.expand((r) => r).fold(double.infinity, math.min);
  double get maxV => z.expand((r) => r).fold(-double.infinity, math.max);

  void setCell(int r, int c, double v) => z[r][c] = v;

  int _nearest(List<double> axis, double v) {
    var idx = 0;
    var best = double.infinity;
    for (var i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double at(double x, double y) => z[_nearest(yValues, y)][_nearest(xValues, x)];

  List<List<String>> toCsvRows() {
    final rows = <List<String>>[];
    rows.add(['${def.name} [${def.units}]', ...xValues.map((e) => e.toStringAsFixed(1))]);
    for (var r = 0; r < def.rows; r++) {
      rows.add([
        yValues.isNotEmpty ? yValues[r].toStringAsFixed(1) : '$r',
        ...z[r].map((e) => e.toStringAsFixed(3)),
      ]);
    }
    return rows;
  }
}
''')
print('OK  lib/models/rom_table.dart')
print()
print('=' * 64)
print('  Модели готовы. Далее -> ячейка 3/10 (SSM2-over-CAN протокол)')
print('=' * 64)


OK  lib/constants.dart
OK  lib/models/live_snapshot.dart
OK  lib/models/rom_table.dart

  Модели готовы. Далее -> ячейка 3/10 (SSM2-over-CAN протокол)


In [ ]:
# @title 🔌 Ячейка 3/10: SSM2-over-CAN стек (ELM327) + блочный опрос
# ============================================================================
# ГЛАВНОЕ УСКОРЕНИЕ ОТНОСИТЕЛЬНО V7:
#   V7: 1 CAN-запрос = 1 PID  => ~150 запросов на цикл (6-9 сек, 0.1 Гц)
#   V8: SSM2 читает ДИАПАЗОН адресов одним A8-запросом. Соседние PID
#       склеиваются в блоки (до maxBlock байт) => весь набор = 8-14 запросов.
#       + приоритетные ярусы: fast каждый цикл, mid 1/4, slow 1/25.
#       + ответ-ориентированный пайплайн (НИ ОДНОГО sleep между кадрами).
#       + count-1 семантика байта длины (по спецификации SSM2) с фолбэком.
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

with open('lib/ssm/ssm_elm.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:convert';
import 'dart:typed_data';

import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/live_snapshot.dart';

// ─────────────────────────────────────────────────────────────
// Статистика протокола (чтобы тормоза было ВИДНО на экране)
// ─────────────────────────────────────────────────────────────
class SsmStats {
  int framesOk = 0;
  int framesErr = 0;
  int noData = 0;
  double lastMs = 0;
  double avgMs = 0;
  double hz = 0;
  int _snapCounter = 0;
  DateTime _hzStart = DateTime.now();

  void ok(double ms) {
    framesOk++;
    lastMs = ms;
    avgMs = avgMs == 0 ? ms : avgMs * 0.85 + ms * 0.15;
  }

  void err() => framesErr++;
  void nodata() => noData++;

  void snap() {
    _snapCounter++;
    final d = DateTime.now().difference(_hzStart).inMilliseconds;
    if (d >= 1000) {
      hz = _snapCounter * 1000.0 / d;
      _snapCounter = 0;
      _hzStart = DateTime.now();
    }
  }

  void reset() {
    framesOk = 0;
    framesErr = 0;
    noData = 0;
    avgMs = 0;
    hz = 0;
  }
}

// ─────────────────────────────────────────────────────────────
// Блок чтения: непрерывный диапазон адресов + PID внутри
// ─────────────────────────────────────────────────────────────
class SsmBlock {
  final int start;
  final int len;
  final List<SubaruPid> pids;
  final int prio; // 1 fast, 2 mid, 3 slow
  int consecErr = 0;
  int skipUntilCycle = 0;

  SsmBlock(this.start, this.len, this.pids, this.prio);

  String get rangeHex => '0x${start.toRadixString(16).toUpperCase().padLeft(6, '0')}'
      '+${len.toRadixString(16).toUpperCase()}';
}

enum SsmState { disconnected, connecting, elmReady, ecuReady, polling, error }

// ─────────────────────────────────────────────────────────────
// Низкоуровневый ELM327 -> SSM2 over CAN (0x7E0 / 0x7E8, 500k, 11bit)
// ─────────────────────────────────────────────────────────────
class SsmElm {
  BluetoothConnection? _conn;
  StreamSubscription<Uint8List>? _sub;
  final StringBuffer _rx = StringBuffer();
  Completer<void>? _promptWaiter;

  final stats = SsmStats();
  SsmState state = SsmState.disconnected;

  int stTimeoutCode = 0x08; // AT ST (x4 мс): 0x08 = 32 мс ожидание байтов ответа
  String elmVersion = '';
  String ecuId = '';

  final StreamController<SsmState> stateCtl = StreamController<SsmState>.broadcast();
  Stream<SsmState> get onState => stateCtl.stream;

  void _setState(SsmState s) {
    state = s;
    if (!stateCtl.isClosed) stateCtl.add(s);
  }

  /// Публичный маркер для поллера: опрос идёт / остановлен
  void markPolling(bool v) => _setState(v ? SsmState.polling : SsmState.ecuReady);

  Future<bool> connect(String address) async {
    _setState(SsmState.connecting);
    try {
      _conn = await BluetoothConnection.toAddress(address)
          .timeout(const Duration(seconds: 12));
    } catch (_) {
      _setState(SsmState.error);
      return false;
    }
    _sub = _conn!.input!.listen(_onData, onDone: () => disconnect(), onError: (_) => disconnect());
    final ok = await setupElm();
    return ok;
  }

  void _onData(Uint8List chunk) {
    for (final b in chunk) {
      if (b == 0x3E) { // '>'
        _promptWaiter?.complete();
        _promptWaiter = null;
      } else {
        _rx.writeCharCode(b);
      }
    }
  }

  Future<String> transact(String cmd, {int timeoutMs = AppConstants.elmCmdTimeoutMs}) async {
    if (_conn == null) throw StateError('not connected');
    _rx.clear();
    _promptWaiter = Completer<void>();
    _conn!.output.add(Uint8List.fromList(ascii.encode('$cmd\\r')));
    await _conn!.output.allSent;
    try {
      await _promptWaiter!.future.timeout(Duration(milliseconds: timeoutMs));
    } catch (_) {
      // таймаут: вернём что успели накопить (парсер сам решит)
    }
    return _rx.toString();
  }

  Future<String> _expectOk(String cmd, {int timeoutMs = 900}) async {
    final r = await transact(cmd, timeoutMs: timeoutMs);
    return r.toUpperCase();
  }

  Future<bool> setupElm() async {
    // Сброс: ждём баннер ELM327
    String resp = '';
    for (var i = 0; i < 3; i++) {
      resp = await transact('ATZ', timeoutMs: AppConstants.elmInitTimeoutMs);
      if (resp.toUpperCase().contains('ELM327')) break;
      await Future.delayed(const Duration(milliseconds: 300));
    }
    if (!resp.toUpperCase().contains('ELM327')) {
      _setState(SsmState.error);
      return false;
    }
    elmVersion = RegExp(r'ELM327[^\\r\\n]*').firstMatch(resp)?.group(0) ?? 'ELM327';

    final st = stTimeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase();
    final steps = <String>[
      'ATE0',   // без эха
      'ATL0',   // без linefeed
      'ATS0',   // без пробелов — чистый hex-поток
      'ATH0',   // без CAN-заголовков — данные сразу
      'ATAL',   // allow long (>7 байт) — ISO-TP мультифрейм SSM2
      'ATST$st',// короткий таймаут ожидания (дефолт 200мс — тормоз!)
      'ATAT0',  // фиксированный тайминг, без адаптации
      'ATSP6',  // ISO 15765-4 CAN, 11 bit, 500 kbit
      'ATSH7E0' // наш моторный ECU
    ];
    for (final cmd in steps) {
      final r = await _expectOk(cmd);
      if (!r.contains('OK') && !r.contains('ELM')) {
        // ATSH на тонких клонах иногда ругается — пробуем продолжить
        if (!cmd.startsWith('ATSH')) { /* мягко игнорируем */ }
      }
    }
    // Фильтр приёма 7E8 — на части клонов CRA капризен, делаем best-effort
    await _expectOk('ATCRA7E8');
    _setState(SsmState.elmReady);
    return true;
  }

  /// SSM2 init. По CAN шлём тот же кадр, что и по K-line:
  /// 80 10 F0 01 BF 40  ->  ответ содержит 0xFF + ECU ID (ASCII)
  Future<String?> ecuInit() async {
    for (var attempt = 0; attempt < 3; attempt++) {
      final r = await transact('8010F001BF40', timeoutMs: 1400);
      final hex = r.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
      final idx = hex.indexOf('FF');
      if (idx >= 0 && hex.length >= idx + 2 + 22) {
        final idBytes = <int>[];
        for (var i = idx + 2; i + 2 <= hex.length && idBytes.length < 11; i += 2) {
          final code = int.tryParse(hex.substring(i, i + 2), radix: 16) ?? 0;
          if (code >= 0x30 && code <= 0x39) idBytes.add(code); // только цифры
        }
        ecuId = ascii.decode(idBytes);
        _setState(SsmState.ecuReady);
        return ecuId;
      }
      // некоторые ECU отвечают FF без ID в первом кадре — всё равно сессия жива
      if (idx >= 0) {
        _setState(SsmState.ecuReady);
        return ecuId.isEmpty ? 'OK' : ecuId;
      }
      await Future.delayed(const Duration(milliseconds: 250));
    }
    return null;
  }

  /// Чтение len байт начиная с адреса addr одним SSM2 A8-запросом.
  /// Байт счётчика в SSM2 = (количество байт - 1). Фолбэк: если ECU вернул меньше,
  /// повторяем с "прямым" счётчиком (некоторые прошивки трактуют его как abs).
  Future<Uint8List?> readBytes(int addr, int len) async {
    if (len < 1 || len > 255) return null;
    final a = addr.toRadixString(16).padLeft(6, '0').toUpperCase();
    final cnt1 = ((len - 1) & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
    final sw = Stopwatch()..start();

    var data = _parseRead(await transact('A8' '00' '$a$cnt1'), addr, len);
    if (data == null) {
      final cntAbs = (len & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
      data = _parseRead(await transact('A8' '00' '$a$cntAbs'), addr, len);
    }
    sw.stop();

    if (data != null) {
      stats.ok(sw.elapsedMicroseconds / 1000.0);
    } else {
      stats.err();
    }
    return data;
  }

  Uint8List? _parseRead(String resp, int addr, int len) {
    final hex = resp.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
    if (hex.contains('NODATA') || hex.isEmpty) { stats.nodata(); return null; }
    if (hex.contains('7F')) return null; // negative response
    var from = 0;
    final wantHi = (addr >> 16) & 0xFF, wantMid = (addr >> 8) & 0xFF, wantLo = addr & 0xFF;
    while (true) {
      final idx = hex.indexOf('E8', from);
      if (idx < 0) return null;
      // E8 00 00 00 aHi aMid aLo [data...]
      if (hex.length >= idx + 14) {
        final h = int.tryParse(hex.substring(idx + 2, idx + 4), radix: 16) ?? -1;
        final m = int.tryParse(hex.substring(idx + 4, idx + 6), radix: 16) ?? -1;
        final a1 = int.tryParse(hex.substring(idx + 8, idx + 10), radix: 16) ?? -1;
        final a2 = int.tryParse(hex.substring(idx + 10, idx + 12), radix: 16) ?? -1;
        final a3 = int.tryParse(hex.substring(idx + 12, idx + 14), radix: 16) ?? -1;
        final addrOk = (h == 0 || h == 0xFF) && m == 0 && a1 == wantHi && a2 == wantMid && a3 == wantLo;
        if (addrOk) {
          final avail = (hex.length - (idx + 14)) ~/ 2;
          if (avail >= len) {
            final out = Uint8List(len);
            for (var i = 0; i < len; i++) {
              out[i] = int.parse(hex.substring(idx + 14 + i * 2, idx + 16 + i * 2), radix: 16);
            }
            return out;
          }
        }
      }
      from = idx + 2;
      if (from >= hex.length) return null;
    }
  }

  Future<void> disconnect() async {
    try { await _sub?.cancel(); } catch (_) {}
    _sub = null;
    try { await _conn?.close(); } catch (_) {}
    _conn = null;
    _rx.clear();
    _setState(SsmState.disconnected);
  }
}

// ─────────────────────────────────────────────────────────────
// Поллер: блочный план + приоритетные ярусы + реконнект
// ─────────────────────────────────────────────────────────────
class SsmPoller {
  final SsmElm elm;
  int maxBlock = AppConstants.maxBlockBytes;
  int gapTol = 2;       // склеивать адреса, если разрыв <= gapTol байт
  int midEveryN = 4;    // средний ярус: каждый 4-й цикл
  int slowEveryN = 25;  // медленный ярус: каждый 25-й цикл

  List<SsmBlock> _blocks = [];
  final Map<String, double> _canon = {};
  final Map<String, double> _byId = {};

  Timer? _timer;
  bool _running = false;
  int _cycle = 0;
  int _globalErrStreak = 0;

  final StreamController<LiveSnapshot> _snapCtl = StreamController<LiveSnapshot>.broadcast();
  Stream<LiveSnapshot> get snapshots => _snapCtl.stream;
  LiveSnapshot? last;
  bool get isRunning => _running;
  int get blockCount => _blocks.length;
  int get pidCount => _blocks.fold(0, (a, b) => a + b.pids.length);

  SsmPoller(this.elm);

  /// Строим блоки: сортировка по адресу + склейка соседей.
  /// ЭТО главный ускоритель: вместо N запросов — ceil(N/плотность).
  List<SsmBlock> buildBlocks(List<SubaruPid> selected, {bool log = false}) {
    final list = selected.where((p) => p.len > 0).toList()
      ..sort((a, b) => a.address.compareTo(b.address));
    final blocks = <SsmBlock>[];
    var cur = <SubaruPid>[];
    var start = 0, end = 0, prio = 3;
    void flush() {
      if (cur.isEmpty) return;
      blocks.add(SsmBlock(start, end - start, List.of(cur), prio));
      cur = [];
    }
    for (final p in list) {
      final pStart = p.address, pEnd = p.address + p.len;
      if (cur.isEmpty) {
        start = pStart; end = pEnd; prio = p.priority; cur.add(p); continue;
      }
      if (pStart <= end + gapTol && (pEnd - start) <= maxBlock) {
        end = pEnd > end ? pEnd : end;
        if (p.priority < prio) prio = p.priority;
        cur.add(p);
      } else {
        flush();
        start = pStart; end = pEnd; prio = p.priority; cur.add(p);
      }
    }
    flush();
    _blocks = blocks;
    return blocks;
  }

  void start() {
    if (_running || _blocks.isEmpty) return;
    _running = true;
    _globalErrStreak = 0;
    elm.markPolling(true);
    _tick();
  }

  void stop() {
    _running = false;
    _timer?.cancel();
    if (elm.state == SsmState.polling) elm.markPolling(false);
  }

  Iterable<SsmBlock> _cycleBlocks(int cycle) sync* {
    for (final b in _blocks) {
      final due = b.prio == 1 || (b.prio == 2 && cycle % midEveryN == 0) || (b.prio == 3 && cycle % slowEveryN == 0);
      if (due && cycle >= b.skipUntilCycle) yield b;
    }
  }

  Future<void> _tick() async {
    if (!_running) return;
    _cycle++;
    for (final b in _cycleBlocks(_cycle)) {
      if (!_running) return;
      Uint8List? data;
      try {
        data = await elm.readBytes(b.start, b.len);
      } catch (_) {
        data = null; // отвал BT/адаптера — считаем ошибочным кадром
      }
      if (data != null) {
        b.consecErr = 0;
        _globalErrStreak = 0;
        for (final p in b.pids) {
          final off = p.address - b.start;
          if (off < 0 || off + p.len > data.length) continue;
          final sub = Uint8List.fromList(data.sublist(off, off + p.len));
          try {
            final v = p.formula(sub);
            if (v.isNaN || v.isInfinite) continue;
            _byId[p.id] = v;
            if (p.canon.isNotEmpty) _canon[p.canon] = v;
          } catch (_) {}
        }
      } else {
        b.consecErr++;
        _globalErrStreak++;
        if (b.consecErr >= 4) {
          b.skipUntilCycle = _cycle + 60; // блок «молчит» — отложим, не будем стоять в очереди
          b.consecErr = 0;
        }
      }
    }
    // снапшот после цикла быстрых блоков
    last = LiveSnapshot(DateTime.now(), Map.of(_canon), Map.of(_byId));
    elm.stats.snap();
    if (!_snapCtl.isClosed) _snapCtl.add(last!);

    // самолечение: много ошибок подряд -> re-init ECU
    if (_globalErrStreak > 40) {
      _globalErrStreak = 0;
      try {
        final id = await elm.ecuInit();
        if (id == null) { stop(); return; }
      } catch (_) {
        stop();
        return;
      }
    }
    // сразу следующий цикл — без искусственных задержек.
    // Темп ограничен реальным временем ответа ECU, а не sleep().
    _timer = Timer(Duration.zero, _tick);
  }

  Future<void> dispose() async {
    stop();
    await _snapCtl.close();
  }
}
''')
print('OK  lib/ssm/ssm_elm.dart  (ELM + A8-блоки + ярусы + статистика)')
print()
print('=' * 64)
print('  Протокол готов. Далее -> ячейка 4/10 (генератор PID + ROM-дефиниций)')
print('=' * 64)


OK  lib/ssm/ssm_elm.dart  (ELM + A8-блоки + ярусы + статистика)

  Протокол готов. Далее -> ячейка 4/10 (генератор PID + ROM-дефиниций)


In [ ]:
# @title 🧬 Ячейка 4/10: Генератор PID-библиотеки и ROM-дефиниций A2TB100B
# ============================================================================
# v8.1.2 — поддержка ДВУХ форматов ECUFlash-дефинишен:
#   А) стандартный: type/category/scaling инлайн в <table>
#   Б) SubMerp (этот репозиторий): в CAL-файле только имя+адрес+подоси,
#      а type/category/scaling/elements — в Bases/32BITBASE.xml.
#      Мерж по нормализованному имени ("Target Boost_" == "Target Boost").
# Цепочка: A2TB100B -> A2TB100K -> 32BITBASE (качаем ВСЕ три файла).
# ============================================================================
import os, re, json, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter

os.chdir('/content/suba_run_v8')

ECU_IDS = ['5204584007', '5204784007']
TARGET_CAL = 'A2TB100B'
LOGGER_URL = 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/RomRaider/logger/metric/logger.xml'
CAL_URLS = {
    'A2TB100B': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Legacy%20GT/A2TB100B.xml',
    'A2TB100K': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Legacy%20GT%20spec.B/A2TB100K.xml',
    '32BITBASE': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Bases/32BITBASE.xml',
}

def fetch(url, path, min_warn=800):
    if not os.path.exists(path) or os.path.getsize(path) < 400:
        print('  загрузка %s ...' % url.split('/')[-1])
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90) as r, open(path, 'wb') as f:
            f.write(r.read())
    sz = os.path.getsize(path)
    print('  %s: %.0f KB%s' % (path, sz / 1024, '  <- МАЛО' if sz < min_warn else ''))
    return sz

print('=' * 64)
print('  [1/4] Источники определений')
print('=' * 64)
fetch(LOGGER_URL, '/content/logger.xml', 500000)
for cal, url in CAL_URLS.items():
    fetch(url, '/content/%s.xml' % cal)

# ─────────────────────────────────────────────────────────────
# expr вида x*0.01953125 -> текст внутри Dart-замыкания
# ─────────────────────────────────────────────────────────────
def dart_body(expr):
    if not expr:
        return None
    e = str(expr).strip()
    # только числа/операторы/скобки/x; идентификаторы кроме одинокого x запрещены
    # (числа 1e-3 не считаются идентификатором — защита отгораживается lookbehind)
    if re.search(r'[^0-9\.\+\-\*/\(\)\sa-zA-Z]', e):
        return None
    idents = re.findall(r'(?<![0-9.])[a-zA-Z_]\w*', e)
    if any(i != 'x' for i in idents):
        return None
    # Нормализация числовых литералов ECUFlash -> канонические Dart-double:
    # .84 -> 0.84 · 5. -> 5.0 · 50 -> 50.0 · 1e-3 -> 0.001 · 14.7 -> 14.7
    num_pat = re.compile(r'(?<![0-9A-Za-z_.])(\d*\.\d+|\d+\.?\d*(?:[eE][+-]?\d+)?)')
    def _num(m):
        try:
            return repr(float(m.group(1)))
        except ValueError:
            return m.group(1)
    return num_pat.sub(_num, e)

def esc(s):
    return (s or '').replace('\\', '\\\\').replace("'", "\\'")

def fnum(v):
    try:
        return repr(float(v))
    except (TypeError, ValueError):
        return 'double.nan'

def norm_name(s):
    return re.sub(r'[^a-z0-9]+', '', (s or '').lower())

# ════════════════════════════════════════════════════════════
# ЧАСТЬ А: logger.xml -> PID
# ════════════════════════════════════════════════════════════
print()
print('=' * 64)
print('  [2/4] RomRaider logger.xml -> subaru_pids.g.dart')
print('=' * 64)

lroot = ET.parse('/content/logger.xml').getroot()

def store_default(length, addr_int):
    if length == 4 and addr_int >= 0xFF0000:
        return 'float'
    return 'uint8' if length == 1 else ('uint16' if length == 2 else 'bytes')

def pick_conv(el, length, addr_int):
    convs = el.findall('.//conversion')
    if not convs:
        return None
    if store_default(length, addr_int) == 'float':
        for c in convs:
            if (c.get('storagetype') or '').lower() == 'float':
                return c
    return convs[0]

CANON_RULES = [
    (r'^engine speed$', 'rpm'),
    (r'^vehicle speed$', 'speed'),
    (r'^coolant temperature$', 'ect'),
    (r'^intake air temperature|^air temp', 'iat'),
    (r'^throttle opening angle$|^throttle plate opening angle', 'tps'),
    (r'^manifold relative pressure', 'boost'),
    (r'^target boost relative|^target boost', 'tboost'),
    (r'^knock correction advance', 'kca'),
    (r'^feedback knock correction', 'fbkc'),
    (r'^fine learning knock correction', 'fkl'),
    (r'^iam(\s|\(|$)', 'iam'),
    (r'^a/f sensor #1$', 'afr'),
    (r'^a/f correction #1$', 'stft'),
    (r'^a/f learning #1$', 'ltft'),
    (r'^mass air flow', 'maf'),
    (r'^mass airflow sensor voltage', 'mafV'),
    (r'injector.*pulse|pulse width', 'injms'),
    (r'^primary wastegate duty cycle$', 'wgd'),
    (r'^avcs (intake|inlet) left', 'avcs'),
    (r'^knock sum', 'knocksum'),
    (r'^battery voltage|battery.*voltage', 'batt'),
]
FAST = {'rpm', 'speed', 'boost', 'tps', 'kca', 'fbkc', 'iam', 'afr', 'maf'}
MID = {'ect', 'iat', 'tboost', 'fkl', 'wgd', 'stft', 'ltft', 'injms', 'avcs', 'mafV', 'batt'}

def canon_for(name):
    n = name.lower().strip()
    for rx, key in CANON_RULES:
        if re.search(rx, n):
            return key
    return ''

def cat_of(unit, name):
    u = (unit + ' ' + name).lower()
    if re.search(r'boost|wastegate|turbo|tgv', u): return 'turbo'
    if re.search(r'timing|knock|ignition|iam|spark|advance|misfire', u): return 'ignition'
    if re.search(r'fuel|injector|a/f|lambda|o2|oxygen|ethanol', u): return 'fuel'
    if re.search(r'temp|coolant', u): return 'temp'
    if re.search(r'maf|airflow|manifold|pressure|baro', u): return 'air'
    if re.search(r'throttle|pedal|accelerator', u): return 'throttle'
    if re.search(r'battery|voltage|alternator', u): return 'electric'
    if re.search(r'gear|speed|rpm|engine|avcs|cam', u): return 'engine'
    return 'other'

pids = []
seen = set()

def collect(el, xmlid, addr_int, length, name, desc, conv):
    body = dart_body(conv.get('expr') or 'x')
    if body is None:
        return
    base = re.sub(r'[^0-9A-Za-z]+', '_', name.upper()).strip('_')[:28] or xmlid
    pid_id, k = base, 1
    while pid_id in seen:
        k += 1
        pid_id = '%s_%d' % (base, k)
    seen.add(pid_id)
    storage = (conv.get('storagetype') or store_default(length, addr_int)).lower()
    pids.append(dict(src=xmlid[0], xmlid=xmlid, id=pid_id, name=name,
                     desc=(desc or name)[:90], unit=conv.get('units') or '',
                     addr=addr_int, len=length, storage=storage, body=body))

for param in lroot.iter('parameter'):
    xmlid = param.get('id') or ''
    if not xmlid.startswith('P'):
        continue
    a_el = param.find('address')
    if a_el is None or not (a_el.text or '').strip():
        continue
    addr_int = int(a_el.text.strip(), 16)
    length = int(a_el.get('length') or '1')
    conv = pick_conv(param, length, addr_int)
    if conv is None:
        continue
    collect(param, xmlid, addr_int, length,
            (param.get('name') or xmlid).strip(),
            param.findtext('description'), conv)

for ecuparam in lroot.iter('ecuparam'):
    xmlid = ecuparam.get('id') or ''
    if not xmlid.startswith('E'):
        continue
    matched, mlen = None, 1
    for ecu in ecuparam.findall('ecu'):
        ids = [x.strip() for x in (ecu.get('id') or '').split(',') if x.strip()]
        if any(x in ids for x in ECU_IDS):
            a = ecu.find('address')
            if a is not None and (a.text or '').strip():
                matched, mlen = a.text.strip(), int(a.get('length') or '1')
                break
    if not matched:
        continue
    conv = pick_conv(ecuparam, mlen, int(matched, 16))
    if conv is None:
        continue
    collect(ecuparam, xmlid, int(matched, 16), mlen,
            (ecuparam.get('name') or xmlid).strip(),
            ecuparam.get('name'), conv)

by_canon = {}
for p in sorted(pids, key=lambda q: (0 if q['addr'] < 0xFF0000 else 1, q['addr'], -q['len'])):
    key = canon_for(p['name'])
    if key and key not in by_canon:
        by_canon[key] = p['id']
        p['canon'] = key
for p in pids:
    p.setdefault('canon', '')

def prio_of(p):
    if p['canon'] in FAST: return 1
    if p['canon'] in MID: return 2
    if p['canon']: return 2
    return 3

pids.sort(key=lambda q: (q['addr'], q['len']))

if len(pids) < 100:
    raise SystemExit('ОШИБКА: распознано только %d PID.' % len(pids))

dart = []
dart.append('''// АВТОСГЕНЕРИРОВАНО ячейкой 4/10 SUBA RUN V8
// RomRaider logger.xml (metric), ECU 5204584007 / 5204784007 (A2TB100B)
// ignore_for_file: constant_identifier_names
import 'dart:typed_data';

typedef PidFormula = double Function(List<int> b);

double _f32(List<int> b) {
  final l = b.length >= 4 ? b.sublist(0, 4) : <int>[...b, ...List.filled(4 - b.length, 0)];
  return ByteData.sublistView(Uint8List.fromList(l)).getFloat32(0, Endian.big);
}

double _raw(List<int> b, String storage) {
  switch (storage) {
    case 'float':
      return _f32(b);
    case 'int8':
      final v = b[0];
      return (v < 128 ? v : v - 256).toDouble();
    case 'uint16':
      return ((b[0] << 8) | b[1]).toDouble();
    case 'int16':
      var v = (b[0] << 8) | b[1];
      if (v >= 32768) v -= 65536;
      return v.toDouble();
    case 'bytes':
      var v = 0;
      for (final x in b) { v = (v << 8) | x; }
      return v.toDouble();
    case 'uint8':
    default:
      return b[0].toDouble();
  }
}

class SubaruPid {
  final String id;
  final String xmlId;
  final String name;
  final String desc;
  final String unit;
  final String category;
  final int address;
  final int len;
  final String storage;
  final int priority; // 1 fast / 2 mid / 3 slow
  final String canon;
  final PidFormula formula;
  const SubaruPid({
    required this.id, required this.xmlId, required this.name, required this.desc,
    required this.unit, required this.category, required this.address, required this.len,
    required this.storage, required this.priority, required this.canon, required this.formula,
  });
  String get addrHex => '0x' + address.toRadixString(16).toUpperCase().padLeft(6, '0');
}

class SubaruPids {
  static final List<SubaruPid> all = [
''')
for p in pids:
    dart.append("    SubaruPid(id: '%s', xmlId: '%s', name: '%s', desc: '%s',\n"
                "        unit: '%s', category: '%s', address: 0x%06X, len: %d,\n"
                "        storage: '%s', priority: %d, canon: '%s',\n"
                "        formula: (b) { final x = _raw(b, '%s'); return %s; }),\n"
                % (esc(p['id']), esc(p['xmlid']), esc(p['name']), esc(p['desc']),
                   esc(p['unit']), cat_of(p['unit'], p['name']), p['addr'], p['len'],
                   p['storage'], prio_of(p), p['canon'], p['storage'], p['body']))
dart.append('''  ];

  static SubaruPid? byId(String id) {
    for (final p in all) { if (p.id == id) return p; }
    return null;
  }

  static List<SubaruPid> get defaults => all.where((p) => p.canon.isNotEmpty).toList();

  static Map<String, List<SubaruPid>> byCategory() {
    final m = <String, List<SubaruPid>>{};
    for (final p in all) { m.putIfAbsent(p.category, () => []).add(p); }
    return m;
  }
}
''')
os.makedirs('lib/generated', exist_ok=True)
with open('lib/generated/subaru_pids.g.dart', 'w') as f:
    f.write(''.join(dart))

print('  PID всего: %d (P: %d, E: %d) · канонических: %d' % (
    len(pids), sum(1 for p in pids if p['src'] == 'P'),
    sum(1 for p in pids if p['src'] == 'E'), len(by_canon)))

# ════════════════════════════════════════════════════════════
# ЧАСТЬ Б: CAL + 32BITBASE -> карты (двухформатный парсер)
# ════════════════════════════════════════════════════════════
print()
print('=' * 64)
print('  [3/4] ECUFlash: мерж CAL-файлов с базой 32BITBASE')
print('=' * 64)

roots = {cal: ET.parse('/content/%s.xml' % cal).getroot() for cal in CAL_URLS}

def rom_list(root):
    return [root] if root.tag == 'rom' else list(root.iter('rom'))

roms_by_id = {}
for cal, root in roots.items():
    for rom in rom_list(root):
        romid = rom.find('romid')
        if romid is None:
            continue
        xmlid = (romid.findtext('xmlid') or '').strip()
        if xmlid:
            roms_by_id[xmlid] = rom

# Реестр именованных скейлингов + реестр "шаблонных" таблиц (с type) — из ВСЕХ файлов
scalings = {}     # name -> ET scaling element
base_tables = {}  # norm_name -> ET table element (с type attr)
for root in roots.values():
    for rom_el in rom_list(root):
        for sc in rom_el.findall('scaling'):
            if sc.get('name'):
                scalings[sc.get('name')] = sc
        for t in rom_el.iter('table'):
            if t.get('type'):
                base_tables.setdefault(norm_name(t.get('name')), t)
print('  скейлингов в реестре: %d · шаблонных таблиц: %d' % (len(scalings), len(base_tables)))

def resolve_chain(xmlid, seen, depth=0):
    """CAL-таблицы (с адресами): include сначала, свои перекрывают по имени"""
    if xmlid in seen or depth > 8 or xmlid not in roms_by_id:
        return {}, []
    seen.add(xmlid)
    rom = roms_by_id[xmlid]
    tables, chain = {}, [xmlid]
    for inc in rom.findall('include'):
        inc_id = (inc.text or '').strip()
        if not inc_id:
            continue
        sub_t, sub_c = resolve_chain(inc_id, seen, depth + 1)
        for k, v in sub_t.items():
            tables.setdefault(k, v)
        chain += sub_c
    for t in rom.findall('table'):
        tables[t.get('name') or 'unnamed'] = t
    return tables, chain

if TARGET_CAL not in roms_by_id:
    raise SystemExit('ОШИБКА: CAL %s не найден (%s)' % (TARGET_CAL, ', '.join(sorted(roms_by_id))))

tmap, chain = resolve_chain(TARGET_CAL, set())
romid_t = roms_by_id[TARGET_CAL].find('romid')
cpu = ((romid_t.findtext('cpu') or romid_t.findtext('memmodel') or '') if romid_t is not None else '')
fsize = ((romid_t.findtext('filesize') or '') if romid_t is not None else '')
mm = re.match(r'(\d+)\s*kb', (fsize or '').lower())
rom_bytes = int(mm.group(1)) * 1024 if mm else 1310720
print('  цепочка: %s · кандидатов: %d · CPU %s' % (' -> '.join(chain), len(tmap), cpu))

AX_X = ('X', 'x', 'X Axis', 'Static X Axis')
AX_Y = ('Y', 'y', 'Y Axis', 'Static Y Axis')

def sub_kind(sub):
    lbl = sub.get('type') or sub.get('name') or ''
    if lbl in AX_X: return 'x'
    if lbl in AX_Y: return 'y'
    return None

static_variants = ('Static X Axis', 'Static Y Axis')

def axis_resolve(cal_sub, base_sub):
    """Объединение информации об оси из CAL (адрес/elements) и базы (scaling/elements)."""
    info = dict(count=0, addr=None, sc=None, stat=None)
    for src in (cal_sub, base_sub):
        if src is None:
            continue
        if info['count'] == 0 and src.get('elements'):
            info['count'] = int(src.get('elements'))
        data = src.findall('data')
        if data and info['stat'] is None:
            vals = []
            for d in data:
                try:
                    vals.append(float((d.text or '0').strip().split()[0]))
                except (ValueError, IndexError):
                    vals.append(0.0)
            if info['count'] == 0:
                info['count'] = len(vals)
            info['stat'] = vals
        if info['addr'] is None and src.get('address'):
            try:
                info['addr'] = int(src.get('address'), 16)
            except ValueError:
                pass
        if info['sc'] is None:
            sc = src.find('scaling')
            if sc is None and src.get('scaling') and src.get('scaling') in scalings:
                sc = scalings[src.get('scaling')]
            if sc is not None:
                info['sc'] = sc
    return info

def sc_fields(sc):
    if sc is None:
        return None
    storage = (sc.get('storagetype') or 'float').lower()
    if storage not in ('float', 'uint8', 'int8', 'uint16', 'int16'):
        storage = 'float'
    return dict(
        storage=storage,
        endian=(sc.get('endian') or 'big').lower(),
        units=sc.get('units') or '',
        to=dart_body(sc.get('toexpr') or 'x'),
        fr=dart_body(sc.get('frexpr') or ''),
        vmin=fnum(sc.get('min')),
        vmax=fnum(sc.get('max')),
    )

def table_scaling(t, base):
    sc = t.find('scaling')
    if sc is None:
        ref = t.get('scaling') or (base.get('scaling') if base is not None else None)
        sc = scalings.get(ref) if ref else None
    return sc

all_tables = []
skipped = []
for name_raw, t in tmap.items():
    # карты без адреса (шаблоны базы) не тащим
    try:
        addr = int(t.get('address') or '', 16)
    except ValueError:
        skipped.append((name_raw, 'no address'))
        continue
    base = base_tables.get(norm_name(name_raw))
    cal_subs = {}
    for s in t.findall('table'):
        k = sub_kind(s)
        if k and k not in cal_subs:
            cal_subs[k] = s
    base_subs = {}
    if base is not None:
        for s in base.findall('table'):
            k = sub_kind(s)
            if k and k not in base_subs:
                base_subs[k] = s

    ttype = (t.get('type') or (base.get('type') if base is not None else '') or '').strip()
    if ttype not in ('1D', '2D', '3D'):
        # выводим по подосям (компактный SubMerp-формат)
        has_x, has_y = ('x' in cal_subs), ('y' in cal_subs)
        if has_x and has_y:
            ttype = '3D'
        elif has_x or has_y:
            ttype = '2D'
        else:
            ttype = '1D'
    cat = t.get('category') or (base.get('category') if base is not None else '') or 'other'
    swap = ((t.get('swapxy') or (base.get('swapxy') if base is not None else '') or 'false').lower() == 'true')

    sc = table_scaling(t, base)
    scf = sc_fields(sc)
    if scf is None or scf['to'] is None:
        skipped.append((name_raw, 'no scaling'))
        continue

    def build_axis(kind_label):
        cal_s = cal_subs.get(kind_label)
        base_s = base_subs.get(kind_label)
        if base_s is None and kind_label == 'x' and 'y' in base_subs:
            base_s = base_subs['y']  # у 2D базы может быть "одна" ось
        return axis_resolve(cal_s, base_s)

    x_el = y_el = None
    if ttype == '3D' or ttype == '2D':
        x_el = cal_subs.get('x') or cal_subs.get('y')  # у 2D в дампе ось бывает подписана "Y"
        y_el = cal_subs.get('y') if ttype == '3D' else None
        if x_el is None and 'x' in base_subs:
            x_el = None  # возьмём из base_subs позже
    # решаем оси
    if ttype == '1D':
        rows, cols = 1, 1
        xd, yd = 'null', 'null'
        xs, ys = 'null', 'null'
    else:
        ax_info = axis_resolve(cal_subs.get('x') or cal_subs.get('y'),
                               base_subs.get('x') or base_subs.get('y'))
        ay_info = axis_resolve(cal_subs.get('y'), base_subs.get('y')) if ttype == '3D' else None
        cols = ax_info['count']
        rows = (ay_info['count'] if ay_info else 1)

        def axis_dart(a):
            if a is None or a['count'] == 0:
                return 'null'
            if a['stat'] is not None and a['addr'] is None:
                return "RomCol(address: -1, count: %d, storage: 'static')" % a['count']
            if a['addr'] is None:
                return 'null'
            f = sc_fields(a['sc'])
            stx = f['storage'] if f else 'float'
            enx = f['endian'] if f else 'big'
            body = (f['to'] if f and f['to'] else 'x')
            return ("RomCol(address: 0x%X, count: %d, storage: '%s', endian: '%s', "
                    "to: (x) => (%s))" % (a['addr'], a['count'], stx, enx, body))

        def axis_vals(a):
            if a is not None and a['stat'] is not None:
                return '[' + ','.join(('%g' % v) for v in a['stat']) + ']'
            return 'null'

        xd, yd = axis_dart(ax_info), axis_dart(ay_info)
        xs, ys = axis_vals(ax_info), axis_vals(ay_info)
        if cols == 0 or rows == 0:
            skipped.append((name_raw, 'axis count %dx%d' % (rows, cols)))
            continue
    if rows * cols > 2048:
        skipped.append((name_raw, 'too big %dx%d' % (rows, cols)))
        continue

    all_tables.append(dict(
        name=name_raw, cat=cat, addr=addr, rows=rows, cols=cols, swapxy=swap,
        units=scf['units'], storage=scf['storage'], endian=scf['endian'],
        to=scf['to'], fr=('(x) => (%s)' % scf['fr']) if scf['fr'] else 'null',
        vmin=scf['vmin'], vmax=scf['vmax'], x=xd, y=yd, xs=xs, ys=ys,
        ttype=ttype))

n1 = sum(1 for t in all_tables if t['ttype'] == '1D')
n2 = sum(1 for t in all_tables if t['ttype'] == '2D')
n3 = sum(1 for t in all_tables if t['ttype'] == '3D')
print('  Карт распознано: %d (1D: %d · 2D: %d · 3D: %d) | пропущено: %d'
      % (len(all_tables), n1, n2, n3, len(skipped)))
for c, n in sorted(Counter(t['cat'] for t in all_tables).items(), key=lambda x: -x[1])[:16]:
    print('    %-30s %d' % (c, n))
if skipped:
    print('  примеры пропусков: ' + '; '.join('%s(%s)' % s for s in skipped[:10]))
# контрольные карты
for probe in ['targetboost', 'basetimingprimarycruise', 'primaryopenloopfueling',
              'knockcorrectionadvancemaxcruise', 'maxwastegateduty']:
    hit = next((t for t in all_tables if norm_name(t['name']) == probe), None)
    if hit:
        print('  ✓ %s: %dx%d [%s] @ 0x%X, %s' % (hit['name'], hit['rows'], hit['cols'],
                                                   hit['storage'], hit['addr'], hit['units']))

if len(all_tables) < 20:
    raise SystemExit('ОШИБКА: распознано только %d карт (цепочка %s).'
                     % (len(all_tables), ' -> '.join(chain)))

print()
print('=' * 64)
print('  [4/4] Генерация subaru_rom.g.dart')
print('=' * 64)

rd = []
rd.append('''// АВТОСГЕНЕРИРОВАНО ячейкой 4/10 SUBA RUN V8
// ECUFlash %s (цепочка: %s) — SH7058, subarucan
// ignore_for_file: constant_identifier_names
import '../models/rom_table.dart';

class SubaruRom {
  static const String calId = '%s';
  static const int romSize = %d;

  static final List<RomTableDef> tables = [
''' % (TARGET_CAL, ' / '.join(chain), TARGET_CAL, rom_bytes))
for i, t in enumerate(all_tables):
    rd.append("    RomTableDef(\n"
              "      name: '%s', category: '%s', address: 0x%X,\n"
              "      rows: %d, cols: %d, swapxy: %s,\n"
              "      units: '%s', minHint: %s, maxHint: %s,\n"
              "      data: RomCol(address: 0x%X, count: %d, storage: '%s', endian: '%s',\n"
              "          to: (x) => (%s), fr: %s),\n"
              "      xAxis: %s, yAxis: %s),\n"
              % (esc(t['name']), esc(t['cat']), t['addr'],
                 t['rows'], t['cols'], str(t['swapxy']).lower(),
                 esc(t['units']), t['vmin'], t['vmax'],
                 t['addr'], t['rows'] * t['cols'], t['storage'], t['endian'],
                 t['to'], t['fr'], t['x'], t['y']))
rd.append('''  ];

  /// Статические оси (Static Axis): ключ = x/y + индекс в tables
  static final Map<String, List<double>> staticAxes = {
''')
for i, t in enumerate(all_tables):
    if t['xs'] != 'null':
        rd.append("    'x%d': %s,\n" % (i, t['xs']))
    if t['ys'] != 'null':
        rd.append("    'y%d': %s,\n" % (i, t['ys']))
rd.append('''  };
}
''')
with open('lib/generated/subaru_rom.g.dart', 'w') as f:
    f.write(''.join(rd))

report = dict(pids=len(pids), canon=by_canon, tables=len(all_tables), chain=chain,
              types={'1D': n1, '2D': n2, '3D': n3},
              categories=dict(Counter(t['cat'] for t in all_tables)),
              skipped=skipped[:40], romBytes=rom_bytes)
with open('/content/v8_definitions_report.json', 'w') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print()
print('=' * 64)
print('  СГЕНЕРИРОВАНО: %d PID + %d карт (%s)' % (len(pids), len(all_tables), ' -> '.join(chain)))
print('  Далее -> ячейка 5/10 (сервисы ядра)')
print('=' * 64)


  [1/4] Источники определений
  загрузка logger.xml ...
  /content/logger.xml: 2026 KB
  загрузка A2TB100B.xml ...
  /content/A2TB100B.xml: 2 KB
  загрузка A2TB100K.xml ...
  /content/A2TB100K.xml: 28 KB
  загрузка 32BITBASE.xml ...
  /content/32BITBASE.xml: 489 KB

  [2/4] RomRaider logger.xml -> subaru_pids.g.dart
  PID всего: 150 (P: 91, E: 59) · канонических: 19

  [3/4] ECUFlash: мерж CAL-файлов с базой 32BITBASE
  скейлингов в реестре: 399 · шаблонных таблиц: 813
  цепочка: A2TB100B -> A2TB100K -> 32BITBASE · кандидатов: 807 · CPU SH7058
  Карт распознано: 289 (1D: 104 · 2D: 143 · 3D: 42) | пропущено: 518
    Diagnostic Trouble Codes       103
    Ignition Timing - Knock Control 23
    Fueling - Warm-Up Enrichment   16
    Ignition Timing - Compensation 14
    Fueling - CL/OL Transition     13
    Boost Control - Turbo Dynamics 12
    Ignition Timing - Advance      12
    Fueling - Cranking             9
    Fueling - Tip-in Enrichment    9
    Drive-by-Wire Throttle (DBW)   9
  

/tmp/ipykernel_2102/1753225597.py:497: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  x_el = cal_subs.get('x') or cal_subs.get('y')  # у 2D в дампе ось бывает подписана "Y"
/tmp/ipykernel_2102/1753225597.py:507: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  ax_info = axis_resolve(cal_subs.get('x') or cal_subs.get('y'),
/tmp/ipykernel_2102/1753225597.py:508: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  base_subs.get('x') or base_subs.get('y'))


In [ ]:
# @title 🧰 Ячейка 5/10: Сервисы ядра (настройки / логгер / алерты / ROM / анализатор)
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/services/settings_service.dart ============
with open('lib/services/settings_service.dart', 'w') as f:
    f.write('''import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';

class SettingsService extends ChangeNotifier {
  static final SettingsService I = SettingsService._();
  SettingsService._();

  String btAddress = '';
  String btName = '';
  Set<String> enabledIds = {};
  int maxBlock = 0x50;
  int gapTol = 2;
  int midEveryN = 4;
  int slowEveryN = 25;
  int stCode = 8;
  bool autoLog = true;
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    btAddress = p.getString('btAddress') ?? '';
    btName = p.getString('btName') ?? '';
    enabledIds = (p.getStringList('enabledIds') ?? SubaruPids.defaults.map((e) => e.id).toList()).toSet();
    maxBlock = p.getInt('maxBlock') ?? 0x50;
    gapTol = p.getInt('gapTol') ?? 2;
    midEveryN = p.getInt('midEveryN') ?? 4;
    slowEveryN = p.getInt('slowEveryN') ?? 25;
    stCode = p.getInt('stCode') ?? 8;
    autoLog = p.getBool('autoLog') ?? true;
    notifyListeners();
  }

  Future<void> save() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('btAddress', btAddress);
    await p.setString('btName', btName);
    await p.setStringList('enabledIds', enabledIds.toList());
    await p.setInt('maxBlock', maxBlock);
    await p.setInt('gapTol', gapTol);
    await p.setInt('midEveryN', midEveryN);
    await p.setInt('slowEveryN', slowEveryN);
    await p.setInt('stCode', stCode);
    await p.setBool('autoLog', autoLog);
    notifyListeners();
  }

  List<SubaruPid> get selectedPids {
    final list = SubaruPids.all.where((p) => enabledIds.contains(p.id)).toList();
    if (list.isEmpty) return SubaruPids.defaults;
    return list;
  }
}
''')
print('OK  settings_service.dart')

# ============ lib/services/logger_service.dart ============
with open('lib/services/logger_service.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:io';

import 'package:csv/csv.dart';
import 'package:intl/intl.dart';
import 'package:path_provider/path_provider.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/live_snapshot.dart';

/// CSV-логгер. Исправлено относительно V6: IOSink всегда в try/catch,
/// flush дозированный, подписка корректно закрывается (нет утечек).
class LoggerService {
  StreamSubscription<LiveSnapshot>? _sub;
  IOSink? _sink;
  File? _file;
  int rows = 0;
  DateTime? startedAt;
  List<SubaruPid> _cols = [];
  bool get logging => _sink != null;
  String? get filePath => _file?.path;
  Timer? _flushTimer;

  Future<void> start(Stream<LiveSnapshot> stream, List<SubaruPid> cols) async {
    if (logging) return;
    _cols = cols;
    final dir = await getApplicationDocumentsDirectory();
    final logsDir = Directory('${dir.path}/logs');
    if (!logsDir.existsSync()) logsDir.createSync(recursive: true);
    final name = 'V8_${DateFormat('yyyyMMdd_HHmmss').format(DateTime.now())}.csv';
    _file = File('${logsDir.path}/$name');
    _sink = _file!.openWrite();
    rows = 0;
    startedAt = DateTime.now();
    _sink!.writeln(const ListToCsvConverter().convert([
      <String>['ts_ms', ...cols.map((c) => c.id), 'boostErr', 'fuelLph', 'mode']
    ]));
    _sub = stream.listen(_write, onError: (_) {});
    _flushTimer = Timer.periodic(const Duration(seconds: 2), (_) {
      try { _sink?.flush(); } catch (_) {}
    });
  }

  void _write(LiveSnapshot s) {
    final sk = _sink;
    if (sk == null) return;
    try {
      sk.writeln(const ListToCsvConverter().convert([
        <Object>[
          s.ts.millisecondsSinceEpoch,
          ..._cols.map((c) {
            final v = s.byId[c.id];
            return v == null ? '' : v.toStringAsFixed(4);
          }),
          s.has('tboost') ? s.boostErr.toStringAsFixed(3) : '',
          s.fuelLph.toStringAsFixed(3),
          s.mode,
        ]
      ]));
      rows++;
    } catch (_) {}
  }

  Future<String?> stop() async {
    _flushTimer?.cancel();
    await _sub?.cancel();
    _sub = null;
    try {
      await _sink?.flush();
      await _sink?.close();
    } catch (_) {}
    _sink = null;
    return _file?.path;
  }

  static Future<List<FileSystemEntity>> listLogs() async {
    final dir = await getApplicationDocumentsDirectory();
    final logsDir = Directory('${dir.path}/logs');
    if (!logsDir.existsSync()) return [];
    final list = logsDir.listSync().where((e) => e.path.endsWith('.csv')).toList()
      ..sort((a, b) => b.statSync().modified.compareTo(a.statSync().modified));
    return list;
  }

  /// Парсинг CSV обратно в строки canon->value (для анализатора «Из лога»)
  static Future<List<Map<String, double>>> parseLog(String path) async {
    final out = <Map<String, double>>[];
    try {
      final content = await File(path).readAsString();
      final rows = const CsvToListConverter(shouldParseNumbers: false).convert(content);
      if (rows.length < 2) return out;
      final head = rows.first.map((e) => e.toString()).toList();
      final canonIdx = <int, String>{};
      for (var i = 0; i < head.length; i++) {
        final h = head[i];
        if (h == 'boostErr' || h == 'fuelLph') {
          canonIdx[i] = h;
        } else {
          final pid = SubaruPids.byId(h);
          if (pid != null && pid.canon.isNotEmpty) canonIdx[i] = pid.canon;
        }
      }
      for (var r = 1; r < rows.length; r++) {
        final row = rows[r];
        final m = <String, double>{};
        canonIdx.forEach((i, key) {
          if (i < row.length) {
            final v = double.tryParse(row[i].toString());
            if (v != null) m[key] = v;
          }
        });
        if (m.isNotEmpty) out.add(m);
      }
    } catch (_) {}
    return out;
  }
}
''')
print('OK  logger_service.dart')

# ============ lib/services/alert_service.dart ============
with open('lib/services/alert_service.dart', 'w') as f:
    f.write('''import 'dart:async';

import '../constants.dart';
import '../models/live_snapshot.dart';

class AlertEvent {
  final DateTime ts;
  final int level; // 1 warn, 2 danger
  final String code;
  final String text;
  final Map<String, double> snapshot;
  AlertEvent(this.ts, this.level, this.code, this.text, this.snapshot);
}

/// Турбо-алерты Subaru: FBKC/FKL/IAM/буст — то, что реально важно на EJ20X,
/// а не Nissan-ориентированные knockRetard/VTC.
class AlertService {
  final List<AlertEvent> events = [];
  final _ctl = StreamController<AlertEvent>.broadcast();
  Stream<AlertEvent> get stream => _ctl.stream;
  final Map<String, DateTime> _cooldown = {};

  bool _cool(String code, [int sec = 5]) {
    final now = DateTime.now();
    final lastT = _cooldown[code];
    if (lastT != null && now.difference(lastT).inSeconds < sec) return false;
    _cooldown[code] = now;
    return true;
  }

  void _fire(int level, String code, String text, LiveSnapshot s) {
    if (!_cool(code)) return;
    final e = AlertEvent(s.ts, level, code, text, Map.of(s.c));
    events.insert(0, e);
    if (events.length > 200) events.removeLast();
    if (!_ctl.isClosed) _ctl.add(e);
  }

  void check(LiveSnapshot s) {
    if (s.rpm > 2500 && s.underBoost) {
      if (s.has('fbkc') && s.fbkc <= AppConstants.fbkcDanger) {
        _fire(2, 'FBKC', 'FBKC ${s.fbkc.toStringAsFixed(1)}° при ${s.boost.toStringAsFixed(2)} бар — детонация!', s);
      } else if (s.has('fbkc') && s.fbkc <= AppConstants.fbkcWarn) {
        _fire(1, 'FBKC', 'FBKC ${s.fbkc.toStringAsFixed(1)}° — лёгкая коррекция', s);
      }
      if (s.has('fkl') && s.fkl <= AppConstants.fklDanger) {
        _fire(2, 'FKL', 'FKL ${s.fkl.toStringAsFixed(1)}° — обученная коррекция, проверь топливо/настройку', s);
      }
      if (s.has('afr') && s.afr >= AppConstants.afrBoostLean) {
        _fire(2, 'AFR', 'Смесь ${s.afr.toStringAsFixed(1)} под бустом! Опасно бедно', s);
      }
    }
    if (s.has('iam') && s.iam < AppConstants.iamDanger) {
      _fire(2, 'IAM', 'IAM упал до ${s.iam.toStringAsFixed(2)} — ECU режет углы', s);
    } else if (s.has('iam') && s.iam < AppConstants.iamWarn) {
      _fire(1, 'IAM', 'IAM ${s.iam.toStringAsFixed(2)} < 1.0', s);
    }
    if (s.has('tboost') && s.tps > 50) {
      final err = s.boostErr;
      if (err >= AppConstants.boostErrDanger) {
        _fire(2, 'BSTERR', 'Перебуст: +${err.toStringAsFixed(2)} бар к цели', s);
      } else if (err <= -AppConstants.boostErrDanger) {
        _fire(1, 'BSTLOW', 'Недобуст: ${err.toStringAsFixed(2)} бар к цели', s);
      }
    }
    if (s.has('boost') && s.boost >= AppConstants.overboostDanger) {
      _fire(2, 'OVBST', 'Овербуст ${s.boost.toStringAsFixed(2)} бар!', s);
    }
    if (s.ect >= AppConstants.ectDanger) {
      _fire(2, 'ECT', 'Температура ОЖ ${s.ect.toStringAsFixed(0)}°C!', s);
    } else if (s.ect >= AppConstants.ectWarn) {
      _fire(1, 'ECT', 'Температура ОЖ ${s.ect.toStringAsFixed(0)}°C', s);
    }
  }

  void clear() => events.clear();
}
''')
print('OK  alert_service.dart')

# ============ lib/services/rom_service.dart ============
with open('lib/services/rom_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:typed_data';

import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';

class RomService {
  List<int>? rom;
  String fileName = '';
  ByteData? _bd;

  bool get loaded => rom != null;
  List<RomTableDef> get defs => SubaruRom.tables;

  Future<String?> pickAndLoad() async {
    final res = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['bin', 'hex', 'rom']);
    if (res == null || res.files.single.path == null) return null;
    final bytes = await File(res.files.single.path!).readAsBytes();
    return loadBytes(bytes, res.files.single.name);
  }

  String? loadBytes(List<int> bytes, String name) {
    if (bytes.length < 0x40000) return 'Файл слишком мал для SH7058 ROM';
    rom = bytes;
    fileName = name;
    _bd = ByteData.sublistView(Uint8List.fromList(bytes));
    return null; // null = ок
  }

  List<double> _readAxis(RomCol? col, int tableIndex, bool isX, int fallbackCount) {
    if (col == null) return List<double>.generate(fallbackCount, (i) => i.toDouble());
    if (col.storage == 'static') {
      return SubaruRom.staticAxes['${isX ? 'x' : 'y'}$tableIndex'] ??
          List<double>.generate(fallbackCount, (i) => i.toDouble());
    }
    final values = <double>[];
    for (var i = 0; i < col.count; i++) {
      try {
        values.add(col.value(rom!, _bd!, i));
      } catch (_) {
        values.add(double.nan);
      }
    }
    return values;
  }

  RomTable readTable(RomTableDef def) {
    final idx = SubaruRom.tables.indexOf(def);
    final xValues = _readAxis(def.xAxis, idx, true, def.cols);
    final yValues = _readAxis(def.yAxis, idx, false, def.rows);
    final z = List.generate(def.rows, (_) => List<double>.filled(def.cols, 0));
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        try {
          z[r][c] = def.data.value(rom!, _bd!, i);
        } catch (_) {
          z[r][c] = double.nan;
        }
      }
    }
    return RomTable(def: def, xValues: xValues, yValues: yValues, z: z);
  }

  /// Запись правок в НОВЫЙ файл (оригинал не трогаем) — как NLP_MOD1 в V6
  Future<String?> saveMod(RomTable table) async {
    if (rom == null || !table.def.editable) return null;
    final mod = List<int>.of(rom!);
    final fr = table.def.data.fr!;
    final sizeOf = table.def.data.sizeOf;
    for (var r = 0; r < table.def.rows; r++) {
      for (var c = 0; c < table.def.cols; c++) {
        final i = table.def.swapxy ? (c * table.def.rows + r) : (r * table.def.cols + c);
        final off = table.def.address + i * sizeOf;
        final raw = fr(table.z[r][c]).round();
        switch (table.def.data.storage) {
          case 'uint16':
            mod[off] = (raw >> 8) & 0xFF; mod[off + 1] = raw & 0xFF; break;
          case 'int16':
            final v = raw < 0 ? raw + 65536 : raw;
            mod[off] = (v >> 8) & 0xFF; mod[off + 1] = v & 0xFF; break;
          case 'int8':
            mod[off] = raw < 0 ? raw + 256 : raw; break;
          default:
            mod[off] = raw.clamp(0, 255);
        }
      }
    }
    final dir = await getApplicationDocumentsDirectory();
    final base = fileName.replaceAll(RegExp(r'\.(bin|hex|rom)$', caseSensitive: false), '');
    // ignore: avoid_escaping_inner_quotes
    final f = File('${dir.path}/V8MOD_$base.bin');
    await f.writeAsBytes(mod);
    return f.path;
  }
}
''')
print('OK  rom_service.dart')

# ============ lib/services/analyzer_service.dart ============
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:math' as math;

import '../models/live_snapshot.dart';

/// Ось анализатора: источник значения + диапазон + число корзин
class AxisOpt {
  final String key;
  final String label;
  final double min;
  final double max;
  final int bins;
  const AxisOpt(this.key, this.label, this.min, this.max, this.bins);
}

/// Тепловая сетка "как в V6": биннинг X×Y, усреднение, счётчик попаданий,
/// слияние сеток (сплиттер логов) и живой режим из потока опроса.
class HeatGrid {
  final int cols;
  final int rows;
  final double xmin, xmax, ymin, ymax;
  late List<List<double>> sum;
  late List<List<int>> n;

  HeatGrid(this.cols, this.rows, this.xmin, this.xmax, this.ymin, this.ymax) {
    reset();
  }

  void reset() {
    sum = List.generate(rows, (_) => List.filled(cols, 0.0));
    n = List.generate(rows, (_) => List.filled(cols, 0));
  }

  void add(double x, double y, double v) {
    if (v.isNaN || x.isNaN || y.isNaN) return;
    if (x < xmin || x > xmax || y < ymin || y > ymax) return;
    final c = (((x - xmin) / (xmax - xmin)) * cols).floor().clamp(0, cols - 1);
    final r = (((y - ymin) / (ymax - ymin)) * rows).floor().clamp(0, rows - 1);
    sum[r][c] += v;
    n[r][c]++;
  }

  void merge(HeatGrid other) {
    if (other.cols != cols || other.rows != rows) return;
    for (var r = 0; r < rows; r++) {
      for (var c = 0; c < cols; c++) {
        sum[r][c] += other.sum[r][c];
        n[r][c] += other.n[r][c];
      }
    }
  }

  double? avg(int r, int c) => n[r][c] > 0 ? sum[r][c] / n[r][c] : null;

  int get totalHits => n.expand((e) => e).fold(0, (a, b) => a + b);

  ({double lo, double hi}) range({double? hintLo, double? hintHi}) {
    var lo = double.infinity, hi = -double.infinity;
    for (var r = 0; r < rows; r++) {
      for (var c = 0; c < cols; c++) {
        final a = avg(r, c);
        if (a != null) { lo = math.min(lo, a); hi = math.max(hi, a); }
      }
    }
    if (lo == double.infinity) return (lo: hintLo ?? 0, hi: hintHi ?? 1);
    return (lo: lo, hi: hi == lo ? lo + 1 : hi);
  }

  String xLabel(int c) => (xmin + (c + 0.5) * (xmax - xmin) / cols).toStringAsFixed(0);
  String yLabel(int r) => (ymin + (rows - r - 0.5) * (ymax - ymin) / rows).toStringAsFixed(2);
}

class AnalyzerService {
  static const List<AxisOpt> axisX = [
    AxisOpt('rpm', 'Обороты, об/мин', 400, 8000, 16),
    AxisOpt('maf', 'MAF, г/с', 0, 200, 16),
    AxisOpt('tps', 'Дроссель, %', 0, 100, 10),
    AxisOpt('speed', 'Скорость, км/ч', 0, 200, 16),
  ];
  static const List<AxisOpt> axisY = [
    AxisOpt('boost', 'Буст, бар', -0.65, 1.5, 14),
    AxisOpt('maf', 'MAF, г/с', 0, 200, 14),
    AxisOpt('tps', 'Дроссель, %', 0, 100, 10),
    AxisOpt('injms', 'Время впрыска, мс', 0, 20, 14),
  ];

  static const Map<String, String> metricNames = {
    'kca': 'УОЗ итоговый (KCA)',
    'fbkc': 'FBKC, °',
    'fkl': 'FKL, °',
    'iam': 'IAM',
    'boost': 'Буст, бар',
    'boostErr': 'Ошибка буста, бар',
    'afr': 'AFR',
    'maf': 'MAF, г/с',
    'wgd': 'Wastegate duty, %',
    'avcs': 'AVCS впуск, °',
    'stft': 'STFT (AF Corr), %',
    'ltft': 'LTFT (AF Learn), %',
    'fuelLph': 'Расход, л/ч',
    'iat': 'Темп. впуска, °C',
    'ect': 'Темп. ОЖ, °C',
    'injms': 'Время впрыска, мс',
    'tps': 'Дроссель, %',
    'speed': 'Скорость, км/ч',
    'ect2': '—',
  };

  AxisOpt xOpt = axisX.first;
  AxisOpt yOpt = axisY.first;
  String metric = 'kca';
  HeatGrid grid = HeatGrid(axisX.first.bins, axisY.first.bins,
      axisX.first.min, axisX.first.max, axisY.first.min, axisY.first.max);

  StreamSubscription<LiveSnapshot>? _sub;
  bool get online => _sub != null;

  void setAxes(AxisOpt x, AxisOpt y, String m) {
    xOpt = x; yOpt = y; metric = m;
    grid = HeatGrid(x.bins, y.bins, x.min, x.max, y.min, y.max);
  }

  double _val(Map<String, double> c, String k) {
    if (k == 'boostErr') {
      final b = c['boost'], t = c['tboost'];
      return (b != null && t != null) ? b - t : double.nan;
    }
    if (k == 'fuelLph') {
      final maf = c['maf'], afr = c['afr'];
      if (maf == null || afr == null || afr <= 0) return double.nan;
      return maf / afr * 3600.0 / 745.0;
    }
    return c[k] ?? double.nan;
  }

  void addRow(Map<String, double> c) {
    final v = _val(c, metric);
    if (v.isNaN) return;
    grid.add(_val(c, xOpt.key), _val(c, yOpt.key), v);
  }

  void startOnline(Stream<LiveSnapshot> stream) {
    stopOnline();
    _sub = stream.listen((s) => addRow(s.c));
  }

  void stopOnline() {
    _sub?.cancel();
    _sub = null;
  }

  /// «Из лога»: экран вызывает LoggerService.parseLog(path) и скармливает
  /// строки через addRow(). Здесь — только построение из готовых строк.
  void buildFromRows(List<Map<String, double>> rows) {
    for (final r in rows) {
      addRow(r);
    }
  }
}
''')
print('OK  analyzer_service.dart')
print()
print('=' * 64)
print('  Сервисы ядра готовы. Далее -> ячейка 6/10 (DTC, PID CRUD, профили, экспорт, связь)')
print('=' * 64)


OK  settings_service.dart
OK  logger_service.dart
OK  alert_service.dart
OK  rom_service.dart
OK  analyzer_service.dart

  Сервисы ядра готовы. Далее -> ячейка 6/10 (DTC, PID CRUD, профили, экспорт, связь)


In [ ]:
# @title 🧩 Ячейка 6/10: DTC на русском · PID CRUD · Профили · Экспорт · Связь
# ============================================================================
# Сервисы, которые были в V6 (17 экранов), переписанные под параметры V7/Subaru.
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/services/expr_eval.dart ============
with open('lib/services/expr_eval.dart', 'w') as f:
    f.write(r'''import 'dart:typed_data';

/// Мини-вычислитель выражений (замена math_expressions, как в V6-CRUD):
/// поддержка: числа, x, + - * /, скобки, унарный минус. RPN через shunting-yard.
class ExprEval {
  final List<String> _rpn;
  ExprEval(String expr) : _rpn = _toRpn(expr);

  static List<String> _tok(String e) {
    final out = <String>[];
    var i = 0, prevOp = true;
    while (i < e.length) {
      final ch = e[i];
      if (ch.trim().isEmpty) { i++; continue; }
      if (ch == 'x' || ch == 'X') { out.add('x'); prevOp = false; i++; continue; }
      if (RegExp(r'[0-9.]').hasMatch(ch)) {
        final m = RegExp(r'[0-9]*\.?[0-9]+([eE][+-]?[0-9]+)?').matchAsPrefix(e.substring(i));
        out.add(m!.group(0)!); prevOp = false; i += m.group(0)!.length; continue;
      }
      if ('+-*/'.contains(ch)) {
        if (ch == '-' && prevOp) { out.add('u-'); } else { out.add(ch); }
        prevOp = true; i++; continue;
      }
      if (ch == '(') { out.add('('); prevOp = true; i++; continue; }
      if (ch == ')') { out.add(')'); prevOp = false; i++; continue; }
      throw FormatException('bad char $ch');
    }
    return out;
  }

  static int _prec(String op) => op == 'u-' ? 3 : (op == '*' || op == '/' ? 2 : (op == '+' || op == '-' ? 1 : 0));

  static List<String> _toRpn(String e) {
    final out = <String>[], ops = <String>[];
    for (final t in _tok(e)) {
      if (t == 'x' || RegExp(r'^[0-9]').hasMatch(t)) {
        out.add(t);
      } else if (t == '(') {
        ops.add(t);
      } else if (t == ')') {
        while (ops.isNotEmpty && ops.last != '(') { out.add(ops.removeLast()); }
        if (ops.isEmpty) { throw const FormatException('parens'); }
        ops.removeLast();
      } else {
        while (ops.isNotEmpty && ops.last != '(' && _prec(ops.last) >= _prec(t)) {
          out.add(ops.removeLast());
        }
        ops.add(t);
      }
    }
    while (ops.isNotEmpty) {
      final o = ops.removeLast();
      if (o == '(') throw const FormatException('parens');
      out.add(o);
    }
    return out;
  }

  double call(double x) {
    final st = <double>[];
    for (final t in _rpn) {
      if (t == 'x') { st.add(x); continue; }
      final num = double.tryParse(t);
      if (num != null) { st.add(num); continue; }
      if (t == 'u-') { st.add(-st.removeLast()); continue; }
      final b = st.removeLast();
      final a = st.removeLast();
      switch (t) {
        case '+': st.add(a + b); break;
        case '-': st.add(a - b); break;
        case '*': st.add(a * b); break;
        case '/': st.add(b == 0 ? double.nan : a / b); break;
      }
    }
    return st.isEmpty ? double.nan : st.last;
  }

  static double rawOf(List<int> b, String storage) {
    switch (storage) {
      case 'float':
        return ByteData.sublistView(Uint8List.fromList(
                b.length >= 4 ? b.sublist(0, 4) : <int>[...b, ...List.filled(4 - b.length, 0)]))
            .getFloat32(0, Endian.big);
      case 'int8':
        return (b[0] < 128 ? b[0] : b[0] - 256).toDouble();
      case 'uint16':
        return ((b[0] << 8) | b[1]).toDouble();
      case 'int16':
        var v = (b[0] << 8) | b[1];
        if (v >= 32768) v -= 65536;
        return v.toDouble();
      case 'uint8':
      default:
        return b[0].toDouble();
    }
  }

  static double Function(List<int>) closure(String expr, String storage) {
    final ev = ExprEval(expr);
    return (b) {
      try {
        final v = ev(rawOf(b, storage));
        return v;
      } catch (_) {
        return double.nan;
      }
    };
  }
}
''')
print('OK  expr_eval.dart')

# ============ lib/services/dtc_service.dart ============
with open('lib/services/dtc_service.dart', 'w') as f:
    f.write(r'''import '../ssm/ssm_elm.dart';

class DtcItem {
  final String code;
  final String text;
  DtcItem(this.code, this.text);
}

/// DTC через стандартный OBD-II (режим 03/04) — на CAN-Субару работает
/// на той же шине 0x7E0 параллельно с SSM2. Описания — Subaru-специфичные.
class DtcService {
  static const Map<String, String> db = {
    'P0011': 'AVCS впуск (банк 1): синхронизация опережения',
    'P0021': 'AVCS впуск (банк 2): синхронизация опережения',
    'P0030': 'Подогрев датчика O2 (б1 д1): цепь',
    'P0031': 'Подогрев A/F датчика (б1 д1): низкий уровень',
    'P0032': 'Подогрев A/F датчика (б1 д1): высокий уровень',
    'P0037': 'Подогрев O2 (б1 д2): низкий уровень',
    'P0038': 'Подогрев O2 (б1 д2): высокий уровень',
    'P0101': 'MAF: диапазон/производительность',
    'P0102': 'MAF: низкий сигнал',
    'P0103': 'MAF: высокий сигнал',
    'P0112': 'IAT: низкий сигнал',
    'P0113': 'IAT: высокий сигнал',
    'P0117': 'ECT: низкий сигнал',
    'P0118': 'ECT: высокий сигнал',
    'P0121': 'TPS: диапазон/производительность',
    'P0122': 'TPS: низкий сигнал',
    'P0123': 'TPS: высокий сигнал',
    'P0130': 'O2 датчик (б1 д1): цепь',
    'P0131': 'A/F датчик (б1 д1): низкое напряжение',
    'P0132': 'A/F датчик (б1 д1): высокое напряжение',
    'P0171': 'Система слишком бедная (банк 1)',
    'P0172': 'Система слишком богатая (банк 1)',
    'P0244': 'Соленоид wastegate «A»: диапазон/производительность',
    'P0245': 'Соленоид wastegate «A»: низкий уровень',
    'P0246': 'Соленоид wastegate «A»: высокий уровень',
    'P0301': 'Пропуски зажигания: цилиндр 1',
    'P0302': 'Пропуски зажигания: цилиндр 2',
    'P0303': 'Пропуски зажигания: цилиндр 3',
    'P0304': 'Пропуски зажигания: цилиндр 4',
    'P0327': 'Датчик детонации (б1): низкий сигнал',
    'P0328': 'Датчик детонации (б1): высокий сигнал',
    'P0335': 'Датчик коленвала (CKP): цепь',
    'P0340': 'Датчик распредвала (CMP): цепь',
    'P0420': 'Катализатор: эффективность ниже порога (б1)',
    'P0500': 'Датчик скорости авто (VSS): цепь',
    'P0604': 'ECU: ошибка внутренней памяти (RAM)',
    'P0607': 'ECU: производительность модуля управления',
    'P0851': 'Датчик нейтрали: низкий сигнал',
    'P0852': 'Датчик нейтрали: высокий сигнал',
    'P1443': 'EVAP: клапан вентиляции — цепь',
    'P2004': 'TGV заслонки (банк 1): заклинило открытыми',
    'P2006': 'TGV заслонки (банк 1): заклинило закрытыми',
    'P2008': 'TGV: электроцепь (банк 1)',
    'P2011': 'TGV: цепь управления (банк 2)',
    'P2016': 'TGV датчик положения (б1): низкий',
    'P2021': 'TGV датчик положения (б2): низкий',
    'P2088': 'OCV AVCS (банк 1): низкий уровень',
    'P2090': 'OCV AVCS (банк 1): низкий уровень цепи',
    'P2091': 'OCV AVCS (банк 1): высокий уровень цепи',
    'P2101': 'ETC привод дросселя: диапазон/производительность',
    'P2111': 'ETC дроссель: заклинил открытым',
    'P2119': 'ETC дроссель: диапазон/производительность',
    'P2122': 'APP датчик педали «D»: низкий сигнал',
    'P2123': 'APP датчик педали «D»: высокий сигнал',
    'P2127': 'APP датчик педали «E»: низкий сигнал',
    'P2128': 'APP датчик педали «E»: высокий сигнал',
    'P2135': 'TPS сенсоры «A»/«B»: рассогласование',
    'P2138': 'APP сенсоры «D»/«E»: рассогласование',
    'P2226': 'BARO датчик: цепь',
    'P2227': 'BARO датчик: диапазон/производительность',
    'U0073': 'CAN: модуль отключён от шины (bus off)',
    'U0101': 'Нет связи с TCM',
    'U0122': 'Нет связи с VDC/ABS',
  };

  /// Разбор кадров режима 03: ищем все вхождения '43', читаем пары байт
  static List<DtcItem> parseMode03(String resp) {
    final hex = resp.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
    final out = <DtcItem>[];
    var from = 0;
    while (true) {
      final idx = hex.indexOf('43', from);
      if (idx < 0 || idx + 2 >= hex.length) break;
      var p = idx + 2;
      while (p + 4 <= hex.length) {
        final b1 = int.tryParse(hex.substring(p, p + 2), radix: 16) ?? 0;
        final b2 = int.tryParse(hex.substring(p + 2, p + 4), radix: 16) ?? 0;
        p += 4;
        if (b1 == 0 && b2 == 0) break;
        const sys = ['P', 'C', 'B', 'U'];
        final code = '${sys[(b1 >> 6) & 3]}${(b1 >> 4) & 3}${(b1 & 0xF).toRadixString(16).toUpperCase()}'
            '${b2.toRadixString(16).padLeft(2, '0').toUpperCase()}';
        out.add(DtcItem(code, db[code] ?? 'нет описания в базе V8'));
        if (out.length > 40) return out;
      }
      from = idx + 2;
    }
    return out;
  }

  static Future<List<DtcItem>> read(SsmElm elm) async {
    final r = await elm.transact('03', timeoutMs: 1800);
    return parseMode03(r);
  }

  static Future<bool> clear(SsmElm elm) async {
    final r = await elm.transact('04', timeoutMs: 1800);
    return r.toUpperCase().contains('44');
  }
}
''')
print('OK  dtc_service.dart (40+ Subaru-кодов на русском)')

# ============ lib/services/custom_pid_service.dart ============
with open('lib/services/custom_pid_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';

import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';
import 'expr_eval.dart';

class CustomPid {
  String id;
  String name;
  String unit;
  String category;
  int address;
  int len;
  String storage; // uint8 | int8 | uint16 | int16 | float
  String expr;    // выражение от x
  int priority;   // 1 fast / 2 mid / 3 slow

  CustomPid({
    required this.id, required this.name, this.unit = '', this.category = 'custom',
    required this.address, this.len = 1, this.storage = 'uint8',
    this.expr = 'x', this.priority = 2,
  });

  Map<String, dynamic> toJson() => {
        'id': id, 'name': name, 'unit': unit, 'category': category,
        'address': address, 'len': len, 'storage': storage, 'expr': expr, 'priority': priority,
      };

  factory CustomPid.fromJson(Map<String, dynamic> j) => CustomPid(
        id: j['id'] as String, name: j['name'] as String,
        unit: (j['unit'] ?? '') as String, category: (j['category'] ?? 'custom') as String,
        address: (j['address'] as num).toInt(), len: (j['len'] ?? 1).toInt(),
        storage: (j['storage'] ?? 'uint8') as String,
        expr: (j['expr'] ?? 'x') as String, priority: (j['priority'] ?? 2).toInt(),
      );

  SubaruPid toSubaruPid() => SubaruPid(
        id: 'C_$id', xmlId: 'custom', name: name, desc: 'custom PID (user)',
        unit: unit, category: category.isEmpty ? 'custom' : category,
        address: address, len: len, storage: storage, priority: priority, canon: '',
        formula: ExprEval.closure(expr, storage),
      );
}

/// CRUD кастомных PID + живой тест значения (как PID-редактор в V6)
class CustomPidService extends ChangeNotifier {
  static final CustomPidService I = CustomPidService._();
  CustomPidService._();

  final List<CustomPid> items = [];
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    final raw = p.getString('customPids');
    if (raw != null && raw.isNotEmpty) {
      try {
        final list = (jsonDecode(raw) as List).cast<Map<String, dynamic>>();
        items.addAll(list.map(CustomPid.fromJson));
      } catch (_) {}
    }
  }

  Future<void> _save() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('customPids', jsonEncode(items.map((e) => e.toJson()).toList()));
    notifyListeners();
  }

  Future<void> upsert(CustomPid pid) async {
    final i = items.indexWhere((e) => e.id == pid.id);
    if (i >= 0) { items[i] = pid; } else { items.add(pid); }
    await _save();
  }

  Future<void> remove(String id) async {
    items.removeWhere((e) => e.id == id);
    await _save();
  }

  List<SubaruPid> asPids() => items.map((e) => e.toSubaruPid()).toList();
}
''')
print('OK  custom_pid_service.dart')

# ============ lib/services/profile_service.dart ============
with open('lib/services/profile_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';

import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';
import 'settings_service.dart';

/// Профили = ВСЕ настройки опроса и алертов (как в V6): набор PID,
/// параметры блоков/ярусов, калибровки. Сиды: сток A2TB100B.
class ProfileService extends ChangeNotifier {
  static final ProfileService I = ProfileService._();
  ProfileService._();

  final Map<String, Map<String, dynamic>> profiles = {};
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    final raw = p.getString('profiles');
    if (raw != null && raw.isNotEmpty) {
      try {
        final m = (jsonDecode(raw) as Map).cast<String, dynamic>();
        m.forEach((k, v) => profiles[k] = (v as Map).cast<String, dynamic>());
      } catch (_) {}
    }
    profiles.putIfAbsent('A2TB100B · сток (канон)', () => _captureDefaults());
  }

  Map<String, dynamic> _captureDefaults() => {
        'enabledIds': SubaruPids.defaults.map((e) => e.id).toList(),
        'maxBlock': 0x50, 'gapTol': 2, 'midEveryN': 4, 'slowEveryN': 25, 'stCode': 8,
        'autoLog': true,
      };

  Map<String, dynamic> _capture(SettingsService st) => {
        'enabledIds': st.enabledIds.toList(),
        'maxBlock': st.maxBlock, 'gapTol': st.gapTol,
        'midEveryN': st.midEveryN, 'slowEveryN': st.slowEveryN,
        'stCode': st.stCode, 'autoLog': st.autoLog,
      };

  Future<void> saveCurrent(String name) async {
    profiles[name] = _capture(SettingsService.I);
    await _persist();
  }

  Future<void> apply(String name) async {
    final d = profiles[name];
    if (d == null) return;
    final st = SettingsService.I;
    st.enabledIds = ((d['enabledIds'] as List?) ?? []).map((e) => e.toString()).toSet();
    st.maxBlock = (d['maxBlock'] ?? st.maxBlock) as int;
    st.gapTol = (d['gapTol'] ?? st.gapTol) as int;
    st.midEveryN = (d['midEveryN'] ?? st.midEveryN) as int;
    st.slowEveryN = (d['slowEveryN'] ?? st.slowEveryN) as int;
    st.stCode = (d['stCode'] ?? st.stCode) as int;
    st.autoLog = (d['autoLog'] ?? st.autoLog) as bool;
    await st.save();
    notifyListeners();
  }

  Future<void> remove(String name) async {
    profiles.remove(name);
    await _persist();
  }

  Future<void> _persist() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('profiles', jsonEncode(profiles));
    notifyListeners();
  }
}
''')
print('OK  profile_service.dart')

# ============ lib/services/export_service.dart ============
with open('lib/services/export_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';

import 'package:path_provider/path_provider.dart';

import 'rom_service.dart';

/// Экспорт карт и ROM (WinOLS/ecuEdit/HEX) — как экран Экспорт в V6.
class ExportService {
  /// Все карты одним CSV в стиле ecuEdit (разделитель ; для RU-Excel)
  static Future<File> exportAllTablesCsv(RomService rom) async {
    final buf = StringBuffer();
    buf.writeln('\uFEFF'); // BOM для Excel
    for (final def in rom.defs) {
      try {
        final t = rom.readTable(def);
        buf.writeln('"${def.name}";"${def.category}";${def.addrHex};${def.rows}x${def.cols};"${def.units}"');
        buf.writeln('"X\\Y";${t.xValues.map((e) => e.toStringAsFixed(1)).join(';')}');
        for (var r = 0; r < def.rows; r++) {
          buf.writeln('"${t.yValues.isNotEmpty ? t.yValues[r].toStringAsFixed(1) : r}";'
              '${t.z[r].map((e) => e.toStringAsFixed(3)).join(';')}');
        }
        buf.writeln();
      } catch (_) {
        buf.writeln('"${def.name}";"READ ERROR"');
        buf.writeln();
      }
    }
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/V8_maps_ecuEdit.csv');
    await f.writeAsString(buf.toString());
    return f;
  }

  /// WinOLS-подобный дамп отдельной карты: адрес + сырые значения сетки
  static Future<File> exportTableWinols(RomService rom, String tableName) async {
    final def = rom.defs.firstWhere((d) => d.name == tableName,
        orElse: () => throw ArgumentError('no table'));
    final t = rom.readTable(def);
    final buf = StringBuffer();
    buf.writeln(';MAP ${def.name}');
    buf.writeln(';ADDR ${def.addrHex} SIZE ${def.rows * def.cols * def.data.sizeOf}');
    buf.writeln(';AXES X=[${t.xValues.join(', ')}] Y=[${t.yValues.join(', ')}]');
    for (final row in t.toCsvRows()) {
      buf.writeln(row.join(';'));
    }
    final dir = await getApplicationDocumentsDirectory();
    final safe = tableName.replaceAll(RegExp(r'[^0-9A-Za-zА-Яа-я]+'), '_');
    final f = File('${dir.path}/V8_winols_$safe.csv');
    await f.writeAsString(buf.toString());
    return f;
  }

  /// HEX-дамп ROM (первые N байт постранично, 16 байт/строка)
  static Future<File> hexDump(List<int> rom, {int maxBytes = 0x10000}) async {
    final n = rom.length < maxBytes ? rom.length : maxBytes;
    final buf = StringBuffer();
    for (var off = 0; off < n; off += 16) {
      final chunk = rom.sublist(off, off + 16 > n ? n : off + 16);
      final hexs = chunk.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ');
      final ascii = chunk.map((b) => (b >= 32 && b < 127) ? String.fromCharCode(b) : '.').join();
      buf.writeln('${off.toRadixString(16).padLeft(8, '0')}  ${hexs.padRight(47)}  $ascii');
    }
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/V8_rom_hexdump.txt');
    await f.writeAsString(buf.toString());
    return f;
  }
}
''')
print('OK  export_service.dart')

# ============ lib/services/connection_service.dart (обновлён: эксклюзив + кастомные PID) ============
with open('lib/services/connection_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/foundation.dart';

import '../ssm/ssm_elm.dart';
import 'alert_service.dart';
import 'analyzer_service.dart';
import 'custom_pid_service.dart';
import 'logger_service.dart';
import 'rom_service.dart';
import 'settings_service.dart';

/// Центральный узел: ELM + поллер + алерты + логгер + ROM.
/// exclusive() — монопольный доступ к шине для DTC/терминала/чтения карт:
/// опрос приостанавливается, операция выполняется, опрос продолжается.
class ConnectionService extends ChangeNotifier {
  static final ConnectionService I = ConnectionService._();
  ConnectionService._();

  final SsmElm elm = SsmElm();
  final AlertService alerts = AlertService();
  final LoggerService logger = LoggerService();
  final AnalyzerService analyzer = AnalyzerService();
  final RomService rom = RomService();

  SsmPoller? poller;
  StreamSubscription? _alertSub;
  bool connecting = false;

  Future<String> connectAndInit() async {
    if (connecting) return 'уже подключаюсь...';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выбери ELM327 на экране настроек';
    connecting = true;
    notifyListeners();
    elm.stTimeoutCode = st.stCode;
    final okBt = await elm.connect(st.btAddress);
    if (!okBt) {
      connecting = false;
      notifyListeners();
      return 'Bluetooth: не удалось подключиться к ${st.btName}';
    }
    final ecuId = await elm.ecuInit();
    connecting = false;
    if (ecuId == null) {
      await elm.disconnect();
      notifyListeners();
      return 'ECU не ответил на SSM2 init (8010F001BF40)';
    }
    buildPoller();
    startPolling();
    notifyListeners();
    return 'OK: ECU ${elm.ecuId.isEmpty ? ecuId : elm.ecuId}';
  }

  /// Перестроение блочного плана: библиотека + кастомные PID
  void buildPoller() {
    final st = SettingsService.I;
    poller?.stop();
    final p = SsmPoller(elm)
      ..maxBlock = st.maxBlock
      ..gapTol = st.gapTol
      ..midEveryN = st.midEveryN
      ..slowEveryN = st.slowEveryN;
    p.buildBlocks([...st.selectedPids, ...CustomPidService.I.asPids()]);
    poller = p;
    _alertSub?.cancel();
    _alertSub = p.snapshots.listen(alerts.check);
  }

  bool get isPolling => poller?.isRunning ?? false;

  void startPolling() {
    if (poller != null && (elm.state == SsmState.ecuReady || elm.state == SsmState.polling)) {
      poller!.start();
    }
    notifyListeners();
  }

  void stopPolling() {
    poller?.stop();
    notifyListeners();
  }

  /// Монопольная операция на шине (DTC, терминал, чтение карты из ECU).
  Future<T> exclusive<T>(Future<T> Function(SsmElm elm) job) async {
    final was = isPolling;
    if (was) stopPolling();
    await Future<void>.delayed(const Duration(milliseconds: 180)); // дожать in-flight кадр
    try {
      return await job(elm);
    } finally {
      if (was) startPolling();
    }
  }

  Future<void> disconnect() async {
    stopPolling();
    if (logger.logging) await logger.stop();
    await elm.disconnect();
    notifyListeners();
  }
}
''')
print('OK  connection_service.dart (exclusive + custom PID merge)')
print()
print('=' * 64)
print('  Доп. сервисы готовы. Далее -> ячейка 7/10 (главный + приборы + графики + лог)')
print('=' * 64)


OK  expr_eval.dart
OK  dtc_service.dart (40+ Subaru-кодов на русском)
OK  custom_pid_service.dart
OK  profile_service.dart
OK  export_service.dart
OK  connection_service.dart (exclusive + custom PID merge)

  Доп. сервисы готовы. Далее -> ячейка 7/10 (главный + приборы + графики + лог)


In [ ]:
# @title 🖥️ Ячейка 7/10: main (17 вкладок) + Приборы + Графики + ЛогГраф + Лог + События
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/widgets/heat_colors.dart ============
with open('lib/widgets/heat_colors.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

/// Цветная шкала "как в V6": синий -> циан -> зелёный -> жёлтый -> красный
const _stops = <Color>[
  Color(0xFF0D47A1),
  Color(0xFF0288D1),
  Color(0xFF00BCD4),
  Color(0xFF4CAF50),
  Color(0xFFFFEB3B),
  Color(0xFFF44336),
];

Color heatColor(double t) {
  final tt = t.isNaN ? 0.0 : t.clamp(0.0, 1.0).toDouble();
  final pos = tt * (_stops.length - 1);
  final i = pos.floor();
  final frac = pos - i;
  if (i >= _stops.length - 1) return _stops.last;
  return Color.lerp(_stops[i], _stops[i + 1], frac)!;
}

class HeatColors {
  static const bg = Color(0xFF070D1F);
  static const panel = Color(0xFF0D1630);
  static const grid = Color(0xFF1B2A52);
  static const accent = Color(0xFF3D7BFF);
  static const gold = Color(0xFFE7B93C);
  static const text = Color(0xFFDCE6FF);
  static const dim = Color(0xFF7C8BB5);
}
''')
print('OK  widgets/heat_colors.dart')

# ============ lib/widgets/heat_map.dart ============
with open('lib/widgets/heat_map.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/analyzer_service.dart';
import 'heat_colors.dart';

/// Цветная карта (тепловая таблица) с осями, легендой и тапом по ячейке.
/// Единый виджет для Анализатора, ROM/ЭБУ-карт и ROM Diff — как анализатор V6.
class HeatMapView extends StatefulWidget {
  final int cols;
  final int rows;
  final double? Function(int r, int c) value; // null = нет данных
  final int Function(int r, int c)? hits;
  final String Function(int c)? xLabel;
  final String Function(int r)? yLabel;
  final String title;
  final String unit;
  final double? hintLo;
  final double? hintHi;

  const HeatMapView({
    super.key,
    required this.cols,
    required this.rows,
    required this.value,
    this.hits,
    this.xLabel,
    this.yLabel,
    this.title = '',
    this.unit = '',
    this.hintLo,
    this.hintHi,
  });

  factory HeatMapView.fromGrid(HeatGrid g,
      {String title = '', String unit = '', double? hintLo, double? hintHi}) {
    final range = g.range(hintLo: hintLo, hintHi: hintHi);
    final lo = range.lo, hi = range.hi;
    return HeatMapView(
      cols: g.cols,
      rows: g.rows,
      title: title,
      unit: unit,
      hintLo: lo,
      hintHi: hi,
      value: (r, c) => g.avg(g.rows - 1 - r, c),
      hits: (r, c) => g.n[g.rows - 1 - r][c],
      xLabel: (c) => g.xLabel(c),
      yLabel: (r) => g.yLabel(g.rows - 1 - r),
    );
  }

  @override
  State<HeatMapView> createState() => _HeatMapViewState();
}

class _HeatMapViewState extends State<HeatMapView> {
  ({int r, int c})? _sel;

  double get _lo {
    var lo = double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v < lo) lo = v;
      }
    }
    if (lo == double.infinity) return widget.hintLo ?? 0;
    return lo;
  }

  double get _hi {
    var hi = -double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v > hi) hi = v;
      }
    }
    if (hi == -double.infinity) return widget.hintHi ?? 1;
    return hi;
  }

  String _fmt(double v) {
    final a = v.abs();
    if (a >= 1000) return v.toStringAsFixed(0);
    if (a >= 100) return v.toStringAsFixed(1);
    return v.toStringAsFixed(2);
  }

  @override
  Widget build(BuildContext context) {
    final lo = widget.hintLo ?? _lo;
    var hi = widget.hintHi ?? _hi;
    if (hi <= lo) hi = lo + 1;
    return LayoutBuilder(builder: (ctx, cons) {
      const axW = 46.0, axH = 26.0;
      final cw = (cons.maxWidth - axW) / widget.cols;
      final ch = ((cons.maxHeight - axH) / widget.rows).clamp(14.0, 60.0);
      return Column(
        crossAxisAlignment: CrossAxisAlignment.stretch,
        children: [
          Expanded(
            child: Row(
              children: [
                SizedBox(
                  width: axW,
                  child: Column(
                    children: List.generate(widget.rows, (r) {
                      return SizedBox(
                        height: ch,
                        child: Align(
                          alignment: Alignment.centerRight,
                          child: Padding(
                            padding: const EdgeInsets.only(right: 4),
                            child: Text(widget.yLabel?.call(r) ?? '$r',
                                style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                          ),
                        ),
                      );
                    }),
                  ),
                ),
                Expanded(
                  child: GestureDetector(
                    onTapDown: (d) {
                      final c = (d.localPosition.dx / cw).floor().clamp(0, widget.cols - 1);
                      final r = (d.localPosition.dy / ch).floor().clamp(0, widget.rows - 1);
                      setState(() => _sel = (r: r, c: c));
                    },
                    child: Column(
                      children: List.generate(widget.rows, (r) {
                        return SizedBox(
                          height: ch,
                          child: Row(
                            children: List.generate(widget.cols, (c) {
                              final v = widget.value(r, c);
                              final t = v == null ? null : (v - lo) / (hi - lo);
                              final sel = _sel != null && _sel!.r == r && _sel!.c == c;
                              return Container(
                                width: cw,
                                height: ch,
                                decoration: BoxDecoration(
                                  color: v == null ? HeatColors.bg : heatColor(t!),
                                  border: Border.all(
                                    color: sel ? Colors.white : HeatColors.bg,
                                    width: sel ? 2 : 1,
                                  ),
                                ),
                                alignment: Alignment.center,
                                child: cw > 26 && v != null
                                    ? Text(
                                        _fmt(v),
                                        style: TextStyle(
                                          fontSize: 8,
                                          color: t! > 0.68 || t < 0.25
                                              ? Colors.white
                                              : Colors.black87,
                                          fontWeight: FontWeight.w600,
                                        ),
                                      )
                                    : null,
                              );
                            }),
                          ),
                        );
                      }),
                    ),
                  ),
                ),
              ],
            ),
          ),
          SizedBox(
            height: axH,
            child: Row(
              children: [
                const SizedBox(width: 46),
                ...List.generate(widget.cols, (c) {
                  return Expanded(
                    child: Text(widget.xLabel?.call(c) ?? '',
                        textAlign: TextAlign.center,
                        style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                  );
                }),
              ],
            ),
          ),
          if (_sel != null && widget.value(_sel!.r, _sel!.c) != null)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(
                'ячейка [${widget.yLabel?.call(_sel!.r) ?? ''} × ${widget.xLabel?.call(_sel!.c) ?? ''}] = '
                '${_fmt(widget.value(_sel!.r, _sel!.c)!)} ${widget.unit}'
                '${widget.hits != null ? ' · попаданий: ${widget.hits!(_sel!.r, _sel!.c)}' : ''}',
                style: const TextStyle(fontSize: 11, color: HeatColors.gold),
              ),
            ),
        ],
      );
    });
  }
}
''')
print('OK  widgets/heat_map.dart (цветные карты как в V6)')

# ============ lib/main.dart ============
with open('lib/main.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:permission_handler/permission_handler.dart';

import 'screens/analyzer_screen.dart';
import 'screens/custom_pid_screen.dart';
import 'screens/dashboard_screen.dart';
import 'screens/dtc_screen.dart';
import 'screens/ecu_maps_screen.dart';
import 'screens/events_screen.dart';
import 'screens/export_screen.dart';
import 'screens/graphs_screen.dart';
import 'screens/log_graph_screen.dart';
import 'screens/logging_screen.dart';
import 'screens/perf_screen.dart';
import 'screens/profile_screen.dart';
import 'screens/rom_diff_screen.dart';
import 'screens/rom_screen.dart';
import 'screens/service_screen.dart';
import 'screens/settings_screen.dart';
import 'screens/terminal_screen.dart';
import 'services/custom_pid_service.dart';
import 'services/profile_service.dart';
import 'services/settings_service.dart';
import 'widgets/heat_colors.dart';

void main() {
  WidgetsFlutterBinding.ensureInitialized();
  runApp(const SubaApp());
}

class SubaApp extends StatelessWidget {
  const SubaApp({super.key});

  @override
  Widget build(BuildContext context) {
    final base = ThemeData.dark(useMaterial3: true);
    return MaterialApp(
      title: 'SUBA RUN V8',
      debugShowCheckedModeBanner: false,
      theme: base.copyWith(
        scaffoldBackgroundColor: HeatColors.bg,
        colorScheme: const ColorScheme.dark(
          primary: HeatColors.accent,
          secondary: HeatColors.gold,
          surface: HeatColors.panel,
        ),
        appBarTheme: const AppBarTheme(
            backgroundColor: HeatColors.bg, foregroundColor: HeatColors.text, elevation: 0),
        tabBarTheme: const TabBarThemeData(
          labelColor: HeatColors.gold,
          unselectedLabelColor: HeatColors.dim,
          indicatorColor: HeatColors.gold,
          labelStyle: TextStyle(fontSize: 10, fontWeight: FontWeight.w700),
          unselectedLabelStyle: TextStyle(fontSize: 10),
        ),
      ),
      home: const HomeShell(),
    );
  }
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});

  @override
  State<HomeShell> createState() => _HomeShellState();
}

class _HomeShellState extends State<HomeShell> {
  @override
  void initState() {
    super.initState();
    _boot();
  }

  Future<void> _boot() async {
    await SettingsService.I.load();
    await CustomPidService.I.load();
    await ProfileService.I.load();
    await Permission.bluetoothConnect.request();
    await Permission.bluetoothScan.request();
    await Permission.locationWhenInUse.request();
  }

  static const _tabs = [
    Tab(text: 'ПРИБОРЫ'),
    Tab(text: 'ГРАФИКИ'),
    Tab(text: 'ЛОГГРАФ'),
    Tab(text: 'ЛОГ'),
    Tab(text: 'СОБЫТИЯ'),
    Tab(text: 'DTC'),
    Tab(text: 'АНАЛИЗАТОР'),
    Tab(text: 'ЭБУ КАРТЫ'),
    Tab(text: 'ROM'),
    Tab(text: 'ROM DIFF'),
    Tab(text: 'СЕРВИС'),
    Tab(text: 'ЗАМЕР'),
    Tab(text: 'ЭКСПОРТ'),
    Tab(text: 'PID'),
    Tab(text: 'ПРОФИЛИ'),
    Tab(text: 'ТЕРМИНАЛ'),
    Tab(text: 'НАСТРОЙКИ'),
  ];

  static const _views = [
    DashboardScreen(),
    GraphsScreen(),
    LogGraphScreen(),
    LoggingScreen(),
    EventsScreen(),
    DtcScreen(),
    AnalyzerScreen(),
    EcuMapsScreen(),
    RomScreen(),
    RomDiffScreen(),
    ServiceScreen(),
    PerfScreen(),
    ExportScreen(),
    CustomPidScreen(),
    ProfileScreen(),
    TerminalScreen(),
    SettingsScreen(),
  ];

  @override
  Widget build(BuildContext context) {
    return DefaultTabController(
      length: _tabs.length,
      child: Scaffold(
        appBar: AppBar(
          titleSpacing: 12,
          title: const Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Text('SUBA RUN V8',
                  style: TextStyle(fontSize: 15, fontWeight: FontWeight.w800, letterSpacing: 1.2)),
              Text('EJ20X · A2TB100B · SSM2/CAN · 17 экранов',
                  style: TextStyle(fontSize: 9.5, color: HeatColors.dim)),
            ],
          ),
          bottom: const TabBar(isScrollable: true, tabAlignment: TabAlignment.start, tabs: _tabs),
        ),
        body: const TabBarView(children: _views),
      ),
    );
  }
}
''')
print('OK  main.dart (17 вкладок)')

# ============ lib/screens/dashboard_screen.dart ============
with open('lib/screens/dashboard_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';

import '../constants.dart';
import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class DashboardScreen extends StatefulWidget {
  const DashboardScreen({super.key});

  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  Timer? _ticker;

  @override
  void initState() {
    super.initState();
    _ticker = Timer.periodic(const Duration(milliseconds: 500), (_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void dispose() {
    _ticker?.cancel();
    super.dispose();
  }

  Color _stateColor(SsmState s) {
    switch (s) {
      case SsmState.polling:
        return Colors.greenAccent;
      case SsmState.ecuReady:
      case SsmState.elmReady:
        return HeatColors.gold;
      case SsmState.connecting:
        return HeatColors.accent;
      case SsmState.error:
        return Colors.redAccent;
      case SsmState.disconnected:
        return HeatColors.dim;
    }
  }

  static String _mode(String m) {
    switch (m) {
      case 'IDLE':
        return 'ХОЛОСТОЙ';
      case 'BOOST':
        return 'БУСТ';
      case 'WOT':
        return 'ПОЛНЫЙ ГАЗ';
      case 'COAST':
        return 'НАКАТ';
      default:
        return 'КРУИЗ';
    }
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    final tiles = SettingsService.I.selectedPids;
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
          child: Row(
            children: [
              Icon(Icons.circle, size: 10, color: _stateColor(svc.elm.state)),
              const SizedBox(width: 8),
              Expanded(
                child: Text(
                  svc.elm.state == SsmState.polling
                      ? 'ОПРОС · ${svc.elm.stats.hz.toStringAsFixed(1)} Гц · '
                          '${svc.elm.stats.avgMs.toStringAsFixed(0)} мс/кадр · '
                          '${svc.poller?.blockCount ?? 0} блоков'
                      : svc.elm.state.name.toUpperCase(),
                  style: const TextStyle(fontSize: 12, color: HeatColors.text),
                  overflow: TextOverflow.ellipsis,
                ),
              ),
              if (svc.poller?.last != null)
                Text(_mode((svc.poller!.last!).mode),
                    style: const TextStyle(fontSize: 11, color: HeatColors.gold)),
            ],
          ),
        ),
        StreamBuilder<LiveSnapshot>(
          stream: svc.poller?.snapshots,
          builder: (ctx, snap) {
            final s = snap.data ?? svc.poller?.last;
            final boost = s?.boost ?? 0;
            final target = s?.has('tboost') == true ? s!.tboost : null;
            final t = ((boost + 0.65) / 2.15).clamp(0.0, 1.0);
            return Container(
              margin: const EdgeInsets.all(10),
              padding: const EdgeInsets.all(12),
              decoration: BoxDecoration(
                  color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
              child: Column(
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(
                    children: [
                      const Text('BOOST', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
                      const Spacer(),
                      Text('${boost.toStringAsFixed(2)} бар',
                          style: TextStyle(
                              fontSize: 22,
                              fontWeight: FontWeight.w800,
                              color: boost > AppConstants.overboostDanger
                                  ? Colors.redAccent
                                  : HeatColors.text)),
                      if (target != null)
                        Text('  / цель ${target.toStringAsFixed(2)}',
                            style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
                    ],
                  ),
                  const SizedBox(height: 6),
                  ClipRRect(
                    borderRadius: BorderRadius.circular(6),
                    child: LinearProgressIndicator(
                      value: t,
                      minHeight: 10,
                      backgroundColor: HeatColors.bg,
                      valueColor: AlwaysStoppedAnimation(
                          boost < 0 ? HeatColors.accent : heatColor(t)),
                    ),
                  ),
                ],
              ),
            );
          },
        ),
        Expanded(
          child: StreamBuilder<LiveSnapshot>(
            stream: svc.poller?.snapshots,
            builder: (ctx, snap) {
              final s = snap.data ?? svc.poller?.last;
              return GridView.builder(
                padding: const EdgeInsets.fromLTRB(10, 0, 10, 10),
                gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
                    crossAxisCount: 3,
                    childAspectRatio: 1.25,
                    mainAxisSpacing: 8,
                    crossAxisSpacing: 8),
                itemCount: tiles.length,
                itemBuilder: (ctx, i) => _Tile(pid: tiles[i], s: s),
              );
            },
          ),
        ),
      ],
    );
  }
}

class _Tile extends StatelessWidget {
  final dynamic pid;
  final LiveSnapshot? s;
  const _Tile({required this.pid, required this.s});

  Color? _alertColor() {
    if (s == null) return null;
    switch (pid.canon) {
      case 'fbkc':
        if (s!.fbkc <= AppConstants.fbkcDanger) return Colors.redAccent;
        if (s!.fbkc <= AppConstants.fbkcWarn) return Colors.orangeAccent;
        break;
      case 'fkl':
        if (s!.fkl <= AppConstants.fklDanger) return Colors.redAccent;
        if (s!.fkl <= AppConstants.fklWarn) return Colors.orangeAccent;
        break;
      case 'iam':
        if (s!.iam < AppConstants.iamDanger) return Colors.redAccent;
        if (s!.iam < AppConstants.iamWarn) return Colors.orangeAccent;
        break;
      case 'ect':
        if (s!.ect >= AppConstants.ectDanger) return Colors.redAccent;
        if (s!.ect >= AppConstants.ectWarn) return Colors.orangeAccent;
        break;
      case 'boost':
        if (s!.boost >= AppConstants.overboostDanger) return Colors.redAccent;
        break;
    }
    return null;
  }

  @override
  Widget build(BuildContext context) {
    final v = s?.byId[pid.id];
    final alert = _alertColor();
    final frac = v == null
        ? ''
        : (v.abs() >= 100
            ? v.toStringAsFixed(0)
            : v.abs() >= 10
                ? v.toStringAsFixed(1)
                : v.toStringAsFixed(2));
    return Container(
      padding: const EdgeInsets.all(8),
      decoration: BoxDecoration(
        color: HeatColors.panel,
        borderRadius: BorderRadius.circular(10),
        border: Border.all(color: alert ?? Colors.transparent, width: 1.5),
      ),
      child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(pid.name,
              maxLines: 1,
              overflow: TextOverflow.ellipsis,
              style: TextStyle(fontSize: 10, color: alert ?? HeatColors.dim)),
          const Spacer(),
          FittedBox(
            fit: BoxFit.scaleDown,
            alignment: Alignment.centerLeft,
            child: Text(v == null ? '--' : frac,
                style: TextStyle(
                    fontSize: 26,
                    fontWeight: FontWeight.w800,
                    color: alert ?? HeatColors.text)),
          ),
          Text(pid.unit, style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
        ],
      ),
    );
  }
}
''')
print('OK  dashboard_screen.dart')

# ============ lib/screens/graphs_screen.dart ============
with open('lib/screens/graphs_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:math' as math;

import 'package:fl_chart/fl_chart.dart';
import 'package:flutter/material.dart';

import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// Живые графики реального времени (как вкладка Графики в V6):
/// 1-4 параметра одновременно, скользящее окно ~240 снапшотов.
class GraphsScreen extends StatefulWidget {
  const GraphsScreen({super.key});

  @override
  State<GraphsScreen> createState() => _GraphsScreenState();
}

class _GraphsScreenState extends State<GraphsScreen> {
  static const _colors = [Color(0xFF4ADE80), Color(0xFF60A5FA), Color(0xFFE7B93C), Color(0xFFF472B6)];
  static const _params = <String, String>{
    'rpm': 'RPM',
    'boost': 'Буст',
    'kca': 'KCA °',
    'fbkc': 'FBKC °',
    'iam': 'IAM',
    'afr': 'AFR',
    'tps': 'TPS %',
    'maf': 'MAF г/с',
    'iat': 'IAT',
    'ect': 'ECT',
    'wgd': 'WG duty',
    'speed': 'Скорость',
  };

  final List<LiveSnapshot> _buf = [];
  final Set<String> _sel = {'rpm', 'boost', 'kca', 'fbkc'};
  StreamSubscription? _sub;
  Timer? _ticker;

  @override
  void initState() {
    super.initState();
    final p = ConnectionService.I.poller;
    if (p != null) {
      _sub = p.snapshots.listen(_add);
    }
    _ticker = Timer.periodic(const Duration(milliseconds: 700), (_) {
      if (mounted) setState(() {});
    });
  }

  void _add(LiveSnapshot s) {
    _buf.add(s);
    if (_buf.length > 240) _buf.removeRange(0, _buf.length - 240);
  }

  @override
  void dispose() {
    _sub?.cancel();
    _ticker?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    if (svc.poller == null) {
      return const Center(
          child: Text('Подключись в настройках — графики живые из опроса',
              style: TextStyle(color: HeatColors.dim)));
    }
    final sel = _sel.where((k) => _params.containsKey(k)).toList();
    return Column(
      children: [
        SizedBox(
          height: 44,
          child: ListView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 6),
            children: _params.entries.map((e) {
              final on = _sel.contains(e.key);
              return Padding(
                padding: const EdgeInsets.only(right: 6),
                child: FilterChip(
                  label: Text(e.value, style: const TextStyle(fontSize: 11)),
                  selected: on,
                  onSelected: (v) {
                    setState(() {
                      if (v) {
                        if (_sel.length < 4) _sel.add(e.key);
                      } else {
                        _sel.remove(e.key);
                      }
                    });
                  },
                ),
              );
            }).toList(),
          ),
        ),
        Expanded(
          child: Padding(
            padding: const EdgeInsets.fromLTRB(4, 4, 12, 12),
            child: _buf.length < 4
                ? const Center(child: Text('жду данные опроса...', style: TextStyle(color: HeatColors.dim)))
                : LineChart(
                    LineChartData(
                      clipData: const FlClipData.all(),
                      gridData: FlGridData(
                        show: true,
                        drawVerticalLine: false,
                        getDrawingHorizontalLine: (_) =>
                            const FlLine(color: HeatColors.grid, strokeWidth: 0.5),
                      ),
                      borderData: FlBorderData(show: false),
                      titlesData: const FlTitlesData(
                        topTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        bottomTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        rightTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        leftTitles: AxisTitles(
                            sideTitles: SideTitles(showTitles: true, reservedSize: 40)),
                      ),
                      lineBarsData: List.generate(sel.length, (i) {
                        final key = sel[i];
                        final spots = <FlSpot>[];
                        for (var j = 0; j < _buf.length; j++) {
                          final v = _buf[j].c[key];
                          if (v != null) spots.add(FlSpot(j.toDouble(), v));
                        }
                        return LineChartBarData(
                          spots: spots,
                          color: _colors[i % _colors.length],
                          barWidth: 1.6,
                          isCurved: false,
                          dotData: const FlDotData(show: false),
                        );
                      }),
                    ),
                    duration: Duration.zero,
                  ),
          ),
        ),
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 8),
          child: Row(
            children: [
              for (var i = 0; i < sel.length; i++)
                Expanded(
                  child: Column(
                    children: [
                      Text('${_params[sel[i]]}',
                          style: TextStyle(fontSize: 9, color: _colors[i % _colors.length])),
                      Text(
                        _buf.isEmpty || _buf.last.c[sel[i]] == null
                            ? '--'
                            : _fmt(_buf.last.c[sel[i]]!),
                        style: TextStyle(
                            fontSize: 17, fontWeight: FontWeight.w800, color: _colors[i % _colors.length]),
                      ),
                      if (_buf.isNotEmpty) _minMax(sel[i], _colors[i % _colors.length]),
                    ],
                  ),
                ),
            ],
          ),
        ),
      ],
    );
  }

  Widget _minMax(String key, Color color) {
    var lo = double.infinity, hi = -double.infinity;
    for (final s in _buf) {
      final v = s.c[key];
      if (v != null) { lo = math.min(lo, v); hi = math.max(hi, v); }
    }
    if (lo == double.infinity) return const SizedBox.shrink();
    return Text('${_fmt(lo)} … ${_fmt(hi)}',
        style: const TextStyle(fontSize: 8.5, color: HeatColors.dim));
  }

  String _fmt(double v) =>
      v.abs() >= 1000 ? v.toStringAsFixed(0) : v.abs() >= 100 ? v.toStringAsFixed(0) : v.abs() >= 10 ? v.toStringAsFixed(1) : v.toStringAsFixed(2);
}
''')
print('OK  graphs_screen.dart')

# ============ lib/screens/log_graph_screen.dart ============
with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math' as math;

import 'package:fl_chart/fl_chart.dart';
import 'package:flutter/material.dart';

import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';

/// ЛогГраф (как в V6): загрузка CSV, до 4 параметров, статистика min/max/avg,
/// децимация до 1500 точек. Zoom — через InteractiveViewer.
class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});

  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  static const _colors = [Color(0xFF4ADE80), Color(0xFF60A5FA), Color(0xFFE7B93C), Color(0xFFF472B6)];

  List<Map<String, double>> _rows = [];
  String _fileName = '';
  final Set<String> _sel = {'rpm', 'boost', 'kca'};
  String _status = 'Открой CSV лог из вкладки ниже';

  List<String> get _keys {
    final s = <String>{};
    for (final r in _rows.take(200)) { s.addAll(r.keys); }
    s.removeWhere((k) => k == 'ts');
    return s.toList()..sort();
  }

  Future<void> _pick() async {
    final files = await LoggerService.listLogs();
    if (!mounted) return;
    if (files.isEmpty) {
      setState(() => _status = 'Логов нет — запиши на вкладке ЛОГ');
      return;
    }
    final chosen = await showModalBottomSheet<File>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (ctx) => ListView(
        children: files
            .map((f) => ListTile(
                  leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                  title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                  onTap: () => Navigator.pop(ctx, File(f.path)),
                ))
            .toList(),
      ),
    );
    if (chosen == null) return;
    final rows = await LoggerService.parseLog(chosen.path);
    setState(() {
      _rows = rows;
      _fileName = chosen.path.split('/').last;
      _status = '$_fileName · ${rows.length} строк';
    });
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            FilledButton.icon(
                onPressed: _pick,
                icon: const Icon(Icons.folder_open, size: 18),
                label: const Text('ОТКРЫТЬ ЛОГ')),
            const SizedBox(width: 10),
            Expanded(
                child: Text(_status,
                    style: const TextStyle(fontSize: 11, color: HeatColors.dim),
                    overflow: TextOverflow.ellipsis)),
          ]),
        ),
        if (_rows.isNotEmpty)
          SizedBox(
            height: 40,
            child: ListView(
              scrollDirection: Axis.horizontal,
              padding: const EdgeInsets.symmetric(horizontal: 8),
              children: _keys.map((k) {
                final on = _sel.contains(k);
                return Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: FilterChip(
                    label: Text(k, style: const TextStyle(fontSize: 11)),
                    selected: on,
                    onSelected: (v) => setState(() {
                      if (v) {
                        if (_sel.length < 4) _sel.add(k);
                      } else {
                        _sel.remove(k);
                      }
                    }),
                  ),
                );
              }).toList(),
            ),
          ),
        Expanded(
          child: _rows.isEmpty
              ? const Center(
                  child: Icon(Icons.show_chart, size: 48, color: HeatColors.grid))
              : Padding(
                  padding: const EdgeInsets.fromLTRB(4, 6, 12, 8),
                  child: InteractiveViewer(
                    constrained: false,
                    scaleEnabled: true,
                    panEnabled: true,
                    minScale: 0.5,
                    maxScale: 8,
                    child: SizedBox(
                      width: MediaQuery.of(context).size.width - 24,
                      height: double.infinity,
                      child: _buildChart(),
                    ),
                  ),
                ),
        ),
        if (_rows.isNotEmpty)
          Container(
            color: HeatColors.panel,
            padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 8),
            child: Row(
              children: [
                for (var i = 0; i < _sel.length; i++)
                  Expanded(child: _stat(_sel.elementAt(i), _colors[i % 4])),
              ],
            ),
          ),
      ],
    );
  }

  Widget _buildChart() {
    final sel = _sel.where((k) => _rows.any((r) => r.containsKey(k))).toList();
    final step = math.max(1, (_rows.length / 1500).floor());
    return LineChart(
      LineChartData(
        clipData: const FlClipData.all(),
        gridData: FlGridData(
          show: true,
          drawVerticalLine: false,
          getDrawingHorizontalLine: (_) => const FlLine(color: HeatColors.grid, strokeWidth: 0.5),
        ),
        borderData: FlBorderData(show: false),
        titlesData: const FlTitlesData(
          topTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          rightTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          bottomTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true, reservedSize: 40)),
        ),
        lineBarsData: List.generate(sel.length, (i) {
          final key = sel[i];
          final spots = <FlSpot>[];
          for (var j = 0; j < _rows.length; j += step) {
            final v = _rows[j][key];
            if (v != null) spots.add(FlSpot(j.toDouble(), v));
          }
          return LineChartBarData(
            spots: spots,
            color: _colors[i % _colors.length],
            barWidth: 1.4,
            isCurved: false,
            dotData: const FlDotData(show: false),
          );
        }),
      ),
      duration: Duration.zero,
    );
  }

  Widget _stat(String key, Color color) {
    var lo = double.infinity, hi = -double.infinity, sum = 0.0;
    var n = 0;
    for (final r in _rows) {
      final v = r[key];
      if (v != null) { lo = math.min(lo, v); hi = math.max(hi, v); sum += v; n++; }
    }
    if (n == 0) return const SizedBox.shrink();
    return Column(children: [
      Text(key, style: TextStyle(fontSize: 9, color: color)),
      Text('${hi == lo ? '' : ''}${_f(lo)} / ${_f(sum / n)} / ${_f(hi)}',
          style: TextStyle(fontSize: 10.5, fontWeight: FontWeight.w700, color: color)),
      const Text('min / avg / max', style: TextStyle(fontSize: 7.5, color: HeatColors.dim)),
    ]);
  }

  String _f(double v) => v.abs() >= 1000
      ? v.toStringAsFixed(0)
      : v.abs() >= 100
          ? v.toStringAsFixed(0)
          : v.abs() >= 10
              ? v.toStringAsFixed(1)
              : v.toStringAsFixed(2);
}
''')
print('OK  log_graph_screen.dart')

# ============ lib/screens/logging_screen.dart ============
with open('lib/screens/logging_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../constants.dart';
import '../services/connection_service.dart';
import '../services/logger_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class LoggingScreen extends StatefulWidget {
  const LoggingScreen({super.key});

  @override
  State<LoggingScreen> createState() => _LoggingScreenState();
}

class _LoggingScreenState extends State<LoggingScreen> {
  Timer? _t;
  int _filesVersion = 0;
  DateTime? _lastActive;

  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(seconds: 1), (_) {
      if (mounted) setState(() {});
      _autoTick();
    });
  }

  void _autoTick() {
    if (!SettingsService.I.autoLog) return;
    final svc = ConnectionService.I;
    final snap = svc.poller?.last;
    if (snap == null || svc.elm.state != SsmState.polling) return;
    if (snap.rpm > AppConstants.autoLogRpm || snap.underBoost) {
      _lastActive = DateTime.now();
      if (!svc.logger.logging) _startLog();
    } else if (svc.logger.logging && _lastActive != null) {
      if (DateTime.now().difference(_lastActive!).inSeconds > AppConstants.autoLogIdleSec) {
        _stopLog();
      }
    }
  }

  Future<void> _startLog() async {
    final svc = ConnectionService.I;
    final p = svc.poller;
    if (p == null) return;
    await svc.logger.start(p.snapshots, SettingsService.I.selectedPids);
    if (mounted) setState(() {});
  }

  Future<void> _stopLog() async {
    await ConnectionService.I.logger.stop();
    if (mounted) setState(() => _filesVersion++);
  }

  @override
  void dispose() {
    _t?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    final logging = svc.logger.logging;
    return Column(
      children: [
        Container(
          margin: const EdgeInsets.all(12),
          padding: const EdgeInsets.all(14),
          decoration:
              BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Row(
            children: [
              Icon(logging ? Icons.fiber_manual_record : Icons.stop_circle_outlined,
                  color: logging ? Colors.redAccent : HeatColors.dim),
              const SizedBox(width: 10),
              Expanded(
                child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: [
                    Text(logging ? 'ЗАПИСЬ · ${svc.logger.rows} строк' : 'Лог остановлен',
                        style: const TextStyle(fontWeight: FontWeight.w700)),
                    Text(
                      logging
                          ? svc.logger.filePath?.split('/').last ?? ''
                          : 'CSV: выбранные PID + ошибка буста + расход',
                      style: const TextStyle(fontSize: 11, color: HeatColors.dim),
                    ),
                  ],
                ),
              ),
              Switch(
                value: SettingsService.I.autoLog,
                onChanged: (v) {
                  SettingsService.I.autoLog = v;
                  SettingsService.I.save();
                  setState(() {});
                },
              ),
              const Text('авто', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
            ],
          ),
        ),
        Padding(
          padding: const EdgeInsets.symmetric(horizontal: 12),
          child: Row(
            children: [
              Expanded(
                child: FilledButton.icon(
                  style: FilledButton.styleFrom(
                      backgroundColor: logging ? Colors.red.shade900 : HeatColors.accent),
                  onPressed:
                      svc.elm.state == SsmState.polling ? (logging ? _stopLog : _startLog) : null,
                  icon: Icon(logging ? Icons.stop : Icons.play_arrow),
                  label: Text(logging ? 'СТОП' : 'СТАРТ'),
                ),
              ),
            ],
          ),
        ),
        const SizedBox(height: 8),
        Expanded(
          child: FutureBuilder<List<FileSystemEntity>>(
            key: ValueKey(_filesVersion),
            future: LoggerService.listLogs(),
            builder: (ctx, snap) {
              final files = snap.data ?? [];
              if (files.isEmpty) {
                return const Center(
                    child: Text('Логов пока нет', style: TextStyle(color: HeatColors.dim)));
              }
              return ListView.builder(
                itemCount: files.length,
                itemBuilder: (ctx, i) {
                  final f = files[i];
                  final sizeKb = (f.statSync().size / 1024).toStringAsFixed(0);
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                    title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                    subtitle: Text('$sizeKb KB',
                        style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
                    trailing: Row(
                      mainAxisSize: MainAxisSize.min,
                      children: [
                        IconButton(
                            icon: const Icon(Icons.share, size: 20),
                            onPressed: () => Share.shareXFiles([XFile(f.path)])),
                        IconButton(
                            icon: const Icon(Icons.delete_outline, size: 20),
                            onPressed: () async {
                              await f.delete();
                              setState(() => _filesVersion++);
                            }),
                      ],
                    ),
                  );
                },
              );
            },
          ),
        ),
      ],
    );
  }
}
''')
print('OK  logging_screen.dart')

# ============ lib/screens/events_screen.dart ============
with open('lib/screens/events_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../services/alert_service.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// События: турбо-алерты со снимком всех параметров в момент срабатывания (как в V6)
class EventsScreen extends StatefulWidget {
  const EventsScreen({super.key});

  @override
  State<EventsScreen> createState() => _EventsScreenState();
}

class _EventsScreenState extends State<EventsScreen> {
  StreamSubscription<AlertEvent>? _sub;

  @override
  void initState() {
    super.initState();
    _sub = ConnectionService.I.alerts.stream.listen((_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void dispose() {
    _sub?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final events = ConnectionService.I.alerts.events;
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
          child: Row(
            children: [
              Text('${events.length} событий',
                  style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
              const Spacer(),
              TextButton.icon(
                onPressed: () => setState(() => ConnectionService.I.alerts.clear()),
                icon: const Icon(Icons.delete_sweep_outlined, size: 18),
                label: const Text('очистить'),
              ),
            ],
          ),
        ),
        Expanded(
          child: events.isEmpty
              ? const Center(
                  child: Text('Событий нет — и это хорошо',
                      style: TextStyle(color: HeatColors.dim)))
              : ListView.builder(
                  itemCount: events.length,
                  itemBuilder: (ctx, i) {
                    final e = events[i];
                    final danger = e.level == 2;
                    return ListTile(
                      dense: true,
                      leading: Icon(
                          danger ? Icons.warning_amber_rounded : Icons.info_outline,
                          color: danger ? Colors.redAccent : Colors.orangeAccent),
                      title: Text(e.text, style: const TextStyle(fontSize: 12.5)),
                      subtitle: Text(
                          '${DateFormat('HH:mm:ss').format(e.ts)} · ${e.code}${e.snapshot.isEmpty ? '' : ' · snapshot: rpm ${e.snapshot['rpm']?.toStringAsFixed(0) ?? '--'}, boost ${e.snapshot['boost']?.toStringAsFixed(2) ?? '--'}'}',
                          style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                      onTap: () => _showSnapshot(e),
                    );
                  },
                ),
        ),
      ],
    );
  }

  void _showSnapshot(AlertEvent e) {
    showDialog(
      context: context,
      builder: (ctx) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: Text('${e.code} · ${DateFormat('HH:mm:ss').format(e.ts)}',
            style: const TextStyle(fontSize: 15)),
        content: SizedBox(
          width: 320,
          child: SingleChildScrollView(
            child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Text(e.text, style: const TextStyle(fontSize: 13, color: HeatColors.gold)),
                const Divider(),
                ...e.snapshot.entries.map((kv) => Padding(
                      padding: const EdgeInsets.symmetric(vertical: 1.5),
                      child: Row(children: [
                        Expanded(
                            child: Text(kv.key,
                                style: const TextStyle(fontSize: 11, color: HeatColors.dim))),
                        Text(kv.value.toStringAsFixed(2), style: const TextStyle(fontSize: 12)),
                      ]),
                    )),
              ],
            ),
          ),
        ),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx), child: const Text('ЗАКРЫТЬ')),
        ],
      ),
    );
  }
}
''')
print('OK  events_screen.dart')
print()
print('=' * 64)
print('  Ядро UI готово (6 экранов). Далее -> ячейка 8/10 (настройческие экраны)')
print('=' * 64)


OK  widgets/heat_colors.dart
OK  widgets/heat_map.dart (цветные карты как в V6)
OK  main.dart (17 вкладок)
OK  dashboard_screen.dart
OK  graphs_screen.dart
OK  log_graph_screen.dart
OK  logging_screen.dart
OK  events_screen.dart

  Ядро UI готово (6 экранов). Далее -> ячейка 8/10 (настройческие экраны)


In [ ]:
# @title 🎛️ Ячейка 8/10: DTC · Анализатор(+сплиттер) · ЭБУ Карты · ROM · ROM Diff · Сервис · Замер · Терминал
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/screens/dtc_screen.dart ============
with open('lib/screens/dtc_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/dtc_service.dart';
import '../widgets/heat_colors.dart';

class DtcScreen extends StatefulWidget {
  const DtcScreen({super.key});

  @override
  State<DtcScreen> createState() => _DtcScreenState();
}

class _DtcScreenState extends State<DtcScreen> {
  List<DtcItem>? _codes;
  bool _busy = false;
  String _msg = '';

  Future<void> _read() async {
    setState(() { _busy = true; _msg = 'читаю DTC (опрос на паузе)...'; });
    try {
      final codes = await ConnectionService.I.exclusive((elm) => DtcService.read(elm));
      setState(() {
        _codes = codes;
        _msg = codes.isEmpty ? 'Ошибок нет (mode 03 пуст)' : 'Найдено: ${codes.length}';
      });
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  Future<void> _clear() async {
    setState(() { _busy = true; _msg = 'стираю (mode 04)...'; });
    try {
      final ok = await ConnectionService.I.exclusive((elm) => DtcService.clear(elm));
      setState(() {
        _msg = ok ? 'CEL стёрт. Дай мотору поработать и перечитай.' : 'ECU не подтвердил стирание';
        if (ok) _codes = [];
      });
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Container(
          margin: const EdgeInsets.all(12),
          padding: const EdgeInsets.all(12),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Column(children: [
            Row(children: [
              Expanded(
                child: FilledButton.icon(
                    onPressed: _busy ? null : _read,
                    icon: const Icon(Icons.search),
                    label: const Text('ЧИТАТЬ DTC')),
              ),
              const SizedBox(width: 8),
              Expanded(
                child: OutlinedButton.icon(
                    onPressed: _busy ? null : _clear,
                    icon: const Icon(Icons.cleaning_services_outlined),
                    label: const Text('СТЕРЕТЬ CEL')),
              ),
            ]),
            if (_msg.isNotEmpty)
              Padding(
                padding: const EdgeInsets.only(top: 8),
                child: Align(
                    alignment: Alignment.centerLeft,
                    child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold))),
              ),
          ]),
        ),
        Expanded(
          child: _codes == null
              ? const Center(
                  child: Text('OBD mode 03/04 на шине 0x7E0 · описания Subaru на русском',
                      style: TextStyle(color: HeatColors.dim, fontSize: 12)))
              : _codes!.isEmpty
                  ? const Center(child: Text('Ошибок нет', style: TextStyle(color: Colors.greenAccent)))
                  : ListView.builder(
                      itemCount: _codes!.length,
                      itemBuilder: (ctx, i) {
                        final c = _codes![i];
                        return ListTile(
                          dense: true,
                          leading: const Icon(Icons.error_outline, color: Colors.redAccent),
                          title: Text(c.code,
                              style: const TextStyle(fontWeight: FontWeight.w800, fontSize: 15)),
                          subtitle: Text(c.text,
                              style: const TextStyle(fontSize: 12, color: HeatColors.text)),
                        );
                      },
                    ),
        ),
      ],
    );
  }
}
''')
print('OK  dtc_screen.dart')

# ============ lib/screens/analyzer_screen.dart (сплиттер слияния как в V6) ============
with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:io';

import 'package:flutter/material.dart';

import '../services/analyzer_service.dart';
import '../services/connection_service.dart';
import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// Анализатор V8: Subaru-метрики по сетке (по умолчанию Обороты × Буст).
/// Онлайн + из лога + СЛИЯНИЕ нескольких CSV (сплиттер V6: суммы+счётчики складываются).
class AnalyzerScreen extends StatefulWidget {
  const AnalyzerScreen({super.key});

  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen> {
  bool _onlineMode = true;
  Timer? _ticker;
  String _status = '';
  final List<String> _mergedLogs = [];

  AnalyzerService get a => ConnectionService.I.analyzer;

  @override
  void initState() {
    super.initState();
    _ticker = Timer.periodic(const Duration(milliseconds: 600), (_) {
      if (mounted && _onlineMode) setState(() {});
    });
  }

  @override
  void dispose() {
    _ticker?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final metrics =
        AnalyzerService.metricNames.entries.where((e) => e.key != 'ect2').toList();
    final range = a.grid.range();
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        SegmentedButton<bool>(
          segments: const [
            ButtonSegment(value: true, icon: Icon(Icons.wifi_tethering), label: Text('ОНЛАЙН')),
            ButtonSegment(value: false, icon: Icon(Icons.description_outlined), label: Text('ИЗ ЛОГА')),
          ],
          selected: {_onlineMode},
          onSelectionChanged: (s) => setState(() => _onlineMode = s.first),
        ),
        const SizedBox(height: 10),
        Wrap(spacing: 8, runSpacing: 8, children: [
          _drop<AxisOpt>('Ось X', AnalyzerService.axisX, a.xOpt, (o) => o.label,
              (o) => setState(() => a.setAxes(o, a.yOpt, a.metric))),
          _drop<AxisOpt>('Ось Y', AnalyzerService.axisY, a.yOpt, (o) => o.label,
              (o) => setState(() => a.setAxes(a.xOpt, o, a.metric))),
          _drop<MapEntry<String, String>>(
              'Метрика',
              metrics,
              metrics.firstWhere((e) => e.key == a.metric, orElse: () => metrics.first),
              (o) => o.value,
              (o) => setState(() => a.setAxes(a.xOpt, a.yOpt, o.key))),
        ]),
        const SizedBox(height: 8),
        Wrap(spacing: 8, runSpacing: 8, crossAxisAlignment: WrapCrossAlignment.center, children: [
          if (!_onlineMode)
            FilledButton.icon(
              onPressed: () => _pickLogAndMerge(reset: true),
              icon: const Icon(Icons.folder_open, size: 18),
              label: const Text('Открыть CSV'),
            )
          else
            FilledButton.icon(
              onPressed: _toggleOnline,
              icon: Icon(a.online ? Icons.pause : Icons.play_arrow, size: 18),
              label: Text(a.online ? 'Пауза' : 'Захват из опроса'),
            ),
          if (!_onlineMode && a.grid.totalHits > 0)
            FilledButton.tonalIcon(
              onPressed: () => _pickLogAndMerge(reset: false),
              icon: const Icon(Icons.merge, size: 18),
              label: const Text('СЛИТЬ ЕЩЁ ЛОГ'),
            ),
          OutlinedButton.icon(
            onPressed: () => setState(() { a.grid.reset(); _mergedLogs.clear(); }),
            icon: const Icon(Icons.cleaning_services_outlined, size: 18),
            label: const Text('Сброс'),
          ),
          Text('точек: ${a.grid.totalHits}',
              style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
        ]),
        if (_status.isNotEmpty)
          Padding(
            padding: const EdgeInsets.only(top: 6),
            child: Text(_status, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
          ),
        if (_mergedLogs.isNotEmpty)
          Padding(
            padding: const EdgeInsets.only(top: 4),
            child: Text('слитые логи: ${_mergedLogs.join(" + ")}',
                style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
          ),
        const SizedBox(height: 12),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 420,
          child: HeatMapView.fromGrid(a.grid,
              title: AnalyzerService.metricNames[a.metric] ?? a.metric),
        ),
        const SizedBox(height: 8),
        Text(
          'X: ${a.xOpt.label}  ·  Y: ${a.yOpt.label}  ·  '
          'диапазон ${range.lo.toStringAsFixed(2)} … ${range.hi.toStringAsFixed(2)}\n'
          'Ячейка = среднее. Синий→красный = min→max. Тап — значение и число замеров.',
          style: const TextStyle(fontSize: 11, color: HeatColors.dim),
        ),
      ],
    );
  }

  Widget _drop<T>(String hint, List<T> items, T value, String Function(T) label, void Function(T) onSel) {
    return Container(
      padding: const EdgeInsets.symmetric(horizontal: 10),
      decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(10)),
      child: DropdownButtonHideUnderline(
        child: DropdownButton<T>(
          hint: Text(hint, style: const TextStyle(fontSize: 12)),
          value: value,
          isDense: true,
          dropdownColor: HeatColors.panel,
          items: items
              .map((e) => DropdownMenuItem<T>(
                  value: e, child: Text('$hint: ${label(e)}', style: const TextStyle(fontSize: 12))))
              .toList(),
          onChanged: (v) { if (v != null) onSel(v); },
        ),
      ),
    );
  }

  void _toggleOnline() {
    final svc = ConnectionService.I;
    if (a.online) {
      a.stopOnline();
    } else {
      final p = svc.poller;
      if (p == null) {
        setState(() => _status = 'Нет опроса: подключись в настройках');
        return;
      }
      a.startOnline(p.snapshots);
      setState(() => _status = 'Захват из живого опроса...');
    }
    setState(() {});
  }

  Future<void> _pickLogAndMerge({required bool reset}) async {
    final files = await LoggerService.listLogs();
    if (!mounted) return;
    if (files.isEmpty) {
      setState(() => _status = 'Логов нет');
      return;
    }
    final chosen = await showModalBottomSheet<File>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (ctx) => ListView(
        children: files
            .map((f) => ListTile(
                  leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                  title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                  onTap: () => Navigator.pop(ctx, File(f.path)),
                ))
            .toList(),
      ),
    );
    if (chosen == null) return;
    if (reset) { a.grid.reset(); _mergedLogs.clear(); }
    final rows = await LoggerService.parseLog(chosen.path);
    a.buildFromRows(rows); // суммы и счётчики накапливаются => это и есть слияние
    final name = chosen.path.split('/').last;
    setState(() {
      _mergedLogs.add(name);
      _status = '$name: +${rows.length} строк → всего ${a.grid.totalHits} точек';
    });
  }
}
''')
print('OK  analyzer_screen.dart (сплиттер слияния CSV)')

# ============ lib/screens/ecu_maps_screen.dart ============
with open('lib/screens/ecu_maps_screen.dart', 'w') as f:
    f.write(r'''import 'dart:typed_data';

import 'package:flutter/material.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/connection_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ЭБУ Карты: чтение таблиц ЖИВЬЁМ из ECU по SSM2-блоками (как «ЭБУ Карты» V6,
/// но по A8-чтению диапазонов). Просмотр; правка — только через ROM-экран в .bin.
class EcuMapsScreen extends StatefulWidget {
  const EcuMapsScreen({super.key});

  @override
  State<EcuMapsScreen> createState() => _EcuMapsScreenState();
}

class _EcuMapsScreenState extends State<EcuMapsScreen> {
  String _filter = 'all';
  bool _reading = false;
  double _progress = 0;
  String _status = '';

  @override
  Widget build(BuildContext context) {
    final defs = SubaruRom.tables;
    final cats = <String>{for (final d in defs) d.category}.toList()..sort();
    final filtered = defs.where((d) => _filter == 'all' || d.category == _filter).toList();
    final connected = ConnectionService.I.elm.state == SsmState.ecuReady ||
        ConnectionService.I.elm.state == SsmState.polling;

    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Text(
              connected
                  ? 'Чтение карт напрямую из ECU (SSM2 A8-блоками, ${defs.length} карт в дефинишне)'
                  : 'Сначала подключись в Настройках',
              style: const TextStyle(fontSize: 12),
            ),
            if (_reading) ...[
              const SizedBox(height: 8),
              LinearProgressIndicator(value: _progress, backgroundColor: HeatColors.bg),
              const SizedBox(height: 4),
              Text(_status, style: const TextStyle(fontSize: 11, color: HeatColors.gold)),
            ],
          ]),
        ),
        SizedBox(
          height: 40,
          child: ListView(scrollDirection: Axis.horizontal, padding: const EdgeInsets.symmetric(horizontal: 8), children: [
            Padding(
              padding: const EdgeInsets.only(right: 6),
              child: ChoiceChip(
                label: Text('ВСЕ (${defs.length})', style: const TextStyle(fontSize: 11)),
                selected: _filter == 'all',
                onSelected: (_) => setState(() => _filter = 'all'),
              ),
            ),
            ...cats.map((c) => Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: ChoiceChip(
                    label: Text('$c (${defs.where((d) => d.category == c).length})',
                        style: const TextStyle(fontSize: 11)),
                    selected: _filter == c,
                    onSelected: (_) => setState(() => _filter = c),
                  ),
                )),
          ]),
        ),
        Expanded(
          child: ListView.builder(
            itemCount: filtered.length,
            itemBuilder: (ctx, i) {
              final d = filtered[i];
              final bytes = d.rows * d.cols * d.data.sizeOf;
              return ListTile(
                dense: true,
                enabled: connected && !_reading,
                leading: const Icon(Icons.memory, size: 20, color: HeatColors.accent),
                title: Text(d.name, style: const TextStyle(fontSize: 13)),
                subtitle: Text('${d.rows}×${d.cols} · ${d.units} · ${d.addrHex} · $bytes B',
                    style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                trailing: const Icon(Icons.download_for_offline_outlined, size: 18),
                onTap: () => _readFromEcu(d),
              );
            },
          ),
        ),
      ],
    );
  }

  Future<void> _readFromEcu(RomTableDef def) async {
    setState(() { _reading = true; _progress = 0; _status = 'пауза опроса...'; });
    try {
      final table = await ConnectionService.I.exclusive((elm) => _readTable(elm, def));
      if (!mounted || table == null) return;
      setState(() { _reading = false; _status = ''; });
      await Navigator.push(
          context, MaterialPageRoute(builder: (_) => EcuTablePage(table: table)));
    } catch (e) {
      setState(() { _reading = false; _status = 'ошибка: $e'; });
    }
  }

  /// Собираем «виртуальный ROM» из адресных кусков ECU и декодируем как файл
  Future<RomTable?> _readTable(SsmElm elm, RomTableDef def) async {
    final jobs = <_RangeJob>[];
    jobs.add(_RangeJob(def.address, def.data.count * def.data.sizeOf));
    if (def.xAxis != null && def.xAxis!.address >= 0) {
      jobs.add(_RangeJob(def.xAxis!.address, def.xAxis!.byteLen));
    }
    if (def.yAxis != null && def.yAxis!.address >= 0) {
      jobs.add(_RangeJob(def.yAxis!.address, def.yAxis!.byteLen));
    }
    var minAddr = 1 << 30, maxAddr = 0;
    for (final j in jobs) {
      minAddr = j.addr < minAddr ? j.addr : minAddr;
      maxAddr = (j.addr + j.len) > maxAddr ? j.addr + j.len : maxAddr;
    }
    final shift = minAddr;
    final rom = Uint8List(maxAddr - shift);
    final idx = SubaruRom.tables.indexOf(def);

    final totalChunks = jobs.fold<int>(0, (a, j) => a + ((j.len + 0x7F) >> 7));
    var done = 0;
    for (final j in jobs) {
      for (var off = 0; off < j.len; off += 0x80) {
        final want = (j.len - off) > 0x80 ? 0x80 : (j.len - off);
        final chunk = await elm.readBytes(j.addr + off, want);
        done++;
        if (mounted) {
          setState(() {
            _progress = done / totalChunks;
            _status = '${def.name}  ·  кадр $done/$totalChunks';
          });
        }
        if (chunk == null) continue; // дырки оставляем нулями — ячейки покажут 0/NaN
        final base = j.addr + off - shift;
        for (var i = 0; i < chunk.length && base + i < rom.length; i++) {
          rom[base + i] = chunk[i];
        }
      }
    }

    // decode (те же правила, что у ROM-файла)
    ByteData bd = ByteData.sublistView(rom);
    double colVal(RomCol? col, int i, List<double> fallback) {
      if (col == null) return fallback[i];
      if (col.storage == 'static') {
        final vals = SubaruRom.staticAxes['${col == def.xAxis ? 'x' : 'y'}$idx'];
        if (vals != null && i < vals.length) return vals[i];
        return fallback[i];
      }
      final shifted = RomCol(
        address: col.address - shift, count: col.count,
        storage: col.storage, endian: col.endian, to: col.to, fr: col.fr);
      try {
        return shifted.value(rom, bd, i);
      } catch (_) {
        return double.nan;
      }
    }

    final xs = List<double>.generate(def.cols, (i) => i.toDouble());
    final ys = List<double>.generate(def.rows, (i) => i.toDouble());
    final xVals = List<double>.generate(def.cols, (i) => colVal(def.xAxis, i, xs));
    final yVals = List<double>.generate(def.rows, (i) => colVal(def.yAxis, i, ys));
    final dataShifted = RomCol(
        address: def.address - shift, count: def.data.count,
        storage: def.data.storage, endian: def.data.endian, to: def.data.to);
    final z = List.generate(def.rows, (_) => List<double>.filled(def.cols, 0));
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        try {
          z[r][c] = dataShifted.value(rom, bd, i);
        } catch (_) {
          z[r][c] = double.nan;
        }
      }
    }
    final tdef = RomTableDef(
        name: def.name, category: def.category, address: def.address,
        rows: def.rows, cols: def.cols, swapxy: def.swapxy, units: def.units,
        data: def.data, xAxis: def.xAxis, yAxis: def.yAxis,
        minHint: def.minHint, maxHint: def.maxHint);
    return RomTable(def: tdef, xValues: xVals, yValues: yVals, z: z);
  }
}

class _RangeJob {
  final int addr;
  final int len;
  _RangeJob(this.addr, this.len);
}

/// Просмотр карты, прочитанной из ECU (read-only)
class EcuTablePage extends StatelessWidget {
  final RomTable table;
  const EcuTablePage({super.key, required this.table});

  @override
  Widget build(BuildContext context) {
    final d = table.def;
    return Scaffold(
      appBar: AppBar(title: Text('${d.name} · ECU live', style: const TextStyle(fontSize: 14))),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Wrap(spacing: 12, runSpacing: 4, children: [
          _kv('адрес', d.addrHex), _kv('размер', '${d.rows}×${d.cols}'),
          _kv('ед.', d.units), _kv('min', table.minV.toStringAsFixed(2)),
          _kv('max', table.maxV.toStringAsFixed(2)),
        ]),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 440,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: d.units,
            hintLo: d.minHint.isNaN ? null : d.minHint,
            hintHi: d.maxHint.isNaN ? null : d.maxHint,
            value: (r, c) => table.z[d.rows - 1 - r][c],
            xLabel: (c) => c < table.xValues.length ? _ax(table.xValues[c]) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < table.yValues.length ? _ax(table.yValues[i]) : '$i';
            },
          ),
        ),
        const SizedBox(height: 8),
        const Text('Прочитано живьём из ECU. Read-only: правка карт — на экране ROM (в .bin-файл).',
            style: TextStyle(fontSize: 11, color: HeatColors.dim)),
      ]),
    );
  }

  String _ax(double v) => v.abs() >= 100 ? v.toStringAsFixed(0) : v.toStringAsFixed(1);

  Widget _kv(String k, String v) => RichText(
        text: TextSpan(children: [
          TextSpan(text: '$k: ', style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextSpan(text: v, style: const TextStyle(fontSize: 11, color: HeatColors.text)),
        ]),
      );
}
''')
print('OK  ecu_maps_screen.dart (живое чтение карт из ECU)')

# ============ lib/screens/rom_screen.dart ============
with open('lib/screens/rom_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ROM-экран: загрузка .bin A2TB100B, все карты из дефинишна,
/// цветной просмотр, правка ячеек, сохранение V8MOD_*.bin.
class RomScreen extends StatefulWidget {
  const RomScreen({super.key});

  @override
  State<RomScreen> createState() => _RomScreenState();
}

class _RomScreenState extends State<RomScreen> {
  String _filter = 'all';
  String _query = '';

  @override
  Widget build(BuildContext context) {
    final rom = ConnectionService.I.rom;
    if (!rom.loaded) {
      return Center(
        child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
          const Icon(Icons.memory, size: 56, color: HeatColors.dim),
          const SizedBox(height: 12),
          const Text('Загрузи ROM (.bin) A2TB100B\n— карты поднимутся из встроенного дефинишна',
              textAlign: TextAlign.center, style: TextStyle(color: HeatColors.dim)),
          const SizedBox(height: 16),
          FilledButton.icon(
              onPressed: _load, icon: const Icon(Icons.file_open), label: const Text('ЗАГРУЗИТЬ ROM')),
        ]),
      );
    }

    final defs = rom.defs;
    final cats = <String>{for (final d in defs) d.category}.toList()..sort();
    final filtered = defs.where((d) {
      if (_filter != 'all' && d.category != _filter) return false;
      if (_query.isNotEmpty && !d.name.toLowerCase().contains(_query.toLowerCase())) return false;
      return true;
    }).toList();

    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            const Icon(Icons.check_circle, color: Colors.greenAccent, size: 16),
            const SizedBox(width: 8),
            Expanded(
                child: Text('${rom.fileName} · ${defs.length} карт · ${SubaruRom.calId}',
                    style: const TextStyle(fontSize: 12))),
            TextButton(onPressed: _load, child: const Text('другой ROM')),
          ]),
        ),
        SizedBox(
          height: 40,
          child: ListView(scrollDirection: Axis.horizontal, padding: const EdgeInsets.symmetric(horizontal: 8), children: [
            Padding(
              padding: const EdgeInsets.only(right: 6),
              child: ChoiceChip(
                label: Text('ВСЕ (${defs.length})', style: const TextStyle(fontSize: 11)),
                selected: _filter == 'all',
                onSelected: (_) => setState(() => _filter = 'all'),
              ),
            ),
            ...cats.map((c) => Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: ChoiceChip(
                    label: Text('$c (${defs.where((d) => d.category == c).length})',
                        style: const TextStyle(fontSize: 11)),
                    selected: _filter == c,
                    onSelected: (_) => setState(() => _filter = c),
                  ),
                )),
          ]),
        ),
        Padding(
          padding: const EdgeInsets.all(8),
          child: TextField(
            style: const TextStyle(fontSize: 13),
            decoration: InputDecoration(
              hintText: 'поиск карты...',
              isDense: true,
              prefixIcon: const Icon(Icons.search, size: 18),
              filled: true,
              fillColor: HeatColors.panel,
              border: OutlineInputBorder(
                  borderRadius: BorderRadius.circular(10), borderSide: BorderSide.none),
            ),
            onChanged: (v) => setState(() => _query = v),
          ),
        ),
        Expanded(
          child: ListView.builder(
            itemCount: filtered.length,
            itemBuilder: (ctx, i) {
              final d = filtered[i];
              return ListTile(
                dense: true,
                leading: Icon(d.editable ? Icons.grid_on : Icons.grid_off,
                    size: 20, color: d.editable ? HeatColors.gold : HeatColors.dim),
                title: Text(d.name, style: const TextStyle(fontSize: 13)),
                subtitle: Text('${d.rows}×${d.cols} · ${d.units} · ${d.addrHex} · ${d.category}',
                    style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                trailing: const Icon(Icons.chevron_right, size: 18),
                onTap: () => Navigator.push(
                    context, MaterialPageRoute(builder: (_) => RomTableScreen(def: d))),
              );
            },
          ),
        ),
      ],
    );
  }

  Future<void> _load() async {
    final rom = ConnectionService.I.rom;
    final err = await rom.pickAndLoad();
    if (!mounted) return;
    if (err != null) {
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(err)));
    }
    setState(() {});
  }
}

class RomTableScreen extends StatefulWidget {
  final RomTableDef def;
  const RomTableScreen({super.key, required this.def});

  @override
  State<RomTableScreen> createState() => _RomTableScreenState();
}

class _RomTableScreenState extends State<RomTableScreen> {
  RomTable? _table;
  bool _modified = false;

  @override
  void initState() {
    super.initState();
    _table = ConnectionService.I.rom.readTable(widget.def);
  }

  @override
  Widget build(BuildContext context) {
    final d = widget.def;
    final t = _table;
    if (t == null) {
      return Scaffold(appBar: AppBar(title: Text(d.name)), body: const Center(child: CircularProgressIndicator()));
    }
    return Scaffold(
      appBar: AppBar(
        title: Text(d.name, style: const TextStyle(fontSize: 15)),
        actions: [
          if (d.editable)
            TextButton.icon(
              onPressed: _modified ? _saveMod : null,
              icon: const Icon(Icons.save_outlined, size: 18),
              label: const Text('V8MOD'),
            ),
          IconButton(icon: const Icon(Icons.ios_share, size: 18), onPressed: _exportCsv),
        ],
      ),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Wrap(spacing: 12, runSpacing: 4, children: [
          _info('адрес', d.addrHex), _info('размер', '${d.rows}×${d.cols}'),
          _info('ед.', d.units), _info('min', t.minV.toStringAsFixed(2)),
          _info('max', t.maxV.toStringAsFixed(2)), _info('storage', d.data.storage),
        ]),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 440,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: d.units,
            hintLo: d.minHint.isNaN ? null : d.minHint,
            hintHi: d.maxHint.isNaN ? null : d.maxHint,
            value: (r, c) => t.z[d.rows - 1 - r][c],
            xLabel: (c) => c < t.xValues.length ? _ax(t.xValues[c]) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < t.yValues.length ? _ax(t.yValues[i]) : '$i';
            },
          ),
        ),
        const SizedBox(height: 10),
        if (d.editable)
          FilledButton.icon(
            onPressed: _editCell,
            icon: const Icon(Icons.edit, size: 18),
            label: const Text('Править ячейку'),
          )
        else
          const Text('Карта read-only (нет обратной формулы frexpr)',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
        if (_modified)
          const Padding(
            padding: EdgeInsets.only(top: 6),
            child: Text('Есть несохранённые правки — жми V8MOD',
                style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          ),
      ]),
    );
  }

  String _ax(double v) => v.abs() >= 1000
      ? v.toStringAsFixed(0)
      : v.abs() >= 100
          ? v.toStringAsFixed(0)
          : v.toStringAsFixed(1);

  Widget _info(String k, String v) => RichText(
        text: TextSpan(children: [
          TextSpan(text: '$k: ', style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextSpan(text: v, style: const TextStyle(fontSize: 11, color: HeatColors.text)),
        ]),
      );

  Future<void> _editCell() async {
    final t = _table!;
    final d = widget.def;
    final rCtl = TextEditingController();
    final cCtl = TextEditingController();
    final vCtl = TextEditingController();
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: const Text('Правка ячейки', style: TextStyle(fontSize: 15)),
        content: Column(mainAxisSize: MainAxisSize.min, children: [
          TextField(controller: rCtl, keyboardType: TextInputType.number,
              decoration: InputDecoration(labelText: 'Строка 0..${d.rows - 1} (Y)')),
          TextField(controller: cCtl, keyboardType: TextInputType.number,
              decoration: InputDecoration(labelText: 'Колонка 0..${d.cols - 1} (X)')),
          TextField(controller: vCtl,
              keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
              decoration: InputDecoration(labelText: 'Новое значение, ${d.units}')),
        ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('ОТМЕНА')),
          FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('ЗАПИСАТЬ')),
        ],
      ),
    );
    if (ok != true) return;
    final r = int.tryParse(rCtl.text) ?? -1;
    final c = int.tryParse(cCtl.text) ?? -1;
    final v = double.tryParse(vCtl.text.replaceAll(',', '.'));
    if (r < 0 || r >= d.rows || c < 0 || c >= d.cols || v == null) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Некорректные данные')));
      return;
    }
    setState(() {
      t.setCell(r, c, v);
      _modified = true;
    });
  }

  Future<void> _saveMod() async {
    final path = await ConnectionService.I.rom.saveMod(_table!);
    if (!mounted) return;
    if (path != null) {
      setState(() => _modified = false);
      ScaffoldMessenger.of(context)
          .showSnackBar(SnackBar(content: Text('Сохранено: ${path.split('/').last}')));
      await Share.shareXFiles([XFile(path)]);
    } else {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Карта read-only')));
    }
  }

  Future<void> _exportCsv() async {
    final t = _table!;
    final buf = StringBuffer();
    for (final row in t.toCsvRows()) {
      buf.writeln(row.join(';'));
    }
    await Share.share(buf.toString(), subject: widget.def.name);
  }
}
''')
print('OK  rom_screen.dart')

# ============ lib/screens/rom_diff_screen.dart ============
with open('lib/screens/rom_diff_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/rom_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ROM Diff: сравнение двух прошивок (как в V6) — дельта-карты, отчёт по всем картам.
class RomDiffScreen extends StatefulWidget {
  const RomDiffScreen({super.key});

  @override
  State<RomDiffScreen> createState() => _RomDiffScreenState();
}

class _Diff {
  final RomTableDef def;
  final int cells;
  final double maxDiff;
  final RomTable ta;
  final RomTable tb;
  _Diff(this.def, this.cells, this.maxDiff, this.ta, this.tb);
}

class _RomDiffScreenState extends State<RomDiffScreen> {
  final _romA = RomService();
  final _romB = RomService();
  List<_Diff>? _report;
  bool _busy = false;
  String _err = '';

  Future<void> _pick(bool isA) async {
    final rom = isA ? _romA : _romB;
    final err = await rom.pickAndLoad();
    setState(() {
      _err = err ?? '';
      _report = null;
    });
    if (err == null && _romA.loaded && _romB.loaded) {
      await _compare();
    }
  }

  Future<void> _compare() async {
    setState(() { _busy = true; _err = ''; });
    await Future.delayed(const Duration(milliseconds: 30));
    final out = <_Diff>[];
    for (final def in SubaruRom.tables) {
      try {
        final ta = _romA.readTable(def);
        final tb = _romB.readTable(def);
        var cells = 0;
        var maxDiff = 0.0;
        for (var r = 0; r < def.rows; r++) {
          for (var c = 0; c < def.cols; c++) {
            final dv = (ta.z[r][c] - tb.z[r][c]).abs();
            if (dv > 0.0001) {
              cells++;
              if (dv > maxDiff) maxDiff = dv;
            }
          }
        }
        if (cells > 0) out.add(_Diff(def, cells, maxDiff, ta, tb));
      } catch (_) {}
    }
    out.sort((a, b) => b.maxDiff.compareTo(a.maxDiff));
    setState(() { _report = out; _busy = false; });
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            Expanded(
              child: OutlinedButton.icon(
                onPressed: () => _pick(true),
                icon: Icon(_romA.loaded ? Icons.check_circle : Icons.file_upload_outlined,
                    size: 16, color: _romA.loaded ? Colors.greenAccent : null),
                label: Text(_romA.loaded ? _romA.fileName : 'ROM A (сток)',
                    overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 11)),
              ),
            ),
            const Padding(
                padding: EdgeInsets.symmetric(horizontal: 6),
                child: Icon(Icons.compare_arrows, color: HeatColors.dim)),
            Expanded(
              child: OutlinedButton.icon(
                onPressed: () => _pick(false),
                icon: Icon(_romB.loaded ? Icons.check_circle : Icons.file_upload_outlined,
                    size: 16, color: _romB.loaded ? Colors.greenAccent : null),
                label: Text(_romB.loaded ? _romB.fileName : 'ROM B (мод)',
                    overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 11)),
              ),
            ),
          ]),
        ),
        if (_busy) const LinearProgressIndicator(minHeight: 3),
        if (_err.isNotEmpty)
          Padding(padding: const EdgeInsets.all(8), child: Text(_err, style: const TextStyle(color: Colors.redAccent))),
        Expanded(
          child: _report == null
              ? const Center(
                  child: Text('Выбери два .bin — покажу карты с различиями\n(ячейки, max Δ, цветная дельта)',
                      textAlign: TextAlign.center, style: TextStyle(color: HeatColors.dim, fontSize: 12)))
              : _report!.isEmpty
                  ? const Center(child: Text('Различий нет', style: TextStyle(color: Colors.greenAccent)))
                  : ListView.builder(
                      itemCount: _report!.length,
                      itemBuilder: (ctx, i) {
                        final d = _report![i];
                        return ListTile(
                          dense: true,
                          leading: const Icon(Icons.difference, color: HeatColors.gold, size: 20),
                          title: Text(d.def.name, style: const TextStyle(fontSize: 13)),
                          subtitle: Text(
                              '${d.def.category} · ячеек отличается: ${d.cells}/${d.def.rows * d.def.cols} · max Δ ${d.maxDiff.toStringAsFixed(2)} ${d.def.units}',
                              style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                          trailing: const Icon(Icons.chevron_right, size: 18),
                          onTap: () => Navigator.push(
                              context, MaterialPageRoute(builder: (_) => _DiffPage(diff: d))),
                        );
                      },
                    ),
        ),
      ],
    );
  }
}

class _DiffPage extends StatelessWidget {
  final _Diff diff;
  const _DiffPage({required this.diff});

  @override
  Widget build(BuildContext context) {
    final d = diff.def;
    final hi = diff.maxDiff <= 0 ? 1.0 : diff.maxDiff;
    return Scaffold(
      appBar: AppBar(title: Text('Δ ${d.name}', style: const TextStyle(fontSize: 14))),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Text('A − B · синий = ниже, красный = выше · шкала ±${hi.toStringAsFixed(2)} ${d.units}',
            style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 420,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: 'Δ ${d.units}',
            hintLo: -hi,
            hintHi: hi,
            value: (r, c) => diff.ta.z[d.rows - 1 - r][c] - diff.tb.z[d.rows - 1 - r][c],
            xLabel: (c) => c < diff.ta.xValues.length ? diff.ta.xValues[c].toStringAsFixed(0) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < diff.ta.yValues.length ? diff.ta.yValues[i].toStringAsFixed(1) : '$i';
            },
          ),
        ),
      ]),
    );
  }
}
''')
print('OK  rom_diff_screen.dart')

# ============ lib/screens/service_screen.dart ============
with open('lib/screens/service_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:math' as math;

import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/dtc_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

/// Сервис: диагностические операции с монопольным доступом к шине.
/// Запись в ECU из приложения НЕ выполняется (только .bin через ROM-экран).
class ServiceScreen extends StatefulWidget {
  const ServiceScreen({super.key});

  @override
  State<ServiceScreen> createState() => _ServiceScreenState();
}

class _ServiceScreenState extends State<ServiceScreen> {
  final List<String> _log = ['готов. операции ниже — опрос при этом на паузе.'];
  bool _busy = false;

  void _say(String s) => setState(() => _log.insert(0, s));

  Future<void> _run(String label, Future<String> Function(SsmElm elm) job) async {
    if (_busy) return;
    setState(() => _busy = true);
    _say('▶ $label ...');
    try {
      final r = await ConnectionService.I.exclusive(job);
      _say('✔ $label: $r');
    } catch (e) {
      _say('✖ $label: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final elm = ConnectionService.I.elm;
    final connected = elm.state == SsmState.ecuReady || elm.state == SsmState.polling;
    return Column(
      children: [
        Expanded(
          flex: 5,
          child: ListView(padding: const EdgeInsets.all(12), children: [
            _op('SSM PING · 20 кадров чтения ОЖ', Icons.speed, connected && !_busy,
                () => _run('PING', (elm) async {
                      final times = <double>[];
                      var err = 0;
                      for (var i = 0; i < 20; i++) {
                        final d = await elm.readBytes(0x000008, 1);
                        if (d == null) {
                          err++;
                        } else {
                          times.add(elm.stats.lastMs);
                        }
                        await Future<void>.delayed(const Duration(milliseconds: 5));
                      }
                      if (times.isEmpty) return 'все 20 кадров потеряны';
                      final mn = times.reduce(math.min), mx = times.reduce(math.max);
                      final avg = times.fold(0.0, (a, b) => a + b) / times.length;
                      return 'min ${mn.toStringAsFixed(0)} / avg ${avg.toStringAsFixed(0)} / max ${mx.toStringAsFixed(0)} мс · потерь $err';
                    })),
            _op('Повторный ECU INIT (BF)', Icons.restart_alt, connected && !_busy,
                () => _run('ECU INIT', (elm) async {
                      final id = await elm.ecuInit();
                      return id == null ? 'ECU не ответил' : 'OK, ECU ID = ${elm.ecuId.isEmpty ? id : elm.ecuId}';
                    })),
            _op('Стирание CEL (OBD mode 04)', Icons.cleaning_services_outlined, connected && !_busy,
                () => _run('CLEAR CEL', (elm) async {
                      final ok = await DtcService.clear(elm);
                      return ok ? 'подтверждено (44)' : 'ECU не подтвердил';
                    })),
            _op('Напряжение адаптера (ATRV)', Icons.bolt, connected && !_busy,
                () => _run('ATRV', (elm) async => (await elm.transact('ATRV')).trim())),
            _op('Чтение 6 ключевых параметров разом', Icons.punch_clock, connected && !_busy,
                () => _run('SNAPSHOT', (elm) async {
                      // блок 0x000008..0x000031 одним кадром — демонстрация блочного чтения
                      final d = await elm.readBytes(0x000008, 0x2A);
                      if (d == null) return 'нет ответа';
                      final ect = d[0] - 40;
                      final iat = d[1] - 40;
                      final tps = d[0x15 - 0x08] * 100 / 255;
                      final rpm = ((d[0x0E - 0x08] << 8) | d[0x0F - 0x08]) / 4;
                      final spd = d[0x10 - 0x08];
                      final kca = d[0x22 - 0x08] / 2 - 20;
                      return 'ECT $ect°C · IAT $iat°C · RPM ${rpm.toStringAsFixed(0)} · '
                          '$spd км/ч · TPS ${tps.toStringAsFixed(0)}% · KCA ${kca.toStringAsFixed(1)}°';
                    })),
            const Padding(
              padding: EdgeInsets.symmetric(vertical: 8),
              child: Text(
                'Правка прошивки в самом ECU из V8 не выполняется: только чтение карт (ЭБУ Карты) '
                'и правка .bin-файла (ROM → V8MOD). Это осознанно безопасно для первой версии.',
                style: TextStyle(fontSize: 11, color: HeatColors.dim),
              ),
            ),
          ]),
        ),
        Expanded(
          flex: 4,
          child: Container(
            color: HeatColors.panel,
            child: ListView.builder(
              reverse: true,
              padding: const EdgeInsets.all(10),
              itemCount: _log.length,
              itemBuilder: (ctx, i) => Text(_log[i],
                  style: const TextStyle(fontSize: 11, color: HeatColors.text, height: 1.5)),
            ),
          ),
        ),
      ],
    );
  }

  Widget _op(String label, IconData icon, bool enabled, VoidCallback onTap) {
    return Padding(
      padding: const EdgeInsets.only(bottom: 8),
      child: ListTile(
        tileColor: HeatColors.panel,
        shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(10)),
        leading: Icon(icon, color: HeatColors.accent),
        title: Text(label, style: const TextStyle(fontSize: 13)),
        trailing: const Icon(Icons.play_arrow, size: 18),
        enabled: enabled,
        onTap: onTap,
      ),
    );
  }
}
''')
print('OK  service_screen.dart')

# ============ lib/screens/perf_screen.dart ============
with open('lib/screens/perf_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';

import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// Замер (как в V6): 0-60, 0-100, 100-200, 402 м — по скорости VSS.
class PerfScreen extends StatefulWidget {
  const PerfScreen({super.key});

  @override
  State<PerfScreen> createState() => _PerfScreenState();
}

class _Meas {
  final Map<int, double> times; // speed -> seconds
  final double q402m;
  final double q1km;
  _Meas(this.times, this.q402m, this.q1km);
}

class _PerfScreenState extends State<PerfScreen> {
  static const _targets = [60, 100, 150, 200];
  String _state = 'disarmed';
  final Map<int, double> _times = {};
  double _dist = 0;
  double _t0 = 0, _t100 = -1;
  DateTime? _lastTs;
  double _lastSpeed = 0;
  double? _q402, _q1km;
  final List<_Meas> _results = [];
  StreamSubscription? _sub;

  void _reset() {
    setState(() {
      _state = 'armed';
      _times.clear();
      _dist = 0;
      _q402 = null;
      _q1km = null;
      _t100 = -1;
      _lastTs = null;
    });
  }

  void _onSnap(LiveSnapshot s) {
    final v = s.speed;
    final now = s.ts;
    if (_state == 'armed') {
      if (v < 2 && _lastSpeed > 5) { _dist = 0; _times.clear(); }
      if (v > 3 && _lastSpeed <= 3) {
        _state = 'run';
        _t0 = now.millisecondsSinceEpoch / 1000.0;
        _dist = 0;
      }
    } else if (_state == 'run') {
      if (_lastTs != null) {
        final dt = now.difference(_lastTs!).inMilliseconds / 1000.0;
        _dist += (_lastSpeed / 3.6) * dt;
      }
      final t = now.millisecondsSinceEpoch / 1000.0 - _t0;
      for (final tg in _targets) {
        if (!_times.containsKey(tg) && v >= tg) {
          _times[tg] = t;
          if (tg == 100 && _t100 < 0) _t100 = t;
        }
      }
      if (_q402 == null && _dist >= 402) _q402 = t;
      if (_q1km == null && _dist >= 1000) _q1km = t;
      if (v >= 200 || (_times.length == _targets.length && _dist >= 1000) ||
          (v < 2 && _dist > 30)) {
        _finish();
      }
    }
    _lastSpeed = v;
    _lastTs = now;
    if (mounted) setState(() {});
  }

  void _finish() {
    _state = 'done';
    _results.insert(0, _Meas(Map.of(_times), _q402 ?? 0, _q1km ?? 0));
  }

  @override
  void initState() {
    super.initState();
    final p = ConnectionService.I.poller;
    if (p != null) _sub = p.snapshots.listen(_onSnap);
  }

  @override
  void dispose() {
    _sub?.cancel();
    super.dispose();
  }

  String _fmt(double? v) => v == null ? '--' : v.toStringAsFixed(2);

  @override
  Widget build(BuildContext context) {
    final connected = ConnectionService.I.poller != null;
    if (!connected) {
      return const Center(
          child: Text('Подключись в настройках', style: TextStyle(color: HeatColors.dim)));
    }
    return ListView(padding: const EdgeInsets.all(12), children: [
      Container(
        padding: const EdgeInsets.all(14),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Row(children: [
          Expanded(
            child: Text(
              _state == 'armed'
                  ? 'ЖДУ СТАРТ: газ в пол с места'
                  : _state == 'run'
                      ? 'ИДЁТ ЗАМЕР · ${_dist.toStringAsFixed(0)} м'
                      : _state == 'done'
                          ? 'ЗАМЕР ЗАВЕРШЁН'
                          : 'Нажми АРМИРОВАТЬ',
              style: const TextStyle(fontWeight: FontWeight.w700),
            ),
          ),
          FilledButton(
              onPressed: _reset,
              child: Text(_state == 'armed' ? 'СБРОС' : 'АРМИРОВАТЬ')),
        ]),
      ),
      if (_state == 'run' || _state == 'done')
        Container(
          margin: const EdgeInsets.only(top: 12),
          padding: const EdgeInsets.all(14),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Row(
            children: [
              for (final tg in _targets)
                Expanded(
                  child: Column(children: [
                    Text('0-$tg', style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                    Text(_fmt(_times[tg]),
                        style: const TextStyle(fontSize: 17, fontWeight: FontWeight.w800)),
                  ]),
                ),
              Expanded(
                child: Column(children: [
                  const Text('402м', style: TextStyle(fontSize: 9, color: HeatColors.dim)),
                  Text(_fmt(_q402),
                      style: const TextStyle(fontSize: 17, fontWeight: FontWeight.w800, color: HeatColors.gold)),
                ]),
              ),
            ],
          ),
        ),
      const SizedBox(height: 12),
      ..._results.map((r) => Container(
            margin: const EdgeInsets.only(bottom: 8),
            padding: const EdgeInsets.all(12),
            decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(10)),
            child: Text(
              '0-60: ${_fmt(r.times[60])}c · 0-100: ${_fmt(r.times[100])}c · '
              '0-150: ${_fmt(r.times[150])}c · 0-200: ${_fmt(r.times[200])}c · '
              '402м: ${r.q402m > 0 ? r.q402m.toStringAsFixed(2) : '--'}c',
              style: const TextStyle(fontSize: 12),
            ),
          )),
    ]);
  }
}
''')
print('OK  perf_screen.dart')

# ============ lib/screens/terminal_screen.dart ============
with open('lib/screens/terminal_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

/// Терминал: сырые AT/hex-команды ELM327 (как в V6), ответ как есть.
class TerminalScreen extends StatefulWidget {
  const TerminalScreen({super.key});

  @override
  State<TerminalScreen> createState() => _TerminalScreenState();
}

class _TerminalScreenState extends State<TerminalScreen> {
  final _ctl = TextEditingController();
  final List<String> _log = ['SSM2/ELM терминал. Примеры ниже. Отправка ставит опрос на паузу.'];
  bool _busy = false;

  static const _quick = [
    'ATRV',
    '8010F001BF40',
    'A8000000080E',
    'A8000000F900',
    'A8 00 00 01 99 00',
  ];

  Future<void> _send(String cmdRaw) async {
    final cmd = cmdRaw.replaceAll(' ', '').toUpperCase();
    if (cmd.isEmpty || _busy) return;
    setState(() { _busy = true; _log.add('> $cmd'); });
    try {
      final r = await ConnectionService.I.exclusive((elm) => elm.transact(cmd, timeoutMs: 1500));
      final text = r.trim().isEmpty ? '(пусто/timeout)' : r.trim();
      _log.add(text);
    } catch (e) {
      _log.add('ERR $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final connected = ConnectionService.I.elm.state == SsmState.ecuReady ||
        ConnectionService.I.elm.state == SsmState.polling;
    return Column(
      children: [
        SizedBox(
          height: 42,
          child: ListView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 6),
            children: _quick
                .map((q) => Padding(
                      padding: const EdgeInsets.only(right: 6),
                      child: ActionChip(
                        label: Text(q, style: const TextStyle(fontSize: 10)),
                        onPressed: connected && !_busy ? () => _send(q) : null,
                      ),
                    ))
                .toList(),
          ),
        ),
        Expanded(
          child: Container(
            margin: const EdgeInsets.all(8),
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(
                color: const Color(0xFF04070F), borderRadius: BorderRadius.circular(10)),
            child: ListView.builder(
              reverse: true,
              itemCount: _log.length,
              itemBuilder: (ctx, i) {
                final line = _log[_log.length - 1 - i];
                final isCmd = line.startsWith('>');
                return Text(line,
                    style: TextStyle(
                        fontSize: 11.5,
                        height: 1.45,
                        color: isCmd ? HeatColors.gold : HeatColors.text));
              },
            ),
          ),
        ),
        Padding(
          padding: const EdgeInsets.fromLTRB(8, 0, 8, 8),
          child: Row(children: [
            Expanded(
              child: TextField(
                controller: _ctl,
                style: const TextStyle(fontSize: 13),
                decoration: InputDecoration(
                  hintText: 'hex без пробелов: A8000000080E',
                  isDense: true,
                  filled: true,
                  fillColor: HeatColors.panel,
                  border: OutlineInputBorder(
                      borderRadius: BorderRadius.circular(8), borderSide: BorderSide.none),
                ),
                onSubmitted: connected ? _send : null,
              ),
            ),
            const SizedBox(width: 8),
            IconButton.filled(
              onPressed: connected && !_busy ? () { _send(_ctl.text); _ctl.clear(); } : null,
              icon: const Icon(Icons.send, size: 18),
            ),
          ]),
        ),
      ],
    );
  }
}
''')
print('OK  terminal_screen.dart')
print()
print('=' * 64)
print('  Настройческие экраны готовы (8 шт). Далее -> ячейка 9/10 (PID/Профили/Экспорт/Настройки)')
print('=' * 64)


OK  dtc_screen.dart
OK  analyzer_screen.dart (сплиттер слияния CSV)
OK  ecu_maps_screen.dart (живое чтение карт из ECU)
OK  rom_screen.dart
OK  rom_diff_screen.dart
OK  service_screen.dart
OK  perf_screen.dart
OK  terminal_screen.dart

  Настройческие экраны готовы (8 шт). Далее -> ячейка 9/10 (PID/Профили/Экспорт/Настройки)


In [ ]:
# @title 🗃️ Ячейка 9/10: PID CRUD · Профили · Экспорт · Настройки (+ тест PID живьём)
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/screens/custom_pid_screen.dart ============
with open('lib/screens/custom_pid_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/custom_pid_service.dart';
import '../services/expr_eval.dart';
import '../widgets/heat_colors.dart';

/// PID CRUD (как в V6): добавить/изменить/удалить кастомные PID,
/// живой тест значения с ECU, автовключение в блочный план опроса.
class CustomPidScreen extends StatefulWidget {
  const CustomPidScreen({super.key});

  @override
  State<CustomPidScreen> createState() => _CustomPidScreenState();
}

class _CustomPidScreenState extends State<CustomPidScreen> {
  static const _storages = ['uint8', 'int8', 'uint16', 'int16', 'float'];
  static const _lens = [1, 2, 4];

  Future<void> _edit([CustomPid? existing]) async {
    final pid = existing ??
        CustomPid(id: DateTime.now().millisecondsSinceEpoch.toString(), name: '', address: 0x000000);
    final name = TextEditingController(text: pid.name);
    final addr = TextEditingController(
        text: pid.address.toRadixString(16).toUpperCase().padLeft(6, '0'));
    final expr = TextEditingController(text: pid.expr);
    final unit = TextEditingController(text: pid.unit);
    var len = pid.len;
    var storage = pid.storage;
    var prio = pid.priority;
    String testResult = '';

    final saved = await showDialog<bool>(
      context: context,
      builder: (ctx) => StatefulBuilder(
        builder: (ctx, setS) => AlertDialog(
          backgroundColor: HeatColors.panel,
          title: Text(existing == null ? 'Новый PID' : 'Правка PID', style: const TextStyle(fontSize: 15)),
          content: SingleChildScrollView(
            child: Column(mainAxisSize: MainAxisSize.min, children: [
              TextField(controller: name, decoration: const InputDecoration(labelText: 'Имя (например "EGT")')),
              TextField(controller: addr,
                  decoration: const InputDecoration(labelText: 'Адрес hex, 6 знаков (0xFF70C6)')),
              Row(children: [
                Expanded(
                  child: DropdownButtonFormField<int>(
                    initialValue: len,
                    decoration: const InputDecoration(labelText: 'Байт'),
                    items: _lens.map((l) => DropdownMenuItem(value: l, child: Text('$l'))).toList(),
                    onChanged: (v) => setS(() => len = v ?? 1),
                  ),
                ),
                const SizedBox(width: 8),
                Expanded(
                  child: DropdownButtonFormField<String>(
                    initialValue: storage,
                    decoration: const InputDecoration(labelText: 'Тип'),
                    items: _storages.map((s) => DropdownMenuItem(value: s, child: Text(s))).toList(),
                    onChanged: (v) => setS(() => storage = v ?? 'uint8'),
                  ),
                ),
              ]),
              TextField(controller: expr,
                  decoration: const InputDecoration(
                      labelText: 'Формула от x (напр. x*0.01953125  или  (x-40))')),
              Row(children: [
                Expanded(
                  child: TextField(controller: unit,
                      decoration: const InputDecoration(labelText: 'Единицы')),
                ),
                const SizedBox(width: 8),
                Expanded(
                  child: DropdownButtonFormField<int>(
                    initialValue: prio,
                    decoration: const InputDecoration(labelText: 'Ярус'),
                    items: const [
                      DropdownMenuItem(value: 1, child: Text('fast')),
                      DropdownMenuItem(value: 2, child: Text('mid')),
                      DropdownMenuItem(value: 3, child: Text('slow')),
                    ],
                    onChanged: (v) => setS(() => prio = v ?? 2),
                  ),
                ),
              ]),
              const SizedBox(height: 10),
              Row(children: [
                Expanded(
                  child: OutlinedButton.icon(
                    icon: const Icon(Icons.bolt, size: 16),
                    label: const Text('ТЕСТ С ECU'),
                    onPressed: () async {
                      try {
                        final a = int.parse(addr.text, radix: 16);
                        final v = await ConnectionService.I.exclusive((elm) async {
                          final d = await elm.readBytes(a, len);
                          if (d == null) return 'нет ответа';
                          final ev = ExprEval(expr.text);
                          final raw = ExprEval.rawOf(d, storage);
                          return 'raw=$raw → ${ev(raw).toStringAsFixed(3)}';
                        });
                        setS(() => testResult = v);
                      } catch (e) {
                        setS(() => testResult = 'ERR $e');
                      }
                    },
                  ),
                ),
              ]),
              if (testResult.isNotEmpty)
                Padding(
                  padding: const EdgeInsets.only(top: 6),
                  child: Text(testResult,
                      style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
                ),
            ]),
          ),
          actions: [
            TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('ОТМЕНА')),
            FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('СОХРАНИТЬ')),
          ],
        ),
      ),
    );
    if (saved != true) return;
    try {
      ExprEval(expr.text); // валидация формулы
    } catch (_) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Формула не парсится')));
      return;
    }
    final a = int.tryParse(addr.text, radix: 16);
    if (a == null || name.text.trim().isEmpty) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Проверь адрес и имя')));
      return;
    }
    await CustomPidService.I.upsert(CustomPid(
      id: pid.id, name: name.text.trim(), unit: unit.text.trim(),
      category: 'custom', address: a, len: len, storage: storage,
      expr: expr.text.trim(), priority: prio,
    ));
    ConnectionService.I.buildPoller();
    ConnectionService.I.startPolling();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final items = CustomPidService.I.items;
    return Scaffold(
      appBar: AppBar(
          title: const Text('Кастомные PID', style: TextStyle(fontSize: 15)),
          automaticallyImplyLeading: false),
      floatingActionButton: FloatingActionButton.small(
          onPressed: () => _edit(), child: const Icon(Icons.add)),
      body: items.isEmpty
          ? const Center(
              child: Text(
                  'Свои PID: адрес + формула → попадут в блочный план опроса\n'
                  'и на приборную панель через канон. Тест с ECU — прямо в редакторе.',
                  textAlign: TextAlign.center,
                  style: TextStyle(color: HeatColors.dim, fontSize: 12)))
          : ListView.builder(
              itemCount: items.length,
              itemBuilder: (ctx, i) {
                final p = items[i];
                return ListTile(
                  dense: true,
                  leading: const Icon(Icons.tune, color: HeatColors.accent),
                  title: Text('${p.name}  [${p.unit}]', style: const TextStyle(fontSize: 13)),
                  subtitle: Text(
                      '0x${p.address.toRadixString(16).toUpperCase().padLeft(6, '0')} · ${p.len}B $p.storage · x → ${p.expr} · prio ${p.priority}'
                          .replaceAll('p.storage', p.storage),
                      style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    IconButton(icon: const Icon(Icons.edit, size: 18), onPressed: () => _edit(p)),
                    IconButton(
                        icon: const Icon(Icons.delete_outline, size: 18),
                        onPressed: () async {
                          await CustomPidService.I.remove(p.id);
                          ConnectionService.I.buildPoller();
                          setState(() {});
                        }),
                  ]),
                );
              },
            ),
    );
  }
}
''')
print('OK  custom_pid_screen.dart')

# ============ lib/screens/profile_screen.dart ============
with open('lib/screens/profile_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/profile_service.dart';
import '../widgets/heat_colors.dart';

/// Профили: ВСЕ настройки опроса одним набором (как в V6).
class ProfileScreen extends StatefulWidget {
  const ProfileScreen({super.key});

  @override
  State<ProfileScreen> createState() => _ProfileScreenState();
}

class _ProfileScreenState extends State<ProfileScreen> {
  final _nameCtl = TextEditingController();

  @override
  Widget build(BuildContext context) {
    final names = ProfileService.I.profiles.keys.toList()..sort();
    return ListView(padding: const EdgeInsets.all(12), children: [
      Container(
        padding: const EdgeInsets.all(12),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Row(children: [
          Expanded(
            child: TextField(
              controller: _nameCtl,
              style: const TextStyle(fontSize: 13),
              decoration: const InputDecoration(
                  hintText: 'имя профиля (напр. "стрит 98 бензин")', isDense: true),
            ),
          ),
          const SizedBox(width: 8),
          FilledButton(
            onPressed: () async {
              final n = _nameCtl.text.trim();
              if (n.isEmpty) return;
              await ProfileService.I.saveCurrent(n);
              _nameCtl.clear();
              setState(() {});
            },
            child: const Text('СОХР. ТЕКУЩИЕ'),
          ),
        ]),
      ),
      const SizedBox(height: 10),
      ...names.map((n) => Padding(
            padding: const EdgeInsets.only(bottom: 8),
            child: ListTile(
              tileColor: HeatColors.panel,
              shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(10)),
              leading: const Icon(Icons.bookmark_outline, color: HeatColors.gold),
              title: Text(n, style: const TextStyle(fontSize: 13.5)),
              subtitle: Text(
                  '${(ProfileService.I.profiles[n]?['enabledIds'] as List?)?.length ?? '?'} PID · '
                  'блок 0x${(ProfileService.I.profiles[n]?['maxBlock'] ?? 0x50).toString()}',
                  style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
              trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                IconButton(
                  icon: const Icon(Icons.play_circle_outline, color: Colors.greenAccent),
                  onPressed: () async {
                    await ProfileService.I.apply(n);
                    ConnectionService.I.buildPoller();
                    ConnectionService.I.startPolling();
                    if (mounted) {
                      ScaffoldMessenger.of(context)
                          .showSnackBar(SnackBar(content: Text('Профиль «$n» применён')));
                    }
                    setState(() {});
                  },
                ),
                IconButton(
                  icon: const Icon(Icons.delete_outline),
                  onPressed: () async {
                    await ProfileService.I.remove(n);
                    setState(() {});
                  },
                ),
              ]),
            ),
          )),
      const Padding(
        padding: EdgeInsets.all(8),
        child: Text('Профиль включает: набор PID, размер блока, ярусы, AT ST, автолог.',
            style: TextStyle(fontSize: 11, color: HeatColors.dim)),
      ),
    ]);
  }
}
''')
print('OK  profile_screen.dart')

# ============ lib/screens/export_screen.dart ============
with open('lib/screens/export_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';

import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../services/connection_service.dart';
import '../services/export_service.dart';
import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';

/// Экспорт (как в V6): карты → CSV (ecuEdit/WinOLS), ROM → HEX, логи.
class ExportScreen extends StatefulWidget {
  const ExportScreen({super.key});

  @override
  State<ExportScreen> createState() => _ExportScreenState();
}

class _ExportScreenState extends State<ExportScreen> {
  bool _busy = false;
  String _msg = '';
  String? _table;

  Future<void> _run(String label, Future<File> Function() job) async {
    if (_busy) return;
    setState(() { _busy = true; _msg = '$label ...'; });
    try {
      final f = await job();
      setState(() => _msg = 'готово: ${f.path.split('/').last}');
      await Share.shareXFiles([XFile(f.path)]);
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final rom = ConnectionService.I.rom;
    final tables = rom.loaded ? rom.defs.map((d) => d.name).toList() : <String>[];
    return ListView(padding: const EdgeInsets.all(12), children: [
      _tile(
        'ВСЕ карты → CSV (ecuEdit-стиль, Excel)',
        Icons.table_view,
        rom.loaded && !_busy,
        () => _run('Экспорт всех карт', () => ExportService.exportAllTablesCsv(rom)),
      ),
      Container(
        margin: const EdgeInsets.only(bottom: 12),
        padding: const EdgeInsets.all(10),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('ОДНА карта → WinOLS CSV', style: TextStyle(fontSize: 11, color: HeatColors.gold)),
          const SizedBox(height: 8),
          DropdownButtonHideUnderline(
            child: DropdownButton<String>(
              isExpanded: true,
              dropdownColor: HeatColors.panel,
              hint: const Text('выбери карту', style: TextStyle(fontSize: 12)),
              value: _table,
              items: tables.map((t) => DropdownMenuItem(value: t, child: Text(t, style: const TextStyle(fontSize: 12)))).toList(),
              onChanged: (v) => setState(() => _table = v),
            ),
          ),
          const SizedBox(height: 8),
          FilledButton.tonal(
            onPressed: rom.loaded && _table != null && !_busy
                ? () => _run('Экспорт карты', () => ExportService.exportTableWinols(rom, _table!))
                : null,
            child: const Text('ЭКСПОРТ КАРТЫ'),
          ),
        ]),
      ),
      _tile(
        'ROM → HEX-дамп (64 KB, текст)',
        Icons.code,
        rom.loaded && !_busy,
        () => _run('HEX-дамп', () => ExportService.hexDump(rom.rom!)),
      ),
      _tile(
        'Последний лог → поделиться',
        Icons.share,
        !_busy,
        () => _run('Лог', () async {
          final logs = await LoggerService.listLogs();
          if (logs.isEmpty) throw StateError('нет логов');
          return File(logs.first.path);
        }),
      ),
      if (_msg.isNotEmpty)
        Padding(
          padding: const EdgeInsets.all(8),
          child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
        ),
      if (!rom.loaded)
        const Padding(
          padding: EdgeInsets.all(8),
          child: Text('Для экспорта карт сначала загрузи ROM на экране ROM.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
        ),
    ]);
  }

  Widget _tile(String label, IconData icon, bool enabled, VoidCallback onTap) {
    return Padding(
      padding: const EdgeInsets.only(bottom: 12),
      child: ListTile(
        tileColor: HeatColors.panel,
        shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(12)),
        leading: Icon(icon, color: HeatColors.accent),
        title: Text(label, style: const TextStyle(fontSize: 13)),
        trailing: const Icon(Icons.chevron_right),
        enabled: enabled,
        onTap: onTap,
      ),
    );
  }
}
''')
print('OK  export_screen.dart')

# ============ lib/screens/settings_screen.dart ============
with open('lib/screens/settings_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class SettingsScreen extends StatefulWidget {
  const SettingsScreen({super.key});

  @override
  State<SettingsScreen> createState() => _SettingsScreenState();
}

class _SettingsScreenState extends State<SettingsScreen> {
  List<BluetoothDevice> _devices = [];
  bool _busy = false;
  String _msg = '';

  SettingsService get st => SettingsService.I;
  ConnectionService get svc => ConnectionService.I;

  @override
  void initState() {
    super.initState();
    _refreshDevices();
  }

  Future<void> _refreshDevices() async {
    try {
      await FlutterBluetoothSerial.instance.requestEnable();
      final list = await FlutterBluetoothSerial.instance.getBondedDevices();
      if (mounted) setState(() => _devices = list);
    } catch (_) {}
  }

  @override
  Widget build(BuildContext context) {
    final stats = svc.elm.stats;
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        _card('ПОДКЛЮЧЕНИЕ ELM327', [
          DropdownButtonHideUnderline(
            child: DropdownButton<String>(
              isExpanded: true,
              dropdownColor: HeatColors.panel,
              hint: const Text('выбери сопряжённый ELM327'),
              value: st.btAddress.isEmpty ? null : st.btAddress,
              items: _devices
                  .map((d) => DropdownMenuItem(
                      value: d.address,
                      child: Text('${d.name ?? '?'} · ${d.address}', style: const TextStyle(fontSize: 13))))
                  .toList(),
              onChanged: (v) {
                if (v == null) return;
                final d = _devices.firstWhere((e) => e.address == v);
                st.btAddress = v;
                st.btName = d.name ?? '';
                st.save();
                setState(() {});
              },
            ),
          ),
          const SizedBox(height: 8),
          Row(children: [
            Expanded(
              child: FilledButton.icon(
                onPressed: _busy ? null : _connect,
                icon: const Icon(Icons.electric_bolt),
                label: Text(svc.elm.state == SsmState.polling ? 'ПЕРЕПОДКЛЮЧИТЬ' : 'CONNECT + INIT ECU'),
              ),
            ),
            const SizedBox(width: 8),
            IconButton(onPressed: _refreshDevices, icon: const Icon(Icons.refresh)),
            IconButton(
              onPressed: () async { await svc.disconnect(); setState(() {}); },
              icon: const Icon(Icons.link_off, color: Colors.redAccent),
            ),
          ]),
          if (_msg.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 8),
              child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
            ),
          if (svc.elm.ecuId.isNotEmpty)
            _kv('ECU ID', '${svc.elm.ecuId} (ожидался ${AppConstants.ecuIdExpected})'),
          if (svc.elm.elmVersion.isNotEmpty) _kv('Адаптер', svc.elm.elmVersion),
        ]),
        _card('СТАТИСТИКА ПРОТОКОЛА (монитор скорости)', [
          Row(children: [
            _stat('${stats.hz.toStringAsFixed(1)}', 'снапш./с'),
            _stat('${stats.avgMs.toStringAsFixed(0)} мс', 'на CAN-кадр'),
            _stat('${svc.poller?.blockCount ?? 0}', 'блока'),
            _stat('${svc.poller?.pidCount ?? 0}', 'PID'),
          ]),
          const SizedBox(height: 6),
          _kv('Кадров OK / ошибок / NO DATA', '${stats.framesOk} / ${stats.framesErr} / ${stats.noData}'),
          Row(children: [
            Expanded(
              child: OutlinedButton(
                onPressed: () { svc.startPolling(); setState(() {}); },
                child: const Text('СТАРТ ОПРОС'),
              ),
            ),
            const SizedBox(width: 8),
            Expanded(
              child: OutlinedButton(
                onPressed: () { svc.stopPolling(); setState(() {}); },
                child: const Text('СТОП'),
              ),
            ),
          ]),
        ]),
        _card('ТЮНИНГ ОПРОСА (блоки SSM2)', [
          _slider('Макс. размер блока', st.maxBlock.toDouble(), 0x20, 0x80, (v) {
            st.maxBlock = v.round() & ~0xF;
          }, '0x${st.maxBlock.toRadixString(16).toUpperCase()}'),
          _slider('Средний ярус: каждый N цикл', st.midEveryN.toDouble(), 2, 10, (v) {
            st.midEveryN = v.round();
          }, '${st.midEveryN}'),
          _slider('Медленный ярус: каждый N цикл', st.slowEveryN.toDouble(), 10, 60, (v) {
            st.slowEveryN = v.round();
          }, '${st.slowEveryN}'),
          _slider('AT ST (×4 мс ожидание байта)', st.stCode.toDouble(), 2, 30, (v) {
            st.stCode = v.round();
            svc.elm.stTimeoutCode = st.stCode;
          }, '${st.stCode * 4} мс'),
          FilledButton.tonalIcon(
            onPressed: () {
              st.save();
              svc.buildPoller();
              svc.startPolling();
              setState(() {});
            },
            icon: const Icon(Icons.autorenew, size: 18),
            label: const Text('ПЕРЕСТРОИТЬ БЛОКИ И ОПРОС'),
          ),
        ]),
        _card('PID В ОПРОСЕ (${st.enabledIds.length})', [
          Row(children: [
            TextButton(
              onPressed: () {
                st.enabledIds = SubaruPids.defaults.map((e) => e.id).toSet();
                st.save();
                svc.buildPoller();
                setState(() {});
              },
              child: const Text('только канонические'),
            ),
            TextButton(
              onPressed: () {
                st.enabledIds = SubaruPids.all.map((e) => e.id).toSet();
                st.save();
                setState(() {});
              },
              child: const Text('ВСЕ 150+'),
            ),
            TextButton(
              onPressed: () {
                st.enabledIds = {};
                st.save();
                setState(() {});
              },
              child: const Text('ничего'),
            ),
          ]),
          ...SubaruPids.byCategory().entries.map((entry) => ExpansionTile(
                dense: true,
                title: Text('${entry.key} (${entry.value.length})', style: const TextStyle(fontSize: 13)),
                children: entry.value
                    .map((p) => CheckboxListTile(
                          dense: true,
                          value: st.enabledIds.contains(p.id),
                          onChanged: (v) {
                            if (v == true) {
                              st.enabledIds.add(p.id);
                            } else {
                              st.enabledIds.remove(p.id);
                            }
                            st.save();
                            setState(() {});
                          },
                          title: Text(p.name, style: const TextStyle(fontSize: 12)),
                          subtitle: Text(
                              '${p.addrHex} · ${p.unit} · prio ${p.priority}${p.canon.isNotEmpty ? ' · canon ${p.canon}' : ''}',
                              style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                        ))
                    .toList(),
              )),
        ]),
        _card('О ПРОШИВКЕ', [
          _kv('CAL ID', AppConstants.calId),
          _kv('ECU ID', AppConstants.ecuIdExpected),
          _kv('Двигатель', AppConstants.engine),
          _kv('Протокол', 'SSM2 over CAN · ISO 15765-4 · 0x7E0/0x7E8 · 500 kbit · 11 bit'),
          _kv('Генератор', 'RomRaider logger.xml + ECUFlash A2TB100B(+K) → встроено'),
        ]),
      ],
    );
  }

  Widget _card(String title, List<Widget> children) {
    return Container(
      margin: const EdgeInsets.only(bottom: 12),
      padding: const EdgeInsets.all(12),
      decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Text(title,
            style: const TextStyle(
                fontSize: 12, fontWeight: FontWeight.w800, letterSpacing: 1, color: HeatColors.gold)),
        const SizedBox(height: 8),
        ...children,
      ]),
    );
  }

  Widget _kv(String k, String v) => Padding(
        padding: const EdgeInsets.only(top: 4),
        child: RichText(
          text: TextSpan(children: [
            TextSpan(text: '$k: ', style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
            TextSpan(text: v, style: const TextStyle(fontSize: 12, color: HeatColors.text)),
          ]),
        ),
      );

  Widget _stat(String v, String l) => Expanded(
        child: Column(children: [
          Text(v, style: const TextStyle(fontSize: 18, fontWeight: FontWeight.w800, color: HeatColors.text)),
          Text(l, style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
        ]),
      );

  Widget _slider(String label, double v, double min, double max, void Function(double) on, String shown) {
    return Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
      Row(children: [
        Expanded(child: Text(label, style: const TextStyle(fontSize: 12))),
        Text(shown, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
      ]),
      Slider(
        value: v.clamp(min, max), min: min, max: max,
        divisions: (max - min).round(),
        onChanged: (nv) { on(nv); setState(() {}); },
      ),
    ]);
  }

  Future<void> _connect() async {
    setState(() { _busy = true; _msg = 'подключаюсь...'; });
    final r = await svc.connectAndInit();
    if (!mounted) return;
    setState(() { _busy = false; _msg = r; });
  }
}
''')
print('OK  settings_screen.dart')
print()
print('=' * 64)
print('  Все 17 экранов готовы. Далее -> ячейка 10/10 (сборка APK)')
print('=' * 64)


OK  custom_pid_screen.dart
OK  profile_screen.dart
OK  export_screen.dart
OK  settings_screen.dart

  Все 17 экранов готовы. Далее -> ячейка 10/10 (сборка APK)


In [ ]:
# @title 🩹 ФИКС 1/3: Bluetooth — список сопряжённых устройств (settings_screen + main)
# Запускать ПОСЛЕ ячейки 9/10 и ПЕРЕД ячейкой 10/10 (сборка APK)
import os, re
os.chdir('/content/suba_run_v8')

# ── 1. Новый сервис разрешений ───────────────────────────────────────────────
with open('lib/services/bt_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'package:permission_handler/permission_handler.dart';

/// Единая точка работы с классическим Bluetooth.
/// В V8 список устройств был пуст, потому что getBondedDevices() вызывался
/// РАНЬШЕ, чем пользователь выдавал BLUETOOTH_CONNECT (Android 12+),
/// а исключение глушилось пустым catch.
class BtService {
  static final BtService I = BtService._();
  BtService._();

  String lastError = '';

  Future<bool> ensurePermissions() async {
    lastError = '';
    if (!Platform.isAndroid) return true;

    // Android 12+ : SCAN + CONNECT. Android <= 11: location.
    final res = await [
      Permission.bluetoothConnect,
      Permission.bluetoothScan,
    ].request();

    var granted = res.values.every((s) => s.isGranted);

    if (!granted) {
      // старые прошивки: этих permission нет -> проверяем гео
      final loc = await Permission.locationWhenInUse.request();
      granted = loc.isGranted;
    }
    if (!granted) {
      lastError = 'Нет разрешения Bluetooth. Открой настройки приложения '
          'и разреши «Устройства поблизости».';
    }
    return granted;
  }

  Future<bool> ensureEnabled() async {
    try {
      final on = await FlutterBluetoothSerial.instance.isEnabled ?? false;
      if (on) return true;
      final ok = await FlutterBluetoothSerial.instance.requestEnable() ?? false;
      if (!ok) lastError = 'Bluetooth выключен';
      return ok;
    } catch (e) {
      lastError = 'Bluetooth недоступен: $e';
      return false;
    }
  }

  /// Полный цикл: разрешения -> включение -> список сопряжённых.
  Future<List<BluetoothDevice>> bondedDevices() async {
    if (!await ensurePermissions()) return const [];
    if (!await ensureEnabled()) return const [];
    try {
      final list = await FlutterBluetoothSerial.instance.getBondedDevices();
      if (list.isEmpty) {
        lastError = 'Сопряжённых устройств нет. Сопряги ELM327 в настройках '
            'Android (PIN 1234/0000), потом жми «Обновить».';
      }
      // ELM-подобные — наверх
      list.sort((a, b) {
        int rank(BluetoothDevice d) {
          final n = (d.name ?? '').toUpperCase();
          return (n.contains('OBD') || n.contains('ELM') || n.contains('VLINK') ||
                  n.contains('VGATE') || n.contains('KONNWEI')) ? 0 : 1;
        }
        return rank(a).compareTo(rank(b));
      });
      return list;
    } catch (e) {
      lastError = 'getBondedDevices: $e';
      return const [];
    }
  }

  Future<void> openAndroidSettings() => FlutterBluetoothSerial.instance.openSettings();
  Future<void> openAppSettings() => openAppSettings();
}
''')
print('OK  lib/services/bt_service.dart')

# ── 2. Патч settings_screen.dart: блок выбора устройства ─────────────────────
p = 'lib/screens/settings_screen.dart'
s = open(p).read()

s = s.replace(
    "import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';",
    "import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';\n"
    "import 'package:permission_handler/permission_handler.dart' as ph;\n"
    "import '../services/bt_service.dart';", 1)

# новый _refreshDevices с диагностикой
s = re.sub(
    r"  Future<void> _refreshDevices\(\) async \{.*?\n  \}\n",
    r'''  bool _scanning = false;
  String _btError = '';

  Future<void> _refreshDevices() async {
    if (_scanning) return;
    setState(() { _scanning = true; _btError = ''; });
    final list = await BtService.I.bondedDevices();
    if (!mounted) return;
    setState(() {
      _devices = list;
      _btError = BtService.I.lastError;
      _scanning = false;
      // если сохранённый адрес больше не сопряжён — сбрасываем,
      // иначе Dropdown падает с "value not in items" и тянет весь экран
      if (st.btAddress.isNotEmpty &&
          !_devices.any((d) => d.address == st.btAddress)) {
        st.btAddress = '';
        st.btName = '';
        st.save();
      }
    });
  }
''', s, flags=re.S, count=1)

# Dropdown -> устойчивый список + кнопки
old_dd_start = "          DropdownButtonHideUnderline("
i = s.index(old_dd_start)
j = s.index("          const SizedBox(height: 8),", i)
new_dd = '''          if (_scanning)
            const Padding(
              padding: EdgeInsets.symmetric(vertical: 10),
              child: Row(children: [
                SizedBox(width: 16, height: 16, child: CircularProgressIndicator(strokeWidth: 2)),
                SizedBox(width: 10),
                Text('ищу сопряжённые устройства...', style: TextStyle(fontSize: 12)),
              ]),
            )
          else if (_devices.isEmpty)
            Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
              Text(_btError.isEmpty ? 'Сопряжённых устройств не найдено' : _btError,
                  style: const TextStyle(fontSize: 12, color: Colors.orangeAccent)),
              const SizedBox(height: 8),
              Wrap(spacing: 8, children: [
                OutlinedButton.icon(
                  onPressed: _refreshDevices,
                  icon: const Icon(Icons.refresh, size: 16),
                  label: const Text('Обновить'),
                ),
                OutlinedButton.icon(
                  onPressed: () => BtService.I.openAndroidSettings(),
                  icon: const Icon(Icons.bluetooth_searching, size: 16),
                  label: const Text('Сопряжение Android'),
                ),
                OutlinedButton.icon(
                  onPressed: () => ph.openAppSettings(),
                  icon: const Icon(Icons.settings, size: 16),
                  label: const Text('Разрешения'),
                ),
              ]),
            ])
          else
            Column(
              children: _devices.map((d) {
                final sel = d.address == st.btAddress;
                return ListTile(
                  dense: true,
                  contentPadding: EdgeInsets.zero,
                  leading: Icon(sel ? Icons.bluetooth_connected : Icons.bluetooth,
                      color: sel ? HeatColors.gold : HeatColors.dim, size: 20),
                  title: Text(d.name?.isNotEmpty == true ? d.name! : 'без имени',
                      style: const TextStyle(fontSize: 13)),
                  subtitle: Text(d.address,
                      style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
                  trailing: sel ? const Icon(Icons.check, color: HeatColors.gold, size: 18) : null,
                  onTap: () {
                    st.btAddress = d.address;
                    st.btName = d.name ?? '';
                    st.save();
                    setState(() {});
                  },
                );
              }).toList(),
            ),
'''
s = s[:i] + new_dd + s[j:]
open(p, 'w').write(s)
print('OK  settings_screen.dart — реальный список сопряжённых ELM327')

# ── 3. main.dart: разрешения ДО построения UI ────────────────────────────────
m = 'lib/main.dart'
s = open(m).read()
s = s.replace('''    await Permission.bluetoothConnect.request();
    await Permission.bluetoothScan.request();
    await Permission.locationWhenInUse.request();''',
'''    await BtService.I.ensurePermissions();
    if (mounted) setState(() => _ready = true);''')
s = s.replace("import 'services/settings_service.dart';",
              "import 'services/settings_service.dart';\nimport 'services/bt_service.dart';")
s = s.replace("class _HomeShellState extends State<HomeShell> {",
              "class _HomeShellState extends State<HomeShell> {\n  bool _ready = false;")
open(m, 'w').write(s)
print('OK  main.dart — разрешения спрашиваются до открытия вкладки Настройки')
print('\n>>> Теперь запусти ячейку 10/10 (сборка APK)')

OK  lib/services/bt_service.dart
OK  settings_screen.dart — реальный список сопряжённых ELM327
OK  main.dart — разрешения спрашиваются до открытия вкладки Настройки

>>> Теперь запусти ячейку 10/10 (сборка APK)


In [ ]:
# @title 🩹 ФИКС 2/3: 3D/тепловые карты в стиле V6 (heat_map.dart)
# Фиксированный размер ячейки, шапка осей, скролл в обе стороны,
# зум ±, полноэкранный режим, редактирование тапом.
import os
os.chdir('/content/suba_run_v8')

with open('lib/widgets/heat_map.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/analyzer_service.dart';
import 'heat_colors.dart';

/// Таблица карты как в Nissan V6: ячейки фиксированного размера,
/// синяя шапка осей, скролл/зум, крупные читаемые цифры.
class HeatMapView extends StatefulWidget {
  final int cols;
  final int rows;
  final double? Function(int r, int c) value;
  final int Function(int r, int c)? hits;
  final String Function(int c)? xLabel;
  final String Function(int r)? yLabel;
  final String title;
  final String unit;
  final double? hintLo;
  final double? hintHi;
  /// Если задан — ячейку можно править тапом (ROM-экран).
  final void Function(int r, int c, double v)? onEdit;
  /// Исходное значение до правки — рисуется зачёркнутым сверху (как в V6).
  final double? Function(int r, int c)? original;

  const HeatMapView({
    super.key,
    required this.cols,
    required this.rows,
    required this.value,
    this.hits,
    this.xLabel,
    this.yLabel,
    this.title = '',
    this.unit = '',
    this.hintLo,
    this.hintHi,
    this.onEdit,
    this.original,
  });

  factory HeatMapView.fromGrid(HeatGrid g,
      {String title = '', String unit = '', double? hintLo, double? hintHi}) {
    final range = g.range(hintLo: hintLo, hintHi: hintHi);
    return HeatMapView(
      cols: g.cols,
      rows: g.rows,
      title: title,
      unit: unit,
      hintLo: range.lo,
      hintHi: range.hi,
      value: (r, c) => g.avg(g.rows - 1 - r, c),
      hits: (r, c) => g.n[g.rows - 1 - r][c],
      xLabel: (c) => g.xLabel(c),
      yLabel: (r) => g.yLabel(g.rows - 1 - r),
    );
  }

  @override
  State<HeatMapView> createState() => _HeatMapViewState();
}

class _HeatMapViewState extends State<HeatMapView> {
  ({int r, int c})? _sel;
  double _cell = 58;          // как _cellSize в V6
  static const double _headH = 30;
  static const double _axW = 62;

  double get _lo {
    if (widget.hintLo != null) return widget.hintLo!;
    var lo = double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v < lo) lo = v;
      }
    }
    return lo == double.infinity ? 0 : lo;
  }

  double get _hi {
    if (widget.hintHi != null) return widget.hintHi!;
    var hi = -double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v > hi) hi = v;
      }
    }
    return hi == -double.infinity ? 1 : hi;
  }

  String _fmt(double v) {
    final a = v.abs();
    if (a >= 1000) return v.toStringAsFixed(0);
    if (a >= 100) return v.toStringAsFixed(1);
    return v.toStringAsFixed(1);
  }

  @override
  Widget build(BuildContext context) {
    final lo = _lo;
    var hi = _hi;
    if (hi <= lo) hi = lo + 1;

    return Column(
      crossAxisAlignment: CrossAxisAlignment.stretch,
      children: [
        // ── панель управления (как в V6) ────────────────────────────────
        Row(children: [
          const Icon(Icons.grid_on, color: HeatColors.accent, size: 16),
          const SizedBox(width: 6),
          Expanded(
            child: Text(widget.title.isEmpty ? '${widget.rows}×${widget.cols}' : widget.title,
                overflow: TextOverflow.ellipsis,
                style: const TextStyle(
                    color: HeatColors.accent, fontSize: 12, fontWeight: FontWeight.bold)),
          ),
          Text('${widget.rows}x${widget.cols}',
              style: const TextStyle(color: HeatColors.dim, fontSize: 11)),
          const SizedBox(width: 8),
          IconButton(
            visualDensity: VisualDensity.compact,
            onPressed: () => setState(() => _cell = (_cell - 8).clamp(32, 110)),
            icon: const Icon(Icons.remove_circle_outline, size: 20),
          ),
          Text('${_cell.toInt()}', style: const TextStyle(color: HeatColors.dim, fontSize: 11)),
          IconButton(
            visualDensity: VisualDensity.compact,
            onPressed: () => setState(() => _cell = (_cell + 8).clamp(32, 110)),
            icon: const Icon(Icons.add_circle_outline, size: 20),
          ),
          IconButton(
            visualDensity: VisualDensity.compact,
            tooltip: 'Во весь экран',
            onPressed: _openFullscreen,
            icon: const Icon(Icons.fullscreen, size: 22, color: HeatColors.accent),
          ),
        ]),
        const SizedBox(height: 4),

        // ── сама таблица: скролл по X и Y, ничего не сжимается ──────────
        Expanded(
          child: Scrollbar(
            child: SingleChildScrollView(
              scrollDirection: Axis.vertical,
              child: SingleChildScrollView(
                scrollDirection: Axis.horizontal,
                child: _grid(lo, hi),
              ),
            ),
          ),
        ),

        if (_sel != null && widget.value(_sel!.r, _sel!.c) != null)
          Padding(
            padding: const EdgeInsets.only(top: 6),
            child: Text(
              'ячейка [${widget.yLabel?.call(_sel!.r) ?? _sel!.r} × '
              '${widget.xLabel?.call(_sel!.c) ?? _sel!.c}] = '
              '${_fmt(widget.value(_sel!.r, _sel!.c)!)} ${widget.unit}'
              '${widget.hits != null ? ' · попаданий: ${widget.hits!(_sel!.r, _sel!.c)}' : ''}',
              style: const TextStyle(fontSize: 12, color: HeatColors.gold),
            ),
          ),
      ],
    );
  }

  Widget _grid(double lo, double hi) {
    return Column(
      crossAxisAlignment: CrossAxisAlignment.start,
      children: [
        // шапка колонок
        Row(children: [
          _head(_axW, _headH, 'RPM/%'),
          ...List.generate(widget.cols,
              (c) => _head(_cell, _headH, widget.xLabel?.call(c) ?? '$c')),
        ]),
        ...List.generate(widget.rows, (r) {
          return Row(children: [
            _head(_axW, _cell, widget.yLabel?.call(r) ?? '$r'),
            ...List.generate(widget.cols, (c) => _cellBox(r, c, lo, hi)),
          ]);
        }),
      ],
    );
  }

  Widget _head(double w, double h, String t) => Container(
        width: w,
        height: h,
        decoration: BoxDecoration(
          color: const Color(0xFF0F3460),
          border: Border.all(color: Colors.white24, width: 0.5),
        ),
        alignment: Alignment.center,
        child: Text(t,
            maxLines: 1,
            overflow: TextOverflow.clip,
            style: const TextStyle(fontSize: 11, color: Colors.white70)),
      );

  Widget _cellBox(int r, int c, double lo, double hi) {
    final v = widget.value(r, c);
    final orig = widget.original?.call(r, c);
    final changed = v != null && orig != null && (orig - v).abs() > 1e-9;
    final sel = _sel != null && _sel!.r == r && _sel!.c == c;
    final t = v == null ? null : ((v - lo) / (hi - lo)).clamp(0.0, 1.0);

    return GestureDetector(
      onTap: () {
        setState(() => _sel = (r: r, c: c));
      },
      onLongPress: widget.onEdit == null || v == null ? null : () => _edit(r, c, v),
      child: Container(
        width: _cell,
        height: _cell,
        decoration: BoxDecoration(
          color: v == null
              ? const Color(0xFF11182B)
              : changed
                  ? const Color(0xFF1E88E5)
                  : heatColor(t!),
          border: Border.all(
            color: sel ? Colors.white : (changed ? Colors.white70 : Colors.white12),
            width: sel ? 2 : (changed ? 1 : 0.5),
          ),
        ),
        alignment: Alignment.center,
        child: v == null
            ? null
            : changed
                ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
                    Text(_fmt(orig),
                        style: TextStyle(
                            fontSize: _cell < 46 ? 8 : 10,
                            color: Colors.white70,
                            decoration: TextDecoration.lineThrough)),
                    Text(_fmt(v),
                        style: TextStyle(
                            fontSize: _cell < 46 ? 9 : 12,
                            color: Colors.white,
                            fontWeight: FontWeight.bold)),
                  ])
                : Text(_fmt(v),
                    style: TextStyle(
                      fontSize: _cell < 40 ? 9 : (_cell < 56 ? 10 : 12),
                      color: Colors.white,
                      fontWeight: FontWeight.w600,
                      shadows: const [Shadow(color: Colors.black54, blurRadius: 2)],
                    )),
      ),
    );
  }

  Future<void> _edit(int r, int c, double cur) async {
    final ctl = TextEditingController(text: cur.toStringAsFixed(2));
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: Text('[${widget.yLabel?.call(r) ?? r} × ${widget.xLabel?.call(c) ?? c}]',
            style: const TextStyle(fontSize: 15)),
        content: TextField(
          controller: ctl,
          autofocus: true,
          keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
          decoration: InputDecoration(labelText: 'Значение, ${widget.unit}'),
        ),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('ОТМЕНА')),
          FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('ОК')),
        ],
      ),
    );
    if (ok != true) return;
    final v = double.tryParse(ctl.text.replaceAll(',', '.'));
    if (v == null) return;
    widget.onEdit!(r, c, v);
    setState(() {});
  }

  void _openFullscreen() {
    Navigator.of(context).push(MaterialPageRoute(
      fullscreenDialog: true,
      builder: (_) => Scaffold(
        backgroundColor: HeatColors.bg,
        appBar: AppBar(
          title: Text(widget.title.isEmpty ? 'Карта' : widget.title,
              style: const TextStyle(fontSize: 16)),
          leading: IconButton(
              icon: const Icon(Icons.close), onPressed: () => Navigator.pop(context)),
        ),
        body: Padding(
          padding: const EdgeInsets.all(8),
          child: HeatMapView(
            cols: widget.cols,
            rows: widget.rows,
            value: widget.value,
            hits: widget.hits,
            xLabel: widget.xLabel,
            yLabel: widget.yLabel,
            title: widget.title,
            unit: widget.unit,
            hintLo: widget.hintLo,
            hintHi: widget.hintHi,
            onEdit: widget.onEdit,
            original: widget.original,
          ),
        ),
      ),
    ));
  }
}
''')
print('OK  lib/widgets/heat_map.dart — таблица карт как в V6')

# высота контейнеров 440 -> адаптивная, чтобы карта не резалась
import glob, re
for p in glob.glob('lib/screens/*.dart'):
    s = open(p).read()
    if 'HeatMapView' in s and 'height: 440,' in s:
        s = s.replace('height: 440,', 'height: MediaQuery.of(context).size.height * 0.62,')
        open(p, 'w').write(s)
        print('   patched height:', p)
print('\n>>> дальше ФИКС 3/3, потом ячейка 10/10')

OK  lib/widgets/heat_map.dart — таблица карт как в V6
   patched height: lib/screens/rom_screen.dart
   patched height: lib/screens/ecu_maps_screen.dart

>>> дальше ФИКС 3/3, потом ячейка 10/10


In [ ]:
# @title 🩹 ФИКС 3/3: вкладки не теряют состояние + ROM правится тапом
import os, glob, re
os.chdir('/content/suba_run_v8')

# ── 1. KeepAlive-обёртка для всех 17 вкладок ────────────────────────────────
with open('lib/widgets/keep_alive.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

/// TabBarView по умолчанию уничтожает соседние экраны:
/// поэтому в V8 «слетали» терминал, анализатор, загруженный ROM,
/// выбранные PID и результаты замера при свайпе между вкладками.
class KeepAlive extends StatefulWidget {
  final Widget child;
  const KeepAlive({super.key, required this.child});

  @override
  State<KeepAlive> createState() => _KeepAliveState();
}

class _KeepAliveState extends State<KeepAlive> with AutomaticKeepAliveClientMixin {
  @override
  bool get wantKeepAlive => true;

  @override
  Widget build(BuildContext context) {
    super.build(context);
    return widget.child;
  }
}
''')
print('OK  lib/widgets/keep_alive.dart')

m = 'lib/main.dart'
s = open(m).read()
s = s.replace("import 'widgets/heat_colors.dart';",
              "import 'widgets/heat_colors.dart';\nimport 'widgets/keep_alive.dart';")
s = s.replace('body: const TabBarView(children: _views),',
              'body: TabBarView(\n          children: _views.map((w) => KeepAlive(child: w)).toList(),\n        ),')
open(m, 'w').write(s)
print('OK  main.dart — состояние вкладок сохраняется')

# ── 2. ROM: правка ячейки тапом по карте вместо ввода индексов ──────────────
p = 'lib/screens/rom_screen.dart'
s = open(p).read()
s = s.replace('''            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < t.yValues.length ? _ax(t.yValues[i]) : '$i';
            },
          ),''',
'''            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < t.yValues.length ? _ax(t.yValues[i]) : '$i';
            },
            title: d.name,
            onEdit: d.editable
                ? (r, c, v) {
                    t.setCell(d.rows - 1 - r, c, v);
                    setState(() => _modified = true);
                  }
                : null,
          ),''', 1)
s = s.replace("        if (d.editable)\n          FilledButton.icon(",
              "        if (d.editable)\n          const Text('Долгий тап по ячейке — правка значения',\n              style: TextStyle(fontSize: 11, color: HeatColors.dim)),\n        if (d.editable)\n          FilledButton.icon(", 1)
open(p, 'w').write(s)
print('OK  rom_screen.dart — правка ячейки долгим тапом')

# ── 3. Экраны обновляются по событиям соединения (а не «мертвые» вкладки) ───
for p in ['lib/screens/dtc_screen.dart', 'lib/screens/terminal_screen.dart',
          'lib/screens/perf_screen.dart', 'lib/screens/service_screen.dart']:
    if not os.path.exists(p):
        continue
    s = open(p).read()
    if 'ConnectionService.I.addListener' in s:
        continue
    s = re.sub(r'(\n  @override\n  void initState\(\) \{\n    super\.initState\(\);\n)',
               r'\1    ConnectionService.I.addListener(_onSvc);\n', s, count=1)
    if '_onSvc' in s:
        s = s.replace('  @override\n  void dispose() {',
                      '  void _onSvc() { if (mounted) setState(() {}); }\n\n  @override\n  void dispose() {\n    ConnectionService.I.removeListener(_onSvc);', 1)
        if 'void dispose()' not in s:
            s = s.rstrip()[:-4] + "\n  void _onSvc() { if (mounted) setState(() {}); }\n}\n"
        open(p, 'w').write(s)
        print('   patched listener:', p)

print('\n>>> Готово. Запускай ячейку 10/10 — сборка APK')

OK  lib/widgets/keep_alive.dart
OK  main.dart — состояние вкладок сохраняется
OK  rom_screen.dart — правка ячейки долгим тапом
   patched listener: lib/screens/perf_screen.dart

>>> Готово. Запускай ячейку 10/10 — сборка APK


In [ ]:
# @title 🩹 ФИКС 4/4: Downloads + ретрай NO DATA + AT ST ≥ 0x20 + SnackBar + страховка buildPoller
# Закрывает все 5 пунктов из «что ещё стоит доделать».
# Запускать ПОСЛЕ ячейки 9/10 (и после ФИКС 1–3, если они уже применены),
# ПЕРЕД ячейкой 10/10 (сборка APK). Идемпотентна — можно перезапускать.
import os
os.chdir('/content/suba_run_v8')

def edit(path, old, new, tag):
    s = open(path, encoding='utf-8').read()
    if new in s:
        print(f'   · {tag}: уже применено, пропуск')
        return
    assert old in s, f'ЯКОРЬ НЕ НАЙДЕН [{tag}] в {path} — ячейки запускались не по порядку?'
    open(path, 'w', encoding='utf-8').write(s.replace(old, new, 1))
    print(f'OK  {tag}')

# ══════════════════════════════════════════════════════════════════════════
# 1/5. Сохранение в /Download (перенос фикса хранилища из Nissan V6)
# ══════════════════════════════════════════════════════════════════════════
with open('lib/services/save_service.dart', 'w', encoding='utf-8') as f:
    f.write(r'''import 'dart:io';

import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:shared_preferences/shared_preferences.dart';

/// Сохранение файлов в ВИДИМЫЕ папки (перенесено из Nissan V6).
/// Логгер/экспорт/моды пишут во внутреннюю папку приложения,
/// а готовые файлы дублируются сюда: выбранная папка -> /Download -> папка приложения.
class V8Saver {
  static const _kDirKey = 'v8_save_dir';
  static const downloads = '/storage/emulated/0/Download';

  static Future<bool> requestStoragePermission() async {
    try {
      if (await Permission.manageExternalStorage.isGranted) return true;
      final s = await Permission.manageExternalStorage.request();
      if (s.isGranted) return true;
      if (await Permission.storage.isGranted) return true;
      final s2 = await Permission.storage.request();
      return s2.isGranted;
    } catch (_) {
      return false;
    }
  }

  static Future<String?> getSavedDir() async =>
      (await SharedPreferences.getInstance()).getString(_kDirKey);

  static Future<void> setSavedDir(String p) async =>
      (await SharedPreferences.getInstance()).setString(_kDirKey, p);

  static Future<String?> pickDirectory() async {
    try {
      final path = await FilePicker.platform.getDirectoryPath(
        dialogTitle: 'Папка для логов и экспорта V8',
      );
      if (path != null) await setSavedDir(path);
      return path;
    } catch (_) {
      return null;
    }
  }

  static Future<V8SaveResult> saveBytes(List<int> bytes, String name) async {
    try { await requestStoragePermission(); } catch (_) {}
    final tried = <String>[];
    final userDir = await getSavedDir();
    if (userDir != null) {
      tried.add(userDir);
      final r = await _tryWrite('$userDir/$name', bytes);
      if (r != null) return V8SaveResult(path: r, where: 'выбранная папка');
    }
    if (await Directory(downloads).exists()) {
      tried.add(downloads);
      final r = await _tryWrite('$downloads/$name', bytes);
      if (r != null) return V8SaveResult(path: r, where: 'Downloads');
    }
    try {
      final dir = await getApplicationDocumentsDirectory();
      tried.add(dir.path);
      final r = await _tryWrite('${dir.path}/$name', bytes);
      if (r != null) return V8SaveResult(path: r, where: 'папка приложения');
    } catch (_) {}
    return V8SaveResult.error('Не сохранилось. Проверено: ${tried.join(', ')}');
  }

  static Future<V8SaveResult> saveCopy(File src) =>
      saveBytes(src.readAsBytesSync(), src.path.split('/').last);

  static Future<String?> _tryWrite(String path, List<int> bytes) async {
    try {
      final f = File(path);
      await f.writeAsBytes(bytes, flush: true);
      if (await f.exists() && await f.length() == bytes.length) return path;
    } catch (_) {}
    return null;
  }
}

class V8SaveResult {
  final String? path;
  final String where;
  final String? error;
  bool get ok => path != null;
  V8SaveResult({required this.path, required this.where}) : error = null;
  V8SaveResult.error(this.error) : path = null, where = '';
}
''')
print('OK  lib/services/save_service.dart (как RomSaver в V6)')

# — Manifest: MANAGE_EXTERNAL_STORAGE + READ + legacy —
m = 'android/app/src/main/AndroidManifest.xml'
s = open(m, encoding='utf-8').read()
if 'MANAGE_EXTERNAL_STORAGE' not in s:
    s = s.replace('<manifest xmlns:android="http://schemas.android.com/apk/res/android">',
                  '<manifest xmlns:android="http://schemas.android.com/apk/res/android"\n'
                  '    xmlns:tools="http://schemas.android.com/tools">')
    s = s.replace('<uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE" android:maxSdkVersion="28"/>',
                  '<uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE" android:maxSdkVersion="32"/>\n'
                  '    <uses-permission android:name="android.permission.READ_EXTERNAL_STORAGE" android:maxSdkVersion="32"/>\n'
                  '    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE" tools:ignore="ScopedStorage"/>')
    s = s.replace('android:label="SUBA RUN V8"',
                  'android:label="SUBA RUN V8"\n        android:requestLegacyExternalStorage="true"')
    open(m, 'w', encoding='utf-8').write(s)
    print('OK  AndroidManifest.xml — MANAGE_EXTERNAL_STORAGE + legacy')
else:
    print('   · AndroidManifest.xml: уже применено, пропуск')

# — Логгер: готовый CSV дублируется в /Download при stop() —
edit('lib/services/logger_service.dart',
     "import 'package:path_provider/path_provider.dart';",
     "import 'package:path_provider/path_provider.dart';\nimport 'save_service.dart';",
     'logger_service.dart — импорт V8Saver')
edit('lib/services/logger_service.dart',
     """    _sink = null;
    return _file?.path;
  }""",
     """    _sink = null;
    // ФИКС 4: копия готового лога в /Download — иначе файл не виден в проводнике
    final f = _file;
    if (f != null && await f.exists()) {
      try {
        final r = await V8Saver.saveCopy(f);
        if (r.ok) return r.path;
      } catch (_) {}
    }
    return _file?.path;
  }""",
     'logger_service.stop() — копия лога в /Download')

# — Экспорт: все 3 файла (ecuEdit CSV, WinOLS CSV, HEX) — в /Download —
edit('lib/services/export_service.dart',
     "import 'rom_service.dart';",
     "import 'rom_service.dart';\nimport 'save_service.dart';",
     'export_service.dart — импорт V8Saver')
s = open('lib/services/export_service.dart', encoding='utf-8').read()
n = s.count('await f.writeAsString(buf.toString());\n    return f;')
if n > 0:
    s = s.replace('await f.writeAsString(buf.toString());\n    return f;',
                  'await f.writeAsString(buf.toString());\n'
                  '    // ФИКС 4: копия экспорта в /Download\n'
                  '    try {\n'
                  '      final r = await V8Saver.saveCopy(f);\n'
                  '      if (r.ok) return File(r.path!);\n'
                  '    } catch (_) {}\n'
                  '    return f;')
    open('lib/services/export_service.dart', 'w', encoding='utf-8').write(s)
    print(f'OK  export_service.dart — {n} файла дублируются в /Download')
else:
    print('   · export_service.dart: уже применено, пропуск')

# — ROM-мод V8MOD_*.bin — в /Download —
edit('lib/services/rom_service.dart',
     "import '../models/rom_table.dart';",
     "import '../models/rom_table.dart';\nimport 'save_service.dart';",
     'rom_service.dart — импорт V8Saver')
edit('lib/services/rom_service.dart',
     """    final f = File('${dir.path}/V8MOD_$base.bin');
    await f.writeAsBytes(mod);
    return f.path;""",
     """    final f = File('${dir.path}/V8MOD_$base.bin');
    await f.writeAsBytes(mod);
    // ФИКС 4: копия мода в /Download
    try {
      final r = await V8Saver.saveBytes(mod, 'V8MOD_$base.bin');
      if (r.ok) return r.path;
    } catch (_) {}
    return f.path;""",
     'rom_service.saveMod() — копия V8MOD в /Download')

# ══════════════════════════════════════════════════════════════════════════
# 2/5. buildPoller() после сохранения PID и применения профиля — ПРОВЕРКА
# (в текущей редакции ячейки 9/10 вызовы уже есть — страхуемся на случай
#  старой редакции: вставляем недостающие)
# ══════════════════════════════════════════════════════════════════════════
s = open('lib/screens/custom_pid_screen.dart', encoding='utf-8').read()
if 'ConnectionService.I.buildPoller()' in s:
    print('OK  custom_pid_screen.dart — buildPoller() уже вызывается после upsert/remove')
else:
    s = s.replace('    ));\n    setState(() {});\n  }\n\n  @override',
                  '    ));\n    ConnectionService.I.buildPoller();\n'
                  '    ConnectionService.I.startPolling();\n    setState(() {});\n  }\n\n  @override')
    s = s.replace('await CustomPidService.I.remove(p.id);\n',
                  'await CustomPidService.I.remove(p.id);\n'
                  '                          ConnectionService.I.buildPoller();\n'
                  '                          ConnectionService.I.startPolling();\n')
    open('lib/screens/custom_pid_screen.dart', 'w', encoding='utf-8').write(s)
    print('OK  custom_pid_screen.dart — buildPoller() ВСТАВЛЕН')

s = open('lib/screens/profile_screen.dart', encoding='utf-8').read()
if 'ConnectionService.I.buildPoller()' in s:
    print('OK  profile_screen.dart — buildPoller() уже вызывается после apply')
else:
    s = s.replace('await ProfileService.I.apply(n);\n',
                  'await ProfileService.I.apply(n);\n'
                  '                    ConnectionService.I.buildPoller();\n'
                  '                    ConnectionService.I.startPolling();\n')
    open('lib/screens/profile_screen.dart', 'w', encoding='utf-8').write(s)
    print('OK  profile_screen.dart — buildPoller() ВСТАВЛЕН')

# ══════════════════════════════════════════════════════════════════════════
# 3/5. Ретрай кадра при NO DATA внутри SSM2-блока
# ══════════════════════════════════════════════════════════════════════════
edit('lib/ssm/ssm_elm.dart',
     '  int stTimeoutCode = 0x08; // AT ST (x4 мс): 0x08 = 32 мс ожидание байтов ответа',
     '  int stTimeoutCode = 0x20; // AT ST (x4 мс): 0x20 = 128 мс — минимум для клонов ELM327\n'
     '  int readRetries = 1;      // ФИКС 4: повторы кадра при NO DATA',
     'ssm_elm.dart — stTimeoutCode 0x20 + readRetries')
edit('lib/ssm/ssm_elm.dart',
     """    var data = _parseRead(await transact('A8' '00' '$a$cnt1'), addr, len);
    if (data == null) {
      final cntAbs = (len & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
      data = _parseRead(await transact('A8' '00' '$a$cntAbs'), addr, len);
    }
    sw.stop();""",
     """    Uint8List? data;
    // ФИКС 4: повтор кадра при NO DATA — клоны ELM327 периодически роняют первый ответ.
    // Без ретрая одна ошибка обнуляла весь блок PID на дашборде.
    for (var attempt = 0; attempt <= readRetries; attempt++) {
      data = _parseRead(await transact('A8' '00' '$a$cnt1'), addr, len);
      if (data == null) {
        final cntAbs = (len & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
        data = _parseRead(await transact('A8' '00' '$a$cntAbs'), addr, len);
      }
      if (data != null) break;
      if (attempt < readRetries) {
        await Future<void>.delayed(const Duration(milliseconds: 150));
      }
    }
    sw.stop();""",
     'ssm_elm.readBytes() — ретрай кадра')

# ══════════════════════════════════════════════════════════════════════════
# 4/5. AT ST: минимум 0x20 (128 мс) — слайдер + дефолты + кламп старых значений
# ══════════════════════════════════════════════════════════════════════════
edit('lib/services/settings_service.dart',
     '  int stCode = 8;',
     '  int stCode = 0x20; // 0x20 x 4 мс = 128 мс — минимум для клонов (ФИКС 4)',
     'settings_service.dart — дефолт stCode 0x20')
edit('lib/services/settings_service.dart',
     "    stCode = p.getInt('stCode') ?? 8;",
     "    stCode = (p.getInt('stCode') ?? 0x20).clamp(0x20, 0x40);",
     'settings_service.load() — кламп старого значения')
edit('lib/services/profile_service.dart',
     "'stCode': 8,",
     "'stCode': 0x20,",
     'profile_service.dart — дефолт профиля 0x20')
edit('lib/services/profile_service.dart',
     '    st.stCode = (d[\'stCode\'] ?? st.stCode) as int;',
     '    st.stCode = ((d[\'stCode\'] ?? st.stCode) as int).clamp(0x20, 0x40);',
     'profile_service.apply() — кламп stCode')
edit('lib/screens/settings_screen.dart',
     'st.stCode.toDouble(), 2, 30, (v) {',
     'st.stCode.toDouble(), 0x20, 0x40, (v) {',
     'settings_screen — слайдер AT ST 0x20..0x40')
edit('lib/screens/settings_screen.dart',
     '            st.stCode = v.round();',
     '            st.stCode = v.round().clamp(0x20, 0x40);',
     'settings_screen — кламп слайдера AT ST')

# ══════════════════════════════════════════════════════════════════════════
# 5/5. Причина неудачного connectAndInit() — в SnackBar, а не мелким текстом
# ══════════════════════════════════════════════════════════════════════════
edit('lib/screens/settings_screen.dart',
     """  Future<void> _connect() async {
    setState(() { _busy = true; _msg = 'подключаюсь...'; });
    final r = await svc.connectAndInit();
    if (!mounted) return;
    setState(() { _busy = false; _msg = r; });
  }""",
     """  Future<void> _connect() async {
    setState(() { _busy = true; _msg = 'подключаюсь...'; });
    final r = await svc.connectAndInit();
    if (!mounted) return;
    setState(() { _busy = false; _msg = r; });
    // ФИКС 4: результат подключения — в SnackBar, ошибку видно сразу
    final ok = r.startsWith('OK');
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(r),
      backgroundColor: ok ? Colors.green.shade700 : Colors.red.shade700,
      duration: Duration(seconds: ok ? 2 : 5),
      action: ok
          ? null
          : SnackBarAction(label: 'ПОНЯТНО', textColor: Colors.white, onPressed: () {}),
    ));
  }""",
     'settings_screen._connect() — SnackBar результата')

print()
print('=' * 64)
print('  ФИКС 4/4 применён. Запускай ячейку 10/10 — сборка APK')
print('=' * 64)


OK  lib/services/save_service.dart (как RomSaver в V6)
OK  AndroidManifest.xml — MANAGE_EXTERNAL_STORAGE + legacy
OK  logger_service.dart — импорт V8Saver
OK  logger_service.stop() — копия лога в /Download
OK  export_service.dart — импорт V8Saver
OK  export_service.dart — 3 файла дублируются в /Download
OK  rom_service.dart — импорт V8Saver
OK  rom_service.saveMod() — копия V8MOD в /Download
OK  custom_pid_screen.dart — buildPoller() уже вызывается после upsert/remove
OK  profile_screen.dart — buildPoller() уже вызывается после apply
OK  ssm_elm.dart — stTimeoutCode 0x20 + readRetries
OK  ssm_elm.readBytes() — ретрай кадра
OK  settings_service.dart — дефолт stCode 0x20
OK  settings_service.load() — кламп старого значения
OK  profile_service.dart — дефолт профиля 0x20
OK  profile_service.apply() — кламп stCode
OK  settings_screen — слайдер AT ST 0x20..0x40
OK  settings_screen — кламп слайдера AT ST
OK  settings_screen._connect() — SnackBar результата

  ФИКС 4/4 применён. Запускай ячейк

In [ ]:
# @title ⚡ ЭКСПРЕСС-ФИКС: 'KeepAlive' name clash (1 секунда)
# Запусти эту ячейку, если сборка APK упала на:
# "Error: 'KeepAlive' is imported from both 'package:flutter/src/widgets/sliver.dart' and '.../keep_alive.dart'"
# После выполнения сразу перезапусти ячейку 10/10 (Сборка APK)!
import os
os.chdir('/content/suba_run_v8')

# 1. Переименовываем виджет в lib/widgets/keep_alive.dart: KeepAlive -> KeepAliveTab
with open('lib/widgets/keep_alive.dart', 'w', encoding='utf-8') as f:
    f.write(r'''import 'package:flutter/material.dart';

class KeepAliveTab extends StatefulWidget {
  final Widget child;
  const KeepAliveTab({super.key, required this.child});

  @override
  State<KeepAliveTab> createState() => _KeepAliveTabState();
}

class _KeepAliveTabState extends State<KeepAliveTab> with AutomaticKeepAliveClientMixin {
  @override
  bool get wantKeepAlive => true;

  @override
  Widget build(BuildContext context) {
    super.build(context);
    return widget.child;
  }
}
''')
print('OK  lib/widgets/keep_alive.dart -> KeepAliveTab')

# 2. Переименовываем вызов в lib/main.dart
m = 'lib/main.dart'
s = open(m, encoding='utf-8').read()
s = s.replace('KeepAlive(child: w)', 'KeepAliveTab(child: w)')
open(m, 'w', encoding='utf-8').write(s)
print('OK  lib/main.dart: KeepAlive -> KeepAliveTab')

print('\n' + '=' * 60)
print('  КОНФЛИКТ УСТРАНЁН! Перезапусти ячейку 10/10 (Сборка APK)')
print('=' * 60)


OK  lib/widgets/keep_alive.dart -> KeepAliveTab
OK  lib/main.dart: KeepAlive -> KeepAliveTab

  КОНФЛИКТ УСТРАНЁН! Перезапусти ячейку 10/10 (Сборка APK)


In [ ]:
# @title SUBA RUN V8: Bluetooth permissions v2 (V6 compatibility)
# Run after cells 0-9 and the existing fixes, BEFORE build cell 10/10.
# This cell changes source files only. It does not build or install an APK.
from pathlib import Path
from datetime import datetime, timezone
import re
import xml.etree.ElementTree as ET

ROOT = Path('/content/suba_run_v8')
ANDROID = 'http://schemas.android.com/apk/res/android'
TOOLS = 'http://schemas.android.com/tools'
ET.register_namespace('android', ANDROID)
ET.register_namespace('tools', TOOLS)

BT_SERVICE = r'''import 'dart:async';
import 'dart:io';
import 'package:flutter/services.dart';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'package:permission_handler/permission_handler.dart' as ph;

class BtService {
  static final BtService I = BtService._();
  BtService._();

  static const revision = 'BT-PERM-V2';
  static const _native = MethodChannel('suba_run_v8/bt_permissions_v2');
  String lastError = '';
  String diagnostic = '$revision\nРазрешения ещё не проверены.';
  Future<bool>? _permissionJob;
  Future<List<BluetoothDevice>>? _devicesJob;

  Future<Map<String, dynamic>> _status() async {
    final state = await _native.invokeMapMethod<String, dynamic>('status');
    if (state == null || state['sdkInt'] is! int) {
      throw StateError('Нет ответа от Android-проверки разрешений.');
    }
    final sdk = state['sdkInt'] as int;
    String flag(String key) => state[key] == true ? 'granted' : 'denied';
    diagnostic = '$revision\nAndroid API: $sdk\n'
        'ACCESS_COARSE_LOCATION: ${flag('coarse')}\n'
        'ACCESS_FINE_LOCATION: ${flag('fine')}\n'
        'BLUETOOTH_CONNECT: ${sdk >= 31 ? flag('connect') : 'not required (<31)'}\n'
        'BLUETOOTH_SCAN: ${sdk >= 31 ? flag('scan') : 'not required (<31)'}';
    return state;
  }

  bool _locationGranted(Map<String, dynamic> s) =>
      s['fine'] == true && s['coarse'] == true;

  // Share permission requests across callers; never open concurrent dialogs.
  Future<bool> ensurePermissions() => _permissionJob ??=
      _checkPermissions().whenComplete(() { _permissionJob = null; });

  Future<bool> _checkPermissions() async {
    lastError = '';
    if (!Platform.isAndroid) {
      lastError = 'Этот Bluetooth Classic плагин поддерживается только на Android.';
      return false;
    }
    try {
      var state = await _status();
      // Compatibility with the old plugin: location is NOT a Bluetooth fallback.
      if (!_locationGranted(state)) {
        final permission = ph.Permission.locationWhenInUse;
        if (!(await permission.status).isPermanentlyDenied) {
          await permission.request();
        }
        state = await _status();
      }
      if (!_locationGranted(state)) {
        lastError = 'Откройте Разрешения > Местоположение и разрешите доступ '
            'при использовании приложения. На Android 12+ включите точное '
            'местоположение. Это проверка старого flutter_bluetooth_serial, '
            'а не запрос координат. Если пункта нет, установите новый APK.';
        return false;
      }

      final sdk = state['sdkInt'] as int;
      if (sdk >= 31) {
        if (state['connect'] != true &&
            !(await ph.Permission.bluetoothConnect.status).isPermanentlyDenied) {
          await ph.Permission.bluetoothConnect.request();
        }
        state = await _status();
        if (state['scan'] != true &&
            !(await ph.Permission.bluetoothScan.status).isPermanentlyDenied) {
          await ph.Permission.bluetoothScan.request();
        }
        state = await _status();
        if (state['connect'] != true || state['scan'] != true) {
          lastError = 'Разрешите Устройства поблизости в настройках SUBA RUN V8. '
              'Разрешение геолокации не заменяет разрешение Bluetooth.';
          return false;
        }
      }
      return _locationGranted(state);
    } on MissingPluginException {
      lastError = 'Нет Android-проверки BT-PERM-V2. Пересоберите и установите '
          'новый APK; одного перезапуска старого приложения недостаточно.';
      diagnostic = '$revision\nNative permission bridge: missing';
      return false;
    } catch (e) {
      lastError = 'Не удалось проверить или запросить разрешения: $e';
      return false;
    }
  }

  Future<bool> ensureEnabled() async {
    try {
      final bt = FlutterBluetoothSerial.instance;
      if ((await bt.isAvailable) != true) {
        lastError = 'Bluetooth недоступен на этом устройстве.';
        return false;
      }
      if ((await bt.isEnabled) != true) {
        if ((await bt.requestEnable()) != true) {
          lastError = 'Включение Bluetooth отменено. Включите его в Android.';
          return false;
        }
      }
      diagnostic += '\nBluetooth: enabled';
      return true;
    } catch (e) {
      lastError = 'Не удалось включить Bluetooth: $e';
      return false;
    }
  }

  Future<List<BluetoothDevice>> bondedDevices() => _devicesJob ??=
      _readBondedDevices().whenComplete(() { _devicesJob = null; });

  Future<List<BluetoothDevice>> _readBondedDevices() async {
    if (!await ensurePermissions()) return const [];
    if (!await ensureEnabled()) return const [];
    try {
      final devices = await FlutterBluetoothSerial.instance
          .getBondedDevices().timeout(const Duration(seconds: 15));
      final unique = <String, BluetoothDevice>{
        for (final d in devices) d.address: d,
      }.values.toList();
      int rank(BluetoothDevice d) {
        final name = (d.name ?? '').toUpperCase();
        return ['ELM', 'OBD', 'VLINK', 'VGATE', 'KONNWEI']
            .any(name.contains) ? 0 : 1;
      }
      unique.sort((a, b) {
        final result = rank(a).compareTo(rank(b));
        return result != 0 ? result : (a.name ?? a.address).compareTo(b.name ?? b.address);
      });
      diagnostic += '\ngetBondedDevices: ${unique.length} devices';
      lastError = unique.isEmpty
          ? 'Сопряжённых устройств нет. Добавьте ELM327 через Сопряжение Android '
              'и вернитесь сюда, затем нажмите Обновить.'
          : '';
      return unique;
    } on PlatformException catch (e) {
      diagnostic += '\ngetBondedDevices: ${e.code}';
      lastError = 'getBondedDevices: ${e.code}: ${e.message ?? ''}. '
          'Скопируйте диагностику ниже, если все разрешения уже выданы.';
      return const [];
    } on TimeoutException {
      diagnostic += '\ngetBondedDevices: timeout';
      lastError = 'Плагин не ответил за 15 секунд. Перезапустите приложение '
          'и повторите Обновить. Подключение к ЭБУ ещё не выполнялось.';
      return const [];
    } catch (e) {
      diagnostic += '\ngetBondedDevices: failed';
      lastError = 'Не удалось получить список устройств: $e';
      return const [];
    }
  }

  Future<void> openAndroidSettings() => FlutterBluetoothSerial.instance.openSettings();
  Future<void> openAppSettings() async {
    final opened = await ph.openAppSettings();
    if (!opened) throw StateError('Не удалось открыть настройки приложения.');
  }
}
'''

NATIVE_BRIDGE = r'''package __PACKAGE__

import android.Manifest
import android.app.Activity
import android.content.pm.PackageManager
import android.os.Build
import androidx.core.content.ContextCompat
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.MethodChannel

// Read-only status bridge: never grants permissions or reads location data.
object SubaBluetoothPermissionsV2 {
    fun register(activity: Activity, engine: FlutterEngine) {
        MethodChannel(engine.dartExecutor.binaryMessenger, "suba_run_v8/bt_permissions_v2")
            .setMethodCallHandler { call, result ->
                if (call.method != "status") {
                    result.notImplemented()
                } else {
                    fun granted(name: String): Boolean =
                        ContextCompat.checkSelfPermission(activity, name) == PackageManager.PERMISSION_GRANTED
                    result.success(mapOf(
                        "sdkInt" to Build.VERSION.SDK_INT,
                        "coarse" to granted(Manifest.permission.ACCESS_COARSE_LOCATION),
                        "fine" to granted(Manifest.permission.ACCESS_FINE_LOCATION),
                        "connect" to (Build.VERSION.SDK_INT < 31 || granted(Manifest.permission.BLUETOOTH_CONNECT)),
                        "scan" to (Build.VERSION.SDK_INT < 31 || granted(Manifest.permission.BLUETOOTH_SCAN))
                    ))
                }
            }
    }
}
'''

REFRESH = r'''  Future<void> _refreshDevices() async {
    if (_scanning || _busy) return;
    setState(() { _scanning = true; _btError = ''; _msg = ''; });
    try {
      final list = await BtService.I.bondedDevices();
      if (!mounted) return;
      setState(() {
        _devices = list;
        _btError = BtService.I.lastError;
      });
      // Do not erase the saved adapter address after a denied permission.
    } catch (e) {
      if (mounted) setState(() {
        _devices = [];
        _btError = 'Ошибка обновления устройств: $e';
      });
    } finally {
      if (mounted) setState(() => _scanning = false);
    }
  }'''

CONNECT = r'''  Future<void> _connect() async {
    if (_busy || _scanning) return;
    setState(() { _busy = true; _msg = 'Проверка разрешений...'; });
    var message = '';
    try {
      if (!await BtService.I.ensurePermissions()) {
        message = BtService.I.lastError;
      } else if (st.btAddress.isEmpty) {
        message = 'Нажмите Обновить и выберите сопряжённый ELM327.';
      } else if (!await BtService.I.ensureEnabled()) {
        message = BtService.I.lastError;
      } else {
        message = await svc.connectAndInit();
      }
    } catch (e) {
      message = 'Ошибка подключения: $e';
    } finally {
      if (mounted) setState(() => _busy = false);
    }
    if (!mounted) return;
    setState(() => _msg = message);
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(message),
      backgroundColor: message.startsWith('OK') ? Colors.green.shade800 : Colors.red.shade800,
      duration: const Duration(seconds: 6),
    ));
  }'''

ACTIONS = r'''  // BT_PERM_V2_ACTIONS_BEGIN
  Future<void> _openBtOptions(bool permissions) async {
    try {
      if (permissions) {
        await BtService.I.openAppSettings();
      } else {
        await BtService.I.openAndroidSettings();
      }
      if (mounted) setState(() => _msg = 'После возврата из Android нажмите Обновить.');
    } catch (e) {
      if (mounted) setState(() => _msg = 'Не удалось открыть настройки: $e');
    }
  }

  Future<void> _copyBtDiagnostic() async {
    try {
      final report = <String>[
        BtService.I.diagnostic,
        if (BtService.I.lastError.isNotEmpty) 'Error: ${BtService.I.lastError}',
        if (_msg.isNotEmpty) 'Last action: $_msg',
      ].join('\n');
      await Clipboard.setData(ClipboardData(text: report));
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(content: Text('Диагностика разрешений скопирована')),
      );
    } catch (e) {
      if (mounted) setState(() => _msg = 'Ошибка копирования: $e');
    }
  }
  // BT_PERM_V2_ACTIONS_END

'''

CONNECTION_UI = r'''        _card('ПОДКЛЮЧЕНИЕ ELM327', [
          const Text('BT-PERM-V2 | Сопряжённые устройства',
              style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          const SizedBox(height: 8),
          const Text('Старому Bluetooth-плагину нужно разрешение Местоположение. '
              'На Android 12+ также нужны Устройства поблизости. '
              'Этот фикс не читает координаты.',
              style: TextStyle(fontSize: 12, color: HeatColors.dim)),
          const SizedBox(height: 10),
          if (_scanning)
            const LinearProgressIndicator()
          else if (_devices.isEmpty)
            Text(_btError.isEmpty ? 'Нажмите Обновить для получения списка устройств.' : _btError,
                style: const TextStyle(fontSize: 12, color: Colors.orangeAccent))
          else
            ..._devices.map((d) => ListTile(
              dense: true,
              contentPadding: EdgeInsets.zero,
              leading: Icon(Icons.bluetooth,
                  color: d.address == st.btAddress ? HeatColors.gold : HeatColors.dim),
              title: Text(d.name?.isNotEmpty == true ? d.name! : 'Без имени',
                  style: const TextStyle(fontSize: 13)),
              subtitle: Text(d.address, style: const TextStyle(fontSize: 11)),
              trailing: d.address == st.btAddress ? const Icon(Icons.check, color: HeatColors.gold) : null,
              onTap: _busy ? null : () async {
                st.btAddress = d.address;
                st.btName = d.name ?? '';
                setState(() {});
                try { await st.save(); }
                catch (e) { if (mounted) setState(() => _msg = 'Не удалось сохранить выбор: $e'); }
              },
            )),
          const SizedBox(height: 8),
          Wrap(spacing: 8, runSpacing: 4, children: [
            OutlinedButton.icon(onPressed: _scanning || _busy ? null : _refreshDevices,
                icon: const Icon(Icons.refresh, size: 16), label: const Text('Обновить')),
            OutlinedButton.icon(onPressed: _scanning || _busy ? null : () => _openBtOptions(false),
                icon: const Icon(Icons.bluetooth_searching, size: 16), label: const Text('Сопряжение Android')),
            OutlinedButton.icon(onPressed: _scanning || _busy ? null : () => _openBtOptions(true),
                icon: const Icon(Icons.settings, size: 16), label: const Text('Разрешения')),
          ]),
          const SizedBox(height: 8),
          Wrap(spacing: 8, children: [
            FilledButton.icon(onPressed: _busy || _scanning ? null : _connect,
                icon: const Icon(Icons.electric_bolt), label: const Text('CONNECT + INIT ECU')),
            IconButton(
              tooltip: 'Отключить',
              onPressed: _busy || _scanning ? null : () async {
                try {
                  await svc.disconnect();
                  if (mounted) setState(() => _msg = 'Отключено');
                } catch (e) {
                  if (mounted) setState(() => _msg = 'Ошибка отключения: $e');
                }
              },
              icon: const Icon(Icons.link_off, color: Colors.redAccent),
            ),
          ]),
          if (_msg.isNotEmpty)
            Padding(padding: const EdgeInsets.only(top: 8), child: Text(_msg,
                style: const TextStyle(fontSize: 12, color: HeatColors.gold))),
          if (svc.elm.ecuId.isNotEmpty) _kv('ECU ID', svc.elm.ecuId),
          if (svc.elm.elmVersion.isNotEmpty) _kv('Адаптер', svc.elm.elmVersion),
          const Divider(),
          SelectableText(BtService.I.diagnostic,
              style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: HeatColors.dim)),
          TextButton.icon(onPressed: _copyBtDiagnostic,
              icon: const Icon(Icons.copy, size: 16), label: const Text('Копировать диагностику')),
        ]),
'''


def require(condition, message):
    if not condition:
        raise ValueError(message)


def patch_manifest(source):
    parser = ET.XMLParser(target=ET.TreeBuilder(insert_comments=True))
    document = ET.fromstring(source, parser=parser)
    require(document.tag == 'manifest', 'Expected an Android manifest.')
    name_attr = '{' + ANDROID + '}name'
    max_attr = '{' + ANDROID + '}maxSdkVersion'
    for short in ('ACCESS_FINE_LOCATION', 'ACCESS_COARSE_LOCATION'):
        name = 'android.permission.' + short
        nodes = [e for e in document.findall('uses-permission') if e.get(name_attr) == name]
        require(len(nodes) <= 1, 'Duplicate permission: ' + name)
        if nodes:
            node = nodes[0]
        else:
            node = ET.Element('uses-permission', {name_attr: name})
            document.insert(0, node)
        node.attrib.pop(max_attr, None)
        # Prevent a lower-priority plugin manifest from restoring a version cap.
        for attr in ('remove', 'replace', 'strict', 'selector'):
            node.attrib.pop('{' + TOOLS + '}' + attr, None)
        node.set('{' + TOOLS + '}node', 'replace')
    for short in ('BLUETOOTH', 'BLUETOOTH_ADMIN', 'BLUETOOTH_CONNECT', 'BLUETOOTH_SCAN'):
        name = 'android.permission.' + short
        nodes = [e for e in document.findall('uses-permission') if e.get(name_attr) == name]
        require(len(nodes) == 1, 'Expected exactly one permission: ' + name)
        require(nodes[0].get('{' + TOOLS + '}node') != 'remove', 'Bluetooth permission is removed: ' + name)
        if short in ('BLUETOOTH', 'BLUETOOTH_ADMIN'):
            nodes[0].set(max_attr, '30')
        else:
            nodes[0].attrib.pop(max_attr, None)
    return ET.tostring(document, encoding='unicode') + '\n'


def patch_activity(source):
    registration = 'SubaBluetoothPermissionsV2.register(this, flutterEngine)'
    if registration in source:
        require(source.count(registration) == 1, 'Duplicate permission bridge registration.')
        return source
    if 'configureFlutterEngine' in source:
        anchor = 'super.configureFlutterEngine(flutterEngine)'
        require(source.count(anchor) == 1, 'Unexpected configureFlutterEngine: adapt the bridge manually.')
        return source.replace(anchor, anchor + '\n        ' + registration, 1)
    header = r'(class\s+MainActivity\s*:\s*FlutterActivity\(\)\s*\{)'
    method = '\n    override fun configureFlutterEngine(flutterEngine: io.flutter.embedding.engine.FlutterEngine) {\n'
    method += '        super.configureFlutterEngine(flutterEngine)\n        ' + registration + '\n    }\n'
    result, count = re.subn(header, lambda match: match.group(0) + method, source)
    require(count == 1, 'Expected the standard Kotlin MainActivity from SUBA V8.')
    return result


def replace_method(source, name, replacement):
    pattern = r'(?ms)^  Future<void> ' + re.escape(name) + r'\(\) async \{.*?^  \}'
    result, count = re.subn(pattern, lambda _: replacement, source)
    require(count == 1, 'Expected exactly one method: ' + name)
    return result


def patch_screen(source):
    for declaration in ('bool _scanning = false;', "String _btError = '';"):
        variable = '_scanning' if declaration.startswith('bool') else '_btError'
        pattern = r'(?m)^  (?:bool|String) ' + variable + r'\s*='
        count = len(re.findall(pattern, source))
        require(count <= 1, 'Duplicate field from an old fix: ' + variable)
        if count == 0:
            anchor = '  bool _busy = false;'
            require(source.count(anchor) == 1, 'Settings busy flag was not found.')
            source = source.replace(anchor, anchor + '\n  ' + declaration, 1)
    source = replace_method(source, '_refreshDevices', REFRESH)
    source = replace_method(source, '_connect', CONNECT)
    start = "        _card('ПОДКЛЮЧЕНИЕ ELM327', ["
    end = "        _card('СТАТИСТИКА ПРОТОКОЛА"
    require(source.count(start) == 1 and source.count(end) == 1, 'Connection panel anchors were not found.')
    first, last = source.index(start), source.index(end)
    require(first < last, 'Unexpected settings panel order.')
    source = source[:first] + CONNECTION_UI + source[last:]
    source = re.sub(r'(?ms)^  // BT_PERM_V2_ACTIONS_BEGIN\n.*?^  // BT_PERM_V2_ACTIONS_END\n\n', '', source)
    anchor = '  Future<void> _connect() async {'
    source = source.replace(anchor, ACTIONS + anchor, 1)
    for statement in (
        "import 'package:flutter/services.dart';",
        "import '../services/bt_service.dart';",
    ):
        if statement not in source:
            source = statement + '\n' + source
    return source


def patch_main(source):
    # Request access in Settings, not in a competing startup permission dialog.
    source = re.sub(r'(?m)^[ \t]*await BtService\.I\.ensurePermissions\(\);\n', '', source)
    source = re.sub(
        r'(?m)^[ \t]*await Permission\.(?:bluetoothConnect|bluetoothScan|locationWhenInUse)\.request\(\);\n',
        '', source,
    )
    if 'BtService.' not in source:
        source = source.replace("import 'services/bt_service.dart';\n", '')
    if 'Permission.' not in source:
        source = source.replace("import 'package:permission_handler/permission_handler.dart';\n", '')
    return source


def run_source_tests():
    permissions = ('BLUETOOTH', 'BLUETOOTH_ADMIN', 'BLUETOOTH_CONNECT', 'BLUETOOTH_SCAN',
                   'ACCESS_COARSE_LOCATION', 'ACCESS_FINE_LOCATION')
    manifest = '<manifest xmlns:android="' + ANDROID + '">'
    manifest += ''.join('<uses-permission android:name="android.permission.' + p +
                        '" android:maxSdkVersion="30" />' for p in permissions)
    manifest += '<application android:label="Preserve me" /></manifest>'
    patched = patch_manifest(manifest)
    require(patch_manifest(patched) == patched, 'Manifest repeat test failed.')
    document = ET.fromstring(patched)
    require(document.find('application').get('{' + ANDROID + '}label') == 'Preserve me',
            'Unrelated manifest attributes changed.')
    for element in document.findall('uses-permission'):
        name = element.get('{' + ANDROID + '}name')
        cap = element.get('{' + ANDROID + '}maxSdkVersion')
        legacy = name in ('android.permission.BLUETOOTH', 'android.permission.BLUETOOTH_ADMIN')
        require(cap == ('30' if legacy else None), 'Permission version-cap test failed.')

    activity = 'package test.app\nclass MainActivity : FlutterActivity() {\n}\n'
    patched = patch_activity(activity)
    require(patch_activity(patched) == patched, 'Activity repeat test failed.')
    require(patched.count('SubaBluetoothPermissionsV2.register') == 1, 'Duplicate native registration.')

    screen = """class _SettingsScreenState extends State<SettingsScreen> {
  bool _busy = false;
  Future<void> _refreshDevices() async {
    await something();
  }
  Widget build(BuildContext context) {
    return ListView(children: [
        _card('ПОДКЛЮЧЕНИЕ ELM327', [Text('old')]),
        _card('СТАТИСТИКА ПРОТОКОЛА', [Text('preserve')]),
    ]);
  }
  Future<void> _connect() async {
    await something();
  }
}
"""
    patched = patch_screen(screen)
    require(patch_screen(patched) == patched, 'Screen repeat test failed.')
    require(patched.count('bool _scanning = false;') == 1, 'Duplicate loading flag.')
    require("Text('preserve')" in patched, 'Unrelated settings panel changed.')
    try:
        patch_screen(screen.replace("_card('СТАТИСТИКА ПРОТОКОЛА'", "_card('OTHER'"))
    except ValueError:
        pass
    else:
        raise ValueError('Missing screen anchor must stop the patch.')
    print('Source self-checks passed: manifest, repeat application, missing-anchor rejection.')
    print('These checks do not compile Dart/Kotlin or test Android Bluetooth.')


def apply_fix(root):
    manifest_path = root / 'android/app/src/main/AndroidManifest.xml'
    screen_path = root / 'lib/screens/settings_screen.dart'
    main_path = root / 'lib/main.dart'
    for path in (manifest_path, screen_path, main_path, root / 'pubspec.yaml'):
        require(path.is_file(), 'Missing project file: ' + str(path))
    pubspec = (root / 'pubspec.yaml').read_text(encoding='utf-8')
    require(re.search(r'(?m)^\s+flutter_bluetooth_serial:\s*[\^~]?0\.4\.0\s*$', pubspec),
            'This compatibility fix targets flutter_bluetooth_serial 0.4.0.')
    candidates = list((root / 'android/app/src/main/kotlin').rglob('MainActivity.kt'))
    require(len(candidates) == 1, 'Expected exactly one Kotlin MainActivity.')
    activity_path = candidates[0]
    activity = activity_path.read_text(encoding='utf-8')
    package = re.search(r'(?m)^package\s+([\w.]+)\s*$', activity)
    require(package is not None, 'Kotlin package declaration not found.')

    patchers = {
        manifest_path: patch_manifest,
        activity_path: patch_activity,
        screen_path: patch_screen,
        main_path: patch_main,
    }
    outputs = {}
    for path, transform in patchers.items():
        original = path.read_text(encoding='utf-8')
        updated = transform(original)
        require(transform(updated) == updated, 'Repeat-application check failed: ' + str(path))
        outputs[path] = updated
    outputs[root / 'lib/services/bt_service.dart'] = BT_SERVICE
    outputs[activity_path.with_name('SubaBluetoothPermissionsV2.kt')] = NATIVE_BRIDGE.replace('__PACKAGE__', package.group(1))

    # Validate the final manifest before creating backups or writing any source.
    final_manifest = ET.fromstring(outputs[manifest_path])
    for short in ('ACCESS_COARSE_LOCATION', 'ACCESS_FINE_LOCATION'):
        nodes = [e for e in final_manifest.findall('uses-permission')
                 if e.get('{' + ANDROID + '}name') == 'android.permission.' + short]
        require(len(nodes) == 1 and '{' + ANDROID + '}maxSdkVersion' not in nodes[0].attrib,
                'Location permission is still version-limited: ' + short)

    changes = []
    for path, content in outputs.items():
        old = path.read_bytes() if path.exists() else None
        new = content.encode('utf-8')
        if old != new:
            changes.append((path, old, new))
    if not changes:
        print('BT-PERM-V2 is already applied. No files changed.')
        return

    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    backup = root / 'patch_backups' / ('bt_perm_v2_' + stamp)
    backup.mkdir(parents=True, exist_ok=True)
    for path, old, _ in changes:
        if old is not None:
            saved = backup / path.relative_to(root)
            saved.parent.mkdir(parents=True, exist_ok=True)
            saved.write_bytes(old)
    try:
        for path, _, new in changes:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(new)
    except OSError:
        for path, old, _ in changes:
            if old is None:
                path.unlink(missing_ok=True)
            else:
                path.write_bytes(old)
        raise
    for path, _, _ in changes:
        print('Patched:', path.relative_to(root))
    print('Backups:', backup)
    print('Source checks passed; Android permissions have NOT been tested on a phone.')
    print('Next: run cell 10/10, install the NEW APK and open Settings.')
    print('Look for BT-PERM-V2. Grant Location (precise on Android 12+) and Nearby devices.')
    print('Tap Refresh, select the paired ELM327, then CONNECT + INIT ECU.')
    print('The Subaru SSM2/CAN protocol and calibration maps were not changed.')


try:
    run_source_tests()
    apply_fix(ROOT)
except (ValueError, ET.ParseError) as error:
    raise SystemExit('Preflight failed; source files were not changed: ' + str(error)) from error

Source self-checks passed: manifest, repeat application, missing-anchor rejection.
These checks do not compile Dart/Kotlin or test Android Bluetooth.
Patched: android/app/src/main/AndroidManifest.xml
Patched: android/app/src/main/kotlin/com/subarun/suba_run_v8/MainActivity.kt
Patched: lib/screens/settings_screen.dart
Patched: lib/main.dart
Patched: lib/services/bt_service.dart
Patched: android/app/src/main/kotlin/com/subarun/suba_run_v8/SubaBluetoothPermissionsV2.kt
Backups: /content/suba_run_v8/patch_backups/bt_perm_v2_20260911_080129_256556
Source checks passed; Android permissions have NOT been tested on a phone.
Next: run cell 10/10, install the NEW APK and open Settings.
Look for BT-PERM-V2. Grant Location (precise on Android 12+) and Nearby devices.
Tap Refresh, select the paired ELM327, then CONNECT + INIT ECU.
The Subaru SSM2/CAN protocol and calibration maps were not changed.


In [ ]:
# @title SUBA RUN V8: MAP-AXES-V2 (reversible display orientation)
# Run after the existing fixes, before cell 10/10. No ROM bytes are changed.
from pathlib import Path
from datetime import datetime, timezone
import subprocess

ROOT = Path('/content/suba_run_v8')
RUN_FLUTTER_TESTS = True

LAYOUT = r'''/// Coordinates are in the callbacks supplied to HeatMapView, not ROM offsets.
class SubaMapLayout {
  final int sourceRows;
  final int sourceCols;
  final bool transpose;
  final bool flipRows;
  final bool flipCols;

  SubaMapLayout({
    required this.sourceRows,
    required this.sourceCols,
    this.transpose = false,
    this.flipRows = false,
    this.flipCols = false,
  }) {
    if (sourceRows <= 0 || sourceCols <= 0) {
      throw ArgumentError('Map dimensions must be positive.');
    }
  }

  int get rows => transpose ? sourceCols : sourceRows;
  int get cols => transpose ? sourceRows : sourceCols;

  ({int r, int c}) toSource(int displayRow, int displayCol) {
    if (displayRow < 0 || displayRow >= rows || displayCol < 0 || displayCol >= cols) {
      throw RangeError('Display cell is outside the map.');
    }
    final r = flipRows ? rows - 1 - displayRow : displayRow;
    final c = flipCols ? cols - 1 - displayCol : displayCol;
    return transpose ? (r: c, c: r) : (r: r, c: c);
  }

  ({int r, int c}) toDisplay(int sourceRow, int sourceCol) {
    if (sourceRow < 0 || sourceRow >= sourceRows || sourceCol < 0 || sourceCol >= sourceCols) {
      throw RangeError('Source cell is outside the map.');
    }
    final r = transpose ? sourceCol : sourceRow;
    final c = transpose ? sourceRow : sourceCol;
    return (r: flipRows ? rows - 1 - r : r, c: flipCols ? cols - 1 - c : c);
  }
}
'''

WIDGET = r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/analyzer_service.dart';
import 'heat_colors.dart';
import 'map_orientation.dart';

class HeatMapView extends StatefulWidget {
  final int cols;
  final int rows;
  final double? Function(int r, int c) value;
  final int Function(int r, int c)? hits;
  final String Function(int c)? xLabel;
  final String Function(int r)? yLabel;
  final String title;
  final String unit;
  final String xAxisName;
  final String yAxisName;
  final double? hintLo;
  final double? hintHi;
  final void Function(int r, int c, double v)? onEdit;
  final double? Function(int r, int c)? original;
  final bool initialTranspose;
  final bool initialFlipRows;
  final bool initialFlipCols;
  final double initialCellSize;
  final bool isFullscreen;
  final void Function(bool transpose, bool flipRows, bool flipCols, double cell)? onViewChanged;

  const HeatMapView({
    super.key,
    required this.cols,
    required this.rows,
    required this.value,
    this.hits,
    this.xLabel,
    this.yLabel,
    this.title = '',
    this.unit = '',
    this.xAxisName = 'X',
    this.yAxisName = 'Y',
    this.hintLo,
    this.hintHi,
    this.onEdit,
    this.original,
    this.initialTranspose = false,
    this.initialFlipRows = false,
    this.initialFlipCols = false,
    this.initialCellSize = 58,
    this.isFullscreen = false,
    this.onViewChanged,
  });

  factory HeatMapView.fromGrid(HeatGrid g,
      {String title = '', String unit = '', double? hintLo, double? hintHi}) {
    final range = g.range(hintLo: hintLo, hintHi: hintHi);
    return HeatMapView(
      cols: g.cols,
      rows: g.rows,
      title: title,
      unit: unit,
      hintLo: range.lo,
      hintHi: range.hi,
      // Retain the caller contract; do not silently undo existing Y inversions.
      value: (r, c) => g.avg(g.rows - 1 - r, c),
      hits: (r, c) => g.n[g.rows - 1 - r][c],
      xLabel: (c) => g.xLabel(c),
      yLabel: (r) => g.yLabel(g.rows - 1 - r),
    );
  }

  @override
  State<HeatMapView> createState() => _HeatMapViewState();
}

class _HeatMapViewState extends State<HeatMapView> {
  late bool _transpose;
  late bool _flipRows;
  late bool _flipCols;
  late double _cell;
  ({int r, int c})? _selected;
  final TransformationController _pan = TransformationController();

  @override
  void initState() {
    super.initState();
    _transpose = widget.initialTranspose;
    _flipRows = widget.initialFlipRows;
    _flipCols = widget.initialFlipCols;
    _cell = widget.initialCellSize.clamp(36.0, 100.0).toDouble();
  }

  @override
  void didUpdateWidget(covariant HeatMapView oldWidget) {
    super.didUpdateWidget(oldWidget);
    if (oldWidget.rows != widget.rows || oldWidget.cols != widget.cols ||
        oldWidget.title != widget.title || oldWidget.key != widget.key) {
      _selected = null;
      _transpose = widget.initialTranspose;
      _flipRows = widget.initialFlipRows;
      _flipCols = widget.initialFlipCols;
      _pan.value = Matrix4.identity();
    }
  }

  @override
  void dispose() {
    _pan.dispose();
    super.dispose();
  }

  SubaMapLayout get _layout => SubaMapLayout(
      sourceRows: widget.rows, sourceCols: widget.cols,
      transpose: _transpose, flipRows: _flipRows, flipCols: _flipCols);

  String get _rowName => _transpose ? widget.xAxisName : widget.yAxisName;
  String get _colName => _transpose ? widget.yAxisName : widget.xAxisName;

  String _columnLabel(SubaMapLayout layout, int c) {
    final p = layout.toSource(0, c);
    return _transpose
        ? (widget.yLabel?.call(p.r) ?? '${p.r}')
        : (widget.xLabel?.call(p.c) ?? '${p.c}');
  }

  String _rowLabel(SubaMapLayout layout, int r) {
    final p = layout.toSource(r, 0);
    return _transpose
        ? (widget.xLabel?.call(p.c) ?? '${p.c}')
        : (widget.yLabel?.call(p.r) ?? '${p.r}');
  }

  String _fmt(double? value) => value == null || !value.isFinite
      ? 'n/a' : value.toStringAsFixed(value.abs() >= 1000 ? 0 : 1);

  void _changeView(VoidCallback change) {
    setState(change);
    _pan.value = Matrix4.identity();
    widget.onViewChanged?.call(_transpose, _flipRows, _flipCols, _cell);
  }

  @override
  Widget build(BuildContext context) {
    if (widget.rows <= 0 || widget.cols <= 0 || widget.rows * widget.cols > 16384) {
      return const Center(child: Text('MAP-AXES-V2: некорректный или слишком большой размер карты.'));
    }
    try {
      final layout = _layout;
      // Snapshot callbacks once per build; all display transforms use the same data.
      final data = List.generate(widget.rows,
          (r) => List<double?>.generate(widget.cols, (c) => widget.value(r, c)));
      final originals = widget.original == null ? null : List.generate(widget.rows,
          (r) => List<double?>.generate(widget.cols, (c) => widget.original!(r, c)));
      final xLabels = List.generate(layout.cols, (c) => _columnLabel(layout, c));
      final yLabels = List.generate(layout.rows, (r) => _rowLabel(layout, r));
      var minimum = double.infinity;
      var maximum = -double.infinity;
      for (final row in data) {
        for (final value in row) {
          if (value == null || !value.isFinite) continue;
          if (value < minimum) minimum = value;
          if (value > maximum) maximum = value;
        }
      }
      final lo = widget.hintLo?.isFinite == true ? widget.hintLo! : (minimum.isFinite ? minimum : 0.0);
      var hi = widget.hintHi?.isFinite == true ? widget.hintHi! : (maximum.isFinite ? maximum : 1.0);
      if (hi <= lo) hi = lo + 1.0;

      return Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
        Row(children: [
          Expanded(child: Text(widget.title.isEmpty ? 'MAP-AXES-V2' : '${widget.title} | MAP-AXES-V2',
              overflow: TextOverflow.ellipsis,
              style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold, color: HeatColors.accent))),
          Text('${layout.rows} x ${layout.cols}', style: const TextStyle(fontSize: 11)),
          if (!widget.isFullscreen)
            IconButton(tooltip: 'Во весь экран', onPressed: _openFullscreen,
                icon: const Icon(Icons.fullscreen, size: 20)),
        ]),
        Wrap(spacing: 2, children: [
          TextButton.icon(
            key: const ValueKey('map-transpose'),
            onPressed: () => _changeView(() => _transpose = !_transpose),
            icon: Icon(Icons.swap_horiz, size: 16, color: _transpose ? HeatColors.gold : null),
            label: Text(_transpose ? 'X/Y поменяны' : 'Поменять X/Y', style: const TextStyle(fontSize: 11)),
          ),
          TextButton(
            key: const ValueKey('map-flip-rows'),
            onPressed: () => _changeView(() => _flipRows = !_flipRows),
            child: Text(_flipRows ? 'Строки: обратно' : 'Развернуть строки',
                style: TextStyle(fontSize: 11, color: _flipRows ? HeatColors.gold : null)),
          ),
          TextButton(
            key: const ValueKey('map-flip-cols'),
            onPressed: () => _changeView(() => _flipCols = !_flipCols),
            child: Text(_flipCols ? 'Столбцы: обратно' : 'Развернуть столбцы',
                style: TextStyle(fontSize: 11, color: _flipCols ? HeatColors.gold : null)),
          ),
          TextButton(onPressed: () => _changeView(() {
            _transpose = false; _flipRows = false; _flipCols = false;
          }), child: const Text('Исходный вид', style: TextStyle(fontSize: 11))),
          IconButton(tooltip: 'Уменьшить ячейки', onPressed: () => _changeView(() {
            _cell = (_cell - 8).clamp(36.0, 100.0).toDouble();
          }), icon: const Icon(Icons.remove, size: 18)),
          IconButton(tooltip: 'Увеличить ячейки', onPressed: () => _changeView(() {
            _cell = (_cell + 8).clamp(36.0, 100.0).toDouble();
          }), icon: const Icon(Icons.add, size: 18)),
          IconButton(tooltip: 'Копировать ориентацию', onPressed: _copyDiagnostic,
              icon: const Icon(Icons.copy, size: 18)),
        ]),
        Text('Строки: $_rowName | Столбцы: $_colName | Только отображение',
            style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
        const SizedBox(height: 6),
        Expanded(child: InteractiveViewer(
          transformationController: _pan,
          constrained: false,
          alignment: Alignment.topLeft,
          minScale: 0.4,
          maxScale: 3,
          boundaryMargin: const EdgeInsets.all(16),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: [
              _header('$_rowName / $_colName', 72, 32),
              ...List.generate(layout.cols, (c) => _header(xLabels[c], _cell, 32,
                  key: ValueKey('map-col-label-$c'))),
            ]),
            ...List.generate(layout.rows, (r) => Row(children: [
              _header(yLabels[r], 72, _cell, key: ValueKey('map-row-label-$r')),
              ...List.generate(layout.cols, (c) {
                final p = layout.toSource(r, c);
                final value = data[p.r][p.c];
                final original = originals?[p.r][p.c];
                final finite = value != null && value.isFinite;
                final changed = finite && original != null && original.isFinite && (original - value).abs() > 1e-9;
                final selected = _selected == p;
                final normalized = finite ? ((value - lo) / (hi - lo)).clamp(0.0, 1.0).toDouble() : 0.0;
                return GestureDetector(
                  key: ValueKey('map-cell-$r-$c'),
                  onTap: () => setState(() => _selected = p),
                  onLongPress: widget.onEdit == null || !finite ? null : () => _edit(p.r, p.c, value),
                  child: Container(
                    width: _cell, height: _cell,
                    alignment: Alignment.center,
                    decoration: BoxDecoration(
                      color: !finite ? HeatColors.bg : changed ? const Color(0xFF1E6AE1) : heatColor(normalized),
                      border: Border.all(color: selected ? Colors.cyanAccent : Colors.white24, width: selected ? 2 : 0.5),
                    ),
                    child: changed
                      ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
                          Text(_fmt(original), style: const TextStyle(fontSize: 10, color: Colors.white70,
                              decoration: TextDecoration.lineThrough)),
                          Text(_fmt(value), style: const TextStyle(fontSize: 12, color: Colors.white, fontWeight: FontWeight.bold)),
                        ])
                      : Text(_fmt(value), style: TextStyle(fontSize: _cell < 46 ? 10 : 12,
                          color: Colors.white, shadows: const [Shadow(color: Colors.black, blurRadius: 2)])),
                  ),
                );
              }),
            ])),
          ]),
        )),
        if (_selected != null)
          Padding(padding: const EdgeInsets.only(top: 6), child: Text(
            'callback[r=${_selected!.r}, c=${_selected!.c}] = ${_fmt(data[_selected!.r][_selected!.c])} ${widget.unit}'
            '${widget.hits == null ? '' : ' | hits: ${widget.hits!(_selected!.r, _selected!.c)}'}',
            style: const TextStyle(fontSize: 11, color: HeatColors.gold),
          )),
      ]);
    } catch (e) {
      return Center(child: Padding(padding: const EdgeInsets.all(12), child: SelectableText(
          'MAP-AXES-V2: ошибка данных или подписей осей: $e\n'
          'Проверьте rows/cols и длины X/Y. Транспонирование не исправляет декодер.',
          style: const TextStyle(fontSize: 12, color: Colors.orangeAccent))));
    }
  }

  Widget _header(String text, double width, double height, {Key? key}) => Container(
    key: key, width: width, height: height,
    alignment: Alignment.center,
    decoration: BoxDecoration(color: const Color(0xFF0F3460), border: Border.all(color: Colors.white24, width: 0.5)),
    child: Text(text, textAlign: TextAlign.center, maxLines: 2,
        overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 10, color: Colors.white70)),
  );

  Future<void> _edit(int sourceRow, int sourceCol, double current) async {
    var input = current.toString();
    final result = await showDialog<double>(context: context, builder: (dialogContext) {
        String? error;
        return StatefulBuilder(builder: (dialogContext, update) => AlertDialog(
          title: const Text('Правка выбранной ячейки'),
          content: Column(mainAxisSize: MainAxisSize.min, crossAxisAlignment: CrossAxisAlignment.start, children: [
            Text('callback[r=$sourceRow, c=$sourceCol]\n'
                '${widget.yAxisName}: ${widget.yLabel?.call(sourceRow) ?? sourceRow}\n'
                '${widget.xAxisName}: ${widget.xLabel?.call(sourceCol) ?? sourceCol}',
                style: const TextStyle(fontSize: 12)),
            const Text('Сначала сверьте карту с исходным определением.', style: TextStyle(fontSize: 11)),
            TextFormField(initialValue: input, autofocus: true, onChanged: (value) => input = value,
                keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
                decoration: InputDecoration(labelText: 'Значение ${widget.unit}', errorText: error)),
          ]),
          actions: [
            TextButton(onPressed: () => Navigator.pop(dialogContext), child: const Text('Отмена')),
            FilledButton(onPressed: () {
              final value = double.tryParse(input.replaceAll(',', '.'));
              if (value == null || !value.isFinite) {
                update(() => error = 'Введите конечное число');
                return;
              }
              Navigator.pop(dialogContext, value);
            }, child: const Text('Применить к ячейке')),
          ],
        ));
      });
    if (result == null || !mounted) return;
    try {
      // Map display to source exactly once; caller retains its own ROM mapping.
      widget.onEdit?.call(sourceRow, sourceCol, result);
      setState(() => _selected = (r: sourceRow, c: sourceCol));
    } catch (e) {
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text('Правка не применена: $e')));
    }
  }

  Future<void> _copyDiagnostic() async {
    try {
      final layout = _layout;
      final report = StringBuffer('MAP-AXES-V2\n')
        ..writeln('title: ${widget.title}')
        ..writeln('callback size: ${widget.rows} rows x ${widget.cols} columns')
        ..writeln('display size: ${layout.rows} rows x ${layout.cols} columns')
        ..writeln('transpose=$_transpose flipRows=$_flipRows flipCols=$_flipCols')
        ..writeln('source X first/last: ${widget.xLabel?.call(0)} / ${widget.xLabel?.call(widget.cols - 1)}')
        ..writeln('source Y first/last: ${widget.yLabel?.call(0)} / ${widget.yLabel?.call(widget.rows - 1)}')
        ..writeln('Rows/columns are callback indices, NOT ROM addresses.');
      for (final corner in [(0, 0), (0, layout.cols - 1), (layout.rows - 1, 0), (layout.rows - 1, layout.cols - 1)]) {
        final p = layout.toSource(corner.$1, corner.$2);
        report.writeln('display[${corner.$1},${corner.$2}] -> callback[${p.r},${p.c}] = ${widget.value(p.r, p.c)}');
      }
      await Clipboard.setData(ClipboardData(text: report.toString()));
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Ориентация скопирована')));
    } catch (e) {
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text('Ошибка диагностики: $e')));
    }
  }

  Future<void> _openFullscreen() async {
    await Navigator.of(context).push<void>(MaterialPageRoute(builder: (pageContext) => Scaffold(
      appBar: AppBar(title: Text(widget.title.isEmpty ? 'Карта' : widget.title),
          leading: IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.pop(pageContext))),
      body: Padding(padding: const EdgeInsets.all(8), child: HeatMapView(
        rows: widget.rows, cols: widget.cols, value: widget.value, hits: widget.hits,
        xLabel: widget.xLabel, yLabel: widget.yLabel, xAxisName: widget.xAxisName, yAxisName: widget.yAxisName,
        title: widget.title, unit: widget.unit, hintLo: widget.hintLo, hintHi: widget.hintHi,
        original: widget.original, onEdit: widget.onEdit, isFullscreen: true,
        initialTranspose: _transpose, initialFlipRows: _flipRows, initialFlipCols: _flipCols, initialCellSize: _cell,
        onViewChanged: (transpose, flipRows, flipCols, cell) {
          if (!mounted) return;
          _changeView(() { _transpose = transpose; _flipRows = flipRows; _flipCols = flipCols; _cell = cell; });
        },
      )),
    )));
    if (mounted) setState(() {});
  }
}
'''

TESTS = r'''import 'package:flutter/material.dart';
import 'package:flutter_test/flutter_test.dart';
import '../lib/widgets/map_orientation.dart';
import '../lib/widgets/heat_map.dart';

void main() {
  test('All eight orientations are bijections, including non-square maps', () {
    for (final shape in [(2, 3), (16, 16), (22, 20), (1, 5), (5, 1), (1, 1)]) {
      for (var bits = 0; bits < 8; bits++) {
        final layout = SubaMapLayout(sourceRows: shape.$1, sourceCols: shape.$2,
            transpose: (bits & 1) != 0, flipRows: (bits & 2) != 0, flipCols: (bits & 4) != 0);
        final seen = <String>{};
        for (var r = 0; r < layout.rows; r++) {
          for (var c = 0; c < layout.cols; c++) {
            final source = layout.toSource(r, c);
            expect(source.r, inInclusiveRange(0, shape.$1 - 1));
            expect(source.c, inInclusiveRange(0, shape.$2 - 1));
            expect(layout.toDisplay(source.r, source.c), (r: r, c: c));
            seen.add('${source.r},${source.c}');
          }
        }
        expect(seen.length, shape.$1 * shape.$2);
      }
    }
  });

  test('Transpose, row reversal, and column reversal are different operations', () {
    final source = [[1, 2, 3], [4, 5, 6]];
    List<List<int>> view(SubaMapLayout layout) => List.generate(layout.rows, (r) =>
        List.generate(layout.cols, (c) { final p = layout.toSource(r, c); return source[p.r][p.c]; }));
    expect(view(SubaMapLayout(sourceRows: 2, sourceCols: 3, transpose: true)), [[1, 4], [2, 5], [3, 6]]);
    expect(view(SubaMapLayout(sourceRows: 2, sourceCols: 3, flipRows: true)), [[4, 5, 6], [1, 2, 3]]);
    expect(view(SubaMapLayout(sourceRows: 2, sourceCols: 3, flipCols: true)), [[3, 2, 1], [6, 5, 4]]);
    expect(view(SubaMapLayout(sourceRows: 2, sourceCols: 3, transpose: true, flipRows: true)), [[3, 6], [2, 5], [1, 4]]);
    expect(source, [[1, 2, 3], [4, 5, 6]]);
    expect(() => SubaMapLayout(sourceRows: 0, sourceCols: 3), throwsArgumentError);
    expect(() => SubaMapLayout(sourceRows: 2, sourceCols: 3).toSource(2, 0), throwsRangeError);
  });

  testWidgets('Values, labels, and edits retain the same source cell after transpose', (tester) async {
    final source = <List<double>>[[1, 2, 3], [4, 5, 6]];
    await tester.pumpWidget(MaterialApp(home: Scaffold(body: HeatMapView(
      rows: 2, cols: 3, value: (r, c) => source[r][c],
      xLabel: (c) => ['10', '20', '30'][c], yLabel: (r) => ['1000', '2000'][r],
      onEdit: (r, c, value) => source[r][c] = value,
    ))));
    await tester.tap(find.byKey(const ValueKey('map-transpose')));
    await tester.pump();
    expect(find.descendant(of: find.byKey(const ValueKey('map-col-label-1')),
        matching: find.text('2000')), findsOneWidget);
    expect(find.descendant(of: find.byKey(const ValueKey('map-row-label-2')),
        matching: find.text('30')), findsOneWidget);
    expect(find.descendant(of: find.byKey(const ValueKey('map-cell-2-1')),
        matching: find.text('6.0')), findsOneWidget);
    await tester.longPress(find.byKey(const ValueKey('map-cell-2-1')));
    await tester.pumpAndSettle();
    await tester.enterText(find.byType(TextField), '9');
    await tester.tap(find.text('Применить к ячейке'));
    await tester.pumpAndSettle();
    expect(source, [[1.0, 2.0, 3.0], [4.0, 5.0, 9.0]]);
    expect(tester.takeException(), isNull);
  });

  testWidgets('An existing caller Y inversion is preserved, not applied twice', (tester) async {
    final rom = <List<double>>[[1, 2, 3], [4, 5, 6]];
    await tester.pumpWidget(MaterialApp(home: Scaffold(body: HeatMapView(
      rows: 2, cols: 3,
      value: (r, c) => rom[1 - r][c],
      xLabel: (c) => ['10', '20', '30'][c],
      yLabel: (r) => ['1000', '2000'][1 - r],
      hits: (r, c) => rom[1 - r][c].toInt(),
      onEdit: (r, c, value) => rom[1 - r][c] = value,
    ))));
    await tester.tap(find.byKey(const ValueKey('map-transpose')));
    await tester.pump();
    await tester.tap(find.byKey(const ValueKey('map-flip-cols')));
    await tester.pump();
    expect(find.descendant(of: find.byKey(const ValueKey('map-col-label-1')),
        matching: find.text('2000')), findsOneWidget);
    await tester.tap(find.byKey(const ValueKey('map-cell-2-1')));
    await tester.pump();
    expect(find.textContaining('hits: 6'), findsOneWidget);
    await tester.longPress(find.byKey(const ValueKey('map-cell-2-1')));
    await tester.pumpAndSettle();
    await tester.enterText(find.byType(TextField), '12');
    await tester.tap(find.text('Применить к ячейке'));
    await tester.pumpAndSettle();
    expect(rom, [[1.0, 2.0, 3.0], [4.0, 5.0, 12.0]]);
    expect(tester.takeException(), isNull);
  });
}
'''

widget_path = ROOT / 'lib/widgets/heat_map.dart'
if not widget_path.is_file():
    raise SystemExit('No files changed: restore the existing SUBA V8 project first.')
old_widget = widget_path.read_text(encoding='utf-8')
if 'class HeatMapView' not in old_widget or 'HeatGrid' not in old_widget:
    raise SystemExit('No files changed: unexpected heat_map.dart. Send this file before applying a replacement.')
if not (ROOT / 'lib/services/analyzer_service.dart').is_file():
    raise SystemExit('No files changed: analyzer_service.dart is missing.')

outputs = {
    ROOT / 'lib/widgets/map_orientation.dart': LAYOUT,
    widget_path: WIDGET,
    ROOT / 'test/suba_map_axes_test.dart': TESTS,
}
changes = []
for path, text in outputs.items():
    before = path.read_bytes() if path.exists() else None
    after = text.encode('utf-8')
    if before != after:
        changes.append((path, before, after))

if changes:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    backup = ROOT / 'patch_backups' / ('map_axes_v2_' + stamp)
    backup.mkdir(parents=True, exist_ok=True)
    for path, before, _ in changes:
        if before is not None:
            saved = backup / path.relative_to(ROOT)
            saved.parent.mkdir(parents=True, exist_ok=True)
            saved.write_bytes(before)
    try:
        for path, _, after in changes:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(after)
    except OSError:
        for path, before, _ in changes:
            if before is None:
                path.unlink(missing_ok=True)
            else:
                path.write_bytes(before)
        raise
    print('Backups:', backup)
    for path, _, _ in changes:
        print('Patched:', path.relative_to(ROOT))
else:
    print('MAP-AXES-V2 sources are already applied.')

if RUN_FLUTTER_TESTS:
    flutter = Path('/content/flutter/bin/flutter')
    if not flutter.is_file() or not (ROOT / '.dart_tool/package_config.json').is_file():
        raise SystemExit('Sources are patched, but tests were NOT run. Flutter and pub get are required in this runtime.')
    print('\nRunning mapping and widget tests (no Bluetooth hardware used)...')
    try:
        result = subprocess.run(
            [str(flutter), 'test', '--no-pub', 'test/suba_map_axes_test.dart'],
            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=240,
        )
    except subprocess.TimeoutExpired as error:
        raise SystemExit('Sources are patched; tests timed out. Send the output before building.') from error
    print(result.stdout)
    if result.returncode != 0:
        raise SystemExit('Tests FAILED. Sources remain patched and backups are available. Send this output; do not use the patch for tuning yet.')
    print('Mapping and widget tests passed IN THIS COLAB RUNTIME. ROM decoding is not validated by these tests.')
else:
    print('Tests were explicitly skipped. Dart/Kotlin compilation is NOT verified.')

print('\nNext: run cell 10/10 and install the new APK. Look for MAP-AXES-V2 on a map.')
print('Use Swap X/Y only for swapped axes; use Reverse rows/columns for descending axes.')
print('Default orientation is unchanged. Existing caller inversions are retained intentionally.')
print('The decoder, generated definitions, CSV export, saveMod, ROM bytes, and Bluetooth were NOT modified.')
print('Compare the same map in a trusted editor before editing or saving any calibration.')

Backups: /content/suba_run_v8/patch_backups/map_axes_v2_20260911_080136_729955
Patched: lib/widgets/map_orientation.dart
Patched: lib/widgets/heat_map.dart
Patched: test/suba_map_axes_test.dart

Running mapping and widget tests (no Bluetooth hardware used)...
00:00 +0: loading /content/suba_run_v8/test/suba_map_axes_test.dart
00:00 +0: All eight orientations are bijections, including non-square maps
00:00 +1: Transpose, row reversal, and column reversal are different operations
00:00 +2: Values, labels, and edits retain the same source cell after transpose
00:02 +3: An existing caller Y inversion is preserved, not applied twice
00:03 +4: All tests passed!

Mapping and widget tests passed IN THIS COLAB RUNTIME. ROM decoding is not validated by these tests.

Next: run cell 10/10 and install the new APK. Look for MAP-AXES-V2 on a map.
Use Swap X/Y only for swapped axes; use Reverse rows/columns for descending axes.
Default orientation is unchanged. Existing caller inversions are retained 

In [ ]:
# @title 🩹 ФИКС V8: SSM2-over-CAN init без K-line кадра (правит "ECU не ответил на SSM2 init")
# ============================================================================
# ПРИЧИНА ОШИБКИ "ECU не ответил на SSM2 init (8010F001BF40)":
#   В V8 метод ecuInit() слал СЫРОЙ K-line SSM2-кадр 80 10 F0 01 BF 40.
#   Это формат ISO-14230 (K-line), а НЕ CAN. По CAN-шине (ATSP6) ЭБУ Subaru
#   этот заголовок не понимает и молчит -> init всегда фейлится.
#
#   В рабочей V7 инициализация для CAN сделана иначе:
#     - живость шины проверяется OBD-II запросом 0100 (ищем ответ 4100)
#     - реальный ECU ID читается прямым SSM2 A8-запросом (readBytes), а НЕ
#       K-line init-кадром. Если ID не пришёл — сессия всё равно живая.
#
# ЧТО ДЕЛАЕТ ЭТОТ ФИКС:
#   Переписывает ТОЛЬКО ecuInit() в lib/ssm/ssm_elm.dart.
#   Всё остальное (transact / readBytes / poller / ATCAF1) не трогаем —
#   оно уже корректно для SSM2-over-CAN.
# ============================================================================
import os, re
os.chdir('/content/suba_run_v8')

path = 'lib/ssm/ssm_elm.dart'
code = open(path).read()

# --- Новый метод ecuInit для CAN ---
new_init = r'''  /// SSM2-over-CAN init.
  /// ВАЖНО: по CAN НЕ шлём K-line кадр 80 10 F0 01 BF 40 — ЭБУ его не понимает.
  /// 1) Проверяем живость шины OBD-II запросом 0100 (ждём 4100).
  /// 2) Best-effort читаем ECU ID прямым SSM2 A8-запросом (адрес 0x000200, 5 байт).
  ///    Если ID не пришёл — сессия всё равно считается живой (как в V7).
  Future<String?> ecuInit() async {
    // 1. Живость CAN-шины через штатный OBD-II PID 0100
    bool busAlive = false;
    for (var attempt = 0; attempt < 3; attempt++) {
      final r = await transact('0100', timeoutMs: 2500);
      final hex = r.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
      if (hex.contains('4100')) { busAlive = true; break; }
      // явные признаки живого ELM без ответа ECU — пробуем ещё
      await Future.delayed(const Duration(milliseconds: 200));
    }
    if (!busAlive) {
      _setState(SsmState.error);
      return null;
    }

    // Шина отвечает — считаем ECU готовым (SSM2 A8-чтение работает и без init).
    _setState(SsmState.ecuReady);

    // 2. Best-effort: читаем ROM/ECU ID прямым SSM2-запросом. НЕ роняет сессию.
    try {
      final id = await _readEcuIdSsm2();
      if (id != null && id.isNotEmpty) ecuId = id;
    } catch (_) {}

    return ecuId.isEmpty ? 'OK' : ecuId;
  }

  /// Читает 5-байтный ECU/ROM ID через SSM2 A8-чтение (адрес 0x000200).
  /// Возвращает ID в виде hex-строки (напр. "5204584007") или null.
  Future<String?> _readEcuIdSsm2() async {
    final data = await readBytes(0x000200, 5);
    if (data == null || data.length < 5) return null;
    final sb = StringBuffer();
    for (final b in data) {
      sb.write(b.toRadixString(16).padLeft(2, '0'));
    }
    return sb.toString().toUpperCase();
  }'''

# Заменяем старый ecuInit (от сигнатуры до закрывающей '}' метода)
pattern = re.compile(
    r'  /// SSM2 init\..*?\n  Future<String\?> ecuInit\(\) async \{.*?\n  \}',
    re.DOTALL,
)

if pattern.search(code):
    code = pattern.sub(new_init, code, count=1)
    print('✅ ecuInit() заменён на CAN-совместимую версию (0100 + A8 ECU ID)')
else:
    # Фолбэк: ищем только по сигнатуре
    pattern2 = re.compile(r'  Future<String\?> ecuInit\(\) async \{.*?\n  \}', re.DOTALL)
    if pattern2.search(code):
        code = pattern2.sub(new_init, code, count=1)
        print('✅ ecuInit() заменён (фолбэк по сигнатуре)')
    else:
        raise RuntimeError('Не нашёл метод ecuInit() в ssm_elm.dart — проверь путь/версию.')

open(path, 'w').write(code)
print('✅ Записан', path)




✅ ecuInit() заменён на CAN-совместимую версию (0100 + A8 ECU ID)
✅ Записан lib/ssm/ssm_elm.dart


In [ ]:
# @title 🩹 ФИКС V8: Инициализация ЭБУ как в рабочем V7 + Сборка APK
# ============================================================
# Портирует в V8 проверенный стек V7 (ячейки 24 + 35 final fix):
#  1. AT: ATCAF0 + ATH1 + ATFCS* + ATST>=0x20 + ATAT1
#  2. Ворота: OBD 0100->4100 (а не BF-конверт 8010F001BF40)
#  3. BF пробуется 3 способами; решает живой A8-probe (ECT)
#  4. A8 строго по спецификации: адрес НА КАЖДЫЙ байт, без count
#  5. Парсер V7-style: E8 после 7E8, ошибки ДО hex-чистки
# Запускать ПОСЛЕ ячеек 0-9 и всех фиксов, ВМЕСТО/ПЕРЕД ячейкой 10/10.
# Идемпотентна: можно перезапускать.
# ============================================================
import os
os.chdir('/content/suba_run_v8')

print("=" * 64)
print("  ФИКС V8: протокол SSM2-over-CAN как в рабочем V7")
print("=" * 64)

# ---------- 0. Бэкап текущего файла ----------
import shutil, datetime
bak = 'lib/ssm/ssm_elm.dart.bak_{}'.format(datetime.datetime.now().strftime('%H%M%S'))
shutil.copy('lib/ssm/ssm_elm.dart', bak)
print("  бэкап: {}".format(bak))

# ---------- 1. Переписываем lib/ssm/ssm_elm.dart ----------
with open('lib/ssm/ssm_elm.dart', 'w', encoding='utf-8') as f:
    f.write('''
import 'dart:async';
import 'dart:convert';
import 'dart:typed_data';

import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/live_snapshot.dart';

// ─────────────────────────────────────────────────────────────
// Статистика протокола (чтобы тормоза было ВИДНО на экране)
// ─────────────────────────────────────────────────────────────
class SsmStats {
  int framesOk = 0;
  int framesErr = 0;
  int noData = 0;
  double lastMs = 0;
  double avgMs = 0;
  double hz = 0;
  int _snapCounter = 0;
  DateTime _hzStart = DateTime.now();

  void ok(double ms) {
    framesOk++;
    lastMs = ms;
    avgMs = avgMs == 0 ? ms : avgMs * 0.85 + ms * 0.15;
  }

  void err() => framesErr++;
  void nodata() => noData++;

  void snap() {
    _snapCounter++;
    final d = DateTime.now().difference(_hzStart).inMilliseconds;
    if (d >= 1000) {
      hz = _snapCounter * 1000.0 / d;
      _snapCounter = 0;
      _hzStart = DateTime.now();
    }
  }

  void reset() {
    framesOk = 0;
    framesErr = 0;
    noData = 0;
    avgMs = 0;
    hz = 0;
  }
}

// ─────────────────────────────────────────────────────────────
// Блок чтения: непрерывный диапазон адресов + PID внутри
// ─────────────────────────────────────────────────────────────
class SsmBlock {
  final int start;
  final int len;
  final List<SubaruPid> pids;
  final int prio; // 1 fast, 2 mid, 3 slow
  int consecErr = 0;
  int skipUntilCycle = 0;

  SsmBlock(this.start, this.len, this.pids, this.prio);

  String get rangeHex => '0x${start.toRadixString(16).toUpperCase().padLeft(6, '0')}'
      '+${len.toRadixString(16).toUpperCase()}';
}

enum SsmState { disconnected, connecting, elmReady, ecuReady, polling, error }

// ─────────────────────────────────────────────────────────────
// Низкоуровневый ELM327 -> SSM2 over CAN (0x7E0 / 0x7E8, 500k, 11bit)
// ФИКС V8 — портировано из рабочего V7 (ячейки 24 + 35 final fix):
//  • AT-последовательность V7: ATCAF0 + ATH1 + ATFCS* + ATST>=0x20 + ATAT1
//  • Шина проверяется через OBD 0100->4100, а не через BF-конверт
//  • BF-init пробуется 3 способами, но НЕ является воротами:
//    решающий тест — живой A8-probe (чтение ECT), как автоскан в V7
//  • A8 строго по спецификации: A8 00 + 3-байтный адрес НА КАЖДЫЙ байт
//    (V8 слал A8 00 + addr + count — такой формат ECU игнорирует)
//  • Парсер V7-style: ищет E8 после 7E8, возвращает ЧИСТЫЕ данные,
//    ошибки детектятся ДО hex-чистки
// ─────────────────────────────────────────────────────────────
class SsmElm {
  BluetoothConnection? _conn;
  StreamSubscription<Uint8List>? _sub;
  final StringBuffer _rx = StringBuffer();
  Completer<void>? _promptWaiter;

  final stats = SsmStats();
  SsmState state = SsmState.disconnected;

  int stTimeoutCode = 0x20; // AT ST (x4 мс): 0x20 = 128 мс — минимум для клонов (V7: ATST32)
  int readRetries = 1;      // повторы однобайтового кадра при NO DATA
  String elmVersion = '';
  String ecuId = '';
  String lastInitLog = '';  // человекочитаемый лог последнего ecuInit
  bool _cafManual = true;   // true = ATCAF0 (ручные ISO-TP кадры для SSM2)

  final StreamController<SsmState> stateCtl = StreamController<SsmState>.broadcast();
  Stream<SsmState> get onState => stateCtl.stream;

  void _setState(SsmState s) {
    state = s;
    if (!stateCtl.isClosed) stateCtl.add(s);
  }

  /// Публичный маркер для поллера: опрос идёт / остановлен
  void markPolling(bool v) => _setState(v ? SsmState.polling : SsmState.ecuReady);

  Future<bool> connect(String address) async {
    _setState(SsmState.connecting);
    try {
      _conn = await BluetoothConnection.toAddress(address)
          .timeout(const Duration(seconds: 12));
    } catch (_) {
      _setState(SsmState.error);
      return false;
    }
    _sub = _conn!.input!.listen(_onData, onDone: () => disconnect(), onError: (_) => disconnect());
    final ok = await setupElm();
    return ok;
  }

  void _onData(Uint8List chunk) {
    for (final b in chunk) {
      if (b == 0x3E) { // '>'
        _promptWaiter?.complete();
        _promptWaiter = null;
      } else {
        _rx.writeCharCode(b);
      }
    }
  }

  Future<String> transact(String cmd, {int timeoutMs = AppConstants.elmCmdTimeoutMs}) async {
    if (_conn == null) throw StateError('not connected');
    _rx.clear();
    _promptWaiter = Completer<void>();
    _conn!.output.add(Uint8List.fromList(ascii.encode('$cmd\\r')));
    await _conn!.output.allSent;
    try {
      await _promptWaiter!.future.timeout(Duration(milliseconds: timeoutMs));
    } catch (_) {
      // таймаут: вернём что успели накопить (парсер сам решит)
    }
    return _rx.toString();
  }

  Future<String> _expectOk(String cmd, {int timeoutMs = 900}) async {
    final r = await transact(cmd, timeoutMs: timeoutMs);
    return r.toUpperCase();
  }

  String _clean(String r) => r.toUpperCase()
      .replaceAll(' ', '')
      .replaceAll('\\r', '')
      .replaceAll('\\n', '')
      .replaceAll('\\t', '')
      .replaceAll('>', '')
      .replaceAll('SEARCHING...', '')
      .replaceAll('STOPPED', '');

  /// Возврат в ручной ISO-TP режим для SSM2 (V7 final fix)
  Future<void> _toManual() async {
    await transact('ATCAF0', timeoutMs: 500);
    await transact('ATH1', timeoutMs: 400);
    await transact('ATSH7E0', timeoutMs: 400);
    await transact('ATCRA7E8', timeoutMs: 400);
    _cafManual = true;
  }

  Future<bool> setupElm() async {
    // Сброс: ждём баннер ELM327
    String resp = '';
    for (var i = 0; i < 3; i++) {
      resp = await transact('ATZ', timeoutMs: AppConstants.elmInitTimeoutMs);
      if (resp.toUpperCase().contains('ELM327')) break;
      await Future.delayed(const Duration(milliseconds: 400));
    }
    if (!resp.toUpperCase().contains('ELM327')) {
      lastInitLog = 'ATZ: нет баннера ELM327';
      _setState(SsmState.error);
      return false;
    }
    elmVersion = RegExp(r'ELM327[^\\r\\n]*').firstMatch(resp)?.group(0) ?? 'ELM327';

    // ST: минимум 0x20 (V7 работал на ATST32 = 200 мс; 0x08 = 32 мс роняет ответы)
    final stCode = stTimeoutCode < 0x20 ? 0x20 : (stTimeoutCode > 0xFF ? 0xFF : stTimeoutCode);
    final st = stCode.toRadixString(16).padLeft(2, '0').toUpperCase();

    // База (V7 working set): тишина ELM + адаптивный тайминг
    for (final cmd in <String>['ATE0', 'ATL0', 'ATS0', 'ATAT1', 'ATST$st']) {
      await _expectOk(cmd);
    }
    // CAN 500k + ручной ISO-TP (V7 final fix, ячейка 35)
    for (final cmd in <String>[
      'ATSP6',
      'ATCAF0',       // ручная сборка кадров — шлём сырые ISO-TP single-frame
      'ATH1',         // заголовки видны → парсим 7E8/E8 надёжно
      'ATSH7E0',      // запросы к моторному ECU
      'ATCRA7E8',     // фильтр ответов ECU
      'ATFCSH7E0',    // flow-control для мультифреймов
      'ATFCSD300000',
      'ATFCSM1',
    ]) {
      await _expectOk(cmd);
    }
    _cafManual = true;
    _setState(SsmState.elmReady);
    return true;
  }

  /// SSM2 init — каскад как в рабочем V7.
  /// ВАЖНО: ворота — OBD 0100->4100 + живой A8-probe, а НЕ BF-конверт.
  /// Старый V8 требовал именно '8010F001BF40' (K-line конверт поверх CAN) —
  /// этот ECU на него молчит, отсюда красный баннер на скриншоте.
  Future<String?> ecuInit() async {
    final log = StringBuffer();

    // ── Стадия 1: жива ли CAN-шина? (V7: временный ATCAF1 + 0100) ──
    await transact('ATCAF1', timeoutMs: 500);
    _cafManual = false;
    await transact('ATSH7E0', timeoutMs: 400);
    await transact('ATCRA7E8', timeoutMs: 400);
    var canOk = false;
    for (var i = 0; i < 2; i++) {
      final r = await transact('0100', timeoutMs: 3000);
      if (_clean(r).contains('4100')) { canOk = true; break; }
      if (i == 0) {
        await transact('ATSP0', timeoutMs: 600); // авто-протокол, одна попытка
        final r2 = await transact('0100', timeoutMs: 4000);
        if (_clean(r2).contains('4100')) {
          canOk = true;
          await transact('ATSP6', timeoutMs: 800);
          await transact('ATSH7E0', timeoutMs: 400);
          await transact('ATCRA7E8', timeoutMs: 400);
          break;
        }
      }
    }
    if (!canOk) {
      await _toManual();
      lastInitLog = 'CAN: 0100 без 4100 — шина/адаптер не отвечают';
      return null;
    }
    log.writeln('CAN 0100->4100: OK');

    // ── Стадия 2: BF-init тремя способами (любой успех — хорошо) ──
    // 2a. BF в авто-режиме: ELM сам обернёт в ISO-TP
    var r = await transact('BF', timeoutMs: 2500);
    var c = _clean(r);
    log.writeln('BF(auto): len=${c.length} e8=${c.contains('E8')}');
    if (_looksLikeInitOk(c)) {
      ecuId = _parseEcuId(c);
      await _toManual();
      _setState(SsmState.ecuReady);
      lastInitLog = 'SSM2 BF(auto) OK · ECU ${ecuId.isEmpty ? 'OK' : ecuId}';
      return ecuId.isEmpty ? 'OK' : ecuId;
    }
    // 2b. BF как ручной single-frame: PCI 01 + BF + паддинг
    await _toManual();
    r = await transact('01BF000000000000', timeoutMs: 2500);
    c = _clean(r);
    log.writeln('BF(manual 01BF): len=${c.length} e8=${c.contains('E8')}');
    if (_looksLikeInitOk(c)) {
      ecuId = _parseEcuId(c);
      _setState(SsmState.ecuReady);
      lastInitLog = 'SSM2 BF(manual) OK · ECU ${ecuId.isEmpty ? 'OK' : ecuId}';
      return ecuId.isEmpty ? 'OK' : ecuId;
    }
    // 2c. Legacy K-конверт (некоторые шлюзы его проксируют на CAN)
    await transact('ATCAF1', timeoutMs: 400);
    _cafManual = false;
    r = await transact('8010F001BF40', timeoutMs: 2500);
    c = _clean(r);
    log.writeln('BF(legacy 8010F001BF40): len=${c.length} e8=${c.contains('E8')}');
    await _toManual();
    if (_looksLikeInitOk(c)) {
      ecuId = _parseEcuId(c);
      _setState(SsmState.ecuReady);
      lastInitLog = 'SSM2 BF(legacy) OK · ECU ${ecuId.isEmpty ? 'OK' : ecuId}';
      return ecuId.isEmpty ? 'OK' : ecuId;
    }

    // ── Стадия 3 (РЕШАЮЩАЯ, как автоскан в V7): живой A8-probe ──
    // Читаем 1 байт ECT (0x000008). Если ECU отдал байт по E8 —
    // сессия жива, BF просто не поддерживается этим ECU по CAN.
    final probe = await readBytes(0x000008, 1);
    if (probe != null && probe.isNotEmpty) {
      final ectRaw = probe[0].toRadixString(16).padLeft(2, '0').toUpperCase();
      _setState(SsmState.ecuReady);
      lastInitLog = 'SSM2 A8-probe OK (ECT=0x$ectRaw) · BF пропущен как в V7';
      return ecuId.isEmpty ? 'OK' : ecuId;
    }

    lastInitLog = 'SSM2: BF и A8-probe без ответа:\n$log';
    return null;
  }

  bool _looksLikeInitOk(String c) {
    if (c.isEmpty) return false;
    if (c.contains('NODATA') || c.contains('UNABLE') || c.contains('CANERROR') ||
        c.contains('BUSINIT') || c.contains('STOPPED')) return false;
    if (c == 'ERROR' || c.contains('?')) return false;
    if (c.contains('E8') && c.length > 12) return true;
    if (c.contains('80F010')) return true;
    if (c.contains('FF') && c.length > 20) return true;
    final hexOnly = c.replaceAll(RegExp(r'[^0-9A-F]'), '');
    if (hexOnly.length > 40) return true; // длинный осмысленный ответ
    return false;
  }

  String _parseEcuId(String c) {
    try {
      final hex = c.replaceAll(RegExp(r'[^0-9A-F]'), '');
      final idx = hex.indexOf('FF');
      if (idx >= 0 && hex.length >= idx + 2 + 10) {
        final digits = <int>[];
        for (var i = idx + 2; i + 2 <= hex.length && digits.length < 10; i += 2) {
          final v = int.tryParse(hex.substring(i, i + 2), radix: 16) ?? -1;
          if (v >= 0x30 && v <= 0x39) digits.add(v);
        }
        if (digits.length >= 5) return ascii.decode(digits);
      }
    } catch (_) {}
    return '';
  }

  /// Чтение len байт с адреса addr — СТРОГО по спецификации SSM2:
  /// A8 00 + 3-байтный адрес НА КАЖДЫЙ байт (RomRaider/FreeSSM-style).
  /// Старый V8 слал A8 00 + addr + count — ECU такой формат игнорирует
  /// (в V7 это починила ячейка 24: 'A8 БЕЗ count').
  /// Single-frame CAN вмещает только 1 адрес (5 байт payload),
  /// поэтому блок читаем побайтово и склеиваем — интерфейс readBytes
  /// для поллера не меняется, блоки продолжают работать.
  Future<Uint8List?> readBytes(int addr, int len) async {
    if (len < 1) return null;
    if (len > 64) len = 64; // защита от 80-байтных монстров
    if (!_cafManual) await _toManual();
    final sw = Stopwatch()..start();
    final out = <int>[];
    for (var off = 0; off < len; off++) {
      final frame = _isoFrame(addr + off);
      List<int>? one;
      for (var a = 0; a <= readRetries; a++) {
        final resp = await transact(frame, timeoutMs: AppConstants.elmCmdTimeoutMs);
        one = _parseSingle(resp, addr + off);
        if (one != null && one.isNotEmpty) break;
        if (a < readRetries) {
          await Future<void>.delayed(const Duration(milliseconds: 60));
        }
      }
      if (one == null || one.isEmpty) {
        sw.stop();
        stats.err();
        return null;
      }
      out.add(one[0]);
    }
    sw.stop();
    stats.ok(sw.elapsedMicroseconds / 1000.0);
    return Uint8List.fromList(out);
  }

  /// ISO-TP single-frame для чтения ОДНОГО байта:
  /// PCI(05) + A8 00 + addr(3) + паддинг до 8 байт CAN.
  /// Пример ECT (0x000008): 05 A8 00 00 00 08 00 00 → '05A8000000080000'
  String _isoFrame(int addr) {
    final a = addr.toRadixString(16).padLeft(6, '0').toUpperCase();
    return '05A800${a}0000';
  }

  /// Парсер ответа в режиме ATCAF0+ATH1 (V7-style, ячейка 35):
  /// сырые кадры '7E8 06 E8 <data...>' → чистые data-байты после E8.
  /// В отличие от старого V8: ошибки проверяются ДО hex-чистки,
  /// а эхо адреса (если ECU его вернул) аккуратно снимается.
  List<int>? _parseSingle(String resp, int addr) {
    final up = resp.toUpperCase();
    // 1. Ошибки — ДО stripping (в старом V8 эта проверка была мёртвой:
    //    'NODATA' после удаления не-hex превращалось в 'DAA')
    for (final e in <String>[
      'NO DATA', 'NODATA', 'UNABLE', 'CAN ERROR', 'CANERROR',
      'BUSINIT', 'FB ERROR', 'DATA ERROR', 'ERROR', 'STOPPED', '?',
    ]) {
      if (up.contains(e)) { stats.nodata(); return null; }
    }
    var s = up
        .replaceAll('SEARCHING...', '')
        .replaceAll('BUSINIT:OK', '')
        .replaceAll('BUSINIT', '')
        .replaceAll('STOPPED', '')
        .replaceAll(' ', '')
        .replaceAll('\\r', '')
        .replaceAll('\\n', '')
        .replaceAll('>', '');
    // 2. Отрезаем всё до заголовка ответа ECU
    final hdr = s.indexOf('7E8');
    if (hdr >= 0) s = s.substring(hdr);
    // 3. Позитивный ответ SSM2
    final e8 = s.indexOf('E8');
    if (e8 < 0) return null;
    final hex = s.substring(e8 + 2).replaceAll(RegExp(r'[^0-9A-F]'), '');
    if (hex.length < 2) return null;
    final all = <int>[];
    for (var i = 0; i + 1 < hex.length; i += 2) {
      final v = int.tryParse(hex.substring(i, i + 2), radix: 16);
      if (v == null) break;
      all.add(v);
    }
    if (all.isEmpty) return null;
    // 4. Снимаем эхо адреса, если ECU его вернул:
    //    E8 00 <pp> <AA BB CC> <DATA...> vs E8 <DATA...> (V7-стиль).
    if (all.length >= 6 && all[0] == 0x00) {
      final hi = (addr >> 16) & 0xFF, mid = (addr >> 8) & 0xFF, lo = addr & 0xFF;
      final echo6 = all[3] == hi && all[4] == mid && all[5] == lo;
      final echo5 = all[2] == hi && all[3] == mid && all[4] == lo;
      if (echo6) return all.sublist(6);
      if (echo5) return all.sublist(5);
      // Эха нет, но первый байт 00 и хвост длинный (паддинг single-frame):
      // это чистые данные, где DATA[0]==0x00 — возвращаем как есть.
      return all;
    }
    return all;
  }

  Future<void> disconnect() async {
    try { await _sub?.cancel(); } catch (_) {}
    _sub = null;
    try { await _conn?.close(); } catch (_) {}
    _conn = null;
    _rx.clear();
    _setState(SsmState.disconnected);
  }
}

// ─────────────────────────────────────────────────────────────
// Поллер: блочный план + приоритетные ярусы + реконнект
// ─────────────────────────────────────────────────────────────
class SsmPoller {
  final SsmElm elm;
  int maxBlock = AppConstants.maxBlockBytes;
  int gapTol = 2;       // склеивать адреса, если разрыв <= gapTol байт
  int midEveryN = 4;    // средний ярус: каждый 4-й цикл
  int slowEveryN = 25;  // медленный ярус: каждый 25-й цикл

  List<SsmBlock> _blocks = [];
  final Map<String, double> _canon = {};
  final Map<String, double> _byId = {};

  Timer? _timer;
  bool _running = false;
  int _cycle = 0;
  int _globalErrStreak = 0;

  final StreamController<LiveSnapshot> _snapCtl = StreamController<LiveSnapshot>.broadcast();
  Stream<LiveSnapshot> get snapshots => _snapCtl.stream;
  LiveSnapshot? last;
  bool get isRunning => _running;
  int get blockCount => _blocks.length;
  int get pidCount => _blocks.fold(0, (a, b) => a + b.pids.length);

  SsmPoller(this.elm);

  /// Строим блоки: сортировка по адресу + склейка соседей.
  /// ЭТО главный ускоритель: вместо N запросов — ceil(N/плотность).
  List<SsmBlock> buildBlocks(List<SubaruPid> selected, {bool log = false}) {
    final list = selected.where((p) => p.len > 0).toList()
      ..sort((a, b) => a.address.compareTo(b.address));
    final blocks = <SsmBlock>[];
    var cur = <SubaruPid>[];
    var start = 0, end = 0, prio = 3;
    void flush() {
      if (cur.isEmpty) return;
      blocks.add(SsmBlock(start, end - start, List.of(cur), prio));
      cur = [];
    }
    for (final p in list) {
      final pStart = p.address, pEnd = p.address + p.len;
      if (cur.isEmpty) {
        start = pStart; end = pEnd; prio = p.priority; cur.add(p); continue;
      }
      if (pStart <= end + gapTol && (pEnd - start) <= maxBlock) {
        end = pEnd > end ? pEnd : end;
        if (p.priority < prio) prio = p.priority;
        cur.add(p);
      } else {
        flush();
        start = pStart; end = pEnd; prio = p.priority; cur.add(p);
      }
    }
    flush();
    _blocks = blocks;
    return blocks;
  }

  void start() {
    if (_running || _blocks.isEmpty) return;
    _running = true;
    _globalErrStreak = 0;
    elm.markPolling(true);
    _tick();
  }

  void stop() {
    _running = false;
    _timer?.cancel();
    if (elm.state == SsmState.polling) elm.markPolling(false);
  }

  Iterable<SsmBlock> _cycleBlocks(int cycle) sync* {
    for (final b in _blocks) {
      final due = b.prio == 1 || (b.prio == 2 && cycle % midEveryN == 0) || (b.prio == 3 && cycle % slowEveryN == 0);
      if (due && cycle >= b.skipUntilCycle) yield b;
    }
  }

  Future<void> _tick() async {
    if (!_running) return;
    _cycle++;
    for (final b in _cycleBlocks(_cycle)) {
      if (!_running) return;
      Uint8List? data;
      try {
        data = await elm.readBytes(b.start, b.len);
      } catch (_) {
        data = null; // отвал BT/адаптера — считаем ошибочным кадром
      }
      if (data != null) {
        b.consecErr = 0;
        _globalErrStreak = 0;
        for (final p in b.pids) {
          final off = p.address - b.start;
          if (off < 0 || off + p.len > data.length) continue;
          final sub = Uint8List.fromList(data.sublist(off, off + p.len));
          try {
            final v = p.formula(sub);
            if (v.isNaN || v.isInfinite) continue;
            _byId[p.id] = v;
            if (p.canon.isNotEmpty) _canon[p.canon] = v;
          } catch (_) {}
        }
      } else {
        b.consecErr++;
        _globalErrStreak++;
        if (b.consecErr >= 4) {
          b.skipUntilCycle = _cycle + 60; // блок «молчит» — отложим, не будем стоять в очереди
          b.consecErr = 0;
        }
      }
    }
    // снапшот после цикла быстрых блоков
    last = LiveSnapshot(DateTime.now(), Map.of(_canon), Map.of(_byId));
    elm.stats.snap();
    if (!_snapCtl.isClosed) _snapCtl.add(last!);

    // самолечение: много ошибок подряд -> re-init ECU
    if (_globalErrStreak > 40) {
      _globalErrStreak = 0;
      try {
        final id = await elm.ecuInit();
        if (id == null) { stop(); return; }
      } catch (_) {
        stop();
        return;
      }
    }
    // сразу следующий цикл — без искусственных задержек.
    // Темп ограничен реальным временем ответа ECU, а не sleep().
    _timer = Timer(Duration.zero, _tick);
  }

  Future<void> dispose() async {
    stop();
    await _snapCtl.close();
  }
}
''')
print("OK  lib/ssm/ssm_elm.dart — протокол как в рабочем V7")

# ---------- 2. Терминал: правильные примеры кадров ----------
tp = 'lib/screens/terminal_screen.dart'
if os.path.exists(tp):
    t = open(tp, encoding='utf-8').read()
    old_quick = """  static const _quick = [
    'ATRV',
    '8010F001BF40',
    'A8000000080E',
    'A8000000F900',
    'A8 00 00 01 99 00',
  ];"""
    new_quick = """  static const _quick = [
    'ATRV',
    '0100',
    'BF',
    '01BF000000000000',
    '05A8000000080000',
    '05A80000000E0000',
  ];"""
    if old_quick in t:
        open(tp, 'w', encoding='utf-8').write(t.replace(old_quick, new_quick, 1))
        print("OK  terminal_screen.dart — примеры ручных ISO-TP кадров")
    else:
        print("  · terminal_screen.dart: примеры уже обновлены или файл менялся вручную")

# ---------- 3. Настройки: ST минимум 0x20 (страховка) ----------
sp = 'lib/services/settings_service.dart'
if os.path.exists(sp):
    s = open(sp, encoding='utf-8').read()
    s2 = s.replace('int stCode = 8;', 'int stCode = 0x20;')
    s2 = s2.replace("p.getInt('stCode') ?? 8", "p.getInt('stCode') ?? 0x20")
    if s2 != s:
        open(sp, 'w', encoding='utf-8').write(s2)
        print("OK  settings_service.dart — дефолт ST 0x20")
    else:
        print("  · settings_service.dart: ST уже >= 0x20")

# ---------- 4. Быстрая проверка синтаксиса ----------
import subprocess
r = subprocess.run(['/content/flutter/bin/flutter', 'analyze', '--no-fatal-infos', 'lib/ssm/ssm_elm.dart'],
                   capture_output=True, text=True)
print("--- flutter analyze lib/ssm/ssm_elm.dart ---")
print((r.stdout + r.stderr)[-1500:])




  ФИКС V8: протокол SSM2-over-CAN как в рабочем V7
  бэкап: lib/ssm/ssm_elm.dart.bak_080210
OK  lib/ssm/ssm_elm.dart — протокол как в рабочем V7
OK  terminal_screen.dart — примеры ручных ISO-TP кадров
  · settings_service.dart: ST уже >= 0x20
--- flutter analyze lib/ssm/ssm_elm.dart ---
_foundation 2.5.7
+ shared_preferences_linux 2.4.1
+ shared_preferences_platform_interface 2.4.2
+ shared_preferences_web 2.4.3
+ shared_preferences_windows 2.4.1
  test_api 0.7.12 (0.7.14 available)
+ typed_data 1.4.0
+ url_launcher_linux 3.2.3
+ url_launcher_platform_interface 2.3.2
+ url_launcher_web 2.4.3
+ url_launcher_windows 3.1.6
+ uuid 4.6.0
  vector_math 2.4.0 (2.4.2 available)
+ web 1.1.1
+ win32 5.15.0 (6.4.0 available)
+ xdg_directories 1.1.0
+ xml 6.6.1 (7.0.1 available)
+ yaml 3.1.4
Changed 63 dependencies!
23 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.
Analyzing ssm_elm.dart...                                    

In [ ]:
# @title SUBA RUN V8: CAN-V3 (fixed CAN 6, raw ISO-TP, selected PID diagnostics)
# Run LAST among fixes, before the existing build cell 10/10.
# Existing BT-PERM-V2 is required. This cell NEVER builds/flashes an ECU ROM.
from pathlib import Path
from datetime import datetime, timezone
import re
import subprocess

ROOT = Path('/content/suba_run_v8')
FLUTTER = Path('/content/flutter/bin/flutter')
CODEC = r'''import 'dart:typed_data';

class CanV3Exception implements Exception {
  final String code;
  final String message;
  const CanV3Exception(this.code, this.message);
  @override
  String toString() => '$code: $message';
}

class CanV3Flow {
  final int blockSize;
  final Duration separation;
  const CanV3Flow(this.blockSize, this.separation);
}

class CanV3Codec {
  static String hex(List<int> bytes) => bytes
      .map((b) => b.toRadixString(16).padLeft(2, '0').toUpperCase()).join();

  static void validateBytes(List<int> bytes) {
    if (bytes.any((b) => b < 0 || b > 255)) {
      throw const CanV3Exception('BAD_BYTE', 'Byte outside 0..255.');
    }
  }

  static List<int> addressRead(int address, int count) {
    if (address < 0 || address > 0xFFFFFF || count < 1 || count > 128 || address + count - 1 > 0xFFFFFF) {
      throw const CanV3Exception('BAD_RANGE', 'A8 requires a 24-bit address and 1..128 bytes.');
    }
    final payload = <int>[0xA8, 0x00];
    for (var offset = 0; offset < count; offset++) {
      final a = address + offset;
      payload.addAll([(a >> 16) & 255, (a >> 8) & 255, a & 255]);
    }
    return payload;
  }

  static List<int> _pad(List<int> bytes) => [...bytes, ...List.filled(8 - bytes.length, 0)];

  static List<List<int>> encode(List<int> payload) {
    validateBytes(payload);
    if (payload.isEmpty || payload.length > 4095) {
      throw const CanV3Exception('BAD_LENGTH', 'Classical CAN ISO-TP length must be 1..4095.');
    }
    if (payload.length <= 7) return [_pad([payload.length, ...payload])];
    final frames = <List<int>>[
      [0x10 | (payload.length >> 8), payload.length & 255, ...payload.take(6)],
    ];
    var offset = 6;
    var sequence = 1;
    while (offset < payload.length) {
      final end = (offset + 7 < payload.length) ? offset + 7 : payload.length;
      frames.add(_pad([0x20 | sequence, ...payload.sublist(offset, end)]));
      sequence = (sequence + 1) & 15;
      offset = end;
    }
    return frames;
  }

  static String? elmError(String reply) {
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.replaceAll(RegExp(r'\s+'), '');
      if (line.contains('NODATA')) return 'NO_DATA';
      if (line.contains('BUFFERFULL')) return 'BUFFER_FULL';
      if (line.contains('CANERROR') || line.contains('BUSERROR')) return 'CAN_ERROR';
      if (line.contains('UNABLETOCONNECT')) return 'UNABLE_TO_CONNECT';
      if (line.contains('STOPPED')) return 'STOPPED';
      if (line.contains('ERROR') || line == '?') return 'ELM_ERROR';
    }
    return null;
  }

  // Parse each physical line. Never search for E8 inside a header or payload.
  static List<List<int>> frames(String reply, {String request = '', int rxId = 0x7E8}) {
    final error = elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    final result = <List<int>>[];
    final echo = request.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final expected = rxId.toRadixString(16).padLeft(3, '0').toUpperCase();
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.trim();
      if (line.startsWith('SEARCHING...')) line = line.substring(12).trim();
      final compact = line.replaceAll(RegExp(r'\s+'), '');
      if (compact.isEmpty || compact == echo || compact == 'OK') continue;
      if (!compact.startsWith(expected)) {
        // Other 11-bit responders are filtered, not joined into this ECU response.
        if (RegExp(r'^[0-9A-F]{3}:?[0-9A-F]+$').hasMatch(compact) &&
            int.parse(compact.substring(0, 3), radix: 16) <= 0x7FF) continue;
        throw CanV3Exception('MALFORMED_LINE', 'Unexpected ELM line: $line');
      }
      var body = compact.substring(3);
      if (body.startsWith(':')) body = body.substring(1);
      int? dlc;
      if (body.length.isOdd && RegExp(r'^[0-8]').hasMatch(body)) {
        dlc = int.parse(body[0], radix: 16);
        body = body.substring(1);
      }
      if (body.isEmpty || body.length.isOdd || body.length > 16 || !RegExp(r'^[0-9A-F]+$').hasMatch(body)) {
        throw CanV3Exception('MALFORMED_FRAME', 'Invalid raw CAN frame: $line');
      }
      final bytes = <int>[
        for (var i = 0; i < body.length; i += 2) int.parse(body.substring(i, i + 2), radix: 16),
      ];
      if (dlc != null && bytes.length != dlc) {
        throw const CanV3Exception('DLC_MISMATCH', 'DLC does not match CAN data length.');
      }
      result.add(bytes);
    }
    if (result.isEmpty) throw const CanV3Exception('NO_FRAMES', 'No complete 7E8 raw CAN frames.');
    return result;
  }

  static List<Uint8List> assemble(List<List<int>> frames) {
    final messages = <Uint8List>[];
    List<int>? pending;
    var length = 0;
    var sequence = 1;
    for (final frame in frames) {
      validateBytes(frame);
      if (frame.isEmpty || frame.length > 8) throw const CanV3Exception('BAD_FRAME', 'CAN frame length.');
      final type = frame[0] >> 4;
      if (type == 0) {
        if (pending != null) throw const CanV3Exception('INTERRUPTED', 'SF interrupted a multi-frame response.');
        final n = frame[0] & 15;
        if (n < 1 || n > 7 || n > frame.length - 1) throw const CanV3Exception('SF_LENGTH', 'Invalid SF length.');
        messages.add(Uint8List.fromList(frame.sublist(1, n + 1)));
      } else if (type == 1) {
        if (pending != null || frame.length != 8) throw const CanV3Exception('BAD_FF', 'Unexpected or incomplete FF.');
        length = ((frame[0] & 15) << 8) | frame[1];
        if (length < 8 || length > 4095) throw const CanV3Exception('FF_LENGTH', 'Invalid FF length.');
        pending = frame.sublist(2);
        sequence = 1;
      } else if (type == 2) {
        if (pending == null || frame.length < 2) throw const CanV3Exception('ORPHAN_CF', 'CF without FF.');
        if ((frame[0] & 15) != sequence) throw const CanV3Exception('SEQUENCE', 'Wrong, duplicate or missing CF sequence.');
        sequence = (sequence + 1) & 15;
        final needed = length - pending.length;
        final available = frame.length - 1;
        if (available < needed && frame.length != 8) throw const CanV3Exception('SHORT_CF', 'Truncated CF.');
        pending.addAll(frame.skip(1).take(needed));
        if (pending.length == length) {
          messages.add(Uint8List.fromList(pending));
          pending = null;
        }
      } else {
        throw CanV3Exception('UNEXPECTED_PCI', 'Expected SF/FF/CF, got PCI ${hex([frame[0]])}.');
      }
    }
    if (pending != null) throw const CanV3Exception('INCOMPLETE', 'Response ended before declared ISO-TP length.');
    if (messages.isEmpty) throw const CanV3Exception('NO_PDU', 'No ISO-TP message.');
    return messages;
  }

  static CanV3Flow flow(List<List<int>> frames) {
    CanV3Flow? accepted;
    for (final bytes in frames) {
      if (bytes.length < 3 || bytes[0] >> 4 != 3) throw const CanV3Exception('EXPECTED_FC', 'No valid flow control.');
      final status = bytes[0] & 15;
      if (status == 1) continue;
      if (status == 2) throw const CanV3Exception('FC_OVERFLOW', 'ECU cannot accept the request.');
      if (status != 0 || accepted != null) throw const CanV3Exception('BAD_FC', 'Invalid or repeated CTS.');
      final st = bytes[2];
      if (st > 0x7F && (st < 0xF1 || st > 0xF9)) throw const CanV3Exception('STMIN', 'Reserved STmin value.');
      // A millisecond is a safe lower bound for sub-millisecond STmin on ELM serial.
      accepted = CanV3Flow(bytes[1], Duration(milliseconds: st <= 0x7F ? st : 1));
    }
    if (accepted == null) throw const CanV3Exception('FC_WAIT', 'WAIT received without CTS before the ELM prompt.');
    return accepted;
  }

  static Uint8List positive(List<Uint8List> messages, int requestSid, List<int> prefix, int length) {
    Uint8List? answer;
    CanV3Exception? negative;
    for (final bytes in messages) {
      if (bytes.length >= 2 && bytes[0] == 0x7F && bytes[1] == requestSid) {
        if (bytes.length != 3) throw const CanV3Exception('NEGATIVE_LENGTH', 'Invalid negative response length.');
        negative = CanV3Exception(bytes[2] == 0x78 ? 'PENDING' : 'NEGATIVE_RESPONSE',
            'SID ${hex([requestSid])}, NRC ${hex([bytes[2]])}.');
        if (bytes[2] != 0x78 || answer != null) throw negative;
        continue;
      }
      if (bytes.length < prefix.length || !List.generate(prefix.length, (i) => bytes[i] == prefix[i]).every((v) => v)) {
        throw CanV3Exception('WRONG_SID', 'Unexpected PDU ${hex(bytes)}.');
      }
      if (bytes.length != length) throw CanV3Exception('RESPONSE_LENGTH', 'Expected $length bytes; got ${bytes.length}.');
      if (answer != null) throw const CanV3Exception('AMBIGUOUS', 'Multiple positive responses for one request.');
      answer = bytes;
    }
    if (answer != null) return answer;
    throw negative ?? const CanV3Exception('NO_POSITIVE_RESPONSE', 'No matching positive response.');
  }
}'''
SESSION = r'''import 'dart:async';
import 'dart:typed_data';
import 'can_v3_codec.dart';

typedef CanV3Send = Future<String> Function(String command, int timeoutMs);

class CanV3Probe {
  final String id;
  final int address;
  final int length;
  const CanV3Probe(this.id, this.address, this.length);
}

class CanV3Session {
  final CanV3Send send;
  final Future<void> Function(Duration) delay;
  Future<void> _tail = Future<void>.value();
  final List<String> _trace = [];
  List<String> _initTrace = [];
  int _generation = 0;
  bool _responses = true;
  bool _poisoned = false;
  bool obdReady = false;
  bool ssmReady = false;
  String protocolNumber = '';
  String lastError = '';
  String lastProbe = '';
  int timeoutCode = 0x32;
  int txFrames = 0;
  int readOk = 0;
  int readErrors = 0;
  int noData = 0;
  int _readMilliseconds = 0;
  double get averageReadMs => readOk == 0 ? 0 : _readMilliseconds / readOk;

  CanV3Session(this.send, {Future<void> Function(Duration)? delay})
      : delay = delay ?? ((d) => Future<void>.delayed(d));

  String get diagnostic => [
    'CAN-V3 | ISO 15765-4 / 11-bit / 500 kbit/s',
    'Required configuration: protocol 6, TX=7E0, RX=7E8, CAF=0, CFC=1',
    'Last ATDPN reply in current connection: $protocolNumber',
    'OBD=$obdReady; SSM2=$ssmReady; last probe=$lastProbe',
    'ECU hardware ID: not read (no synthetic ID, no shared PID cache)',
    'AT ST=0x${timeoutCode.toRadixString(16).toUpperCase()}; nominal ${timeoutCode * 4} ms',
    'TX CAN frames=$txFrames; complete reads=$readOk; read errors=$readErrors; NO DATA=$noData',
    'Average complete A8 read: ${averageReadMs.toStringAsFixed(1)} ms (not time per physical frame)',
    'last error=$lastError',
    '--- INIT TRACE ---', ..._initTrace,
    '--- RECENT TRACE ---', ..._trace,
  ].join('\n');

  void record(String text) {
    _trace.add('${DateTime.now().toIso8601String()} $text');
    if (_trace.length > 240) _trace.removeRange(0, _trace.length - 240);
  }

  void invalidate(String reason) {
    _generation++;
    obdReady = false;
    ssmReady = false;
    protocolNumber = '';
    _poisoned = true;
    record('INVALIDATE $reason');
  }

  Future<T> _serial<T>(Future<T> Function() task) {
    final run = _tail.then((_) => task());
    _tail = run.then<void>((_) {}, onError: (Object _, StackTrace __) {});
    return run;
  }

  Future<void> whenIdle() => _tail;

  Future<String> setupCommand(String command, int timeoutMs) =>
      _serial(() => _wire(command, timeoutMs: timeoutMs));

  Future<String> _wire(String command, {int timeoutMs = 1500, bool responseRequired = true}) async {
    final generation = _generation;
    record('TX $command');
    try {
      final response = await send(command, timeoutMs).timeout(Duration(milliseconds: timeoutMs + 500));
      record('RX ${response.replaceAll('\r', '<CR>').replaceAll('\n', '<LF>')}');
      if (generation != _generation) throw const CanV3Exception('CANCELLED', 'Connection changed during request.');
      if (responseRequired && response.replaceAll('>', '').trim().isEmpty) {
        _poisoned = true;
        throw const CanV3Exception('EMPTY_REPLY', 'No response; reconnect before continuing.');
      }
      return response;
    } on TimeoutException {
      _poisoned = true;
      ssmReady = false;
      throw const CanV3Exception('WIRE_TIMEOUT', 'ELM prompt/command timeout. Reconnect to avoid a late reply.');
    }
  }

  Future<void> _ok(String command) async {
    final response = await _wire(command);
    final error = CanV3Codec.elmError(response);
    final lines = response.toUpperCase().replaceAll('>', '').split(RegExp(r'[\r\n]+'));
    if (error != null || !lines.any((line) => line.trim() == 'OK')) {
      throw CanV3Exception('AT_REJECTED', '$command -> ${response.trim()}');
    }
  }

  Future<void> _setResponses(bool enabled) async {
    if (_responses == enabled) return;
    await _ok(enabled ? 'ATR1' : 'ATR0');
    _responses = enabled;
  }

  Future<String> _frame(List<int> bytes, {required bool expectResponse}) async {
    txFrames++;
    final reply = await _wire(CanV3Codec.hex(bytes), timeoutMs: 3000, responseRequired: expectResponse);
    final error = CanV3Codec.elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    return reply;
  }

  Future<List<Uint8List>> _exchange(List<int> payload) async {
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Transport is not synchronized.');
    final frames = CanV3Codec.encode(payload);
    try {
      await _setResponses(true);
      var reply = await _frame(frames.first, expectResponse: true);
      if (frames.length == 1) {
        return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      }
      var flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      var sentInBlock = 0;
      for (var i = 1; i < frames.length; i++) {
        final last = i == frames.length - 1;
        final boundary = flow.blockSize > 0 && sentInBlock + 1 == flow.blockSize;
        // Suppress response waits only for CFs that cannot require FC or a final reply.
        await _setResponses(last || boundary);
        await delay(flow.separation);
        reply = await _frame(frames[i], expectResponse: last || boundary);
        sentInBlock++;
        if (last) return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
        if (boundary) {
          flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
          sentInBlock = 0;
        }
      }
      throw const CanV3Exception('TX_INCOMPLETE', 'Not all CFs sent.');
    } catch (e) {
      if (frames.length > 1 || e is! CanV3Exception || e.code != 'NO_DATA') {
        _poisoned = true;
        ssmReady = false;
        record('ISO-TP aborted; reconnect required.');
      }
      rethrow;
    } finally {
      if (!_responses && !_poisoned) {
        try { await _setResponses(true); }
        catch (e) { _poisoned = true; ssmReady = false; record('RESTORE ATR1 FAILED $e'); }
      }
    }
  }

  Future<Uint8List> _read(int address, int length) async {
    final reply = await _exchange(CanV3Codec.addressRead(address, length));
    try {
      final pdu = CanV3Codec.positive(reply, 0xA8, [0xE8], length + 1);
      return Uint8List.fromList(pdu.sublist(1));
    } on CanV3Exception catch (e) {
      if (e.code != 'NEGATIVE_RESPONSE') {
        _poisoned = true;
        ssmReady = false;
      }
      rethrow;
    }
  }

  Future<bool> initialize(List<CanV3Probe> probes, {int stCode = 0x32}) => _serial(() async {
    _generation++;
    _poisoned = false;
    obdReady = false;
    ssmReady = false;
    lastError = '';
    protocolNumber = '';
    lastProbe = '';
    timeoutCode = stCode.clamp(0x32, 0xFF).toInt();
    _trace.clear();
    txFrames = 0; readOk = 0; readErrors = 0; noData = 0; _readMilliseconds = 0;
    try {
      for (final command in [
        'ATE0', 'ATL0', 'ATS0', 'ATSP6', 'ATH1', 'ATD0', 'ATCAF0',
        'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
        'ATR1', 'ATST${timeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase()}',
      ]) { await _ok(command); }
      _responses = true;
      // Same on-wire 01 00 as CAF1 + 0100, without changing CAF mode mid-session.
      final obd = await _exchange([0x01, 0x00]);
      CanV3Codec.positive(obd, 0x01, [0x41, 0x00], 6);
      final number = await _wire('ATDPN');
      protocolNumber = number.replaceAll(RegExp(r'[\s>]'), '').toUpperCase();
      if (protocolNumber != '6') {
        throw CanV3Exception('WRONG_PROTOCOL', 'Expected fixed protocol 6, got $protocolNumber (A6 is auto).');
      }
      obdReady = true;
      record('OBD CAN confirmed; SSM2 is not confirmed yet.');
      for (final probe in probes.take(4)) {
        try {
          await _read(probe.address, probe.length);
          lastProbe = probe.id;
          ssmReady = true;
          record('SSM2 confirmed: ${probe.id}, ${probe.length} bytes, exact E8 response.');
          return true;
        } catch (e) {
          record('PROBE ${probe.id} FAILED $e');
          if (_poisoned) rethrow;
        }
      }
      throw CanV3Exception('SSM_NOT_CONFIRMED', probes.isEmpty
          ? 'Select at least one PID before connecting.'
          : 'OBD CAN works, but selected A8 probes did not return valid E8 data.');
    } catch (e) {
      ssmReady = false;
      lastError = e.toString();
      record('INIT FAILED $lastError');
      return false;
    } finally {
      _initTrace = List.of(_trace);
    }
  });

  Future<Uint8List?> readBytes(int address, int length) => _serial(() async {
    lastError = '';
    final clock = Stopwatch()..start();
    try {
      if (!ssmReady || _poisoned) throw const CanV3Exception('NOT_READY', 'Connect and confirm CAN/SSM2 first.');
      final result = await _read(address, length);
      readOk++;
      _readMilliseconds += clock.elapsedMilliseconds;
      return result;
    } catch (e) {
      readErrors++;
      if (e is CanV3Exception && e.code == 'NO_DATA') noData++;
      lastError = e.toString();
      record('READ FAILED $lastError');
      return null; // No partial bytes, padding or fabricated zeroes.
    }
  });

  Future<String> terminalQuery(String command, int timeoutMs) => _serial(() async {
    final normalized = command.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (!const ['ATI', 'ATDP', 'ATDPN', 'ATRV', 'AT@1'].contains(normalized)) {
      throw const CanV3Exception('MANUAL_TX_DISABLED', 'Use PID CAN for reads; arbitrary terminal TX is blocked.');
    }
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Reconnect before sending terminal queries.');
    return _wire(normalized, timeoutMs: timeoutMs);
  });
}'''
TESTS = r'''import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import '../lib/ssm/can_v3_codec.dart';
import '../lib/ssm/can_v3_session.dart';
import '../lib/ssm/ssm_elm.dart';
import '../lib/main.dart' as application;

String raw(List<int> pdu) => CanV3Codec.encode(pdu)
    .map((f) => '7E8${CanV3Codec.hex(f)}\r').join() + '>';

class FakeElm {
  final commands = <String>[];
  final request = <List<int>>[];
  int required = 0;
  bool responses = true;
  String dpn = '6';
  bool rejectSsm = false;
  int blockSize = 0;
  int inBlock = 0;

  Future<String> call(String command, int timeout) async {
    commands.add(command);
    if (command == 'ATDPN') return '$dpn\r>';
    if (command == 'ATR0') responses = false;
    if (command == 'ATR1') responses = true;
    if (command.startsWith('AT')) return 'OK\r>';
    final bytes = <int>[
      for (var i = 0; i < command.length; i += 2) int.parse(command.substring(i, i + 2), radix: 16),
    ];
    if (bytes[0] >> 4 == 0) return respond(CanV3Codec.assemble([bytes]).single);
    if (bytes[0] >> 4 == 1) {
      request.clear();
      request.add(bytes);
      required = ((bytes[0] & 15) << 8) | bytes[1];
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    request.add(bytes);
    inBlock++;
    final received = 6 + (request.length - 1) * 7;
    if (received >= required) {
      if (!responses) throw StateError('Final CF sent with responses disabled.');
      return respond(CanV3Codec.assemble(request).single);
    }
    if (blockSize > 0 && inBlock == blockSize) {
      if (!responses) throw StateError('FC boundary sent with responses disabled.');
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    if (responses) throw StateError('Intermediate CF unexpectedly waits for a response.');
    return '>';
  }

  String respond(List<int> payload) {
    if (payload.length == 2 && payload[0] == 1 && payload[1] == 0) return raw([0x41, 0, 0xBE, 0x3F, 0xA8, 0x13]);
    if (payload[0] != 0xA8 || payload[1] != 0 || (payload.length - 2) % 3 != 0) {
      throw StateError('Incorrect SSM payload: ${CanV3Codec.hex(payload)}');
    }
    if (rejectSsm) return raw([0x7F, 0xA8, 0x31]);
    return raw([0xE8, for (var i = 4; i < payload.length; i += 3) payload[i]]);
  }
}

void main() {
  test('Application API links against the patched SsmElm', () {
    // Referencing main forces compile-time checking of all imported screens/services.
    expect(application.main, isNotNull);
    final elm = SsmElm();
    expect(elm.canV3Ready, isFalse);
    expect(elm.canV3Diagnostic, contains('CAN-V3'));
  });

  test('Exact SF and FF/CF for one and two A8 addresses', () {
    expect(CanV3Codec.hex(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 1)).single), '05A80000000E0000');
    expect(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 2)).map(CanV3Codec.hex).toList(),
        ['1008A80000000E00', '21000F0000000000']);
    expect(CanV3Codec.hex(CanV3Codec.encode([1, 0]).single), '0201000000000000');
  });

  test('Lengths 1..4095, padding removed, CF sequence wraps through zero', () {
    for (final n in [1, 7, 8, 20, 80, 128, 242, 386, 4095]) {
      final data = List.generate(n, (i) => i & 255);
      final encoded = CanV3Codec.encode(data);
      expect(encoded.every((f) => f.length == 8), isTrue);
      expect(CanV3Codec.assemble(encoded).single, data);
    }
    expect(() => CanV3Codec.encode([]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.addressRead(0xFFFFFF, 2), throwsA(isA<CanV3Exception>()));
  });

  test('7E8 is a header, not a response SID; foreign frames and TX echo are not data', () {
    final reply = '05A80000000E0000\r7E9 02 E8 88 00 00 00 00 00\r7E8 02 E8 12 00 00 00 00 00\r>';
    final frames = CanV3Codec.frames(reply, request: '05A80000000E0000');
    expect(CanV3Codec.positive(CanV3Codec.assemble(frames), 0xA8, [0xE8], 2), [0xE8, 0x12]);
    expect(() => CanV3Codec.positive(CanV3Codec.assemble(CanV3Codec.frames('7E8 02 62 12 00 00 00 00 00\r>')), 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31])], 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>().having((e) => e.code, 'code', 'NEGATIVE_RESPONSE')));
    expect(CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x78]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), [0xE8, 1]);
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), throwsA(isA<CanV3Exception>()));
  });

  test('Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail', () {
    expect(() => CanV3Codec.frames('NO DATA\r>'), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([[6, 0xE8, 1]]), throwsA(isA<CanV3Exception>()));
    final full = CanV3Codec.encode(List.generate(25, (i) => i));
    expect(() => CanV3Codec.assemble(full.take(2).toList()), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[2]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[1], full[1]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x32, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x31, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(CanV3Codec.flow([[0x30, 3, 0xF5]]).separation.inMicroseconds, greaterThanOrEqualTo(500));
  });

  test('CAN-only initialization does not invent an ECU ID or confirm SSM from 4100', () async {
    final fake = FakeElm()..rejectSsm = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(fake.commands.contains('ATSP6'), isTrue);
    expect(fake.commands.any((c) => c.startsWith('8010') || c == 'ATSP0' || c == 'ATSP3' || c == 'ATSP5'), isFalse);
    final auto = CanV3Session((FakeElm()..dpn = 'A6').call, delay: (_) async {});
    expect(await auto.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(auto.lastError, contains('WRONG_PROTOCOL'));
  });

  test('Required AT commands are verified, timeout code is hexadecimal', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)], stCode: 8), isTrue);
    expect(fake.commands.contains('ATST32'), isTrue);
    expect(fake.commands.contains('ATST08'), isFalse);
    final bad = CanV3Session((command, timeout) async {
      if (command == 'ATFCSM1') return '?\r>';
      return await fake.call(command, timeout);
    }, delay: (_) async {});
    expect(await bad.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(bad.lastError, contains('ATFCSM1'));
    expect(bad.ssmReady, isFalse);
  });

  test('MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction', () async {
    for (final blockSize in [0, 1, 4]) {
      final fake = FakeElm()..blockSize = blockSize;
      final session = CanV3Session(fake.call, delay: (_) async {});
      expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
      expect(await session.readBytes(0xE, 4), [14, 15, 16, 17]);
      expect(await session.readBytes(0x100, 80), List.generate(80, (i) => i));
      expect(fake.responses, isTrue);
    }
  });

  test('Concurrent reads are serialized; no data is returned after disconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    final results = await Future.wait([session.readBytes(10, 4), session.readBytes(20, 4)]);
    expect(results, [[10, 11, 12, 13], [20, 21, 22, 23]]);
    session.invalidate('test disconnect');
    expect(await session.readBytes(10, 4), isNull);
  });
}'''
PID_SCREEN = r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';

class CanV3PidScreen extends StatefulWidget {
  const CanV3PidScreen({super.key});
  @override
  State<CanV3PidScreen> createState() => _CanV3PidScreenState();
}

class _CanV3PidScreenState extends State<CanV3PidScreen> {
  final List<_CanPidResult> _results = [];
  bool _running = false;
  bool _cancel = false;
  int _total = 0;
  String _message = '';

  @override
  void dispose() {
    _cancel = true;
    super.dispose();
  }

  Future<void> _scan() async {
    final svc = ConnectionService.I;
    if (_running) return;
    if (!svc.elm.canV3Ready) {
      setState(() => _message = 'Сначала подтвердите CAN и SSM2 через CONNECT + INIT ECU.');
      return;
    }
    final pids = SettingsService.I.selectedPids.toList();
    if (pids.isEmpty) { setState(() => _message = 'Нет выбранных PID.'); return; }
    setState(() { _running = true; _cancel = false; _results.clear(); _total = pids.length; _message = ''; });
    try {
      await svc.exclusive((elm) async {
        await elm.canV3Idle();
        for (final pid in pids) {
          if (_cancel || !mounted || !elm.canV3Ready) break;
          final clock = Stopwatch()..start();
          final bytes = await elm.readBytes(pid.address, pid.len);
          double? value;
          var status = 'READ_ERROR';
          var detail = elm.canV3LastError;
          if (bytes != null && bytes.length == pid.len) {
            try {
              final decoded = pid.formula(bytes);
              if (!decoded.isFinite) {
                status = 'FORMULA_ERROR'; detail = 'Формула вернула NaN/Infinity.';
              } else {
                value = decoded; status = 'OK'; detail = 'E8 и длина проверены; физическая достоверность не подтверждена.';
              }
            } catch (e) { status = 'FORMULA_ERROR'; detail = e.toString(); }
          }
          clock.stop();
          final result = _CanPidResult(
            pid.id, pid.address, pid.len, status, value,
            bytes?.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ').toUpperCase() ?? '',
            clock.elapsedMilliseconds, detail,
          );
          if (!mounted) break;
          setState(() => _results.add(result));
        }
      });
      if (mounted) setState(() => _message = _cancel ? 'Проверка остановлена после текущего запроса.' : 'Проверка завершена. Кэш рабочих PID не изменён.');
    } catch (e) {
      if (mounted) setState(() => _message = 'Ошибка диагностики: $e');
    } finally {
      if (mounted) setState(() => _running = false);
    }
  }

  Future<void> _copy() async {
    final report = [
      'CAN-V3 selected PID diagnostics (not a calibration validation)',
      ..._results.map((r) => '${r.id} address=0x${r.address.toRadixString(16)} length=${r.length} '
          'status=${r.status} value=${r.value} ms=${r.ms} bytes=${r.bytes} detail=${r.detail}'),
      ConnectionService.I.elm.canV3Diagnostic,
    ].join('\n');
    try {
      await Clipboard.setData(ClipboardData(text: report));
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Отчёт CAN скопирован')));
    } catch (e) { if (mounted) setState(() => _message = 'Копирование не удалось: $e'); }
  }

  @override
  Widget build(BuildContext context) => Scaffold(
    appBar: AppBar(title: const Text('CAN-V3: выбранные PID'), actions: [
      IconButton(tooltip: 'Копировать отчёт', onPressed: _copy, icon: const Icon(Icons.copy)),
    ]),
    body: Column(children: [
      Padding(padding: const EdgeInsets.all(12), child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
        const Text('Проверяется текущий набор PID. Значение не считается физически достоверным только из-за ответа E8. '
            'Опрос приостанавливается на время проверки. Кэш по общему имени Subaru-CAN не создаётся.', style: TextStyle(fontSize: 12)),
        const SizedBox(height: 8),
        Text('${_results.length} / $_total; OK: ${_results.where((r) => r.status == 'OK').length}'),
        if (_running) LinearProgressIndicator(value: _total == 0 ? null : _results.length / _total),
        const SizedBox(height: 8),
        FilledButton(onPressed: _running ? null : _scan, child: const Text('Проверить выбранные PID')),
        if (_running) TextButton(onPressed: () => setState(() => _cancel = true),
            child: Text(_cancel ? 'Завершаем текущий запрос...' : 'Остановить')),
        if (_message.isNotEmpty) Padding(padding: const EdgeInsets.only(top: 8), child: Text(_message)),
      ])),
      Expanded(child: ListView.builder(itemCount: _results.length, itemBuilder: (context, index) {
        final r = _results[index];
        return ExpansionTile(
          title: Text('${r.id}: ${r.status}', style: TextStyle(color: r.status == 'OK' ? Colors.greenAccent : Colors.orangeAccent)),
          subtitle: Text('${r.ms} ms${r.value == null ? '' : ' | ${r.value}'}'),
          children: [Padding(padding: const EdgeInsets.all(12), child: SelectableText(
            'Address: 0x${r.address.toRadixString(16)}\nLength: ${r.length}\nBytes: ${r.bytes}\n${r.detail}',
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ))],
        );
      })),
    ]),
  );
}

class _CanPidResult {
  final String id;
  final int address;
  final int length;
  final String status;
  final double? value;
  final String bytes;
  final int ms;
  final String detail;
  _CanPidResult(this.id, this.address, this.length, this.status, this.value, this.bytes, this.ms, this.detail);
}'''


def require(ok, message):
    if not ok:
        raise ValueError(message)


def dart_mask(text):
    # Keep positions/newlines while hiding strings and comments for brace matching.
    result = list(text)

    def blank(a, b):
        for k in range(a, b):
            if result[k] != '\n':
                result[k] = ' '

    def string_end(start):
        raw = text[start] in 'rR' and start + 1 < len(text) and text[start + 1] in "'\""
        q = start + 1 if raw else start
        delimiter = text[q] * (3 if text.startswith(text[q] * 3, q) else 1)
        i = q + len(delimiter)
        while i < len(text):
            if text.startswith(delimiter, i):
                return i + len(delimiter)
            if not raw and text[i] == '\\':
                i += 2
            elif not raw and text.startswith('${', i):
                i = interpolation_end(i + 2)
            else:
                i += 1
        raise ValueError('Unterminated Dart string.')

    def interpolation_end(i):
        depth = 1
        while i < len(text):
            if text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
                i = string_end(i)
            elif text.startswith('//', i):
                end = text.find('\n', i)
                i = len(text) if end < 0 else end
            elif text.startswith('/*', i):
                end = text.find('*/', i + 2)
                require(end >= 0, 'Unterminated Dart comment.')
                i = end + 2
            else:
                if text[i] == '{': depth += 1
                if text[i] == '}': depth -= 1
                i += 1
                if depth == 0: return i
        raise ValueError('Unterminated Dart interpolation.')

    i = 0
    while i < len(text):
        if text.startswith('//', i):
            end = text.find('\n', i)
            end = len(text) if end < 0 else end
        elif text.startswith('/*', i):
            end = text.find('*/', i + 2)
            require(end >= 0, 'Unterminated Dart comment.')
            end += 2
        elif text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
            end = string_end(i)
        else:
            i += 1
            continue
        blank(i, end)
        i = end
    return ''.join(result)


def close_bracket(mask, index, left='{', right='}'):
    require(mask[index] == left, 'Expected opening bracket.')
    depth = 0
    for i in range(index, len(mask)):
        if mask[i] == left: depth += 1
        if mask[i] == right: depth -= 1
        if depth == 0: return i
    raise ValueError('Unbalanced Dart brackets.')


def class_region(source, name):
    mask = dart_mask(source)
    found = re.search(r'\bclass\s+' + re.escape(name) + r'\b[^\{]*\{', mask)
    require(found is not None, 'Class not found: ' + name)
    opening = found.end() - 1
    return opening, close_bracket(mask, opening)


def methods(source, name):
    opening, closing = class_region(source, name)
    mask = dart_mask(source)
    pattern = r'(?m)^  Future<([^\n]+?)>\s+(\w+)(?:<[^>]+>)?\s*\('
    result = {}
    for match in re.finditer(pattern, mask[opening + 1:closing]):
        start = opening + 1 + match.start()
        paren = opening + match.end()
        end_params = close_bracket(mask, paren, '(', ')')
        suffix = re.match(r'\s*async\s*\{', mask[end_params + 1:])
        if suffix is None: continue
        body_start = end_params + 1 + suffix.end() - 1
        body_end = close_bracket(mask, body_start)
        require(match.group(2) not in result, 'Duplicate method: ' + match.group(2))
        result[match.group(2)] = dict(
            start=start, end=body_end + 1, header=source[start:body_start + 1],
            body=source[body_start + 1:body_end], params=source[paren + 1:end_params],
            returns=match.group(1), name=match.group(2),
        )
    return result


def edits(source, changes):
    for start, end, replacement in sorted(changes, reverse=True):
        source = source[:start] + replacement + source[end:]
    return source


def method_text(method, body):
    return method['header'] + '\n' + body.rstrip() + '\n  }'


def patch_elm(source):
    if '// CAN_V3_ADAPTER_BEGIN' in source:
        require('canV3Ready' in source and '_canV3Physical' in source, 'Partial CAN-V3 adapter detected.')
        return source
    found = methods(source, 'SsmElm')
    for name in ('ecuInit', 'readBytes', 'connect', 'disconnect'):
        require(name in found, 'Expected async SsmElm method missing: ' + name)
    require(found['ecuInit']['returns'] == 'String?' and not found['ecuInit']['params'].strip(),
            'Unexpected ecuInit signature; send can_v3_source_probe.txt.')
    require(found['readBytes']['returns'] == 'Uint8List?', 'Unexpected readBytes return type.')
    candidates = [m for m in found.values() if m['returns'] == 'String'
                  and re.search(r'\.output\s*\.\s*add\s*\(', dart_mask(m['body']))]
    require(len(candidates) == 1, 'Cannot identify one raw ELM send method without guessing. Send can_v3_source_probe.txt.')
    wire = candidates[0]
    require(not re.search(r'\b' + re.escape(wire['name']) + r'\s*\(', dart_mask(wire['body'])),
            'Raw sender is recursive; manual adaptation is required.')
    first = re.match(r'\s*String\s+(\w+)', wire['params'])
    require(first is not None, 'Expected a String command as the first wire parameter.')
    command_var = first.group(1)
    timeout = re.search(r'\b(int|Duration)\s+(timeoutMs|timeout)\s*=', wire['params'])
    require(timeout is not None and '{' in wire['params'], 'Expected a named timeout parameter in raw sender.')
    timeout_type, timeout_name = timeout.groups()
    passed_ms = timeout_name if timeout_type == 'int' else timeout_name + '.inMilliseconds'
    core_args = 'command, ' + timeout_name + (': ms' if timeout_type == 'int' else ': Duration(milliseconds: ms)')
    rest = wire['params'][first.end():]
    require(not re.search(r'\brequired\b', rest), 'Raw sender has unsupported required parameters.')
    for flattened in ("replaceAll('\\r', '')", "replaceAll('\\n', '')"):
        require(flattened not in wire['body'], 'Raw sender removes CAN frame boundaries. Send the source report.')

    old_init = found['ecuInit']['body']
    state_call = re.search(r'\b(\w+)\(SsmState\.ecuReady\);', dart_mask(old_init))
    state_assignment = re.search(r'\b(\w+)\s*=\s*SsmState\.ecuReady;', dart_mask(old_init))
    require(state_call is not None or state_assignment is not None, 'Cannot locate SsmState update without guessing.')
    ready = (state_call.group(1) + '(SsmState.ecuReady);') if state_call else (state_assignment.group(1) + ' = SsmState.ecuReady;')
    not_ready = ready.replace('SsmState.ecuReady', 'SsmState.elmReady')
    failed_state = ready.replace('SsmState.ecuReady', 'SsmState.error')
    a, b = class_region(source, 'SsmElm')
    ecu_id = re.search(r'\bString\s+(ecuId|_ecuId)\s*=', dart_mask(source[a:b]))
    require(ecu_id is not None, 'Cannot locate ECU-ID storage without guessing.')
    byte_params = re.findall(r'\bint\s+(\w+)', found['readBytes']['params'])
    require(len(byte_params) == 2, 'Expected readBytes(address, length).')

    physical_header = wire['header'].replace(wire['name'] + '(', '_canV3Physical(', 1)
    if physical_header == wire['header']:
        physical_header = re.sub(r'\b' + wire['name'] + r'\s*\(', '_canV3Physical(', wire['header'], count=1)
    wrapper = fr'''    final _canV3Command = {command_var}.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (_canV3Configuring) {{
      // Adapter setup may reset defaults, but no vehicle packet is allowed before ATSP6.
      if (!_canV3Command.startsWith('AT') ||
          _canV3Command.startsWith('ATIB') || _canV3Command == 'ATSI' || _canV3Command == 'ATFI') {{
        throw StateError('CAN-V3 rejected startup command: $_canV3Command');
      }}
      final _canV3Sent = _canV3Command.startsWith('ATSP') || _canV3Command.startsWith('ATTP')
          ? 'ATSP6' : _canV3Command.startsWith('ATSH') ? 'ATSH7E0' : _canV3Command;
      if (_canV3Sent != _canV3Command) _canV3.record('FORCED $_canV3Command -> $_canV3Sent');
      return _canV3.setupCommand(_canV3Sent, {passed_ms});
    }}
    return _canV3.terminalQuery({command_var}, {passed_ms});'''
    physical = physical_header + wire['body'] + '}\n'

    new_init = f'''    final st = can_settings.SettingsService.I;
    st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
    stTimeoutCode = st.stCode;
    await st.save();
    final probes = st.selectedPids
        .where((p) => p.len > 0 && p.len <= 4 && p.address >= 0 && p.address + p.len - 1 <= 0xFFFFFF)
        .take(4).map((p) => can_proto.CanV3Probe('${{p.id}}:byte0', p.address, 1)).toList();
    final ok = await _canV3.initialize(probes, stCode: stTimeoutCode);
    {ecu_id.group(1)} = ''; // A successful OBD probe is not an ECU hardware ID.
    if (!ok) {{ {not_ready} return null; }}
    {ready}
    return 'CAN-V3: SSM2 confirmed, hardware ID not read';'''
    new_read = f'''    final bytes = await _canV3.readBytes({byte_params[0]}, {byte_params[1]});
    if (bytes == null && !_canV3.ssmReady) {{ {failed_state} }}
    return bytes;'''
    new_connect = '''    _canV3Configuring = true;
    _canV3.invalidate('new Bluetooth connection');
    try {''' + found['connect']['body'] + '''
    } finally { _canV3Configuring = false; }'''
    new_disconnect = "    _canV3.invalidate('Bluetooth disconnect');\n" + found['disconnect']['body']
    changes = []
    for name, body in [('ecuInit', new_init), ('readBytes', new_read), ('connect', new_connect), ('disconnect', new_disconnect)]:
        m = found[name]
        changes.append((m['start'], m['end'], method_text(m, body)))
    changes.append((wire['start'], wire['end'], method_text(wire, wrapper) + '\n\n  ' + physical.lstrip()))
    source = edits(source, changes)
    opening, _ = class_region(source, 'SsmElm')
    members = f'''
  // CAN_V3_ADAPTER_BEGIN
  bool _canV3Configuring = false;
  late final can_proto.CanV3Session _canV3 = can_proto.CanV3Session(
      (command, ms) => _canV3Physical({core_args}));
  bool get canV3Ready => _canV3.ssmReady;
  String get canV3LastError => _canV3.lastError;
  String get canV3Diagnostic => _canV3.diagnostic;
  int get canV3ReadOk => _canV3.readOk;
  int get canV3ReadErrors => _canV3.readErrors;
  int get canV3NoData => _canV3.noData;
  double get canV3AvgReadMs => _canV3.averageReadMs;
  Future<void> canV3Idle() => _canV3.whenIdle();
  // CAN_V3_ADAPTER_END
'''
    source = source[:opening + 1] + members + source[opening + 1:]
    source = "import 'can_v3_session.dart' as can_proto;\nimport '../services/settings_service.dart' as can_settings;\n" + source
    poller = methods(source, 'SsmPoller')
    require('_tick' in poller, 'Expected SsmPoller._tick to stop on a CAN failure.')
    tick = poller['_tick']
    tick_body, tick_count = re.subn(
        r'(data\s*=\s*await\s+elm\.readBytes\([^;]+\);)',
        r'\1\n        if (!elm.canV3Ready) { stop(); return; }', tick['body'])
    require(tick_count == 1, 'Cannot safely add the poller CAN failure guard.')
    tick_body = '    if (!elm.canV3Ready) { stop(); return; }\n' + tick_body
    source = edits(source, [(tick['start'], tick['end'], method_text(tick, tick_body))])
    print('Detected ELM sender:', wire['name'], '| timeout:', timeout_type, timeout_name)
    print('Replaced ECU init and readBytes; all SSM address requests now use CAN ISO-TP.')
    return source


def patch_connection(source):
    if '// CAN_V3_CONNECTION' in source: return source
    found = methods(source, 'ConnectionService')
    require('connectAndInit' in found, 'ConnectionService.connectAndInit not found.')
    m = found['connectAndInit']
    body = r'''    // CAN_V3_CONNECTION
    if (connecting) return 'CAN-V3: подключение уже выполняется.';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выберите сопряжённый ELM327.';
    connecting = true;
    notifyListeners();
    try {
      stopPolling();
      await elm.canV3Idle();
      if (logger.logging) await logger.stop();
      await elm.disconnect();
      st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
      await st.save();
      elm.stTimeoutCode = st.stCode;
      if (!await elm.connect(st.btAddress)) return 'CAN-V3: Bluetooth/ELM setup failed. Check the selected adapter.';
      final initialized = await elm.ecuInit();
      if (initialized == null || !elm.canV3Ready) {
        final reason = elm.canV3LastError;
        await elm.disconnect();
        return 'CAN-V3: $reason';
      }
      buildPoller();
      // Do not immediately poll an unverified large PID library on a new transport.
      return 'OK: CAN 500 kbit/s, ATDPN=6, SSM2 подтверждён. '
          'ECU ID не считан. Проверьте PID CAN, затем нажмите СТАРТ ОПРОС.';
    } catch (e) {
      try { await elm.disconnect(); } catch (_) {}
      return 'CAN-V3: $e';
    } finally {
      connecting = false;
      notifyListeners();
    }'''
    source = edits(source, [(m['start'], m['end'], method_text(m, body))])
    # Replace a fixed sleep with an actual drain of the serialized CAN transaction.
    source, count = re.subn(
        r'await Future<void>\.delayed\(const Duration\(milliseconds: 180\)\);[^\n]*',
        'await elm.canV3Idle(); // Wait for a complete CAN transaction, not 180 ms.', source)
    require(count == 1 or 'await elm.canV3Idle(); // Wait' in source,
            'The original exclusive() synchronization differs. Send the source report.')
    source = source.replace('if (was) startPolling();', 'if (was && elm.canV3Ready) startPolling();')
    return source


PANEL = r'''          // CAN_V3_PANEL_BEGIN
          const Text('CAN-V3 | CAN 11-bit, 500 kbit/s | 7E0 / 7E8',
              style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          const Text('После подключения опрос запускается вручную. '
              'В терминале разрешены только ATI / ATDP / ATDPN / ATRV / AT@1.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextButton.icon(
            onPressed: _busy || _scanning ? null : () async {
              await Navigator.of(context).push(MaterialPageRoute(builder: (_) => const CanV3PidScreen()));
              if (mounted) setState(() {});
            },
            icon: const Icon(Icons.fact_check_outlined), label: const Text('PID CAN: выбранные параметры'),
          ),
          TextButton.icon(onPressed: () async {
            try {
              await Clipboard.setData(ClipboardData(text: svc.elm.canV3Diagnostic));
              if (mounted) ScaffoldMessenger.of(context).showSnackBar(
                const SnackBar(content: Text('Журнал CAN скопирован')));
            } catch (e) { if (mounted) setState(() => _msg = 'Ошибка копирования CAN: $e'); }
          }, icon: const Icon(Icons.copy), label: const Text('Копировать журнал CAN')),
          ExpansionTile(title: const Text('Журнал CAN-V3', style: TextStyle(fontSize: 12)),
            children: [SelectableText(svc.elm.canV3Diagnostic,
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace'))]),
          // CAN_V3_PANEL_END
'''


def patch_settings(source):
    require('BT-PERM-V2' in source, 'Apply the existing BT-PERM-V2 fix before CAN-V3.')
    if '// CAN_V3_PANEL_BEGIN' in source: return source
    anchor = "        _card('ПОДКЛЮЧЕНИЕ ELM327', [\n"
    require(source.count(anchor) == 1, 'Connection panel anchor not found.')
    source = source.replace(anchor, anchor + PANEL, 1)
    for statement in ("import 'can_v3_pid_screen.dart';", "import 'package:flutter/services.dart';"):
        if statement not in source: source = statement + '\n' + source
    source = re.sub(r"(st\.stCode\.toDouble\(\),\s*)(?:0x[\da-fA-F]+|\d+),\s*(?:0x[\da-fA-F]+|\d+),",
                    r'\g<1>0x32, 0xFF,', source)
    source = re.sub(r'st\.stCode = v\.round\(\)(?: & ~1)?;[^\n]*', 'st.stCode = v.round();', source)
    source = source.replace("_slider('AT ST (×4 мс ожидание байта)'", "_slider('AT ST (CAN; после переподключения)'")
    source = source.replace('stats.avgMs', 'svc.elm.canV3AvgReadMs')
    source = source.replace('stats.framesOk', 'svc.elm.canV3ReadOk')
    source = source.replace('stats.framesErr', 'svc.elm.canV3ReadErrors')
    source = source.replace('stats.noData', 'svc.elm.canV3NoData')
    source = source.replace('на CAN-кадр', 'на чтение A8')
    source = source.replace('Кадров OK / ошибок / NO DATA', 'Чтений A8 OK / ошибок / NO DATA')
    return source


def preflight_tests():
    sample = """class X {
  Future<String> raw(String command, {int timeoutMs = 700}) async {
    final s = '${1 + 2}'; // } ignored
    return s;
  }
}
"""
    parsed = methods(sample, 'X')
    require('raw' in parsed and parsed['raw']['params'].startswith('String command'), 'Dart source parser self-test failed.')
    require(sample[parsed['raw']['start']:parsed['raw']['end']].endswith('}'), 'Method boundary self-test failed.')


paths = {
    ROOT / 'lib/ssm/ssm_elm.dart': patch_elm,
    ROOT / 'lib/services/connection_service.dart': patch_connection,
    ROOT / 'lib/screens/settings_screen.dart': patch_settings,
}
try:
    preflight_tests()
    require(FLUTTER.is_file(), 'Flutter SDK is not available in this Colab session.')
    require((ROOT / '.dart_tool/package_config.json').is_file(), 'Run pub get/build cell setup in this project first.')
    originals = {path: path.read_text(encoding='utf-8') for path in paths}
    outputs = {path: transform(originals[path]) for path, transform in paths.items()}
    for path, transform in paths.items():
        require(transform(outputs[path]) == outputs[path], 'Idempotence check failed: ' + str(path))
except (ValueError, FileNotFoundError) as error:
    if ROOT.is_dir():
        report = ROOT / 'can_v3_source_probe.txt'
        report.write_text('\n\n'.join(str(p.relative_to(ROOT)) + '\n' + p.read_text(encoding='utf-8')
                                     for p in paths if p.is_file()), encoding='utf-8')
        print('Source report for manual adaptation:', report)
    raise SystemExit('Preflight failed; project sources were NOT changed: ' + str(error)) from error

outputs.update({
    ROOT / 'lib/ssm/can_v3_codec.dart': CODEC,
    ROOT / 'lib/ssm/can_v3_session.dart': SESSION,
    ROOT / 'lib/screens/can_v3_pid_screen.dart': PID_SCREEN,
    ROOT / 'test/can_v3_test.dart': TESTS,
})
changes = [(path, path.read_bytes() if path.exists() else None, text.encode('utf-8'))
           for path, text in outputs.items() if not path.exists() or path.read_text(encoding='utf-8') != text]
backup = ROOT / 'patch_backups' / ('can_v3_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for path, before, _ in changes:
    if before is not None:
        saved = backup / path.relative_to(ROOT)
        saved.parent.mkdir(parents=True, exist_ok=True)
        saved.write_bytes(before)


def rollback():
    for path, before, _ in changes:
        if before is None: path.unlink(missing_ok=True)
        else: path.write_bytes(before)


try:
    for path, _, after in changes:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(after)
    print('Running CAN codec, flow-control, session and application-link tests...')
    result = subprocess.run([str(FLUTTER), 'test', '--no-pub', 'test/can_v3_test.dart'],
                            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300)
    (ROOT / 'can_v3_test_output.txt').write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    require(result.returncode == 0, 'Tests failed. Send can_v3_test_output.txt.')
except (OSError, ValueError, subprocess.TimeoutExpired) as error:
    rollback()
    raise SystemExit('CAN-V3 was NOT installed; changed source files were rolled back. ' + str(error)) from error

print('CAN-V3 source patch and simulated tests passed IN THIS COLAB RUNTIME.')
print('Backups:', backup if changes else 'no changes (already applied)')
print('Now run the EXISTING cell 10/10 and install the NEW APK. Look for CAN-V3.')
print('OBD 4100 alone does not confirm SSM2. A valid E8 probe is required.')
print('No ECU hardware ID is invented. No global Subaru-CAN PID cache is reused.')
print('Polling starts manually. First verify a small selected PID set using PID CAN.')
print('Terminal protocol changes and arbitrary TX are blocked in this stabilization patch.')
print('Legacy DTC/service routines require separate CAN validation; do not assume they work.')
print('Real ELM clone, flow-control timing, CAN bus and ECU have NOT been tested by these tests.')

Detected ELM sender: transact | timeout: int timeoutMs
Replaced ECU init and readBytes; all SSM address requests now use CAN ISO-TP.
Running CAN codec, flow-control, session and application-link tests...
00:00 +0: loading /content/suba_run_v8/test/can_v3_test.dart
00:00 +0: Application API links against the patched SsmElm
00:00 +1: Exact SF and FF/CF for one and two A8 addresses
00:00 +2: Lengths 1..4095, padding removed, CF sequence wraps through zero
00:00 +3: 7E8 is a header, not a response SID; foreign frames and TX echo are not data
00:00 +4: Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail
00:00 +5: CAN-only initialization does not invent an ECU ID or confirm SSM from 4100
00:00 +6: Required AT commands are verified, timeout code is hexadecimal
00:00 +7: MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction
00:00 +8: Concurrent reads are serialized; no data is returned after disconnect
00:00 +9: All tests passed!

CAN-V3 source pa

In [ ]:
# @title SUBA RUN V8: CAN-V3 (fixed CAN 6, raw ISO-TP, selected PID diagnostics)
# Run LAST among fixes, before the existing build cell 10/10.
# Existing BT-PERM-V2 is required. This cell NEVER builds/flashes an ECU ROM.
from pathlib import Path
from datetime import datetime, timezone
import re
import subprocess

ROOT = Path('/content/suba_run_v8')
FLUTTER = Path('/content/flutter/bin/flutter')
CODEC = r'''import 'dart:typed_data';

class CanV3Exception implements Exception {
  final String code;
  final String message;
  const CanV3Exception(this.code, this.message);
  @override
  String toString() => '$code: $message';
}

class CanV3Flow {
  final int blockSize;
  final Duration separation;
  const CanV3Flow(this.blockSize, this.separation);
}

class CanV3Codec {
  static String hex(List<int> bytes) => bytes
      .map((b) => b.toRadixString(16).padLeft(2, '0').toUpperCase()).join();

  static void validateBytes(List<int> bytes) {
    if (bytes.any((b) => b < 0 || b > 255)) {
      throw const CanV3Exception('BAD_BYTE', 'Byte outside 0..255.');
    }
  }

  static List<int> addressRead(int address, int count) {
    if (address < 0 || address > 0xFFFFFF || count < 1 || count > 128 || address + count - 1 > 0xFFFFFF) {
      throw const CanV3Exception('BAD_RANGE', 'A8 requires a 24-bit address and 1..128 bytes.');
    }
    final payload = <int>[0xA8, 0x00];
    for (var offset = 0; offset < count; offset++) {
      final a = address + offset;
      payload.addAll([(a >> 16) & 255, (a >> 8) & 255, a & 255]);
    }
    return payload;
  }

  static List<int> _pad(List<int> bytes, bool pad) =>
      pad ? [...bytes, ...List.filled(8 - bytes.length, 0)] : bytes;

  // pad=false keeps frames <= 7 bytes, which an ELM327 without AT AL still accepts.
  static List<List<int>> encode(List<int> payload, {bool pad = true}) {
    validateBytes(payload);
    if (payload.isEmpty || payload.length > 4095) {
      throw const CanV3Exception('BAD_LENGTH', 'Classical CAN ISO-TP length must be 1..4095.');
    }
    if (payload.length <= 7) return [_pad([payload.length, ...payload], pad)];
    if (!pad) {
      throw const CanV3Exception('AL_REQUIRED',
          'Multi-frame ISO-TP needs full 8-byte frames, but this adapter rejected them (AT AL unsupported).');
    }
    final frames = <List<int>>[
      [0x10 | (payload.length >> 8), payload.length & 255, ...payload.take(6)],
    ];
    var offset = 6;
    var sequence = 1;
    while (offset < payload.length) {
      final end = (offset + 7 < payload.length) ? offset + 7 : payload.length;
      frames.add(_pad([0x20 | sequence, ...payload.sublist(offset, end)], true));
      sequence = (sequence + 1) & 15;
      offset = end;
    }
    return frames;
  }

  static String? elmError(String reply) {
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.replaceAll(RegExp(r'\s+'), '');
      if (line.contains('NODATA')) return 'NO_DATA';
      if (line.contains('BUFFERFULL')) return 'BUFFER_FULL';
      if (line.contains('CANERROR') || line.contains('BUSERROR')) return 'CAN_ERROR';
      if (line.contains('UNABLETOCONNECT')) return 'UNABLE_TO_CONNECT';
      if (line.contains('STOPPED')) return 'STOPPED';
      if (line.contains('ERROR') || line == '?') return 'ELM_ERROR';
    }
    return null;
  }

  // Parse each physical line. Never search for E8 inside a header or payload.
  static List<List<int>> frames(String reply, {String request = '', int rxId = 0x7E8}) {
    final error = elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    final result = <List<int>>[];
    final echo = request.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final expected = rxId.toRadixString(16).padLeft(3, '0').toUpperCase();
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.trim();
      if (line.startsWith('SEARCHING...')) line = line.substring(12).trim();
      final compact = line.replaceAll(RegExp(r'\s+'), '');
      if (compact.isEmpty || compact == echo || compact == 'OK') continue;
      if (!compact.startsWith(expected)) {
        // Other 11-bit responders are filtered, not joined into this ECU response.
        if (RegExp(r'^[0-9A-F]{3}:?[0-9A-F]+$').hasMatch(compact) &&
            int.parse(compact.substring(0, 3), radix: 16) <= 0x7FF) continue;
        throw CanV3Exception('MALFORMED_LINE', 'Unexpected ELM line: $line');
      }
      var body = compact.substring(3);
      if (body.startsWith(':')) body = body.substring(1);
      int? dlc;
      if (body.length.isOdd && RegExp(r'^[0-8]').hasMatch(body)) {
        dlc = int.parse(body[0], radix: 16);
        body = body.substring(1);
      }
      if (body.isEmpty || body.length.isOdd || body.length > 16 || !RegExp(r'^[0-9A-F]+$').hasMatch(body)) {
        throw CanV3Exception('MALFORMED_FRAME', 'Invalid raw CAN frame: $line');
      }
      final bytes = <int>[
        for (var i = 0; i < body.length; i += 2) int.parse(body.substring(i, i + 2), radix: 16),
      ];
      if (dlc != null && bytes.length != dlc) {
        throw const CanV3Exception('DLC_MISMATCH', 'DLC does not match CAN data length.');
      }
      result.add(bytes);
    }
    if (result.isEmpty) throw const CanV3Exception('NO_FRAMES', 'No complete 7E8 raw CAN frames.');
    return result;
  }

  static List<Uint8List> assemble(List<List<int>> frames) {
    final messages = <Uint8List>[];
    List<int>? pending;
    var length = 0;
    var sequence = 1;
    for (final frame in frames) {
      validateBytes(frame);
      if (frame.isEmpty || frame.length > 8) throw const CanV3Exception('BAD_FRAME', 'CAN frame length.');
      final type = frame[0] >> 4;
      if (type == 0) {
        if (pending != null) throw const CanV3Exception('INTERRUPTED', 'SF interrupted a multi-frame response.');
        final n = frame[0] & 15;
        if (n < 1 || n > 7 || n > frame.length - 1) throw const CanV3Exception('SF_LENGTH', 'Invalid SF length.');
        messages.add(Uint8List.fromList(frame.sublist(1, n + 1)));
      } else if (type == 1) {
        if (pending != null || frame.length != 8) throw const CanV3Exception('BAD_FF', 'Unexpected or incomplete FF.');
        length = ((frame[0] & 15) << 8) | frame[1];
        if (length < 8 || length > 4095) throw const CanV3Exception('FF_LENGTH', 'Invalid FF length.');
        pending = frame.sublist(2);
        sequence = 1;
      } else if (type == 2) {
        if (pending == null || frame.length < 2) throw const CanV3Exception('ORPHAN_CF', 'CF without FF.');
        if ((frame[0] & 15) != sequence) throw const CanV3Exception('SEQUENCE', 'Wrong, duplicate or missing CF sequence.');
        sequence = (sequence + 1) & 15;
        final needed = length - pending.length;
        final available = frame.length - 1;
        if (available < needed && frame.length != 8) throw const CanV3Exception('SHORT_CF', 'Truncated CF.');
        pending.addAll(frame.skip(1).take(needed));
        if (pending.length == length) {
          messages.add(Uint8List.fromList(pending));
          pending = null;
        }
      } else {
        throw CanV3Exception('UNEXPECTED_PCI', 'Expected SF/FF/CF, got PCI ${hex([frame[0]])}.');
      }
    }
    if (pending != null) throw const CanV3Exception('INCOMPLETE', 'Response ended before declared ISO-TP length.');
    if (messages.isEmpty) throw const CanV3Exception('NO_PDU', 'No ISO-TP message.');
    return messages;
  }

  static CanV3Flow flow(List<List<int>> frames) {
    CanV3Flow? accepted;
    for (final bytes in frames) {
      if (bytes.length < 3 || bytes[0] >> 4 != 3) throw const CanV3Exception('EXPECTED_FC', 'No valid flow control.');
      final status = bytes[0] & 15;
      if (status == 1) continue;
      if (status == 2) throw const CanV3Exception('FC_OVERFLOW', 'ECU cannot accept the request.');
      if (status != 0 || accepted != null) throw const CanV3Exception('BAD_FC', 'Invalid or repeated CTS.');
      final st = bytes[2];
      if (st > 0x7F && (st < 0xF1 || st > 0xF9)) throw const CanV3Exception('STMIN', 'Reserved STmin value.');
      // A millisecond is a safe lower bound for sub-millisecond STmin on ELM serial.
      accepted = CanV3Flow(bytes[1], Duration(milliseconds: st <= 0x7F ? st : 1));
    }
    if (accepted == null) throw const CanV3Exception('FC_WAIT', 'WAIT received without CTS before the ELM prompt.');
    return accepted;
  }

  static Uint8List positive(List<Uint8List> messages, int requestSid, List<int> prefix, int length) {
    Uint8List? answer;
    CanV3Exception? negative;
    for (final bytes in messages) {
      if (bytes.length >= 2 && bytes[0] == 0x7F && bytes[1] == requestSid) {
        if (bytes.length != 3) throw const CanV3Exception('NEGATIVE_LENGTH', 'Invalid negative response length.');
        negative = CanV3Exception(bytes[2] == 0x78 ? 'PENDING' : 'NEGATIVE_RESPONSE',
            'SID ${hex([requestSid])}, NRC ${hex([bytes[2]])}.');
        if (bytes[2] != 0x78 || answer != null) throw negative;
        continue;
      }
      if (bytes.length < prefix.length || !List.generate(prefix.length, (i) => bytes[i] == prefix[i]).every((v) => v)) {
        throw CanV3Exception('WRONG_SID', 'Unexpected PDU ${hex(bytes)}.');
      }
      if (bytes.length != length) throw CanV3Exception('RESPONSE_LENGTH', 'Expected $length bytes; got ${bytes.length}.');
      if (answer != null) throw const CanV3Exception('AMBIGUOUS', 'Multiple positive responses for one request.');
      answer = bytes;
    }
    if (answer != null) return answer;
    throw negative ?? const CanV3Exception('NO_POSITIVE_RESPONSE', 'No matching positive response.');
  }
}'''
SESSION = r'''import 'dart:async';
import 'dart:typed_data';
import 'can_v3_codec.dart';

typedef CanV3Send = Future<String> Function(String command, int timeoutMs);

class CanV3Probe {
  final String id;
  final int address;
  final int length;
  const CanV3Probe(this.id, this.address, this.length);
}

class CanV3Session {
  final CanV3Send send;
  final Future<void> Function(Duration) delay;
  Future<void> _tail = Future<void>.value();
  final List<String> _trace = [];
  List<String> _initTrace = [];
  int _generation = 0;
  bool _responses = true;
  bool _poisoned = false;
  bool obdReady = false;
  bool ssmReady = false;
  String protocolNumber = '';
  String lastError = '';
  String lastProbe = '';
  int timeoutCode = 0x32;
  bool padFrames = true;
  bool allowLong = false;
  int txFrames = 0;
  int readOk = 0;
  int readErrors = 0;
  int noData = 0;
  int _readMilliseconds = 0;
  double get averageReadMs => readOk == 0 ? 0 : _readMilliseconds / readOk;

  CanV3Session(this.send, {Future<void> Function(Duration)? delay})
      : delay = delay ?? ((d) => Future<void>.delayed(d));

  String get diagnostic => [
    'CAN-V3 | ISO 15765-4 / 11-bit / 500 kbit/s',
    'Required configuration: protocol 6, TX=7E0, RX=7E8, CAF=0, CFC=1',
    'AT AL accepted=$allowLong; 8-byte TX padding=$padFrames',
    if (!padFrames) 'Unpadded mode: multi-byte PIDs use separate requests and are not sampled atomically.',
    'Last ATDPN reply in current connection: $protocolNumber',
    'OBD=$obdReady; SSM2=$ssmReady; last probe=$lastProbe',
    'ECU hardware ID: not read (no synthetic ID, no shared PID cache)',
    'AT ST=0x${timeoutCode.toRadixString(16).toUpperCase()}; nominal ${timeoutCode * 4} ms',
    'TX CAN frames=$txFrames; complete reads=$readOk; read errors=$readErrors; NO DATA=$noData',
    'Average complete A8 read: ${averageReadMs.toStringAsFixed(1)} ms (not time per physical frame)',
    'last error=$lastError',
    '--- INIT TRACE ---', ..._initTrace,
    '--- RECENT TRACE ---', ..._trace,
  ].join('\n');

  void record(String text) {
    _trace.add('${DateTime.now().toIso8601String()} $text');
    if (_trace.length > 240) _trace.removeRange(0, _trace.length - 240);
  }

  void invalidate(String reason) {
    _generation++;
    obdReady = false;
    ssmReady = false;
    protocolNumber = '';
    _poisoned = true;
    record('INVALIDATE $reason');
  }

  Future<T> _serial<T>(Future<T> Function() task) {
    final run = _tail.then((_) => task());
    _tail = run.then<void>((_) {}, onError: (Object _, StackTrace __) {});
    return run;
  }

  Future<void> whenIdle() => _tail;

  Future<String> setupCommand(String command, int timeoutMs) =>
      _serial(() => _wire(command, timeoutMs: timeoutMs));

  Future<String> _wire(String command, {int timeoutMs = 1500, bool responseRequired = true}) async {
    final generation = _generation;
    record('TX $command');
    try {
      final response = await send(command, timeoutMs).timeout(Duration(milliseconds: timeoutMs + 500));
      record('RX ${response.replaceAll('\r', '<CR>').replaceAll('\n', '<LF>')}');
      if (generation != _generation) throw const CanV3Exception('CANCELLED', 'Connection changed during request.');
      if (responseRequired && response.replaceAll('>', '').trim().isEmpty) {
        _poisoned = true;
        throw const CanV3Exception('EMPTY_REPLY', 'No response; reconnect before continuing.');
      }
      return response;
    } on TimeoutException {
      _poisoned = true;
      ssmReady = false;
      throw const CanV3Exception('WIRE_TIMEOUT', 'ELM prompt/command timeout. Reconnect to avoid a late reply.');
    }
  }

  Future<void> _ok(String command) async {
    final response = await _wire(command);
    final error = CanV3Codec.elmError(response);
    final lines = response.toUpperCase().replaceAll('>', '').split(RegExp(r'[\r\n]+'));
    if (error != null || !lines.any((line) => line.trim() == 'OK')) {
      throw CanV3Exception('AT_REJECTED', '$command -> ${response.trim()}');
    }
  }

  // Optional commands must not abort setup, but the outcome is recorded.
  Future<bool> _tryOk(String command) async {
    try {
      await _ok(command);
      return true;
    } on CanV3Exception catch (e) {
      if (e.code == 'WIRE_TIMEOUT' || e.code == 'EMPTY_REPLY' || e.code == 'CANCELLED') rethrow;
      record('OPTIONAL $command rejected: ${e.message}');
      return false;
    }
  }

  Future<void> _setResponses(bool enabled) async {
    if (_responses == enabled) return;
    await _ok(enabled ? 'ATR1' : 'ATR0');
    _responses = enabled;
  }

  Future<String> _frame(List<int> bytes, {required bool expectResponse}) async {
    txFrames++;
    final reply = await _wire(CanV3Codec.hex(bytes), timeoutMs: 3000, responseRequired: expectResponse);
    final error = CanV3Codec.elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    return reply;
  }

  Future<List<Uint8List>> _exchange(List<int> payload) async {
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Transport is not synchronized.');
    final frames = CanV3Codec.encode(payload, pad: padFrames);
    try {
      await _setResponses(true);
      var reply = await _frame(frames.first, expectResponse: true);
      if (frames.length == 1) {
        return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      }
      var flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      var sentInBlock = 0;
      for (var i = 1; i < frames.length; i++) {
        final last = i == frames.length - 1;
        final boundary = flow.blockSize > 0 && sentInBlock + 1 == flow.blockSize;
        // Suppress response waits only for CFs that cannot require FC or a final reply.
        await _setResponses(last || boundary);
        await delay(flow.separation);
        reply = await _frame(frames[i], expectResponse: last || boundary);
        sentInBlock++;
        if (last) return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
        if (boundary) {
          flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
          sentInBlock = 0;
        }
      }
      throw const CanV3Exception('TX_INCOMPLETE', 'Not all CFs sent.');
    } catch (e) {
      if (frames.length > 1 || e is! CanV3Exception || e.code != 'NO_DATA') {
        _poisoned = true;
        ssmReady = false;
        record('ISO-TP aborted; reconnect required.');
      }
      rethrow;
    } finally {
      if (!_responses && !_poisoned) {
        try { await _setResponses(true); }
        catch (e) { _poisoned = true; ssmReady = false; record('RESTORE ATR1 FAILED $e'); }
      }
    }
  }

  Future<Uint8List> _read(int address, int length) async {
    if (!padFrames && length > 1) {
      // Only one address fits an unpadded single frame, so bytes are read one by one.
      // These bytes are NOT sampled at the same instant inside the ECU.
      final combined = <int>[];
      for (var offset = 0; offset < length; offset++) {
        combined.addAll(await _read(address + offset, 1));
      }
      record('Non-atomic read: $length separate A8 requests (adapter lacks AT AL).');
      return Uint8List.fromList(combined);
    }
    final reply = await _exchange(CanV3Codec.addressRead(address, length));
    try {
      final pdu = CanV3Codec.positive(reply, 0xA8, [0xE8], length + 1);
      return Uint8List.fromList(pdu.sublist(1));
    } on CanV3Exception catch (e) {
      if (e.code != 'NEGATIVE_RESPONSE') {
        _poisoned = true;
        ssmReady = false;
      }
      rethrow;
    }
  }

  Future<bool> initialize(List<CanV3Probe> probes, {int stCode = 0x32}) => _serial(() async {
    _generation++;
    _poisoned = false;
    obdReady = false;
    ssmReady = false;
    lastError = '';
    protocolNumber = '';
    lastProbe = '';
    timeoutCode = stCode.clamp(0x32, 0xFF).toInt();
    _trace.clear();
    txFrames = 0; readOk = 0; readErrors = 0; noData = 0; _readMilliseconds = 0;
    try {
      for (final command in [
        'ATE0', 'ATL0', 'ATS0', 'ATSP6', 'ATH1', 'ATD0', 'ATCAF0',
        'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
        'ATR1', 'ATST${timeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase()}',
      ]) { await _ok(command); }
      _responses = true;
      // Without AL many adapters reject the 8-byte padded frames ISO-TP normally uses.
      allowLong = await _tryOk('ATAL');
      padFrames = true;
      // Same on-wire 01 00 as CAF1 + 0100, without switching CAF mode mid-session.
      List<Uint8List> obd;
      try {
        obd = await _exchange([0x01, 0x00]);
      } on CanV3Exception catch (e) {
        if (e.code != 'ELM_ERROR') rethrow;
        // The adapter refused a padded frame; retry once with a short frame.
        padFrames = false;
        _poisoned = false;
        record('Padded 8-byte TX rejected. Retrying unpadded; multi-frame requests stay unavailable.');
        obd = await _exchange([0x01, 0x00]);
      }
      CanV3Codec.positive(obd, 0x01, [0x41, 0x00], 6);
      final number = await _wire('ATDPN');
      protocolNumber = number.replaceAll(RegExp(r'[\s>]'), '').toUpperCase();
      if (protocolNumber != '6') {
        throw CanV3Exception('WRONG_PROTOCOL', 'Expected fixed protocol 6, got $protocolNumber (A6 is auto).');
      }
      obdReady = true;
      record('OBD CAN confirmed; SSM2 is not confirmed yet.');
      for (final probe in probes.take(4)) {
        try {
          await _read(probe.address, probe.length);
          lastProbe = probe.id;
          ssmReady = true;
          record('SSM2 confirmed: ${probe.id}, ${probe.length} bytes, exact E8 response.');
          return true;
        } catch (e) {
          record('PROBE ${probe.id} FAILED $e');
          if (_poisoned) rethrow;
        }
      }
      throw CanV3Exception('SSM_NOT_CONFIRMED', probes.isEmpty
          ? 'Select at least one PID before connecting.'
          : 'OBD CAN works, but selected A8 probes did not return valid E8 data.');
    } catch (e) {
      ssmReady = false;
      lastError = e.toString();
      record('INIT FAILED $lastError');
      return false;
    } finally {
      _initTrace = List.of(_trace);
    }
  });

  Future<Uint8List?> readBytes(int address, int length) => _serial(() async {
    lastError = '';
    final clock = Stopwatch()..start();
    try {
      if (!ssmReady || _poisoned) throw const CanV3Exception('NOT_READY', 'Connect and confirm CAN/SSM2 first.');
      final result = await _read(address, length);
      readOk++;
      _readMilliseconds += clock.elapsedMilliseconds;
      return result;
    } catch (e) {
      readErrors++;
      if (e is CanV3Exception && e.code == 'NO_DATA') noData++;
      lastError = e.toString();
      record('READ FAILED $lastError');
      return null; // No partial bytes, padding or fabricated zeroes.
    }
  });

  Future<String> terminalQuery(String command, int timeoutMs) => _serial(() async {
    final normalized = command.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (!const ['ATI', 'ATDP', 'ATDPN', 'ATRV', 'AT@1'].contains(normalized)) {
      throw const CanV3Exception('MANUAL_TX_DISABLED', 'Use PID CAN for reads; arbitrary terminal TX is blocked.');
    }
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Reconnect before sending terminal queries.');
    return _wire(normalized, timeoutMs: timeoutMs);
  });
}'''
TESTS = r'''import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import '../lib/ssm/can_v3_codec.dart';
import '../lib/ssm/can_v3_session.dart';
import '../lib/ssm/ssm_elm.dart';
import '../lib/main.dart' as application;

String raw(List<int> pdu) => CanV3Codec.encode(pdu)
    .map((f) => '7E8${CanV3Codec.hex(f)}\r').join() + '>';

class FakeElm {
  final commands = <String>[];
  final request = <List<int>>[];
  int required = 0;
  bool responses = true;
  String dpn = '6';
  bool rejectSsm = false;
  int blockSize = 0;
  int inBlock = 0;
  bool supportsAl = true;
  bool rejectPadded = false;

  Future<String> call(String command, int timeout) async {
    commands.add(command);
    if (command == 'ATDPN') return '$dpn\r>';
    if (command == 'ATAL') return supportsAl ? 'OK\r>' : '?\r>';
    if (rejectPadded && !command.startsWith('AT') && command.length > 14) return '?\r>';
    if (command == 'ATR0') responses = false;
    if (command == 'ATR1') responses = true;
    if (command.startsWith('AT')) return 'OK\r>';
    final bytes = <int>[
      for (var i = 0; i < command.length; i += 2) int.parse(command.substring(i, i + 2), radix: 16),
    ];
    if (bytes[0] >> 4 == 0) return respond(CanV3Codec.assemble([bytes]).single);
    if (bytes[0] >> 4 == 1) {
      request.clear();
      request.add(bytes);
      required = ((bytes[0] & 15) << 8) | bytes[1];
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    request.add(bytes);
    inBlock++;
    final received = 6 + (request.length - 1) * 7;
    if (received >= required) {
      if (!responses) throw StateError('Final CF sent with responses disabled.');
      return respond(CanV3Codec.assemble(request).single);
    }
    if (blockSize > 0 && inBlock == blockSize) {
      if (!responses) throw StateError('FC boundary sent with responses disabled.');
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    if (responses) throw StateError('Intermediate CF unexpectedly waits for a response.');
    return '>';
  }

  String respond(List<int> payload) {
    if (payload.length == 2 && payload[0] == 1 && payload[1] == 0) return raw([0x41, 0, 0xBE, 0x3F, 0xA8, 0x13]);
    if (payload[0] != 0xA8 || payload[1] != 0 || (payload.length - 2) % 3 != 0) {
      throw StateError('Incorrect SSM payload: ${CanV3Codec.hex(payload)}');
    }
    if (rejectSsm) return raw([0x7F, 0xA8, 0x31]);
    return raw([0xE8, for (var i = 4; i < payload.length; i += 3) payload[i]]);
  }
}

void main() {
  test('Application API links against the patched SsmElm', () {
    // Referencing main forces compile-time checking of all imported screens/services.
    expect(application.main, isNotNull);
    final elm = SsmElm();
    expect(elm.canV3Ready, isFalse);
    expect(elm.canV3Diagnostic, contains('CAN-V3'));
  });

  test('Exact SF and FF/CF for one and two A8 addresses', () {
    expect(CanV3Codec.hex(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 1)).single), '05A80000000E0000');
    expect(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 2)).map(CanV3Codec.hex).toList(),
        ['1008A80000000E00', '21000F0000000000']);
    expect(CanV3Codec.hex(CanV3Codec.encode([1, 0]).single), '0201000000000000');
  });

  test('Lengths 1..4095, padding removed, CF sequence wraps through zero', () {
    for (final n in [1, 7, 8, 20, 80, 128, 242, 386, 4095]) {
      final data = List.generate(n, (i) => i & 255);
      final encoded = CanV3Codec.encode(data);
      expect(encoded.every((f) => f.length == 8), isTrue);
      expect(CanV3Codec.assemble(encoded).single, data);
    }
    expect(() => CanV3Codec.encode([]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.addressRead(0xFFFFFF, 2), throwsA(isA<CanV3Exception>()));
  });

  test('7E8 is a header, not a response SID; foreign frames and TX echo are not data', () {
    final reply = '05A80000000E0000\r7E9 02 E8 88 00 00 00 00 00\r7E8 02 E8 12 00 00 00 00 00\r>';
    final frames = CanV3Codec.frames(reply, request: '05A80000000E0000');
    expect(CanV3Codec.positive(CanV3Codec.assemble(frames), 0xA8, [0xE8], 2), [0xE8, 0x12]);
    expect(() => CanV3Codec.positive(CanV3Codec.assemble(CanV3Codec.frames('7E8 02 62 12 00 00 00 00 00\r>')), 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31])], 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>().having((e) => e.code, 'code', 'NEGATIVE_RESPONSE')));
    expect(CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x78]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), [0xE8, 1]);
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), throwsA(isA<CanV3Exception>()));
  });

  test('Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail', () {
    expect(() => CanV3Codec.frames('NO DATA\r>'), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([[6, 0xE8, 1]]), throwsA(isA<CanV3Exception>()));
    final full = CanV3Codec.encode(List.generate(25, (i) => i));
    expect(() => CanV3Codec.assemble(full.take(2).toList()), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[2]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[1], full[1]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x32, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x31, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(CanV3Codec.flow([[0x30, 3, 0xF5]]).separation.inMicroseconds, greaterThanOrEqualTo(500));
  });

  test('CAN-only initialization does not invent an ECU ID or confirm SSM from 4100', () async {
    final fake = FakeElm()..rejectSsm = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(fake.commands.contains('ATSP6'), isTrue);
    expect(fake.commands.any((c) => c.startsWith('8010') || c == 'ATSP0' || c == 'ATSP3' || c == 'ATSP5'), isFalse);
    final auto = CanV3Session((FakeElm()..dpn = 'A6').call, delay: (_) async {});
    expect(await auto.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(auto.lastError, contains('WRONG_PROTOCOL'));
  });

  test('Required AT commands are verified, timeout code is hexadecimal', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)], stCode: 8), isTrue);
    expect(fake.commands.contains('ATST32'), isTrue);
    expect(fake.commands.contains('ATST08'), isFalse);
    final bad = CanV3Session((command, timeout) async {
      if (command == 'ATFCSM1') return '?\r>';
      return await fake.call(command, timeout);
    }, delay: (_) async {});
    expect(await bad.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(bad.lastError, contains('ATFCSM1'));
    expect(bad.ssmReady, isFalse);
  });

  test('AT AL is requested; a padded-frame refusal falls back to unpadded single frames', () async {
    final fake = FakeElm()..supportsAl = false..rejectPadded = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(fake.commands.contains('ATAL'), isTrue);
    expect(session.allowLong, isFalse);
    expect(session.padFrames, isFalse);
    // One address per frame; multi-byte PIDs become separate, non-atomic requests.
    expect(CanV3Codec.hex(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 1), pad: false).single), '05A80000000E');
    expect(() => CanV3Codec.encode(CanV3Codec.addressRead(0xE, 2), pad: false),
        throwsA(isA<CanV3Exception>().having((e) => e.code, 'code', 'AL_REQUIRED')));
    expect(await session.readBytes(0x10, 3), [0x10, 0x11, 0x12]);
    expect(session.diagnostic, contains('not sampled atomically'));
  });

  test('With AT AL the padded 8-byte frames stay in use', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(session.allowLong, isTrue);
    expect(session.padFrames, isTrue);
    expect(fake.commands.contains('0201000000000000'), isTrue);
  });

  test('MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction', () async {
    for (final blockSize in [0, 1, 4]) {
      final fake = FakeElm()..blockSize = blockSize;
      final session = CanV3Session(fake.call, delay: (_) async {});
      expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
      expect(await session.readBytes(0xE, 4), [14, 15, 16, 17]);
      expect(await session.readBytes(0x100, 80), List.generate(80, (i) => i));
      expect(fake.responses, isTrue);
    }
  });

  test('Concurrent reads are serialized; no data is returned after disconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    final results = await Future.wait([session.readBytes(10, 4), session.readBytes(20, 4)]);
    expect(results, [[10, 11, 12, 13], [20, 21, 22, 23]]);
    session.invalidate('test disconnect');
    expect(await session.readBytes(10, 4), isNull);
  });
}'''
PID_SCREEN = r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';

class CanV3PidScreen extends StatefulWidget {
  const CanV3PidScreen({super.key});
  @override
  State<CanV3PidScreen> createState() => _CanV3PidScreenState();
}

class _CanV3PidScreenState extends State<CanV3PidScreen> {
  final List<_CanPidResult> _results = [];
  bool _running = false;
  bool _cancel = false;
  int _total = 0;
  String _message = '';

  @override
  void dispose() {
    _cancel = true;
    super.dispose();
  }

  Future<void> _scan() async {
    final svc = ConnectionService.I;
    if (_running) return;
    if (!svc.elm.canV3Ready) {
      setState(() => _message = 'Сначала подтвердите CAN и SSM2 через CONNECT + INIT ECU.');
      return;
    }
    final pids = SettingsService.I.selectedPids.toList();
    if (pids.isEmpty) { setState(() => _message = 'Нет выбранных PID.'); return; }
    setState(() { _running = true; _cancel = false; _results.clear(); _total = pids.length; _message = ''; });
    try {
      await svc.exclusive((elm) async {
        await elm.canV3Idle();
        for (final pid in pids) {
          if (_cancel || !mounted || !elm.canV3Ready) break;
          final clock = Stopwatch()..start();
          final bytes = await elm.readBytes(pid.address, pid.len);
          double? value;
          var status = 'READ_ERROR';
          var detail = elm.canV3LastError;
          if (bytes != null && bytes.length == pid.len) {
            try {
              final decoded = pid.formula(bytes);
              if (!decoded.isFinite) {
                status = 'FORMULA_ERROR'; detail = 'Формула вернула NaN/Infinity.';
              } else {
                value = decoded; status = 'OK'; detail = 'E8 и длина проверены; физическая достоверность не подтверждена.';
              }
            } catch (e) { status = 'FORMULA_ERROR'; detail = e.toString(); }
          }
          clock.stop();
          final result = _CanPidResult(
            pid.id, pid.address, pid.len, status, value,
            bytes?.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ').toUpperCase() ?? '',
            clock.elapsedMilliseconds, detail,
          );
          if (!mounted) break;
          setState(() => _results.add(result));
        }
      });
      if (mounted) setState(() => _message = _cancel ? 'Проверка остановлена после текущего запроса.' : 'Проверка завершена. Кэш рабочих PID не изменён.');
    } catch (e) {
      if (mounted) setState(() => _message = 'Ошибка диагностики: $e');
    } finally {
      if (mounted) setState(() => _running = false);
    }
  }

  Future<void> _copy() async {
    final report = [
      'CAN-V3 selected PID diagnostics (not a calibration validation)',
      ..._results.map((r) => '${r.id} address=0x${r.address.toRadixString(16)} length=${r.length} '
          'status=${r.status} value=${r.value} ms=${r.ms} bytes=${r.bytes} detail=${r.detail}'),
      ConnectionService.I.elm.canV3Diagnostic,
    ].join('\n');
    try {
      await Clipboard.setData(ClipboardData(text: report));
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Отчёт CAN скопирован')));
    } catch (e) { if (mounted) setState(() => _message = 'Копирование не удалось: $e'); }
  }

  @override
  Widget build(BuildContext context) => Scaffold(
    appBar: AppBar(title: const Text('CAN-V3: выбранные PID'), actions: [
      IconButton(tooltip: 'Копировать отчёт', onPressed: _copy, icon: const Icon(Icons.copy)),
    ]),
    body: Column(children: [
      Padding(padding: const EdgeInsets.all(12), child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
        const Text('Проверяется текущий набор PID. Значение не считается физически достоверным только из-за ответа E8. '
            'Опрос приостанавливается на время проверки. Кэш по общему имени Subaru-CAN не создаётся.', style: TextStyle(fontSize: 12)),
        const SizedBox(height: 8),
        Text('${_results.length} / $_total; OK: ${_results.where((r) => r.status == 'OK').length}'),
        if (_running) LinearProgressIndicator(value: _total == 0 ? null : _results.length / _total),
        const SizedBox(height: 8),
        FilledButton(onPressed: _running ? null : _scan, child: const Text('Проверить выбранные PID')),
        if (_running) TextButton(onPressed: () => setState(() => _cancel = true),
            child: Text(_cancel ? 'Завершаем текущий запрос...' : 'Остановить')),
        if (_message.isNotEmpty) Padding(padding: const EdgeInsets.only(top: 8), child: Text(_message)),
      ])),
      Expanded(child: ListView.builder(itemCount: _results.length, itemBuilder: (context, index) {
        final r = _results[index];
        return ExpansionTile(
          title: Text('${r.id}: ${r.status}', style: TextStyle(color: r.status == 'OK' ? Colors.greenAccent : Colors.orangeAccent)),
          subtitle: Text('${r.ms} ms${r.value == null ? '' : ' | ${r.value}'}'),
          children: [Padding(padding: const EdgeInsets.all(12), child: SelectableText(
            'Address: 0x${r.address.toRadixString(16)}\nLength: ${r.length}\nBytes: ${r.bytes}\n${r.detail}',
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ))],
        );
      })),
    ]),
  );
}

class _CanPidResult {
  final String id;
  final int address;
  final int length;
  final String status;
  final double? value;
  final String bytes;
  final int ms;
  final String detail;
  _CanPidResult(this.id, this.address, this.length, this.status, this.value, this.bytes, this.ms, this.detail);
}'''


def require(ok, message):
    if not ok:
        raise ValueError(message)


def dart_mask(text):
    # Keep positions/newlines while hiding strings and comments for brace matching.
    result = list(text)

    def blank(a, b):
        for k in range(a, b):
            if result[k] != '\n':
                result[k] = ' '

    def string_end(start):
        raw = text[start] in 'rR' and start + 1 < len(text) and text[start + 1] in "'\""
        q = start + 1 if raw else start
        delimiter = text[q] * (3 if text.startswith(text[q] * 3, q) else 1)
        i = q + len(delimiter)
        while i < len(text):
            if text.startswith(delimiter, i):
                return i + len(delimiter)
            if not raw and text[i] == '\\':
                i += 2
            elif not raw and text.startswith('${', i):
                i = interpolation_end(i + 2)
            else:
                i += 1
        raise ValueError('Unterminated Dart string.')

    def interpolation_end(i):
        depth = 1
        while i < len(text):
            if text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
                i = string_end(i)
            elif text.startswith('//', i):
                end = text.find('\n', i)
                i = len(text) if end < 0 else end
            elif text.startswith('/*', i):
                end = text.find('*/', i + 2)
                require(end >= 0, 'Unterminated Dart comment.')
                i = end + 2
            else:
                if text[i] == '{': depth += 1
                if text[i] == '}': depth -= 1
                i += 1
                if depth == 0: return i
        raise ValueError('Unterminated Dart interpolation.')

    i = 0
    while i < len(text):
        if text.startswith('//', i):
            end = text.find('\n', i)
            end = len(text) if end < 0 else end
        elif text.startswith('/*', i):
            end = text.find('*/', i + 2)
            require(end >= 0, 'Unterminated Dart comment.')
            end += 2
        elif text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
            end = string_end(i)
        else:
            i += 1
            continue
        blank(i, end)
        i = end
    return ''.join(result)


def close_bracket(mask, index, left='{', right='}'):
    require(mask[index] == left, 'Expected opening bracket.')
    depth = 0
    for i in range(index, len(mask)):
        if mask[i] == left: depth += 1
        if mask[i] == right: depth -= 1
        if depth == 0: return i
    raise ValueError('Unbalanced Dart brackets.')


def class_region(source, name):
    mask = dart_mask(source)
    found = re.search(r'\bclass\s+' + re.escape(name) + r'\b[^\{]*\{', mask)
    require(found is not None, 'Class not found: ' + name)
    opening = found.end() - 1
    return opening, close_bracket(mask, opening)


def methods(source, name):
    opening, closing = class_region(source, name)
    mask = dart_mask(source)
    pattern = r'(?m)^  Future<([^\n]+?)>\s+(\w+)(?:<[^>]+>)?\s*\('
    result = {}
    for match in re.finditer(pattern, mask[opening + 1:closing]):
        start = opening + 1 + match.start()
        paren = opening + match.end()
        end_params = close_bracket(mask, paren, '(', ')')
        suffix = re.match(r'\s*async\s*\{', mask[end_params + 1:])
        if suffix is None: continue
        body_start = end_params + 1 + suffix.end() - 1
        body_end = close_bracket(mask, body_start)
        require(match.group(2) not in result, 'Duplicate method: ' + match.group(2))
        result[match.group(2)] = dict(
            start=start, end=body_end + 1, header=source[start:body_start + 1],
            body=source[body_start + 1:body_end], params=source[paren + 1:end_params],
            returns=match.group(1), name=match.group(2),
        )
    return result


def edits(source, changes):
    for start, end, replacement in sorted(changes, reverse=True):
        source = source[:start] + replacement + source[end:]
    return source


def method_text(method, body):
    return method['header'] + '\n' + body.rstrip() + '\n  }'


def patch_elm(source):
    if '// CAN_V3_ADAPTER_BEGIN' in source:
        require('canV3Ready' in source and '_canV3Physical' in source, 'Partial CAN-V3 adapter detected.')
        return source
    found = methods(source, 'SsmElm')
    for name in ('ecuInit', 'readBytes', 'connect', 'disconnect'):
        require(name in found, 'Expected async SsmElm method missing: ' + name)
    require(found['ecuInit']['returns'] == 'String?' and not found['ecuInit']['params'].strip(),
            'Unexpected ecuInit signature; send can_v3_source_probe.txt.')
    require(found['readBytes']['returns'] == 'Uint8List?', 'Unexpected readBytes return type.')
    candidates = [m for m in found.values() if m['returns'] == 'String'
                  and re.search(r'\.output\s*\.\s*add\s*\(', dart_mask(m['body']))]
    require(len(candidates) == 1, 'Cannot identify one raw ELM send method without guessing. Send can_v3_source_probe.txt.')
    wire = candidates[0]
    require(not re.search(r'\b' + re.escape(wire['name']) + r'\s*\(', dart_mask(wire['body'])),
            'Raw sender is recursive; manual adaptation is required.')
    first = re.match(r'\s*String\s+(\w+)', wire['params'])
    require(first is not None, 'Expected a String command as the first wire parameter.')
    command_var = first.group(1)
    timeout = re.search(r'\b(int|Duration)\s+(timeoutMs|timeout)\s*=', wire['params'])
    require(timeout is not None and '{' in wire['params'], 'Expected a named timeout parameter in raw sender.')
    timeout_type, timeout_name = timeout.groups()
    passed_ms = timeout_name if timeout_type == 'int' else timeout_name + '.inMilliseconds'
    core_args = 'command, ' + timeout_name + (': ms' if timeout_type == 'int' else ': Duration(milliseconds: ms)')
    rest = wire['params'][first.end():]
    require(not re.search(r'\brequired\b', rest), 'Raw sender has unsupported required parameters.')
    for flattened in ("replaceAll('\\r', '')", "replaceAll('\\n', '')"):
        require(flattened not in wire['body'], 'Raw sender removes CAN frame boundaries. Send the source report.')

    old_init = found['ecuInit']['body']
    state_call = re.search(r'\b(\w+)\(SsmState\.ecuReady\);', dart_mask(old_init))
    state_assignment = re.search(r'\b(\w+)\s*=\s*SsmState\.ecuReady;', dart_mask(old_init))
    require(state_call is not None or state_assignment is not None, 'Cannot locate SsmState update without guessing.')
    ready = (state_call.group(1) + '(SsmState.ecuReady);') if state_call else (state_assignment.group(1) + ' = SsmState.ecuReady;')
    not_ready = ready.replace('SsmState.ecuReady', 'SsmState.elmReady')
    failed_state = ready.replace('SsmState.ecuReady', 'SsmState.error')
    a, b = class_region(source, 'SsmElm')
    ecu_id = re.search(r'\bString\s+(ecuId|_ecuId)\s*=', dart_mask(source[a:b]))
    require(ecu_id is not None, 'Cannot locate ECU-ID storage without guessing.')
    byte_params = re.findall(r'\bint\s+(\w+)', found['readBytes']['params'])
    require(len(byte_params) == 2, 'Expected readBytes(address, length).')

    physical_header = wire['header'].replace(wire['name'] + '(', '_canV3Physical(', 1)
    if physical_header == wire['header']:
        physical_header = re.sub(r'\b' + wire['name'] + r'\s*\(', '_canV3Physical(', wire['header'], count=1)
    wrapper = fr'''    final _canV3Command = {command_var}.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (_canV3Configuring) {{
      // Adapter setup may reset defaults, but no vehicle packet is allowed before ATSP6.
      if (!_canV3Command.startsWith('AT') ||
          _canV3Command.startsWith('ATIB') || _canV3Command == 'ATSI' || _canV3Command == 'ATFI') {{
        throw StateError('CAN-V3 rejected startup command: $_canV3Command');
      }}
      final _canV3Sent = _canV3Command.startsWith('ATSP') || _canV3Command.startsWith('ATTP')
          ? 'ATSP6' : _canV3Command.startsWith('ATSH') ? 'ATSH7E0' : _canV3Command;
      if (_canV3Sent != _canV3Command) _canV3.record('FORCED $_canV3Command -> $_canV3Sent');
      return _canV3.setupCommand(_canV3Sent, {passed_ms});
    }}
    return _canV3.terminalQuery({command_var}, {passed_ms});'''
    physical = physical_header + wire['body'] + '}\n'

    new_init = f'''    final st = can_settings.SettingsService.I;
    st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
    stTimeoutCode = st.stCode;
    await st.save();
    final probes = st.selectedPids
        .where((p) => p.len > 0 && p.len <= 4 && p.address >= 0 && p.address + p.len - 1 <= 0xFFFFFF)
        .take(4).map((p) => can_proto.CanV3Probe('${{p.id}}:byte0', p.address, 1)).toList();
    final ok = await _canV3.initialize(probes, stCode: stTimeoutCode);
    {ecu_id.group(1)} = ''; // A successful OBD probe is not an ECU hardware ID.
    if (!ok) {{ {not_ready} return null; }}
    {ready}
    return 'CAN-V3: SSM2 confirmed, hardware ID not read';'''
    new_read = f'''    final bytes = await _canV3.readBytes({byte_params[0]}, {byte_params[1]});
    if (bytes == null && !_canV3.ssmReady) {{ {failed_state} }}
    return bytes;'''
    new_connect = '''    _canV3Configuring = true;
    _canV3.invalidate('new Bluetooth connection');
    try {''' + found['connect']['body'] + '''
    } finally { _canV3Configuring = false; }'''
    new_disconnect = "    _canV3.invalidate('Bluetooth disconnect');\n" + found['disconnect']['body']
    changes = []
    for name, body in [('ecuInit', new_init), ('readBytes', new_read), ('connect', new_connect), ('disconnect', new_disconnect)]:
        m = found[name]
        changes.append((m['start'], m['end'], method_text(m, body)))
    changes.append((wire['start'], wire['end'], method_text(wire, wrapper) + '\n\n  ' + physical.lstrip()))
    source = edits(source, changes)
    opening, _ = class_region(source, 'SsmElm')
    members = f'''
  // CAN_V3_ADAPTER_BEGIN
  bool _canV3Configuring = false;
  late final can_proto.CanV3Session _canV3 = can_proto.CanV3Session(
      (command, ms) => _canV3Physical({core_args}));
  bool get canV3Ready => _canV3.ssmReady;
  String get canV3LastError => _canV3.lastError;
  String get canV3Diagnostic => _canV3.diagnostic;
  int get canV3ReadOk => _canV3.readOk;
  int get canV3ReadErrors => _canV3.readErrors;
  int get canV3NoData => _canV3.noData;
  double get canV3AvgReadMs => _canV3.averageReadMs;
  Future<void> canV3Idle() => _canV3.whenIdle();
  // CAN_V3_ADAPTER_END
'''
    source = source[:opening + 1] + members + source[opening + 1:]
    source = "import 'can_v3_session.dart' as can_proto;\nimport '../services/settings_service.dart' as can_settings;\n" + source
    poller = methods(source, 'SsmPoller')
    require('_tick' in poller, 'Expected SsmPoller._tick to stop on a CAN failure.')
    tick = poller['_tick']
    tick_body, tick_count = re.subn(
        r'(data\s*=\s*await\s+elm\.readBytes\([^;]+\);)',
        r'\1\n        if (!elm.canV3Ready) { stop(); return; }', tick['body'])
    require(tick_count == 1, 'Cannot safely add the poller CAN failure guard.')
    tick_body = '    if (!elm.canV3Ready) { stop(); return; }\n' + tick_body
    source = edits(source, [(tick['start'], tick['end'], method_text(tick, tick_body))])
    print('Detected ELM sender:', wire['name'], '| timeout:', timeout_type, timeout_name)
    print('Replaced ECU init and readBytes; all SSM address requests now use CAN ISO-TP.')
    return source


def patch_connection(source):
    if '// CAN_V3_CONNECTION' in source: return source
    found = methods(source, 'ConnectionService')
    require('connectAndInit' in found, 'ConnectionService.connectAndInit not found.')
    m = found['connectAndInit']
    body = r'''    // CAN_V3_CONNECTION
    if (connecting) return 'CAN-V3: подключение уже выполняется.';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выберите сопряжённый ELM327.';
    connecting = true;
    notifyListeners();
    try {
      stopPolling();
      await elm.canV3Idle();
      if (logger.logging) await logger.stop();
      await elm.disconnect();
      st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
      await st.save();
      elm.stTimeoutCode = st.stCode;
      if (!await elm.connect(st.btAddress)) return 'CAN-V3: Bluetooth/ELM setup failed. Check the selected adapter.';
      final initialized = await elm.ecuInit();
      if (initialized == null || !elm.canV3Ready) {
        final reason = elm.canV3LastError;
        await elm.disconnect();
        return 'CAN-V3: $reason';
      }
      buildPoller();
      // Do not immediately poll an unverified large PID library on a new transport.
      return 'OK: CAN 500 kbit/s, ATDPN=6, SSM2 подтверждён. '
          'ECU ID не считан. Проверьте PID CAN, затем нажмите СТАРТ ОПРОС.';
    } catch (e) {
      try { await elm.disconnect(); } catch (_) {}
      return 'CAN-V3: $e';
    } finally {
      connecting = false;
      notifyListeners();
    }'''
    source = edits(source, [(m['start'], m['end'], method_text(m, body))])
    # Replace a fixed sleep with an actual drain of the serialized CAN transaction.
    source, count = re.subn(
        r'await Future<void>\.delayed\(const Duration\(milliseconds: 180\)\);[^\n]*',
        'await elm.canV3Idle(); // Wait for a complete CAN transaction, not 180 ms.', source)
    require(count == 1 or 'await elm.canV3Idle(); // Wait' in source,
            'The original exclusive() synchronization differs. Send the source report.')
    source = source.replace('if (was) startPolling();', 'if (was && elm.canV3Ready) startPolling();')
    return source


PANEL = r'''          // CAN_V3_PANEL_BEGIN
          const Text('CAN-V3 | CAN 11-bit, 500 kbit/s | 7E0 / 7E8',
              style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          const Text('После подключения опрос запускается вручную. '
              'В терминале разрешены только ATI / ATDP / ATDPN / ATRV / AT@1.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextButton.icon(
            onPressed: _busy || _scanning ? null : () async {
              await Navigator.of(context).push(MaterialPageRoute(builder: (_) => const CanV3PidScreen()));
              if (mounted) setState(() {});
            },
            icon: const Icon(Icons.fact_check_outlined), label: const Text('PID CAN: выбранные параметры'),
          ),
          TextButton.icon(onPressed: () async {
            try {
              await Clipboard.setData(ClipboardData(text: svc.elm.canV3Diagnostic));
              if (mounted) ScaffoldMessenger.of(context).showSnackBar(
                const SnackBar(content: Text('Журнал CAN скопирован')));
            } catch (e) { if (mounted) setState(() => _msg = 'Ошибка копирования CAN: $e'); }
          }, icon: const Icon(Icons.copy), label: const Text('Копировать журнал CAN')),
          ExpansionTile(title: const Text('Журнал CAN-V3', style: TextStyle(fontSize: 12)),
            children: [SelectableText(svc.elm.canV3Diagnostic,
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace'))]),
          // CAN_V3_PANEL_END
'''


def patch_settings(source):
    require('BT-PERM-V2' in source, 'Apply the existing BT-PERM-V2 fix before CAN-V3.')
    if '// CAN_V3_PANEL_BEGIN' in source: return source
    anchor = "        _card('ПОДКЛЮЧЕНИЕ ELM327', [\n"
    require(source.count(anchor) == 1, 'Connection panel anchor not found.')
    source = source.replace(anchor, anchor + PANEL, 1)
    for statement in ("import 'can_v3_pid_screen.dart';", "import 'package:flutter/services.dart';"):
        if statement not in source: source = statement + '\n' + source
    source = re.sub(r"(st\.stCode\.toDouble\(\),\s*)(?:0x[\da-fA-F]+|\d+),\s*(?:0x[\da-fA-F]+|\d+),",
                    r'\g<1>0x32, 0xFF,', source)
    source = re.sub(r'st\.stCode = v\.round\(\)(?: & ~1)?;[^\n]*', 'st.stCode = v.round();', source)
    source = source.replace("_slider('AT ST (×4 мс ожидание байта)'", "_slider('AT ST (CAN; после переподключения)'")
    source = source.replace('stats.avgMs', 'svc.elm.canV3AvgReadMs')
    source = source.replace('stats.framesOk', 'svc.elm.canV3ReadOk')
    source = source.replace('stats.framesErr', 'svc.elm.canV3ReadErrors')
    source = source.replace('stats.noData', 'svc.elm.canV3NoData')
    source = source.replace('на CAN-кадр', 'на чтение A8')
    source = source.replace('Кадров OK / ошибок / NO DATA', 'Чтений A8 OK / ошибок / NO DATA')
    return source


def preflight_tests():
    sample = """class X {
  Future<String> raw(String command, {int timeoutMs = 700}) async {
    final s = '${1 + 2}'; // } ignored
    return s;
  }
}
"""
    parsed = methods(sample, 'X')
    require('raw' in parsed and parsed['raw']['params'].startswith('String command'), 'Dart source parser self-test failed.')
    require(sample[parsed['raw']['start']:parsed['raw']['end']].endswith('}'), 'Method boundary self-test failed.')


paths = {
    ROOT / 'lib/ssm/ssm_elm.dart': patch_elm,
    ROOT / 'lib/services/connection_service.dart': patch_connection,
    ROOT / 'lib/screens/settings_screen.dart': patch_settings,
}
try:
    preflight_tests()
    require(FLUTTER.is_file(), 'Flutter SDK is not available in this Colab session.')
    require((ROOT / '.dart_tool/package_config.json').is_file(), 'Run pub get/build cell setup in this project first.')
    originals = {path: path.read_text(encoding='utf-8') for path in paths}
    outputs = {path: transform(originals[path]) for path, transform in paths.items()}
    for path, transform in paths.items():
        require(transform(outputs[path]) == outputs[path], 'Idempotence check failed: ' + str(path))
except (ValueError, FileNotFoundError) as error:
    if ROOT.is_dir():
        report = ROOT / 'can_v3_source_probe.txt'
        report.write_text('\n\n'.join(str(p.relative_to(ROOT)) + '\n' + p.read_text(encoding='utf-8')
                                     for p in paths if p.is_file()), encoding='utf-8')
        print('Source report for manual adaptation:', report)
    raise SystemExit('Preflight failed; project sources were NOT changed: ' + str(error)) from error

outputs.update({
    ROOT / 'lib/ssm/can_v3_codec.dart': CODEC,
    ROOT / 'lib/ssm/can_v3_session.dart': SESSION,
    ROOT / 'lib/screens/can_v3_pid_screen.dart': PID_SCREEN,
    ROOT / 'test/can_v3_test.dart': TESTS,
})
changes = [(path, path.read_bytes() if path.exists() else None, text.encode('utf-8'))
           for path, text in outputs.items() if not path.exists() or path.read_text(encoding='utf-8') != text]
backup = ROOT / 'patch_backups' / ('can_v3_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for path, before, _ in changes:
    if before is not None:
        saved = backup / path.relative_to(ROOT)
        saved.parent.mkdir(parents=True, exist_ok=True)
        saved.write_bytes(before)


def rollback():
    for path, before, _ in changes:
        if before is None: path.unlink(missing_ok=True)
        else: path.write_bytes(before)


try:
    for path, _, after in changes:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(after)
    print('Running CAN codec, flow-control, session and application-link tests...')
    result = subprocess.run([str(FLUTTER), 'test', '--no-pub', 'test/can_v3_test.dart'],
                            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300)
    (ROOT / 'can_v3_test_output.txt').write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    require(result.returncode == 0, 'Tests failed. Send can_v3_test_output.txt.')
except (OSError, ValueError, subprocess.TimeoutExpired) as error:
    rollback()
    raise SystemExit('CAN-V3 was NOT installed; changed source files were rolled back. ' + str(error)) from error

print('CAN-V3 source patch and simulated tests passed IN THIS COLAB RUNTIME.')
print('Backups:', backup if changes else 'no changes (already applied)')
print('Now run the EXISTING cell 10/10 and install the NEW APK. Look for CAN-V3.')
print('OBD 4100 alone does not confirm SSM2. A valid E8 probe is required.')
print('No ECU hardware ID is invented. No global Subaru-CAN PID cache is reused.')
print('Polling starts manually. First verify a small selected PID set using PID CAN.')
print('Terminal protocol changes and arbitrary TX are blocked in this stabilization patch.')
print('Legacy DTC/service routines require separate CAN validation; do not assume they work.')
print('Real ELM clone, flow-control timing, CAN bus and ECU have NOT been tested by these tests.')

Running CAN codec, flow-control, session and application-link tests...
00:00 +0: loading /content/suba_run_v8/test/can_v3_test.dart
00:00 +0: Application API links against the patched SsmElm
00:00 +1: Exact SF and FF/CF for one and two A8 addresses
00:00 +2: Lengths 1..4095, padding removed, CF sequence wraps through zero
00:00 +3: 7E8 is a header, not a response SID; foreign frames and TX echo are not data
00:00 +4: Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail
00:00 +5: CAN-only initialization does not invent an ECU ID or confirm SSM from 4100
00:00 +6: Required AT commands are verified, timeout code is hexadecimal
00:00 +7: AT AL is requested; a padded-frame refusal falls back to unpadded single frames
00:00 +8: With AT AL the padded 8-byte frames stay in use
00:00 +9: MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction
00:00 +10: Concurrent reads are serialized; no data is returned after disconnect
00:00 +11: All tests passed!


In [ ]:
# @title SUBA RUN V8: CAN-V3 rev2 (ATAL + early ATDPN + CAF1 OBD fallback)
# Run LAST among fixes, before the existing build cell 10/10.
# Re-running upgrades an existing CAN-V3 install (idempotent: only CAN files change).
# Existing BT-PERM-V2 is required. This cell NEVER builds/flashes an ECU ROM.
from pathlib import Path
from datetime import datetime, timezone
import re
import subprocess

ROOT = Path('/content/suba_run_v8')
FLUTTER = Path('/content/flutter/bin/flutter')
CODEC = r'''import 'dart:typed_data';

class CanV3Exception implements Exception {
  final String code;
  final String message;
  const CanV3Exception(this.code, this.message);
  @override
  String toString() => '$code: $message';
}

class CanV3Flow {
  final int blockSize;
  final Duration separation;
  const CanV3Flow(this.blockSize, this.separation);
}

class CanV3Codec {
  static String hex(List<int> bytes) => bytes
      .map((b) => b.toRadixString(16).padLeft(2, '0').toUpperCase()).join();

  static void validateBytes(List<int> bytes) {
    if (bytes.any((b) => b < 0 || b > 255)) {
      throw const CanV3Exception('BAD_BYTE', 'Byte outside 0..255.');
    }
  }

  static List<int> addressRead(int address, int count) {
    if (address < 0 || address > 0xFFFFFF || count < 1 || count > 128 || address + count - 1 > 0xFFFFFF) {
      throw const CanV3Exception('BAD_RANGE', 'A8 requires a 24-bit address and 1..128 bytes.');
    }
    final payload = <int>[0xA8, 0x00];
    for (var offset = 0; offset < count; offset++) {
      final a = address + offset;
      payload.addAll([(a >> 16) & 255, (a >> 8) & 255, a & 255]);
    }
    return payload;
  }

  static List<int> _pad(List<int> bytes) => [...bytes, ...List.filled(8 - bytes.length, 0)];

  static List<List<int>> encode(List<int> payload) {
    validateBytes(payload);
    if (payload.isEmpty || payload.length > 4095) {
      throw const CanV3Exception('BAD_LENGTH', 'Classical CAN ISO-TP length must be 1..4095.');
    }
    if (payload.length <= 7) return [_pad([payload.length, ...payload])];
    final frames = <List<int>>[
      [0x10 | (payload.length >> 8), payload.length & 255, ...payload.take(6)],
    ];
    var offset = 6;
    var sequence = 1;
    while (offset < payload.length) {
      final end = (offset + 7 < payload.length) ? offset + 7 : payload.length;
      frames.add(_pad([0x20 | sequence, ...payload.sublist(offset, end)]));
      sequence = (sequence + 1) & 15;
      offset = end;
    }
    return frames;
  }

  static String? elmError(String reply) {
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.replaceAll(RegExp(r'\s+'), '');
      if (line.contains('NODATA')) return 'NO_DATA';
      if (line.contains('BUFFERFULL')) return 'BUFFER_FULL';
      if (line.contains('CANERROR') || line.contains('BUSERROR')) return 'CAN_ERROR';
      if (line.contains('UNABLETOCONNECT')) return 'UNABLE_TO_CONNECT';
      if (line.contains('STOPPED')) return 'STOPPED';
      if (line.contains('ERROR') || line == '?') return 'ELM_ERROR';
    }
    return null;
  }

  // Parse each physical line. Never search for E8 inside a header or payload.
  static List<List<int>> frames(String reply, {String request = '', int rxId = 0x7E8}) {
    final error = elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    final result = <List<int>>[];
    final echo = request.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final expected = rxId.toRadixString(16).padLeft(3, '0').toUpperCase();
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.trim();
      if (line.startsWith('SEARCHING...')) line = line.substring(12).trim();
      final compact = line.replaceAll(RegExp(r'\s+'), '');
      if (compact.isEmpty || compact == echo || compact == 'OK') continue;
      if (!compact.startsWith(expected)) {
        // Other 11-bit responders are filtered, not joined into this ECU response.
        if (RegExp(r'^[0-9A-F]{3}:?[0-9A-F]+$').hasMatch(compact) &&
            int.parse(compact.substring(0, 3), radix: 16) <= 0x7FF) continue;
        throw CanV3Exception('MALFORMED_LINE', 'Unexpected ELM line: $line');
      }
      var body = compact.substring(3);
      if (body.startsWith(':')) body = body.substring(1);
      int? dlc;
      if (body.length.isOdd && RegExp(r'^[0-8]').hasMatch(body)) {
        dlc = int.parse(body[0], radix: 16);
        body = body.substring(1);
      }
      if (body.isEmpty || body.length.isOdd || body.length > 16 || !RegExp(r'^[0-9A-F]+$').hasMatch(body)) {
        throw CanV3Exception('MALFORMED_FRAME', 'Invalid raw CAN frame: $line');
      }
      final bytes = <int>[
        for (var i = 0; i < body.length; i += 2) int.parse(body.substring(i, i + 2), radix: 16),
      ];
      if (dlc != null && bytes.length != dlc) {
        throw const CanV3Exception('DLC_MISMATCH', 'DLC does not match CAN data length.');
      }
      result.add(bytes);
    }
    if (result.isEmpty) throw const CanV3Exception('NO_FRAMES', 'No complete 7E8 raw CAN frames.');
    return result;
  }

  static List<Uint8List> assemble(List<List<int>> frames) {
    final messages = <Uint8List>[];
    List<int>? pending;
    var length = 0;
    var sequence = 1;
    for (final frame in frames) {
      validateBytes(frame);
      if (frame.isEmpty || frame.length > 8) throw const CanV3Exception('BAD_FRAME', 'CAN frame length.');
      final type = frame[0] >> 4;
      if (type == 0) {
        if (pending != null) throw const CanV3Exception('INTERRUPTED', 'SF interrupted a multi-frame response.');
        final n = frame[0] & 15;
        if (n < 1 || n > 7 || n > frame.length - 1) throw const CanV3Exception('SF_LENGTH', 'Invalid SF length.');
        messages.add(Uint8List.fromList(frame.sublist(1, n + 1)));
      } else if (type == 1) {
        if (pending != null || frame.length != 8) throw const CanV3Exception('BAD_FF', 'Unexpected or incomplete FF.');
        length = ((frame[0] & 15) << 8) | frame[1];
        if (length < 8 || length > 4095) throw const CanV3Exception('FF_LENGTH', 'Invalid FF length.');
        pending = frame.sublist(2);
        sequence = 1;
      } else if (type == 2) {
        if (pending == null || frame.length < 2) throw const CanV3Exception('ORPHAN_CF', 'CF without FF.');
        if ((frame[0] & 15) != sequence) throw const CanV3Exception('SEQUENCE', 'Wrong, duplicate or missing CF sequence.');
        sequence = (sequence + 1) & 15;
        final needed = length - pending.length;
        final available = frame.length - 1;
        if (available < needed && frame.length != 8) throw const CanV3Exception('SHORT_CF', 'Truncated CF.');
        pending.addAll(frame.skip(1).take(needed));
        if (pending.length == length) {
          messages.add(Uint8List.fromList(pending));
          pending = null;
        }
      } else {
        throw CanV3Exception('UNEXPECTED_PCI', 'Expected SF/FF/CF, got PCI ${hex([frame[0]])}.');
      }
    }
    if (pending != null) throw const CanV3Exception('INCOMPLETE', 'Response ended before declared ISO-TP length.');
    if (messages.isEmpty) throw const CanV3Exception('NO_PDU', 'No ISO-TP message.');
    return messages;
  }

  static CanV3Flow flow(List<List<int>> frames) {
    CanV3Flow? accepted;
    for (final bytes in frames) {
      if (bytes.length < 3 || bytes[0] >> 4 != 3) throw const CanV3Exception('EXPECTED_FC', 'No valid flow control.');
      final status = bytes[0] & 15;
      if (status == 1) continue;
      if (status == 2) throw const CanV3Exception('FC_OVERFLOW', 'ECU cannot accept the request.');
      if (status != 0 || accepted != null) throw const CanV3Exception('BAD_FC', 'Invalid or repeated CTS.');
      final st = bytes[2];
      if (st > 0x7F && (st < 0xF1 || st > 0xF9)) throw const CanV3Exception('STMIN', 'Reserved STmin value.');
      // A millisecond is a safe lower bound for sub-millisecond STmin on ELM serial.
      accepted = CanV3Flow(bytes[1], Duration(milliseconds: st <= 0x7F ? st : 1));
    }
    if (accepted == null) throw const CanV3Exception('FC_WAIT', 'WAIT received without CTS before the ELM prompt.');
    return accepted;
  }

  static Uint8List positive(List<Uint8List> messages, int requestSid, List<int> prefix, int length) {
    Uint8List? answer;
    CanV3Exception? negative;
    for (final bytes in messages) {
      if (bytes.length >= 2 && bytes[0] == 0x7F && bytes[1] == requestSid) {
        if (bytes.length != 3) throw const CanV3Exception('NEGATIVE_LENGTH', 'Invalid negative response length.');
        negative = CanV3Exception(bytes[2] == 0x78 ? 'PENDING' : 'NEGATIVE_RESPONSE',
            'SID ${hex([requestSid])}, NRC ${hex([bytes[2]])}.');
        if (bytes[2] != 0x78 || answer != null) throw negative;
        continue;
      }
      if (bytes.length < prefix.length || !List.generate(prefix.length, (i) => bytes[i] == prefix[i]).every((v) => v)) {
        throw CanV3Exception('WRONG_SID', 'Unexpected PDU ${hex(bytes)}.');
      }
      if (bytes.length != length) throw CanV3Exception('RESPONSE_LENGTH', 'Expected $length bytes; got ${bytes.length}.');
      if (answer != null) throw const CanV3Exception('AMBIGUOUS', 'Multiple positive responses for one request.');
      answer = bytes;
    }
    if (answer != null) return answer;
    throw negative ?? const CanV3Exception('NO_POSITIVE_RESPONSE', 'No matching positive response.');
  }
}'''
SESSION = r'''import 'dart:async';
import 'dart:typed_data';
import 'can_v3_codec.dart';

typedef CanV3Send = Future<String> Function(String command, int timeoutMs);

class CanV3Probe {
  final String id;
  final int address;
  final int length;
  const CanV3Probe(this.id, this.address, this.length);
}

class CanV3Session {
  final CanV3Send send;
  final Future<void> Function(Duration) delay;
  Future<void> _tail = Future<void>.value();
  final List<String> _trace = [];
  List<String> _initTrace = [];
  int _generation = 0;
  bool _responses = true;
  bool _poisoned = false;
  bool obdReady = false;
  bool ssmReady = false;
  bool atalAccepted = false;
  String obdVia = '';
  String protocolNumber = '';
  String adapterInfo = '';
  String lastError = '';
  String lastProbe = '';
  String lastProbeError = '';
  int timeoutCode = 0x32;
  int txFrames = 0;
  int readOk = 0;
  int readErrors = 0;
  int noData = 0;
  int _readMilliseconds = 0;
  double get averageReadMs => readOk == 0 ? 0 : _readMilliseconds / readOk;

  CanV3Session(this.send, {Future<void> Function(Duration)? delay})
      : delay = delay ?? ((d) => Future<void>.delayed(d));

  String get diagnostic => [
    'CAN-V3 rev2 | ISO 15765-4 / 11-bit / 500 kbit/s',
    'Required configuration: protocol 6, TX=7E0, RX=7E8, CAF=0, CFC=1, ATAL',
    'Last ATDPN reply in current connection: $protocolNumber',
    "Adapter: ${adapterInfo.isEmpty ? 'not captured' : adapterInfo}",
    "OBD=$obdReady via ${obdVia.isEmpty ? '-' : obdVia}; SSM2=$ssmReady; last probe=$lastProbe",
    'ATAL accepted=$atalAccepted',
    'ECU hardware ID: not read (no synthetic ID, no shared PID cache)',
    'AT ST=0x${timeoutCode.toRadixString(16).toUpperCase()}; nominal ${timeoutCode * 4} ms',
    'TX CAN frames=$txFrames; complete reads=$readOk; read errors=$readErrors; NO DATA=$noData',
    'Average complete A8 read: ${averageReadMs.toStringAsFixed(1)} ms (not time per physical frame)',
    'last error=$lastError',
    if (lastProbeError.isNotEmpty) 'last probe error=$lastProbeError',
    '--- INIT TRACE ---', ..._initTrace,
    '--- RECENT TRACE ---', ..._trace,
  ].join('\n');

  void record(String text) {
    _trace.add('${DateTime.now().toIso8601String()} $text');
    if (_trace.length > 240) _trace.removeRange(0, _trace.length - 240);
  }

  void invalidate(String reason) {
    _generation++;
    obdReady = false;
    ssmReady = false;
    protocolNumber = '';
    _poisoned = true;
    record('INVALIDATE $reason');
  }

  Future<T> _serial<T>(Future<T> Function() task) {
    final run = _tail.then((_) => task());
    _tail = run.then<void>((_) {}, onError: (Object _, StackTrace __) {});
    return run;
  }

  Future<void> whenIdle() => _tail;

  Future<String> setupCommand(String command, int timeoutMs) =>
      _serial(() => _wire(command, timeoutMs: timeoutMs));

  Future<String> _wire(String command, {int timeoutMs = 1500, bool responseRequired = true}) async {
    final generation = _generation;
    record('TX $command');
    try {
      final response = await send(command, timeoutMs).timeout(Duration(milliseconds: timeoutMs + 500));
      record('RX ${response.replaceAll('\r', '<CR>').replaceAll('\n', '<LF>')}');
      if (generation != _generation) throw const CanV3Exception('CANCELLED', 'Connection changed during request.');
      if (responseRequired && response.replaceAll('>', '').trim().isEmpty) {
        _poisoned = true;
        throw const CanV3Exception('EMPTY_REPLY', 'No response; reconnect before continuing.');
      }
      return response;
    } on TimeoutException {
      _poisoned = true;
      ssmReady = false;
      throw const CanV3Exception('WIRE_TIMEOUT', 'ELM prompt/command timeout. Reconnect to avoid a late reply.');
    }
  }

  Future<void> _ok(String command) async {
    final response = await _wire(command);
    final error = CanV3Codec.elmError(response);
    final lines = response.toUpperCase().replaceAll('>', '').split(RegExp(r'[\r\n]+'));
    if (error != null || !lines.any((line) => line.trim() == 'OK')) {
      throw CanV3Exception('AT_REJECTED', '$command -> ${response.trim()}');
    }
  }

  Future<bool> _tryOk(String command) async {
    try {
      await _ok(command);
      return true;
    } catch (e) {
      record('OPTIONAL $command FAILED $e');
      return false;
    }
  }

  Future<void> _setResponses(bool enabled) async {
    if (_responses == enabled) return;
    await _ok(enabled ? 'ATR1' : 'ATR0');
    _responses = enabled;
  }

  Future<String> _frame(List<int> bytes, {required bool expectResponse}) async {
    txFrames++;
    final reply = await _wire(CanV3Codec.hex(bytes), timeoutMs: 3000, responseRequired: expectResponse);
    final compact = reply.replaceAll(RegExp(r'\s+'), '').replaceAll('>', '').toUpperCase();
    if (compact == '?') {
      throw CanV3Exception('RAW_TX_REJECTED',
          'Adapter answered ? to raw frame ${CanV3Codec.hex(bytes)} without bus activity. '
          '8-byte CAF0 frames require ATAL; if ATAL was accepted, this clone cannot send raw CAN.');
    }
    final error = CanV3Codec.elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    return reply;
  }

  Future<List<Uint8List>> _exchange(List<int> payload) async {
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Transport is not synchronized.');
    final frames = CanV3Codec.encode(payload);
    try {
      await _setResponses(true);
      var reply = await _frame(frames.first, expectResponse: true);
      if (frames.length == 1) {
        return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      }
      var flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      var sentInBlock = 0;
      for (var i = 1; i < frames.length; i++) {
        final last = i == frames.length - 1;
        final boundary = flow.blockSize > 0 && sentInBlock + 1 == flow.blockSize;
        // Suppress response waits only for CFs that cannot require FC or a final reply.
        await _setResponses(last || boundary);
        await delay(flow.separation);
        reply = await _frame(frames[i], expectResponse: last || boundary);
        sentInBlock++;
        if (last) return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
        if (boundary) {
          flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
          sentInBlock = 0;
        }
      }
      throw const CanV3Exception('TX_INCOMPLETE', 'Not all CFs sent.');
    } catch (e) {
      // Single-frame local rejections leave no partial bus transaction: no poison,
      // so init can fall back and probes report precisely. Multi-frame aborts poison.
      final benign = e is CanV3Exception &&
          (e.code == 'NO_DATA' || e.code == 'RAW_TX_REJECTED' || e.code == 'ELM_ERROR');
      if (frames.length > 1 || !benign) {
        _poisoned = true;
        ssmReady = false;
        record('ISO-TP aborted; reconnect required.');
      }
      rethrow;
    } finally {
      if (!_responses && !_poisoned) {
        try { await _setResponses(true); }
        catch (e) { _poisoned = true; ssmReady = false; record('RESTORE ATR1 FAILED $e'); }
      }
    }
  }

  Future<Uint8List> _read(int address, int length) async {
    final reply = await _exchange(CanV3Codec.addressRead(address, length));
    try {
      final pdu = CanV3Codec.positive(reply, 0xA8, [0xE8], length + 1);
      return Uint8List.fromList(pdu.sublist(1));
    } on CanV3Exception catch (e) {
      if (e.code != 'NEGATIVE_RESPONSE') {
        _poisoned = true;
        ssmReady = false;
      }
      rethrow;
    }
  }

  Future<void> _captureProtocol({required bool requireSix}) async {
    final number = await _wire('ATDPN');
    protocolNumber = number.replaceAll(RegExp(r'[\s>]'), '').toUpperCase();
    record('ATDPN=$protocolNumber');
    if (requireSix && protocolNumber != '6') {
      throw CanV3Exception('WRONG_PROTOCOL', 'Expected fixed protocol 6, got $protocolNumber (A6 is auto).');
    }
  }

  Future<void> _captureAdapterInfo() async {
    final parts = <String>[];
    for (final query in ['ATI', 'ATRV']) {
      try {
        final reply = await _wire(query, responseRequired: false);
        final clean = reply.replaceAll(RegExp(r'\s+'), ' ').replaceAll('>', '').trim();
        if (clean.isNotEmpty) parts.add('$query=$clean');
      } catch (e) {
        record('$query FAILED $e');
      }
    }
    adapterInfo = parts.join('; ');
  }

  // V7-proven OBD check: temporary CAF1 + plain 0100, then restore raw mode.
  Future<void> _obdViaCaf1Fallback() async {
    record('FALLBACK OBD via CAF1 (V7 method) after raw TX rejection');
    await _ok('ATCAF1');
    final reply = await _wire('0100', timeoutMs: 3000);
    final compact = reply.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final error = CanV3Codec.elmError(reply);
    if (error != null || !compact.contains('4100')) {
      throw CanV3Exception('OBD_FALLBACK_FAILED', 'CAF1 0100 reply: ${reply.trim()}');
    }
    record('OBD 4100 confirmed via CAF1; restoring raw CAF0 mode');
    for (final command in [
      'ATH1', 'ATD0', 'ATCAF0',
      'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
      'ATR1',
    ]) { await _ok(command); }
    _responses = true;
    if (atalAccepted) atalAccepted = await _tryOk('ATAL');
  }

  Future<bool> initialize(List<CanV3Probe> probes, {int stCode = 0x32}) => _serial(() async {
    _generation++;
    _poisoned = false;
    obdReady = false;
    ssmReady = false;
    atalAccepted = false;
    obdVia = '';
    lastError = '';
    lastProbeError = '';
    protocolNumber = '';
    adapterInfo = '';
    lastProbe = '';
    timeoutCode = stCode.clamp(0x32, 0xFF).toInt();
    _trace.clear();
    txFrames = 0; readOk = 0; readErrors = 0; noData = 0; _readMilliseconds = 0;
    try {
      for (final command in [
        'ATE0', 'ATL0', 'ATS0', 'ATSP6', 'ATH1', 'ATD0',
      ]) { await _ok(command); }
      // Raw 8-byte CAN frames exceed the default 7-byte limit; without ATAL
      // clones answer ? to the transmission itself. Best effort on old firmware.
      atalAccepted = await _tryOk('ATAL');
      for (final command in [
        'ATCAF0',
        'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
        'ATR1', 'ATST${timeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase()}',
      ]) { await _ok(command); }
      _responses = true;
      await _captureProtocol(requireSix: true);
      await _captureAdapterInfo();
      try {
        final obd = await _exchange([0x01, 0x00]);
        CanV3Codec.positive(obd, 0x01, [0x41, 0x00], 6);
        obdVia = 'raw-caf0';
      } on CanV3Exception catch (e) {
        if (e.code != 'RAW_TX_REJECTED' && e.code != 'ELM_ERROR') rethrow;
        await _obdViaCaf1Fallback();
        obdVia = 'caf1-fallback';
      }
      obdReady = true;
      record('OBD CAN confirmed via $obdVia; SSM2 is not confirmed yet.');
      for (final probe in probes.take(4)) {
        try {
          await _read(probe.address, probe.length);
          lastProbe = probe.id;
          ssmReady = true;
          record('SSM2 confirmed: ${probe.id}, ${probe.length} bytes, exact E8 response.');
          return true;
        } catch (e) {
          lastProbeError = e.toString();
          record('PROBE ${probe.id} FAILED $e');
          if (_poisoned) rethrow;
        }
      }
      throw CanV3Exception('SSM_NOT_CONFIRMED', probes.isEmpty
          ? 'Select at least one PID before connecting.'
          : 'OBD CAN works, but selected A8 probes did not return valid E8 data. Last probe error: $lastProbeError');
    } catch (e) {
      ssmReady = false;
      lastError = e.toString();
      record('INIT FAILED $lastError');
      return false;
    } finally {
      _initTrace = List.of(_trace);
    }
  });

  Future<Uint8List?> readBytes(int address, int length) => _serial(() async {
    lastError = '';
    final clock = Stopwatch()..start();
    try {
      if (!ssmReady || _poisoned) throw const CanV3Exception('NOT_READY', 'Connect and confirm CAN/SSM2 first.');
      final result = await _read(address, length);
      readOk++;
      _readMilliseconds += clock.elapsedMilliseconds;
      return result;
    } catch (e) {
      readErrors++;
      if (e is CanV3Exception && e.code == 'NO_DATA') noData++;
      if (e is CanV3Exception && (e.code == 'RAW_TX_REJECTED' || e.code == 'ELM_ERROR')) {
        _poisoned = true;
        ssmReady = false;
        record('TX rejected mid-session; reconnect required.');
      }
      lastError = e.toString();
      record('READ FAILED $lastError');
      return null; // No partial bytes, padding or fabricated zeroes.
    }
  });

  Future<String> terminalQuery(String command, int timeoutMs) => _serial(() async {
    final normalized = command.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (!const ['ATI', 'ATDP', 'ATDPN', 'ATRV', 'AT@1'].contains(normalized)) {
      throw const CanV3Exception('MANUAL_TX_DISABLED', 'Use PID CAN for reads; arbitrary terminal TX is blocked.');
    }
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Reconnect before sending terminal queries.');
    return _wire(normalized, timeoutMs: timeoutMs);
  });
}
'''
TESTS = r'''import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import '../lib/ssm/can_v3_codec.dart';
import '../lib/ssm/can_v3_session.dart';
import '../lib/ssm/ssm_elm.dart';
import '../lib/main.dart' as application;

String raw(List<int> pdu) => CanV3Codec.encode(pdu)
    .map((f) => '7E8${CanV3Codec.hex(f)}\r').join() + '>';

class FakeElm {
  final commands = <String>[];
  final request = <List<int>>[];
  int required = 0;
  bool responses = true;
  bool cafRaw = true;
  String dpn = '6';
  bool rejectSsm = false;
  bool rejectRawTx = false;
  bool rejectAtal = false;
  int blockSize = 0;
  int inBlock = 0;

  Future<String> call(String command, int timeout) async {
    commands.add(command);
    if (command == 'ATDPN') return '$dpn\r>';
    if (command == 'ATI') return 'ELM327 v1.5\r>';
    if (command == 'ATRV') return '12.6V\r>';
    if (command == 'ATR0') responses = false;
    if (command == 'ATR1') responses = true;
    if (command == 'ATAL') return rejectAtal ? '?\r>' : 'OK\r>';
    if (command == 'ATCAF0') { cafRaw = true; return 'OK\r>'; }
    if (command == 'ATCAF1') { cafRaw = false; return 'OK\r>'; }
    if (command.startsWith('AT')) return 'OK\r>';
    if (!cafRaw) {
      if (command == '0100') return '7E8 06 41 00 BE 3F A8 13\r>';
      throw StateError('Unexpected CAF1 data command: $command');
    }
    if (rejectRawTx && command.length == 16) return '?\r>';
    final bytes = <int>[
      for (var i = 0; i < command.length; i += 2) int.parse(command.substring(i, i + 2), radix: 16),
    ];
    if (bytes[0] >> 4 == 0) return respond(CanV3Codec.assemble([bytes]).single);
    if (bytes[0] >> 4 == 1) {
      request.clear();
      request.add(bytes);
      required = ((bytes[0] & 15) << 8) | bytes[1];
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    request.add(bytes);
    inBlock++;
    final received = 6 + (request.length - 1) * 7;
    if (received >= required) {
      if (!responses) throw StateError('Final CF sent with responses disabled.');
      return respond(CanV3Codec.assemble(request).single);
    }
    if (blockSize > 0 && inBlock == blockSize) {
      if (!responses) throw StateError('FC boundary sent with responses disabled.');
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    if (responses) throw StateError('Intermediate CF unexpectedly waits for a response.');
    return '>';
  }

  String respond(List<int> payload) {
    if (payload.length == 2 && payload[0] == 1 && payload[1] == 0) return raw([0x41, 0, 0xBE, 0x3F, 0xA8, 0x13]);
    if (payload[0] != 0xA8 || payload[1] != 0 || (payload.length - 2) % 3 != 0) {
      throw StateError('Incorrect SSM payload: ${CanV3Codec.hex(payload)}');
    }
    if (rejectSsm) return raw([0x7F, 0xA8, 0x31]);
    return raw([0xE8, for (var i = 4; i < payload.length; i += 3) payload[i]]);
  }
}

void main() {
  test('Application API links against the patched SsmElm', () {
    // Referencing main forces compile-time checking of all imported screens/services.
    expect(application.main, isNotNull);
    final elm = SsmElm();
    expect(elm.canV3Ready, isFalse);
    expect(elm.canV3Diagnostic, contains('CAN-V3'));
  });

  test('Exact SF and FF/CF for one and two A8 addresses', () {
    expect(CanV3Codec.hex(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 1)).single), '05A80000000E0000');
    expect(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 2)).map(CanV3Codec.hex).toList(),
        ['1008A80000000E00', '21000F0000000000']);
    expect(CanV3Codec.hex(CanV3Codec.encode([1, 0]).single), '0201000000000000');
  });

  test('Lengths 1..4095, padding removed, CF sequence wraps through zero', () {
    for (final n in [1, 7, 8, 20, 80, 128, 242, 386, 4095]) {
      final data = List.generate(n, (i) => i & 255);
      final encoded = CanV3Codec.encode(data);
      expect(encoded.every((f) => f.length == 8), isTrue);
      expect(CanV3Codec.assemble(encoded).single, data);
    }
    expect(() => CanV3Codec.encode([]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.addressRead(0xFFFFFF, 2), throwsA(isA<CanV3Exception>()));
  });

  test('7E8 is a header, not a response SID; foreign frames and TX echo are not data', () {
    final reply = '05A80000000E0000\r7E9 02 E8 88 00 00 00 00 00\r7E8 02 E8 12 00 00 00 00 00\r>';
    final frames = CanV3Codec.frames(reply, request: '05A80000000E0000');
    expect(CanV3Codec.positive(CanV3Codec.assemble(frames), 0xA8, [0xE8], 2), [0xE8, 0x12]);
    expect(() => CanV3Codec.positive(CanV3Codec.assemble(CanV3Codec.frames('7E8 02 62 12 00 00 00 00 00\r>')), 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31])], 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>().having((e) => e.code, 'code', 'NEGATIVE_RESPONSE')));
    expect(CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x78]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), [0xE8, 1]);
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), throwsA(isA<CanV3Exception>()));
  });

  test('Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail', () {
    expect(() => CanV3Codec.frames('NO DATA\r>'), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([[6, 0xE8, 1]]), throwsA(isA<CanV3Exception>()));
    final full = CanV3Codec.encode(List.generate(25, (i) => i));
    expect(() => CanV3Codec.assemble(full.take(2).toList()), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[2]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[1], full[1]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x32, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x31, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(CanV3Codec.flow([[0x30, 3, 0xF5]]).separation.inMicroseconds, greaterThanOrEqualTo(500));
  });

  test('CAN-only initialization does not invent an ECU ID or confirm SSM from 4100', () async {
    final fake = FakeElm()..rejectSsm = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(fake.commands.contains('ATSP6'), isTrue);
    expect(fake.commands.any((c) => c.startsWith('8010') || c == 'ATSP0' || c == 'ATSP3' || c == 'ATSP5'), isFalse);
    final auto = CanV3Session((FakeElm()..dpn = 'A6').call, delay: (_) async {});
    expect(await auto.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(auto.lastError, contains('WRONG_PROTOCOL'));
  });

  test('Required AT commands are verified, timeout code is hexadecimal', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)], stCode: 8), isTrue);
    expect(fake.commands.contains('ATST32'), isTrue);
    expect(fake.commands.contains('ATST08'), isFalse);
    final bad = CanV3Session((command, timeout) async {
      if (command == 'ATFCSM1') return '?\r>';
      return await fake.call(command, timeout);
    }, delay: (_) async {});
    expect(await bad.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(bad.lastError, contains('ATFCSM1'));
    expect(bad.ssmReady, isFalse);
  });

  test('MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction', () async {
    for (final blockSize in [0, 1, 4]) {
      final fake = FakeElm()..blockSize = blockSize;
      final session = CanV3Session(fake.call, delay: (_) async {});
      expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
      expect(await session.readBytes(0xE, 4), [14, 15, 16, 17]);
      expect(await session.readBytes(0x100, 80), List.generate(80, (i) => i));
      expect(fake.responses, isTrue);
    }
  });

  test('Concurrent reads are serialized; no data is returned after disconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    final results = await Future.wait([session.readBytes(10, 4), session.readBytes(20, 4)]);
    expect(results, [[10, 11, 12, 13], [20, 21, 22, 23]]);
    session.invalidate('test disconnect');
    expect(await session.readBytes(10, 4), isNull);
  });

  test('rev2: ATAL sent, ATDPN captured before OBD, raw path recorded', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(fake.commands.contains('ATAL'), isTrue);
    expect(fake.commands.indexOf('ATDPN'), lessThan(fake.commands.indexOf('0201000000000000')));
    expect(session.obdVia, 'raw-caf0');
    expect(session.atalAccepted, isTrue);
    expect(session.protocolNumber, '6');
    expect(session.adapterInfo, contains('ELM327'));
  });

  test('rev2: bare ? on raw OBD retries via CAF1; persistent raw rejection fails SSM clearly', () async {
    final fake = FakeElm()..rejectRawTx = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('rpm', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(session.obdVia, 'caf1-fallback');
    expect(session.protocolNumber, '6');
    final caf1 = fake.commands.indexOf('ATCAF1');
    final obd = fake.commands.indexOf('0100');
    final restore = fake.commands.lastIndexOf('ATCAF0');
    expect(caf1 >= 0 && obd > caf1 && restore > obd, isTrue);
    expect(session.lastError, contains('SSM_NOT_CONFIRMED'));
    expect(session.lastError, contains('RAW_TX_REJECTED'));
  });

  test('rev2: rejected ATAL does not abort init when raw frames work', () async {
    final fake = FakeElm()..rejectAtal = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(session.atalAccepted, isFalse);
    expect(session.diagnostic, contains('OPTIONAL ATAL FAILED'));
  });

  test('rev2: raw rejection mid-session stops polling until reconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    fake.rejectRawTx = true;
    expect(await session.readBytes(0xE, 1), isNull);
    expect(session.ssmReady, isFalse);
    expect(session.lastError, contains('RAW_TX_REJECTED'));
  });
}
'''
PID_SCREEN = r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';

class CanV3PidScreen extends StatefulWidget {
  const CanV3PidScreen({super.key});
  @override
  State<CanV3PidScreen> createState() => _CanV3PidScreenState();
}

class _CanV3PidScreenState extends State<CanV3PidScreen> {
  final List<_CanPidResult> _results = [];
  bool _running = false;
  bool _cancel = false;
  int _total = 0;
  String _message = '';

  @override
  void dispose() {
    _cancel = true;
    super.dispose();
  }

  Future<void> _scan() async {
    final svc = ConnectionService.I;
    if (_running) return;
    if (!svc.elm.canV3Ready) {
      setState(() => _message = 'Сначала подтвердите CAN и SSM2 через CONNECT + INIT ECU.');
      return;
    }
    final pids = SettingsService.I.selectedPids.toList();
    if (pids.isEmpty) { setState(() => _message = 'Нет выбранных PID.'); return; }
    setState(() { _running = true; _cancel = false; _results.clear(); _total = pids.length; _message = ''; });
    try {
      await svc.exclusive((elm) async {
        await elm.canV3Idle();
        for (final pid in pids) {
          if (_cancel || !mounted || !elm.canV3Ready) break;
          final clock = Stopwatch()..start();
          final bytes = await elm.readBytes(pid.address, pid.len);
          double? value;
          var status = 'READ_ERROR';
          var detail = elm.canV3LastError;
          if (bytes != null && bytes.length == pid.len) {
            try {
              final decoded = pid.formula(bytes);
              if (!decoded.isFinite) {
                status = 'FORMULA_ERROR'; detail = 'Формула вернула NaN/Infinity.';
              } else {
                value = decoded; status = 'OK'; detail = 'E8 и длина проверены; физическая достоверность не подтверждена.';
              }
            } catch (e) { status = 'FORMULA_ERROR'; detail = e.toString(); }
          }
          clock.stop();
          final result = _CanPidResult(
            pid.id, pid.address, pid.len, status, value,
            bytes?.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ').toUpperCase() ?? '',
            clock.elapsedMilliseconds, detail,
          );
          if (!mounted) break;
          setState(() => _results.add(result));
        }
      });
      if (mounted) setState(() => _message = _cancel ? 'Проверка остановлена после текущего запроса.' : 'Проверка завершена. Кэш рабочих PID не изменён.');
    } catch (e) {
      if (mounted) setState(() => _message = 'Ошибка диагностики: $e');
    } finally {
      if (mounted) setState(() => _running = false);
    }
  }

  Future<void> _copy() async {
    final report = [
      'CAN-V3 selected PID diagnostics (not a calibration validation)',
      ..._results.map((r) => '${r.id} address=0x${r.address.toRadixString(16)} length=${r.length} '
          'status=${r.status} value=${r.value} ms=${r.ms} bytes=${r.bytes} detail=${r.detail}'),
      ConnectionService.I.elm.canV3Diagnostic,
    ].join('\n');
    try {
      await Clipboard.setData(ClipboardData(text: report));
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Отчёт CAN скопирован')));
    } catch (e) { if (mounted) setState(() => _message = 'Копирование не удалось: $e'); }
  }

  @override
  Widget build(BuildContext context) => Scaffold(
    appBar: AppBar(title: const Text('CAN-V3: выбранные PID'), actions: [
      IconButton(tooltip: 'Копировать отчёт', onPressed: _copy, icon: const Icon(Icons.copy)),
    ]),
    body: Column(children: [
      Padding(padding: const EdgeInsets.all(12), child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
        const Text('Проверяется текущий набор PID. Значение не считается физически достоверным только из-за ответа E8. '
            'Опрос приостанавливается на время проверки. Кэш по общему имени Subaru-CAN не создаётся.', style: TextStyle(fontSize: 12)),
        const SizedBox(height: 8),
        Text('${_results.length} / $_total; OK: ${_results.where((r) => r.status == 'OK').length}'),
        if (_running) LinearProgressIndicator(value: _total == 0 ? null : _results.length / _total),
        const SizedBox(height: 8),
        FilledButton(onPressed: _running ? null : _scan, child: const Text('Проверить выбранные PID')),
        if (_running) TextButton(onPressed: () => setState(() => _cancel = true),
            child: Text(_cancel ? 'Завершаем текущий запрос...' : 'Остановить')),
        if (_message.isNotEmpty) Padding(padding: const EdgeInsets.only(top: 8), child: Text(_message)),
      ])),
      Expanded(child: ListView.builder(itemCount: _results.length, itemBuilder: (context, index) {
        final r = _results[index];
        return ExpansionTile(
          title: Text('${r.id}: ${r.status}', style: TextStyle(color: r.status == 'OK' ? Colors.greenAccent : Colors.orangeAccent)),
          subtitle: Text('${r.ms} ms${r.value == null ? '' : ' | ${r.value}'}'),
          children: [Padding(padding: const EdgeInsets.all(12), child: SelectableText(
            'Address: 0x${r.address.toRadixString(16)}\nLength: ${r.length}\nBytes: ${r.bytes}\n${r.detail}',
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ))],
        );
      })),
    ]),
  );
}

class _CanPidResult {
  final String id;
  final int address;
  final int length;
  final String status;
  final double? value;
  final String bytes;
  final int ms;
  final String detail;
  _CanPidResult(this.id, this.address, this.length, this.status, this.value, this.bytes, this.ms, this.detail);
}'''


def require(ok, message):
    if not ok:
        raise ValueError(message)


def dart_mask(text):
    # Keep positions/newlines while hiding strings and comments for brace matching.
    result = list(text)

    def blank(a, b):
        for k in range(a, b):
            if result[k] != '\n':
                result[k] = ' '

    def string_end(start):
        raw = text[start] in 'rR' and start + 1 < len(text) and text[start + 1] in "'\""
        q = start + 1 if raw else start
        delimiter = text[q] * (3 if text.startswith(text[q] * 3, q) else 1)
        i = q + len(delimiter)
        while i < len(text):
            if text.startswith(delimiter, i):
                return i + len(delimiter)
            if not raw and text[i] == '\\':
                i += 2
            elif not raw and text.startswith('${', i):
                i = interpolation_end(i + 2)
            else:
                i += 1
        raise ValueError('Unterminated Dart string.')

    def interpolation_end(i):
        depth = 1
        while i < len(text):
            if text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
                i = string_end(i)
            elif text.startswith('//', i):
                end = text.find('\n', i)
                i = len(text) if end < 0 else end
            elif text.startswith('/*', i):
                end = text.find('*/', i + 2)
                require(end >= 0, 'Unterminated Dart comment.')
                i = end + 2
            else:
                if text[i] == '{': depth += 1
                if text[i] == '}': depth -= 1
                i += 1
                if depth == 0: return i
        raise ValueError('Unterminated Dart interpolation.')

    i = 0
    while i < len(text):
        if text.startswith('//', i):
            end = text.find('\n', i)
            end = len(text) if end < 0 else end
        elif text.startswith('/*', i):
            end = text.find('*/', i + 2)
            require(end >= 0, 'Unterminated Dart comment.')
            end += 2
        elif text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
            end = string_end(i)
        else:
            i += 1
            continue
        blank(i, end)
        i = end
    return ''.join(result)


def close_bracket(mask, index, left='{', right='}'):
    require(mask[index] == left, 'Expected opening bracket.')
    depth = 0
    for i in range(index, len(mask)):
        if mask[i] == left: depth += 1
        if mask[i] == right: depth -= 1
        if depth == 0: return i
    raise ValueError('Unbalanced Dart brackets.')


def class_region(source, name):
    mask = dart_mask(source)
    found = re.search(r'\bclass\s+' + re.escape(name) + r'\b[^\{]*\{', mask)
    require(found is not None, 'Class not found: ' + name)
    opening = found.end() - 1
    return opening, close_bracket(mask, opening)


def methods(source, name):
    opening, closing = class_region(source, name)
    mask = dart_mask(source)
    pattern = r'(?m)^  Future<([^\n]+?)>\s+(\w+)(?:<[^>]+>)?\s*\('
    result = {}
    for match in re.finditer(pattern, mask[opening + 1:closing]):
        start = opening + 1 + match.start()
        paren = opening + match.end()
        end_params = close_bracket(mask, paren, '(', ')')
        suffix = re.match(r'\s*async\s*\{', mask[end_params + 1:])
        if suffix is None: continue
        body_start = end_params + 1 + suffix.end() - 1
        body_end = close_bracket(mask, body_start)
        require(match.group(2) not in result, 'Duplicate method: ' + match.group(2))
        result[match.group(2)] = dict(
            start=start, end=body_end + 1, header=source[start:body_start + 1],
            body=source[body_start + 1:body_end], params=source[paren + 1:end_params],
            returns=match.group(1), name=match.group(2),
        )
    return result


def edits(source, changes):
    for start, end, replacement in sorted(changes, reverse=True):
        source = source[:start] + replacement + source[end:]
    return source


def method_text(method, body):
    return method['header'] + '\n' + body.rstrip() + '\n  }'


def patch_elm(source):
    if '// CAN_V3_ADAPTER_BEGIN' in source:
        require('canV3Ready' in source and '_canV3Physical' in source, 'Partial CAN-V3 adapter detected.')
        return source
    found = methods(source, 'SsmElm')
    for name in ('ecuInit', 'readBytes', 'connect', 'disconnect'):
        require(name in found, 'Expected async SsmElm method missing: ' + name)
    require(found['ecuInit']['returns'] == 'String?' and not found['ecuInit']['params'].strip(),
            'Unexpected ecuInit signature; send can_v3_source_probe.txt.')
    require(found['readBytes']['returns'] == 'Uint8List?', 'Unexpected readBytes return type.')
    candidates = [m for m in found.values() if m['returns'] == 'String'
                  and re.search(r'\.output\s*\.\s*add\s*\(', dart_mask(m['body']))]
    require(len(candidates) == 1, 'Cannot identify one raw ELM send method without guessing. Send can_v3_source_probe.txt.')
    wire = candidates[0]
    require(not re.search(r'\b' + re.escape(wire['name']) + r'\s*\(', dart_mask(wire['body'])),
            'Raw sender is recursive; manual adaptation is required.')
    first = re.match(r'\s*String\s+(\w+)', wire['params'])
    require(first is not None, 'Expected a String command as the first wire parameter.')
    command_var = first.group(1)
    timeout = re.search(r'\b(int|Duration)\s+(timeoutMs|timeout)\s*=', wire['params'])
    require(timeout is not None and '{' in wire['params'], 'Expected a named timeout parameter in raw sender.')
    timeout_type, timeout_name = timeout.groups()
    passed_ms = timeout_name if timeout_type == 'int' else timeout_name + '.inMilliseconds'
    core_args = 'command, ' + timeout_name + (': ms' if timeout_type == 'int' else ': Duration(milliseconds: ms)')
    rest = wire['params'][first.end():]
    require(not re.search(r'\brequired\b', rest), 'Raw sender has unsupported required parameters.')
    for flattened in ("replaceAll('\\r', '')", "replaceAll('\\n', '')"):
        require(flattened not in wire['body'], 'Raw sender removes CAN frame boundaries. Send the source report.')

    old_init = found['ecuInit']['body']
    state_call = re.search(r'\b(\w+)\(SsmState\.ecuReady\);', dart_mask(old_init))
    state_assignment = re.search(r'\b(\w+)\s*=\s*SsmState\.ecuReady;', dart_mask(old_init))
    require(state_call is not None or state_assignment is not None, 'Cannot locate SsmState update without guessing.')
    ready = (state_call.group(1) + '(SsmState.ecuReady);') if state_call else (state_assignment.group(1) + ' = SsmState.ecuReady;')
    not_ready = ready.replace('SsmState.ecuReady', 'SsmState.elmReady')
    failed_state = ready.replace('SsmState.ecuReady', 'SsmState.error')
    a, b = class_region(source, 'SsmElm')
    ecu_id = re.search(r'\bString\s+(ecuId|_ecuId)\s*=', dart_mask(source[a:b]))
    require(ecu_id is not None, 'Cannot locate ECU-ID storage without guessing.')
    byte_params = re.findall(r'\bint\s+(\w+)', found['readBytes']['params'])
    require(len(byte_params) == 2, 'Expected readBytes(address, length).')

    physical_header = wire['header'].replace(wire['name'] + '(', '_canV3Physical(', 1)
    if physical_header == wire['header']:
        physical_header = re.sub(r'\b' + wire['name'] + r'\s*\(', '_canV3Physical(', wire['header'], count=1)
    wrapper = fr'''    final _canV3Command = {command_var}.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (_canV3Configuring) {{
      // Adapter setup may reset defaults, but no vehicle packet is allowed before ATSP6.
      if (!_canV3Command.startsWith('AT') ||
          _canV3Command.startsWith('ATIB') || _canV3Command == 'ATSI' || _canV3Command == 'ATFI') {{
        throw StateError('CAN-V3 rejected startup command: $_canV3Command');
      }}
      final _canV3Sent = _canV3Command.startsWith('ATSP') || _canV3Command.startsWith('ATTP')
          ? 'ATSP6' : _canV3Command.startsWith('ATSH') ? 'ATSH7E0' : _canV3Command;
      if (_canV3Sent != _canV3Command) _canV3.record('FORCED $_canV3Command -> $_canV3Sent');
      return _canV3.setupCommand(_canV3Sent, {passed_ms});
    }}
    return _canV3.terminalQuery({command_var}, {passed_ms});'''
    physical = physical_header + wire['body'] + '}\n'

    new_init = f'''    final st = can_settings.SettingsService.I;
    st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
    stTimeoutCode = st.stCode;
    await st.save();
    final probes = st.selectedPids
        .where((p) => p.len > 0 && p.len <= 4 && p.address >= 0 && p.address + p.len - 1 <= 0xFFFFFF)
        .take(4).map((p) => can_proto.CanV3Probe('${{p.id}}:byte0', p.address, 1)).toList();
    final ok = await _canV3.initialize(probes, stCode: stTimeoutCode);
    {ecu_id.group(1)} = ''; // A successful OBD probe is not an ECU hardware ID.
    if (!ok) {{ {not_ready} return null; }}
    {ready}
    return 'CAN-V3: SSM2 confirmed, hardware ID not read';'''
    new_read = f'''    final bytes = await _canV3.readBytes({byte_params[0]}, {byte_params[1]});
    if (bytes == null && !_canV3.ssmReady) {{ {failed_state} }}
    return bytes;'''
    new_connect = '''    _canV3Configuring = true;
    _canV3.invalidate('new Bluetooth connection');
    try {''' + found['connect']['body'] + '''
    } finally { _canV3Configuring = false; }'''
    new_disconnect = "    _canV3.invalidate('Bluetooth disconnect');\n" + found['disconnect']['body']
    changes = []
    for name, body in [('ecuInit', new_init), ('readBytes', new_read), ('connect', new_connect), ('disconnect', new_disconnect)]:
        m = found[name]
        changes.append((m['start'], m['end'], method_text(m, body)))
    changes.append((wire['start'], wire['end'], method_text(wire, wrapper) + '\n\n  ' + physical.lstrip()))
    source = edits(source, changes)
    opening, _ = class_region(source, 'SsmElm')
    members = f'''
  // CAN_V3_ADAPTER_BEGIN
  bool _canV3Configuring = false;
  late final can_proto.CanV3Session _canV3 = can_proto.CanV3Session(
      (command, ms) => _canV3Physical({core_args}));
  bool get canV3Ready => _canV3.ssmReady;
  String get canV3LastError => _canV3.lastError;
  String get canV3Diagnostic => _canV3.diagnostic;
  int get canV3ReadOk => _canV3.readOk;
  int get canV3ReadErrors => _canV3.readErrors;
  int get canV3NoData => _canV3.noData;
  double get canV3AvgReadMs => _canV3.averageReadMs;
  Future<void> canV3Idle() => _canV3.whenIdle();
  // CAN_V3_ADAPTER_END
'''
    source = source[:opening + 1] + members + source[opening + 1:]
    source = "import 'can_v3_session.dart' as can_proto;\nimport '../services/settings_service.dart' as can_settings;\n" + source
    poller = methods(source, 'SsmPoller')
    require('_tick' in poller, 'Expected SsmPoller._tick to stop on a CAN failure.')
    tick = poller['_tick']
    tick_body, tick_count = re.subn(
        r'(data\s*=\s*await\s+elm\.readBytes\([^;]+\);)',
        r'\1\n        if (!elm.canV3Ready) { stop(); return; }', tick['body'])
    require(tick_count == 1, 'Cannot safely add the poller CAN failure guard.')
    tick_body = '    if (!elm.canV3Ready) { stop(); return; }\n' + tick_body
    source = edits(source, [(tick['start'], tick['end'], method_text(tick, tick_body))])
    print('Detected ELM sender:', wire['name'], '| timeout:', timeout_type, timeout_name)
    print('Replaced ECU init and readBytes; all SSM address requests now use CAN ISO-TP.')
    return source


def patch_connection(source):
    if '// CAN_V3_CONNECTION' in source: return source
    found = methods(source, 'ConnectionService')
    require('connectAndInit' in found, 'ConnectionService.connectAndInit not found.')
    m = found['connectAndInit']
    body = r'''    // CAN_V3_CONNECTION
    if (connecting) return 'CAN-V3: подключение уже выполняется.';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выберите сопряжённый ELM327.';
    connecting = true;
    notifyListeners();
    try {
      stopPolling();
      await elm.canV3Idle();
      if (logger.logging) await logger.stop();
      await elm.disconnect();
      st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
      await st.save();
      elm.stTimeoutCode = st.stCode;
      if (!await elm.connect(st.btAddress)) return 'CAN-V3: Bluetooth/ELM setup failed. Check the selected adapter.';
      final initialized = await elm.ecuInit();
      if (initialized == null || !elm.canV3Ready) {
        final reason = elm.canV3LastError;
        await elm.disconnect();
        return 'CAN-V3: $reason';
      }
      buildPoller();
      // Do not immediately poll an unverified large PID library on a new transport.
      return 'OK: CAN 500 kbit/s, ATDPN=6, SSM2 подтверждён. '
          'ECU ID не считан. Проверьте PID CAN, затем нажмите СТАРТ ОПРОС.';
    } catch (e) {
      try { await elm.disconnect(); } catch (_) {}
      return 'CAN-V3: $e';
    } finally {
      connecting = false;
      notifyListeners();
    }'''
    source = edits(source, [(m['start'], m['end'], method_text(m, body))])
    # Replace a fixed sleep with an actual drain of the serialized CAN transaction.
    source, count = re.subn(
        r'await Future<void>\.delayed\(const Duration\(milliseconds: 180\)\);[^\n]*',
        'await elm.canV3Idle(); // Wait for a complete CAN transaction, not 180 ms.', source)
    require(count == 1 or 'await elm.canV3Idle(); // Wait' in source,
            'The original exclusive() synchronization differs. Send the source report.')
    source = source.replace('if (was) startPolling();', 'if (was && elm.canV3Ready) startPolling();')
    return source


PANEL = r'''          // CAN_V3_PANEL_BEGIN
          const Text('CAN-V3 rev2 | CAN 11-bit, 500 kbit/s | 7E0 / 7E8',
              style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          const Text('После подключения опрос запускается вручную. '
              'В терминале разрешены только ATI / ATDP / ATDPN / ATRV / AT@1.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextButton.icon(
            onPressed: _busy || _scanning ? null : () async {
              await Navigator.of(context).push(MaterialPageRoute(builder: (_) => const CanV3PidScreen()));
              if (mounted) setState(() {});
            },
            icon: const Icon(Icons.fact_check_outlined), label: const Text('PID CAN: выбранные параметры'),
          ),
          TextButton.icon(onPressed: () async {
            try {
              await Clipboard.setData(ClipboardData(text: svc.elm.canV3Diagnostic));
              if (mounted) ScaffoldMessenger.of(context).showSnackBar(
                const SnackBar(content: Text('Журнал CAN скопирован')));
            } catch (e) { if (mounted) setState(() => _msg = 'Ошибка копирования CAN: $e'); }
          }, icon: const Icon(Icons.copy), label: const Text('Копировать журнал CAN')),
          ExpansionTile(title: const Text('Журнал CAN-V3', style: TextStyle(fontSize: 12)),
            children: [SelectableText(svc.elm.canV3Diagnostic,
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace'))]),
          // CAN_V3_PANEL_END
'''


def patch_settings(source):
    require('BT-PERM-V2' in source, 'Apply the existing BT-PERM-V2 fix before CAN-V3.')
    if '// CAN_V3_PANEL_BEGIN' in source: return source
    anchor = "        _card('ПОДКЛЮЧЕНИЕ ELM327', [\n"
    require(source.count(anchor) == 1, 'Connection panel anchor not found.')
    source = source.replace(anchor, anchor + PANEL, 1)
    for statement in ("import 'can_v3_pid_screen.dart';", "import 'package:flutter/services.dart';"):
        if statement not in source: source = statement + '\n' + source
    source = re.sub(r"(st\.stCode\.toDouble\(\),\s*)(?:0x[\da-fA-F]+|\d+),\s*(?:0x[\da-fA-F]+|\d+),",
                    r'\g<1>0x32, 0xFF,', source)
    source = re.sub(r'st\.stCode = v\.round\(\)(?: & ~1)?;[^\n]*', 'st.stCode = v.round();', source)
    source = source.replace("_slider('AT ST (×4 мс ожидание байта)'", "_slider('AT ST (CAN; после переподключения)'")
    source = source.replace('stats.avgMs', 'svc.elm.canV3AvgReadMs')
    source = source.replace('stats.framesOk', 'svc.elm.canV3ReadOk')
    source = source.replace('stats.framesErr', 'svc.elm.canV3ReadErrors')
    source = source.replace('stats.noData', 'svc.elm.canV3NoData')
    source = source.replace('на CAN-кадр', 'на чтение A8')
    source = source.replace('Кадров OK / ошибок / NO DATA', 'Чтений A8 OK / ошибок / NO DATA')
    return source


def preflight_tests():
    sample = """class X {
  Future<String> raw(String command, {int timeoutMs = 700}) async {
    final s = '${1 + 2}'; // } ignored
    return s;
  }
}
"""
    parsed = methods(sample, 'X')
    require('raw' in parsed and parsed['raw']['params'].startswith('String command'), 'Dart source parser self-test failed.')
    require(sample[parsed['raw']['start']:parsed['raw']['end']].endswith('}'), 'Method boundary self-test failed.')


paths = {
    ROOT / 'lib/ssm/ssm_elm.dart': patch_elm,
    ROOT / 'lib/services/connection_service.dart': patch_connection,
    ROOT / 'lib/screens/settings_screen.dart': patch_settings,
}
try:
    preflight_tests()
    require(FLUTTER.is_file(), 'Flutter SDK is not available in this Colab session.')
    require((ROOT / '.dart_tool/package_config.json').is_file(), 'Run pub get/build cell setup in this project first.')
    originals = {path: path.read_text(encoding='utf-8') for path in paths}
    outputs = {path: transform(originals[path]) for path, transform in paths.items()}
    for path, transform in paths.items():
        require(transform(outputs[path]) == outputs[path], 'Idempotence check failed: ' + str(path))
except (ValueError, FileNotFoundError) as error:
    if ROOT.is_dir():
        report = ROOT / 'can_v3_source_probe.txt'
        report.write_text('\n\n'.join(str(p.relative_to(ROOT)) + '\n' + p.read_text(encoding='utf-8')
                                     for p in paths if p.is_file()), encoding='utf-8')
        print('Source report for manual adaptation:', report)
    raise SystemExit('Preflight failed; project sources were NOT changed: ' + str(error)) from error

outputs.update({
    ROOT / 'lib/ssm/can_v3_codec.dart': CODEC,
    ROOT / 'lib/ssm/can_v3_session.dart': SESSION,
    ROOT / 'lib/screens/can_v3_pid_screen.dart': PID_SCREEN,
    ROOT / 'test/can_v3_test.dart': TESTS,
})
changes = [(path, path.read_bytes() if path.exists() else None, text.encode('utf-8'))
           for path, text in outputs.items() if not path.exists() or path.read_text(encoding='utf-8') != text]
backup = ROOT / 'patch_backups' / ('can_v3_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for path, before, _ in changes:
    if before is not None:
        saved = backup / path.relative_to(ROOT)
        saved.parent.mkdir(parents=True, exist_ok=True)
        saved.write_bytes(before)


def rollback():
    for path, before, _ in changes:
        if before is None: path.unlink(missing_ok=True)
        else: path.write_bytes(before)


try:
    for path, _, after in changes:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(after)
    print('Running CAN codec, flow-control, session and application-link tests...')
    result = subprocess.run([str(FLUTTER), 'test', '--no-pub', 'test/can_v3_test.dart'],
                            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300)
    (ROOT / 'can_v3_test_output.txt').write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    require(result.returncode == 0, 'Tests failed. Send can_v3_test_output.txt.')
except (OSError, ValueError, subprocess.TimeoutExpired) as error:
    rollback()
    raise SystemExit('CAN-V3 was NOT installed; changed source files were rolled back. ' + str(error)) from error

print('CAN-V3 rev2 source patch and simulated tests passed IN THIS COLAB RUNTIME.')
print('Backups:', backup if changes else 'no changes (already applied)')
print('Now run the EXISTING cell 10/10 and install the NEW APK. Look for CAN-V3 rev2.')
print('rev2 sends ATAL for 8-byte raw frames and captures ATDPN before OBD.')
print('If raw OBD is rejected with ?, OBD retries via CAF1+0100 (V7 method), then CAF0 is restored.')
print('OBD 4100 alone does not confirm SSM2. A valid E8 probe is required.')
print('No ECU hardware ID is invented. No global Subaru-CAN PID cache is reused.')
print('Polling starts manually. First verify a small selected PID set using PID CAN.')
print('Terminal protocol changes and arbitrary TX are blocked in this stabilization patch.')
print('Legacy DTC/service routines require separate CAN validation; do not assume they work.')
print('Real ELM clone, flow-control timing, CAN bus and ECU have NOT been tested by these tests.')

Running CAN codec, flow-control, session and application-link tests...
00:00 +0: loading /content/suba_run_v8/test/can_v3_test.dart
00:00 +0: Application API links against the patched SsmElm
00:00 +1: Exact SF and FF/CF for one and two A8 addresses
00:00 +2: Lengths 1..4095, padding removed, CF sequence wraps through zero
00:00 +3: 7E8 is a header, not a response SID; foreign frames and TX echo are not data
00:00 +4: Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail
00:00 +5: CAN-only initialization does not invent an ECU ID or confirm SSM from 4100
00:00 +6: Required AT commands are verified, timeout code is hexadecimal
00:00 +7: MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction
00:00 +8: Concurrent reads are serialized; no data is returned after disconnect
00:00 +9: rev2: ATAL sent, ATDPN captured before OBD, raw path recorded
00:00 +10: rev2: bare ? on raw OBD retries via CAF1; persistent raw rejection fails SSM clearly
00:00 +11:

In [ ]:
# @title SUBA RUN V8: CAN-V3 rev3 (split A8 blocks + do not poison on NO DATA)
# Run LAST among fixes, before the existing build cell 10/10.
# Re-running upgrades an existing CAN-V3 install (idempotent: only CAN files change).
# Existing BT-PERM-V2 is required. This cell NEVER builds/flashes an ECU ROM.
from pathlib import Path
from datetime import datetime, timezone
import re
import subprocess

ROOT = Path('/content/suba_run_v8')
FLUTTER = Path('/content/flutter/bin/flutter')
CODEC = r'''import 'dart:typed_data';

class CanV3Exception implements Exception {
  final String code;
  final String message;
  const CanV3Exception(this.code, this.message);
  @override
  String toString() => '$code: $message';
}

class CanV3Flow {
  final int blockSize;
  final Duration separation;
  const CanV3Flow(this.blockSize, this.separation);
}

class CanV3Codec {
  static String hex(List<int> bytes) => bytes
      .map((b) => b.toRadixString(16).padLeft(2, '0').toUpperCase()).join();

  static void validateBytes(List<int> bytes) {
    if (bytes.any((b) => b < 0 || b > 255)) {
      throw const CanV3Exception('BAD_BYTE', 'Byte outside 0..255.');
    }
  }

  static List<int> addressRead(int address, int count) {
    if (address < 0 || address > 0xFFFFFF || count < 1 || count > 128 || address + count - 1 > 0xFFFFFF) {
      throw const CanV3Exception('BAD_RANGE', 'A8 requires a 24-bit address and 1..128 bytes.');
    }
    final payload = <int>[0xA8, 0x00];
    for (var offset = 0; offset < count; offset++) {
      final a = address + offset;
      payload.addAll([(a >> 16) & 255, (a >> 8) & 255, a & 255]);
    }
    return payload;
  }

  static List<int> _pad(List<int> bytes) => [...bytes, ...List.filled(8 - bytes.length, 0)];

  static List<List<int>> encode(List<int> payload) {
    validateBytes(payload);
    if (payload.isEmpty || payload.length > 4095) {
      throw const CanV3Exception('BAD_LENGTH', 'Classical CAN ISO-TP length must be 1..4095.');
    }
    if (payload.length <= 7) return [_pad([payload.length, ...payload])];
    final frames = <List<int>>[
      [0x10 | (payload.length >> 8), payload.length & 255, ...payload.take(6)],
    ];
    var offset = 6;
    var sequence = 1;
    while (offset < payload.length) {
      final end = (offset + 7 < payload.length) ? offset + 7 : payload.length;
      frames.add(_pad([0x20 | sequence, ...payload.sublist(offset, end)]));
      sequence = (sequence + 1) & 15;
      offset = end;
    }
    return frames;
  }

  static String? elmError(String reply) {
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.replaceAll(RegExp(r'\s+'), '');
      if (line.contains('NODATA')) return 'NO_DATA';
      if (line.contains('BUFFERFULL')) return 'BUFFER_FULL';
      if (line.contains('CANERROR') || line.contains('BUSERROR')) return 'CAN_ERROR';
      if (line.contains('UNABLETOCONNECT')) return 'UNABLE_TO_CONNECT';
      if (line.contains('STOPPED')) return 'STOPPED';
      if (line.contains('ERROR') || line == '?') return 'ELM_ERROR';
    }
    return null;
  }

  // Parse each physical line. Never search for E8 inside a header or payload.
  static List<List<int>> frames(String reply, {String request = '', int rxId = 0x7E8}) {
    final error = elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    final result = <List<int>>[];
    final echo = request.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final expected = rxId.toRadixString(16).padLeft(3, '0').toUpperCase();
    for (var line in reply.toUpperCase().replaceAll('>', '\n').split(RegExp(r'[\r\n]+'))) {
      line = line.trim();
      if (line.startsWith('SEARCHING...')) line = line.substring(12).trim();
      final compact = line.replaceAll(RegExp(r'\s+'), '');
      if (compact.isEmpty || compact == echo || compact == 'OK') continue;
      if (!compact.startsWith(expected)) {
        // Other 11-bit responders are filtered, not joined into this ECU response.
        if (RegExp(r'^[0-9A-F]{3}:?[0-9A-F]+$').hasMatch(compact) &&
            int.parse(compact.substring(0, 3), radix: 16) <= 0x7FF) continue;
        throw CanV3Exception('MALFORMED_LINE', 'Unexpected ELM line: $line');
      }
      var body = compact.substring(3);
      if (body.startsWith(':')) body = body.substring(1);
      int? dlc;
      if (body.length.isOdd && RegExp(r'^[0-8]').hasMatch(body)) {
        dlc = int.parse(body[0], radix: 16);
        body = body.substring(1);
      }
      if (body.isEmpty || body.length.isOdd || body.length > 16 || !RegExp(r'^[0-9A-F]+$').hasMatch(body)) {
        throw CanV3Exception('MALFORMED_FRAME', 'Invalid raw CAN frame: $line');
      }
      final bytes = <int>[
        for (var i = 0; i < body.length; i += 2) int.parse(body.substring(i, i + 2), radix: 16),
      ];
      if (dlc != null && bytes.length != dlc) {
        throw const CanV3Exception('DLC_MISMATCH', 'DLC does not match CAN data length.');
      }
      result.add(bytes);
    }
    if (result.isEmpty) throw const CanV3Exception('NO_FRAMES', 'No complete 7E8 raw CAN frames.');
    return result;
  }

  static List<Uint8List> assemble(List<List<int>> frames) {
    final messages = <Uint8List>[];
    List<int>? pending;
    var length = 0;
    var sequence = 1;
    for (final frame in frames) {
      validateBytes(frame);
      if (frame.isEmpty || frame.length > 8) throw const CanV3Exception('BAD_FRAME', 'CAN frame length.');
      final type = frame[0] >> 4;
      if (type == 0) {
        if (pending != null) throw const CanV3Exception('INTERRUPTED', 'SF interrupted a multi-frame response.');
        final n = frame[0] & 15;
        if (n < 1 || n > 7 || n > frame.length - 1) throw const CanV3Exception('SF_LENGTH', 'Invalid SF length.');
        messages.add(Uint8List.fromList(frame.sublist(1, n + 1)));
      } else if (type == 1) {
        if (pending != null || frame.length != 8) throw const CanV3Exception('BAD_FF', 'Unexpected or incomplete FF.');
        length = ((frame[0] & 15) << 8) | frame[1];
        if (length < 8 || length > 4095) throw const CanV3Exception('FF_LENGTH', 'Invalid FF length.');
        pending = frame.sublist(2);
        sequence = 1;
      } else if (type == 2) {
        if (pending == null || frame.length < 2) throw const CanV3Exception('ORPHAN_CF', 'CF without FF.');
        if ((frame[0] & 15) != sequence) throw const CanV3Exception('SEQUENCE', 'Wrong, duplicate or missing CF sequence.');
        sequence = (sequence + 1) & 15;
        final needed = length - pending.length;
        final available = frame.length - 1;
        if (available < needed && frame.length != 8) throw const CanV3Exception('SHORT_CF', 'Truncated CF.');
        pending.addAll(frame.skip(1).take(needed));
        if (pending.length == length) {
          messages.add(Uint8List.fromList(pending));
          pending = null;
        }
      } else {
        throw CanV3Exception('UNEXPECTED_PCI', 'Expected SF/FF/CF, got PCI ${hex([frame[0]])}.');
      }
    }
    if (pending != null) throw const CanV3Exception('INCOMPLETE', 'Response ended before declared ISO-TP length.');
    if (messages.isEmpty) throw const CanV3Exception('NO_PDU', 'No ISO-TP message.');
    return messages;
  }

  static CanV3Flow flow(List<List<int>> frames) {
    CanV3Flow? accepted;
    for (final bytes in frames) {
      if (bytes.length < 3 || bytes[0] >> 4 != 3) throw const CanV3Exception('EXPECTED_FC', 'No valid flow control.');
      final status = bytes[0] & 15;
      if (status == 1) continue;
      if (status == 2) throw const CanV3Exception('FC_OVERFLOW', 'ECU cannot accept the request.');
      if (status != 0 || accepted != null) throw const CanV3Exception('BAD_FC', 'Invalid or repeated CTS.');
      final st = bytes[2];
      if (st > 0x7F && (st < 0xF1 || st > 0xF9)) throw const CanV3Exception('STMIN', 'Reserved STmin value.');
      // A millisecond is a safe lower bound for sub-millisecond STmin on ELM serial.
      accepted = CanV3Flow(bytes[1], Duration(milliseconds: st <= 0x7F ? st : 1));
    }
    if (accepted == null) throw const CanV3Exception('FC_WAIT', 'WAIT received without CTS before the ELM prompt.');
    return accepted;
  }

  static Uint8List positive(List<Uint8List> messages, int requestSid, List<int> prefix, int length) {
    Uint8List? answer;
    CanV3Exception? negative;
    for (final bytes in messages) {
      if (bytes.length >= 2 && bytes[0] == 0x7F && bytes[1] == requestSid) {
        if (bytes.length != 3) throw const CanV3Exception('NEGATIVE_LENGTH', 'Invalid negative response length.');
        negative = CanV3Exception(bytes[2] == 0x78 ? 'PENDING' : 'NEGATIVE_RESPONSE',
            'SID ${hex([requestSid])}, NRC ${hex([bytes[2]])}.');
        if (bytes[2] != 0x78 || answer != null) throw negative;
        continue;
      }
      if (bytes.length < prefix.length || !List.generate(prefix.length, (i) => bytes[i] == prefix[i]).every((v) => v)) {
        throw CanV3Exception('WRONG_SID', 'Unexpected PDU ${hex(bytes)}.');
      }
      if (bytes.length != length) throw CanV3Exception('RESPONSE_LENGTH', 'Expected $length bytes; got ${bytes.length}.');
      if (answer != null) throw const CanV3Exception('AMBIGUOUS', 'Multiple positive responses for one request.');
      answer = bytes;
    }
    if (answer != null) return answer;
    throw negative ?? const CanV3Exception('NO_POSITIVE_RESPONSE', 'No matching positive response.');
  }
}'''
SESSION = r'''import 'dart:async';
import 'dart:typed_data';
import 'can_v3_codec.dart';

typedef CanV3Send = Future<String> Function(String command, int timeoutMs);

class CanV3Probe {
  final String id;
  final int address;
  final int length;
  const CanV3Probe(this.id, this.address, this.length);
}

class CanV3Session {
  final CanV3Send send;
  final Future<void> Function(Duration) delay;
  Future<void> _tail = Future<void>.value();
  final List<String> _trace = [];
  List<String> _initTrace = [];
  int _generation = 0;
  bool _responses = true;
  bool _poisoned = false;
  bool obdReady = false;
  bool ssmReady = false;
  bool atalAccepted = false;
  String obdVia = '';
  String protocolNumber = '';
  String adapterInfo = '';
  String lastError = '';
  String lastProbe = '';
  String lastProbeError = '';
  int timeoutCode = 0x32;
  int txFrames = 0;
  int readOk = 0;
  int readErrors = 0;
  int noData = 0;
  int _readMilliseconds = 0;
  double get averageReadMs => readOk == 0 ? 0 : _readMilliseconds / readOk;

  CanV3Session(this.send, {Future<void> Function(Duration)? delay})
      : delay = delay ?? ((d) => Future<void>.delayed(d));

  String get diagnostic => [
    'CAN-V3 rev3 | ISO 15765-4 / 11-bit / 500 kbit/s',
    'Required configuration: protocol 6, TX=7E0, RX=7E8, CAF=0, CFC=1, ATAL',
    'Last ATDPN reply in current connection: $protocolNumber',
    "Adapter: ${adapterInfo.isEmpty ? 'not captured' : adapterInfo}",
    "OBD=$obdReady via ${obdVia.isEmpty ? '-' : obdVia}; SSM2=$ssmReady; last probe=$lastProbe",
    'ATAL accepted=$atalAccepted',
    'ECU hardware ID: not read (no synthetic ID, no shared PID cache)',
    'AT ST=0x${timeoutCode.toRadixString(16).toUpperCase()}; nominal ${timeoutCode * 4} ms',
    'TX CAN frames=$txFrames; complete reads=$readOk; read errors=$readErrors; NO DATA=$noData',
    'Average complete A8 read: ${averageReadMs.toStringAsFixed(1)} ms (not time per physical frame)',
    'last error=$lastError',
    if (lastProbeError.isNotEmpty) 'last probe error=$lastProbeError',
    '--- INIT TRACE ---', ..._initTrace,
    '--- RECENT TRACE ---', ..._trace,
  ].join('\n');

  void record(String text) {
    _trace.add('${DateTime.now().toIso8601String()} $text');
    if (_trace.length > 240) _trace.removeRange(0, _trace.length - 240);
  }

  void invalidate(String reason) {
    _generation++;
    obdReady = false;
    ssmReady = false;
    protocolNumber = '';
    _poisoned = true;
    record('INVALIDATE $reason');
  }

  Future<T> _serial<T>(Future<T> Function() task) {
    final run = _tail.then((_) => task());
    _tail = run.then<void>((_) {}, onError: (Object _, StackTrace __) {});
    return run;
  }

  Future<void> whenIdle() => _tail;

  Future<String> setupCommand(String command, int timeoutMs) =>
      _serial(() => _wire(command, timeoutMs: timeoutMs));

  Future<String> _wire(String command, {int timeoutMs = 1500, bool responseRequired = true}) async {
    final generation = _generation;
    record('TX $command');
    try {
      final response = await send(command, timeoutMs).timeout(Duration(milliseconds: timeoutMs + 500));
      record('RX ${response.replaceAll('\r', '<CR>').replaceAll('\n', '<LF>')}');
      if (generation != _generation) throw const CanV3Exception('CANCELLED', 'Connection changed during request.');
      if (responseRequired && response.replaceAll('>', '').trim().isEmpty) {
        _poisoned = true;
        throw const CanV3Exception('EMPTY_REPLY', 'No response; reconnect before continuing.');
      }
      return response;
    } on TimeoutException {
      _poisoned = true;
      ssmReady = false;
      throw const CanV3Exception('WIRE_TIMEOUT', 'ELM prompt/command timeout. Reconnect to avoid a late reply.');
    }
  }

  Future<void> _ok(String command) async {
    final response = await _wire(command);
    final error = CanV3Codec.elmError(response);
    final lines = response.toUpperCase().replaceAll('>', '').split(RegExp(r'[\r\n]+'));
    if (error != null || !lines.any((line) => line.trim() == 'OK')) {
      throw CanV3Exception('AT_REJECTED', '$command -> ${response.trim()}');
    }
  }

  Future<bool> _tryOk(String command) async {
    try {
      await _ok(command);
      return true;
    } catch (e) {
      record('OPTIONAL $command FAILED $e');
      return false;
    }
  }

  Future<void> _setResponses(bool enabled) async {
    if (_responses == enabled) return;
    await _ok(enabled ? 'ATR1' : 'ATR0');
    _responses = enabled;
  }

  Future<String> _frame(List<int> bytes, {required bool expectResponse}) async {
    txFrames++;
    final reply = await _wire(CanV3Codec.hex(bytes), timeoutMs: 3000, responseRequired: expectResponse);
    final compact = reply.replaceAll(RegExp(r'\s+'), '').replaceAll('>', '').toUpperCase();
    if (compact == '?') {
      throw CanV3Exception('RAW_TX_REJECTED',
          'Adapter answered ? to raw frame ${CanV3Codec.hex(bytes)} without bus activity. '
          '8-byte CAF0 frames require ATAL; if ATAL was accepted, this clone cannot send raw CAN.');
    }
    final error = CanV3Codec.elmError(reply);
    if (error != null) throw CanV3Exception(error, reply.trim());
    return reply;
  }

  Future<List<Uint8List>> _exchange(List<int> payload) async {
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Transport is not synchronized.');
    final frames = CanV3Codec.encode(payload);
    try {
      await _setResponses(true);
      var reply = await _frame(frames.first, expectResponse: true);
      if (frames.length == 1) {
        return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      }
      var flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames.first)));
      var sentInBlock = 0;
      for (var i = 1; i < frames.length; i++) {
        final last = i == frames.length - 1;
        final boundary = flow.blockSize > 0 && sentInBlock + 1 == flow.blockSize;
        // Suppress response waits only for CFs that cannot require FC or a final reply.
        await _setResponses(last || boundary);
        await delay(flow.separation);
        reply = await _frame(frames[i], expectResponse: last || boundary);
        sentInBlock++;
        if (last) return CanV3Codec.assemble(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
        if (boundary) {
          flow = CanV3Codec.flow(CanV3Codec.frames(reply, request: CanV3Codec.hex(frames[i])));
          sentInBlock = 0;
        }
      }
      throw const CanV3Exception('TX_INCOMPLETE', 'Not all CFs sent.');
    } catch (e) {
      // NO DATA on one request is not a desync: ATR1 is restored in finally.
      // Sequence errors / truncated CF still poison. Do not drop a working SSM2
      // session after a single oversized block times out.
      final benign = e is CanV3Exception &&
          (e.code == 'NO_DATA' || e.code == 'RAW_TX_REJECTED' || e.code == 'ELM_ERROR'
              || e.code == 'NEGATIVE_RESPONSE');
      if (!benign) {
        _poisoned = true;
        ssmReady = false;
        record('ISO-TP aborted; reconnect required.');
      } else {
        record('Request failed without session poison: $e');
      }
      rethrow;
    } finally {
      if (!_responses && !_poisoned) {
        try { await _setResponses(true); }
        catch (e) { _poisoned = true; ssmReady = false; record('RESTORE ATR1 FAILED $e'); }
      }
    }
  }

  // FF holds 6 payload bytes, one CF holds 7 → max 13 bytes A8 payload.
  // A8 00 + 3 addresses = 11 bytes → FF + 1 CF (proven on this clone).
  // 4 addresses = 14 bytes → 2 CFs; ATR0/ATR1 between them exceeds ISO-TP N_Cr.
  static const int maxAddressesPerA8 = 3;

  Future<Uint8List> _readChunk(int address, int length) async {
    final reply = await _exchange(CanV3Codec.addressRead(address, length));
    final pdu = CanV3Codec.positive(reply, 0xA8, [0xE8], length + 1);
    return Uint8List.fromList(pdu.sublist(1));
  }

  Future<Uint8List> _read(int address, int length) async {
    if (length <= maxAddressesPerA8) return _readChunk(address, length);
    final out = BytesBuilder(copy: false);
    var remaining = length;
    var addr = address;
    while (remaining > 0) {
      final n = remaining > maxAddressesPerA8 ? maxAddressesPerA8 : remaining;
      out.add(await _readChunk(addr, n));
      addr += n;
      remaining -= n;
    }
    return Uint8List.fromList(out.takeBytes());
  }

  Future<void> _captureProtocol({required bool requireSix}) async {
    final number = await _wire('ATDPN');
    protocolNumber = number.replaceAll(RegExp(r'[\s>]'), '').toUpperCase();
    record('ATDPN=$protocolNumber');
    if (requireSix && protocolNumber != '6') {
      throw CanV3Exception('WRONG_PROTOCOL', 'Expected fixed protocol 6, got $protocolNumber (A6 is auto).');
    }
  }

  Future<void> _captureAdapterInfo() async {
    final parts = <String>[];
    for (final query in ['ATI', 'ATRV']) {
      try {
        final reply = await _wire(query, responseRequired: false);
        final clean = reply.replaceAll(RegExp(r'\s+'), ' ').replaceAll('>', '').trim();
        if (clean.isNotEmpty) parts.add('$query=$clean');
      } catch (e) {
        record('$query FAILED $e');
      }
    }
    adapterInfo = parts.join('; ');
  }

  // V7-proven OBD check: temporary CAF1 + plain 0100, then restore raw mode.
  Future<void> _obdViaCaf1Fallback() async {
    record('FALLBACK OBD via CAF1 (V7 method) after raw TX rejection');
    await _ok('ATCAF1');
    final reply = await _wire('0100', timeoutMs: 3000);
    final compact = reply.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    final error = CanV3Codec.elmError(reply);
    if (error != null || !compact.contains('4100')) {
      throw CanV3Exception('OBD_FALLBACK_FAILED', 'CAF1 0100 reply: ${reply.trim()}');
    }
    record('OBD 4100 confirmed via CAF1; restoring raw CAF0 mode');
    for (final command in [
      'ATH1', 'ATD0', 'ATCAF0',
      'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
      'ATR1',
    ]) { await _ok(command); }
    _responses = true;
    if (atalAccepted) atalAccepted = await _tryOk('ATAL');
  }

  Future<bool> initialize(List<CanV3Probe> probes, {int stCode = 0x32}) => _serial(() async {
    _generation++;
    _poisoned = false;
    obdReady = false;
    ssmReady = false;
    atalAccepted = false;
    obdVia = '';
    lastError = '';
    lastProbeError = '';
    protocolNumber = '';
    adapterInfo = '';
    lastProbe = '';
    timeoutCode = stCode.clamp(0x32, 0xFF).toInt();
    _trace.clear();
    txFrames = 0; readOk = 0; readErrors = 0; noData = 0; _readMilliseconds = 0;
    try {
      for (final command in [
        'ATE0', 'ATL0', 'ATS0', 'ATSP6', 'ATH1', 'ATD0',
      ]) { await _ok(command); }
      // Raw 8-byte CAN frames exceed the default 7-byte limit; without ATAL
      // clones answer ? to the transmission itself. Best effort on old firmware.
      atalAccepted = await _tryOk('ATAL');
      for (final command in [
        'ATCAF0',
        'ATSH7E0', 'ATCRA7E8', 'ATCFC1', 'ATFCSH7E0', 'ATFCSD300000', 'ATFCSM1',
        'ATR1', 'ATST${timeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase()}',
      ]) { await _ok(command); }
      _responses = true;
      await _captureProtocol(requireSix: true);
      await _captureAdapterInfo();
      try {
        final obd = await _exchange([0x01, 0x00]);
        CanV3Codec.positive(obd, 0x01, [0x41, 0x00], 6);
        obdVia = 'raw-caf0';
      } on CanV3Exception catch (e) {
        if (e.code != 'RAW_TX_REJECTED' && e.code != 'ELM_ERROR') rethrow;
        await _obdViaCaf1Fallback();
        obdVia = 'caf1-fallback';
      }
      obdReady = true;
      record('OBD CAN confirmed via $obdVia; SSM2 is not confirmed yet.');
      for (final probe in probes.take(4)) {
        try {
          await _read(probe.address, probe.length);
          lastProbe = probe.id;
          ssmReady = true;
          record('SSM2 confirmed: ${probe.id}, ${probe.length} bytes, exact E8 response.');
          return true;
        } catch (e) {
          lastProbeError = e.toString();
          record('PROBE ${probe.id} FAILED $e');
          if (_poisoned) rethrow;
        }
      }
      throw CanV3Exception('SSM_NOT_CONFIRMED', probes.isEmpty
          ? 'Select at least one PID before connecting.'
          : 'OBD CAN works, but selected A8 probes did not return valid E8 data. Last probe error: $lastProbeError');
    } catch (e) {
      ssmReady = false;
      lastError = e.toString();
      record('INIT FAILED $lastError');
      return false;
    } finally {
      _initTrace = List.of(_trace);
    }
  });

  Future<Uint8List?> readBytes(int address, int length) => _serial(() async {
    lastError = '';
    final clock = Stopwatch()..start();
    try {
      if (!ssmReady || _poisoned) throw const CanV3Exception('NOT_READY', 'Connect and confirm CAN/SSM2 first.');
      final result = await _read(address, length);
      readOk++;
      _readMilliseconds += clock.elapsedMilliseconds;
      return result;
    } catch (e) {
      readErrors++;
      if (e is CanV3Exception && e.code == 'NO_DATA') noData++;
      if (e is CanV3Exception && (e.code == 'RAW_TX_REJECTED' || e.code == 'ELM_ERROR')) {
        _poisoned = true;
        ssmReady = false;
        record('TX rejected mid-session; reconnect required.');
      }
      lastError = e.toString();
      record('READ FAILED $lastError');
      return null; // No partial bytes, padding or fabricated zeroes.
    }
  });

  Future<String> terminalQuery(String command, int timeoutMs) => _serial(() async {
    final normalized = command.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (!const ['ATI', 'ATDP', 'ATDPN', 'ATRV', 'AT@1'].contains(normalized)) {
      throw const CanV3Exception('MANUAL_TX_DISABLED', 'Use PID CAN for reads; arbitrary terminal TX is blocked.');
    }
    if (_poisoned) throw const CanV3Exception('RECONNECT_REQUIRED', 'Reconnect before sending terminal queries.');
    return _wire(normalized, timeoutMs: timeoutMs);
  });
}
'''
TESTS = r'''import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import '../lib/ssm/can_v3_codec.dart';
import '../lib/ssm/can_v3_session.dart';
import '../lib/ssm/ssm_elm.dart';
import '../lib/main.dart' as application;

String raw(List<int> pdu) => CanV3Codec.encode(pdu)
    .map((f) => '7E8${CanV3Codec.hex(f)}\r').join() + '>';

class FakeElm {
  final commands = <String>[];
  final request = <List<int>>[];
  int required = 0;
  bool responses = true;
  bool cafRaw = true;
  String dpn = '6';
  bool rejectSsm = false;
  bool rejectRawTx = false;
  bool rejectAtal = false;
  bool noDataNext = false;
  int blockSize = 0;
  int inBlock = 0;

  Future<String> call(String command, int timeout) async {
    commands.add(command);
    if (command == 'ATDPN') return '$dpn\r>';
    if (command == 'ATI') return 'ELM327 v1.5\r>';
    if (command == 'ATRV') return '12.6V\r>';
    if (command == 'ATR0') responses = false;
    if (command == 'ATR1') responses = true;
    if (command == 'ATAL') return rejectAtal ? '?\r>' : 'OK\r>';
    if (command == 'ATCAF0') { cafRaw = true; return 'OK\r>'; }
    if (command == 'ATCAF1') { cafRaw = false; return 'OK\r>'; }
    if (command.startsWith('AT')) return 'OK\r>';
    if (noDataNext && command.length >= 4 && !command.startsWith('AT')) {
      noDataNext = false;
      return 'NO DATA\r>';
    }
    if (!cafRaw) {
      if (command == '0100') return '7E8 06 41 00 BE 3F A8 13\r>';
      throw StateError('Unexpected CAF1 data command: $command');
    }
    if (rejectRawTx && command.length == 16) return '?\r>';
    final bytes = <int>[
      for (var i = 0; i < command.length; i += 2) int.parse(command.substring(i, i + 2), radix: 16),
    ];
    if (bytes[0] >> 4 == 0) return respond(CanV3Codec.assemble([bytes]).single);
    if (bytes[0] >> 4 == 1) {
      request.clear();
      request.add(bytes);
      required = ((bytes[0] & 15) << 8) | bytes[1];
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    request.add(bytes);
    inBlock++;
    final received = 6 + (request.length - 1) * 7;
    if (received >= required) {
      if (!responses) throw StateError('Final CF sent with responses disabled.');
      return respond(CanV3Codec.assemble(request).single);
    }
    if (blockSize > 0 && inBlock == blockSize) {
      if (!responses) throw StateError('FC boundary sent with responses disabled.');
      inBlock = 0;
      return '7E8${CanV3Codec.hex([0x30, blockSize, 0, 0, 0, 0, 0, 0])}\r>';
    }
    if (responses) throw StateError('Intermediate CF unexpectedly waits for a response.');
    return '>';
  }

  String respond(List<int> payload) {
    if (payload.length == 2 && payload[0] == 1 && payload[1] == 0) return raw([0x41, 0, 0xBE, 0x3F, 0xA8, 0x13]);
    if (payload[0] != 0xA8 || payload[1] != 0 || (payload.length - 2) % 3 != 0) {
      throw StateError('Incorrect SSM payload: ${CanV3Codec.hex(payload)}');
    }
    if (rejectSsm) return raw([0x7F, 0xA8, 0x31]);
    return raw([0xE8, for (var i = 4; i < payload.length; i += 3) payload[i]]);
  }
}

void main() {
  test('Application API links against the patched SsmElm', () {
    // Referencing main forces compile-time checking of all imported screens/services.
    expect(application.main, isNotNull);
    final elm = SsmElm();
    expect(elm.canV3Ready, isFalse);
    expect(elm.canV3Diagnostic, contains('CAN-V3'));
  });

  test('Exact SF and FF/CF for one and two A8 addresses', () {
    expect(CanV3Codec.hex(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 1)).single), '05A80000000E0000');
    expect(CanV3Codec.encode(CanV3Codec.addressRead(0xE, 2)).map(CanV3Codec.hex).toList(),
        ['1008A80000000E00', '21000F0000000000']);
    expect(CanV3Codec.hex(CanV3Codec.encode([1, 0]).single), '0201000000000000');
  });

  test('Lengths 1..4095, padding removed, CF sequence wraps through zero', () {
    for (final n in [1, 7, 8, 20, 80, 128, 242, 386, 4095]) {
      final data = List.generate(n, (i) => i & 255);
      final encoded = CanV3Codec.encode(data);
      expect(encoded.every((f) => f.length == 8), isTrue);
      expect(CanV3Codec.assemble(encoded).single, data);
    }
    expect(() => CanV3Codec.encode([]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.addressRead(0xFFFFFF, 2), throwsA(isA<CanV3Exception>()));
  });

  test('7E8 is a header, not a response SID; foreign frames and TX echo are not data', () {
    final reply = '05A80000000E0000\r7E9 02 E8 88 00 00 00 00 00\r7E8 02 E8 12 00 00 00 00 00\r>';
    final frames = CanV3Codec.frames(reply, request: '05A80000000E0000');
    expect(CanV3Codec.positive(CanV3Codec.assemble(frames), 0xA8, [0xE8], 2), [0xE8, 0x12]);
    expect(() => CanV3Codec.positive(CanV3Codec.assemble(CanV3Codec.frames('7E8 02 62 12 00 00 00 00 00\r>')), 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31])], 0xA8, [0xE8], 2),
        throwsA(isA<CanV3Exception>().having((e) => e.code, 'code', 'NEGATIVE_RESPONSE')));
    expect(CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x78]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), [0xE8, 1]);
    expect(() => CanV3Codec.positive([Uint8List.fromList([0x7F, 0xA8, 0x31]), Uint8List.fromList([0xE8, 1])],
        0xA8, [0xE8], 2), throwsA(isA<CanV3Exception>()));
  });

  test('Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail', () {
    expect(() => CanV3Codec.frames('NO DATA\r>'), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([[6, 0xE8, 1]]), throwsA(isA<CanV3Exception>()));
    final full = CanV3Codec.encode(List.generate(25, (i) => i));
    expect(() => CanV3Codec.assemble(full.take(2).toList()), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[2]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.assemble([full.first, full[1], full[1]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x32, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(() => CanV3Codec.flow([[0x31, 0, 0]]), throwsA(isA<CanV3Exception>()));
    expect(CanV3Codec.flow([[0x30, 3, 0xF5]]).separation.inMicroseconds, greaterThanOrEqualTo(500));
  });

  test('CAN-only initialization does not invent an ECU ID or confirm SSM from 4100', () async {
    final fake = FakeElm()..rejectSsm = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(fake.commands.contains('ATSP6'), isTrue);
    expect(fake.commands.any((c) => c.startsWith('8010') || c == 'ATSP0' || c == 'ATSP3' || c == 'ATSP5'), isFalse);
    final auto = CanV3Session((FakeElm()..dpn = 'A6').call, delay: (_) async {});
    expect(await auto.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(auto.lastError, contains('WRONG_PROTOCOL'));
  });

  test('Required AT commands are verified, timeout code is hexadecimal', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)], stCode: 8), isTrue);
    expect(fake.commands.contains('ATST32'), isTrue);
    expect(fake.commands.contains('ATST08'), isFalse);
    final bad = CanV3Session((command, timeout) async {
      if (command == 'ATFCSM1') return '?\r>';
      return await fake.call(command, timeout);
    }, delay: (_) async {});
    expect(await bad.initialize([const CanV3Probe('test', 0xE, 1)]), isFalse);
    expect(bad.lastError, contains('ATFCSM1'));
    expect(bad.ssmReady, isFalse);
  });

  test('MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction', () async {
    for (final blockSize in [0, 1, 4]) {
      final fake = FakeElm()..blockSize = blockSize;
      final session = CanV3Session(fake.call, delay: (_) async {});
      expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
      expect(await session.readBytes(0xE, 4), [14, 15, 16, 17]);
      expect(await session.readBytes(0x100, 80), List.generate(80, (i) => i));
      expect(fake.responses, isTrue);
    }
  });

  test('Concurrent reads are serialized; no data is returned after disconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    final results = await Future.wait([session.readBytes(10, 4), session.readBytes(20, 4)]);
    expect(results, [[10, 11, 12, 13], [20, 21, 22, 23]]);
    session.invalidate('test disconnect');
    expect(await session.readBytes(10, 4), isNull);
  });

  test('rev2: ATAL sent, ATDPN captured before OBD, raw path recorded', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(fake.commands.contains('ATAL'), isTrue);
    expect(fake.commands.indexOf('ATDPN'), lessThan(fake.commands.indexOf('0201000000000000')));
    expect(session.obdVia, 'raw-caf0');
    expect(session.atalAccepted, isTrue);
    expect(session.protocolNumber, '6');
    expect(session.adapterInfo, contains('ELM327'));
  });

  test('rev2: bare ? on raw OBD retries via CAF1; persistent raw rejection fails SSM clearly', () async {
    final fake = FakeElm()..rejectRawTx = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('rpm', 0xE, 1)]), isFalse);
    expect(session.obdReady, isTrue);
    expect(session.ssmReady, isFalse);
    expect(session.obdVia, 'caf1-fallback');
    expect(session.protocolNumber, '6');
    final caf1 = fake.commands.indexOf('ATCAF1');
    final obd = fake.commands.indexOf('0100');
    final restore = fake.commands.lastIndexOf('ATCAF0');
    expect(caf1 >= 0 && obd > caf1 && restore > obd, isTrue);
    expect(session.lastError, contains('SSM_NOT_CONFIRMED'));
    expect(session.lastError, contains('RAW_TX_REJECTED'));
  });

  test('rev2: rejected ATAL does not abort init when raw frames work', () async {
    final fake = FakeElm()..rejectAtal = true;
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(session.atalAccepted, isFalse);
    expect(session.diagnostic, contains('OPTIONAL ATAL FAILED'));
  });

  test('rev2: raw rejection mid-session stops polling until reconnect', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    fake.rejectRawTx = true;
    expect(await session.readBytes(0xE, 1), isNull);
    expect(session.ssmReady, isFalse);
    expect(session.lastError, contains('RAW_TX_REJECTED'));
  });

  test('rev3: reads of 4+ bytes are split so A8 never needs two CFs', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(await session.readBytes(0xFF6454, 4), [0x54, 0x55, 0x56, 0x57]);
    expect(fake.commands.any((c) => c.startsWith('100E')), isFalse,
        reason: '14-byte first frame is the failing pattern from the car log');
    expect(fake.commands.any((c) => c.startsWith('100B') || c.startsWith('05A8')), isTrue);
  });

  test('rev3: NO DATA on one block does not drop a confirmed SSM2 session', () async {
    final fake = FakeElm();
    final session = CanV3Session(fake.call, delay: (_) async {});
    expect(await session.initialize([const CanV3Probe('test', 0xE, 1)]), isTrue);
    expect(session.ssmReady, isTrue);
    fake.noDataNext = true;
    expect(await session.readBytes(0x46, 1), isNull);
    expect(session.ssmReady, isTrue);
    expect(session.lastError, contains('NO_DATA'));
    expect(await session.readBytes(0xE, 1), [14]);
    expect(session.ssmReady, isTrue);
  });
}
'''
PID_SCREEN = r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';

class CanV3PidScreen extends StatefulWidget {
  const CanV3PidScreen({super.key});
  @override
  State<CanV3PidScreen> createState() => _CanV3PidScreenState();
}

class _CanV3PidScreenState extends State<CanV3PidScreen> {
  final List<_CanPidResult> _results = [];
  bool _running = false;
  bool _cancel = false;
  int _total = 0;
  String _message = '';

  @override
  void dispose() {
    _cancel = true;
    super.dispose();
  }

  Future<void> _scan() async {
    final svc = ConnectionService.I;
    if (_running) return;
    if (!svc.elm.canV3Ready) {
      setState(() => _message = 'Сначала подтвердите CAN и SSM2 через CONNECT + INIT ECU.');
      return;
    }
    final pids = SettingsService.I.selectedPids.toList();
    if (pids.isEmpty) { setState(() => _message = 'Нет выбранных PID.'); return; }
    setState(() { _running = true; _cancel = false; _results.clear(); _total = pids.length; _message = ''; });
    try {
      await svc.exclusive((elm) async {
        await elm.canV3Idle();
        for (final pid in pids) {
          if (_cancel || !mounted || !elm.canV3Ready) break;
          final clock = Stopwatch()..start();
          final bytes = await elm.readBytes(pid.address, pid.len);
          double? value;
          var status = 'READ_ERROR';
          var detail = elm.canV3LastError;
          if (bytes != null && bytes.length == pid.len) {
            try {
              final decoded = pid.formula(bytes);
              if (!decoded.isFinite) {
                status = 'FORMULA_ERROR'; detail = 'Формула вернула NaN/Infinity.';
              } else {
                value = decoded; status = 'OK'; detail = 'E8 и длина проверены; физическая достоверность не подтверждена.';
              }
            } catch (e) { status = 'FORMULA_ERROR'; detail = e.toString(); }
          }
          clock.stop();
          final result = _CanPidResult(
            pid.id, pid.address, pid.len, status, value,
            bytes?.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ').toUpperCase() ?? '',
            clock.elapsedMilliseconds, detail,
          );
          if (!mounted) break;
          setState(() => _results.add(result));
        }
      });
      if (mounted) setState(() => _message = _cancel ? 'Проверка остановлена после текущего запроса.' : 'Проверка завершена. Кэш рабочих PID не изменён.');
    } catch (e) {
      if (mounted) setState(() => _message = 'Ошибка диагностики: $e');
    } finally {
      if (mounted) setState(() => _running = false);
    }
  }

  Future<void> _copy() async {
    final report = [
      'CAN-V3 selected PID diagnostics (not a calibration validation)',
      ..._results.map((r) => '${r.id} address=0x${r.address.toRadixString(16)} length=${r.length} '
          'status=${r.status} value=${r.value} ms=${r.ms} bytes=${r.bytes} detail=${r.detail}'),
      ConnectionService.I.elm.canV3Diagnostic,
    ].join('\n');
    try {
      await Clipboard.setData(ClipboardData(text: report));
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Отчёт CAN скопирован')));
    } catch (e) { if (mounted) setState(() => _message = 'Копирование не удалось: $e'); }
  }

  @override
  Widget build(BuildContext context) => Scaffold(
    appBar: AppBar(title: const Text('CAN-V3: выбранные PID'), actions: [
      IconButton(tooltip: 'Копировать отчёт', onPressed: _copy, icon: const Icon(Icons.copy)),
    ]),
    body: Column(children: [
      Padding(padding: const EdgeInsets.all(12), child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
        const Text('Проверяется текущий набор PID. Значение не считается физически достоверным только из-за ответа E8. '
            'Опрос приостанавливается на время проверки. Кэш по общему имени Subaru-CAN не создаётся.', style: TextStyle(fontSize: 12)),
        const SizedBox(height: 8),
        Text('${_results.length} / $_total; OK: ${_results.where((r) => r.status == 'OK').length}'),
        if (_running) LinearProgressIndicator(value: _total == 0 ? null : _results.length / _total),
        const SizedBox(height: 8),
        FilledButton(onPressed: _running ? null : _scan, child: const Text('Проверить выбранные PID')),
        if (_running) TextButton(onPressed: () => setState(() => _cancel = true),
            child: Text(_cancel ? 'Завершаем текущий запрос...' : 'Остановить')),
        if (_message.isNotEmpty) Padding(padding: const EdgeInsets.only(top: 8), child: Text(_message)),
      ])),
      Expanded(child: ListView.builder(itemCount: _results.length, itemBuilder: (context, index) {
        final r = _results[index];
        return ExpansionTile(
          title: Text('${r.id}: ${r.status}', style: TextStyle(color: r.status == 'OK' ? Colors.greenAccent : Colors.orangeAccent)),
          subtitle: Text('${r.ms} ms${r.value == null ? '' : ' | ${r.value}'}'),
          children: [Padding(padding: const EdgeInsets.all(12), child: SelectableText(
            'Address: 0x${r.address.toRadixString(16)}\nLength: ${r.length}\nBytes: ${r.bytes}\n${r.detail}',
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ))],
        );
      })),
    ]),
  );
}

class _CanPidResult {
  final String id;
  final int address;
  final int length;
  final String status;
  final double? value;
  final String bytes;
  final int ms;
  final String detail;
  _CanPidResult(this.id, this.address, this.length, this.status, this.value, this.bytes, this.ms, this.detail);
}'''


def require(ok, message):
    if not ok:
        raise ValueError(message)


def dart_mask(text):
    # Keep positions/newlines while hiding strings and comments for brace matching.
    result = list(text)

    def blank(a, b):
        for k in range(a, b):
            if result[k] != '\n':
                result[k] = ' '

    def string_end(start):
        raw = text[start] in 'rR' and start + 1 < len(text) and text[start + 1] in "'\""
        q = start + 1 if raw else start
        delimiter = text[q] * (3 if text.startswith(text[q] * 3, q) else 1)
        i = q + len(delimiter)
        while i < len(text):
            if text.startswith(delimiter, i):
                return i + len(delimiter)
            if not raw and text[i] == '\\':
                i += 2
            elif not raw and text.startswith('${', i):
                i = interpolation_end(i + 2)
            else:
                i += 1
        raise ValueError('Unterminated Dart string.')

    def interpolation_end(i):
        depth = 1
        while i < len(text):
            if text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
                i = string_end(i)
            elif text.startswith('//', i):
                end = text.find('\n', i)
                i = len(text) if end < 0 else end
            elif text.startswith('/*', i):
                end = text.find('*/', i + 2)
                require(end >= 0, 'Unterminated Dart comment.')
                i = end + 2
            else:
                if text[i] == '{': depth += 1
                if text[i] == '}': depth -= 1
                i += 1
                if depth == 0: return i
        raise ValueError('Unterminated Dart interpolation.')

    i = 0
    while i < len(text):
        if text.startswith('//', i):
            end = text.find('\n', i)
            end = len(text) if end < 0 else end
        elif text.startswith('/*', i):
            end = text.find('*/', i + 2)
            require(end >= 0, 'Unterminated Dart comment.')
            end += 2
        elif text[i] in "'\"" or (text[i] in 'rR' and i + 1 < len(text) and text[i + 1] in "'\""):
            end = string_end(i)
        else:
            i += 1
            continue
        blank(i, end)
        i = end
    return ''.join(result)


def close_bracket(mask, index, left='{', right='}'):
    require(mask[index] == left, 'Expected opening bracket.')
    depth = 0
    for i in range(index, len(mask)):
        if mask[i] == left: depth += 1
        if mask[i] == right: depth -= 1
        if depth == 0: return i
    raise ValueError('Unbalanced Dart brackets.')


def class_region(source, name):
    mask = dart_mask(source)
    found = re.search(r'\bclass\s+' + re.escape(name) + r'\b[^\{]*\{', mask)
    require(found is not None, 'Class not found: ' + name)
    opening = found.end() - 1
    return opening, close_bracket(mask, opening)


def methods(source, name):
    opening, closing = class_region(source, name)
    mask = dart_mask(source)
    pattern = r'(?m)^  Future<([^\n]+?)>\s+(\w+)(?:<[^>]+>)?\s*\('
    result = {}
    for match in re.finditer(pattern, mask[opening + 1:closing]):
        start = opening + 1 + match.start()
        paren = opening + match.end()
        end_params = close_bracket(mask, paren, '(', ')')
        suffix = re.match(r'\s*async\s*\{', mask[end_params + 1:])
        if suffix is None: continue
        body_start = end_params + 1 + suffix.end() - 1
        body_end = close_bracket(mask, body_start)
        require(match.group(2) not in result, 'Duplicate method: ' + match.group(2))
        result[match.group(2)] = dict(
            start=start, end=body_end + 1, header=source[start:body_start + 1],
            body=source[body_start + 1:body_end], params=source[paren + 1:end_params],
            returns=match.group(1), name=match.group(2),
        )
    return result


def edits(source, changes):
    for start, end, replacement in sorted(changes, reverse=True):
        source = source[:start] + replacement + source[end:]
    return source


def method_text(method, body):
    return method['header'] + '\n' + body.rstrip() + '\n  }'


def patch_elm(source):
    if '// CAN_V3_ADAPTER_BEGIN' in source:
        require('canV3Ready' in source and '_canV3Physical' in source, 'Partial CAN-V3 adapter detected.')
        return source
    found = methods(source, 'SsmElm')
    for name in ('ecuInit', 'readBytes', 'connect', 'disconnect'):
        require(name in found, 'Expected async SsmElm method missing: ' + name)
    require(found['ecuInit']['returns'] == 'String?' and not found['ecuInit']['params'].strip(),
            'Unexpected ecuInit signature; send can_v3_source_probe.txt.')
    require(found['readBytes']['returns'] == 'Uint8List?', 'Unexpected readBytes return type.')
    candidates = [m for m in found.values() if m['returns'] == 'String'
                  and re.search(r'\.output\s*\.\s*add\s*\(', dart_mask(m['body']))]
    require(len(candidates) == 1, 'Cannot identify one raw ELM send method without guessing. Send can_v3_source_probe.txt.')
    wire = candidates[0]
    require(not re.search(r'\b' + re.escape(wire['name']) + r'\s*\(', dart_mask(wire['body'])),
            'Raw sender is recursive; manual adaptation is required.')
    first = re.match(r'\s*String\s+(\w+)', wire['params'])
    require(first is not None, 'Expected a String command as the first wire parameter.')
    command_var = first.group(1)
    timeout = re.search(r'\b(int|Duration)\s+(timeoutMs|timeout)\s*=', wire['params'])
    require(timeout is not None and '{' in wire['params'], 'Expected a named timeout parameter in raw sender.')
    timeout_type, timeout_name = timeout.groups()
    passed_ms = timeout_name if timeout_type == 'int' else timeout_name + '.inMilliseconds'
    core_args = 'command, ' + timeout_name + (': ms' if timeout_type == 'int' else ': Duration(milliseconds: ms)')
    rest = wire['params'][first.end():]
    require(not re.search(r'\brequired\b', rest), 'Raw sender has unsupported required parameters.')
    for flattened in ("replaceAll('\\r', '')", "replaceAll('\\n', '')"):
        require(flattened not in wire['body'], 'Raw sender removes CAN frame boundaries. Send the source report.')

    old_init = found['ecuInit']['body']
    state_call = re.search(r'\b(\w+)\(SsmState\.ecuReady\);', dart_mask(old_init))
    state_assignment = re.search(r'\b(\w+)\s*=\s*SsmState\.ecuReady;', dart_mask(old_init))
    require(state_call is not None or state_assignment is not None, 'Cannot locate SsmState update without guessing.')
    ready = (state_call.group(1) + '(SsmState.ecuReady);') if state_call else (state_assignment.group(1) + ' = SsmState.ecuReady;')
    not_ready = ready.replace('SsmState.ecuReady', 'SsmState.elmReady')
    failed_state = ready.replace('SsmState.ecuReady', 'SsmState.error')
    a, b = class_region(source, 'SsmElm')
    ecu_id = re.search(r'\bString\s+(ecuId|_ecuId)\s*=', dart_mask(source[a:b]))
    require(ecu_id is not None, 'Cannot locate ECU-ID storage without guessing.')
    byte_params = re.findall(r'\bint\s+(\w+)', found['readBytes']['params'])
    require(len(byte_params) == 2, 'Expected readBytes(address, length).')

    physical_header = wire['header'].replace(wire['name'] + '(', '_canV3Physical(', 1)
    if physical_header == wire['header']:
        physical_header = re.sub(r'\b' + wire['name'] + r'\s*\(', '_canV3Physical(', wire['header'], count=1)
    wrapper = fr'''    final _canV3Command = {command_var}.replaceAll(RegExp(r'\s+'), '').toUpperCase();
    if (_canV3Configuring) {{
      // Adapter setup may reset defaults, but no vehicle packet is allowed before ATSP6.
      if (!_canV3Command.startsWith('AT') ||
          _canV3Command.startsWith('ATIB') || _canV3Command == 'ATSI' || _canV3Command == 'ATFI') {{
        throw StateError('CAN-V3 rejected startup command: $_canV3Command');
      }}
      final _canV3Sent = _canV3Command.startsWith('ATSP') || _canV3Command.startsWith('ATTP')
          ? 'ATSP6' : _canV3Command.startsWith('ATSH') ? 'ATSH7E0' : _canV3Command;
      if (_canV3Sent != _canV3Command) _canV3.record('FORCED $_canV3Command -> $_canV3Sent');
      return _canV3.setupCommand(_canV3Sent, {passed_ms});
    }}
    return _canV3.terminalQuery({command_var}, {passed_ms});'''
    physical = physical_header + wire['body'] + '}\n'

    new_init = f'''    final st = can_settings.SettingsService.I;
    st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
    stTimeoutCode = st.stCode;
    await st.save();
    final probes = st.selectedPids
        .where((p) => p.len > 0 && p.len <= 4 && p.address >= 0 && p.address + p.len - 1 <= 0xFFFFFF)
        .take(4).map((p) => can_proto.CanV3Probe('${{p.id}}:byte0', p.address, 1)).toList();
    final ok = await _canV3.initialize(probes, stCode: stTimeoutCode);
    {ecu_id.group(1)} = ''; // A successful OBD probe is not an ECU hardware ID.
    if (!ok) {{ {not_ready} return null; }}
    {ready}
    return 'CAN-V3: SSM2 confirmed, hardware ID not read';'''
    new_read = f'''    final bytes = await _canV3.readBytes({byte_params[0]}, {byte_params[1]});
    if (bytes == null && !_canV3.ssmReady) {{ {failed_state} }}
    return bytes;'''
    new_connect = '''    _canV3Configuring = true;
    _canV3.invalidate('new Bluetooth connection');
    try {''' + found['connect']['body'] + '''
    } finally { _canV3Configuring = false; }'''
    new_disconnect = "    _canV3.invalidate('Bluetooth disconnect');\n" + found['disconnect']['body']
    changes = []
    for name, body in [('ecuInit', new_init), ('readBytes', new_read), ('connect', new_connect), ('disconnect', new_disconnect)]:
        m = found[name]
        changes.append((m['start'], m['end'], method_text(m, body)))
    changes.append((wire['start'], wire['end'], method_text(wire, wrapper) + '\n\n  ' + physical.lstrip()))
    source = edits(source, changes)
    opening, _ = class_region(source, 'SsmElm')
    members = f'''
  // CAN_V3_ADAPTER_BEGIN
  bool _canV3Configuring = false;
  late final can_proto.CanV3Session _canV3 = can_proto.CanV3Session(
      (command, ms) => _canV3Physical({core_args}));
  bool get canV3Ready => _canV3.ssmReady;
  String get canV3LastError => _canV3.lastError;
  String get canV3Diagnostic => _canV3.diagnostic;
  int get canV3ReadOk => _canV3.readOk;
  int get canV3ReadErrors => _canV3.readErrors;
  int get canV3NoData => _canV3.noData;
  double get canV3AvgReadMs => _canV3.averageReadMs;
  Future<void> canV3Idle() => _canV3.whenIdle();
  // CAN_V3_ADAPTER_END
'''
    source = source[:opening + 1] + members + source[opening + 1:]
    source = "import 'can_v3_session.dart' as can_proto;\nimport '../services/settings_service.dart' as can_settings;\n" + source
    poller = methods(source, 'SsmPoller')
    require('_tick' in poller, 'Expected SsmPoller._tick to stop on a CAN failure.')
    tick = poller['_tick']
    tick_body, tick_count = re.subn(
        r'(data\s*=\s*await\s+elm\.readBytes\([^;]+\);)',
        r'\1\n        if (!elm.canV3Ready) { stop(); return; }', tick['body'])
    require(tick_count == 1, 'Cannot safely add the poller CAN failure guard.')
    tick_body = '    if (!elm.canV3Ready) { stop(); return; }\n' + tick_body
    source = edits(source, [(tick['start'], tick['end'], method_text(tick, tick_body))])
    print('Detected ELM sender:', wire['name'], '| timeout:', timeout_type, timeout_name)
    print('Replaced ECU init and readBytes; all SSM address requests now use CAN ISO-TP.')
    return source


def patch_connection(source):
    if '// CAN_V3_CONNECTION' in source: return source
    found = methods(source, 'ConnectionService')
    require('connectAndInit' in found, 'ConnectionService.connectAndInit not found.')
    m = found['connectAndInit']
    body = r'''    // CAN_V3_CONNECTION
    if (connecting) return 'CAN-V3: подключение уже выполняется.';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выберите сопряжённый ELM327.';
    connecting = true;
    notifyListeners();
    try {
      stopPolling();
      await elm.canV3Idle();
      if (logger.logging) await logger.stop();
      await elm.disconnect();
      st.stCode = st.stCode.clamp(0x32, 0xFF).toInt();
      await st.save();
      elm.stTimeoutCode = st.stCode;
      if (!await elm.connect(st.btAddress)) return 'CAN-V3: Bluetooth/ELM setup failed. Check the selected adapter.';
      final initialized = await elm.ecuInit();
      if (initialized == null || !elm.canV3Ready) {
        final reason = elm.canV3LastError;
        await elm.disconnect();
        return 'CAN-V3: $reason';
      }
      buildPoller();
      // Do not immediately poll an unverified large PID library on a new transport.
      return 'OK: CAN 500 kbit/s, ATDPN=6, SSM2 подтверждён. '
          'ECU ID не считан. Проверьте PID CAN, затем нажмите СТАРТ ОПРОС.';
    } catch (e) {
      try { await elm.disconnect(); } catch (_) {}
      return 'CAN-V3: $e';
    } finally {
      connecting = false;
      notifyListeners();
    }'''
    source = edits(source, [(m['start'], m['end'], method_text(m, body))])
    # Replace a fixed sleep with an actual drain of the serialized CAN transaction.
    source, count = re.subn(
        r'await Future<void>\.delayed\(const Duration\(milliseconds: 180\)\);[^\n]*',
        'await elm.canV3Idle(); // Wait for a complete CAN transaction, not 180 ms.', source)
    require(count == 1 or 'await elm.canV3Idle(); // Wait' in source,
            'The original exclusive() synchronization differs. Send the source report.')
    source = source.replace('if (was) startPolling();', 'if (was && elm.canV3Ready) startPolling();')
    return source


PANEL = r'''          // CAN_V3_PANEL_BEGIN
          const Text('CAN-V3 rev3 | CAN 11-bit, 500 kbit/s | 7E0 / 7E8',
              style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          const Text('После подключения опрос запускается вручную. '
              'В терминале разрешены только ATI / ATDP / ATDPN / ATRV / AT@1.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextButton.icon(
            onPressed: _busy || _scanning ? null : () async {
              await Navigator.of(context).push(MaterialPageRoute(builder: (_) => const CanV3PidScreen()));
              if (mounted) setState(() {});
            },
            icon: const Icon(Icons.fact_check_outlined), label: const Text('PID CAN: выбранные параметры'),
          ),
          TextButton.icon(onPressed: () async {
            try {
              await Clipboard.setData(ClipboardData(text: svc.elm.canV3Diagnostic));
              if (mounted) ScaffoldMessenger.of(context).showSnackBar(
                const SnackBar(content: Text('Журнал CAN скопирован')));
            } catch (e) { if (mounted) setState(() => _msg = 'Ошибка копирования CAN: $e'); }
          }, icon: const Icon(Icons.copy), label: const Text('Копировать журнал CAN')),
          ExpansionTile(title: const Text('Журнал CAN-V3', style: TextStyle(fontSize: 12)),
            children: [SelectableText(svc.elm.canV3Diagnostic,
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace'))]),
          // CAN_V3_PANEL_END
'''


def patch_settings(source):
    require('BT-PERM-V2' in source, 'Apply the existing BT-PERM-V2 fix before CAN-V3.')
    if '// CAN_V3_PANEL_BEGIN' in source: return source
    anchor = "        _card('ПОДКЛЮЧЕНИЕ ELM327', [\n"
    require(source.count(anchor) == 1, 'Connection panel anchor not found.')
    source = source.replace(anchor, anchor + PANEL, 1)
    for statement in ("import 'can_v3_pid_screen.dart';", "import 'package:flutter/services.dart';"):
        if statement not in source: source = statement + '\n' + source
    source = re.sub(r"(st\.stCode\.toDouble\(\),\s*)(?:0x[\da-fA-F]+|\d+),\s*(?:0x[\da-fA-F]+|\d+),",
                    r'\g<1>0x32, 0xFF,', source)
    source = re.sub(r'st\.stCode = v\.round\(\)(?: & ~1)?;[^\n]*', 'st.stCode = v.round();', source)
    source = source.replace("_slider('AT ST (×4 мс ожидание байта)'", "_slider('AT ST (CAN; после переподключения)'")
    source = source.replace('stats.avgMs', 'svc.elm.canV3AvgReadMs')
    source = source.replace('stats.framesOk', 'svc.elm.canV3ReadOk')
    source = source.replace('stats.framesErr', 'svc.elm.canV3ReadErrors')
    source = source.replace('stats.noData', 'svc.elm.canV3NoData')
    source = source.replace('на CAN-кадр', 'на чтение A8')
    source = source.replace('Кадров OK / ошибок / NO DATA', 'Чтений A8 OK / ошибок / NO DATA')
    return source


def preflight_tests():
    sample = """class X {
  Future<String> raw(String command, {int timeoutMs = 700}) async {
    final s = '${1 + 2}'; // } ignored
    return s;
  }
}
"""
    parsed = methods(sample, 'X')
    require('raw' in parsed and parsed['raw']['params'].startswith('String command'), 'Dart source parser self-test failed.')
    require(sample[parsed['raw']['start']:parsed['raw']['end']].endswith('}'), 'Method boundary self-test failed.')


paths = {
    ROOT / 'lib/ssm/ssm_elm.dart': patch_elm,
    ROOT / 'lib/services/connection_service.dart': patch_connection,
    ROOT / 'lib/screens/settings_screen.dart': patch_settings,
}
try:
    preflight_tests()
    require(FLUTTER.is_file(), 'Flutter SDK is not available in this Colab session.')
    require((ROOT / '.dart_tool/package_config.json').is_file(), 'Run pub get/build cell setup in this project first.')
    originals = {path: path.read_text(encoding='utf-8') for path in paths}
    outputs = {path: transform(originals[path]) for path, transform in paths.items()}
    for path, transform in paths.items():
        require(transform(outputs[path]) == outputs[path], 'Idempotence check failed: ' + str(path))
except (ValueError, FileNotFoundError) as error:
    if ROOT.is_dir():
        report = ROOT / 'can_v3_source_probe.txt'
        report.write_text('\n\n'.join(str(p.relative_to(ROOT)) + '\n' + p.read_text(encoding='utf-8')
                                     for p in paths if p.is_file()), encoding='utf-8')
        print('Source report for manual adaptation:', report)
    raise SystemExit('Preflight failed; project sources were NOT changed: ' + str(error)) from error

outputs.update({
    ROOT / 'lib/ssm/can_v3_codec.dart': CODEC,
    ROOT / 'lib/ssm/can_v3_session.dart': SESSION,
    ROOT / 'lib/screens/can_v3_pid_screen.dart': PID_SCREEN,
    ROOT / 'test/can_v3_test.dart': TESTS,
})
changes = [(path, path.read_bytes() if path.exists() else None, text.encode('utf-8'))
           for path, text in outputs.items() if not path.exists() or path.read_text(encoding='utf-8') != text]
backup = ROOT / 'patch_backups' / ('can_v3_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for path, before, _ in changes:
    if before is not None:
        saved = backup / path.relative_to(ROOT)
        saved.parent.mkdir(parents=True, exist_ok=True)
        saved.write_bytes(before)


def rollback():
    for path, before, _ in changes:
        if before is None: path.unlink(missing_ok=True)
        else: path.write_bytes(before)


try:
    for path, _, after in changes:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(after)
    print('Running CAN codec, flow-control, session and application-link tests...')
    result = subprocess.run([str(FLUTTER), 'test', '--no-pub', 'test/can_v3_test.dart'],
                            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300)
    (ROOT / 'can_v3_test_output.txt').write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    require(result.returncode == 0, 'Tests failed. Send can_v3_test_output.txt.')
except (OSError, ValueError, subprocess.TimeoutExpired) as error:
    rollback()
    raise SystemExit('CAN-V3 was NOT installed; changed source files were rolled back. ' + str(error)) from error

print('CAN-V3 rev3 source patch and simulated tests passed IN THIS COLAB RUNTIME.')
print('Backups:', backup if changes else 'no changes (already applied)')
print('Now run the EXISTING cell 10/10 and install the NEW APK. Look for CAN-V3 rev3.')
print('rev2 sends ATAL for 8-byte raw frames and captures ATDPN before OBD.')
print('If raw OBD is rejected with ?, OBD retries via CAF1+0100 (V7 method), then CAF0 is restored.')
print('OBD 4100 alone does not confirm SSM2. A valid E8 probe is required.')
print('No ECU hardware ID is invented. No global Subaru-CAN PID cache is reused.')
print('Polling starts manually. First verify a small selected PID set using PID CAN.')
print('Terminal protocol changes and arbitrary TX are blocked in this stabilization patch.')
print('Legacy DTC/service routines require separate CAN validation; do not assume they work.')
print('Real ELM clone, flow-control timing, CAN bus and ECU have NOT been tested by these tests.')

Running CAN codec, flow-control, session and application-link tests...
00:00 +0: loading /content/suba_run_v8/test/can_v3_test.dart
00:00 +0: Application API links against the patched SsmElm
00:00 +1: Exact SF and FF/CF for one and two A8 addresses
00:00 +2: Lengths 1..4095, padding removed, CF sequence wraps through zero
00:00 +3: 7E8 is a header, not a response SID; foreign frames and TX echo are not data
00:00 +4: Bad lengths, NO DATA, missing/duplicate/out-of-order CF and overflow fail
00:00 +5: CAN-only initialization does not invent an ECU ID or confirm SSM from 4100
00:00 +6: Required AT commands are verified, timeout code is hexadecimal
00:00 +7: MF requests complete with FC block size 0 and 1; 4-byte PID is one transaction
00:00 +8: Concurrent reads are serialized; no data is returned after disconnect
00:00 +9: rev2: ATAL sent, ATDPN captured before OBD, raw path recorded
00:00 +10: rev2: bare ? on raw OBD retries via CAF1; persistent raw rejection fails SSM clearly
00:00 +11:

In [ ]:
# @title 🩹 ФИКС V8-ALL: карты/сохранение/анализатор как в V6 (Subaru) + мгновенный расход + согласование логгер↔анализатор
# ============================================================================
# ЕДИНАЯ фикс-ячейка для SUBA RUN V8. Запускать ПОСЛЕ ячеек 0..9 (после того,
# как lib/ полностью сгенерирован) и ПЕРЕД ячейкой 10/10 (сборка APK).
# Идемпотентна: повторный запуск ничего не ломает (проверяет маркеры).
#
# Что делает:
#   1) rom_service.dart  — чинит запись float/int (setFloat32/endian), пакетную
#      запись, историю ревизий и «Сброс к дефолту» (как MapStorageService в V6).
#   2) models/rom_table  — editable учитывает поддержку storage; безопасный setCell.
#   3) analyzer_service  — паттерны V6 (Максимальная мощность / Экономика /
#      Стабильность), правила по детонации/AFR для Subaru (fbkc/fkl/kca/iam),
#      предложения правок с confidence/reason по ячейкам карты.
#   4) pending_edits.dart — очередь правок карт (как в V6).
#   5) rom_screen.dart   — вид карты как на скриншотах V6: цветная сетка
#      RPM×нагрузка, тап по ячейке -> старое/новое, кнопки Сохранить/Сброс.
#   6) analyzer_screen    — кнопка «В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM (n)» + список правок.
#   7) dashboard_screen  — под ячейкой BOOST добавляет ячейку МГНОВЕННЫЙ РАСХОД.
#   8) Согласование логгера и анализатора по каноническим ключам (единый источник).
# ============================================================================
import os, re
os.chdir('/content/suba_run_v8')

def read(p):  return open(p, encoding='utf-8').read()
def write(p, s): open(p, 'w', encoding='utf-8').write(s); print('  OK', p)
def has(p, marker): return os.path.exists(p) and marker in read(p)

print('=' * 64)
print('  ФИКС V8-ALL: перенос механики V6 в Subaru-проект')
print('=' * 64)

# ── Проверка зависимостей и порядка запуска ──
_required = [
    'lib/models/rom_table.dart',
    'lib/services/rom_service.dart',
    'lib/services/analyzer_service.dart',
    'lib/services/connection_service.dart',
    'lib/services/save_service.dart',           # создаётся в ФИКС 4 — нужен V8Saver
    'lib/screens/rom_screen.dart',
    'lib/screens/analyzer_screen.dart',
    'lib/screens/dashboard_screen.dart',
    'lib/generated/subaru_pids.g.dart',
    'lib/generated/subaru_rom.g.dart',
]
_missing = [p for p in _required if not os.path.exists(p)]
if _missing:
    raise SystemExit(
        'Нет файлов: %s\nЗапусти ячейки 0..9 и все ФИКСы (особенно ФИКС 4 — save_service) '
        'ПЕРЕД этой ячейкой.' % ', '.join(_missing))
if 'class V8Saver' not in read('lib/services/save_service.dart'):
    raise SystemExit('save_service.dart есть, но без V8Saver — прогони ФИКС 4/4.')
print('  зависимости на месте — продолжаю')

# ══════════════════════════════════════════════════════════════════════════
# 1. models/rom_table.dart — editable по поддержке storage + безопасный setCell
# ══════════════════════════════════════════════════════════════════════════
p = 'lib/models/rom_table.dart'
s = read(p)
if 'writableStorage' not in s:
    # набор storage, которые мы умеем корректно кодировать обратно
    s = s.replace(
        "  bool get editable => data.fr != null;",
        "  static const Set<String> writableStorage = {'float','uint8','int8','uint16','int16'};\n"
        "  bool get editable => data.fr != null && writableStorage.contains(data.storage);")
    # setCell с защитой границ
    s = s.replace(
        "  void setCell(int r, int c, double v) => z[r][c] = v;",
        "  void setCell(int r, int c, double v) {\n"
        "    if (r < 0 || r >= z.length || c < 0 || c >= z[r].length) return;\n"
        "    z[r][c] = v;\n"
        "  }")
    write(p, s)
else:
    print('  · rom_table.dart уже пропатчен')

# ══════════════════════════════════════════════════════════════════════════
# 2. rom_service.dart — ПОЛНАЯ замена: запись float/int + история + дефолты
# ══════════════════════════════════════════════════════════════════════════
ROM_SERVICE = r'''import 'dart:convert';
import 'dart:io';
import 'dart:typed_data';

import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import 'save_service.dart';

/// V8-ALL: загрузка/сохранение карт как в V6, адаптировано под Subaru.
/// - корректная запись float/uint/int с учётом endian (в V6 был только u8/u16 BE);
/// - пакетная запись нескольких карт в один .bin (SUBMOD_*.bin);
/// - слой «дефолтов»/правок в SharedPreferences (как MapStorageService V6);
/// - история ревизий на карту (как MapHistoryService V6).
class RomService {
  List<int>? rom;
  String fileName = '';
  ByteData? _bd;
  SharedPreferences? _p;

  bool get loaded => rom != null;
  List<RomTableDef> get defs => SubaruRom.tables;

  Future<SharedPreferences> get _prefs async => _p ??= await SharedPreferences.getInstance();

  Future<String?> pickAndLoad() async {
    final res = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['bin', 'hex', 'rom']);
    if (res == null || res.files.single.path == null) return null;
    final bytes = await File(res.files.single.path!).readAsBytes();
    return loadBytes(bytes, res.files.single.name);
  }

  String? loadBytes(List<int> bytes, String name) {
    if (bytes.length < 0x40000) return 'Файл слишком мал для SH7058 ROM';
    rom = bytes;
    fileName = name;
    _bd = ByteData.sublistView(Uint8List.fromList(bytes));
    return null;
  }

  List<double> _readAxis(RomCol? col, int tableIndex, bool isX, int fallbackCount) {
    if (col == null) return List<double>.generate(fallbackCount, (i) => i.toDouble());
    if (col.storage == 'static') {
      return SubaruRom.staticAxes['${isX ? 'x' : 'y'}$tableIndex'] ??
          List<double>.generate(fallbackCount, (i) => i.toDouble());
    }
    final values = <double>[];
    for (var i = 0; i < col.count; i++) {
      try { values.add(col.value(rom!, _bd!, i)); } catch (_) { values.add(double.nan); }
    }
    return values;
  }

  RomTable readTable(RomTableDef def) {
    final idx = SubaruRom.tables.indexOf(def);
    final xValues = _readAxis(def.xAxis, idx, true, def.cols);
    final yValues = _readAxis(def.yAxis, idx, false, def.rows);
    final z = List.generate(def.rows, (_) => List<double>.filled(def.cols, 0));
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        try { z[r][c] = def.data.value(rom!, _bd!, i); } catch (_) { z[r][c] = double.nan; }
      }
    }
    // Наложим сохранённые правки (если есть) — так это «дефолт» сессии, как в V6
    final saved = _loadEditsSync(def.addrHex);
    if (saved != null) {
      for (var r = 0; r < def.rows && r < saved.length; r++) {
        for (var c = 0; c < def.cols && c < saved[r].length; c++) {
          z[r][c] = saved[r][c];
        }
      }
    }
    return RomTable(def: def, xValues: xValues, yValues: yValues, z: z);
  }

  // ── Кодирование одной ячейки обратно в raw (учитывает ВСЕ storage) ──
  void _writeCell(Uint8List buf, ByteData bd, RomTableDef def, int off, double phys) {
    final st = def.data.storage;
    final little = def.data.endian == 'little';
    if (st == 'float') {
      // bd построен над тем же буфером buf -> запись float32 отражается в buf
      bd.setFloat32(off, phys, little ? Endian.little : Endian.big);
      return;
    }
    final fr = def.data.fr;
    final raw = (fr == null ? phys : fr(phys)).round();
    switch (st) {
      case 'uint16':
        final v = raw.clamp(0, 65535);
        if (little) { buf[off] = v & 0xFF; buf[off + 1] = (v >> 8) & 0xFF; }
        else { buf[off] = (v >> 8) & 0xFF; buf[off + 1] = v & 0xFF; }
        break;
      case 'int16':
        var v = raw.clamp(-32768, 32767);
        if (v < 0) v += 65536;
        if (little) { buf[off] = v & 0xFF; buf[off + 1] = (v >> 8) & 0xFF; }
        else { buf[off] = (v >> 8) & 0xFF; buf[off + 1] = v & 0xFF; }
        break;
      case 'int8':
        var v = raw.clamp(-128, 127);
        if (v < 0) v += 256;
        buf[off] = v & 0xFF;
        break;
      default: // uint8
        buf[off] = raw.clamp(0, 255);
    }
  }

  /// Записывает правки одной карты в переданный буфер (не в оригинал).
  bool _applyTableToBuffer(Uint8List buf, ByteData bd, RomTable table) {
    final def = table.def;
    if (!def.editable) return false;
    final size = def.data.sizeOf;
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        final off = def.address + i * size;
        if (off + size > buf.length) continue;
        _writeCell(buf, bd, def, off, table.z[r][c]);
      }
    }
    return true;
  }

  /// Одна карта -> новый .bin (как раньше, но с корректным float).
  Future<String?> saveMod(RomTable table) async {
    final path = await saveModBatch([table]);
    return path;
  }

  /// ПАКЕТНАЯ запись: несколько карт в один SUBMOD_{n}_*.bin (как V6 NLP_MOD).
  Future<String?> saveModBatch(List<RomTable> tables) async {
    if (rom == null) return null;
    final editable = tables.where((t) => t.def.editable).toList();
    if (editable.isEmpty) return null;
    // единый буфер: bd и out указывают на одну и ту же память
    final out = Uint8List.fromList(rom!);
    final bd = ByteData.sublistView(out);
    for (final t in editable) { _applyTableToBuffer(out, bd, t); }

    final base = fileName.replaceAll(RegExp(r'\.(bin|hex|rom)$', caseSensitive: false), '');
    final m = RegExp(r'SUBMOD_(\d+)_').firstMatch(base);
    final idx = m != null ? int.parse(m.group(1)!) + 1 : 1;
    final clean = base.replaceAll(RegExp(r'SUBMOD_\d+_'), '');
    final name = 'SUBMOD_${idx}_$clean.bin';

    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/$name');
    await f.writeAsBytes(out);
    // + копия в /Download (как ФИКС 4)
    try {
      final r = await V8Saver.saveBytes(out, name);
      if (r.ok) return r.path;
    } catch (_) {}
    return f.path;
  }

  // ══ Слой «дефолтов»/правок (SharedPreferences) — как MapStorageService V6 ══
  static const _editPrefix = 'rom_edit_';
  static const _histPrefix = 'rom_hist_';

  List<List<double>>? _cachedEdits;
  String _cachedAddr = '';
  List<List<double>>? _loadEditsSync(String addr) {
    if (_cachedAddr == addr) return _cachedEdits;
    return null; // синхронный кэш; асинхронная загрузка — в prewarmEdits()
  }

  /// Прогреть кэш правок для карты (вызывать перед readTable в экране).
  Future<void> prewarmEdits(String addr) async {
    final p = await _prefs;
    final s = p.getString('$_editPrefix$addr');
    _cachedAddr = addr;
    _cachedEdits = null;
    if (s == null) return;
    try {
      final j = jsonDecode(s) as List;
      _cachedEdits = j.map<List<double>>(
          (row) => (row as List).map<double>((v) => (v as num).toDouble()).toList()).toList();
    } catch (_) {}
  }

  Future<bool> hasEdits(String addr) async =>
      (await _prefs).containsKey('$_editPrefix$addr');

  /// Сохранить правки карты как «дефолт» + новую ревизию в истории.
  Future<void> saveEdits(RomTable table, {String comment = ''}) async {
    final p = await _prefs;
    final addr = table.def.addrHex;
    final data = table.z.map((r) => List<double>.from(r)).toList();
    await p.setString('$_editPrefix$addr', jsonEncode(data));
    _cachedAddr = addr; _cachedEdits = data;
    // история (до 20 ревизий)
    final histRaw = p.getString('$_histPrefix$addr');
    final hist = <Map<String, dynamic>>[];
    if (histRaw != null) {
      try { for (final e in jsonDecode(histRaw) as List) { hist.add(e as Map<String, dynamic>); } } catch (_) {}
    }
    hist.add({
      'v': hist.isEmpty ? 1 : (hist.last['v'] as int) + 1,
      'at': DateTime.now().toIso8601String(),
      'comment': comment.isEmpty ? 'Правка' : comment,
      'data': data,
    });
    if (hist.length > 20) hist.removeRange(0, hist.length - 20);
    await p.setString('$_histPrefix$addr', jsonEncode(hist));
  }

  /// Сброс правок карты к значениям из .bin (кнопка «Сброс» на скринах V6).
  Future<void> resetEdits(String addr) async {
    final p = await _prefs;
    await p.remove('$_editPrefix$addr');
    if (_cachedAddr == addr) { _cachedEdits = null; }
  }

  /// Полный сброс всех правок (вызывать при загрузке нового ROM).
  Future<void> resetAllEdits() async {
    final p = await _prefs;
    for (final k in p.getKeys().where((k) => k.startsWith(_editPrefix)).toList()) {
      await p.remove(k);
    }
    _cachedAddr = ''; _cachedEdits = null;
  }
}
'''
if not has('lib/services/rom_service.dart', 'saveModBatch'):
    write('lib/services/rom_service.dart', ROM_SERVICE)
else:
    print('  · rom_service.dart уже пропатчен')

# ══════════════════════════════════════════════════════════════════════════
# 3. services/pending_edits.dart — очередь правок карт (как в V6)
# ══════════════════════════════════════════════════════════════════════════
PENDING = r'''import '../models/rom_table.dart';

/// Очередь правок карт до пакетной записи в ROM (перенос из Nissan V6).
class PendingEdits {
  static final PendingEdits I = PendingEdits._();
  PendingEdits._();

  final Map<String, RomTable> _q = {}; // addrHex -> изменённая карта

  int get count => _q.length;
  bool get isEmpty => _q.isEmpty;
  List<RomTable> get all => _q.values.toList();
  bool has(String addr) => _q.containsKey(addr);

  void addOrUpdate(RomTable table) => _q[table.def.addrHex] = table;
  void remove(String addr) => _q.remove(addr);
  void clear() => _q.clear();
}
'''
if not os.path.exists('lib/services/pending_edits.dart'):
    write('lib/services/pending_edits.dart', PENDING)
else:
    print('  · pending_edits.dart уже есть')

# ══════════════════════════════════════════════════════════════════════════
# 4. analyzer_service.dart — паттерны + предложения правок по ячейкам карты
#    (сохраняем существующий HeatGrid, ДОБАВЛЯЕМ интеллектуальный анализ)
# ══════════════════════════════════════════════════════════════════════════
ANALYZER_ADD = r'''

// ═══════════════════════════════════════════════════════════════
// V8-ALL: интеллектуальный анализ по ячейкам карты (перенос из V6)
// Паттерны настройки + предложения правок с confidence/reason.
// Использует канонические ключи логгера (rpm, boost, kca, fbkc, fkl, iam, afr).
// ═══════════════════════════════════════════════════════════════
enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTol;      // порог по FBKC/FKL (модуль, град)
  final double afrLean, afrRich;
  final double timingAggr;    // агрессивность добавки УОЗ 0..1
  const TuningPattern(this.type, this.name, this.description,
      {this.knockTol = 1.0, this.afrLean = 14.7, this.afrRich = 11.5,
       this.timingAggr = 0.5});

  static const maxPower = TuningPattern(TuningPatternType.maxPower,
      'Максимальная мощность', 'Агрессивный УОЗ, богатая WOT смесь',
      knockTol: 0.3, afrLean: 12.5, afrRich: 11.0, timingAggr: 0.9);
  static const economy = TuningPattern(TuningPatternType.economy,
      'Минимальный расход', 'Бедная смесь на круизе',
      knockTol: 0.5, afrLean: 15.2, afrRich: 13.5, timingAggr: 0.3);
  static const stability = TuningPattern(TuningPatternType.stability,
      'Стабильная работа', 'Консервативные настройки',
      knockTol: 1.0, afrLean: 14.7, afrRich: 12.0, timingAggr: 0.15);
  static const List<TuningPattern> all = [maxPower, economy, stability];
}

class MapEditCell {
  final int r, c;
  final double rpm, load;
  final double current, suggested, confidence;
  final int samples;
  final String reason;
  const MapEditCell(this.r, this.c, this.rpm, this.load, this.current,
      this.suggested, this.confidence, this.samples, this.reason);
  double get delta => suggested - current;
}

class MapAnalysis {
  final String mapName;
  final int totalSamples;
  final List<MapEditCell> changes;
  final String summary;
  final String patternName;
  const MapAnalysis(this.mapName, this.totalSamples, this.changes,
      this.summary, this.patternName);
}

/// Какие карты умеем анализировать и по какому канону.
enum TuneMapKind { ignition, fuel, boost }

extension SmartAnalyzer on AnalyzerService {
  static const int _minSamples = 3;
  static const double _minConf = 0.4;

  double _avg(Iterable<double> v) {
    var s = 0.0; var n = 0;
    for (final x in v) { if (!x.isNaN) { s += x; n++; } }
    return n == 0 ? double.nan : s / n;
  }

  int _nearest(List<double> axis, double v) {
    var idx = 0; var best = double.infinity;
    for (var i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  /// Анализ карты по логу: cur = значение карты, sug = предложение.
  MapAnalysis analyzeMap(
    List<Map<String, double>> rows,
    List<List<double>> z,
    List<double> xAxis, // load / нагрузка
    List<double> yAxis, // rpm
    TuneMapKind kind, {
    TuningPattern pattern = TuningPattern.stability,
    String mapName = '',
  }) {
    // Группируем замеры по ячейкам карты
    final buckets = <String, List<Map<String, double>>>{};
    for (final d in rows) {
      final rpm = d['rpm'];
      if (rpm == null || rpm < 400 || rpm > 8000) continue;
      final loadKey = d.containsKey('boost') && kind == TuneMapKind.boost
          ? d['boost']!
          : (d['maf'] ?? d['tps'] ?? d['boost'] ?? 0);
      final ri = _nearest(yAxis, rpm);
      final ci = _nearest(xAxis, loadKey);
      (buckets['$ri,$ci'] ??= []).add(d);
    }

    final changes = <MapEditCell>[];
    buckets.forEach((key, samples) {
      if (samples.length < _minSamples) return;
      final parts = key.split(',');
      final ri = int.parse(parts[0]);
      final ci = int.parse(parts[1]);
      if (ri >= z.length || ci >= z[ri].length) return;
      final cur = z[ri][ci];

      double sug = cur; double conf = 0; String why = '';

      if (kind == TuneMapKind.ignition) {
        // fbkc/fkl отрицательные = была детонация; kca = итоговый УОЗ
        final fbkc = _avg(samples.map((d) => d['fbkc'] ?? 0));
        final fkl  = _avg(samples.map((d) => d['fkl'] ?? 0));
        final knock = [fbkc, fkl].where((v) => !v.isNaN).fold(0.0, (a, b) => a < b ? a : b);
        final afr = _avg(samples.map((d) => d['afr'] ?? double.nan));
        if (!knock.isNaN && knock <= -pattern.knockTol * 2) {
          sug = cur - (knock.abs().clamp(0.0, 3.0));
          why = 'Детонация ${knock.toStringAsFixed(1)}°'; conf = 0.9;
        } else if (!knock.isNaN && knock <= -pattern.knockTol) {
          sug = cur - (1.0 * (1.0 - pattern.timingAggr));
          why = 'FBKC ${knock.toStringAsFixed(1)}°'; conf = 0.7;
        } else if (!knock.isNaN && knock > -pattern.knockTol * 0.2 &&
                   !afr.isNaN && afr > pattern.afrRich && afr < pattern.afrLean) {
          sug = cur + pattern.timingAggr * 2;
          why = '+${(pattern.timingAggr * 2).toStringAsFixed(1)}° (запас есть)'; conf = 0.5;
        }
        sug = sug.clamp(-10.0, 45.0);
      } else if (kind == TuneMapKind.fuel) {
        // подгоняем к целевому AFR паттерна (под нагрузкой — богаче)
        final afr = _avg(samples.map((d) => d['afr'] ?? double.nan));
        final boost = _avg(samples.map((d) => d['boost'] ?? 0));
        if (!afr.isNaN) {
          final targetAfr = boost > 0.05 ? pattern.afrRich : pattern.afrLean;
          final err = (afr - targetAfr) / targetAfr;
          if (err.abs() > 0.03) {
            // карта задаёт целевой множитель λ — двигаем на долю ошибки
            sug = cur * (1 - err.clamp(-0.15, 0.15));
            why = 'AFR ${afr.toStringAsFixed(1)}→${targetAfr.toStringAsFixed(1)}'; conf = 0.6;
          }
        }
      } else {
        // boost: подтягиваем к цели, если фактический стабильно ниже/выше
        final b = _avg(samples.map((d) => d['boost'] ?? double.nan));
        final tb = _avg(samples.map((d) => d['tboost'] ?? double.nan));
        if (!b.isNaN && !tb.isNaN && (b - tb).abs() > 0.08) {
          sug = cur + (tb - b) * 0.5;
          why = 'Буст ${b.toStringAsFixed(2)}→цель ${tb.toStringAsFixed(2)}'; conf = 0.5;
        }
      }

      if ((sug - cur).abs() >= 0.1 && conf >= _minConf) {
        changes.add(MapEditCell(ri, ci, yAxis[ri], xAxis[ci], cur, sug,
            conf, samples.length, why));
      }
    });

    final up = changes.where((e) => e.delta > 0).length;
    final dn = changes.where((e) => e.delta < 0).length;
    return MapAnalysis(
      mapName.isEmpty ? kind.name : mapName,
      rows.length,
      changes,
      'Правок: ${changes.length} (+$up −$dn) · ячеек с данными: ${buckets.length}',
      pattern.name,
    );
  }
}
'''
p = 'lib/services/analyzer_service.dart'
s = read(p)
if 'class TuningPattern' not in s:
    # analyzer_service заканчивается на "}\n" последнего класса — просто дописываем
    s = s.rstrip() + '\n' + ANALYZER_ADD
    write(p, s)
else:
    print('  · analyzer_service.dart уже содержит паттерны')

# ══════════════════════════════════════════════════════════════════════════
# 5. rom_screen.dart — вид карты как на скринах V6:
#    цветная сетка RPM×нагрузка, тап -> старое/новое, Сохранить/Сброс
#    + прогрев правок перед чтением + запись через saveEdits/pending
# ══════════════════════════════════════════════════════════════════════════
# Патчим RomTableScreen: прогрев + кнопки. Делается точечными заменами.
p = 'lib/screens/rom_screen.dart'
s = read(p)
if 'V8ALL_TABLE' not in s:
    # 5.1 прогрев правок и повторное чтение при init
    s = s.replace(
        "  @override\n  void initState() {\n    super.initState();\n    _table = ConnectionService.I.rom.readTable(widget.def);\n  }",
        "  // V8ALL_TABLE\n"
        "  bool _hasSavedEdits = false;\n"
        "  @override\n  void initState() {\n    super.initState();\n"
        "    _reload();\n  }\n\n"
        "  Future<void> _reload() async {\n"
        "    final rom = ConnectionService.I.rom;\n"
        "    await rom.prewarmEdits(widget.def.addrHex);\n"
        "    _hasSavedEdits = await rom.hasEdits(widget.def.addrHex);\n"
        "    if (!mounted) return;\n"
        "    setState(() { _table = rom.readTable(widget.def); });\n"
        "  }")
    # 5.2 в AppBar заменяем действие V8MOD на «Сохранить» (правки-дефолт) и добавляем «Сброс»
    s = s.replace(
        "          if (d.editable)\n"
        "            TextButton.icon(\n"
        "              onPressed: _modified ? _saveMod : null,\n"
        "              icon: const Icon(Icons.save_outlined, size: 18),\n"
        "              label: const Text('V8MOD'),\n"
        "            ),",
        "          if (d.editable)\n"
        "            TextButton.icon(\n"
        "              onPressed: (_modified || _hasSavedEdits) ? _resetEdits : null,\n"
        "              icon: const Icon(Icons.history, size: 18),\n"
        "              label: const Text('Сброс'),\n"
        "              style: TextButton.styleFrom(foregroundColor: HeatColors.gold)),\n"
        "          if (d.editable)\n"
        "            TextButton.icon(\n"
        "              onPressed: _modified ? _saveEdits : null,\n"
        "              icon: const Icon(Icons.save, size: 18),\n"
        "              label: const Text('Сохранить')),\n"
        "          if (d.editable)\n"
        "            TextButton.icon(\n"
        "              onPressed: _modified ? _saveMod : null,\n"
        "              icon: const Icon(Icons.sd_card, size: 18),\n"
        "              label: const Text('В .bin')),")
    # 5.3 добавляем методы _saveEdits/_resetEdits перед _saveMod
    s = s.replace(
        "  Future<void> _saveMod() async {",
        "  Future<void> _saveEdits() async {\n"
        "    await ConnectionService.I.rom.saveEdits(_table!, comment: 'ручная правка');\n"
        "    if (!mounted) return;\n"
        "    setState(() { _modified = false; _hasSavedEdits = true; });\n"
        "    ScaffoldMessenger.of(context).showSnackBar(\n"
        "        const SnackBar(content: Text('Сохранено как дефолт (с историей)')));\n"
        "  }\n\n"
        "  Future<void> _resetEdits() async {\n"
        "    await ConnectionService.I.rom.resetEdits(widget.def.addrHex);\n"
        "    await _reload();\n"
        "    if (!mounted) return;\n"
        "    setState(() { _modified = false; });\n"
        "    ScaffoldMessenger.of(context).showSnackBar(\n"
        "        const SnackBar(content: Text('Сброшено к значениям из .bin')));\n"
        "  }\n\n"
        "  Future<void> _saveMod() async {")
    write(p, s)
else:
    print('  · rom_screen.dart уже пропатчен')

# ══════════════════════════════════════════════════════════════════════════
# 6. analyzer_screen.dart — кнопка «В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM (n)»
#    и вывод списка предложенных правок с confidence/reason.
#    Добавляем нижний блок к существующему экрану анализатора.
# ══════════════════════════════════════════════════════════════════════════
p = 'lib/screens/analyzer_screen.dart'
s = read(p)
if 'V8ALL_ANALYZER' not in s:
    s = s.replace("import '../widgets/heat_map.dart';",
        "import '../widgets/heat_map.dart';\n"
        "import '../services/pending_edits.dart';\n"
        "import '../models/rom_table.dart';\n"
        "import '../generated/subaru_rom.g.dart';")
    # состояние анализа + паттерн
    s = s.replace("  final List<String> _mergedLogs = [];",
        "  final List<String> _mergedLogs = [];\n"
        "  // V8ALL_ANALYZER\n"
        "  TuningPattern _pattern = TuningPattern.stability;\n"
        "  MapAnalysis? _analysis;\n"
        "  RomTable? _analyzedTable;\n"
        "  final List<Map<String, double>> _rawRows = [];")
    # копим сырые строки при слиянии логов
    s = s.replace("    a.buildFromRows(rows); // суммы и счётчики накапливаются => это и есть слияние",
        "    a.buildFromRows(rows);\n    _rawRows.addAll(rows); // V8ALL: копим строки для интеллектуального анализа")
    s = s.replace("    if (reset) { a.grid.reset(); _mergedLogs.clear(); }",
        "    if (reset) { a.grid.reset(); _mergedLogs.clear(); _rawRows.clear(); _analysis = null; }")
    # блок UI перед закрывающими скобками build (перед '      ],\n    );' последнего ListView)
    UI = r'''
        // ── V8ALL: интеллектуальный анализ карты по логу ──
        const SizedBox(height: 14),
        const Divider(color: HeatColors.dim),
        const Text('Интеллектуальная настройка карты',
            style: TextStyle(fontSize: 14, color: HeatColors.gold, fontWeight: FontWeight.bold)),
        const SizedBox(height: 8),
        Wrap(spacing: 8, runSpacing: 8, children:
          TuningPattern.all.map((pt) => ChoiceChip(
            label: Text(pt.name, style: const TextStyle(fontSize: 11)),
            selected: _pattern.type == pt.type,
            onSelected: (_) => setState(() => _pattern = pt),
          )).toList()),
        const SizedBox(height: 8),
        FilledButton.icon(
          onPressed: _rawRows.isEmpty ? null : _runSmart,
          icon: const Icon(Icons.auto_graph, size: 18),
          label: Text('АНАЛИЗ КАРТЫ «${AnalyzerService.metricNames[a.metric] ?? a.metric}» (${_rawRows.length} строк)'),
        ),
        if (_analysis != null) ...[
          const SizedBox(height: 8),
          Container(
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(10)),
            child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
              Text('${_analysis!.mapName} · ${_analysis!.changes.length} правок',
                  style: const TextStyle(fontWeight: FontWeight.bold)),
              Text('Паттерн: ${_analysis!.patternName}',
                  style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
              Text(_analysis!.summary, style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
              const SizedBox(height: 8),
              if (_analysis!.changes.isNotEmpty)
                FilledButton.icon(
                  style: FilledButton.styleFrom(backgroundColor: Colors.redAccent),
                  onPressed: _queueToRom,
                  icon: const Icon(Icons.playlist_add, size: 18),
                  label: Text('В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM (${_analysis!.changes.length})'),
                ),
              const SizedBox(height: 6),
              ..._analysis!.changes.take(40).map((c) => Padding(
                padding: const EdgeInsets.symmetric(vertical: 2),
                child: Row(children: [
                  Expanded(child: Text('RPM ${c.rpm.toStringAsFixed(0)} | ${c.load.toStringAsFixed(1)}',
                      style: const TextStyle(fontSize: 11))),
                  Text('${c.current.toStringAsFixed(1)} → ${c.suggested.toStringAsFixed(1)}',
                      style: TextStyle(fontSize: 11,
                          color: c.delta > 0 ? Colors.greenAccent : Colors.orangeAccent)),
                  const SizedBox(width: 8),
                  Text('${(c.confidence * 100).toStringAsFixed(0)}%',
                      style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
                ]))),
            ]),
          ),
        ],
'''
    # вставим перед последним '      ],\n    );\n  }' метода build
    idx = s.rfind("      ],\n    );\n  }")
    if idx != -1:
        s = s[:idx] + UI + s[idx:]
    # добавим методы _runSmart / _queueToRom / _mapKindFor перед последней '}'
    METHODS = r'''
  TuneMapKind _mapKindForMetric(String m) {
    if (m == 'boost' || m == 'boostErr' || m == 'wgd') return TuneMapKind.boost;
    if (m == 'afr' || m == 'stft' || m == 'ltft') return TuneMapKind.fuel;
    return TuneMapKind.ignition; // kca/fbkc/fkl/iam
  }

  RomTableDef? _bestDefForKind(TuneMapKind k) {
    final defs = SubaruRom.tables.where((d) => d.editable && d.is3d).toList();
    bool m(RomTableDef d, List<String> kw) {
      final n = d.name.toLowerCase();
      return kw.any((w) => n.contains(w));
    }
    RomTableDef? pick(List<String> kw) {
      for (final d in defs) { if (m(d, kw)) return d; }
      return null;
    }
    switch (k) {
      case TuneMapKind.ignition: return pick(['ignition','timing','advance','spark','base timing']);
      case TuneMapKind.fuel:     return pick(['fuel','injector','target','a/f','lambda']);
      case TuneMapKind.boost:    return pick(['boost','wastegate','wgd','duty']);
    }
  }

  Future<void> _runSmart() async {
    final kind = _mapKindForMetric(a.metric);
    final def = _bestDefForKind(kind);
    final rom = ConnectionService.I.rom;
    if (def == null || !rom.loaded) {
      setState(() => _status = def == null
          ? 'Не нашёл подходящую карту в дефинишне'
          : 'Сначала загрузи .bin на экране ROM');
      return;
    }
    await rom.prewarmEdits(def.addrHex);
    final table = rom.readTable(def);
    final an = a.analyzeMap(_rawRows, table.z, table.xValues, table.yValues, kind,
        pattern: _pattern, mapName: def.name);
    // применяем предложения к копии таблицы
    for (final c in an.changes) { table.setCell(c.r, c.c, c.suggested); }
    setState(() { _analysis = an; _analyzedTable = table; });
  }

  void _queueToRom() {
    if (_analyzedTable == null) return;
    PendingEdits.I.addOrUpdate(_analyzedTable!);
    setState(() => _status =
        'Добавлено в очередь. Всего карт в очереди: ${PendingEdits.I.count}. '
        'Запись — на экране ROM.');
  }
'''
    last = s.rfind("}")
    s = s[:last] + METHODS + "\n}" + s[last+1:]
    write(p, s)
else:
    print('  · analyzer_screen.dart уже пропатчен')

# ══════════════════════════════════════════════════════════════════════════
# 6b. rom_screen.dart — кнопка «Записать очередь в ROM (n)» в списке карт
# ══════════════════════════════════════════════════════════════════════════
p = 'lib/screens/rom_screen.dart'
s = read(p)
if 'V8ALL_QUEUE' not in s:
    s = s.replace("import '../services/connection_service.dart';",
        "import '../services/connection_service.dart';\n"
        "import '../services/pending_edits.dart';")
    # вставим кнопку записи очереди в верхнюю панель (рядом с «другой ROM»)
    s = s.replace(
        "            TextButton(onPressed: _load, child: const Text('другой ROM')),\n          ]),",
        "            if (PendingEdits.I.count > 0)\n"
        "              TextButton(\n"
        "                onPressed: _writeQueue,\n"
        "                child: Text('Записать очередь (${PendingEdits.I.count})',\n"
        "                    style: const TextStyle(color: Colors.redAccent))), // V8ALL_QUEUE\n"
        "            TextButton(onPressed: _load, child: const Text('другой ROM')),\n          ]),")
    # метод записи очереди
    s = s.replace("  Future<void> _load() async {",
        "  Future<void> _writeQueue() async {\n"
        "    final rom = ConnectionService.I.rom;\n"
        "    final path = await rom.saveModBatch(PendingEdits.I.all);\n"
        "    if (!mounted) return;\n"
        "    if (path != null) {\n"
        "      PendingEdits.I.clear();\n"
        "      setState(() {});\n"
        "      ScaffoldMessenger.of(context).showSnackBar(\n"
        "          SnackBar(content: Text('Записано в ${path.split('/').last}')));\n"
        "    } else {\n"
        "      ScaffoldMessenger.of(context).showSnackBar(\n"
        "          const SnackBar(content: Text('Нет карт для записи / read-only')));\n"
        "    }\n"
        "  }\n\n"
        "  Future<void> _load() async {")
    # сброс правок при загрузке нового ROM
    s = s.replace("    final err = await rom.pickAndLoad();",
        "    final err = await rom.pickAndLoad();\n"
        "    if (err == null) { await rom.resetAllEdits(); PendingEdits.I.clear(); }")
    write(p, s)
else:
    print('  · rom_screen.dart очередь уже добавлена')

# ══════════════════════════════════════════════════════════════════════════
# 7. dashboard_screen.dart — ячейка МГНОВЕННЫЙ РАСХОД под BOOST
# ══════════════════════════════════════════════════════════════════════════
p = 'lib/screens/dashboard_screen.dart'
s = read(p)
if 'V8ALL_FUEL' not in s:
    # вставляем компактную панель расхода сразу после закрытия панели BOOST
    # (после её StreamBuilder, перед Expanded с GridView)
    anchor = "        Expanded(\n          child: StreamBuilder<LiveSnapshot>("
    fuel_panel = r'''        // V8ALL_FUEL: мгновенный расход топлива под ячейкой BOOST
        StreamBuilder<LiveSnapshot>(
          stream: svc.poller?.snapshots,
          builder: (ctx, snap) {
            final s = snap.data ?? svc.poller?.last;
            final lph = s?.fuelLph ?? 0;
            final spd = s?.speed ?? 0;
            final l100 = (spd > 5 && lph > 0) ? (lph / spd * 100) : null;
            return Container(
              margin: const EdgeInsets.fromLTRB(10, 0, 10, 10),
              padding: const EdgeInsets.all(12),
              decoration: BoxDecoration(
                  color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
              child: Row(children: [
                const Icon(Icons.local_gas_station, size: 18, color: HeatColors.gold),
                const SizedBox(width: 8),
                const Text('РАСХОД', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
                const Spacer(),
                Text('${lph.toStringAsFixed(1)} л/ч',
                    style: const TextStyle(fontSize: 22, fontWeight: FontWeight.w800, color: HeatColors.text)),
                if (l100 != null)
                  Padding(
                    padding: const EdgeInsets.only(left: 10),
                    child: Text('${l100.toStringAsFixed(1)} л/100',
                        style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
                  ),
              ]),
            );
          },
        ),
'''
    s = s.replace(anchor, fuel_panel + anchor, 1)
    write(p, s)
else:
    print('  · dashboard_screen.dart расход уже добавлен')

# ══════════════════════════════════════════════════════════════════════════
# 8. Согласование логгера и анализатора
#    Логгер уже пишет boostErr/fuelLph и канон-ключи, parseLog отдаёт канон.
#    Убедимся, что fuel-канон maf/afr всегда попадают в лог: форсируем добавление
#    в selectedPids при построении поллера (иначе интеллектуальный анализ пуст).
# ══════════════════════════════════════════════════════════════════════════
p = 'lib/services/connection_service.dart'
s = read(p)
if 'V8ALL_SYNC' not in s:
    s = s.replace(
        "    p.buildBlocks([...st.selectedPids, ...CustomPidService.I.asPids()]);",
        "    // V8ALL_SYNC: гарантируем канон-PID, нужные анализатору\n"
        "    final need = {'rpm','boost','tboost','kca','fbkc','fkl','iam','afr','maf','tps','speed'};\n"
        "    final base = [...st.selectedPids, ...CustomPidService.I.asPids()];\n"
        "    final haveCanon = base.map((e) => e.canon).toSet();\n"
        "    final extra = <dynamic>[];\n"
        "    for (final canon in need) {\n"
        "      if (haveCanon.contains(canon)) continue;\n"
        "      final match = SubaruPids.all.where((pp) => pp.canon == canon);\n"
        "      if (match.isNotEmpty) extra.add(match.first);\n"
        "    }\n"
        "    p.buildBlocks([...base, ...extra.cast()]);")
    # импорт сгенерированных PID (если не импортирован)
    if "subaru_pids.g.dart" not in s:
        s = s.replace("import '../ssm/ssm_elm.dart';",
                      "import '../ssm/ssm_elm.dart';\nimport '../generated/subaru_pids.g.dart';")
    write(p, s)
else:
    print('  · connection_service.dart синхронизация уже добавлена')

print()
print('=' * 64)
print('  ГОТОВО. Проверь, что маркеры вставлены, и запусти ячейку 10/10 (сборка).')
print('  Новое: вид/правка карт как в V6, Сохранить/Сброс+история, пакетная')
print('  запись SUBMOD_*.bin, интеллектуальный анализатор с паттернами и')
print('  «В очередь на запись в ROM», ячейка мгновенного расхода на дашборде.')
print('=' * 64)


  ФИКС V8-ALL: перенос механики V6 в Subaru-проект
  зависимости на месте — продолжаю
  OK lib/models/rom_table.dart
  OK lib/services/rom_service.dart
  OK lib/services/pending_edits.dart
  OK lib/services/analyzer_service.dart
  OK lib/screens/rom_screen.dart
  OK lib/screens/analyzer_screen.dart
  OK lib/screens/rom_screen.dart
  OK lib/screens/dashboard_screen.dart
  OK lib/services/connection_service.dart

  ГОТОВО. Проверь, что маркеры вставлены, и запусти ячейку 10/10 (сборка).
  Новое: вид/правка карт как в V6, Сохранить/Сброс+история, пакетная
  запись SUBMOD_*.bin, интеллектуальный анализатор с паттернами и
  «В очередь на запись в ROM», ячейка мгновенного расхода на дашборде.


In [ ]:
# @title 🧭 ФИКС TUNE-BRIDGE: ROM-карты · анализатор · запись .bin как в V6 (Subaru) + расход на приборах + связка логгер↔анализатор
# ============================================================================
# SUBA RUN V8 — ОДНА ячейка, все правки. Идемпотентна: повторный запуск ничего не ломает.
# Запускать ПОСЛЕ ячеек 0–9 и прежних фиксов, ПЕРЕД ячейкой 10/10 (сборка APK).
#
# Что делает:
#  1) lib/services/rom_writer.dart        — единый кодек записи ячеек (float/int32/uint16/… + endian),
#                                           имена V8MOD{n}_*.bin, контрольные суммы Subaru SH7058
#                                           (формат подтверждается по ОРИГИНАЛУ — иначе байты не трогаем)
#  2) lib/services/map_storage_service     — «дефолтные значения» карт (SharedPrefs) = MapStorageService из V6
#  3) lib/services/map_history_service     — до 20 ревизий на карту, откат к любой
#  4) lib/services/pending_edits           — очередь карт на пакетную запись
#  5) lib/services/rom_holder              — новый ROM = сброс кеша правок; Cal ID из образа vs дефинишен
#  6) lib/models/tuning_types + lib/services/tuning_analyzer
#                                         — анализатор V6 (паттерны, confidence, reason) по Subaru-каналам:
#                                           FBKC/FKL/IAM → УОЗ, AFR под бустом → OL-топливо,
#                                           STFT+LTFT → MAF scaling, буст−цель → wastegate duty
#  7) lib/widgets/map_table_view + lib/screens/rom_table_edit_screen
#                                         — вид карты как в V6: старое зачёркнуто / новое жирным,
#                                           Сохранить / Сброс / зум, тап по ячейке
#  8) lib/screens/analyzer_screen (новый: ОНЛАЙН · ИЗ ЛОГА · ТЕПЛОКАРТА-старый V8)
#     lib/screens/write_rom_screen (вкладка ЗАПИСЬ ROM)
#  9) lib/widgets/fuel_card + патч dashboard — ячейка мгновенного расхода ПОД бустом
#                                           (л/ч, л/100 км, загрузка форсунок)
# 10) патчи: rom_service.saveMod → RomWriter (фикс float),
#            connection_service.buildPoller + logger_service.start → TuningChannels.ensure()
#            (поллер и CSV гарантированно содержат каналы, нужные анализатору)
# ============================================================================
import os, re, subprocess
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/suba_run_v8')
FLUTTER = Path('/content/flutter/bin/flutter')
MARK = '// TUNE-V6-BRIDGE'


def require(cond, msg):
    if not cond:
        raise ValueError(msg)


def add_import(src, line):
    """Вставить import после последнего import-а (если его ещё нет)."""
    if line in src:
        return src
    last = None
    for m in re.finditer(r'^import .*?;\s*$', src, re.M):
        last = m
    require(last is not None, 'нет import-секции')
    return src[:last.end()] + '\n' + line + src[last.end():]


# ════════════════════════════════════════════════════════════════════════════
# НОВЫЕ ФАЙЛЫ
# ════════════════════════════════════════════════════════════════════════════
NEW_FILES = {}

NEW_FILES['lib/models/tuning_types.dart'] = r'''// TUNE-V6-BRIDGE
import '../models/rom_table.dart';

enum PatternType { maxPower, economy, stability }

/// Паттерн настройки — как в V6, параметры под турбо-Subaru (EJ20X).
class TuningPattern {
  final PatternType type;
  final String name;
  final String desc;
  final double knockWarn;     // FBKC/FKL, ° (отрицательные): начинаем снимать УОЗ
  final double knockDanger;   // сильная детонация
  final double timingStep;    // шаг добавления УОЗ в «чистых» ячейках, °
  final double afrBoost;      // цель AFR под бустом (>0.3 бар)
  final double afrWot;        // цель AFR при полном газе без большого буста
  final double afrCruise;     // цель AFR круиз
  final double trimThreshold; // % (STFT+LTFT), выше — правим MAF
  final double boostErrTol;   // бар, допуск буст−цель
  final double wgdGain;       // % duty на 0.1 бар ошибки буста

  const TuningPattern({
    required this.type,
    required this.name,
    required this.desc,
    required this.knockWarn,
    required this.knockDanger,
    required this.timingStep,
    required this.afrBoost,
    required this.afrWot,
    required this.afrCruise,
    required this.trimThreshold,
    required this.boostErrTol,
    required this.wgdGain,
  });

  static const maxPower = TuningPattern(
    type: PatternType.maxPower,
    name: 'Максимальная мощность',
    desc: 'УОЗ к порогу детонации, AFR 11.2 под бустом',
    knockWarn: -0.7, knockDanger: -2.0, timingStep: 1.0,
    afrBoost: 11.2, afrWot: 12.0, afrCruise: 14.7,
    trimThreshold: 3.0, boostErrTol: 0.08, wgdGain: 2.5,
  );
  static const economy = TuningPattern(
    type: PatternType.economy,
    name: 'Минимальный расход',
    desc: 'Бедный круиз, консервативный буст',
    knockWarn: -1.0, knockDanger: -2.8, timingStep: 0.5,
    afrBoost: 11.8, afrWot: 12.8, afrCruise: 15.2,
    trimThreshold: 4.0, boostErrTol: 0.12, wgdGain: 1.5,
  );
  static const stability = TuningPattern(
    type: PatternType.stability,
    name: 'Стабильная работа',
    desc: 'Запас по детонации, AFR 11.5 под бустом',
    knockWarn: -1.0, knockDanger: -2.8, timingStep: 0.5,
    afrBoost: 11.5, afrWot: 12.5, afrCruise: 14.7,
    trimThreshold: 3.0, boostErrTol: 0.10, wgdGain: 2.0,
  );
  static const List<TuningPattern> all = [maxPower, economy, stability];
}

/// Что анализируем и в какую карту ROM это ложится.
enum TuningKind { ignition, fuelOl, maf, wgd, targetBoost, avcs }

class TuningTarget {
  final TuningKind kind;
  final String title;
  final RomTableDef? def;
  final String xCanon;
  final String yCanon;

  const TuningTarget({
    required this.kind,
    required this.title,
    required this.def,
    required this.xCanon,
    required this.yCanon,
  });

  TuningTarget copyWith({RomTableDef? def, String? xCanon, String? yCanon}) => TuningTarget(
        kind: kind,
        title: title,
        def: def ?? this.def,
        xCanon: xCanon ?? this.xCanon,
        yCanon: yCanon ?? this.yCanon,
      );
}

class MapCellChange {
  final int row;
  final int col;
  final double x;
  final double y;
  final double current;
  final double suggested;
  final double confidence;
  final int samples;
  final String reason;

  const MapCellChange({
    required this.row,
    required this.col,
    required this.x,
    required this.y,
    required this.current,
    required this.suggested,
    required this.confidence,
    required this.samples,
    required this.reason,
  });

  double get delta => suggested - current;
  double get deltaPct => current != 0 ? delta / current.abs() * 100 : 0;
}

class TuningResult {
  final String mapName;
  final String patternName;
  final int totalRows;
  final int usedRows;
  final int cellsHit;
  final List<MapCellChange> changes;
  final List<String> notes;
  final DateTime at;

  TuningResult({
    required this.mapName,
    required this.patternName,
    required this.totalRows,
    required this.usedRows,
    required this.cellsHit,
    required this.changes,
    required this.notes,
  }) : at = DateTime.now();

  int get up => changes.where((c) => c.delta > 0).length;
  int get down => changes.where((c) => c.delta < 0).length;
  double get avgAbsDelta => changes.isEmpty
      ? 0
      : changes.map((c) => c.delta.abs()).reduce((a, b) => a + b) / changes.length;
  double get maxAbsDelta =>
      changes.isEmpty ? 0 : changes.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);

  String get summary {
    final dir = up > down ? '↑ увеличение' : (down > up ? '↓ уменьшение' : '');
    return 'Данных: $usedRows/$totalRows | Клеток: $cellsHit\n'
        'Правок: ${changes.length} (+$up −$down) $dir\n'
        'Средняя |Δ|: ${avgAbsDelta.toStringAsFixed(2)} | Макс: ${maxAbsDelta.toStringAsFixed(2)}';
  }
}
'''

NEW_FILES['lib/services/tuning_analyzer.dart'] = r'''// TUNE-V6-BRIDGE
import 'dart:math' as math;

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/rom_table.dart';
import '../models/tuning_types.dart';

/// Каналы, без которых интеллектуальная настройка невозможна.
/// Поллер и логгер добавляют их к выбранным PID автоматически (ensure()).
class TuningChannels {
  static const List<String> required = [
    'rpm', 'maf', 'afr', 'fbkc', 'fkl', 'iam', 'kca',
    'boost', 'tboost', 'wgd', 'tps', 'stft', 'ltft', 'iat', 'ect', 'injms',
  ];
  static const List<String> helpful = ['mafV', 'load', 'avcs', 'speed'];

  static const Map<String, String> labels = {
    'rpm': 'Обороты', 'maf': 'MAF г/с', 'afr': 'AFR', 'fbkc': 'FBKC', 'fkl': 'FKL',
    'iam': 'IAM', 'kca': 'УОЗ (KCA)', 'boost': 'Буст', 'tboost': 'Цель буста',
    'wgd': 'WG duty', 'tps': 'Дроссель', 'stft': 'STFT', 'ltft': 'LTFT',
    'iat': 'IAT', 'ect': 'ECT', 'injms': 'Впрыск мс', 'mafV': 'MAF В',
    'load': 'Нагрузка', 'avcs': 'AVCS', 'speed': 'Скорость',
  };

  /// Дополняет список PID недостающими каналами (самый лёгкий PID на канал).
  static List<SubaruPid> ensure(List<SubaruPid> selected) {
    final out = List<SubaruPid>.of(selected);
    final have = <String>{};
    for (final p in out) {
      if (p.canon.isNotEmpty) have.add(p.canon);
    }
    for (final canon in [...required, ...helpful]) {
      if (have.contains(canon)) continue;
      final cands = SubaruPids.all.where((p) => p.canon == canon).toList();
      if (cands.isEmpty) continue;
      cands.sort((a, b) {
        final pr = a.priority.compareTo(b.priority);
        return pr != 0 ? pr : a.len.compareTo(b.len);
      });
      out.add(cands.first);
      have.add(canon);
    }
    return out;
  }

  static List<String> missing(Iterable<Map<String, double>> rows) {
    final seen = <String>{};
    for (final r in rows) {
      seen.addAll(r.keys);
    }
    return required.where((k) => !seen.contains(k)).toList();
  }
}

/// Статистика лога для экрана анализатора
class LogStats {
  final int rows;
  final int idle;
  final int cruise;
  final int boost;
  final int wot;
  final List<String> missing;
  const LogStats(this.rows, this.idle, this.cruise, this.boost, this.wot, this.missing);

  static LogStats of(List<Map<String, double>> rows) {
    var i = 0, c = 0, b = 0, w = 0;
    for (final r in rows) {
      final rpm = r['rpm'] ?? 0.0;
      final tps = r['tps'] ?? 0.0;
      final bst = r['boost'] ?? -1.0;
      if (rpm < 1100) {
        i++;
      } else if (tps > 85) {
        w++;
      } else if (bst >= AppConstants.boostActive) {
        b++;
      } else {
        c++;
      }
    }
    return LogStats(rows.length, i, c, b, w, TuningChannels.missing(rows));
  }
}

class _Cell {
  final List<Map<String, double>> s = [];
}

/// Анализатор «как в V6», но по Subaru-каналам.
class TuningAnalyzer {
  static const int minSamples = 3;
  static const double minConfidence = 0.4;
  static const List<String> yCanonChoices = ['auto', 'load', 'tps', 'boost', 'tboost', 'maf', 'injms'];

  // ── подбор карт ROM по имени из дефинишена ──────────────────────────────
  static List<TuningTarget> detectTargets(List<RomTableDef> defs) {
    RomTableDef? find(List<String> patterns, {bool need3d = true}) {
      for (final pat in patterns) {
        final re = RegExp(pat, caseSensitive: false);
        for (final d in defs) {
          if (!d.editable) continue;
          if (need3d && !d.is3d) continue;
          if (re.hasMatch(d.name)) return d;
        }
      }
      return null;
    }

    return [
      TuningTarget(kind: TuningKind.ignition, title: 'Зажигание (Base Timing)',
          def: find([r'^base timing.*primary', r'^base timing', r'timing.*primary']),
          xCanon: 'rpm', yCanon: 'auto'),
      TuningTarget(kind: TuningKind.fuelOl, title: 'Топливо OL (Primary Open Loop Fueling)',
          def: find([r'^primary open loop fuel', r'open loop fuel']),
          xCanon: 'rpm', yCanon: 'auto'),
      TuningTarget(kind: TuningKind.maf, title: 'MAF Sensor Scaling',
          def: find([r'^maf sensor scal', r'maf.*scal'], need3d: false),
          xCanon: 'mafV', yCanon: ''),
      TuningTarget(kind: TuningKind.wgd, title: 'Wastegate Duty (Initial)',
          def: find([r'^initial wastegate', r'wastegate duty']),
          xCanon: 'rpm', yCanon: 'auto'),
      TuningTarget(kind: TuningKind.targetBoost, title: 'Target Boost',
          def: find([r'^target boost']),
          xCanon: 'rpm', yCanon: 'auto'),
      TuningTarget(kind: TuningKind.avcs, title: 'AVCS впуск (справочно)',
          def: find([r'cam advance', r'avcs']),
          xCanon: 'rpm', yCanon: 'auto'),
    ];
  }

  /// Канал лога для оси Y по её значениям (override — чипами на экране)
  static String guessYCanon(RomTable t) {
    if (t.yValues.isEmpty) return '';
    final mx = t.yValues.reduce(math.max);
    if (mx <= 6.5) return 'load';
    if (mx <= 105) return 'tps';
    if (mx <= 400) return 'maf';
    return 'tps';
  }

  // ── значения из строки лога ─────────────────────────────────────────────
  static double? val(Map<String, double> r, String k) {
    if (k == 'load') {
      final l = r['load'];
      if (l != null) return l;
      final maf = r['maf'];
      final rpm = r['rpm'];
      if (maf == null || rpm == null || rpm < 300) return null;
      return maf * 60.0 / rpm; // Engine Load, г/об
    }
    return r[k];
  }

  static int _nearest(List<double> axis, double v) {
    if (axis.isEmpty) return -1;
    if (axis.length == 1) return 0;
    var idx = 0;
    var best = double.infinity;
    for (var i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) {
        best = d;
        idx = i;
      }
    }
    final step = (axis.last - axis.first).abs() / (axis.length - 1);
    final lo = math.min(axis.first, axis.last) - step * 0.6;
    final hi = math.max(axis.first, axis.last) + step * 0.6;
    if (v < lo || v > hi) return -1;
    return idx;
  }

  static double _avg(Iterable<double> xs) {
    var n = 0;
    var s = 0.0;
    for (final x in xs) {
      s += x;
      n++;
    }
    return n == 0 ? double.nan : s / n;
  }

  static double _min(Iterable<double> xs) {
    var m = double.infinity;
    for (final x in xs) {
      if (x < m) m = x;
    }
    return m == double.infinity ? double.nan : m;
  }

  static Iterable<double> _col(List<Map<String, double>> s, String k) =>
      s.map((r) => r[k]).whereType<double>();

  // ── главный вход ────────────────────────────────────────────────────────
  static TuningResult analyze({
    required List<Map<String, double>> rows,
    required RomTable table,
    required TuningTarget target,
    required TuningPattern pattern,
  }) {
    final notes = <String>[];
    final def = table.def;
    final yCanon = target.yCanon == 'auto' ? guessYCanon(table) : target.yCanon;
    final is1d = def.rows <= 1;
    if (!is1d) notes.add('Ось Y ← канал «$yCanon»');
    final missing = TuningChannels.missing(rows);
    if (missing.isNotEmpty) {
      notes.add('Нет каналов: ${missing.map((m) => TuningChannels.labels[m] ?? m).join(', ')}');
    }

    final cells = <int, _Cell>{};
    var used = 0;
    for (final r in rows) {
      final rpm = r['rpm'] ?? 0.0;
      if (rpm < 500 || rpm > 8000) continue;
      final xv = val(r, target.xCanon);
      if (xv == null) continue;
      final ci = _nearest(table.xValues, xv);
      if (ci < 0) continue;
      var ri = 0;
      if (!is1d) {
        final yv = val(r, yCanon);
        if (yv == null) continue;
        ri = _nearest(table.yValues, yv);
        if (ri < 0) continue;
      }
      cells.putIfAbsent(ri * def.cols + ci, () => _Cell()).s.add(r);
      used++;
    }

    final changes = <MapCellChange>[];
    cells.forEach((key, cell) {
      final s = cell.s;
      if (s.length < minSamples) return;
      final ri = key ~/ def.cols;
      final ci = key % def.cols;
      if (ri >= table.z.length || ci >= table.z[ri].length) return;
      final cur = table.z[ri][ci];
      if (cur.isNaN) return;
      final x = ci < table.xValues.length ? table.xValues[ci] : ci.toDouble();
      final y = ri < table.yValues.length ? table.yValues[ri] : ri.toDouble();
      MapCellChange? ch;
      switch (target.kind) {
        case TuningKind.ignition:
          ch = _ignition(s, cur, ri, ci, x, y, pattern, table);
          break;
        case TuningKind.fuelOl:
          ch = _fuelOl(s, cur, ri, ci, x, y, pattern);
          break;
        case TuningKind.maf:
          ch = _maf(s, cur, ri, ci, x, y, pattern);
          break;
        case TuningKind.wgd:
          ch = _wgd(s, cur, ri, ci, x, y, pattern);
          break;
        case TuningKind.targetBoost:
          ch = _targetBoost(s, cur, ri, ci, x, y, pattern);
          break;
        case TuningKind.avcs:
          final a = _avg(_col(s, 'avcs'));
          if (!a.isNaN && (a - cur).abs() > 5) {
            notes.add('AVCS ${x.toStringAsFixed(0)}/${y.toStringAsFixed(1)}: '
                'карта ${cur.toStringAsFixed(1)}°, факт ${a.toStringAsFixed(1)}°');
          }
          break;
      }
      if (ch != null && ch.confidence >= minConfidence && ch.delta.abs() > 1e-6) changes.add(ch);
    });

    changes.sort((a, b) => b.confidence.compareTo(a.confidence));
    if (target.kind == TuningKind.avcs) {
      notes.add('AVCS правится по приросту момента на стенде — автоправок нет');
    }
    if (used == 0) notes.add('Ни одна строка не попала в оси карты — проверь канал оси Y');
    return TuningResult(
      mapName: def.name,
      patternName: pattern.name,
      totalRows: rows.length,
      usedRows: used,
      cellsHit: cells.length,
      changes: changes,
      notes: notes,
    );
  }

  // ── правила ─────────────────────────────────────────────────────────────
  static MapCellChange? _ignition(List<Map<String, double>> s, double cur, int ri, int ci,
      double x, double y, TuningPattern p, RomTable t) {
    final fb = _avg(_col(s, 'fbkc'));
    final fk = _avg(_col(s, 'fkl'));
    final iam = _min(_col(s, 'iam'));
    final afr = _avg(_col(s, 'afr'));
    final boost = _avg(_col(s, 'boost'));
    final iat = _avg(_col(s, 'iat'));
    if (fb.isNaN && fk.isNaN) return null;
    final knock = math.min(fb.isNaN ? 0.0 : fb, fk.isNaN ? 0.0 : fk);
    var sug = cur;
    var conf = 0.0;
    var why = '';
    final boosted = !boost.isNaN && boost > 0.3;
    if (!iam.isNaN && iam < 0.9 && knock <= p.knockWarn) {
      sug = cur - 2.0;
      conf = 0.92;
      why = 'IAM ${iam.toStringAsFixed(2)} + детонация ${knock.toStringAsFixed(1)}°';
    } else if (knock <= p.knockDanger) {
      sug = cur - math.min(-knock, 3.0);
      conf = 0.9;
      why = 'Детонация ${knock.toStringAsFixed(1)}°';
    } else if (knock <= p.knockWarn) {
      sug = cur - p.timingStep;
      conf = 0.7;
      why = 'FBKC/FKL ${knock.toStringAsFixed(1)}°';
    } else {
      final clean = knock > -0.2 && (iam.isNaN || iam >= 0.999);
      final hot = !iat.isNaN && iat > 60;
      final afrOk = afr.isNaN || !boosted || (afr >= p.afrBoost - 0.6 && afr <= p.afrBoost + 0.6);
      final highLoad = t.yValues.isEmpty ? true : (y >= t.yValues.reduce(math.max) * 0.45);
      if (clean && !hot && afrOk && highLoad && s.length >= minSamples * 2) {
        sug = cur + p.timingStep;
        conf = 0.5;
        why = '+${p.timingStep.toStringAsFixed(1)}° чисто (IAM 1.0, ${s.length} точек)';
      } else {
        return null;
      }
    }
    sug = sug.clamp(cur - 3.0, cur + p.timingStep).toDouble();
    sug = sug.clamp(-10.0, 60.0).toDouble();
    if ((sug - cur).abs() < 0.25) return null;
    return MapCellChange(row: ri, col: ci, x: x, y: y, current: cur, suggested: sug,
        confidence: conf, samples: s.length, reason: why);
  }

  static MapCellChange? _fuelOl(List<Map<String, double>> s, double cur, int ri, int ci,
      double x, double y, TuningPattern p) {
    // только открытый контур: полный газ или буст
    final ol = s.where((r) => (r['tps'] ?? 0.0) > 60 || (r['boost'] ?? -1.0) > 0.05).toList();
    if (ol.length < minSamples) return null;
    final m = _avg(_col(ol, 'afr'));
    if (m.isNaN || m < 8 || m > 20) return null;
    final boost = _avg(_col(ol, 'boost'));
    final lambdaTable = cur < 2.0; // карта в λ, а не в AFR
    final curAfr = lambdaTable ? cur * AppConstants.stoich : cur;
    final desired = (!boost.isNaN && boost > 0.3) ? p.afrBoost : p.afrWot;
    if ((m - desired).abs() < 0.3) return null;
    var sugAfr = desired * curAfr / m; // компенсируем ошибку доставки топлива
    sugAfr = sugAfr.clamp(curAfr * 0.85, curAfr * 1.15).toDouble();
    sugAfr = sugAfr.clamp(9.5, 16.0).toDouble();
    var conf = math.min(0.9, 0.5 + ol.length / 20.0);
    var why = 'AFR ${m.toStringAsFixed(1)} → цель ${desired.toStringAsFixed(1)}';
    if (!boost.isNaN && boost > 0.3 && m > AppConstants.afrBoostLean) {
      conf = 0.95;
      why = 'БЕДНО ${m.toStringAsFixed(1)} под бустом!';
    }
    final sug = lambdaTable ? sugAfr / AppConstants.stoich : sugAfr;
    if ((sug - cur).abs() < (lambdaTable ? 0.005 : 0.05)) return null;
    return MapCellChange(row: ri, col: ci, x: x, y: y, current: cur, suggested: sug,
        confidence: conf, samples: ol.length, reason: why);
  }

  static MapCellChange? _maf(List<Map<String, double>> s, double cur, int ri, int ci,
      double x, double y, TuningPattern p) {
    // замкнутый контур: без буста и не полный газ
    final cl = s.where((r) => (r['tps'] ?? 0.0) < 60 && (r['boost'] ?? -1.0) < 0.05).toList();
    if (cl.length < minSamples * 2) return null;
    final tr = _avg(cl.map((r) => (r['stft'] ?? 0.0) + (r['ltft'] ?? 0.0)));
    if (tr.isNaN || tr.abs() < p.trimThreshold) return null;
    final sug = (cur * (1 + tr / 100.0)).clamp(cur * 0.8, cur * 1.2).toDouble();
    final conf = math.min(0.9, 0.45 + cl.length / 30.0);
    return MapCellChange(row: ri, col: ci, x: x, y: y, current: cur, suggested: sug,
        confidence: conf, samples: cl.length,
        reason: 'STFT+LTFT ${tr > 0 ? '+' : ''}${tr.toStringAsFixed(1)}%');
  }

  static MapCellChange? _wgd(List<Map<String, double>> s, double cur, int ri, int ci,
      double x, double y, TuningPattern p) {
    final b = s.where((r) => r['boost'] != null && r['tboost'] != null && r['tboost']! > 0.1).toList();
    if (b.length < minSamples) return null;
    final err = _avg(b.map((r) => r['boost']! - r['tboost']!));
    final boost = _avg(_col(b, 'boost'));
    if (err.isNaN || err.abs() < p.boostErrTol) return null;
    var sug = cur - (err / 0.1) * p.wgdGain; // перебуст → меньше duty
    sug = sug.clamp(cur - 10.0, cur + 10.0).toDouble();
    sug = sug.clamp(0.0, 100.0).toDouble();
    var conf = math.min(0.85, 0.5 + b.length / 20.0);
    var why = 'Буст ${boost.toStringAsFixed(2)} vs цель, Δ ${err > 0 ? '+' : ''}${err.toStringAsFixed(2)} бар';
    if (!boost.isNaN && boost >= AppConstants.overboostDanger) {
      conf = 0.95;
      why = 'ПЕРЕБУСТ ${boost.toStringAsFixed(2)} бар';
    }
    if ((sug - cur).abs() < 0.5) return null;
    return MapCellChange(row: ri, col: ci, x: x, y: y, current: cur, suggested: sug,
        confidence: conf, samples: b.length, reason: why);
  }

  static MapCellChange? _targetBoost(List<Map<String, double>> s, double cur, int ri, int ci,
      double x, double y, TuningPattern p) {
    final fb = _avg(_col(s, 'fbkc'));
    final fk = _avg(_col(s, 'fkl'));
    final boost = _avg(_col(s, 'boost'));
    if (boost.isNaN || boost < 0.3) return null;
    final knock = math.min(fb.isNaN ? 0.0 : fb, fk.isNaN ? 0.0 : fk);
    if (knock > p.knockDanger) return null; // цель буста снижаем только при детонации
    final psi = cur > 5.0; // карта в psi или в бар
    final step = psi ? 1.5 : 0.1;
    final sug = math.max(psi ? 0.0 : -0.5, cur - step);
    return MapCellChange(row: ri, col: ci, x: x, y: y, current: cur, suggested: sug,
        confidence: 0.8, samples: s.length,
        reason: 'Детонация ${knock.toStringAsFixed(1)}° под бустом ${boost.toStringAsFixed(2)}');
  }

  /// Применить правки к копии карты
  static RomTable apply(RomTable base, List<MapCellChange> changes) {
    final t = RomTable(
      def: base.def,
      xValues: List<double>.of(base.xValues),
      yValues: List<double>.of(base.yValues),
      z: base.z.map((r) => List<double>.of(r)).toList(),
    );
    for (final c in changes) {
      if (c.row < t.z.length && c.col < t.z[c.row].length) t.z[c.row][c.col] = c.suggested;
    }
    return t;
  }
}
'''

NEW_FILES['lib/services/rom_writer.dart'] = r'''// TUNE-V6-BRIDGE
import 'dart:typed_data';

import '../models/rom_table.dart';

/// Единый кодек записи ячеек в ROM: все storage/endian из ECUFlash-дефинишена.
class RomWriter {
  static Endian _e(RomCol c) => c.endian == 'little' ? Endian.little : Endian.big;

  static List<int> encode(RomCol col, double physical) {
    final fr = col.fr;
    if (fr == null || physical.isNaN) return const <int>[];
    final raw = fr(physical);
    final bd = ByteData(4);
    switch (col.storage) {
      case 'float':
        bd.setFloat32(0, raw, _e(col));
        return bd.buffer.asUint8List(0, 4).toList();
      case 'uint32':
        bd.setUint32(0, raw.round().clamp(0, 0xFFFFFFFF).toInt(), _e(col));
        return bd.buffer.asUint8List(0, 4).toList();
      case 'int32':
        bd.setInt32(0, raw.round().clamp(-2147483648, 2147483647).toInt(), _e(col));
        return bd.buffer.asUint8List(0, 4).toList();
      case 'uint16':
        bd.setUint16(0, raw.round().clamp(0, 65535).toInt(), _e(col));
        return bd.buffer.asUint8List(0, 2).toList();
      case 'int16':
        bd.setInt16(0, raw.round().clamp(-32768, 32767).toInt(), _e(col));
        return bd.buffer.asUint8List(0, 2).toList();
      case 'int8':
        return <int>[raw.round().clamp(-128, 127).toInt() & 0xFF];
      default:
        return <int>[raw.round().clamp(0, 255).toInt()];
    }
  }

  /// Записывает значения таблицы в буфер (индексация как в RomService.readTable)
  static int applyTable(List<int> buf, RomTable t) {
    final def = t.def;
    if (!def.editable) return 0;
    final sz = def.data.sizeOf;
    var n = 0;
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        if (r >= t.z.length || c >= t.z[r].length) continue;
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        final off = def.address + i * sz;
        final bytes = encode(def.data, t.z[r][c]);
        if (bytes.isEmpty || off + bytes.length > buf.length) continue;
        for (var k = 0; k < bytes.length; k++) {
          buf[off + k] = bytes[k];
        }
        n++;
      }
    }
    return n;
  }

  /// Копия ROM с записанными картами (оригинал не трогаем)
  static List<int> buildMod(List<int> rom, Iterable<RomTable> tables) {
    final mod = List<int>.of(rom);
    for (final t in tables) {
      applyTable(mod, t);
    }
    return mod;
  }

  /// V8MOD{n}_{base}.bin — номер растёт, если исходник уже был модом
  static String modName(String fileName) {
    var base = fileName.replaceAll(RegExp(r'\.(bin|hex|rom)$', caseSensitive: false), '');
    var n = 1;
    final m = RegExp(r'^V8MOD(\d*)_').firstMatch(base);
    if (m != null) {
      n = (int.tryParse(m.group(1) ?? '') ?? 1) + 1;
      base = base.substring(m.end);
    }
    if (base.isEmpty) base = 'rom';
    return 'V8MOD${n}_$base.bin';
  }
}

class ChecksumReport {
  final bool supported;
  final bool inclusiveEnd;
  final int blocks;
  final int valid;
  final int fixed;
  final String message;
  const ChecksumReport({
    required this.supported,
    required this.inclusiveEnd,
    required this.blocks,
    required this.valid,
    required this.fixed,
    required this.message,
  });
  bool get ok => supported && blocks > 0 && valid == blocks;
}

/// Контрольные суммы Subaru 32-bit (SH7055/SH7058): таблица из 8 записей
/// (start, end, value) по адресу romSize−0x480; сумма 32-битных слов блока
/// (с учётом value) должна давать 0x5AA5A55A. Формат ПОДТВЕРЖДАЕТСЯ по оригиналу
/// перед пересчётом — если оригинал не сходится, байты мода не трогаем.
class SubaruChecksum {
  static const int magic = 0x5AA5A55A;
  static const int entries = 8;
  static const int mask = 0xFFFFFFFF;

  static int _u32(List<int> b, int off) =>
      ((b[off] & 0xFF) << 24) | ((b[off + 1] & 0xFF) << 16) | ((b[off + 2] & 0xFF) << 8) | (b[off + 3] & 0xFF);

  static void _put32(List<int> b, int off, int v) {
    b[off] = (v >> 24) & 0xFF;
    b[off + 1] = (v >> 16) & 0xFF;
    b[off + 2] = (v >> 8) & 0xFF;
    b[off + 3] = v & 0xFF;
  }

  static int tableOffset(int len) => len - 0x480;

  static const _unsupported = ChecksumReport(
      supported: false, inclusiveEnd: false, blocks: 0, valid: 0, fixed: 0,
      message: 'таблица КС не распознана');

  static bool _layoutOk(List<int> rom) {
    if (rom.length < 0x80000 || rom.length % 4 != 0) return false;
    final t = tableOffset(rom.length);
    for (var i = 0; i < entries; i++) {
      final s = _u32(rom, t + i * 12);
      final e = _u32(rom, t + i * 12 + 4);
      if (s % 4 != 0 || e % 4 != 0 || s >= e || e > rom.length) return false;
    }
    return true;
  }

  static int _blockSum(List<int> rom, int s, int e) {
    var sum = 0;
    for (var a = s; a + 4 <= e; a += 4) {
      sum = (sum + _u32(rom, a)) & mask;
    }
    return sum;
  }

  static ChecksumReport _verify(List<int> rom, bool incl) {
    if (!_layoutOk(rom)) return _unsupported;
    final t = tableOffset(rom.length);
    var valid = 0;
    for (var i = 0; i < entries; i++) {
      final s = _u32(rom, t + i * 12);
      var e = _u32(rom, t + i * 12 + 4);
      final chkOff = t + i * 12 + 8;
      final stored = _u32(rom, chkOff);
      if (incl) e += 4;
      if (e > rom.length) return _unsupported;
      final inside = chkOff >= s && chkOff < e;
      final sum = _blockSum(rom, s, e);
      final total = inside ? sum : (sum + stored) & mask;
      if (total == magic) valid++;
    }
    return ChecksumReport(
        supported: true, inclusiveEnd: incl, blocks: entries, valid: valid, fixed: 0,
        message: valid == entries ? 'КС сходится ($entries/$entries)' : 'КС не сходится: $valid/$entries');
  }

  /// Проверка: пробует обе трактовки границы блока
  static ChecksumReport verify(List<int> rom) {
    final a = _verify(rom, false);
    if (a.ok) return a;
    final b = _verify(rom, true);
    return b.ok ? b : a;
  }

  static ChecksumReport _fixAll(List<int> rom, bool incl) {
    final t = tableOffset(rom.length);
    var fixed = 0;
    for (var pass = 0; pass < 2; pass++) {
      for (var i = 0; i < entries; i++) {
        final s = _u32(rom, t + i * 12);
        var e = _u32(rom, t + i * 12 + 4);
        final chkOff = t + i * 12 + 8;
        final stored = _u32(rom, chkOff);
        if (incl) e += 4;
        final inside = chkOff >= s && chkOff < e;
        final sum = _blockSum(rom, s, e);
        final want = inside ? (magic - ((sum - stored) & mask)) & mask : (magic - sum) & mask;
        if (want != stored) {
          _put32(rom, chkOff, want);
          if (pass == 0) fixed++;
        }
      }
    }
    final v = _verify(rom, incl);
    return ChecksumReport(
        supported: true, inclusiveEnd: incl, blocks: entries, valid: v.valid, fixed: fixed,
        message: v.ok
            ? 'КС пересчитана: исправлено $fixed блок(ов), сходится $entries/$entries'
            : 'КС пересчитать не удалось (${v.valid}/$entries)');
  }

  /// Пересчёт КС в [mod]. [reference] — исходный .bin: по нему подтверждаем формат.
  static ChecksumReport fix(List<int> mod, {required List<int> reference}) {
    if (mod.length != reference.length) {
      return const ChecksumReport(supported: false, inclusiveEnd: false, blocks: 0, valid: 0, fixed: 0,
          message: 'размер мода не равен оригиналу');
    }
    final ref = verify(reference);
    if (!ref.ok) {
      return ChecksumReport(supported: false, inclusiveEnd: false, blocks: ref.blocks, valid: ref.valid, fixed: 0,
          message: 'Формат КС оригинала не подтверждён (${ref.message}) — байты не тронуты');
    }
    return _fixAll(mod, ref.inclusiveEnd);
  }

  /// Для тестов/синтетики: пересчёт без сверки с оригиналом
  static ChecksumReport forceFix(List<int> rom, {bool inclusiveEnd = false}) =>
      _layoutOk(rom) ? _fixAll(rom, inclusiveEnd) : _unsupported;
}
'''

NEW_FILES['lib/services/map_storage_service.dart'] = r'''// TUNE-V6-BRIDGE
import 'dart:convert';

import 'package:shared_preferences/shared_preferences.dart';

import '../models/rom_table.dart';

/// Слой «дефолтных значений» как в V6: правки карт хранятся по адресу карты
/// и накладываются поверх значений из .bin при открытии/анализе.
class MapStorageService {
  static const _prefix = 'v8_map_edit_';
  static const _listKey = 'v8_map_edit_list';
  static SharedPreferences? _p;

  static Future<SharedPreferences> _sp() async => _p ??= await SharedPreferences.getInstance();

  static List<List<double?>> _clean(List<List<double>> z) =>
      z.map((r) => r.map((v) => v.isNaN ? null : v).toList()).toList();

  static Future<void> saveMap(RomTable t) async {
    final p = await _sp();
    final key = _prefix + t.def.addrHex;
    await p.setString(key, jsonEncode({
      'name': t.def.name,
      'address': t.def.addrHex,
      'rows': t.def.rows,
      'cols': t.def.cols,
      'z': _clean(t.z),
      'updatedAt': DateTime.now().toIso8601String(),
    }));
    final list = p.getStringList(_listKey) ?? <String>[];
    if (!list.contains(t.def.addrHex)) {
      list.add(t.def.addrHex);
      await p.setStringList(_listKey, list);
    }
  }

  static Future<List<List<double>>?> loadZ(String addrHex) async {
    final p = await _sp();
    final s = p.getString(_prefix + addrHex);
    if (s == null) return null;
    try {
      final j = jsonDecode(s) as Map<String, dynamic>;
      final raw = j['z'] as List;
      return raw
          .map<List<double>>((row) =>
              (row as List).map<double>((v) => v == null ? double.nan : (v as num).toDouble()).toList())
          .toList();
    } catch (_) {
      return null;
    }
  }

  static Future<DateTime?> updatedAt(String addrHex) async {
    final p = await _sp();
    final s = p.getString(_prefix + addrHex);
    if (s == null) return null;
    try {
      final j = jsonDecode(s) as Map<String, dynamic>;
      return DateTime.tryParse((j['updatedAt'] as String?) ?? '');
    } catch (_) {
      return null;
    }
  }

  static Future<bool> hasEdits(String addrHex) async => (await _sp()).containsKey(_prefix + addrHex);

  static Future<List<String>> editedAddresses() async =>
      (await _sp()).getStringList(_listKey) ?? <String>[];

  static Future<void> resetMap(String addrHex) async {
    final p = await _sp();
    await p.remove(_prefix + addrHex);
    final list = p.getStringList(_listKey) ?? <String>[];
    list.remove(addrHex);
    await p.setStringList(_listKey, list);
  }

  static Future<void> resetAll() async {
    final p = await _sp();
    final list = p.getStringList(_listKey) ?? <String>[];
    for (final a in list) {
      await p.remove(_prefix + a);
    }
    await p.remove(_listKey);
  }

  /// Наложить сохранённые правки на таблицу из ROM (если размеры совпали)
  static Future<bool> applyTo(RomTable t) async {
    final z = await loadZ(t.def.addrHex);
    if (z == null || z.length != t.def.rows) return false;
    for (var r = 0; r < t.def.rows; r++) {
      if (z[r].length != t.def.cols) return false;
    }
    for (var r = 0; r < t.def.rows; r++) {
      for (var c = 0; c < t.def.cols; c++) {
        t.z[r][c] = z[r][c];
      }
    }
    return true;
  }
}
'''

NEW_FILES['lib/services/map_history_service.dart'] = r'''// TUNE-V6-BRIDGE
import 'dart:convert';

import 'package:intl/intl.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../models/rom_table.dart';

class MapRevision {
  final String addrHex;
  final String name;
  final int version;
  final DateTime savedAt;
  final List<List<double>> z;
  final String comment;

  const MapRevision({
    required this.addrHex,
    required this.name,
    required this.version,
    required this.savedAt,
    required this.z,
    this.comment = '',
  });

  String get label =>
      'v$version • ${DateFormat('dd.MM HH:mm').format(savedAt)}${comment.isNotEmpty ? ' — $comment' : ''}';

  Map<String, dynamic> toJson() => {
        'a': addrHex,
        'n': name,
        'v': version,
        't': savedAt.toIso8601String(),
        'z': z.map((r) => r.map((v) => v.isNaN ? null : v).toList()).toList(),
        'c': comment,
      };

  factory MapRevision.fromJson(Map<String, dynamic> j) => MapRevision(
        addrHex: j['a'] as String,
        name: j['n'] as String,
        version: (j['v'] as num).toInt(),
        savedAt: DateTime.parse(j['t'] as String),
        z: (j['z'] as List)
            .map<List<double>>((r) =>
                (r as List).map<double>((v) => v == null ? double.nan : (v as num).toDouble()).toList())
            .toList(),
        comment: (j['c'] as String?) ?? '',
      );
}

/// История правок карты — до 20 ревизий, как в V6.
class MapHistoryService {
  static const _prefix = 'v8_map_hist_';
  static const maxRevs = 20;
  static SharedPreferences? _p;

  static Future<SharedPreferences> _sp() async => _p ??= await SharedPreferences.getInstance();

  static Future<List<MapRevision>> getHistory(String addrHex) async {
    final s = (await _sp()).getString(_prefix + addrHex);
    if (s == null) return [];
    try {
      return (jsonDecode(s) as List).map((j) => MapRevision.fromJson(j as Map<String, dynamic>)).toList();
    } catch (_) {
      return [];
    }
  }

  static Future<int> saveRevision(RomTable t, {String comment = ''}) async {
    final p = await _sp();
    final hist = await getHistory(t.def.addrHex);
    final v = hist.isEmpty ? 1 : hist.last.version + 1;
    hist.add(MapRevision(
      addrHex: t.def.addrHex,
      name: t.def.name,
      version: v,
      savedAt: DateTime.now(),
      z: t.z.map((r) => List<double>.of(r)).toList(),
      comment: comment.isEmpty ? 'Правка $v' : comment,
    ));
    if (hist.length > maxRevs) hist.removeRange(0, hist.length - maxRevs);
    await p.setString(_prefix + t.def.addrHex, jsonEncode(hist.map((r) => r.toJson()).toList()));
    return v;
  }

  static Future<void> clear(String addrHex) async {
    final p = await _sp();
    await p.remove(_prefix + addrHex);
  }
}
'''

NEW_FILES['lib/services/pending_edits.dart'] = r'''// TUNE-V6-BRIDGE
import 'package:flutter/foundation.dart';

import '../models/rom_table.dart';

class PendingEntry {
  final RomTable original;
  final RomTable updated;
  final String source;
  final DateTime at;
  bool selected;

  PendingEntry({required this.original, required this.updated, required this.source, this.selected = true})
      : at = DateTime.now();

  String get key => original.def.addrHex;
  String get name => original.def.name;

  int get changeCount {
    var n = 0;
    for (var r = 0; r < updated.z.length && r < original.z.length; r++) {
      for (var c = 0; c < updated.z[r].length && c < original.z[r].length; c++) {
        final a = original.z[r][c], b = updated.z[r][c];
        if (a.isNaN || b.isNaN) continue;
        if ((a - b).abs() > 1e-9) n++;
      }
    }
    return n;
  }
}

/// Очередь карт на пакетную запись в .bin (как PendingEdits в V6).
class PendingEdits extends ChangeNotifier {
  static final PendingEdits I = PendingEdits._();
  PendingEdits._();

  final Map<String, PendingEntry> _m = {};

  int get count => _m.length;
  bool get isEmpty => _m.isEmpty;
  List<PendingEntry> get all => _m.values.toList();
  bool has(String key) => _m.containsKey(key);
  PendingEntry? get(String key) => _m[key];

  void addOrUpdate(RomTable original, RomTable updated, {String source = 'ручная правка'}) {
    final e = PendingEntry(original: original, updated: updated, source: source);
    if (e.changeCount == 0) {
      _m.remove(e.key);
    } else {
      _m[e.key] = e;
    }
    notifyListeners();
  }

  void remove(String key) {
    _m.remove(key);
    notifyListeners();
  }

  void toggle(String key) {
    final e = _m[key];
    if (e != null) {
      e.selected = !e.selected;
      notifyListeners();
    }
  }

  void clear() {
    _m.clear();
    notifyListeners();
  }
}
'''

NEW_FILES['lib/services/rom_holder.dart'] = r'''// TUNE-V6-BRIDGE
import '../generated/subaru_rom.g.dart';
import 'connection_service.dart';
import 'map_storage_service.dart';
import 'pending_edits.dart';
import 'rom_service.dart';

/// Глобальный держатель .bin: загрузка нового ROM = сброс кеша правок (как в V6).
class RomHolder {
  static final RomHolder I = RomHolder._();
  RomHolder._();

  RomService get rom => ConnectionService.I.rom;
  bool get loaded => rom.loaded;
  String get fileName => rom.fileName;

  Future<String?> loadNewRom() async {
    final err = await rom.pickAndLoad();
    if (err == null && rom.loaded) {
      await MapStorageService.resetAll();
      PendingEdits.I.clear();
    }
    return err;
  }

  /// Cal ID из образа (ASCII в районе 0x2000) — для сверки с дефинишеном
  String? calIdInRom() {
    final b = rom.rom;
    if (b == null || b.length < 0x4000) return null;
    final re = RegExp(r'[A-Z]\d[A-Z]{2}\d{3}[A-Z]');
    String ascii(int from, int len) =>
        String.fromCharCodes(b.sublist(from, from + len).map((c) => (c >= 32 && c < 127) ? c : 46));
    for (final off in [0x2000, 0x2004, 0x200C]) {
      final s = ascii(off, 8);
      if (re.hasMatch(s)) return s;
    }
    final m = re.firstMatch(ascii(0, 0x4000));
    return m?.group(0);
  }

  bool get calIdMatches {
    final id = calIdInRom();
    return id != null && id == SubaruRom.calId;
  }
}
'''

NEW_FILES['lib/widgets/map_table_view.dart'] = r'''// TUNE-V6-BRIDGE
import 'package:flutter/material.dart';

import '../models/rom_table.dart';
import 'heat_colors.dart';

/// Таблица карты «как в V6»: заголовки осей, тепловая заливка, изменённые
/// ячейки — синие со старым (зачёркнутым) и новым значением.
class MapTableView extends StatelessWidget {
  final RomTable table;
  final RomTable? baseline;
  final double cell;
  final void Function(int r, int c)? onTap;
  final String cornerLabel;

  const MapTableView({
    super.key,
    required this.table,
    this.baseline,
    this.cell = 62,
    this.onTap,
    this.cornerLabel = 'Y \\ X',
  });

  static String fmt(double v) {
    if (v.isNaN) return '—';
    final a = v.abs();
    if (a >= 100) return v.toStringAsFixed(0);
    if (a >= 10) return v.toStringAsFixed(1);
    return v.toStringAsFixed(2);
  }

  static String fmtAxis(double v) {
    if (v.isNaN) return '—';
    final a = v.abs();
    if (a >= 100) return v.toStringAsFixed(0);
    if (a >= 10) return v.toStringAsFixed(1);
    return v.toStringAsFixed(2);
  }

  static const _head = Color(0xFF14264F);

  @override
  Widget build(BuildContext context) {
    final def = table.def;
    var lo = double.infinity, hi = -double.infinity;
    for (final row in table.z) {
      for (final v in row) {
        if (v.isNaN) continue;
        if (v < lo) lo = v;
        if (v > hi) hi = v;
      }
    }
    if (lo == double.infinity) {
      lo = 0;
      hi = 1;
    }
    if (hi - lo < 1e-9) hi = lo + 1;
    final hasY = def.rows > 1;
    final labelW = cell * 0.9;
    final h = cell * 0.78;

    Widget label(String s, double w, double hh, {double fs = 10}) => Container(
          width: w,
          height: hh,
          alignment: Alignment.center,
          decoration: BoxDecoration(color: _head, border: Border.all(color: HeatColors.grid, width: 0.5)),
          child: Text(s, style: TextStyle(fontSize: fs, color: HeatColors.dim)),
        );

    final rows = <Widget>[
      Row(children: [
        label(cornerLabel, labelW, h * 0.7, fs: 9),
        for (var c = 0; c < def.cols; c++)
          label(c < table.xValues.length ? fmtAxis(table.xValues[c]) : '$c', cell, h * 0.7),
      ]),
    ];
    for (var r = 0; r < def.rows; r++) {
      rows.add(Row(children: [
        label(hasY && r < table.yValues.length ? fmtAxis(table.yValues[r]) : (hasY ? '$r' : ''), labelW, h),
        for (var c = 0; c < def.cols; c++) _cell(r, c, lo, hi, h),
      ]));
    }
    return SingleChildScrollView(
      scrollDirection: Axis.horizontal,
      child: SingleChildScrollView(
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: rows),
      ),
    );
  }

  Widget _cell(int r, int c, double lo, double hi, double h) {
    final v = table.z[r][c];
    final b = baseline;
    final old = (b != null && r < b.z.length && c < b.z[r].length) ? b.z[r][c] : v;
    final changed = !v.isNaN && !old.isNaN && (v - old).abs() > 1e-9;
    final t = v.isNaN ? 0.0 : ((v - lo) / (hi - lo)).clamp(0.0, 1.0).toDouble();
    final bg = changed
        ? const Color(0xFF2962FF)
        : Color.alphaBlend(heatColor(t).withAlpha(150), HeatColors.panel);
    return GestureDetector(
      onTap: onTap == null ? null : () => onTap!(r, c),
      child: Container(
        width: cell,
        height: h,
        alignment: Alignment.center,
        decoration: BoxDecoration(color: bg, border: Border.all(color: HeatColors.grid, width: 0.5)),
        child: changed
            ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
                Text(fmt(old),
                    style: const TextStyle(
                        fontSize: 9, color: Color(0xFFB0BEC5), decoration: TextDecoration.lineThrough)),
                Text(fmt(v),
                    style: const TextStyle(fontSize: 12, fontWeight: FontWeight.w800, color: Color(0xFFFFEB3B))),
              ])
            : Text(fmt(v), style: const TextStyle(fontSize: 11, color: HeatColors.text)),
      ),
    );
  }
}
'''

NEW_FILES['lib/widgets/fuel_card.dart'] = r'''// TUNE-V6-BRIDGE
import 'package:flutter/material.dart';

import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import 'heat_colors.dart';

/// Ячейка мгновенного расхода под картой буста: л/ч, л/100 км, загрузка форсунок.
class FuelCard extends StatelessWidget {
  const FuelCard({super.key});

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    return StreamBuilder<LiveSnapshot>(
      stream: svc.poller?.snapshots,
      builder: (ctx, snap) {
        final s = snap.data ?? svc.poller?.last;
        final lph = s?.fuelLph ?? 0.0;
        final speed = s?.speed ?? 0.0;
        final l100 = (s != null && speed > 5) ? lph / speed * 100.0 : null;
        // 4-тактный: 1 впрыск на 2 оборота → duty% = мс × об/мин / 1200
        final duty = (s != null && s.rpm > 0) ? (s.injMs * s.rpm / 1200.0) : 0.0;
        final hasSrc = s != null && s.has('maf') && s.has('afr');
        final t = (lph / 60.0).clamp(0.0, 1.0).toDouble();
        final dutyColor = duty >= 85
            ? Colors.redAccent
            : (duty >= 75 ? Colors.orangeAccent : HeatColors.dim);
        return Container(
          margin: const EdgeInsets.fromLTRB(10, 0, 10, 10),
          padding: const EdgeInsets.all(12),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Row(children: [
                const Text('РАСХОД', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
                const SizedBox(width: 6),
                Text(hasSrc ? 'MAF/AFR' : 'нет MAF/AFR',
                    style: TextStyle(fontSize: 9, color: hasSrc ? HeatColors.dim : Colors.orangeAccent)),
                const Spacer(),
                Text(lph.toStringAsFixed(1),
                    style: const TextStyle(fontSize: 22, fontWeight: FontWeight.w800, color: HeatColors.text)),
                const Text(' л/ч', style: TextStyle(fontSize: 12, color: HeatColors.dim)),
              ]),
              const SizedBox(height: 4),
              Row(children: [
                Text(l100 == null ? '— л/100 км (стоим)' : '${l100.toStringAsFixed(1)} л/100 км',
                    style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
                const Spacer(),
                Text('форсунки ${duty.toStringAsFixed(0)} %', style: TextStyle(fontSize: 12, color: dutyColor)),
              ]),
              const SizedBox(height: 6),
              ClipRRect(
                borderRadius: BorderRadius.circular(6),
                child: LinearProgressIndicator(
                  value: t,
                  minHeight: 8,
                  backgroundColor: HeatColors.bg,
                  valueColor: AlwaysStoppedAnimation(heatColor(t)),
                ),
              ),
            ],
          ),
        );
      },
    );
  }
}
'''

NEW_FILES['lib/screens/rom_table_edit_screen.dart'] = r'''// TUNE-V6-BRIDGE
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../models/rom_table.dart';
import '../services/connection_service.dart';
import '../services/map_history_service.dart';
import '../services/map_storage_service.dart';
import '../services/pending_edits.dart';
import '../widgets/heat_colors.dart';
import '../widgets/map_table_view.dart';

/// Редактор карты «как в V6»: Сохранить / Сброс / зум, тап по ячейке — правка.
class RomTableEditScreen extends StatefulWidget {
  final RomTableDef def;
  final RomTable? proposal; // предложение анализатора (ещё не сохранено)
  final String? proposalLabel;
  const RomTableEditScreen({super.key, required this.def, this.proposal, this.proposalLabel});

  @override
  State<RomTableEditScreen> createState() => _RomTableEditScreenState();
}

class _RomTableEditScreenState extends State<RomTableEditScreen> {
  RomTable? _base;
  RomTable? _cur;
  DateTime? _updatedAt;
  double _cell = 62;
  bool _dirty = false;
  String? _err;

  static RomTable copyOf(RomTable t) => RomTable(
        def: t.def,
        xValues: List<double>.of(t.xValues),
        yValues: List<double>.of(t.yValues),
        z: t.z.map((r) => List<double>.of(r)).toList(),
      );

  @override
  void initState() {
    super.initState();
    _load();
  }

  Future<void> _load() async {
    final rom = ConnectionService.I.rom;
    if (!rom.loaded) {
      if (mounted) setState(() => _err = 'ROM не загружен: вкладка ROM → «Загрузить .bin»');
      return;
    }
    final base = rom.readTable(widget.def);
    final cur = copyOf(widget.proposal ?? base);
    DateTime? at;
    if (widget.proposal == null) {
      final ok = await MapStorageService.applyTo(cur);
      if (ok) at = await MapStorageService.updatedAt(widget.def.addrHex);
    }
    if (!mounted) return;
    setState(() {
      _base = base;
      _cur = cur;
      _updatedAt = at;
      _dirty = widget.proposal != null;
    });
  }

  int get _changes {
    final b = _base, c = _cur;
    if (b == null || c == null) return 0;
    var n = 0;
    for (var r = 0; r < c.z.length && r < b.z.length; r++) {
      for (var k = 0; k < c.z[r].length && k < b.z[r].length; k++) {
        if ((c.z[r][k] - b.z[r][k]).abs() > 1e-9) n++;
      }
    }
    return n;
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  Future<void> _save() async {
    final cur = _cur, base = _base;
    if (cur == null || base == null) return;
    await MapStorageService.saveMap(cur);
    await MapHistoryService.saveRevision(cur, comment: widget.proposalLabel ?? '');
    PendingEdits.I.addOrUpdate(base, cur, source: widget.proposalLabel ?? 'ручная правка');
    if (!mounted) return;
    setState(() {
      _updatedAt = DateTime.now();
      _dirty = false;
    });
    _snack(
        _changes == 0
            ? 'Сохранено (совпадает со стоком — из очереди убрано)'
            : 'Сохранено · в очереди на запись: ${PendingEdits.I.count}',
        Colors.green);
  }

  Future<void> _reset() async {
    final choice = await showDialog<String>(
      context: context,
      builder: (c) => SimpleDialog(backgroundColor: HeatColors.panel, title: const Text('Сброс'), children: [
        SimpleDialogOption(onPressed: () => Navigator.pop(c, 'stock'), child: const Text('К стоку из .bin')),
        SimpleDialogOption(onPressed: () => Navigator.pop(c, 'hist'), child: const Text('Выбрать из истории правок')),
        SimpleDialogOption(onPressed: () => Navigator.pop(c), child: const Text('Отмена')),
      ]),
    );
    if (choice == 'stock') {
      await _toStock();
    } else if (choice == 'hist') {
      await _fromHistory();
    }
  }

  Future<void> _toStock() async {
    final base = _base;
    if (base == null) return;
    await MapStorageService.resetMap(widget.def.addrHex);
    PendingEdits.I.remove(widget.def.addrHex);
    if (!mounted) return;
    setState(() {
      _cur = copyOf(base);
      _updatedAt = null;
      _dirty = false;
    });
    _snack('Сброшено к стоку', Colors.orange);
  }

  Future<void> _fromHistory() async {
    final hist = await MapHistoryService.getHistory(widget.def.addrHex);
    if (!mounted) return;
    if (hist.isEmpty) {
      _snack('История пуста', Colors.orange);
      return;
    }
    final rev = await showModalBottomSheet<MapRevision>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (_) => ListView(children: [
        for (final r in hist.reversed)
          ListTile(
              leading: const Icon(Icons.history, color: HeatColors.gold),
              title: Text(r.label, style: const TextStyle(fontSize: 13)),
              onTap: () => Navigator.pop(context, r)),
      ]),
    );
    if (rev == null) return;
    final cur = _cur;
    if (cur == null) return;
    if (rev.z.length != cur.z.length) {
      _snack('Размер ревизии не совпадает с картой', Colors.red);
      return;
    }
    setState(() {
      for (var r = 0; r < cur.z.length; r++) {
        for (var c = 0; c < cur.z[r].length && c < rev.z[r].length; c++) {
          cur.z[r][c] = rev.z[r][c];
        }
      }
      _dirty = true;
    });
  }

  Future<void> _edit(int r, int c) async {
    final cur = _cur;
    if (cur == null || !widget.def.editable) return;
    final v0 = cur.z[r][c];
    final ctl = TextEditingController(text: v0.isNaN ? '' : MapTableView.fmt(v0));
    final mag = v0.abs();
    final steps = mag >= 100
        ? <double>[-10, -1, 1, 10]
        : (mag >= 10 ? <double>[-1, -0.5, 0.5, 1] : <double>[-0.1, -0.05, 0.05, 0.1]);
    void bump(double d) {
      final p = double.tryParse(ctl.text.replaceAll(',', '.')) ?? v0;
      ctl.text = (p + d).toStringAsFixed(3);
    }

    final x = c < cur.xValues.length ? MapTableView.fmtAxis(cur.xValues[c]) : '$c';
    final y = r < cur.yValues.length ? MapTableView.fmtAxis(cur.yValues[r]) : '$r';
    final stock = _base?.z[r][c] ?? double.nan;
    final ok = await showDialog<bool>(
      context: context,
      builder: (dc) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: Text('X $x · Y $y', style: const TextStyle(fontSize: 15)),
        content: Column(mainAxisSize: MainAxisSize.min, children: [
          Text('${widget.def.name} [${widget.def.units}]',
              style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextField(
            controller: ctl,
            autofocus: true,
            keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
            style: const TextStyle(fontSize: 22, fontWeight: FontWeight.bold),
            textAlign: TextAlign.center,
          ),
          const SizedBox(height: 8),
          Row(mainAxisAlignment: MainAxisAlignment.spaceEvenly, children: [
            for (final s in steps)
              OutlinedButton(
                  onPressed: () => bump(s),
                  child: Text(s > 0 ? '+$s' : '$s', style: const TextStyle(fontSize: 12))),
          ]),
          const SizedBox(height: 6),
          Text('Сток: ${MapTableView.fmt(stock)}', style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
        ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(dc, false), child: const Text('ОТМЕНА')),
          FilledButton(onPressed: () => Navigator.pop(dc, true), child: const Text('ПРИМЕНИТЬ')),
        ],
      ),
    );
    if (ok != true) return;
    final v = double.tryParse(ctl.text.replaceAll(',', '.'));
    if (v == null) {
      _snack('Не число', Colors.red);
      return;
    }
    setState(() {
      cur.z[r][c] = v;
      _dirty = true;
    });
  }

  @override
  Widget build(BuildContext context) {
    final def = widget.def;
    final cur = _cur, base = _base;
    final isProposal = widget.proposal != null;
    String status;
    Color sc;
    if (isProposal) {
      status = 'Предложение анализатора • не сохранено';
      sc = Colors.amber;
    } else if (_updatedAt != null) {
      status = 'С правками • ${DateFormat('dd.MM HH:mm').format(_updatedAt!)}${_dirty ? ' • не сохранено' : ''}';
      sc = Colors.orange;
    } else if (_dirty) {
      status = 'Изменено • не сохранено';
      sc = Colors.orange;
    } else {
      status = 'Сток';
      sc = HeatColors.dim;
    }
    final err = _err;
    return Scaffold(
      backgroundColor: HeatColors.bg,
      appBar: AppBar(
        backgroundColor: HeatColors.bg,
        leading: IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.pop(context)),
        title: Text(def.name, overflow: TextOverflow.ellipsis),
      ),
      body: err != null
          ? Center(child: Text(err, style: const TextStyle(color: Colors.orange)))
          : cur == null
              ? const Center(child: CircularProgressIndicator())
              : Column(children: [
                  Padding(
                    padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
                    child: Row(children: [
                      const Icon(Icons.grid_on, color: Colors.cyan, size: 20),
                      const SizedBox(width: 8),
                      Expanded(
                        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                          Text(def.name,
                              style: const TextStyle(color: Colors.cyan, fontSize: 16, fontWeight: FontWeight.bold),
                              overflow: TextOverflow.ellipsis),
                          Text(status, style: TextStyle(color: sc, fontSize: 12)),
                        ]),
                      ),
                      Text('${def.rows}x${def.cols}', style: const TextStyle(color: HeatColors.dim, fontSize: 13)),
                      IconButton(
                          icon: const Icon(Icons.remove_circle_outline, color: HeatColors.dim),
                          onPressed: () => setState(() => _cell = (_cell - 6).clamp(36.0, 110.0).toDouble())),
                      Text(_cell.toStringAsFixed(0), style: const TextStyle(color: HeatColors.dim, fontSize: 12)),
                      IconButton(
                          icon: const Icon(Icons.add_circle_outline, color: HeatColors.dim),
                          onPressed: () => setState(() => _cell = (_cell + 6).clamp(36.0, 110.0).toDouble())),
                    ]),
                  ),
                  Padding(
                    padding: const EdgeInsets.fromLTRB(12, 6, 12, 8),
                    child: Row(children: [
                      Expanded(
                        child: ElevatedButton.icon(
                          onPressed: def.editable ? _save : null,
                          icon: const Icon(Icons.save),
                          label: const Text('Сохранить', style: TextStyle(fontSize: 15)),
                          style: ElevatedButton.styleFrom(
                              backgroundColor: const Color(0xFF43A047),
                              foregroundColor: Colors.white,
                              minimumSize: const Size.fromHeight(46),
                              shape: const StadiumBorder()),
                        ),
                      ),
                      const SizedBox(width: 10),
                      Expanded(
                        child: ElevatedButton.icon(
                          onPressed: def.editable ? _reset : null,
                          icon: const Icon(Icons.history),
                          label: const Text('Сброс', style: TextStyle(fontSize: 15)),
                          style: ElevatedButton.styleFrom(
                              backgroundColor: const Color(0xFFFB8C00),
                              foregroundColor: Colors.white,
                              minimumSize: const Size.fromHeight(46),
                              shape: const StadiumBorder()),
                        ),
                      ),
                    ]),
                  ),
                  if (!def.editable)
                    Container(
                      margin: const EdgeInsets.symmetric(horizontal: 12),
                      padding: const EdgeInsets.all(8),
                      decoration: BoxDecoration(color: Colors.orange.withAlpha(30), borderRadius: BorderRadius.circular(6)),
                      child: const Text('Карта только для чтения: в дефинишене нет обратной формулы (frexpr)',
                          style: TextStyle(color: Colors.orange, fontSize: 11)),
                    ),
                  Padding(
                    padding: const EdgeInsets.symmetric(horizontal: 12),
                    child: Row(children: [
                      Text('Правок: $_changes${PendingEdits.I.has(def.addrHex) ? ' • в очереди на запись' : ''}',
                          style: const TextStyle(color: HeatColors.dim, fontSize: 11)),
                      const Spacer(),
                      Text('${def.units} · ${def.addrHex}', style: const TextStyle(color: HeatColors.dim, fontSize: 11)),
                    ]),
                  ),
                  const SizedBox(height: 4),
                  Expanded(
                    child: MapTableView(
                      table: cur,
                      baseline: base,
                      cell: _cell,
                      onTap: def.editable ? _edit : null,
                      cornerLabel: def.rows > 1 ? 'Y \\ X' : 'X',
                    ),
                  ),
                  Container(
                    color: HeatColors.panel,
                    padding: const EdgeInsets.all(6),
                    width: double.infinity,
                    child: const Text('Тап по ячейке — правка · синие ячейки: старое ⟶ новое',
                        textAlign: TextAlign.center, style: TextStyle(fontSize: 10, color: HeatColors.dim)),
                  ),
                ]),
    );
  }
}
'''

NEW_FILES['lib/screens/write_rom_screen.dart'] = r'''// TUNE-V6-BRIDGE
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';

import '../generated/subaru_rom.g.dart';
import '../services/pending_edits.dart';
import '../services/rom_holder.dart';
import '../services/rom_writer.dart';
import '../services/save_service.dart';
import '../widgets/heat_colors.dart';
import 'rom_table_edit_screen.dart';

/// Вкладка ЗАПИСЬ ROM: пакетная запись очереди карт в новый .bin (как WriteRomScreen в V6).
class WriteRomScreen extends StatefulWidget {
  const WriteRomScreen({super.key});

  @override
  State<WriteRomScreen> createState() => _WriteRomScreenState();
}

class _WriteRomScreenState extends State<WriteRomScreen> {
  String? _dir;
  bool _busy = false;
  bool _fixCs = true;

  @override
  void initState() {
    super.initState();
    PendingEdits.I.addListener(_refresh);
    V8Saver.getSavedDir().then((d) {
      if (mounted) setState(() => _dir = d);
    });
  }

  @override
  void dispose() {
    PendingEdits.I.removeListener(_refresh);
    super.dispose();
  }

  void _refresh() {
    if (mounted) setState(() {});
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  Future<void> _loadRom() async {
    final err = await RomHolder.I.loadNewRom();
    if (err != null) _snack(err, Colors.red);
    if (mounted) setState(() {});
  }

  Future<void> _pickDir() async {
    await V8Saver.requestStoragePermission();
    final p = await V8Saver.pickDirectory();
    if (p != null && mounted) setState(() => _dir = p);
  }

  Future<void> _write() async {
    final rom = RomHolder.I.rom;
    final bytesIn = rom.rom;
    if (!rom.loaded || bytesIn == null) {
      _snack('Сначала загрузи .bin прошивку', Colors.orange);
      return;
    }
    final selected = PendingEdits.I.all.where((e) => e.selected).toList();
    if (selected.isEmpty) {
      _snack('Выбери хотя бы одну карту', Colors.orange);
      return;
    }
    setState(() => _busy = true);
    try {
      final bytes = RomWriter.buildMod(bytesIn, selected.map((e) => e.updated));
      final cs = _fixCs ? SubaruChecksum.fix(bytes, reference: bytesIn) : SubaruChecksum.verify(bytes);
      await V8Saver.requestStoragePermission();
      final name = RomWriter.modName(rom.fileName);
      final res = await V8Saver.saveBytes(bytes, name);
      if (!mounted) return;
      if (res.ok) {
        _showSuccess(res, selected, cs);
      } else {
        _snack(res.error ?? 'Не удалось сохранить', Colors.red);
      }
    } finally {
      if (mounted) setState(() => _busy = false);
    }
  }

  void _showSuccess(V8SaveResult res, List<PendingEntry> written, ChecksumReport cs) {
    final csColor = cs.ok ? Colors.cyan : Colors.orange;
    showDialog<void>(
      context: context,
      builder: (c) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: const Row(children: [
          Icon(Icons.check_circle, color: Colors.green, size: 26),
          SizedBox(width: 8),
          Text('ГОТОВО'),
        ]),
        content: SingleChildScrollView(
          child: Column(mainAxisSize: MainAxisSize.min, crossAxisAlignment: CrossAxisAlignment.start, children: [
            Text('Записано карт: ${written.length}',
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ...written.map((e) => Padding(
                padding: const EdgeInsets.only(left: 8, top: 2),
                child: Text('• ${e.name} (${e.changeCount} яч.)', style: const TextStyle(fontSize: 12)))),
            const Divider(color: Colors.white24, height: 20),
            Text('Сохранено в ${res.where}:', style: const TextStyle(color: HeatColors.dim, fontSize: 12)),
            const SizedBox(height: 6),
            SelectableText(res.path ?? '',
                style: const TextStyle(color: Colors.green, fontSize: 11, fontFamily: 'monospace')),
            const SizedBox(height: 10),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                  color: csColor.withAlpha(30),
                  borderRadius: BorderRadius.circular(6),
                  border: Border.all(color: csColor.withAlpha(120))),
              child: Row(children: [
                Icon(cs.ok ? Icons.verified : Icons.warning_amber, color: csColor, size: 16),
                const SizedBox(width: 6),
                Expanded(
                    child: Text(
                        cs.ok ? cs.message : '${cs.message}. Открой мод в EcuFlash — КС пересчитается при записи.',
                        style: TextStyle(color: csColor, fontSize: 11))),
              ]),
            ),
            const SizedBox(height: 6),
            const Text('Прошивать через EcuFlash + OpenPort 2.0. По CAN из приложения ЭБУ не пишется.',
                style: TextStyle(color: HeatColors.dim, fontSize: 10)),
          ]),
        ),
        actions: [
          TextButton(
              onPressed: () async {
                await Clipboard.setData(ClipboardData(text: res.path ?? ''));
                _snack('Путь скопирован', Colors.green);
              },
              child: const Text('КОПИРОВАТЬ ПУТЬ')),
          TextButton(
              onPressed: () {
                Navigator.pop(c);
                PendingEdits.I.clear();
              },
              style: TextButton.styleFrom(foregroundColor: Colors.orange),
              child: const Text('ОЧИСТИТЬ ОЧЕРЕДЬ')),
          TextButton(onPressed: () => Navigator.pop(c), child: const Text('OK')),
        ],
      ),
    );
  }

  Widget _card({required List<Widget> children}) => Card(
        color: HeatColors.panel,
        margin: const EdgeInsets.only(bottom: 8),
        child: Padding(
            padding: const EdgeInsets.all(12),
            child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: children)),
      );

  @override
  Widget build(BuildContext context) {
    final rom = RomHolder.I;
    final pend = PendingEdits.I.all;
    final calId = rom.loaded ? rom.calIdInRom() : null;
    final calOk = calId != null && calId == SubaruRom.calId;
    final selCount = pend.where((e) => e.selected).length;
    return ListView(padding: const EdgeInsets.all(10), children: [
      _card(children: [
        Row(children: [
          const Icon(Icons.memory, color: HeatColors.gold),
          const SizedBox(width: 8),
          Expanded(
              child: Text(rom.loaded ? rom.fileName : 'Прошивка не загружена',
                  style: const TextStyle(fontWeight: FontWeight.bold), overflow: TextOverflow.ellipsis)),
          TextButton.icon(
              onPressed: _busy ? null : _loadRom,
              icon: const Icon(Icons.folder_open, size: 18),
              label: Text(rom.loaded ? 'ДРУГОЙ' : 'ЗАГРУЗИТЬ .BIN')),
        ]),
        if (rom.loaded)
          Text(
              calId == null
                  ? 'Cal ID в образе не найден (дефинишен ${SubaruRom.calId})'
                  : (calOk
                      ? 'Cal ID $calId — совпадает с дефинишеном ✓'
                      : 'Cal ID в образе: $calId ≠ дефинишен ${SubaruRom.calId} — адреса карт могут не совпадать!'),
              style: TextStyle(fontSize: 11, color: calOk ? Colors.green : Colors.orange)),
      ]),
      _card(children: [
        Row(children: [
          const Icon(Icons.folder, color: HeatColors.dim, size: 18),
          const SizedBox(width: 8),
          Expanded(
              child: Text(_dir ?? 'Папка: Downloads (по умолчанию)',
                  style: const TextStyle(fontSize: 12), overflow: TextOverflow.ellipsis)),
          TextButton(onPressed: _pickDir, child: const Text('ПАПКА')),
        ]),
        SwitchListTile(
          dense: true,
          contentPadding: EdgeInsets.zero,
          value: _fixCs,
          onChanged: (v) => setState(() => _fixCs = v),
          title: const Text('Пересчитать контрольные суммы Subaru', style: TextStyle(fontSize: 13)),
          subtitle: const Text('Только если формат КС подтверждён на оригинале',
              style: TextStyle(fontSize: 10, color: HeatColors.dim)),
        ),
      ]),
      if (pend.isEmpty)
        _card(children: const [
          Text('Очередь пуста. Правки добавляются из редактора карты («Сохранить») и из анализатора («В очередь на запись»).',
              style: TextStyle(color: HeatColors.dim, fontSize: 12)),
        ]),
      for (final e in pend)
        Card(
          color: HeatColors.panel,
          child: CheckboxListTile(
            value: e.selected,
            onChanged: (_) => PendingEdits.I.toggle(e.key),
            activeColor: Colors.green,
            title: Text(e.name, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
            subtitle: Text('${e.changeCount} яч. • ${e.source} • ${e.key}',
                style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
            secondary: IconButton(
                icon: const Icon(Icons.grid_on, color: Colors.cyan),
                onPressed: () => Navigator.push(
                    context, MaterialPageRoute<void>(builder: (_) => RomTableEditScreen(def: e.updated.def)))),
          ),
        ),
      const SizedBox(height: 8),
      SizedBox(
        height: 50,
        child: ElevatedButton.icon(
          onPressed: (_busy || selCount == 0) ? null : _write,
          icon: _busy
              ? const SizedBox(width: 18, height: 18, child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : const Icon(Icons.save_alt),
          label: Text('ЗАПИСАТЬ ВЫБРАННОЕ В .BIN ($selCount)', style: const TextStyle(fontWeight: FontWeight.bold)),
          style: ElevatedButton.styleFrom(backgroundColor: Colors.red.shade800, foregroundColor: Colors.white),
        ),
      ),
      const SizedBox(height: 8),
      TextButton(onPressed: pend.isEmpty ? null : () => PendingEdits.I.clear(), child: const Text('ОЧИСТИТЬ ОЧЕРЕДЬ')),
    ]);
  }
}
'''

NEW_FILES['lib/screens/analyzer_screen.dart'] = r'''// TUNE-V6-BRIDGE — анализатор «как в V6», адаптирован под Subaru (FBKC/FKL/IAM, AFR, trims, буст)
import 'dart:async';
import 'dart:convert';

import 'package:file_picker/file_picker.dart';
import 'package:flutter/material.dart';

import '../generated/subaru_rom.g.dart';
import '../models/live_snapshot.dart';
import '../models/rom_table.dart';
import '../models/tuning_types.dart';
import '../services/connection_service.dart';
import '../services/logger_service.dart';
import '../services/map_storage_service.dart';
import '../services/pending_edits.dart';
import '../services/rom_holder.dart';
import '../services/rom_writer.dart';
import '../services/save_service.dart';
import '../services/tuning_analyzer.dart';
import '../widgets/heat_colors.dart';
import 'heatmap_analyzer_screen.dart';
import 'rom_table_edit_screen.dart';

class AnalyzerScreen extends StatefulWidget {
  const AnalyzerScreen({super.key});

  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen> with SingleTickerProviderStateMixin {
  late final TabController _tab;
  TuningPattern _pattern = TuningPattern.stability;
  List<TuningTarget> _targets = [];
  int _ti = 0;
  String _yCanon = 'auto';

  final List<Map<String, double>> _logRows = [];
  final List<String> _logNames = [];
  final List<Map<String, double>> _liveRows = [];
  StreamSubscription<LiveSnapshot>? _liveSub;

  TuningResult? _res;
  RomTable? _orig;
  RomTable? _upd;
  bool _busy = false;

  @override
  void initState() {
    super.initState();
    _tab = TabController(length: 3, vsync: this);
    _targets = TuningAnalyzer.detectTargets(SubaruRom.tables);
  }

  @override
  void dispose() {
    _liveSub?.cancel();
    _tab.dispose();
    super.dispose();
  }

  TuningTarget get _target => _targets[_ti].copyWith(yCanon: _yCanon);

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  // ── источники данных ────────────────────────────────────────────────────
  Future<void> _pickLog({bool merge = false}) async {
    final files = await LoggerService.listLogs();
    if (!mounted) return;
    final chosen = await showModalBottomSheet<String>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (_) => ListView(children: [
        ListTile(
            leading: const Icon(Icons.folder_open, color: HeatColors.gold),
            title: const Text('Выбрать файл CSV…'),
            onTap: () => Navigator.pop(context, '__pick__')),
        for (final f in files)
          ListTile(
              leading: const Icon(Icons.description, color: HeatColors.dim),
              title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
              onTap: () => Navigator.pop(context, f.path)),
      ]),
    );
    if (chosen == null) return;
    var path = chosen;
    if (chosen == '__pick__') {
      final r = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null || r.files.single.path == null) return;
      path = r.files.single.path!;
    }
    final rows = await LoggerService.parseLog(path);
    if (!mounted) return;
    setState(() {
      if (!merge) {
        _logRows.clear();
        _logNames.clear();
      }
      _logRows.addAll(rows);
      _logNames.add(path.split('/').last);
      _res = null;
    });
    if (rows.isEmpty) _snack('Лог пуст или колонки не распознаны', Colors.orange);
  }

  void _toggleLive() {
    final sub = _liveSub;
    if (sub != null) {
      sub.cancel();
      _liveSub = null;
      setState(() {});
      return;
    }
    final p = ConnectionService.I.poller;
    if (p == null) {
      _snack('Нет опроса — подключись к ЭБУ (НАСТРОЙКИ)', Colors.orange);
      return;
    }
    _liveSub = p.snapshots.listen((s) {
      if (_liveRows.length >= 30000) _liveRows.removeAt(0);
      _liveRows.add(Map<String, double>.from(s.c));
    });
    setState(() {});
  }

  // ── анализ ──────────────────────────────────────────────────────────────
  Future<bool> _ensureRom() async {
    if (RomHolder.I.loaded) return true;
    final go = await showDialog<bool>(
      context: context,
      builder: (c) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: const Text('Прошивка не загружена'),
        content: const Text('Для анализа нужны оси и значения карты из .bin. Загрузить сейчас?',
            style: TextStyle(color: HeatColors.dim)),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c, false), child: const Text('НЕТ')),
          FilledButton(onPressed: () => Navigator.pop(c, true), child: const Text('ЗАГРУЗИТЬ')),
        ],
      ),
    );
    if (go != true) return false;
    final err = await RomHolder.I.loadNewRom();
    if (err != null) {
      _snack(err, Colors.red);
      return false;
    }
    if (mounted) setState(() {});
    return RomHolder.I.loaded;
  }

  Future<void> _analyze(List<Map<String, double>> rows) async {
    if (rows.length < 10) {
      _snack('Мало данных: ${rows.length} строк', Colors.orange);
      return;
    }
    if (!await _ensureRom()) return;
    final t = _target;
    final def = t.def;
    if (def == null) {
      _snack('Карта для «${t.title}» не найдена в дефинишене — нажми ВЫБРАТЬ', Colors.orange);
      return;
    }
    setState(() => _busy = true);
    try {
      final base = RomHolder.I.rom.readTable(def);
      await MapStorageService.applyTo(base); // поверх сохранённых правок, как _apply() в V6
      final res = TuningAnalyzer.analyze(rows: rows, table: base, target: t, pattern: _pattern);
      final upd = TuningAnalyzer.apply(base, res.changes);
      if (!mounted) return;
      setState(() {
        _res = res;
        _orig = base;
        _upd = upd;
      });
    } finally {
      if (mounted) setState(() => _busy = false);
    }
  }

  void _openMap() {
    final o = _orig, u = _upd, r = _res;
    if (o == null || u == null || r == null) return;
    Navigator.push(
        context,
        MaterialPageRoute<void>(
            builder: (_) =>
                RomTableEditScreen(def: o.def, proposal: u, proposalLabel: 'анализатор · ${r.patternName}')));
  }

  Future<void> _queue() async {
    final o = _orig, u = _upd, r = _res;
    if (o == null || u == null || r == null || r.changes.isEmpty) return;
    await MapStorageService.saveMap(u);
    PendingEdits.I.addOrUpdate(o, u, source: 'анализатор · ${r.patternName}');
    if (!mounted) return;
    setState(() {});
    _snack('В очереди на запись: ${PendingEdits.I.count} карт(ы) → вкладка ЗАПИСЬ ROM', Colors.green);
  }

  Future<void> _pickTable() async {
    final ctl = TextEditingController();
    final def = await showDialog<RomTableDef>(
      context: context,
      builder: (c) => StatefulBuilder(builder: (c2, setD) {
        final q = ctl.text.trim().toLowerCase();
        final list = SubaruRom.tables
            .where((d) =>
                d.editable &&
                (q.isEmpty || d.name.toLowerCase().contains(q) || d.category.toLowerCase().contains(q)))
            .take(200)
            .toList();
        return AlertDialog(
          backgroundColor: HeatColors.panel,
          title: const Text('Карта ROM'),
          content: SizedBox(
            width: double.maxFinite,
            height: 420,
            child: Column(children: [
              TextField(
                  controller: ctl,
                  decoration: const InputDecoration(hintText: 'Поиск по имени/категории', prefixIcon: Icon(Icons.search)),
                  onChanged: (_) => setD(() {})),
              Expanded(
                child: ListView.builder(
                  itemCount: list.length,
                  itemBuilder: (_, i) => ListTile(
                      dense: true,
                      title: Text(list[i].name, style: const TextStyle(fontSize: 13)),
                      subtitle: Text('${list[i].category} · ${list[i].rows}x${list[i].cols} · ${list[i].units}',
                          style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                      onTap: () => Navigator.pop(c, list[i])),
                ),
              ),
            ]),
          ),
        );
      }),
    );
    if (def == null) return;
    setState(() {
      _targets[_ti] = _targets[_ti].copyWith(def: def);
      _res = null;
    });
  }

  // ── экспорт ─────────────────────────────────────────────────────────────
  Future<void> _export(String kind) async {
    final u = _upd, r = _res;
    if (u == null || r == null) return;
    final safe = u.def.name.replaceAll(RegExp(r'[^A-Za-z0-9]+'), '_');
    final stamp = DateTime.now().millisecondsSinceEpoch;
    List<int> bytes;
    String name;
    if (kind == 'json') {
      name = 'V8_${safe}_$stamp.json';
      bytes = utf8.encode(const JsonEncoder.withIndent('  ').convert({
        'map': u.def.name,
        'address': u.def.addrHex,
        'units': u.def.units,
        'pattern': r.patternName,
        'x': u.xValues,
        'y': u.yValues,
        'z': u.z.map((row) => row.map((v) => v.isNaN ? null : v).toList()).toList(),
        'changes': r.changes
            .map((c) => {
                  'row': c.row, 'col': c.col, 'x': c.x, 'y': c.y,
                  'from': c.current, 'to': c.suggested, 'conf': c.confidence, 'n': c.samples, 'why': c.reason,
                })
            .toList(),
      }));
    } else if (kind == 'hex') {
      name = 'V8_${safe}_$stamp.hex.txt';
      final rom = RomHolder.I.rom.rom;
      if (rom == null) {
        _snack('ROM не загружен', Colors.orange);
        return;
      }
      final mod = RomWriter.buildMod(rom, [u]);
      final sz = u.def.data.sizeOf;
      final len = u.def.rows * u.def.cols * sz;
      final sb = StringBuffer('; ${u.def.name} @ ${u.def.addrHex} (${u.def.data.storage}/${u.def.data.endian})\n');
      for (var i = 0; i < len; i += 16) {
        sb.write('0x${(u.def.address + i).toRadixString(16).toUpperCase().padLeft(6, '0')}: ');
        for (var k = i; k < i + 16 && k < len; k++) {
          sb.write(mod[u.def.address + k].toRadixString(16).toUpperCase().padLeft(2, '0'));
          sb.write(' ');
        }
        sb.write('\n');
      }
      bytes = utf8.encode(sb.toString());
    } else {
      name = 'V8_${safe}_$stamp.csv';
      final sb = StringBuffer();
      sb.write(';${u.xValues.map((v) => v.toStringAsFixed(2)).join(';')}\n');
      for (var i = 0; i < u.z.length; i++) {
        final y = i < u.yValues.length ? u.yValues[i].toStringAsFixed(2) : '$i';
        sb.write('$y;${u.z[i].map((v) => v.isNaN ? '' : v.toStringAsFixed(3)).join(';')}\n');
      }
      bytes = utf8.encode(sb.toString());
    }
    final res = await V8Saver.saveBytes(bytes, name);
    _snack(res.ok ? 'Сохранено (${res.where}): ${res.path}' : (res.error ?? 'Ошибка сохранения'),
        res.ok ? Colors.green : Colors.red);
  }

  // ── UI ──────────────────────────────────────────────────────────────────
  @override
  Widget build(BuildContext context) {
    return Column(children: [
      Container(
        color: HeatColors.panel,
        child: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.wifi), text: 'ОНЛАЙН'),
          Tab(icon: Icon(Icons.folder), text: 'ИЗ ЛОГА'),
          Tab(icon: Icon(Icons.grid_on), text: 'ТЕПЛОКАРТА'),
        ]),
      ),
      Expanded(
        child: TabBarView(controller: _tab, children: [
          _onlineTab(),
          _logTab(),
          HeatmapAnalyzerScreen(),
        ]),
      ),
    ]);
  }

  Widget _onlineTab() {
    final live = _liveSub != null;
    return ListView(padding: const EdgeInsets.all(10), children: [
      _card(children: [
        Row(children: [
          Icon(live ? Icons.fiber_manual_record : Icons.pause_circle_outline,
              color: live ? Colors.redAccent : HeatColors.dim, size: 18),
          const SizedBox(width: 8),
          Expanded(
              child: Text('Сбор точек: ${_liveRows.length}${live ? ' • идёт' : ''}',
                  style: const TextStyle(fontWeight: FontWeight.bold))),
          FilledButton.icon(
              onPressed: _toggleLive,
              icon: Icon(live ? Icons.stop : Icons.play_arrow, size: 18),
              label: Text(live ? 'СТОП' : 'СБОР'),
              style: FilledButton.styleFrom(backgroundColor: live ? Colors.red.shade800 : const Color(0xFF43A047))),
          const SizedBox(width: 6),
          OutlinedButton(
              onPressed: _liveRows.isEmpty
                  ? null
                  : () => setState(() {
                        _liveRows.clear();
                        _res = null;
                      }),
              child: const Text('СБРОС')),
        ]),
        const SizedBox(height: 4),
        const Text(
            'Каналы для анализа добавляются в опрос автоматически (TuningChannels.ensure). Прогрев → круиз → 3-я передача в пол.',
            style: TextStyle(fontSize: 10, color: HeatColors.dim)),
      ]),
      ..._panel(_liveRows),
    ]);
  }

  Widget _logTab() {
    return ListView(padding: const EdgeInsets.all(10), children: [
      _card(children: [
        Row(children: [
          Expanded(
            child: OutlinedButton.icon(
                onPressed: () => _pickLog(),
                icon: const Icon(Icons.folder),
                label: const Text('CSV'),
                style: OutlinedButton.styleFrom(minimumSize: const Size.fromHeight(44), shape: const StadiumBorder())),
          ),
          const SizedBox(width: 10),
          Expanded(
            child: FilledButton.icon(
                onPressed: _logRows.isEmpty ? null : () => _pickLog(merge: true),
                icon: const Icon(Icons.merge_type),
                label: const Text('Объединить'),
                style: FilledButton.styleFrom(
                    backgroundColor: const Color(0xFF8E24AA),
                    minimumSize: const Size.fromHeight(44),
                    shape: const StadiumBorder())),
          ),
        ]),
        const SizedBox(height: 6),
        if (_logNames.isNotEmpty)
          Center(
              child: Text(_logNames.join(' + '),
                  style: const TextStyle(fontSize: 12, color: HeatColors.dim), textAlign: TextAlign.center)),
        Center(
            child: Text('Записей: ${_logRows.length}',
                style: TextStyle(
                    fontSize: 15, fontWeight: FontWeight.bold, color: _logRows.isEmpty ? HeatColors.dim : Colors.green))),
      ]),
      ..._panel(_logRows),
    ]);
  }

  List<Widget> _panel(List<Map<String, double>> rows) {
    final t = _targets[_ti];
    final def = t.def;
    final stats = LogStats.of(rows);
    final rom = RomHolder.I;
    final r = _res;
    return [
      _card(children: [
        Row(children: [
          Icon(Icons.memory, color: rom.loaded ? Colors.green : Colors.orange, size: 18),
          const SizedBox(width: 8),
          Expanded(
              child: Text(rom.loaded ? 'ROM: ${rom.fileName}' : 'ROM не загружен',
                  style: const TextStyle(fontSize: 12), overflow: TextOverflow.ellipsis)),
          TextButton(
              onPressed: () async {
                final err = await RomHolder.I.loadNewRom();
                if (err != null) _snack(err, Colors.red);
                if (mounted) setState(() => _res = null);
              },
              child: Text(rom.loaded ? 'ДРУГОЙ' : 'ЗАГРУЗИТЬ')),
        ]),
        if (rows.isNotEmpty)
          Wrap(spacing: 6, runSpacing: 4, children: [
            _chip('холостой ${stats.idle}', HeatColors.dim),
            _chip('круиз ${stats.cruise}', Colors.cyan),
            _chip('буст ${stats.boost}', Colors.orange),
            _chip('WOT ${stats.wot}', Colors.redAccent),
            for (final m in stats.missing) _chip('нет ${TuningChannels.labels[m] ?? m}', Colors.red),
          ]),
      ]),
      _card(children: [
        Row(children: [
          const Text('Карта: ', style: TextStyle(color: HeatColors.dim)),
          Expanded(
            child: DropdownButton<int>(
              value: _ti,
              isExpanded: true,
              dropdownColor: HeatColors.panel,
              underline: const SizedBox(),
              items: [
                for (var i = 0; i < _targets.length; i++)
                  DropdownMenuItem(value: i, child: Text(_targets[i].title, style: const TextStyle(fontSize: 15))),
              ],
              onChanged: (v) => setState(() {
                _ti = v ?? 0;
                _res = null;
              }),
            ),
          ),
        ]),
        Row(children: [
          Expanded(
              child: Text(
                  def == null
                      ? 'Таблица: не найдена в дефинишене'
                      : 'Таблица: ${def.name} · ${def.rows}x${def.cols} · ${def.units}',
                  style: TextStyle(fontSize: 11, color: def == null ? Colors.orange : HeatColors.dim))),
          TextButton(onPressed: _pickTable, child: const Text('ВЫБРАТЬ')),
        ]),
        if (def != null && def.rows > 1)
          Wrap(spacing: 6, children: [
            const Padding(
                padding: EdgeInsets.only(top: 8),
                child: Text('Ось Y ←', style: TextStyle(fontSize: 11, color: HeatColors.dim))),
            for (final k in TuningAnalyzer.yCanonChoices)
              ChoiceChip(
                  label: Text(k == 'auto' ? 'авто' : (TuningChannels.labels[k] ?? k),
                      style: const TextStyle(fontSize: 11)),
                  selected: _yCanon == k,
                  onSelected: (_) => setState(() {
                        _yCanon = k;
                        _res = null;
                      })),
          ]),
        const SizedBox(height: 6),
        const Text('Паттерн:', style: TextStyle(color: Colors.orange, fontWeight: FontWeight.bold)),
        const SizedBox(height: 4),
        Wrap(spacing: 8, runSpacing: 8, children: [for (final p in TuningPattern.all) _patternChip(p)]),
      ]),
      SizedBox(
        height: 52,
        child: ElevatedButton.icon(
          onPressed: (_busy || rows.isEmpty) ? null : () => _analyze(rows),
          icon: _busy
              ? const SizedBox(width: 20, height: 20, child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : const Icon(Icons.analytics),
          label: const Text('АНАЛИЗИРОВАТЬ', style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
          style: ElevatedButton.styleFrom(
              backgroundColor: const Color(0xFF43A047), foregroundColor: Colors.white, shape: const StadiumBorder()),
        ),
      ),
      const SizedBox(height: 8),
      if (r != null) _resultCard(r),
    ];
  }

  Widget _patternChip(TuningPattern p) {
    final sel = _pattern == p;
    return InkWell(
      onTap: () => setState(() {
        _pattern = p;
        _res = null;
      }),
      borderRadius: BorderRadius.circular(8),
      child: Container(
        padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
        decoration: BoxDecoration(
            color: sel ? const Color(0xFF3E2A10) : const Color(0xFF14264F),
            borderRadius: BorderRadius.circular(8),
            border: Border.all(color: sel ? Colors.orange : HeatColors.grid, width: sel ? 1.5 : 1)),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, mainAxisSize: MainAxisSize.min, children: [
          Text(p.name,
              style: TextStyle(fontSize: 14, fontWeight: FontWeight.bold, color: sel ? Colors.orange : HeatColors.text)),
          Text(p.desc, style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
        ]),
      ),
    );
  }

  Widget _resultCard(TuningResult r) {
    final romLoaded = RomHolder.I.loaded;
    final o = _orig;
    final queued = o != null && PendingEdits.I.has(o.def.addrHex);
    return _card(children: [
      Row(children: [
        const Icon(Icons.assessment, color: Colors.green, size: 20),
        const SizedBox(width: 6),
        Expanded(child: Text(r.mapName, style: const TextStyle(fontSize: 14, fontWeight: FontWeight.bold))),
        Text('${r.changes.length} правок', style: const TextStyle(color: Colors.orange, fontSize: 13)),
      ]),
      Text('Паттерн: ${r.patternName}', style: const TextStyle(color: HeatColors.dim, fontSize: 11)),
      Text(r.summary, style: const TextStyle(color: HeatColors.text, fontSize: 11)),
      for (final n in r.notes) Text('· $n', style: const TextStyle(color: Colors.amber, fontSize: 10)),
      const SizedBox(height: 6),
      Container(
        padding: const EdgeInsets.all(6),
        decoration: BoxDecoration(
            color: Colors.amber.withAlpha(28),
            borderRadius: BorderRadius.circular(4),
            border: Border.all(color: Colors.amber.withAlpha(100))),
        child: const Row(children: [
          Icon(Icons.lightbulb_outline, color: Colors.amber, size: 16),
          SizedBox(width: 6),
          Expanded(
              child: Text('Важно: после прошивки запиши НОВЫЙ лог и повтори анализ — правки итеративны.',
                  style: TextStyle(color: Colors.amber, fontSize: 10))),
        ]),
      ),
      const SizedBox(height: 6),
      SizedBox(
        width: double.infinity,
        child: ElevatedButton.icon(
            onPressed: o == null ? null : _openMap,
            icon: const Icon(Icons.grid_on, size: 20),
            label: const Text('ОТКРЫТЬ КАРТУ', style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
                backgroundColor: Colors.cyan.shade700,
                foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42),
                shape: const StadiumBorder())),
      ),
      if (r.changes.isNotEmpty) ...[
        const SizedBox(height: 4),
        SizedBox(
          width: double.infinity,
          child: ElevatedButton.icon(
              onPressed: (!romLoaded || _busy) ? null : _queue,
              icon: Icon(queued ? Icons.playlist_add_check : Icons.playlist_add, size: 20),
              label: Text('В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM (${PendingEdits.I.count})',
                  style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.red.shade800,
                  foregroundColor: Colors.white,
                  minimumSize: const Size.fromHeight(46),
                  shape: const StadiumBorder())),
        ),
        if (queued)
          const Padding(
              padding: EdgeInsets.only(top: 2),
              child: Text('Эта карта уже в очереди — вкладка ЗАПИСЬ ROM',
                  style: TextStyle(color: Colors.green, fontSize: 10), textAlign: TextAlign.center)),
      ],
      const SizedBox(height: 6),
      SizedBox(
        height: 260,
        child: r.changes.isEmpty
            ? const Center(
                child: Text('Правок нет — ячейки чистые или мало данных',
                    style: TextStyle(color: Colors.green, fontSize: 13)))
            : ListView.builder(
                itemCount: r.changes.length,
                itemBuilder: (_, i) {
                  final c = r.changes[i];
                  final dc = c.delta > 0 ? Colors.green : Colors.orange;
                  return Card(
                    color: const Color(0xFF0F3460),
                    margin: const EdgeInsets.symmetric(vertical: 2),
                    child: ListTile(
                      dense: true,
                      title: Text(
                          '${_target.xCanon.toUpperCase()} ${c.x.toStringAsFixed(0)} | ${c.y.toStringAsFixed(c.y.abs() < 10 ? 2 : 0)}',
                          style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                      subtitle: Text(
                          '${c.current.toStringAsFixed(2)} → ${c.suggested.toStringAsFixed(2)} (${c.reason}) · n=${c.samples}',
                          style: TextStyle(color: dc, fontSize: 10)),
                      trailing: Text('${(c.confidence * 100).toInt()}%',
                          style: TextStyle(
                              color: c.confidence > 0.8 ? Colors.green : Colors.orange,
                              fontSize: 11,
                              fontWeight: FontWeight.bold)),
                    ),
                  );
                },
              ),
      ),
      const SizedBox(height: 6),
      Wrap(spacing: 8, runSpacing: 6, children: [
        _exportBtn('WinOLS', Colors.blue, 'csv'),
        _exportBtn('JSON', Colors.green, 'json'),
        _exportBtn('HEX', Colors.orange, 'hex'),
      ]),
    ]);
  }

  Widget _exportBtn(String label, Color c, String kind) => FilledButton.icon(
      onPressed: _upd == null ? null : () => _export(kind),
      icon: const Icon(Icons.download, size: 16),
      label: Text(label),
      style: FilledButton.styleFrom(backgroundColor: c, shape: const StadiumBorder()));

  Widget _chip(String s, Color c) => Container(
      padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 3),
      decoration: BoxDecoration(
          color: c.withAlpha(30), borderRadius: BorderRadius.circular(10), border: Border.all(color: c.withAlpha(120))),
      child: Text(s, style: TextStyle(fontSize: 10, color: c)));

  Widget _card({required List<Widget> children}) => Card(
      color: HeatColors.panel,
      margin: const EdgeInsets.only(bottom: 8),
      child: Padding(
          padding: const EdgeInsets.all(10),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: children)));
}
'''

NEW_FILES['test/tune_bridge_test.dart'] = r'''// TUNE-V6-BRIDGE — юнит-тесты кодека ROM, КС и правил анализатора (без железа)
import 'package:flutter_test/flutter_test.dart';
import 'package:suba_run_v8/generated/subaru_pids.g.dart';
import 'package:suba_run_v8/models/rom_table.dart';
import 'package:suba_run_v8/models/tuning_types.dart';
import 'package:suba_run_v8/services/rom_service.dart';
import 'package:suba_run_v8/services/rom_writer.dart';
import 'package:suba_run_v8/services/tuning_analyzer.dart';

RomTableDef defOf(String storage, String endian, {bool swap = false}) => RomTableDef(
      name: 'Test $storage $endian',
      category: 'test',
      address: 0x1000,
      rows: 2,
      cols: 3,
      swapxy: swap,
      units: 'u',
      data: RomCol(
          address: 0x1000, count: 6, storage: storage, endian: endian,
          to: (x) => x * 0.5, fr: (x) => x / 0.5),
    );

List<Map<String, double>> rowsOf(int n, Map<String, double> base) =>
    List.generate(n, (_) => Map<String, double>.from(base));

void main() {
  test('RomWriter: round-trip всех storage/endian через RomService.readTable', () {
    final cases = <List<dynamic>>[
      ['uint8', 'big', 12.0], ['int8', 'big', -7.0], ['uint16', 'big', 1234.0], ['uint16', 'little', 1234.0],
      ['int16', 'big', -321.0], ['int16', 'little', -321.0], ['uint32', 'big', 70000.0], ['int32', 'little', -70000.0],
      ['float', 'big', 12.345], ['float', 'little', -0.75],
    ];
    for (final swap in [false, true]) {
      for (final c in cases) {
        final def = defOf(c[0] as String, c[1] as String, swap: swap);
        final rom = List<int>.filled(0x100000, 0);
        final t = RomTable(def: def, xValues: [1, 2, 3], yValues: [10, 20], z: [
          [c[2] as double, 1.0, 2.0],
          [3.0, 4.0, c[2] as double],
        ]);
        expect(RomWriter.applyTable(rom, t), 6);
        final svc = RomService();
        expect(svc.loadBytes(rom, 't.bin'), isNull);
        final back = svc.readTable(def);
        for (var r = 0; r < 2; r++) {
          for (var k = 0; k < 3; k++) {
            expect(back.z[r][k], closeTo(t.z[r][k], 1e-3), reason: '${c[0]}/${c[1]} swap=$swap [$r][$k]');
          }
        }
      }
    }
  });

  test('RomWriter.modName: нумерация модов', () {
    expect(RomWriter.modName('A2TB100B.bin'), 'V8MOD1_A2TB100B.bin');
    expect(RomWriter.modName('V8MOD3_A2TB100B.bin'), 'V8MOD4_A2TB100B.bin');
    expect(RomWriter.modName('V8MOD_A2TB100B.bin'), 'V8MOD2_A2TB100B.bin');
  });

  test('SubaruChecksum: синтетический ROM — verify/fix, формат подтверждается по оригиналу', () {
    const len = 0x100000;
    final rom = List<int>.generate(len, (i) => i < 0x80000 ? (i * 31 + 7) & 0xFF : 0);
    final t = SubaruChecksum.tableOffset(len);
    for (var i = 0; i < 8; i++) {
      final s = i * 0x20000, e = s + 0x20000;
      for (var k = 0; k < 4; k++) {
        rom[t + i * 12 + k] = (s >> (24 - 8 * k)) & 0xFF;
        rom[t + i * 12 + 4 + k] = (e >> (24 - 8 * k)) & 0xFF;
        rom[t + i * 12 + 8 + k] = 0;
      }
    }
    expect(SubaruChecksum.verify(rom).ok, isFalse);
    expect(SubaruChecksum.forceFix(rom).ok, isTrue);
    expect(SubaruChecksum.verify(rom).ok, isTrue);
    final mod = List<int>.of(rom);
    mod[0x1234] ^= 0xFF;
    mod[0x51234] ^= 0x0F;
    expect(SubaruChecksum.verify(mod).ok, isFalse);
    final rep = SubaruChecksum.fix(mod, reference: rom);
    expect(rep.ok, isTrue, reason: rep.message);
    expect(SubaruChecksum.verify(mod).ok, isTrue);
    // оригинал с «неизвестным» форматом — байты не трогаем
    final junk = List<int>.filled(len, 0x11);
    final mod2 = List<int>.of(junk)..[10] = 0x22;
    final rep2 = SubaruChecksum.fix(mod2, reference: junk);
    expect(rep2.supported, isFalse);
    expect(mod2[10], 0x22);
  });

  test('TuningChannels.ensure добавляет обязательные каналы из библиотеки PID', () {
    final out = TuningChannels.ensure(<SubaruPid>[]);
    final canons = out.map((p) => p.canon).toSet();
    final available = SubaruPids.all.map((p) => p.canon).toSet();
    for (final k in TuningChannels.required) {
      if (available.contains(k)) expect(canons.contains(k), isTrue, reason: 'нет канала $k');
    }
    expect(TuningChannels.ensure(out).length, out.length); // идемпотентно
  });

  test('TuningAnalyzer: зажигание — детонация снижает, чистая ячейка добавляет', () {
    final def = RomTableDef(
        name: 'Base Timing Primary', category: 'Timing', address: 0x2000, rows: 2, cols: 2,
        units: 'deg', data: RomCol(address: 0x2000, count: 4, storage: 'uint8', to: (x) => x, fr: (x) => x));
    final table = RomTable(def: def, xValues: [2000, 4000], yValues: [1.0, 2.0], z: [
      [20.0, 25.0],
      [15.0, 18.0],
    ]);
    final rows = <Map<String, double>>[
      ...rowsOf(6, {'rpm': 4000, 'load': 2.0, 'fbkc': -3.5, 'fkl': 0, 'iam': 1.0, 'afr': 11.5, 'boost': 0.8, 'iat': 30, 'tps': 100}),
      ...rowsOf(8, {'rpm': 2000, 'load': 1.0, 'fbkc': 0, 'fkl': 0, 'iam': 1.0, 'afr': 14.7, 'boost': -0.3, 'iat': 30, 'tps': 30}),
    ];
    final target = TuningAnalyzer.detectTargets([def]).first.copyWith(yCanon: 'auto');
    expect(target.def, isNotNull);
    final res = TuningAnalyzer.analyze(rows: rows, table: table, target: target, pattern: TuningPattern.stability);
    expect(res.usedRows, 14);
    expect(res.changes.length, 2);
    final knockCell = res.changes.firstWhere((c) => c.row == 1 && c.col == 1);
    expect(knockCell.delta, closeTo(-3.0, 1e-9));
    expect(knockCell.confidence, greaterThan(0.8));
    final cleanCell = res.changes.firstWhere((c) => c.row == 0 && c.col == 0);
    expect(cleanCell.delta, closeTo(0.5, 1e-9));
    final upd = TuningAnalyzer.apply(table, res.changes);
    expect(upd.z[1][1], closeTo(15.0, 1e-9));
    expect(table.z[1][1], 18.0); // оригинал не тронут
  });

  test('TuningAnalyzer: OL-топливо — бедно под бустом → богаче', () {
    final def = RomTableDef(
        name: 'Primary Open Loop Fueling', category: 'Fuel', address: 0x3000, rows: 1, cols: 2,
        units: 'AFR', data: RomCol(address: 0x3000, count: 2, storage: 'uint8', to: (x) => x, fr: (x) => x));
    final table = RomTable(def: def, xValues: [3000, 5000], yValues: [], z: [
      [11.0, 11.0],
    ]);
    final rows = rowsOf(5, {'rpm': 5000, 'tps': 100, 'afr': 12.6, 'boost': 0.9, 'fbkc': 0, 'fkl': 0, 'iam': 1});
    final target = TuningAnalyzer.detectTargets([def]).firstWhere((t) => t.kind == TuningKind.fuelOl);
    final res = TuningAnalyzer.analyze(rows: rows, table: table, target: target, pattern: TuningPattern.stability);
    expect(res.changes.length, 1);
    expect(res.changes.first.col, 1);
    expect(res.changes.first.suggested, lessThan(11.0));
    expect(res.changes.first.reason, contains('БЕДНО'));
  });
}
'''


# ════════════════════════════════════════════════════════════════════════════
# ПАТЧИ СУЩЕСТВУЮЩИХ ФАЙЛОВ (каждый идемпотентен по маркеру)
# ════════════════════════════════════════════════════════════════════════════
def patch_rom_service(s):
    if MARK in s:
        return s
    s = add_import(s, "import 'rom_writer.dart'; " + MARK)
    pat = re.compile(r"final mod = List<int>\.of\(rom!\);\n.*?\n(\s*)final dir = await getApplicationDocumentsDirectory\(\);", re.S)
    m = pat.search(s)
    require(m, 'saveMod: не найден цикл записи между `final mod = …` и `final dir = …`')
    s = (s[:m.start()]
         + "final mod = List<int>.of(rom!);\n"
         + m.group(1) + "RomWriter.applyTable(mod, table); // единый кодек: float/int32/endian\n"
         + m.group(1) + "final dir = await getApplicationDocumentsDirectory();"
         + s[m.end():])
    return s


def patch_rom_table(s):
    """readRaw не читал uint32/int32 (брал 1 байт вместо 4) — симметрично кодеку RomWriter."""
    if MARK in s:
        return s
    anchor = "      case 'int8':\n        return bd.getInt8(off);"
    require(anchor in s, 'rom_table: не найден case int8 в readRaw')
    return s.replace(anchor,
        "      case 'uint32': " + MARK + "\n"
        "        return endian == 'little' ? bd.getUint32(off, Endian.little) : bd.getUint32(off, Endian.big);\n"
        "      case 'int32':\n"
        "        return endian == 'little' ? bd.getInt32(off, Endian.little) : bd.getInt32(off, Endian.big);\n"
        + anchor, 1)


def patch_dashboard(s):
    if MARK in s:
        return s
    s = add_import(s, "import '../widgets/fuel_card.dart'; " + MARK)
    m = re.search(r"\n(\s*)Expanded\(\s*\n\s*child: StreamBuilder<LiveSnapshot>\(", s)
    require(m, 'dashboard: не найден блок Expanded(StreamBuilder) с сеткой приборов')
    return s[:m.start()] + "\n" + m.group(1) + "const FuelCard(), // расход — под картой буста" + s[m.start():]


def patch_main(s):
    if MARK in s:
        return s
    require("Tab(text: 'ROM DIFF')," in s and "RomDiffScreen()," in s, 'main: нет вкладки ROM DIFF')
    s = add_import(s, "import 'screens/write_rom_screen.dart'; " + MARK)
    s = s.replace("Tab(text: 'ROM DIFF'),", "Tab(text: 'ROM DIFF'),\n    Tab(text: 'ЗАПИСЬ ROM'),", 1)
    s = s.replace("RomDiffScreen(),", "RomDiffScreen(),\n    WriteRomScreen(),", 1)
    return s


def patch_rom_screen(s):
    if MARK in s:
        return s
    require("=> RomTableScreen(def: d)" in s, 'rom_screen: не найден переход RomTableScreen(def: d)')
    s = add_import(s, "import 'rom_table_edit_screen.dart'; " + MARK)
    return s.replace("=> RomTableScreen(def: d)", "=> RomTableEditScreen(def: d)")


def patch_connection(s):
    if MARK in s:
        return s
    m = re.search(r"p\.buildBlocks\(\[\.\.\.st\.selectedPids,", s)
    require(m, 'connection_service: не найден p.buildBlocks([...st.selectedPids, …')
    s = add_import(s, "import 'tuning_analyzer.dart'; " + MARK)
    return s.replace("p.buildBlocks([...st.selectedPids,", "p.buildBlocks([...TuningChannels.ensure(st.selectedPids),", 1)


def patch_logger(s):
    if MARK in s:
        return s
    require("_cols = cols;" in s and "...cols.map((c) => c.id)" in s, 'logger_service: не найдены _cols = cols / заголовок CSV')
    s = add_import(s, "import 'tuning_analyzer.dart'; " + MARK)
    s = s.replace("...cols.map((c) => c.id)", "..._cols.map((c) => c.id)", 1)
    s = s.replace("_cols = cols;", "_cols = TuningChannels.ensure(cols); // те же каналы, что и в поллере", 1)
    return s


PATCHES = {
    'lib/models/rom_table.dart': patch_rom_table,
    'lib/services/rom_service.dart': patch_rom_service,
    'lib/screens/dashboard_screen.dart': patch_dashboard,
    'lib/main.dart': patch_main,
    'lib/screens/rom_screen.dart': patch_rom_screen,
    'lib/services/connection_service.dart': patch_connection,
    'lib/services/logger_service.dart': patch_logger,
}

HEATMAP_STUB = r'''// TUNE-V6-BRIDGE: прежний анализатор не найден — заглушка
import 'package:flutter/material.dart';

class HeatmapAnalyzerScreen extends StatelessWidget {
  const HeatmapAnalyzerScreen({super.key});
  @override
  Widget build(BuildContext context) => const Center(child: Text('Тепловая карта недоступна'));
}
'''


def derive_heatmap(old_src):
    """Старый AnalyzerScreen V8 (HeatGrid) → HeatmapAnalyzerScreen, 3-я вкладка нового анализатора."""
    if 'class AnalyzerScreen' not in old_src:
        return HEATMAP_STUB
    s = re.sub(r'\bAnalyzerScreen\b', 'HeatmapAnalyzerScreen', old_src)
    return '// TUNE-V6-BRIDGE: прежний анализатор V8 (тепловая сетка), встроен третьей вкладкой\n' + s


# ════════════════════════════════════════════════════════════════════════════
# ВЫПОЛНЕНИЕ: preflight → бэкап → запись → тесты + analyze → откат при ошибке
# ════════════════════════════════════════════════════════════════════════════
print('=' * 64)
print('  TUNE-BRIDGE: карты · анализатор · запись ROM как в V6 (Subaru)')
print('=' * 64)
require(ROOT.is_dir(), 'Нет /content/suba_run_v8 — прогони ячейки 0–9')
require(FLUTTER.is_file(), 'Нет Flutter SDK в этой сессии Colab — ячейка 0/10')
os.chdir(ROOT)

warnings = []
outputs = {}
originals = {}
for rel, fn in PATCHES.items():
    p = ROOT / rel
    if not p.is_file():
        warnings.append(f'{rel}: файла нет — патч пропущен')
        continue
    src = p.read_text(encoding='utf-8')
    originals[rel] = src
    try:
        out = fn(src)
        require(fn(out) == out, 'идемпотентность нарушена')
        outputs[rel] = out
    except ValueError as e:
        warnings.append(f'{rel}: {e} — патч пропущен')

# старый анализатор → heatmap_analyzer_screen.dart (только если ещё не сделано)
an_path = ROOT / 'lib/screens/analyzer_screen.dart'
hm_path = ROOT / 'lib/screens/heatmap_analyzer_screen.dart'
if not hm_path.is_file():
    old = an_path.read_text(encoding='utf-8') if an_path.is_file() else ''
    NEW_FILES['lib/screens/heatmap_analyzer_screen.dart'] = HEATMAP_STUB if MARK in old else derive_heatmap(old)

targets = {ROOT / rel: text for rel, text in NEW_FILES.items()}
targets.update({ROOT / rel: text for rel, text in outputs.items()})
changes = []
for path, text in targets.items():
    before = path.read_bytes() if path.exists() else None
    after = text.encode('utf-8')
    if before != after:
        changes.append((path, before, after))

if not changes:
    print('Всё уже применено — изменений нет.')
    for w in warnings:
        print('  ! ' + w)
else:
    backup = ROOT / 'patch_backups' / ('tune_bridge_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
    for path, before, _ in changes:
        if before is not None:
            dst = backup / path.relative_to(ROOT)
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(before)

    def rollback():
        for path, before, _ in changes:
            if before is None:
                path.unlink(missing_ok=True)
            else:
                path.write_bytes(before)

    try:
        for path, _, after in changes:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(after)
        print(f'Записано файлов: {len(changes)} (бэкап: {backup})')
        for w in warnings:
            print('  ! ' + w)

        if not (ROOT / '.dart_tool/package_config.json').is_file():
            print('pub get …')
            subprocess.run([str(FLUTTER), 'pub', 'get'], cwd=ROOT, check=False, timeout=600)

        print('\n[1/2] flutter test test/tune_bridge_test.dart …')
        t = subprocess.run([str(FLUTTER), 'test', '--no-pub', 'test/tune_bridge_test.dart'],
                           cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=900)
        (ROOT / 'tune_bridge_test_output.txt').write_text(t.stdout, encoding='utf-8')
        print('\n'.join(t.stdout.splitlines()[-25:]))
        require(t.returncode == 0, 'тесты не прошли — см. tune_bridge_test_output.txt')

        print('\n[2/2] flutter analyze lib …')
        a = subprocess.run([str(FLUTTER), 'analyze', '--no-pub', '--no-fatal-infos', '--no-fatal-warnings', 'lib'],
                           cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=900)
        (ROOT / 'tune_bridge_analyze_output.txt').write_text(a.stdout, encoding='utf-8')
        errors = [l for l in a.stdout.splitlines() if re.search(r'\berror\b\s*[•-]', l)]
        for l in errors[:40]:
            print('  ' + l.strip())
        require(not errors, f'analyze: {len(errors)} ошибок — см. tune_bridge_analyze_output.txt')
    except (ValueError, OSError, subprocess.TimeoutExpired) as e:
        rollback()
        raise SystemExit('TUNE-BRIDGE НЕ установлен, файлы откачены: ' + str(e)) from e

    print()
    print('=' * 64)
    print('  TUNE-BRIDGE применён и проверен в этой сессии Colab.')
    print('=' * 64)
    print('  ПРИБОРЫ      : под BOOST — карточка РАСХОД (л/ч · л/100 км · duty форсунок)')
    print('  АНАЛИЗАТОР   : ОНЛАЙН / ИЗ ЛОГА (CSV + Объединить) / ТЕПЛОКАРТА(старый)')
    print('                 карта ← дефинишен · паттерн · АНАЛИЗИРОВАТЬ · ОТКРЫТЬ КАРТУ ·')
    print('                 В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM · экспорт WinOLS/JSON/HEX')
    print('  ROM → карта  : вид как в V6: старое зачёркнуто / новое жирным, Сохранить / Сброс / зум')
    print('  ЗАПИСЬ ROM   : новая вкладка — пакетная запись очереди в V8MOD{n}_*.bin + КС Subaru')
    print('  ЛОГГЕР/ПОЛЛЕР: TuningChannels.ensure() — обязательные каналы всегда в опросе и CSV')
    print()
    print('  Далее -> ячейка 10/10 (сборка APK). В ЭБУ по CAN приложение НЕ пишет —')
    print('  прошивка V8MOD*.bin только через EcuFlash + OpenPort 2.0.')


  TUNE-BRIDGE: карты · анализатор · запись ROM как в V6 (Subaru)
Записано файлов: 19 (бэкап: /content/suba_run_v8/patch_backups/tune_bridge_20260911_082224)
  ! lib/services/rom_service.dart: saveMod: не найден цикл записи между `final mod = …` и `final dir = …` — патч пропущен
  ! lib/services/connection_service.dart: connection_service: не найден p.buildBlocks([...st.selectedPids, … — патч пропущен

[1/2] flutter test test/tune_bridge_test.dart …


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



00:00 +0: loading /content/suba_run_v8/test/tune_bridge_test.dart
00:00 +0: RomWriter: round-trip всех storage/endian через RomService.readTable
00:00 +0 -1: RomWriter: round-trip всех storage/endian через RomService.readTable [E]
  Expected: <6>
    Actual: <0>
  
  package:matcher                                     expect
  package:flutter_test/src/widget_tester.dart 473:18  expect
  test/tune_bridge_test.dart 41:9                     main.<fn>
  
00:00 +0 -1: RomWriter.modName: нумерация модов
00:00 +1 -1: SubaruChecksum: синтетический ROM — verify/fix, формат подтверждается по оригиналу
00:00 +2 -1: TuningChannels.ensure добавляет обязательные каналы из библиотеки PID
00:00 +3 -1: TuningAnalyzer: зажигание — детонация снижает, чистая ячейка добавляет
00:00 +4 -1: TuningAnalyzer: OL-топливо — бедно под бустом → богаче
00:00 +5 -1: Some tests failed.

Failing tests:
  /content/suba_run_v8/test/tune_bridge_test.dart: RomWriter: round-trip всех storage/endian через RomService.readTabl

TypeError: object of type 'NoneType' has no len()

In [ ]:
# @title SUBA V8: Анализатор LAB-SUBA (V1+V2 одной ячейкой)
# ЕДИНСТВЕННАЯ нужная ячейка анализатора. Ставит всё сразу:
# V1 (ядро, хранилище, ROM-мост, экран, тесты, crypto) + V2 (CSV без 0 записей,
# набор карт для настройки, вид V6, авто-дефолты).
# Запускать ОДИН раз в ТЕКУЩЕМ рантайме после фиксов CAN/BT/MAP-AXES, ПЕРЕД сборкой 10/10.
# Повторный запуск безопасен: файлы просто перезаписываются тем же содержимым.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import os
import re
import shutil
import subprocess


def find_root():
    """Проект ищется, а не предполагается: переживает смену cwd и отвал рантайма."""
    override = os.environ.get('SUBA_ROOT', '').strip()
    if override:
        p = Path(override)
        if (p / 'lib/main.dart').is_file():
            return p
        print(f'SUBA_ROOT={override} не содержит lib/main.dart — ищу дальше...')
    for c in [Path('/content/suba_run_v8'), Path('suba_run_v8'),
              Path.cwd() / 'suba_run_v8', Path.cwd()]:
        if (c / 'lib/main.dart').is_file():
            return c
    for base in [Path('/content'), Path.cwd()]:
        try:
            hits = list(base.glob('*/lib/main.dart')) + list(base.glob('*/*/lib/main.dart'))
        except OSError:
            hits = []
        if hits:
            return hits[0].parent.parent
    return None


def find_flutter():
    p = Path('/content/flutter/bin/flutter')
    if p.is_file():
        return p
    w = shutil.which('flutter')
    return Path(w) if w else None


ROOT = find_root()
if ROOT is None:
    print('=' * 64)
    print('  Проект suba_run_v8 НЕ найден в этом рантайме Colab.')
    print('=' * 64)
    print('Содержимое /content:')
    try:
        for p in sorted(Path('/content').iterdir()):
            print('  -', p.name, '(папка)' if p.is_dir() else '')
    except OSError as e:
        print('  не могу прочитать /content:', e)
    print()
    print('Что делать:')
    print(' 1) Выполните ячейки 0/10 … 9/10 — они создают /content/suba_run_v8.')
    print(' 2) Затем CAN-V3 / BT-PERM-V2 / MAP-AXES-V2.')
    print(' 3) Только потом эту ячейку.')
    print(' 4) Проект в другом месте? Вставьте перед ячейкой:')
    print('    import os; os.environ["SUBA_ROOT"] = "/content/ваша_папка"')
    print('Файлы НЕ изменены.')
    raise SystemExit('Missing project: suba_run_v8 not found (see steps above).')

FLUTTER = find_flutter()
if FLUTTER is None:
    print('Flutter SDK не найден: выполните ячейку 0/10 (окружение).')
    print('Файлы НЕ изменены.')
    raise SystemExit('Missing Flutter SDK.')

print('ROOT =', ROOT)
print('FLUTTER =', FLUTTER)
print('INSTALLER REV = 2026-09-10-v12-combined (если этой строки нет — копия старая, скопируйте заново)')

CORE = r'''import 'dart:convert';
import 'dart:math' as math;
import 'package:csv/csv.dart';

typedef LabJson = Map<String, dynamic>;

void labRequire(bool ok, String message) {
  if (!ok) throw FormatException(message);
}

String labNormalize(String value) => value.trim().toLowerCase().replaceAll(RegExp(r'[^a-z0-9а-я]'), '');
List<List<double>> labCopy(List<List<double>> data) => data.map((r) => List<double>.of(r)).toList();
List<List<double>> labMatrix(dynamic value, int rows, int cols) {
  labRequire(value is List && value.length == rows, 'Число строк карты не совпадает с Y.');
  return (value as List).map((row) {
    labRequire(row is List && row.length == cols, 'Число столбцов карты не совпадает с X.');
    return (row as List).map((v) {
      labRequire(v is num && v.isFinite, 'Карта содержит нечисловое/бесконечное значение.');
      return (v as num).toDouble();
    }).toList();
  }).toList();
}

class LabAxis {
  final String name, unit;
  final List<double> values;
  LabAxis(this.name, this.unit, List<double> values) : values = List.unmodifiable(values) {
    labRequire(values.isNotEmpty && values.length <= 128 && values.every((v) => v.isFinite), 'Некорректная ось.');
    if (values.length > 1) {
      final sign = (values[1] - values[0]).sign;
      labRequire(sign != 0 && List.generate(values.length - 1, (i) => (values[i + 1] - values[i]).sign == sign).every((v) => v), 'Ось не монотонна или содержит повторы.');
    }
  }
  LabJson toJson() => {'name': name, 'unit': unit, 'values': values};
  factory LabAxis.fromJson(LabJson json) => LabAxis(json['name'] as String, json['unit'] as String? ?? '',
      (json['values'] as List).map((v) => (v as num).toDouble()).toList());
}

class LabMap {
  final String id, name, category, unit;
  final int address;
  final LabAxis x, y;
  final List<List<double>> original;
  List<List<double>> data;
  LabMap({required this.id, required this.name, required this.category, required this.unit,
      required this.address, required this.x, required this.y,
      required List<List<double>> original, required List<List<double>> data})
      : original = List.unmodifiable(original.map((r) => List<double>.unmodifiable(r))), data = labCopy(data) {
    labMatrix(original, y.values.length, x.values.length);
    labMatrix(data, y.values.length, x.values.length);
  }
  int get rows => y.values.length;
  int get cols => x.values.length;
  int get changes => List.generate(rows, (r) => List.generate(cols,
      (c) => data[r][c] != original[r][c] ? 1 : 0).fold(0, (a, b) => a + b)).fold(0, (a, b) => a + b);
  LabJson toJson() => {'id': id, 'name': name, 'category': category, 'unit': unit,
      'address': address, 'x': x.toJson(), 'y': y.toJson(), 'original': original, 'data': data};
  factory LabMap.fromJson(LabJson j) {
    final x = LabAxis.fromJson(Map<String, dynamic>.from(j['x']));
    final y = LabAxis.fromJson(Map<String, dynamic>.from(j['y']));
    return LabMap(id: j['id'], name: j['name'], category: j['category'], unit: j['unit'], address: j['address'], x: x, y: y,
        original: labMatrix(j['original'], y.values.length, x.values.length), data: labMatrix(j['data'], y.values.length, x.values.length));
  }
}

class LabProject {
  final String calId, romHash, definitionHash, sourceName, origin;
  final List<LabMap> maps;
  LabProject({required this.calId, required this.romHash, required this.definitionHash,
      required this.sourceName, required this.origin, required this.maps}) {
    labRequire(maps.isNotEmpty && maps.length <= 500, 'Пакет должен содержать 1..500 карт.');
    labRequire(maps.map((m) => m.id).toSet().length == maps.length, 'Повтор ID карты.');
    labRequire(maps.fold<int>(0, (sum, m) => sum + m.rows * m.cols) <= 250000, 'Лимит 250000 ячеек в пакете.');
  }
  LabJson toJson() => {'schema': 'suba-lab/v1', 'calId': calId, 'romHash': romHash, 'definitionHash': definitionHash,
      'sourceName': sourceName, 'origin': origin, 'maps': maps.map((m) => m.toJson()).toList()};
  factory LabProject.fromJson(LabJson j) {
    labRequire(j['schema'] == 'suba-lab/v1', 'Неверный формат пакета карт.');
    return LabProject(calId: j['calId'], romHash: j['romHash'], definitionHash: j['definitionHash'], sourceName: j['sourceName'],
        origin: j['origin'] ?? 'map-json', maps: (j['maps'] as List).map((m) => LabMap.fromJson(Map<String, dynamic>.from(m))).toList());
  }
}

class LabSample {
  final int timeMs;
  final Map<String, double?> values;
  final String source;
  LabSample(this.timeMs, this.values, this.source);
}

class LabLog {
  final String name;
  final List<LabSample> samples;
  final List<String> columns;
  final int rejected, duplicates;
  final String? timeColumn;
  final String delimiter;
  final List<String> warnings;
  final List<String> rejectedSample;
  LabLog(this.name, this.samples, this.columns, this.rejected, this.duplicates,
      {this.timeColumn, this.delimiter = ',', this.warnings = const [], this.rejectedSample = const []});
}

// Набор карт для правки и настройки (как вкладки в V6).
class LabTuningDef {
  final String key, titleRu, shortRu, category, description;
  final List<String> match;
  const LabTuningDef(this.key, this.titleRu, this.shortRu, this.match, this.category, this.description);
}

const labTuningMaps = [
  LabTuningDef('base-timing', 'Базовое зажигание', 'Зажигание',
      ['base timing', 'base ignition', 'timing primary', 'basetiming'], 'Зажигание', 'Основная карта УОЗ.'),
  LabTuningDef('kca-max', 'Коррекция детонации MAX (KCA)', 'KCA MAX',
      ['knock correction advance max', 'kca max', 'knock advance max', 'kc max'], 'Зажигание', 'Максимальная прибавка УОЗ.'),
  LabTuningDef('ol-fueling', 'Открытый цикл — топливо', 'Топливо OL',
      ['primary open loop', 'open loop fuel', 'fueling primary', 'ol fuel'], 'Топливо', 'Целевая смесь под нагрузкой.'),
  LabTuningDef('target-boost', 'Целевой наддув', 'Буст цель',
      ['target boost', 'boost target', 'desired boost'], 'Наддув', 'Цель по наддуву.'),
  LabTuningDef('wg-initial', 'Wastegate начальный duty', 'WG нач.',
      ['initial wastegate', 'wastegate initial', 'wgdc initial'], 'Наддув', 'Стартовый duty вестгейта.'),
  LabTuningDef('wg-max', 'Wastegate максимальный duty', 'WG макс.',
      ['max wastegate', 'wastegate max', 'wgdc max'], 'Наддув', 'Ограничение duty вестгейта.'),
  LabTuningDef('avcs-intake', 'AVCS впуск — цель', 'AVCS вп.',
      ['intake avcs', 'avcs intake', 'avcs target intake'], 'AVCS', 'Целевой угол впускного AVCS.'),
  LabTuningDef('avcs-exhaust', 'AVCS выпуск — цель', 'AVCS вып.',
      ['exhaust avcs', 'avcs exhaust', 'avcs target exhaust'], 'AVCS', 'Целевой угол выпускного AVCS.'),
  LabTuningDef('maf-scaling', 'Шкалирование MAF', 'MAF',
      ['maf scaling', 'maf sensor scaling', 'airflow scaling'], 'Впуск', 'Кривая расходомера.'),
  LabTuningDef('rev-limit', 'Отсечка / Rev limit', 'Отсечка',
      ['rev limit', 'revlimit', 'fuel cut rpm', 'rpm limit'], 'Защита', 'Обороты отсечки.'),
];

String _normName(String v) => v.toLowerCase().replaceAll(RegExp(r'[^a-z0-9а-я]+'), ' ').trim();

LabTuningDef? labMatchTuning(String name, String category) {
  final hay = '${_normName(name)} ${_normName(category)}';
  LabTuningDef? best;
  var bestLen = 0;
  for (final def in labTuningMaps) {
    for (final raw in def.match) {
      final needle = _normName(raw);
      if (needle.isNotEmpty && hay.contains(needle) && needle.length > bestLen) {
        best = def;
        bestLen = needle.length;
      }
    }
  }
  return best;
}

int labTuningOrder(String name, String category) {
  final def = labMatchTuning(name, category);
  if (def == null) return 999;
  return labTuningMaps.indexWhere((d) => d.key == def.key);
}

double? labNumber(String text, [String delimiter = ',']) {
  final t = text.trim();
  if (t.isEmpty) return null;
  const empties = ['—', '-', '–', 'n/a', 'na', 'nan', 'null', 'none', '--'];
  if (empties.contains(t.toLowerCase())) return null;
  final normalized = delimiter == ',' ? t.replaceAll(RegExp(r'\s+'), '') : t.replaceAll(RegExp(r'\s+'), '').replaceAll(',', '.');
  final value = double.tryParse(normalized);
  return value != null && value.isFinite ? value : null;
}

String _detectDelimiter(List<String> sampleLines) {
  var best = ',';
  var bestScore = -1;
  for (final sep in [';', '\t', ',', '|']) {
    final counts = sampleLines.take(5).map((l) => l.split(sep).length).toList();
    if (counts.isEmpty || counts.first < 2) continue;
    final consistent = counts.every((c) => c == counts.first);
    final score = counts.first * 10 + (consistent ? 5 : 0);
    if (score > bestScore) { bestScore = score; best = sep; }
  }
  return best;
}

int? _parseTime(String raw, String kind, String delimiter) {
  final t = raw.trim();
  if (t.isEmpty) return null;
  if (kind == 'ms') {
    final n = labNumber(t, delimiter);
    if (n == null) return null;
    if (n < 100000 && n >= 0) return (n * 1000).round();
    return n.round();
  }
  if (kind == 's') {
    final n = labNumber(t, delimiter);
    if (n != null && n >= 0) return (n * 1000).round();
  }
  final hms = RegExp(r'^(?:(\d+):)?([0-5]?\d):([0-5]?\d)(?:[.,](\d{1,3}))?$').firstMatch(t);
  if (hms != null) {
    final h = int.tryParse(hms.group(1) ?? '0') ?? 0;
    final m = int.tryParse(hms.group(2)!) ?? 0;
    final s = int.tryParse(hms.group(3)!) ?? 0;
    final ms = int.tryParse((hms.group(4) ?? '0').padRight(3, '0')) ?? 0;
    return ((h * 3600 + m * 60 + s) * 1000) + ms;
  }
  final iso = t.contains('T') ? t : t.replaceFirst(' ', 'T');
  final parsed = DateTime.tryParse(iso)?.millisecondsSinceEpoch;
  if (parsed != null) return parsed;
  final eu = RegExp(r'^(\d{1,2})[./-](\d{1,2})[./-](\d{2,4})[ T](\d{1,2}):(\d{2})(?::(\d{2})(?:[.,](\d{1,3}))?)?$').firstMatch(t);
  if (eu != null) {
    var year = int.parse(eu.group(3)!);
    if (year < 100) year += 2000;
    final d = DateTime(year, int.parse(eu.group(2)!), int.parse(eu.group(1)!),
        int.parse(eu.group(4)!), int.parse(eu.group(5)!), int.parse(eu.group(6) ?? '0'),
        int.parse((eu.group(7) ?? '0').padRight(3, '0')));
    return d.millisecondsSinceEpoch;
  }
  final fallback = labNumber(t, delimiter);
  if (fallback != null && fallback >= 0) return (fallback < 100000 ? fallback * 1000 : fallback).round();
  return null;
}

LabLog labParseCsv(String content, String name) {
  final text = content.replaceFirst(RegExp(r'^\uFEFF'), '');
  final nonEmpty = text.split(RegExp(r'[\r\n]')).where((l) => l.trim().isNotEmpty).take(5).toList();
  final delimiter = _detectDelimiter(nonEmpty.isEmpty ? [text.split(RegExp(r'[\r\n]')).first] : nonEmpty);
  // Нормализуем переводы строк сами — без csvSettingsDetector (его нет в csv 6.0.0).
  final normalized = text.replaceAll('\r\n', '\n').replaceAll('\r', '\n');
  final records = CsvToListConverter(fieldDelimiter: delimiter, eol: '\n', shouldParseNumbers: false,
      allowInvalid: false).convert(normalized);
  labRequire(records.length >= 2, 'CSV не содержит данных.');
  var columns = records.first.map((v) => v.toString().trim()).toList();
  while (columns.isNotEmpty && columns.last.isEmpty) columns.removeLast();
  labRequire(columns.isNotEmpty, 'Пустая строка заголовков.');
  final warnings = <String>[];
  final normed = columns.map(labNormalize).toList();
  if (normed.toSet().length != normed.length) {
    final seen = <String, int>{};
    columns = columns.map((c) {
      final n = labNormalize(c).isEmpty ? 'col' : labNormalize(c);
      final k = seen[n] ?? 0;
      seen[n] = k + 1;
      return k == 0 ? c : '${c}_${k + 1}';
    }).toList();
    warnings.add('Повторяющиеся заголовки переименованы.');
  }
  const timeKinds = {
    'tsms': 'ms', 'timestampms': 'ms', 'timems': 'ms', 'ms': 'ms', 'millis': 'ms',
    'times': 's', 'timesec': 's', 'time': 's', 't': 's', 'sec': 's', 'seconds': 's', 'uptime': 's',
    'timestamp': 'iso', 'datetime': 'iso', 'date': 'iso', 'clock': 'iso',
  };
  var ti = columns.indexWhere((v) => timeKinds.containsKey(labNormalize(v)));
  String? timeColumn = ti >= 0 ? columns[ti] : null;
  if (ti < 0) warnings.add('Столбец времени не найден — используется порядок строк (шаг 100 мс).');
  final samples = <LabSample>[];
  var rejected = 0;
  final rejectedSample = <String>[];
  var lastT = 0;
  var rowNo = 1;
  for (final row in records.skip(1)) {
    rowNo++;
    if (row.every((v) => v.toString().trim().isEmpty)) continue;
    final rec = List<String>.from(row.map((v) => v.toString()));
    while (rec.length < columns.length) rec.add('');
    if (rec.length > columns.length) rec.length = columns.length;
    int? time;
    if (ti >= 0) {
      time = _parseTime(rec[ti], timeKinds[labNormalize(columns[ti])]!, delimiter);
      if (time == null || time < 0) {
        rejected++;
        if (rejectedSample.length < 3) rejectedSample.add('строка $rowNo: время «${rec[ti]}» не распознано');
        continue;
      }
    } else {
      time = lastT + 100;
    }
    lastT = time;
    final values = <String, double?>{};
    for (var i = 0; i < columns.length; i++) {
      if (i != ti) values[columns[i]] = labNumber(rec[i], delimiter);
    }
    samples.add(LabSample(time, values, name));
  }
  if (samples.isEmpty) {
    throw FormatException('Все строки отклонены ($rejected). ${rejectedSample.isEmpty ? '' : rejectedSample.first} Столбцы: ${columns.join(' | ')}');
  }
  final merged = labMerge(samples);
  return LabLog(name, merged.samples, columns.where((c) => c != timeColumn).toList(), rejected, merged.duplicates,
      timeColumn: timeColumn, delimiter: delimiter, warnings: warnings, rejectedSample: rejectedSample);
}

({List<LabSample> samples, int duplicates}) labMerge(List<LabSample> input) {
  final sorted = List<LabSample>.of(input)..sort((a, b) => a.timeMs.compareTo(b.timeMs));
  final keys = <String>{}; final output = <LabSample>[]; var duplicates = 0;
  for (final sample in sorted) {
    final names = sample.values.keys.toList()..sort();
    final key = jsonEncode([sample.timeMs, names.map((n) => [n, sample.values[n]]).toList()]);
    if (keys.add(key)) output.add(sample); else duplicates++;
  }
  return (samples: output, duplicates: duplicates);
}

class LabOptions {
  String x = '', y = '', metric = '', reference = '', focus = 'stability', tps = '', speed = '';
  int minSamples = 5;
  double threshold = 1;
  LabJson toJson() => {'x': x, 'y': y, 'metric': metric, 'reference': reference, 'focus': focus,
      'tps': tps, 'speed': speed, 'minSamples': minSamples, 'threshold': threshold};
}

class LabCell {
  final int row, col;
  int count = 0;
  double _mean = 0, minimum = double.infinity, maximum = -double.infinity;
  LabCell(this.row, this.col);
  void add(double value) { count++; _mean = _mean * ((count - 1) / count) + value / count; minimum = math.min(minimum, value); maximum = math.max(maximum, value); }
  double get mean => count == 0 ? double.nan : _mean;
  LabJson toJson() => {'r': row, 'c': col, 'n': count, 'mean': mean, 'min': minimum, 'max': maximum};
}

class LabResult {
  final Map<String, LabCell> cells = {};
  int total = 0, accepted = 0, missing = 0, outside = 0, filtered = 0;
  LabJson toJson() => {'total': total, 'accepted': accepted, 'missing': missing, 'outside': outside,
      'filtered': filtered, 'cells': cells.values.map((c) => c.toJson()).toList(), 'automaticCalibrationChanges': false};
}

class SubaruLogAnalyzer {
  static int nearest(List<double> axis, double value) {
    if (value < math.min(axis.first, axis.last) || value > math.max(axis.first, axis.last)) return -1;
    var best = 0;
    for (var i = 1; i < axis.length; i++) if ((axis[i] - value).abs() < (axis[best] - value).abs()) best = i;
    return best;
  }
  LabResult analyze(List<LabSample> samples, LabMap map, LabOptions options) {
    labRequire(options.x.isNotEmpty && options.y.isNotEmpty && options.metric.isNotEmpty, 'Выберите X, Y и метрику.');
    final result = LabResult()..total = samples.length;
    for (final sample in samples) {
      if (options.focus != 'stability') {
        final tps = sample.values[options.tps], speed = sample.values[options.speed];
        if (tps == null || (options.focus == 'economy' && speed == null)) { result.missing++; continue; }
        if (options.focus == 'power' ? tps < 80 : tps < 2 || tps > 40 || speed! < 20) { result.filtered++; continue; }
      }
      final x = sample.values[options.x], y = sample.values[options.y], value = sample.values[options.metric];
      final reference = options.reference.isEmpty ? 0.0 : sample.values[options.reference];
      if (x == null || y == null || value == null || reference == null || ![x, y, value, reference].every((v) => v.isFinite)) {
        result.missing++; continue;
      }
      final row = nearest(map.y.values, y), col = nearest(map.x.values, x);
      if (row < 0 || col < 0) { result.outside++; continue; }
      final delta = value - reference;
      if (!delta.isFinite) { result.missing++; continue; }
      result.cells.putIfAbsent('$row:$col', () => LabCell(row, col)).add(delta);
      result.accepted++;
    }
    return result;
  }
}

String labMapCsv(LabMap map) => '\uFEFF' + const ListToCsvConverter(fieldDelimiter: ';').convert([
  ['${map.y.name} (${map.y.unit}) / ${map.x.name} (${map.x.unit})', ...map.x.values],
  for (var r = 0; r < map.rows; r++) [map.y.values[r], ...map.data[r]],
]);'''
STORE = r'''import 'dart:convert';
import 'dart:io';
import 'package:crypto/crypto.dart';
import 'package:path_provider/path_provider.dart';
import 'subaru_lab_core.dart';

class LabRevision {
  final int version;
  final String at, comment;
  final List<List<double>> data;
  LabRevision(this.version, this.at, this.comment, this.data);
  LabJson toJson() => {'version': version, 'at': at, 'comment': comment, 'data': data};
}

class LabStore {
  final Future<Directory> Function()? directory;
  LabStore({this.directory});
  final Map<String, List<LabRevision>> history = {};
  final Map<String, List<List<double>>> defaults = {};
  Future<void> _queue = Future<void>.value();
  String _identity(LabProject p) => sha256.convert(utf8.encode('${p.calId}|${p.romHash}|${p.definitionHash}')).toString();
  Future<Directory> _dir() async {
    if (directory != null) return directory!();
    final base = await getApplicationDocumentsDirectory();
    return Directory('${base.path}/subaru_lab_v1').create(recursive: true);
  }
  Future<void> _atomic(File file, String text) async {
    final temp = File('${file.path}.tmp');
    await temp.writeAsString(text, flush: true);
    await temp.rename(file.path);
  }
  Future<LabProject?> loadLast() async {
    final dir = await _dir();
    final pointer = File('${dir.path}/last.json');
    if (!await pointer.exists()) return null;
    final id = (jsonDecode(await pointer.readAsString()) as Map)['id'] as String;
    if (!RegExp(r'^[0-9a-f]{64}$').hasMatch(id)) throw const FormatException('Повреждён указатель workspace.');
    final file = File('${dir.path}/$id.json');
    if (!await file.exists()) return null;
    final json = Map<String, dynamic>.from(jsonDecode(await file.readAsString()));
    final project = LabProject.fromJson(Map<String, dynamic>.from(json['project']));
    labRequire(_identity(project) == id, 'Идентичность сохранённого ROM не совпадает.');
    _restore(json, project);
    for (final m in project.maps) if (defaults.containsKey(m.id)) m.data = labCopy(defaults[m.id]!);
    return project;
  }
  Future<void> loadFor(LabProject project) async {
    await _queue;
    final dir = await _dir(); final file = File('${dir.path}/${_identity(project)}.json');
    if (!await file.exists()) { history.clear(); defaults.clear(); return; }
    final json = Map<String, dynamic>.from(jsonDecode(await file.readAsString()));
    final stored = LabProject.fromJson(Map<String, dynamic>.from(json['project']));
    labRequire(_identity(stored) == _identity(project), 'Сохранённые карты от другого ROM.');
    final previousMaps = <String, LabMap>{};
    for (final m in project.maps) {
      final candidates = stored.maps.where((v) => v.id == m.id);
      if (candidates.isNotEmpty) {
        final previous = candidates.single;
        labRequire(jsonEncode(previous.x.toJson()) == jsonEncode(m.x.toJson()) && jsonEncode(previous.y.toJson()) == jsonEncode(m.y.toJson()), 'Сохранённые оси изменились.');
        labRequire(jsonEncode(previous.original) == jsonEncode(m.original), 'Оригинал сохранённой карты отличается.');
        previousMaps[m.id] = previous;
      }
    }
    _restore(json, project);
    for (final m in project.maps) if (previousMaps.containsKey(m.id)) m.data = labCopy(defaults[m.id] ?? previousMaps[m.id]!.data);
  }
  void _restore(LabJson json, LabProject project) {
    final nextHistory = <String, List<LabRevision>>{};
    final nextDefaults = <String, List<List<double>>>{};
    final h = Map<String, dynamic>.from(json['history'] ?? {});
    final d = Map<String, dynamic>.from(json['defaults'] ?? {});
    for (final m in project.maps) {
      if (d.containsKey(m.id)) nextDefaults[m.id] = labMatrix(d[m.id], m.rows, m.cols);
      final items = h[m.id] as List? ?? [];
      nextHistory[m.id] = items.map((j) => LabRevision(j['version'] as int, j['at'] as String, j['comment'] as String,
          labMatrix(j['data'], m.rows, m.cols))).toList();
      if (nextHistory[m.id]!.length > 20) nextHistory[m.id] = nextHistory[m.id]!.sublist(nextHistory[m.id]!.length - 20);
    }
    history..clear()..addAll(nextHistory);
    defaults..clear()..addAll(nextDefaults);
  }
  Future<void> save(LabProject project, List<LabMap> maps, {bool asDefault = false, String comment = ''}) {
    // Snapshot before scheduling: a queued save must not observe later UI edits.
    final snapshot = LabProject.fromJson(project.toJson());
    final selected = maps.map((m) => m.id).toSet();
    final run = _queue.then((_) async {
      final nextHistory = {for (final e in history.entries) e.key: List<LabRevision>.of(e.value)};
      final nextDefaults = {for (final e in defaults.entries) e.key: labCopy(e.value)};
      for (final m in snapshot.maps.where((m) => selected.contains(m.id))) {
        final revisions = nextHistory[m.id] ?? [];
        final version = revisions.isEmpty ? 1 : revisions.last.version + 1;
        revisions.add(LabRevision(version, DateTime.now().toUtc().toIso8601String(),
            comment.isEmpty ? asDefault ? 'Пользовательский дефолт' : 'Ручные правки' : comment, labCopy(m.data)));
        nextHistory[m.id] = revisions.length <= 20 ? revisions : revisions.sublist(revisions.length - 20);
        if (asDefault) nextDefaults[m.id] = labCopy(m.data);
      }
      final document = {'project': snapshot.toJson(), 'defaults': nextDefaults,
        'history': {for (final e in nextHistory.entries) e.key: e.value.map((r) => r.toJson()).toList()}};
      final dir = await _dir(); final id = _identity(snapshot);
      await _atomic(File('${dir.path}/$id.json'), jsonEncode(document));
      history..clear()..addAll(nextHistory); defaults..clear()..addAll(nextDefaults);
      await _atomic(File('${dir.path}/last.json'), jsonEncode({'id': id}));
    });
    _queue = run.then<void>((_) {}, onError: (Object _, StackTrace __) {});
    return run;
  }
  Future<void> removeDefaults(LabProject project, List<String> ids) async {
    await _queue;
    final old = {for (final e in defaults.entries) e.key: labCopy(e.value)};
    for (final id in ids) defaults.remove(id);
    try { await save(project, [], comment: 'Удаление дефолтов'); }
    catch (_) { defaults..clear()..addAll(old); rethrow; }
  }
}'''
BRIDGE = r'''import 'dart:convert';
import 'dart:io';
import 'dart:math' as math;
import 'dart:typed_data';
import 'package:crypto/crypto.dart';
import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import '../constants.dart';
import '../models/rom_table.dart';
import 'rom_service.dart';
import 'connection_service.dart';
import 'subaru_lab_core.dart';

List<int> labEncodeRaw(double raw, String storage, int size) {
  labRequire(raw.isFinite, 'Обратная формула вернула NaN/Infinity.');
  final sizes = {'uint8': 1, 'int8': 1, 'uint16': 2, 'int16': 2, 'uint32': 4, 'int32': 4, 'float': 4, 'float32': 4};
  labRequire(sizes[storage] == size, 'Неподдержанный тип или размер $storage / $size.');
  final data = ByteData(size);
  if (storage == 'float' || storage == 'float32') {
    data.setFloat32(0, raw, Endian.big);
    labRequire(data.getFloat32(0, Endian.big).isFinite, 'Переполнение float32.');
  } else {
    final value = raw.round();
    switch (storage) {
      case 'uint8': labRequire(value >= 0 && value <= 255, 'Переполнение uint8.'); data.setUint8(0, value); break;
      case 'int8': labRequire(value >= -128 && value <= 127, 'Переполнение int8.'); data.setInt8(0, value); break;
      case 'uint16': labRequire(value >= 0 && value <= 65535, 'Переполнение uint16.'); data.setUint16(0, value, Endian.big); break;
      case 'int16': labRequire(value >= -32768 && value <= 32767, 'Переполнение int16.'); data.setInt16(0, value, Endian.big); break;
      case 'uint32': labRequire(value >= 0 && value <= 4294967295, 'Переполнение uint32.'); data.setUint32(0, value, Endian.big); break;
      case 'int32': labRequire(value >= -2147483648 && value <= 2147483647, 'Переполнение int32.'); data.setInt32(0, value, Endian.big); break;
      default: throw const FormatException('Identity-fallback при записи запрещён.');
    }
  }
  return data.buffer.asUint8List();
}

class LabRomBridge {
  static const definitionFingerprint = '__DEFINITION_FINGERPRINT__';
  final RomService reader = RomService();
  Uint8List? _originalBytes;
  String _hash = '';
  final Map<String, RomTableDef> definitions = {};
  final List<String> warnings = [];
  bool get canWrite => _originalBytes != null;

  String _axisText(dynamic axis, bool units, String fallback) {
    if (axis == null) return fallback;
    try {
      final value = units ? axis.units : axis.name;
      if (value is String && value.isNotEmpty) return value;
    } catch (_) { /* Older definitions have no axis labels; never invent units. */ }
    return fallback;
  }
  dynamic _axisOf(dynamic def, bool isX) {
    try { return isX ? def.x : def.y; } catch (_) { return null; }
  }

  Future<LabProject?> pickRom() async {
    final picked = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['bin', 'rom']);
    if (picked == null) return null;
    final f = picked.files.single;
    final bytes = f.bytes ?? (f.path == null ? null : await File(f.path!).readAsBytes());
    labRequire(bytes != null, 'Не удалось прочитать ROM.');
    return loadRom(bytes!, f.name);
  }

  LabProject loadRom(Uint8List bytes, String name) {
    labRequire(bytes.length >= 0x2008 && bytes.length <= 4 * 1024 * 1024, 'Некорректный размер бинарного ROM. Intel HEX текст здесь не поддерживается.');
    final id = ascii.decode(bytes.sublist(0x2000, 0x2000 + AppConstants.calId.length), allowInvalid: true);
    labRequire(id == AppConstants.calId, 'CAL ID файла "$id" не совпадает с определением ${AppConstants.calId}.');
    reader.labLoadBytes(Uint8List.fromList(bytes), name);
    final hash = sha256.convert(bytes).toString();
    final maps = <LabMap>[]; final nextDefinitions = <String, RomTableDef>{};
    warnings.clear(); var total = 0;
    for (final def in reader.defs) {
      if (def.rows <= 1 && def.cols <= 1) continue;
      if (def.rows < 1 || def.cols < 1 || def.rows > 128 || def.cols > 128) { warnings.add('${def.name}: размер не поддержан.'); continue; }
      try {
        final table = reader.readTable(def);
        var xv = List<double>.of(table.xValues), yv = List<double>.of(table.yValues);
        if (xv.isEmpty && def.cols == 1) xv = [0];
        if (yv.isEmpty && def.rows == 1) yv = [0];
        labRequire(xv.length == def.cols && yv.length == def.rows, 'Оси не совпадают с размерами.');
        final x = LabAxis(_axisText(_axisOf(def, true), false, 'X'), _axisText(_axisOf(def, true), true, ''), xv);
        final y = LabAxis(_axisText(_axisOf(def, false), false, 'Y'), _axisText(_axisOf(def, false), true, ''), yv);
        final id = sha256.convert(utf8.encode(jsonEncode([def.name, def.address, def.rows, def.cols,
          def.data.storage, def.swapxy, x.toJson(), y.toJson()]))).toString();
        final map = LabMap(id: id, name: def.name, category: def.category, unit: def.units, address: def.address,
            x: x, y: y, original: table.z, data: table.z);
        total += map.rows * map.cols;
        if (total > 250000 || maps.length >= 500) { warnings.add('Лимит 500 карт / 250000 ячеек; остальные пропущены.'); break; }
        maps.add(map); nextDefinitions[id] = def;
      } catch (e) { warnings.add('${def.name}: $e'); }
    }
    labRequire(maps.isNotEmpty, 'Ни одна карта не прочитана. Проверьте определения, оси и типы.');
    _originalBytes = Uint8List.fromList(bytes); _hash = hash;
    definitions..clear()..addAll(nextDefinitions);
    return LabProject(calId: AppConstants.calId, romHash: hash, definitionHash: definitionFingerprint,
        sourceName: name, origin: 'rom', maps: maps);
  }

  void requireMatching(LabProject project) {
    labRequire(_originalBytes != null && _hash == project.romHash && project.definitionHash == definitionFingerprint,
        'Загрузите тот же исходный .bin (SHA-256 и определения должны совпадать). JSON/дефолт не заменяет исходный ROM.');
  }

  List<int> _encode(double physical, RomTableDef def) {
    final inverse = def.data.fr;
    labRequire(inverse != null, '${def.name}: нет обратной формулы, карта read-only.');
    final raw = inverse!(physical);
    // Only the big-endian Subaru writer is supported; actual V8 readback is checked below.
    return labEncodeRaw(raw, def.data.storage, def.data.sizeOf);
  }

  Future<List<File>> exportModified(LabProject project, List<LabMap> selected) async {
    requireMatching(project);
    labRequire(selected.isNotEmpty, 'Выберите карты для экспорта.');
    final original = _originalBytes!; final candidate = Uint8List.fromList(original);
    final writes = <int, int>{}; final changes = <LabJson>[];
    for (final map in selected) {
      final def = definitions[map.id]; labRequire(def != null, 'Карта не принадлежит текущему определению.');
      labRequire(def!.rows == map.rows && def.cols == map.cols, 'Размер карты изменён.');
      labRequire(map.address == def.address, 'Адрес карты изменён.');
      final base = reader.labPreviewBytes(original, def);
      labRequire(jsonEncode(base.z) == jsonEncode(map.original), 'Оригинальные значения не совпадают с загруженным ROM.');
      labRequire(jsonEncode(base.xValues.isEmpty && map.cols == 1 ? [0.0] : base.xValues) == jsonEncode(map.x.values) &&
          jsonEncode(base.yValues.isEmpty && map.rows == 1 ? [0.0] : base.yValues) == jsonEncode(map.y.values), 'Оси карты изменены.');
      for (var r = 0; r < map.rows; r++) {
        for (var c = 0; c < map.cols; c++) {
          if (map.data[r][c] == map.original[r][c]) continue;
          final offset = def.address + (def.swapxy ? c * def.rows + r : r * def.cols + c) * def.data.sizeOf;
          final encoded = _encode(map.data[r][c], def);
          labRequire(offset >= 0 && offset + encoded.length <= candidate.length, 'Запись вне границ ROM.');
          for (var b = 0; b < encoded.length; b++) {
            labRequire(!writes.containsKey(offset + b) || writes[offset + b] == encoded[b], 'Конфликт перекрывающихся карт.');
            writes[offset + b] = encoded[b]; candidate[offset + b] = encoded[b];
          }
          changes.add({'map': map.name, 'r': r, 'c': c, 'offset': offset, 'requested': map.data[r][c], 'original': map.original[r][c]});
        }
      }
    }
    // Re-read after ALL writes to detect overlapping maps, not just each intermediate buffer.
    for (final map in selected) {
      final def = definitions[map.id]!;
      final decoded = reader.labPreviewBytes(candidate, def);
      for (var r = 0; r < map.rows; r++) {
        for (var c = 0; c < map.cols; c++) {
          final actual = decoded.z[r][c]; final target = map.data[r][c];
          if (target == map.original[r][c]) {
            labRequire(actual == target, 'Round-trip изменил невыбранную ячейку.');
            continue;
          }
          labRequire(actual.isFinite, 'Декодирование экспортируемого значения не удалось.');
          final intendedRaw = def.data.fr!(target), actualRaw = def.data.fr!(actual);
          if (def.data.storage == 'float' || def.data.storage == 'float32') {
            labRequire((actual - target).abs() <= math.max(1e-5, target.abs() * 1e-5), 'Float round-trip не совпал. Запись остановлена.');
          } else {
            labRequire((actualRaw - intendedRaw.round()).abs() < 0.01, 'Integer round-trip не совпал (тип/endian/формула).');
          }
          for (final change in changes.where((e) => e['map'] == map.name && e['r'] == r && e['c'] == c)) change['encodedValue'] = actual;
        }
      }
    }
    labRequire(changes.isNotEmpty, 'Нет изменённых ячеек для записи.');
    for (var i = 0; i < candidate.length; i++) labRequire(candidate[i] == original[i] || writes.containsKey(i), 'Изменён байт вне выбранных ячеек.');
    labRequire(sha256.convert(original).toString() == project.romHash, 'Исходный ROM изменился.');
    final stamp = DateTime.now().toUtc().millisecondsSinceEpoch;
    final bin = await exportBytes(candidate, 'SUBA_${project.calId}_${stamp}_CHECKSUM_UNVERIFIED.bin');
    final report = await exportText(jsonEncode({'sourceSha256': project.romHash, 'outputSha256': sha256.convert(candidate).toString(),
      'definitionSha256': project.definitionHash, 'checksum': 'NOT recalculated; validate externally before flashing',
      'sourceUnchanged': true, 'changes': changes}), 'SUBA_${stamp}_changes.json');
    return [bin, report];
  }

  Future<List<List<double>>> readEcuMap(LabProject project, LabMap map, {required bool Function() cancelled,
      required void Function(int, int) progress}) async {
    requireMatching(project);
    final def = definitions[map.id]!;
    final size = def.rows * def.cols * def.data.sizeOf;
    labRequire(size > 0 && size <= 65536 && def.address + size <= _originalBytes!.length && def.address + size <= 0x1000000,
        'Диапазон чтения карты не поддержан.');
    final bytes = Uint8List.fromList(_originalBytes!);
    await ConnectionService.I.exclusive((elm) async {
      for (var offset = 0; offset < size; offset += 32) {
        if (cancelled()) throw StateError('Чтение отменено; частичная карта не используется.');
        final n = math.min(32, size - offset);
        final chunk = await elm.readBytes(def.address + offset, n);
        labRequire(chunk != null && chunk.length == n, 'ЭБУ не вернул полную карту. UNLOCK и запись не выполняются.');
        bytes.setRange(def.address + offset, def.address + offset + n, chunk!);
        progress(offset + n, size);
      }
    });
    return labMatrix(reader.labPreviewBytes(bytes, def).z, map.rows, map.cols);
  }

  static Future<File> exportText(String text, String name) => exportBytes(utf8.encode(text), name);
  static Future<File> exportBytes(List<int> bytes, String name) async {
    final root = await getApplicationDocumentsDirectory();
    final dir = await Directory('${root.path}/subaru_lab_exports').create(recursive: true);
    final safe = name.replaceAll(RegExp(r'[^a-zA-Z0-9_.-]'), '_');
    final file = File('${dir.path}/${DateTime.now().microsecondsSinceEpoch}_$safe');
    await file.writeAsBytes(bytes, flush: true); return file;
  }
}'''
SCREEN = r'''import 'dart:async';
import 'dart:convert';
import 'dart:io';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import 'package:share_plus/share_plus.dart';
import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../services/subaru_lab_core.dart';
import '../services/subaru_lab_store.dart';
import '../services/subaru_lab_bridge.dart';
import '../widgets/heat_map.dart';
import '../widgets/subaru_map_table_v6.dart';

class SubaruV6AnalyzerScreen extends StatefulWidget {
  const SubaruV6AnalyzerScreen({super.key});
  @override
  State<SubaruV6AnalyzerScreen> createState() => _SubaruV6AnalyzerScreenState();
}

class _SubaruV6AnalyzerScreenState extends State<SubaruV6AnalyzerScreen>
    with AutomaticKeepAliveClientMixin<SubaruV6AnalyzerScreen> {
  final _store = LabStore();
  final _bridge = LabRomBridge();
  final _analyzer = SubaruLogAnalyzer();
  final _options = LabOptions();
  final _logs = <LabLog>[];
  final _online = <LabSample>[];
  LabProject? _project;
  String? _mapId;
  LabResult? _result;
  bool _resultOnline = false;
  bool _busy = true, _recording = false, _cancelRead = false, _confirmedUnits = false, _tuningOnly = true, _v6view = true;
  String _message = 'Загрузка сохранённых карт...', _view = 'values';
  String _preset = 'ignition';
  StreamSubscription<LiveSnapshot>? _sub;
  Timer? _ticker;
  Object? _recordingPoller;
  int _dropped = 0;

  @override
  bool get wantKeepAlive => true;

  @override
  void initState() {
    super.initState();
    ConnectionService.I.addListener(_connectionChanged);
    _restore();
  }
  Future<void> _restore() async {
    try {
      final project = await _store.loadLast();
      if (!mounted) return;
      setState(() { _project = project; _mapId = project?.maps.first.id;
        _message = project == null ? 'Загрузите Subaru .bin или пакет карт JSON. Nissan-карты не используются.' : 'Пользовательские карты восстановлены. Для .bin экспорта снова загрузите исходный ROM.'; });
    } catch (e) { if (mounted) setState(() => _message = 'Хранилище не загружено: $e'); }
    finally { if (mounted) setState(() => _busy = false); }
  }
  @override
  void dispose() {
    _cancelRead = true;
    _recording = false;
    _ticker?.cancel(); _sub?.cancel();
    ConnectionService.I.removeListener(_connectionChanged);
    super.dispose();
  }
  void _connectionChanged() {
    if (!mounted) return;
    if (_recording && (!ConnectionService.I.isPolling || !identical(_recordingPoller, ConnectionService.I.poller))) {
      _stopRecord();
      setState(() => _message = 'Запись остановлена: опрос остановлен или изменился поллер.');
    } else { setState(() {}); }
  }
  LabMap? get _map {
    final project = _project;
    if (project == null) return null;
    final visible = _visibleMaps;
    return project.maps.firstWhere((m) => m.id == _mapId,
        orElse: () => visible.isNotEmpty ? visible.first : project.maps.first);
  }

  List<LabMap> get _tuningMaps {
    final project = _project;
    if (project == null) return [];
    final list = project.maps.where((m) => labMatchTuning(m.name, m.category) != null).toList();
    list.sort((a, b) => labTuningOrder(a.name, a.category).compareTo(labTuningOrder(b.name, b.category)));
    return list;
  }

  List<LabMap> get _visibleMaps {
    final project = _project;
    if (project == null) return [];
    if (!_tuningOnly) {
      final all = List<LabMap>.of(project.maps);
      all.sort((a, b) => labTuningOrder(a.name, a.category).compareTo(labTuningOrder(b.name, b.category)));
      return all;
    }
    return _tuningMaps;
  }
  List<String> _columns(bool online) {
    final names = <String>{};
    if (online) { for (final sample in _online) names.addAll(sample.values.keys); }
    else { for (final log in _logs) names.addAll(log.columns); }
    return names.toList()..sort();
  }
  Future<void> _run(Future<void> Function() task) async {
    if (_busy) return;
    setState(() => _busy = true);
    try { await task(); }
    catch (e) { if (mounted) setState(() => _message = e.toString()); }
    finally { if (mounted) setState(() => _busy = false); }
  }
  Future<bool> _confirm(String title, String message) async => await showDialog<bool>(context: context,
      builder: (ctx) => AlertDialog(title: Text(title), content: SingleChildScrollView(child: Text(message)), actions: [
        TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('Отмена')),
        FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('Подтвердить')),
      ])) ?? false;
  Future<void> _share(List<File> files) async {
    await Share.shareXFiles(files.map((f) => XFile(f.path)).toList());
    if (mounted) setState(() => _message = 'Файлы созданы. Через системное меню выберите сохранение в Загрузки/Файлы.');
  }

  Future<void> _loadRom() => _run(() async {
    if (_project?.maps.any((m) => m.changes > 0) == true &&
        !await _confirm('Загрузить ROM?', 'Рабочие правки будут заменены сохранёнными для нового ROM. Несохранённые правки сначала экспортируйте в JSON.')) return;
    final project = await _bridge.pickRom();
    if (project == null || !mounted) return;
    await _store.loadFor(project);
    // Как в V6: при загрузке ROM сохраняем карты для настройки как дефолтные (только новые).
    var autoDefaults = 0;
    for (final m in project.maps.where((m) => labMatchTuning(m.name, m.category) != null)) {
      if (!_store.defaults.containsKey(m.id)) {
        await _store.save(project, [m], asDefault: true, comment: 'Авто-дефолт при загрузке ROM (как V6)');
        autoDefaults++;
      }
    }
    if (!mounted) return;
    final tuningCount = project.maps.where((m) => labMatchTuning(m.name, m.category) != null).length;
    setState(() { _project = project; _mapId = project.maps.first.id; _result = null; _confirmedUnits = false;
      _message = 'Загружено ${project.maps.length} карт, для настройки: $tuningCount. Новых дефолтов: $autoDefaults (как V6). Пропущено: ${_bridge.warnings.length}.'; });
  });
  Future<void> _loadJson() => _run(() async {
    final picked = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['json']);
    if (picked == null) return;
    final f = picked.files.single;
    labRequire(f.size < 30 * 1024 * 1024, 'JSON слишком большой.');
    final text = f.bytes != null ? utf8.decode(f.bytes!) : await File(f.path!).readAsString();
    final project = LabProject.fromJson(Map<String, dynamic>.from(jsonDecode(text)));
    if (!mounted) return;
    if (!await _confirm('Импорт карт?', 'Загрузить ${project.maps.length} карт из JSON? Это не загрузка бинарного ROM. Текущая рабочая копия будет заменена.')) return;
    await _store.loadFor(project);
    if (!mounted) return;
    setState(() { _project = project; _mapId = project.maps.first.id; _result = null; _confirmedUnits = false; _message = 'JSON импортирован. Для .bin экспорта нужен совпадающий исходный ROM.'; });
  });
  Future<void> _loadLogs() => _run(() async {
    final picked = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv', 'txt'], allowMultiple: true);
    if (picked == null) return;
    final incoming = <LabLog>[];
    for (final f in picked.files) {
      labRequire(f.size <= 20 * 1024 * 1024, '${f.name}: лимит файла 20 МБ.');
      final text = f.bytes != null ? utf8.decode(f.bytes!) : await File(f.path!).readAsString();
      incoming.add(labParseCsv(text, f.name));
    }
    final totalRows = incoming.fold<int>(0, (n, l) => n + l.samples.length);
    labRequire(totalRows > 0, 'Файлы прочитаны, но записей 0. ${incoming.map((l) => '${l.name}: отклонено ${l.rejected}${l.rejectedSample.isEmpty ? '' : ' (${l.rejectedSample.first})'}').join(' | ')}');
    labRequire([..._logs, ...incoming].fold<int>(0, (n, l) => n + l.samples.length) <= 150000, 'Лимит 150000 строк. Очистите старые логи.');
    if (!mounted) return;
    final warns = incoming.expand((l) => l.warnings.map((w) => '${l.name}: $w')).join(' | ');
    setState(() { _logs.addAll(incoming); _result = null; _message = 'Импортировано $totalRows строк из ${incoming.length} файлов. ${warns.isEmpty ? 'Пропуски не заменены нулями.' : warns}'; });
  });

  Future<void> _saveTuningDefaults() => _run(() async {
    final project = _project!;
    final tuning = project.maps.where((m) => labMatchTuning(m.name, m.category) != null).toList();
    labRequire(tuning.isNotEmpty, 'Карты для настройки не найдены в этом ROM.');
    if (!await _confirm('Набор в дефолт?', 'Сохранить дефолты для ${tuning.length} карт набора? Остальные карты не тронуты. Заводской оригинал не меняется.')) return;
    await _store.save(project, tuning, asDefault: true, comment: 'Набор для настройки (как V6)');
    if (mounted) setState(() => _message = 'Сохранено дефолтов для настройки: ${tuning.length}.');
  });

  void _startRecord() {
    final poller = ConnectionService.I.poller;
    if (_recording || poller == null || !ConnectionService.I.isPolling) { setState(() => _message = 'Сначала подключитесь и запустите опрос.'); return; }
    _online.clear(); _dropped = 0; _result = null; _recordingPoller = poller;
    _recording = true;
    _sub = poller.snapshots.listen((snapshot) {
      if (!_recording) return;
      final values = <String, double?>{
        ...snapshot.byId.map((k, v) => MapEntry(k, v.isFinite ? v : null)),
        ...snapshot.c.map((k, v) => MapEntry('canon:$k', v.isFinite ? v : null)),
      };
      _online.add(LabSample(snapshot.ts.millisecondsSinceEpoch, values, 'online'));
      if (_online.length > 5000) { _online.removeRange(0, 500); _dropped += 500; }
    }, onError: (Object e) {
      _stopRecord(); if (mounted) setState(() => _message = 'Поток остановлен: $e');
    }, onDone: () { _stopRecord(); });
    _ticker = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) _connectionChanged(); });
    setState(() => _message = 'Запись потока V8. Время snapshot не гарантирует свежесть всех PID; анализ требует проверенных каналов.');
  }
  void _stopRecord() {
    _recording = false; _sub?.cancel(); _sub = null; _ticker?.cancel(); _ticker = null;
    if (mounted) setState(() {});
  }
  void _analyze(bool online) {
    try {
      final map = _map;
      labRequire(map != null && _confirmedUnits, 'Загрузите карту и подтвердите единицы/смысл каналов.');
      final rows = online ? List<LabSample>.of(_online) : labMerge(_logs.expand((l) => l.samples).toList()).samples;
      labRequire(rows.isNotEmpty, 'Нет записей для анализа.');
      final result = _analyzer.analyze(rows, map!, _options);
      setState(() { _result = result; _resultOnline = online; _view = 'hits'; _message = 'Распределено ${result.accepted} из ${result.total} записей. Автоправки калибровок не выполняются.'; });
    } catch (e) { setState(() => _message = e.toString()); }
  }
  Future<void> _save({bool asDefault = false, bool all = false}) => _run(() async {
    final project = _project!;
    final maps = all ? project.maps : [_map!];
    if (!await _confirm(asDefault ? 'Сохранить в дефолт?' : 'Сохранить правки?',
        '${maps.length} карт. Пользовательская база привязана к SHA-256 ROM и определениям. Заводской оригинал и ЭБУ не изменяются. История: до 20 ревизий.')) return;
    await _store.save(project, maps, asDefault: asDefault);
    if (mounted) setState(() => _message = asDefault ? 'Пользовательские дефолты сохранены.' : 'Правки и история сохранены.');
  });
  Future<void> _saveBin() => _run(() async {
    final project = _project!;
    final changed = project.maps.where((m) => m.changes > 0).toList();
    labRequire(changed.isNotEmpty, 'Нет правок.');
    final selected = await showDialog<Set<String>>(context: context, builder: (ctx) {
      final ids = <String>{};
      return StatefulBuilder(builder: (ctx, update) => AlertDialog(
        title: const Text('Выберите карты для нового .bin'),
        content: SizedBox(width: double.maxFinite, child: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min, children: [
          const Text('Checksum НЕ пересчитывается. Файл будет помечен CHECKSUM_UNVERIFIED. Не прошивайте его до внешней проверки определения и checksum.'),
          ...changed.map((m) => CheckboxListTile(value: ids.contains(m.id), title: Text(m.name), subtitle: Text('${m.changes} правок'),
            onChanged: (v) => update(() { if (v == true) ids.add(m.id); else ids.remove(m.id); }))),
        ]))),
        actions: [TextButton(onPressed: () => Navigator.pop(ctx), child: const Text('Отмена')),
          FilledButton(onPressed: ids.isEmpty ? null : () => Navigator.pop(ctx, ids), child: const Text('Создать копию'))],
      ));
    });
    if (selected == null || selected.isEmpty) return;
    final files = await _bridge.exportModified(project, changed.where((m) => selected.contains(m.id)).toList());
    await _share(files);
  });
  Future<void> _readEcu() => _run(() async {
    final project = _project!, map = _map!;
    if (!await _confirm('Прочитать карту из ЭБУ?', 'Нужен загруженный совпадающий Subaru ROM. Аппаратный ECU ID не подтверждён автоматически. Оси берутся из ROM, читаются только значения. Ни UNLOCK, ни запись в ЭБУ не выполняются. Текущая рабочая карта будет заменена только после полного ответа.')) return;
    _cancelRead = false;
    final data = await _bridge.readEcuMap(project, map, cancelled: () => _cancelRead || !mounted,
        progress: (done, total) { if (mounted) setState(() => _message = 'Чтение карты: $done / $total байт'); });
    if (mounted) setState(() { map.data = data; _message = 'Значения ЭБУ загружены в рабочую копию. Оси из ROM; соответствие определению требует проверки.'; });
  });

  Widget _select(String label, String value, List<String> names, void Function(String) on, {bool optional = false}) => Padding(
    padding: const EdgeInsets.only(bottom: 6), child: DropdownButtonFormField<String>(
      isExpanded: true, initialValue: names.contains(value) ? value : null,
      key: ValueKey('$label:$value:${names.length}'),
      decoration: InputDecoration(labelText: label, isDense: true, border: const OutlineInputBorder()),
      items: [if (optional) const DropdownMenuItem(value: '', child: Text('Без вычитания')),
        ...names.map((n) => DropdownMenuItem(value: n, child: Text(n, overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 12))))],
      onChanged: _busy || _recording ? null : (v) { if (v != null) setState(() { on(v); _result = null; }); },
    ));
  Widget _selectors(bool online) {
    final map = _map; final columns = _columns(online);
    if (map == null) return const Padding(padding: EdgeInsets.all(16), child: Text('Откройте вкладку ROM/Карты и загрузите свой Subaru ROM или пакет JSON.'));
    final visible = _visibleMaps;
    if (visible.isEmpty) return const Padding(padding: EdgeInsets.all(16), child: Text('Нет карт под фильтром. Отключите «Только для настройки».'));
    final effectiveId = visible.any((m) => m.id == map.id) ? map.id : visible.first.id;
    return Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
      Row(children: [
        Expanded(child: Text('Для настройки: ${_tuningMaps.length}/${_project!.maps.length}', style: const TextStyle(fontSize: 11))),
        FilterChip(label: const Text('Только для настройки', style: TextStyle(fontSize: 11)), selected: _tuningOnly,
            onSelected: (v) => setState(() { _tuningOnly = v; _result = null; })),
      ]),
      const SizedBox(height: 6),
      DropdownButtonFormField<String>(initialValue: effectiveId, key: ValueKey('map:$effectiveId:${visible.length}:$_tuningOnly'), isExpanded: true,
          decoration: const InputDecoration(labelText: 'Карта из вашего ROM', border: OutlineInputBorder(), isDense: true),
          items: visible.map((m) {
            final t = labMatchTuning(m.name, m.category);
            return DropdownMenuItem(value: m.id, child: Text(t != null ? '${t.shortRu} — ${m.name}' : m.name, overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 12)));
          }).toList(),
          onChanged: _busy || _recording ? null : (v) => setState(() { _mapId = v; _result = null; _confirmedUnits = false; })),
      const SizedBox(height: 8),
      DropdownButtonFormField<String>(initialValue: _preset, key: ValueKey('preset:$_preset'),
        decoration: const InputDecoration(labelText: 'Наблюдение Subaru', border: OutlineInputBorder(), isDense: true),
        items: const [
          DropdownMenuItem(value: 'ignition', child: Text('Зажигание / FBKC')),
          DropdownMenuItem(value: 'learning', child: Text('Изученная коррекция / FKL')),
          DropdownMenuItem(value: 'fuel', child: Text('Lambda wideband минус target')),
          DropdownMenuItem(value: 'avcs', child: Text('AVCS actual минус target')),
          DropdownMenuItem(value: 'boost', child: Text('Boost relative факт минус цель')),
          DropdownMenuItem(value: 'torque', child: Text('Момент: только реальные каналы')),
        ], onChanged: _busy || _recording ? null : (value) {
          if (value == null) return;
          final presets = <String, (List<String>, List<String>, double)>{
            'ignition': (['fbkc', 'canonfbkc', 'feedbackknockcorrection'], [], 1),
            'learning': (['fkl', 'canonfkl', 'finelearningknockcorrection'], [], 1),
            'fuel': (['lambdawideband', 'widebandlambda'], ['lambdatarget', 'targetlambda'], 0.03),
            'avcs': (['avcsactualdeg'], ['avcstargetdeg'], 2),
            'boost': (['boostbarrel'], ['boosttargetbarrel'], 0.1),
            'torque': (['actualtorquenm'], ['requestedtorquenm'], 10),
          };
          final preset = presets[value]!;
          String find(List<String> names) => columns.firstWhere((c) => names.contains(labNormalize(c)), orElse: () => '');
          setState(() { _preset = value; _options.metric = find(preset.$1); _options.reference = find(preset.$2);
            _options.threshold = preset.$3; _confirmedUnits = false; _result = null; });
        }),
      const SizedBox(height: 8),
      Wrap(spacing: 6, children: [for (final p in [('stability', 'Стабильность'), ('power', 'Мощность'), ('economy', 'Экономия')])
        ChoiceChip(label: Text(p.$2), selected: _options.focus == p.$1, onSelected: _recording ? null : (_) => setState(() { _options.focus = p.$1; _result = null; }))]),
      const Text('Фокус = фильтр: мощность TPS >=80%; экономия TPS 2..40% и скорость >=20. Не автотюнинг.', style: TextStyle(fontSize: 10)),
      const SizedBox(height: 8),
      _select('X: ${map.x.name} [${map.x.unit}]', _options.x, columns, (v) => _options.x = v),
      _select('Y: ${map.y.name} [${map.y.unit}]', _options.y, columns, (v) => _options.y = v),
      _select('Метрика (FBKC / lambda / AVCS / boost...)', _options.metric, columns, (v) => _options.metric = v),
      _select('Вычесть target (те же единицы)', _options.reference, columns, (v) => _options.reference = v, optional: true),
      if (_options.focus != 'stability') ...[
        _select('TPS в процентах', _options.tps, columns, (v) => _options.tps = v),
        _select('Скорость км/ч', _options.speed, columns, (v) => _options.speed = v),
      ],
      Row(children: [
        Expanded(child: TextFormField(initialValue: '5', decoration: const InputDecoration(labelText: 'Мин. записей'), keyboardType: TextInputType.number,
          onChanged: (v) { _options.minSamples = (int.tryParse(v) ?? 5).clamp(2, 1000).toInt(); setState(() => _result = null); })),
        const SizedBox(width: 10),
        Expanded(child: TextFormField(initialValue: _options.threshold.toString(), key: ValueKey('threshold:$_preset'), decoration: const InputDecoration(labelText: 'Порог |метрика|'), keyboardType: const TextInputType.numberWithOptions(decimal: true),
          onChanged: (v) { _options.threshold = (labNumber(v) ?? 1).abs(); setState(() => _result = null); })),
      ]),
      CheckboxListTile(contentPadding: EdgeInsets.zero, dense: true, value: _confirmedUnits,
        title: const Text('Проверены единицы, смысл и свежесть каналов. g/rev не заменяется на %, relative boost не равен absolute.', style: TextStyle(fontSize: 11)),
        onChanged: (v) => setState(() => _confirmedUnits = v == true)),
    ]);
  }
  Future<void> _editCellDialog(LabMap map, int r, int c) async {
    final ctl = TextEditingController(text: map.data[r][c].toString());
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        title: Text('Ячейка [$r, $c]', style: const TextStyle(fontSize: 14)),
        content: Column(mainAxisSize: MainAxisSize.min, crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('${map.y.name}: ${map.y.values[r]} ${map.y.unit}\n${map.x.name}: ${map.x.values[c]} ${map.x.unit}', style: const TextStyle(fontSize: 12)),
          const SizedBox(height: 8),
          TextField(controller: ctl, autofocus: true, keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
              decoration: InputDecoration(labelText: 'Значение, ${map.unit}', border: const OutlineInputBorder())),
        ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('Отмена')),
          FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('Применить')),
        ],
      ),
    );
    if (ok != true || !mounted) return;
    final v = double.tryParse(ctl.text.replaceAll(',', '.'));
    if (v == null || !v.isFinite) {
      if (mounted) setState(() => _message = 'Введите конечное число.');
      return;
    }
    setState(() { map.data[r][c] = v; _message = 'Ячейка [$r, $c] изменена. Нажмите Сохранить.'; });
  }

  Widget _mapView({LabResult? result}) {
    final map = _map; if (map == null) return const SizedBox.shrink();
    final tuning = labMatchTuning(map.name, map.category);
    final changes = <String, V6MapChange>{};
    for (var r = 0; r < map.rows; r++) {
      for (var c = 0; c < map.cols; c++) {
        if (map.data[r][c] != map.original[r][c]) {
          final st = result?.cells['$r:$c'];
          changes['$r:$c'] = V6MapChange(r: r, c: c, suggested: map.data[r][c],
              reason: st != null ? 'Ручная правка · n=${st.count}' : 'Ручная правка', samples: st?.count ?? 0);
        }
      }
    }
    final revs = _store.history[map.id] ?? [];
    return Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
      if (tuning != null)
        Container(padding: const EdgeInsets.all(8), margin: const EdgeInsets.only(bottom: 6),
            decoration: BoxDecoration(color: Colors.cyan.withOpacity(0.08), borderRadius: BorderRadius.circular(8)),
            child: Text('${tuning.titleRu} · ${tuning.description}', style: const TextStyle(fontSize: 11, color: Colors.cyan))),
      Row(children: [
        const Text('Вид: ', style: TextStyle(fontSize: 11)),
        ChoiceChip(label: const Text('V6'), selected: _v6view, onSelected: (_) => setState(() => _v6view = true)),
        const SizedBox(width: 6),
        ChoiceChip(label: const Text('Тепло'), selected: !_v6view, onSelected: (_) => setState(() => _v6view = false)),
        const Spacer(),
        if (result != null)
          DropdownButton<String>(
            value: _view, isDense: true,
            items: const [DropdownMenuItem(value: 'values', child: Text('Карта')), DropdownMenuItem(value: 'hits', child: Text('Покрытие')), DropdownMenuItem(value: 'metric', child: Text('Метрика'))],
            onChanged: (v) => setState(() => _view = v ?? 'values'),
          ),
      ]),
      const SizedBox(height: 6),
      if (_v6view)
        SizedBox(
          height: MediaQuery.of(context).size.height * 0.62,
          child: MapTableV6(
            name: tuning != null ? '${tuning.titleRu} (${map.name})' : map.name,
            address: map.address.toRadixString(16).toUpperCase(),
            unit: map.unit,
            xValues: map.x.values,
            yValues: map.y.values,
            original: map.original,
            data: map.data,
            changes: changes,
            hasSavedEdits: _store.defaults.containsKey(map.id) || map.changes > 0,
            savedAt: revs.isEmpty ? null : DateTime.tryParse(revs.last.at),
            onSave: _busy || _recording ? null : () => _save(),
            onResetOne: () async {
              if (await _confirm('К оригиналу?', 'Заменить рабочие значения на исходные из ROM?') && mounted) {
                setState(() { map.data = labCopy(map.original); });
              }
            },
            onResetAll: () async {
              if (await _confirm('Сбросить ВСЕ карты?', 'Все рабочие копии вернутся к оригиналу ROM. История и дефолты сохранятся.') && mounted) {
                setState(() { for (final m in _project!.maps) { m.data = labCopy(m.original); } });
              }
            },
            onEditCell: _busy || _recording ? null : (r, c) => _editCellDialog(map, r, c),
          ),
        )
      else
        SizedBox(height: MediaQuery.of(context).size.height * 0.62, child: HeatMapView(
          rows: map.rows, cols: map.cols, title: '${map.name} | LAB-SUBA-V1',
          unit: _view == 'values' ? map.unit : _view == 'hits' ? 'samples' : _options.metric,
          xAxisName: '${map.x.name} ${map.x.unit}', yAxisName: '${map.y.name} ${map.y.unit}',
          xLabel: (c) => map.x.values[c].toString(), yLabel: (r) => map.y.values[r].toString(),
          value: (r, c) => _view == 'values' ? map.data[r][c] : _view == 'hits' ? result?.cells['$r:$c']?.count.toDouble() : result?.cells['$r:$c']?.mean,
          original: _view == 'values' ? (r, c) => map.original[r][c] : null,
          hits: result == null ? null : (r, c) => result.cells['$r:$c']?.count ?? 0,
          onEdit: _view != 'values' || _busy || _recording ? null : (r, c, value) {
            if (mounted) setState(() { map.data[r][c] = value; _message = 'Рабочая ячейка изменена. Нажмите Сохранить или В дефолт.'; });
          },
        )),
      Text('${map.changes} ручных правок. V6: тап — выбор, двойной тап — правка. Числа в метрике не являются калибровкой.', style: const TextStyle(fontSize: 11)),
      Wrap(spacing: 6, children: [
        FilledButton.icon(onPressed: _busy || _recording ? null : () => _save(), icon: const Icon(Icons.save), label: const Text('Сохранить')),
        OutlinedButton(onPressed: _busy || _recording ? null : () => _save(asDefault: true), child: const Text('В дефолт')),
        TextButton(onPressed: _busy ? null : () => _run(() async { await _share([await LabRomBridge.exportText(labMapCsv(map), '${map.id}.csv')]); }), child: const Text('CSV карты')),
      ]),
    ]);
  }
  Widget _resultView(LabResult? result) {
    if (result == null) return const SizedBox.shrink();
    final marked = result.cells.values.where((c) => c.count >= _options.minSamples &&
        (c.minimum.abs() >= _options.threshold || c.maximum.abs() >= _options.threshold)).toList();
    return Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
      Text('Принято ${result.accepted}/${result.total}; покрыто ${result.cells.length} ячеек', style: const TextStyle(fontWeight: FontWeight.bold)),
      Text('Нет каналов: ${result.missing}; вне осей: ${result.outside}; фильтр: ${result.filtered}', style: const TextStyle(fontSize: 12)),
      Text('Наблюдений выше пользовательского порога: ${marked.length}. Это не диагноз и не автоматические правки.', style: const TextStyle(fontSize: 12)),
      ...marked.take(30).map((c) => ListTile(dense: true,
        title: Text('[${c.row}, ${c.col}] n=${c.count}; mean=${c.mean.toStringAsFixed(3)}'),
        subtitle: Text('min=${c.minimum}, max=${c.maximum}. Требуется просмотр лога.'))),
      if (marked.length > 30) Text('Показано 30 из ${marked.length}. Все ячейки есть в JSON-отчёте.'),
      OutlinedButton(onPressed: _busy ? null : () => _run(() async { await _share([await LabRomBridge.exportText(jsonEncode({
        'romHash': _project!.romHash, 'map': _map!.name, 'options': _options.toJson(), 'result': result.toJson(),
        'freshness': 'not verified by snapshot timestamp',
      }), 'subaru_analysis.json')]); }), child: const Text('Экспорт отчёта JSON')),
    ]);
  }

  Widget _analysisTab(bool online) => ListView(padding: const EdgeInsets.all(12), children: [
    if (online) ...[
      Text('Записей: ${_online.length}; удалено из кольцевого буфера: $_dropped'),
      FilledButton.icon(onPressed: _busy ? null : _recording ? _stopRecord : _startRecord,
          icon: Icon(_recording ? Icons.stop : Icons.play_arrow), label: Text(_recording ? 'СТОП ЗАПИСЬ' : 'ЗАПИСЬ ПОТОКА V8')),
      const Text('Нет автоматической правки каждые 5 секунд. Сначала остановите запись и проверьте каналы.', style: TextStyle(fontSize: 11)),
    ] else ...[
      FilledButton.icon(onPressed: _busy ? null : _loadLogs, icon: const Icon(Icons.folder_open), label: const Text('CSV / объединить несколько')),
      ..._logs.asMap().entries.map((e) => ListTile(dense: true, title: Text(e.value.name),
        subtitle: Text('${e.value.samples.length} строк; пропущено ${e.value.rejected}; дублей ${e.value.duplicates}\n'
            'Разделитель «${e.value.delimiter == '\t' ? 'TAB' : e.value.delimiter}» · время: ${e.value.timeColumn ?? 'по порядку'}'
            '${e.value.warnings.isEmpty ? '' : '\n${e.value.warnings.join(' | ')}'}'
            '${e.value.rejectedSample.isEmpty ? '' : '\n${e.value.rejectedSample.first}'}'),
        trailing: IconButton(icon: const Icon(Icons.close), onPressed: _busy ? null : () => setState(() { _logs.removeAt(e.key); _result = null; })))),
      const Text('Удаляются только точные дубли. Записи с интервалом 200 мс не считаются дублями.', style: TextStyle(fontSize: 11)),
    ],
    const SizedBox(height: 12), _selectors(online),
    FilledButton.icon(onPressed: _busy || _recording || _map == null ? null : () => _analyze(online), icon: const Icon(Icons.analytics), label: const Text('АНАЛИЗИРОВАТЬ')),
    const SizedBox(height: 12), _resultView(_resultOnline == online ? _result : null),
    _mapView(result: _resultOnline == online ? _result : null),
  ]);

  Widget _romTab() => ListView(padding: const EdgeInsets.all(12), children: [
    Wrap(spacing: 8, children: [
      FilledButton.icon(onPressed: _busy || _recording ? null : _loadRom, icon: const Icon(Icons.folder_open), label: const Text('Subaru .bin')),
      OutlinedButton(onPressed: _busy || _recording ? null : _loadJson, child: const Text('Карты JSON')),
    ]),
    if (_project != null) ...[
      SelectableText('${_project!.sourceName}\nCAL: ${_project!.calId}\nROM SHA-256: ${_project!.romHash}\nДля настройки: ${_tuningMaps.length}/${_project!.maps.length}', style: const TextStyle(fontSize: 11)),
      const SizedBox(height: 10),
      Row(children: [
        Expanded(child: Text('Набор для правки и настройки', style: const TextStyle(fontSize: 11))),
        FilterChip(label: const Text('Только для настройки', style: TextStyle(fontSize: 11)), selected: _tuningOnly,
            onSelected: (v) => setState(() { _tuningOnly = v; _result = null; })),
      ]),
      const SizedBox(height: 6),
      DropdownButton<String>(isExpanded: true, value: _visibleMaps.any((m) => m.id == _map!.id) ? _map!.id : _visibleMaps.first.id,
        items: _visibleMaps.map((m) {
          final t = labMatchTuning(m.name, m.category);
          return DropdownMenuItem(value: m.id, child: Text(t != null ? '${t.shortRu} — ${m.name}' : m.name, overflow: TextOverflow.ellipsis));
        }).toList(),
        onChanged: _busy ? null : (v) => setState(() { _mapId = v; _result = null; })),
      _mapView(),
      Wrap(spacing: 8, children: [
        FilledButton.tonalIcon(onPressed: _busy ? null : _saveTuningDefaults, icon: const Icon(Icons.star, size: 16), label: const Text('Набор в дефолт (как V6)')),
        OutlinedButton(onPressed: _busy ? null : () => _save(asDefault: true, all: true), child: const Text('Все в дефолт')),
        OutlinedButton(onPressed: _busy ? null : () => _run(() async { await _share([await LabRomBridge.exportText(jsonEncode(_project!.toJson()), 'subaru_maps.json')]); }), child: const Text('Экспорт пакета JSON')),
        OutlinedButton(onPressed: _busy || _recording ? null : _saveBin, child: const Text('Правки в новый .bin')),
        OutlinedButton(onPressed: _busy || _recording ? null : _readEcu, child: const Text('Читать значения из ЭБУ')),
      ]),
      if (_busy) TextButton(onPressed: () => _cancelRead = true, child: const Text('Отменить чтение ЭБУ')),
      const Text('Пользовательский дефолт не изменяет заводской ROM. .bin экспорт использует только явно выбранные карты; '
          'checksum не пересчитывается. Перед прошивкой обязательна внешняя проверка. Экспорт не отправляется в ЭБУ.', style: TextStyle(fontSize: 11, color: Colors.orangeAccent)),
    ],
    if (_bridge.warnings.isNotEmpty) ExpansionTile(title: Text('Предупреждения определения (${_bridge.warnings.length})'),
      children: [SelectableText(_bridge.warnings.take(100).join('\n'), style: const TextStyle(fontSize: 11))]),
  ]);
  Widget _historyTab() {
    final map = _map; if (map == null) return const Center(child: Text('Сначала загрузите карты.'));
    final revisions = _store.history[map.id] ?? [];
    return ListView(padding: const EdgeInsets.all(12), children: [
      Text(map.name, style: const TextStyle(fontWeight: FontWeight.bold)),
      Text(_store.defaults.containsKey(map.id) ? 'Есть пользовательский дефолт.' : 'Пользовательского дефолта нет.'),
      if (revisions.isEmpty) const Padding(padding: EdgeInsets.all(24), child: Text('История появится после сохранения.')),
      ...revisions.reversed.map((r) => ListTile(title: Text('v${r.version} | ${r.comment}'), subtitle: Text(r.at),
        trailing: TextButton(onPressed: _busy || _recording ? null : () async {
          if (await _confirm('Восстановить ревизию?', 'Заменить рабочую копию на v${r.version}?') && mounted) setState(() { map.data = labCopy(r.data); _message = 'Ревизия восстановлена в рабочую копию.'; });
        }, child: const Text('Вернуть')))),
      OutlinedButton(onPressed: _busy ? null : () => _run(() async {
        if (await _confirm('Удалить дефолт?', 'Удалить пользовательский дефолт только выбранной карты? История сохранится.')) {
          await _store.removeDefaults(_project!, [map.id]); if (mounted) setState(() => _message = 'Дефолт удалён.');
        }
      }), child: const Text('Сбросить дефолт этой карты')),
      OutlinedButton(onPressed: _busy ? null : () => _run(() async {
        if (await _confirm('Удалить все дефолты?', 'Только для этого ROM и определения. История и другие прошивки не изменяются.')) {
          await _store.removeDefaults(_project!, _project!.maps.map((m) => m.id).toList()); if (mounted) setState(() => _message = 'Все дефолты этого ROM удалены.');
        }
      }), child: const Text('Сбросить все дефолты этого ROM')),
    ]);
  }

  @override
  Widget build(BuildContext context) {
    super.build(context);
    return DefaultTabController(length: 4, child: Scaffold(
      appBar: AppBar(title: const Text('Анализатор Subaru | LAB-SUBA-V1', style: TextStyle(fontSize: 14)), automaticallyImplyLeading: false,
        bottom: const TabBar(isScrollable: true, tabs: [Tab(text: 'Онлайн'), Tab(text: 'Из лога'), Tab(text: 'ROM / Карты'), Tab(text: 'История')])),
      body: Column(children: [
        if (_busy) const LinearProgressIndicator(),
        Container(width: double.infinity, padding: const EdgeInsets.all(10), color: Colors.blueGrey.withValues(alpha: 0.1),
          child: Text(_message, style: const TextStyle(fontSize: 11))),
        Expanded(child: TabBarView(physics: const NeverScrollableScrollPhysics(), children: [
          _analysisTab(true), _analysisTab(false), _romTab(), _historyTab(),
        ])),
      ]),
    ));
  }
}'''
V6WIDGET = r'''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

/// Виджет карты в стиле Nissan V6: шапка, Сохранить/Сброс, выбранная ячейка,
/// таблица с осями, градиент зелёный→жёлтый→красный, правки синим/оранжевым.
/// Используется анализатором Subaru вместо generic HeatMapView.
class V6MapChange {
  final int r, c;
  final double suggested;
  final String reason;
  final int samples;
  const V6MapChange({required this.r, required this.c, required this.suggested, this.reason = '', this.samples = 0});
}

class MapTableV6 extends StatefulWidget {
  final String name;
  final String address;
  final String unit;
  final List<double> xValues;
  final List<double> yValues;
  final List<List<double>> original;
  final List<List<double>> data;
  final Map<String, V6MapChange> changes;
  final bool hasSavedEdits;
  final DateTime? savedAt;
  final VoidCallback? onSave;
  final VoidCallback? onResetOne;
  final VoidCallback? onResetAll;
  final void Function(int r, int c)? onEditCell;

  const MapTableV6({
    super.key,
    required this.name,
    required this.address,
    required this.unit,
    required this.xValues,
    required this.yValues,
    required this.original,
    required this.data,
    this.changes = const {},
    this.hasSavedEdits = false,
    this.savedAt,
    this.onSave,
    this.onResetOne,
    this.onResetAll,
    this.onEditCell,
  });

  @override
  State<MapTableV6> createState() => _MapTableV6State();
}

class _MapTableV6State extends State<MapTableV6> {
  int? _selR, _selC;
  double _cell = 52;

  double get _min {
    var m = double.infinity;
    for (final row in widget.original) {
      for (final v in row) {
        if (v < m) m = v;
      }
    }
    return m == double.infinity ? 0 : m;
  }

  double get _max {
    var m = -double.infinity;
    for (final row in widget.original) {
      for (final v in row) {
        if (v > m) m = v;
      }
    }
    return m == -double.infinity ? 1 : m;
  }

  Color _base(double v) {
    final range = _max - _min;
    final norm = range > 0 ? ((v - _min) / range).clamp(0.0, 1.0) : 0.5;
    const green = Color(0xFF1B5E20);
    const yellow = Color(0xFF827717);
    const red = Color(0xFFB71C1C);
    if (norm < 0.5) return Color.lerp(green, yellow, norm * 2)!.withOpacity(0.85);
    return Color.lerp(yellow, red, (norm - 0.5) * 2)!.withOpacity(0.85);
  }

  Color _changed(double pctAbs, double delta) {
    final up = delta > 0;
    if (pctAbs < 3) return up ? Colors.yellow.shade800 : Colors.lightBlue.shade800;
    if (pctAbs < 8) return up ? Colors.orange.shade800 : Colors.blue.shade800;
    return up ? Colors.red.shade800 : Colors.blueAccent.shade700;
  }

  String _ax(double v) => v.abs() >= 100 ? v.toStringAsFixed(0) : v.toStringAsFixed(1);
  String _cellFmt(double v) {
    if (widget.unit.toLowerCase().contains('lambda')) return v.toStringAsFixed(3);
    if (v.abs() >= 100) return v.toStringAsFixed(0);
    return v.toStringAsFixed(1);
  }

  @override
  Widget build(BuildContext context) {
    final rows = widget.data.length;
    final cols = rows > 0 ? widget.data[0].length : 0;
    final selChange = (_selR != null && _selC != null) ? widget.changes['${_selR!}:${_selC!}'] : null;
    return Column(children: [
      Container(
        padding: const EdgeInsets.all(6),
        color: const Color(0xFF16213E),
        child: Column(children: [
          Row(children: [
            const Icon(Icons.grid_on, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Expanded(
              child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                Text(widget.name, style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12), overflow: TextOverflow.ellipsis),
                if (widget.hasSavedEdits)
                  Text('С правками${widget.savedAt != null ? ' • ${DateFormat("dd.MM HH:mm").format(widget.savedAt!)}' : ''}',
                      style: const TextStyle(color: Colors.orange, fontSize: 9)),
              ]),
            ),
            Text('$rows x $cols', style: const TextStyle(color: Colors.white54, fontSize: 10)),
            const SizedBox(width: 6),
            GestureDetector(onTap: () => setState(() => _cell = (_cell - 8).clamp(32, 90)), child: const Icon(Icons.remove_circle_outline, color: Colors.white70, size: 20)),
            Text(_cell.toInt().toString(), style: const TextStyle(color: Colors.white54, fontSize: 10)),
            GestureDetector(onTap: () => setState(() => _cell = (_cell + 8).clamp(32, 90)), child: const Icon(Icons.add_circle_outline, color: Colors.white70, size: 20)),
          ]),
          if (widget.changes.isNotEmpty || widget.hasSavedEdits)
            Padding(
              padding: const EdgeInsets.only(top: 6),
              child: Row(children: [
                if (widget.changes.isNotEmpty && widget.onSave != null)
                  Expanded(
                    child: ElevatedButton.icon(
                      onPressed: widget.onSave,
                      icon: const Icon(Icons.save, size: 14),
                      label: const Text('Сохранить', style: TextStyle(fontSize: 11)),
                      style: ElevatedButton.styleFrom(backgroundColor: Colors.green, padding: const EdgeInsets.symmetric(vertical: 8)),
                    ),
                  ),
                if (widget.changes.isNotEmpty && widget.hasSavedEdits) const SizedBox(width: 6),
                if (widget.hasSavedEdits)
                  Expanded(
                    child: PopupMenuButton<String>(
                      onSelected: (v) => v == 'one' ? widget.onResetOne?.call() : widget.onResetAll?.call(),
                      itemBuilder: (_) => const [PopupMenuItem(value: 'one', child: Text('Сбросить эту')), PopupMenuItem(value: 'all', child: Text('Сбросить ВСЕ'))],
                      child: Container(
                        padding: const EdgeInsets.symmetric(vertical: 8, horizontal: 12),
                        decoration: BoxDecoration(color: Colors.orange, borderRadius: BorderRadius.circular(4)),
                        child: const Row(mainAxisAlignment: MainAxisAlignment.center, children: [
                          Icon(Icons.restore, color: Colors.white, size: 16),
                          SizedBox(width: 4),
                          Text('Сброс', style: TextStyle(color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
                        ]),
                      ),
                    ),
                  ),
              ]),
            ),
        ]),
      ),
      if (_selR != null && _selC != null)
        Container(
          padding: const EdgeInsets.all(6),
          margin: const EdgeInsets.all(4),
          decoration: BoxDecoration(color: Colors.cyan.withOpacity(0.15), borderRadius: BorderRadius.circular(6), border: Border.all(color: Colors.cyan.withOpacity(0.5))),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: [
              Expanded(
                child: Text('R${_selR! + 1} ${_ax(widget.yValues[_selR!])} ${widget.data.isNotEmpty ? '' : ''} | C${_selC! + 1} ${_ax(widget.xValues[_selC!])}',
                    style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12)),
              ),
              GestureDetector(onTap: () => setState(() { _selR = null; _selC = null; }), child: const Icon(Icons.close, size: 14, color: Colors.white54)),
            ]),
            if (selChange != null) ...[
              Text('${_cellFmt(widget.data[_selR!][_selC!])} → ${_cellFmt(selChange.suggested)}',
                  style: TextStyle(color: selChange.suggested - widget.data[_selR!][_selC!] > 0 ? Colors.green : Colors.orange, fontSize: 13)),
              Text(selChange.reason, style: const TextStyle(color: Colors.white70, fontSize: 10)),
            ] else
              Row(children: [
                Expanded(child: Text('${_cellFmt(widget.data[_selR!][_selC!])} ${widget.unit}', style: const TextStyle(color: Colors.white, fontSize: 13, fontWeight: FontWeight.bold))),
                if (widget.onEditCell != null)
                  TextButton(onPressed: () => widget.onEditCell!(_selR!, _selC!), child: const Text('Править', style: TextStyle(fontSize: 11))),
              ]),
          ]),
        ),
      Expanded(
        child: InteractiveViewer(
          constrained: false,
          minScale: 0.5,
          maxScale: 3.0,
          boundaryMargin: const EdgeInsets.all(20),
          child: Padding(
            padding: const EdgeInsets.all(4),
            child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
              Row(children: [
                Container(
                  width: _cell,
                  height: 28,
                  decoration: BoxDecoration(color: const Color(0xFF0F3460), border: Border.all(color: Colors.white24, width: 0.5)),
                  child: const Center(child: Text('RPM/%', style: TextStyle(fontSize: 9, color: Colors.white70))),
                ),
                ...List.generate(cols, (c) => Container(
                      width: _cell,
                      height: 28,
                      decoration: BoxDecoration(color: const Color(0xFF0F3460), border: Border.all(color: Colors.white24, width: 0.5)),
                      child: Center(child: Text(c < widget.xValues.length ? _ax(widget.xValues[c]) : '$c', style: const TextStyle(fontSize: 10, color: Colors.white70))),
                    )),
              ]),
              ...List.generate(rows, (r) => Row(children: [
                    Container(
                      width: _cell,
                      height: _cell,
                      decoration: BoxDecoration(color: const Color(0xFF0F3460), border: Border.all(color: Colors.white24, width: 0.5)),
                      child: Center(child: Text(r < widget.yValues.length ? _ax(widget.yValues[r]) : '$r', style: const TextStyle(fontSize: 10, color: Colors.white70))),
                    ),
                    ...List.generate(cols, (c) {
                      final ch = widget.changes['$r:$c'];
                      final isSel = _selR == r && _selC == c;
                      final orig = widget.data[r][c];
                      Color bg;
                      if (ch == null) {
                        bg = _base(orig);
                      } else {
                        final pct = (ch.suggested - orig).abs() / (orig.abs() + 1e-9) * 100;
                        bg = _changed(pct, ch.suggested - orig);
                      }
                      return GestureDetector(
                        onTap: () => setState(() { _selR = r; _selC = c; }),
                        onDoubleTap: widget.onEditCell == null ? null : () => widget.onEditCell!(r, c),
                        child: Container(
                          width: _cell,
                          height: _cell,
                          decoration: BoxDecoration(color: bg, border: Border.all(color: isSel ? Colors.cyan : ch != null ? Colors.white54 : Colors.white12, width: isSel ? 2 : (ch != null ? 1 : 0.5))),
                          child: ch != null
                              ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
                                  Text(_cellFmt(orig), style: TextStyle(fontSize: 9, color: Colors.white.withOpacity(0.7), decoration: TextDecoration.lineThrough)),
                                  Text(_cellFmt(ch.suggested), style: TextStyle(fontSize: 11, fontWeight: FontWeight.bold, color: ch.suggested - orig > 0 ? Colors.greenAccent : Colors.yellowAccent)),
                                ])
                              : Center(child: Text(_cellFmt(orig), style: const TextStyle(fontSize: 10, color: Colors.white70))),
                        ),
                      );
                    }),
                  ])),
            ]),
          ),
        ),
      ),
    ]);
  }
}
'''
TESTS = r'''import 'dart:convert';
import 'dart:io';
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import '../lib/services/subaru_lab_core.dart';
import '../lib/services/subaru_lab_store.dart';
import '../lib/services/subaru_lab_bridge.dart';
import '../lib/main.dart' as application;

LabMap sampleMap() => LabMap(id: 'test', name: 'Rectangular test', category: 'test', unit: 'test', address: 16,
    x: LabAxis('Load', 'g/rev', [0.5, 1, 2]), y: LabAxis('RPM', 'rpm', [800, 2000]),
    original: [[1, 2, 3], [4, 5, 6]], data: [[1, 2, 3], [4, 5, 6]]);
LabProject project(String hash) => LabProject(calId: 'TEST', romHash: hash, definitionHash: 'test-definition',
    sourceName: 'fixture', origin: 'demo', maps: [sampleMap()]);

void main() {
  test('Application imports the new analyzer without replacing old HeatGrid or CAN', () {
    expect(application.main, isNotNull);
    expect(LabRomBridge().canWrite, isFalse);
    expect(() => LabRomBridge().requireMatching(project('test')), throwsFormatException);
  });
  test('CSV BOM, quoting, semicolon, decimal comma, missing value', () {
    final log = labParseCsv('\uFEFFts_ms;rpm;FBKC\n1000;800;"-1,4"\n1050;900;\n', 'test');
    expect(log.samples.length, 2);
    expect(log.samples.first.values['FBKC'], -1.4);
    expect(log.samples.last.values['FBKC'], isNull);
    expect(log.rejected, 0);
    expect(log.delimiter, ';');
  });
  test('CSV without time keeps rows instead of zero records', () {
    final log = labParseCsv('rpm,load_grev,fbkc\n800,0.5,-1.4\n820,0.6,0\n', 'notime');
    expect(log.samples.length, 2);
    expect(log.timeColumn, isNull);
    expect(log.warnings.isNotEmpty, isTrue);
  });
  test('CSV clock time HH:MM:SS.mmm is parsed', () {
    final log = labParseCsv('time,rpm\n12:00:01.500,800\n12:00:02.000,820\n', 'clock');
    expect(log.samples.length, 2);
    expect(log.samples[1].timeMs - log.samples[0].timeMs, 500);
  });
  test('CSV duplicate headers do not fail import', () {
    final log = labParseCsv('rpm,rpm\n800,810\n', 'dupe');
    expect(log.samples.length, 1);
    expect(log.columns.length, 2);
  });
  test('Tuning set matches Subaru ROM names, other maps excluded', () {
    expect(labMatchTuning('Base Timing Primary', 'Ignition')?.key, 'base-timing');
    expect(labMatchTuning('Primary Open Loop Fueling', 'Fuel'), isNotNull);
    expect(labMatchTuning('Target Boost Primary', 'Boost'), isNotNull);
    expect(labMatchTuning('Some Random Scalar', 'Misc'), isNull);
    expect(labTuningOrder('Base Timing', ''), lessThan(labTuningOrder('Other', '')));
  });
  test('Exact duplicate removal preserves 50ms-spaced samples and different values', () {
    final log = labParseCsv('ts_ms,rpm\n1000,800\n1050,820\n1000,800\n1000,900\n', 'test');
    expect(log.samples.length, 3);
    expect(log.duplicates, 1);
    final merged = labMerge([...log.samples, ...log.samples]);
    expect(merged.samples.length, 3);
    expect(merged.duplicates, 3);
  });
  test('JSON roundtrip validates every row and retains axes and immutable original', () {
    final m = sampleMap(); m.data[1][2] = 9;
    expect(m.original[1][2], 6);
    final copy = LabMap.fromJson(Map<String, dynamic>.from(jsonDecode(jsonEncode(m.toJson()))));
    expect(copy.data[1][2], 9); expect(copy.rows, 2); expect(copy.cols, 3);
    final malformed = m.toJson(); malformed['data'] = [[1, 2, 3], [4, 5]];
    expect(() => LabMap.fromJson(malformed), throwsFormatException);
    expect(() => LabAxis('x', '', [1, 1, 2]), throwsFormatException);
  });
  test('Grouping uses matching units/explicit keys, rejects out-of-range and missing data', () {
    final log = labParseCsv('ts_ms,rpm,load_grev,fbkc\n1000,800,0.5,-1.4\n1050,800,0.5,0\n1100,2000,3,-2\n1150,2000,2,\n', 'test');
    final options = LabOptions()..x = 'load_grev'..y = 'rpm'..metric = 'fbkc';
    final result = SubaruLogAnalyzer().analyze(log.samples, sampleMap(), options);
    expect(result.accepted, 2); expect(result.outside, 1); expect(result.missing, 1);
    expect(result.cells['0:0']!.minimum, -1.4); expect(result.cells['0:0']!.mean, -0.7);
    expect(SubaruLogAnalyzer.nearest([4000, 2000, 800], 2100), 1);
  });
  test('Missing actual or target is not converted into zero error', () {
    final log = labParseCsv('ts_ms,rpm,load_grev,act,target\n1000,800,0.5,20,18\n1050,800,0.5,20,\n', 'test');
    final options = LabOptions()..x = 'load_grev'..y = 'rpm'..metric = 'act'..reference = 'target';
    final result = SubaruLogAnalyzer().analyze(log.samples, sampleMap(), options);
    expect(result.accepted, 1); expect(result.missing, 1); expect(result.cells['0:0']!.mean, 2);
  });
  test('Binary writer retains float32 precision, signed values and rejects overflow', () {
    expect(labEncodeRaw(-2, 'int16', 2), [255, 254]);
    expect(labEncodeRaw(65535, 'uint16', 2), [255, 255]);
    final bytes = Uint8List.fromList(labEncodeRaw(1.25, 'float', 4));
    expect(ByteData.sublistView(bytes).getFloat32(0, Endian.big), 1.25);
    expect(() => labEncodeRaw(65536, 'uint16', 2), throwsFormatException);
    expect(() => labEncodeRaw(double.nan, 'float', 4), throwsFormatException);
    expect(() => labEncodeRaw(1, 'unknown', 1), throwsFormatException);
  });
  test('Defaults/history are scoped by ROM, capped at 20, version numbers monotonic', () async {
    final dir = await Directory.systemTemp.createTemp('subaru_lab_test_');
    try {
      final store = LabStore(directory: () async => dir);
      final p = project('sha-first');
      for (var i = 0; i < 23; i++) {
        p.maps.first.data[0][0] = i.toDouble();
        await store.save(p, p.maps, asDefault: true, comment: 'v${i + 1}');
      }
      expect(store.history['test']!.length, 20);
      expect(store.history['test']!.last.version, 23);
      final loaded = await LabStore(directory: () async => dir).loadLast();
      expect(loaded!.maps.first.data[0][0], 22); expect(loaded.maps.first.original[0][0], 1);
      final other = project('sha-second');
      await store.loadFor(other);
      expect(other.maps.first.data[0][0], 1); expect(store.history, isEmpty);
    } finally { await dir.delete(recursive: true); }
  });
}'''

ROM_HELPERS = r'''
  // LAB_SUBA_V1_ROM_HELPERS_BEGIN
  void labLoadBytes(Uint8List bytes, String name) {
    rom = List<int>.of(bytes);
    fileName = name;
    _bd = ByteData.sublistView(Uint8List.fromList(bytes));
  }

  RomTable labPreviewBytes(List<int> bytes, RomTableDef definition) {
    final isolated = RomService();
    isolated.labLoadBytes(Uint8List.fromList(bytes), 'preview-only.bin');
    return isolated.readTable(definition);
  }
  // LAB_SUBA_V1_ROM_HELPERS_END
'''


def require(ok, message):
    if not ok:
        raise ValueError(message)


main_path = ROOT / 'lib/main.dart'
rom_path = ROOT / 'lib/services/rom_service.dart'
pubspec_path = ROOT / 'pubspec.yaml'
v6_path = ROOT / 'lib/widgets/subaru_map_table_v6.dart'
try:
    needed = [main_path, rom_path, pubspec_path, ROOT / 'lib/generated/subaru_rom.g.dart',
              ROOT / 'lib/models/rom_table.dart', ROOT / 'lib/models/live_snapshot.dart', ROOT / 'lib/widgets/heat_map.dart']
    missing = [str(p) for p in needed if not p.is_file()]
    if missing:
        print('Не хватает файлов:')
        for m in missing:
            print('  -', m)
        print('Похоже, ячейки 0–9 выполнены не полностью. Догоните их и повторите.')
        print('Файлы НЕ изменены.')
        raise SystemExit('Preflight failed: missing project files (see list above).')
    main = main_path.read_text(encoding='utf-8')
    rom = rom_path.read_text(encoding='utf-8')
    model = (ROOT / 'lib/models/rom_table.dart').read_text(encoding='utf-8')
    heatmap = (ROOT / 'lib/widgets/heat_map.dart').read_text(encoding='utf-8')
    require('xAxisName' in heatmap and 'onEdit' in heatmap, 'Apply MAP-AXES-V2 before this analyzer module.')
    for anchor in ('class RomService', 'List<int>? rom', 'ByteData? _bd', 'String fileName', 'RomTable readTable'):
        require(anchor in rom, 'Unexpected RomService API: ' + anchor + '. No source files changed.')
    for anchor in ('class RomTableDef', 'swapxy', 'xValues', 'yValues'):
        require(anchor in model, 'Unexpected RomTable model: ' + anchor)
    require(len(re.findall(r'(?m)^class\s+', rom)) == 1, 'RomService contains additional classes; manual adapter required.')
    # Проверка свежести встроенного кода (защита от обрезанной копии ячейки).
    require('labMatchTuning' in CORE and '_parseTime' in CORE and 'FirstOccurrence' not in CORE,
            'Встроенное ядро повреждено или обрезано — скопируйте ячейку заново целиком.')
    require('_tuningOnly' in SCREEN and 'MapTableV6' in SCREEN,
            'Встроенный экран поврежден или обрезан — скопируйте ячейку заново целиком.')
    require('class MapTableV6' in V6WIDGET, 'Встроенный V6-виджет поврежден — скопируйте ячейку заново целиком.')
    if 'LAB_SUBA_V1_ROM_HELPERS_BEGIN' not in rom:
        closing = rom.rfind('\n}')
        require(closing >= 0, 'RomService closing brace not found.')
        rom = rom[:closing] + '\n' + ROM_HELPERS + rom[closing:]
    else:
        require('labPreviewBytes' in rom and 'labLoadBytes' in rom, 'Partial ROM helper patch detected.')
    if 'SubaruV6AnalyzerScreen' not in main:
        import_anchor = "import 'screens/analyzer_screen.dart';"
        require(main.count(import_anchor) == 1, 'Analyzer import differs in main.dart.')
        require(len(re.findall(r'\bAnalyzerScreen\(\)', main)) == 1, 'Expected one AnalyzerScreen() in main.dart.')
        main = main.replace(import_anchor, "import 'screens/subaru_v6_analyzer_screen.dart';", 1)
        main = re.sub(r'\bAnalyzerScreen\(\)', 'SubaruV6AnalyzerScreen()', main, count=1)
    else:
        require("import 'screens/subaru_v6_analyzer_screen.dart';" in main, 'Partial analyzer navigation patch.')
    definitions = (ROOT / 'lib/generated/subaru_rom.g.dart').read_bytes()
    fingerprint = hashlib.sha256(definitions + b'\n' + model.encode('utf-8')).hexdigest()
    bridge = BRIDGE.replace('__DEFINITION_FINGERPRINT__', fingerprint)
except (OSError, ValueError) as error:
    # Без "from error": иначе Colab показывает пугающий Internal Python error в inspect.
    print('Preflight failed. Project sources NOT changed:', error)
    raise SystemExit(str(error))

outputs = {
    main_path: main, rom_path: rom,
    ROOT / 'lib/services/subaru_lab_core.dart': CORE,
    ROOT / 'lib/services/subaru_lab_store.dart': STORE,
    ROOT / 'lib/services/subaru_lab_bridge.dart': bridge,
    ROOT / 'lib/screens/subaru_v6_analyzer_screen.dart': SCREEN,
    v6_path: V6WIDGET,
    ROOT / 'test/subaru_lab_test.dart': TESTS,
}
previous = {path: path.read_bytes() if path.exists() else None for path in outputs}
lock_path = ROOT / 'pubspec.lock'
previous[pubspec_path] = pubspec_path.read_bytes()
previous[lock_path] = lock_path.read_bytes() if lock_path.exists() else None
backup = ROOT / 'patch_backups' / ('subaru_lab_v12_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for path, content in previous.items():
    if content is not None:
        saved = backup / path.relative_to(ROOT)
        saved.parent.mkdir(parents=True, exist_ok=True)
        saved.write_bytes(content)


def command(args, timeout):
    result = subprocess.run([str(FLUTTER), *args], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout)
    print(result.stdout)
    with (ROOT / 'subaru_lab_install.log').open('a', encoding='utf-8') as log:
        log.write(result.stdout + '\n')
    require(result.returncode == 0, 'Command failed: flutter ' + ' '.join(args))


try:
    (ROOT / 'subaru_lab_install.log').write_text('LAB-SUBA-V1+V2 install\n', encoding='utf-8')
    for path, content in outputs.items():
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists() or path.read_text(encoding='utf-8') != content:
            path.write_text(content, encoding='utf-8')
    if not re.search(r'(?m)^  crypto:', pubspec_path.read_text(encoding='utf-8')):
        command(['pub', 'add', 'crypto:^3.0.6'], 240)
    else:
        command(['pub', 'get'], 240)
    command(['test', '--no-pub', 'test/subaru_lab_test.dart'], 300)
except (OSError, ValueError, subprocess.TimeoutExpired) as error:
    for path, content in previous.items():
        if content is None:
            path.unlink(missing_ok=True)
        else:
            path.write_bytes(content)
    print('Source files and pubspec restored. Backups:', backup)
    print('Analyzer installation failed. Send subaru_lab_install.log:', error)
    raise SystemExit(str(error))

print('LAB-SUBA-V1+V2 установлен одной ячейкой, тесты прошли В ЭТОМ COLAB.')
print('Backups:', backup)
print('Что внутри: анализатор V6-workflow + CSV без 0 записей + набор карт для настройки + вид V6 + авто-дефолты.')
print('Next: run your EXISTING build cell 10/10, install APK, open Analyzer.')
print('New .bin export NEVER overwrites the original. Checksum is NOT recalculated: validate externally before flashing.')
print('These tests do NOT validate ECU addresses, ROM calibration, checksum, Android sharing or the real vehicle.')


ROOT = /content/suba_run_v8
FLUTTER = /content/flutter/bin/flutter
INSTALLER REV = 2026-09-10-v12-combined (если этой строки нет — копия старая, скопируйте заново)
Resolving dependencies...
  code_assets 1.2.1 (2.0.0 available)
  crypto 3.0.7 (from transitive dependency to direct dependency)
  csv 6.0.0 (8.0.0 available)
  file_picker 8.1.2 (12.2.0 available)
  fl_chart 0.68.0 (1.2.0 available)
  flutter_lints 4.0.0 (6.0.0 available)
  hooks 2.0.2 (2.2.0 available)
  intl 0.19.0 (0.20.3 available)
  lints 4.0.0 (6.1.0 available)
  material_color_utilities 0.13.0 (0.13.1 available)
  meta 1.18.3 (1.19.0 available)
  mime 1.0.6 (2.1.0 available)
  objective_c 9.5.0 (9.6.0 available)
  path_provider 2.1.4 (2.1.6 available)
  permission_handler 11.3.1 (13.0.2 available)
  permission_handler_android 12.1.0 (14.1.0 available)
  record_use 0.6.0 (1.1.1 available)
  share_plus 10.0.2 (13.3.0 available)
  share_plus_platform_interface 5.0.2 (7.2.0 available)
  shared_preferences 2.3.2 (2.5.5 av

In [ ]:
# @title SUBA RUN V8: Анализатор как в V6 (Subaru-адаптация)
# Порт механики Nissan V6 в SUBA RUN V8 с адаптацией под Subaru EJ20X / A2TB100B:
# модели TuningMap/AnalysisResult, анализатор (зажигание/топливо/AVCS/буст),
# паттерны, CSV из V8-логов, сплиттер, дефолты+история, запись правок в V8MOD,
# экран анализатора и виджет карты как в V6, кнопки "В дефолт" в ROM и ЭБУ Картах.
# Запускать ПОСЛЕ ячеек 0-9 (и после текущих фиксов), ПЕРЕД 10/10 (сборка APK).
# Идемпотентна — можно перезапускать. Старый HeatGrid-экран заменяется (бэкап .bak).
import os
os.chdir('/content/suba_run_v8')

def edit(path, old, new, tag):
    s = open(path, encoding='utf-8').read()
    if new in s:
        print(f'   · {tag}: уже применено, пропуск')
        return
    assert old in s, f'ЯКОРЬ НЕ НАЙДЕН [{tag}] в {path} — порядок ячеек нарушен?'
    open(path, 'w', encoding='utf-8').write(s.replace(old, new, 1))
    print(f'OK  {tag}')

def write(path, content, tag):
    if os.path.exists(path) and open(path, encoding='utf-8').read() == content:
        print(f'   · {tag}: без изменений, пропуск')
        return
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    open(path, 'w', encoding='utf-8').write(content)
    print(f'OK  {tag} ({len(content)} B)')

# ── Модели: TuningMap + AnalysisResult (механика V6 1:1, без Nissan-специфики) ──
TUNING_MAP = r'''import 'dart:convert';

class TuningMap {
  final String name;
  final String address;
  final int    rows;
  final int    cols;
  final List<double> rpmAxis;
  final List<double> loadAxis;
  List<List<double>> data;
  final String units;
  final double minValue;
  final double maxValue;

  TuningMap({
    required this.name,
    required this.address,
    required this.rows,
    required this.cols,
    required this.rpmAxis,
    required this.loadAxis,
    required this.data,
    required this.units,
    this.minValue = -100,
    this.maxValue =  400,
  });

  double getValue(double rpm, double load) {
    final ri = _closest(rpmAxis,  rpm);
    final li = _closest(loadAxis, load);
    return data[ri][li];
  }

  void setValue(double rpm, double load, double value) {
    final ri = _closest(rpmAxis,  rpm);
    final li = _closest(loadAxis, load);
    data[ri][li] = value;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  TuningMap copy() => TuningMap(
    name: name, address: address, rows: rows, cols: cols,
    rpmAxis:  List<double>.from(rpmAxis),
    loadAxis: List<double>.from(loadAxis),
    data: data.map((r) => List<double>.from(r)).toList(),
    units: units, minValue: minValue, maxValue: maxValue,
  );

  double get avgValue {
    double sum = 0; int n = 0;
    for (final row in data) { for (final v in row) { sum += v; n++; } }
    return n > 0 ? sum / n : 0;
  }

  Map<String, dynamic> toJson() => {
    'name': name, 'address': address, 'rows': rows, 'cols': cols,
    'rpmAxis': rpmAxis, 'loadAxis': loadAxis, 'data': data,
    'units': units, 'minValue': minValue, 'maxValue': maxValue,
  };

  factory TuningMap.fromJson(Map<String, dynamic> j) {
    final rawData = j['data'] as List;
    final data = rawData.map<List<double>>(
      (row) => (row as List).map<double>((v) => (v as num).toDouble()).toList()
    ).toList();
    return TuningMap(
      name:     j['name']    as String,
      address:  j['address'] as String,
      rows:     j['rows']    as int,
      cols:     j['cols']    as int,
      rpmAxis:  (j['rpmAxis']  as List).map<double>((v) => (v as num).toDouble()).toList(),
      loadAxis: (j['loadAxis'] as List).map<double>((v) => (v as num).toDouble()).toList(),
      data:     data,
      units:    j['units']    as String,
      minValue: (j['minValue'] as num).toDouble(),
      maxValue: (j['maxValue'] as num).toDouble(),
    );
  }
}
'''

ANALYSIS_RESULT = r'''class MapCell {
  final int    rpmIndex;
  final int    loadIndex;
  final double rpm;
  final double load;
  final double currentValue;
  final double suggestedValue;
  final double confidence;
  final int    sampleCount;
  final String reason;

  const MapCell({
    required this.rpmIndex,
    required this.loadIndex,
    required this.rpm,
    required this.load,
    required this.currentValue,
    required this.suggestedValue,
    required this.confidence,
    required this.sampleCount,
    required this.reason,
  });

  double get delta        => suggestedValue - currentValue;
  double get deltaPercent =>
      currentValue != 0 ? (delta / currentValue.abs()) * 100 : 0;
}

class AnalysisResult {
  final String       mapName;
  final DateTime     analyzedAt;
  final int          totalSamples;
  final List<MapCell> changes;
  final String       summary;
  final String       patternName;

  const AnalysisResult({
    required this.mapName,
    required this.analyzedAt,
    required this.totalSamples,
    required this.changes,
    required this.summary,
    this.patternName = '',
  });
}
'''

write('lib/models/tuning_map.dart', TUNING_MAP, 'models/tuning_map.dart (как V6)')
write('lib/models/analysis_result.dart', ANALYSIS_RESULT, 'models/analysis_result.dart (как V6)')

# ── Дефолты + история правок (порт V6, свои ключи suba_) ──
MAP_STORAGE = r'''import 'dart:convert';
import 'package:shared_preferences/shared_preferences.dart';
import '../models/tuning_map.dart';

/// Дефолтные карты анализатора Subaru: пользовательские правки поверх базы.
/// Механика 1:1 из Nissan V6, отдельный префикс ключей.
class SubaMapStorageService {
  static const _prefix  = 'suba_map_edit_';
  static const _listKey = 'suba_map_edit_list';
  static SharedPreferences? _p;

  static Future<void> _init() async {
    _p ??= await SharedPreferences.getInstance();
  }

  static Future<void> saveMap(TuningMap map) async {
    await _init();
    final key  = _prefix + map.address;
    final json = jsonEncode({
      'name': map.name, 'address': map.address,
      'rows': map.rows, 'cols': map.cols,
      'data': map.data,
      'updatedAt': DateTime.now().toIso8601String(),
    });
    await _p!.setString(key, json);
    final list = _p!.getStringList(_listKey) ?? [];
    if (!list.contains(map.address)) {
      list.add(map.address);
      await _p!.setStringList(_listKey, list);
    }
  }

  static Future<List<List<double>>?> loadMapData(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return null;
    try {
      final j   = jsonDecode(s) as Map<String, dynamic>;
      final raw = j['data'] as List;
      return raw.map<List<double>>(
        (row) => (row as List).map<double>((v) => (v as num).toDouble()).toList(),
      ).toList();
    } catch (_) { return null; }
  }

  static Future<bool> hasEdits(String address) async {
    await _init();
    return _p!.containsKey(_prefix + address);
  }

  static Future<DateTime?> getUpdatedAt(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return null;
    try {
      final j = jsonDecode(s) as Map<String, dynamic>;
      return DateTime.tryParse(j['updatedAt'] as String? ?? '');
    } catch (_) { return null; }
  }

  static Future<void> resetMap(String address) async {
    await _init();
    await _p!.remove(_prefix + address);
    final list = _p!.getStringList(_listKey) ?? [];
    list.remove(address);
    await _p!.setStringList(_listKey, list);
  }

  static Future<void> resetAll() async {
    await _init();
    final list = _p!.getStringList(_listKey) ?? [];
    for (final a in list) await _p!.remove(_prefix + a);
    await _p!.remove(_listKey);
  }
}
'''

MAP_HISTORY = r'''import 'dart:convert';
import 'package:shared_preferences/shared_preferences.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';

class SubaMapRevision {
  final String   address;
  final String   name;
  final int      version;
  final DateTime savedAt;
  final List<List<double>> data;
  final String   comment;

  const SubaMapRevision({
    required this.address,
    required this.name,
    required this.version,
    required this.savedAt,
    required this.data,
    this.comment = '',
  });

  String get label =>
      'v$version • ${DateFormat("dd.MM HH:mm").format(savedAt)}'
      '${comment.isNotEmpty ? " — $comment" : ""}';

  Map<String, dynamic> toJson() => {
    'address': address, 'name': name, 'version': version,
    'savedAt': savedAt.toIso8601String(),
    'data': data, 'comment': comment,
  };

  factory SubaMapRevision.fromJson(Map<String, dynamic> j) {
    final raw = j['data'] as List;
    return SubaMapRevision(
      address: j['address'] as String,
      name:    j['name']    as String,
      version: j['version'] as int,
      savedAt: DateTime.parse(j['savedAt'] as String),
      data: raw.map<List<double>>(
        (r) => (r as List).map<double>((v) => (v as num).toDouble()).toList(),
      ).toList(),
      comment: j['comment'] as String? ?? '',
    );
  }
}

/// История ревизий карт (до 20 на карту). Порт V6 1:1.
class SubaMapHistoryService {
  static const _prefix    = 'suba_map_hist_';
  static const _maxRevs   = 20;
  static SharedPreferences? _p;

  static Future<void> _init() async {
    _p ??= await SharedPreferences.getInstance();
  }

  static Future<List<SubaMapRevision>> getHistory(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return [];
    try {
      final list = jsonDecode(s) as List;
      return list
          .map((j) => SubaMapRevision.fromJson(j as Map<String, dynamic>))
          .toList();
    } catch (_) { return []; }
  }

  static Future<int> saveRevision(TuningMap map, {String comment = ''}) async {
    await _init();
    final hist    = await getHistory(map.address);
    final version = hist.isEmpty ? 1 : hist.last.version + 1;
    final rev     = SubaMapRevision(
      address: map.address, name: map.name,
      version: version,     savedAt: DateTime.now(),
      data:    map.data.map((r) => List<double>.from(r)).toList(),
      comment: comment.isEmpty ? 'Правка $version' : comment,
    );
    hist.add(rev);
    if (hist.length > _maxRevs) hist.removeRange(0, hist.length - _maxRevs);
    await _p!.setString(
      _prefix + map.address,
      jsonEncode(hist.map((r) => r.toJson()).toList()),
    );
    return version;
  }

  static Future<void> clearHistory(String address) async {
    await _init();
    await _p!.remove(_prefix + address);
  }
}
'''

write('lib/services/suba_map_storage_service.dart', MAP_STORAGE, 'services/suba_map_storage_service.dart')
write('lib/services/suba_map_history_service.dart', MAP_HISTORY, 'services/suba_map_history_service.dart')

# ── Дефолтные карты Subaru EJ20X (стартовые шаблоны + правки поверх) ──
TUNING = r'''import '../models/tuning_map.dart';
import 'suba_map_storage_service.dart';

/// Базовые карты анализатора Subaru.
/// ВНИМАНИЕ: встроенные дефолты — синтетические стартовые шаблоны EJ20X
/// (правдоподобная форма, НЕ заводская прошивка). Основной поток:
/// загрузить ROM/ECU → «В дефолт» → анализ → «Сохранить» → «В прошивку».
class SubaTuningService {
  static const rpm16 = [600.0, 900, 1200, 1600, 2000, 2500, 3000, 3500,
                        4000, 4500, 5000, 5500, 6000, 6500, 7000, 7500];
  static const load16 = [0.0, 7, 14, 20, 27, 34, 40, 47, 54, 60, 67, 74, 80, 87, 93, 100];
  static const rpm8 = [800.0, 1600, 2400, 3200, 4000, 4800, 5600, 6400];
  static const load8 = [0.0, 15, 30, 45, 60, 75, 90, 100];

  static const addrTiming = 'SUBA_TIMING';
  static const addrFuel   = 'SUBA_FUEL';
  static const addrAvcs   = 'SUBA_AVCS';
  static const addrBoost  = 'SUBA_BOOST';

  Future<TuningMap> getTimingMap() async => _apply(_defaultTiming());
  Future<TuningMap> getFuelMap()   async => _apply(_defaultFuel());
  Future<TuningMap> getAvcsMap()   async => _apply(_defaultAvcs());
  Future<TuningMap> getBoostMap()  async => _apply(_defaultBoost());

  Future<TuningMap> _apply(TuningMap def) async {
    final saved = await SubaMapStorageService.loadMapData(def.address);
    if (saved == null) return def;
    if (saved.length != def.rows) return def;
    if (saved.isNotEmpty && saved[0].length != def.cols) return def;
    return TuningMap(
      name: def.name, address: def.address,
      rows: def.rows, cols: def.cols,
      rpmAxis: def.rpmAxis, loadAxis: def.loadAxis,
      data: saved, units: def.units,
      minValue: def.minValue, maxValue: def.maxValue,
    );
  }

  /// Зажигание: много на малой нагрузке, меньше под бустом и на верхах.
  TuningMap _defaultTiming() {
    final data = List.generate(16, (r) {
      final rpm = rpm16[r];
      return List.generate(16, (c) {
        final load = load16[c];
        var v = 34.0 - load * 0.22 - (rpm > 3500 ? (rpm - 3500) * 0.0022 : 0.0);
        return double.parse(v.clamp(-5.0, 40.0).toStringAsFixed(1));
      });
    });
    return TuningMap(
      name: 'Зажигание (Subaru)', address: addrTiming,
      rows: 16, cols: 16,
      rpmAxis: List<double>.from(rpm16), loadAxis: List<double>.from(load16),
      data: data, units: 'deg', minValue: -5, maxValue: 40,
    );
  }

  /// Топливо: целевая AFR — 14.7 на круизе, богаче к полной нагрузке.
  TuningMap _defaultFuel() {
    final data = List.generate(16, (r) {
      return List.generate(16, (c) {
        final load = load16[c];
        var v = 14.7 - (load > 30 ? (load - 30) * 0.045 : 0.0);
        return double.parse(v.clamp(10.5, 15.2).toStringAsFixed(2));
      });
    });
    return TuningMap(
      name: 'Топливо AFR цель (Subaru)', address: addrFuel,
      rows: 16, cols: 16,
      rpmAxis: List<double>.from(rpm16), loadAxis: List<double>.from(load16),
      data: data, units: 'AFR', minValue: 10.0, maxValue: 16.0,
    );
  }

  /// AVCS впуск: пик в середине диапазона, 0 на холостых и отсечке.
  TuningMap _defaultAvcs() {
    final data = List.generate(8, (r) {
      final rpm = rpm8[r];
      return List.generate(8, (c) {
        final load = load8[c];
        double v;
        if (load < 10 || rpm < 1200) {
          v = 0;
        } else {
          final rpmF = 1.0 - ((rpm - 3500).abs() / 3500.0).clamp(0.0, 1.0);
          v = 28.0 * rpmF * (load / 60.0).clamp(0.0, 1.0);
        }
        return double.parse(v.clamp(0.0, 35.0).toStringAsFixed(1));
      });
    });
    return TuningMap(
      name: 'AVCS впуск (Subaru)', address: addrAvcs,
      rows: 8, cols: 8,
      rpmAxis: List<double>.from(rpm8), loadAxis: List<double>.from(load8),
      data: data, units: 'deg', minValue: 0, maxValue: 35,
    );
  }

  /// Target Boost: 0 вне наддува, рампа к ~1.0 бар на полной нагрузке.
  TuningMap _defaultBoost() {
    final data = List.generate(16, (r) {
      final rpm = rpm16[r];
      return List.generate(16, (c) {
        final load = load16[c];
        final lf = ((load - 35) / 65).clamp(0.0, 1.0);
        final rf = ((rpm - 1800) / 1800).clamp(0.0, 1.0);
        final v = (lf * rf * 1.15).clamp(0.0, 1.2);
        return double.parse(v.toStringAsFixed(2));
      });
    });
    return TuningMap(
      name: 'Target Boost (Subaru)', address: addrBoost,
      rows: 16, cols: 16,
      rpmAxis: List<double>.from(rpm16), loadAxis: List<double>.from(load16),
      data: data, units: 'bar', minValue: 0, maxValue: 1.2,
    );
  }
}
'''

write('lib/services/suba_tuning_service.dart', TUNING, 'services/suba_tuning_service.dart (Subaru-дефолты)')

# ── Анализатор Subaru (структура V6, сигналы EJ20X) ──
ANALYZER = r'''import 'dart:io';
import 'dart:math';
import 'package:csv/csv.dart';
import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/analysis_result.dart';
import '../models/tuning_map.dart';

// ═══════════════════════════════════════════════════════════════
// Паттерны настройки Subaru (структура V6, пороги EJ20X)
// ═══════════════════════════════════════════════════════════════
enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTolerance;    // детонация FBKC+FKL, °
  final double afrLean, afrRich;
  final double timingAggression;  // 0..1
  final double fuelTrimThreshold; // суммарный трим, %
  final double avcsAggression;    // 0..1
  final double boostAggression;   // 0..1

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 14.7,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.avcsAggression    = 0.5,
    this.boostAggression   = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Мощность (турбо)',
    description: 'Агрессивный УОЗ, богатая смесь под бустом',
    knockTolerance: 0.5, afrLean: 13.0, afrRich: 11.2,
    timingAggression: 0.8, fuelTrimThreshold: 2.0,
    avcsAggression: 0.8, boostAggression: 0.7,
  );

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе, умеренный УОЗ',
    knockTolerance: 1.0, afrLean: 15.5, afrRich: 13.8,
    timingAggression: 0.3, fuelTrimThreshold: 5.0,
    avcsAggression: 0.3, boostAggression: 0.2,
  );

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативно, минимум детонации',
    knockTolerance: 1.5, afrLean: 14.7, afrRich: 12.5,
    timingAggression: 0.15, fuelTrimThreshold: 3.0,
    avcsAggression: 0.2, boostAggression: 0.2,
  );

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

// ═══════════════════════════════════════════════════════════════
// Сэмпл лога Subaru (канонические ключи V8)
// Нагрузка %: TPS (основа) → фолбэк буст/MAF. Детонация: |FBKC+FKL|.
// ═══════════════════════════════════════════════════════════════
class SubaSample {
  final DateTime ts;
  final double rpm, load;
  final double boost, tboost, kca, fbkc, fkl, iam;
  final double afr, stft, ltft, avcs, tps, maf, wgd, ect;

  const SubaSample({
    required this.ts,
    this.rpm = double.nan, this.load = double.nan,
    this.boost = double.nan, this.tboost = double.nan,
    this.kca = double.nan, this.fbkc = double.nan, this.fkl = double.nan,
    this.iam = double.nan, this.afr = double.nan,
    this.stft = double.nan, this.ltft = double.nan,
    this.avcs = double.nan, this.tps = double.nan,
    this.maf = double.nan, this.wgd = double.nan, this.ect = double.nan,
  });

  bool get hasKnock => fbkc.isFinite || fkl.isFinite;
  double get knockMag {
    final a = fbkc.isFinite ? min(0.0, fbkc) : 0.0;
    final b = fkl.isFinite  ? min(0.0, fkl)  : 0.0;
    return -(a + b);
  }
  bool get underBoost => boost.isFinite && boost >= AppConstants.boostActive;

  factory SubaSample.fromCanons(Map<String, double> c, [DateTime? ts]) {
    double g(String k) => c[k] ?? double.nan;
    return SubaSample(
      ts: ts ?? DateTime.now(),
      rpm: g('rpm'), load: _loadPct(c),
      boost: g('boost'), tboost: g('tboost'),
      kca: g('kca'), fbkc: g('fbkc'), fkl: g('fkl'), iam: g('iam'),
      afr: g('afr'), stft: g('stft'), ltft: g('ltft'),
      avcs: g('avcs'), tps: g('tps'), maf: g('maf'),
      wgd: g('wgd'), ect: g('ect'),
    );
  }

  static double _loadPct(Map<String, double> c) {
    final tps = c['tps'];
    if (tps != null && tps.isFinite) return tps.clamp(0.0, 100.0);
    final b = c['boost'];
    if (b != null && b.isFinite) {
      return ((b + 0.65) / (1.5 + 0.65) * 100.0).clamp(0.0, 100.0);
    }
    final maf = c['maf'];
    if (maf != null && maf.isFinite) {
      return (maf / 200.0 * 100.0).clamp(0.0, 100.0);
    }
    return double.nan;
  }
}

// ═══════════════════════════════════════════════════════════════
// Сервис анализа (логика ветвлений V6, сигналы Subaru)
// ═══════════════════════════════════════════════════════════════
class SubaAnalyzerService {
  static const _minSamples = 2;
  static const _minConf    = 0.4;

  // ── Зажигание: KCA = факт ЭБУ, детонация = |FBKC+FKL|, гейт IAM ──
  Future<AnalysisResult> analyzeTimingMap(
    List<SubaSample> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm.isFinite && d.rpm > 600 && d.rpm < 7500 &&
      d.load.isFinite && d.load >= 0).toList();
    if (valid.length < 10) {
      return _empty('Зажигание', log.length, 'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgKca   = _avgFin(samples.map((d) => d.kca));
      final knockVals = samples.where((d) => d.hasKnock).map((d) => d.knockMag);
      final avgKnock = _avgFin(knockVals);
      final avgAfr   = _avgFin(samples.map((d) => d.afr));
      final avgIam   = _avgFin(samples.map((d) => d.iam));
      final iamLow   = avgIam != null && avgIam < AppConstants.iamDanger;

      double sug = cur;
      double conf = 0;
      String why = '';

      if (avgKnock != null && avgKnock > pattern.knockTolerance * 2) {
        sug = cur - min(avgKnock, 3.0);
        why = 'Детонация FBKC/FKL ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (iamLow && avgKnock != null && avgKnock > pattern.knockTolerance) {
        sug = cur - 1.5;
        why = 'IAM ${avgIam!.toStringAsFixed(2)} — ЭБУ режет углы';
        conf = 0.85;
      } else if (avgKnock != null && avgKnock > pattern.knockTolerance) {
        sug = cur - (1.0 * (1.0 - pattern.timingAggression));
        why = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKca != null && avgKca != 0 && (avgKca - cur).abs() > 2) {
        sug = avgKca + (pattern.timingAggression * 2 - 1);
        why = 'ЭБУ KCA: ${avgKca.toStringAsFixed(1)}°';
        conf = 0.6;
      } else if (avgKnock != null && avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAfr != null && avgAfr > pattern.afrRich &&
                 avgAfr < pattern.afrLean && !iamLow) {
        sug = cur + pattern.timingAggression * 2;
        why = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стаб.)';
        conf = 0.5;
      }

      sug = sug.clamp(map.minValue, map.maxValue);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Зажигание',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Зажигание', changes, valid.length, grouped.length),
    );
  }

  // ── Топливо: суммарный трим STFT+LTFT + AFR, защита от бедной под бустом ──
  Future<AnalysisResult> analyzeFuelMap(
    List<SubaSample> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm.isFinite && d.rpm > 600 && d.rpm < 7500 &&
      d.load.isFinite && d.load >= 0 &&
      (!d.ltft.isFinite || d.ltft.abs() < 30)).toList();
    if (valid.length < 10) {
      return _empty('Топливо', log.length, 'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgSTFT = _avgFin(samples.map((d) => d.stft)) ?? 0.0;
      final avgLTFT = _avgFin(samples.map((d) => d.ltft)) ?? 0.0;
      final hasTrim = samples.any((d) => d.stft.isFinite || d.ltft.isFinite);
      final total   = avgSTFT + avgLTFT;
      final avgAFR  = _avgFin(samples.map((d) => d.afr));
      final avgLoad = _avgFin(samples.map((d) => d.load)) ?? 0.0;
      final avgBoost = _avgFin(samples.map((d) => d.boost)) ?? 0.0;
      final onBoost = avgBoost >= AppConstants.boostActive;

      double sug = cur;
      double conf = 0;
      String why = '';

      if (onBoost && avgAFR != null && avgAFR >= AppConstants.afrBoostLean) {
        // Безопасность первична: бедно под бустом — обогатить.
        sug = cur * 0.94;
        why = 'ОПАСНО бедно под бустом AFR ${avgAFR.toStringAsFixed(1)}';
        conf = 0.85;
      } else if (hasTrim && total > pattern.fuelTrimThreshold) {
        sug = cur * (1 - total / 100.0 * 0.5);
        why = 'Trim +${total.toStringAsFixed(1)}% (бедно) → богаче цель';
        conf = min(0.9, total.abs() / 10);
      } else if (hasTrim && total < -pattern.fuelTrimThreshold) {
        sug = cur * (1 - total / 100.0 * 0.5);
        why = 'Trim ${total.toStringAsFixed(1)}% (богато) → беднее цель';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR != null && avgAFR > pattern.afrLean && avgLoad > 50) {
        sug = cur * 0.98;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} бедно';
        conf = 0.6;
      } else if (avgAFR != null && avgAFR < pattern.afrRich && avgLoad > 50) {
        sug = cur * 1.02;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} богато';
        conf = 0.6;
      }

      sug = sug.clamp(map.minValue, map.maxValue);
      final maxChange = cur.abs() * 0.08;
      sug = cur + (sug - cur).clamp(-maxChange, maxChange);

      if (cur.abs() > 0.001 &&
          (sug - cur).abs() / cur.abs() >= 0.005 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Топливо',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Топливо', changes, valid.length, grouped.length),
    );
  }

  // ── AVCS: факт ЭБУ vs карта + гейт детонации ──
  Future<AnalysisResult> analyzeAvcsMap(
    List<SubaSample> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm.isFinite && d.rpm > 1000 && d.rpm < 7000 &&
      d.load.isFinite && d.load > 5 &&
      d.avcs.isFinite).toList();
    if (valid.length < 10) {
      return _empty('AVCS', log.length, 'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgAct = _avgFin(samples.map((d) => d.avcs)) ?? cur;
      final avgKnock = _avgFin(
        samples.where((d) => d.hasKnock).map((d) => d.knockMag));

      double sug = cur;
      double conf = 0;
      String why = '';

      if ((avgAct - cur).abs() > 3) {
        sug = avgAct + (pattern.avcsAggression * 2 - 1);
        why = 'ЭБУ AVCS: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock != null && avgKnock > pattern.knockTolerance && cur > 10) {
        sug = max(map.minValue, cur - 5);
        why = 'Детонация — снизить AVCS';
        conf = 0.75;
      }

      sug = sug.clamp(map.minValue, map.maxValue);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'AVCS',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('AVCS', changes, valid.length, grouped.length),
    );
  }

  // ── Target Boost: ошибка регулирования + гейт детонации (вместо Torque V6) ──
  Future<AnalysisResult> analyzeBoostMap(
    List<SubaSample> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm.isFinite && d.rpm > 1500 && d.rpm < 7500 &&
      d.load.isFinite && d.load > 30 &&
      d.boost.isFinite && d.tboost.isFinite).toList();
    if (valid.length < 10) {
      return _empty('Target Boost', log.length,
        'Мало данных: ${valid.length} (нужны boost+tboost)');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgErr = _avgFin(
        samples.map((d) => d.boost - d.tboost)) ?? 0.0;
      final avgKnock = _avgFin(
        samples.where((d) => d.hasKnock).map((d) => d.knockMag));

      double sug = cur;
      double conf = 0;
      String why = '';

      if (avgErr > 0.12) {
        sug = cur - min(0.2, avgErr * 0.5);
        why = 'Перебуст +${avgErr.toStringAsFixed(2)} бар';
        conf = 0.7;
      } else if (avgErr < -0.12) {
        if (avgKnock == null || avgKnock < pattern.knockTolerance) {
          sug = cur + min(0.15, -avgErr * (0.2 + pattern.boostAggression * 0.4));
          why = 'Недобуст ${avgErr.toStringAsFixed(2)} бар';
          conf = 0.55;
        }
      }
      if (avgKnock != null && avgKnock > pattern.knockTolerance * 2 && cur > 0.3) {
        sug = cur - 0.1;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить цель';
        conf = 0.7;
      }

      sug = sug.clamp(map.minValue, map.maxValue);
      final maxStep = 0.2;
      sug = cur + (sug - cur).clamp(-maxStep, maxStep);

      if ((sug - cur).abs() >= 0.03 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Target Boost',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Boost', changes, valid.length, grouped.length),
    );
  }

  // ── CSV логов V8 (ts_ms + ID пидов; разделитель , или ;) ──
  static const _canonKeys = {
    'rpm', 'speed', 'ect', 'iat', 'boost', 'tps', 'kca', 'fbkc', 'fkl',
    'iam', 'afr', 'maf', 'mafV', 'tboost', 'wgd', 'injms', 'stft', 'ltft',
    'avcs', 'knocksum', 'batt',
  };

  Future<List<SubaSample>> loadLogFromCSV(String path) async {
    final content = await File(path).readAsString();
    final firstLine = content.split('\n')
        .firstWhere((l) => l.trim().isNotEmpty, orElse: () => '');
    if (firstLine.isEmpty) return [];
    final delim = firstLine.contains(';') ? ';' : ',';
    final rows = CsvToListConverter(
      fieldDelimiter: delim, shouldParseNumbers: false).convert(content);
    if (rows.length < 2) return [];

    final head = rows.first.map((e) => e.toString().trim()).toList();
    final canonIdx = <int, String>{};
    var tsIdx = -1;
    for (var i = 0; i < head.length; i++) {
      final h = head[i];
      if (h == 'ts_ms' || h == 'Timestamp_ms') { tsIdx = i; continue; }
      final pid = SubaruPids.byId(h);
      if (pid != null && pid.canon.isNotEmpty) {
        canonIdx[i] = pid.canon;
      } else if (_canonKeys.contains(h)) {
        canonIdx[i] = h;
      }
    }

    final result = <SubaSample>[];
    for (var r = 1; r < rows.length; r++) {
      try {
        final row = rows[r];
        var ts = DateTime.now();
        if (tsIdx >= 0 && tsIdx < row.length) {
          final ms = int.tryParse(row[tsIdx].toString().trim()) ?? 0;
          if (ms == 0) continue;
          ts = DateTime.fromMillisecondsSinceEpoch(ms);
        }
        final m = <String, double>{};
        canonIdx.forEach((i, key) {
          if (i < row.length) {
            final v = double.tryParse(row[i].toString().trim());
            if (v != null && v.isFinite) m[key] = v;
          }
        });
        if (m['rpm'] == null || !m['rpm']!.isFinite) continue;
        result.add(SubaSample.fromCanons(m, ts));
      } catch (_) {
        continue;
      }
    }
    return result;
  }

  // ── Сплиттер: слияние логов, дедуп ±200 мс (как V6) ──
  Future<List<SubaSample>> mergeLogs(List<List<SubaSample>> logs) async {
    final all = <SubaSample>[];
    for (final log in logs) all.addAll(log);
    all.sort((a, b) => a.ts.compareTo(b.ts));
    final deduped = <SubaSample>[];
    for (final d in all) {
      if (deduped.isEmpty) { deduped.add(d); continue; }
      final diff = d.ts.difference(deduped.last.ts).inMilliseconds.abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  // ── Вспомогательное (структура V6) ──
  Map<String, List<SubaSample>> _group(List<SubaSample> data, TuningMap map) {
    final result = <String, List<SubaSample>>{};
    for (final d in data) {
      if (!d.rpm.isFinite || !d.load.isFinite) continue;
      final ri = _closest(map.rpmAxis,  d.rpm);
      final li = _closest(map.loadAxis, d.load);
      result.putIfAbsent('$ri,$li', () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double? _avgFin(Iterable<double> vals) {
    final f = vals.where((v) => v.isFinite).toList();
    if (f.isEmpty) return null;
    return f.reduce((a, b) => a + b) / f.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(
        mapName: name,
        analyzedAt: DateTime.now(),
        totalSamples: total,
        changes: const [],
        summary: msg,
      );

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) {
      return '$type оптимальна\nДанных: $total | Клеток: $cells';
    }
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) /
        ch.length;
    final maxD =
        ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2
        ? '↑ увеличение'
        : dec > inc * 2
            ? '↓ уменьшение'
            : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
'''

write('lib/services/suba_analyzer_service.dart', ANALYZER, 'services/suba_analyzer_service.dart (Subaru)')

# ── Мост ROM↔анализатор + запись MOD + экспорт (Subaru) ──
BRIDGE = r'''import 'dart:io';
import 'dart:typed_data';
import 'package:path_provider/path_provider.dart';
import '../models/rom_table.dart';
import '../models/tuning_map.dart';
import 'rom_service.dart';

/// Связка карт прошивки A2TB100B с анализатором:
/// RomTable → TuningMap («Из ROM», «В дефолт») и запись правок в V8MOD_*.bin.
/// В отличие от однотабличного saveMod: пакетная запись, все типы хранения
/// (float/uint32/int32/uint16/int16/uint8/int8), endianness и проверка границ.
class SubaRomBridge {
  static TuningMap toTuningMap(RomTable t) {
    final d = t.def;
    var lo = d.minHint, hi = d.maxHint;
    if (lo.isNaN || hi.isNaN) {
      lo = double.infinity; hi = -double.infinity;
      for (final row in t.z) {
        for (final v in row) {
          if (!v.isFinite) continue;
          if (v < lo) lo = v;
          if (v > hi) hi = v;
        }
      }
      if (lo == double.infinity) { lo = 0; hi = 1; }
      if (hi <= lo) hi = lo + 1;
    }
    return TuningMap(
      name: d.name, address: d.addrHex,
      rows: d.rows, cols: d.cols,
      rpmAxis: List<double>.from(t.yValues),
      loadAxis: List<double>.from(t.xValues),
      data: t.z.map((r) => List<double>.from(r)).toList(),
      units: d.units, minValue: lo, maxValue: hi,
    );
  }

  static void applyToRomTable(RomTable t, TuningMap m) {
    final d = t.def;
    if (m.rows != d.rows || m.cols != d.cols) {
      throw StateError(
        '${d.name}: размер ${m.rows}x${m.cols} != ${d.rows}x${d.cols}');
    }
    for (var r = 0; r < d.rows; r++) {
      for (var c = 0; c < d.cols; c++) {
        final v = m.data[r][c];
        if (!v.isFinite) {
          throw StateError('${d.name} [$r,$c]: нечисловое значение');
        }
        t.z[r][c] = v;
      }
    }
  }

  static Future<String?> writeTableToMod(
    RomService rom, RomTable table, {String prefix = 'V8MOD'}) =>
      writeTablesToMod(rom, [table], prefix: prefix);

  static Future<String?> writeTablesToMod(
    RomService rom, List<RomTable> tables, {String prefix = 'V8MOD'}) async {
    if (!rom.loaded || rom.rom == null) {
      throw StateError('ROM не загружен');
    }
    if (tables.isEmpty) throw StateError('Нет карт для записи');
    final buf = Uint8List.fromList(rom.rom!);
    final bd = ByteData.sublistView(buf);

    for (final t in tables) {
      final d = t.def;
      final fr = d.data.fr;
      if (fr == null) {
        throw StateError('${d.name}: read-only (нет frexpr)');
      }
      if (t.z.length != d.rows ||
          t.z.any((r) => r.length != d.cols)) {
        throw StateError('${d.name}: сетка не совпадает с дефинишном');
      }
      final size = d.data.sizeOf;
      final endian =
          d.data.endian == 'little' ? Endian.little : Endian.big;
      for (var r = 0; r < d.rows; r++) {
        for (var c = 0; c < d.cols; c++) {
          final i = d.swapxy ? (c * d.rows + r) : (r * d.cols + c);
          final off = d.address + i * size;
          if (off < 0 || off + size > buf.length) {
            throw StateError(
              '${d.name} [$r,$c]: адрес 0x${off.toRadixString(16)} вне ROM');
          }
          final phys = t.z[r][c];
          if (!phys.isFinite) {
            throw StateError('${d.name} [$r,$c]: нечисловое значение');
          }
          final raw = fr(phys);
          if (!raw.isFinite) {
            throw StateError(
              '${d.name} [$r,$c]: frexpr дала $raw для $phys');
          }
          switch (d.data.storage) {
            case 'float':
              bd.setFloat32(off, raw, endian);
              break;
            case 'uint32':
              bd.setUint32(off, raw.round().clamp(0, 0xFFFFFFFF), endian);
              break;
            case 'int32':
              bd.setInt32(off, raw.round().clamp(-0x80000000, 0x7FFFFFFF), endian);
              break;
            case 'uint16':
              bd.setUint16(off, raw.round().clamp(0, 0xFFFF), endian);
              break;
            case 'int16':
              bd.setInt16(off, raw.round().clamp(-0x8000, 0x7FFF), endian);
              break;
            case 'int8':
              buf[off] = raw.round().clamp(-128, 127) & 0xFF;
              break;
            default:
              buf[off] = raw.round().clamp(0, 255);
          }
        }
      }
    }

    final dir = await getApplicationDocumentsDirectory();
    final base = rom.fileName.replaceAll(
      RegExp(r'\.(bin|hex|rom)$', caseSensitive: false), '');
    final name = '${prefix}_${base.isEmpty ? 'rom' : base}.bin';
    final path = '${dir.path}/$name';
    await File(path).writeAsBytes(buf);
    // Копия в видимую папку (best-effort, как остальные экспорты V8).
    try {
      const dl = '/storage/emulated/0/Download';
      if (await Directory(dl).exists()) {
        await File('$dl/$name').writeAsBytes(buf);
      }
    } catch (_) {}
    return path;
  }
}
'''

EXPORT = r'''import 'dart:convert';
import 'dart:io';
import 'package:path_provider/path_provider.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

/// Экспорт карт анализатора Subaru (порты форматов V6).
class SubaExportService {
  String _ts() => DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
  String _safe(String n) => n.replaceAll(RegExp(r'[^a-zA-Z0-9_]'), '_');

  Future<String> _save(String name, String text) async {
    final dir  = await getApplicationDocumentsDirectory();
    final path = '${dir.path}/$name';
    await File(path).writeAsString(text);
    try {
      const dl = '/storage/emulated/0/Download';
      if (await Directory(dl).exists()) {
        await File('$dl/$name').writeAsString(text);
      }
    } catch (_) {}
    return path;
  }

  Future<String> exportToWinOLS(TuningMap map) async {
    final sb = StringBuffer();
    sb.writeln('# WinOLS export — SUBA RUN V8');
    sb.writeln('MAP_NAME=${map.name}');
    sb.writeln('MAP_ADDRESS=${map.address}');
    sb.writeln('X_AXIS=${map.loadAxis.join(",")}');
    sb.writeln('Y_AXIS=${map.rpmAxis.join(",")}');
    for (int i = 0; i < map.data.length; i++) {
      sb.writeln('ROW_$i=${map.data[i].map((v) => v.toStringAsFixed(2)).join(",")}');
    }
    return _save('${_safe(map.name)}_${_ts()}.ols', sb.toString());
  }

  Future<String> exportToJson(
      AnalysisResult result, TuningMap orig, TuningMap upd) async {
    final data = {
      'meta': {
        'app':      'SUBA RUN V8',
        'exported': DateTime.now().toIso8601String(),
        'pattern':  result.patternName,
      },
      'original': orig.toJson(),
      'updated':  upd.toJson(),
      'changes':  result.changes.map((c) => {
        'rpm': c.rpm, 'load': c.load,
        'from': c.currentValue, 'to': c.suggestedValue,
        'delta': c.delta, 'reason': c.reason,
      }).toList(),
    };
    return _save('suba_tuning_${_ts()}.json',
        const JsonEncoder.withIndent('  ').convert(data));
  }

  Future<String> exportHexPatch(TuningMap map, {int bytesPerCell = 2}) async {
    final base = int.tryParse(
        map.address.replaceAll('0x','').replaceAll('0X',''), radix: 16) ?? 0;
    final sb = StringBuffer();
    sb.writeln('; Hex patch — ${map.name}');
    sb.writeln('; Base: ${map.address}');
    for (int i = 0; i < map.data.length; i++) {
      for (int j = 0; j < map.data[i].length; j++) {
        final addr = base + (i * map.cols + j) * bytesPerCell;
        sb.writeln(
          '${addr.toRadixString(16).padLeft(8,"0").toUpperCase()}: '
          '${map.data[i][j].toStringAsFixed(2)}',
        );
      }
    }
    return _save('${_safe(map.name)}_${_ts()}.hex', sb.toString());
  }
}
'''

write('lib/services/suba_rom_bridge.dart', BRIDGE, 'services/suba_rom_bridge.dart (ROM↔анализ + MOD)')
write('lib/services/suba_export_service.dart', EXPORT, 'services/suba_export_service.dart (WinOLS/JSON/HEX)')

# ── Виджет карты V6: градиент, правки, Сохранить/Сброс, полноэкранный ──
MAPVIEW = r'''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/suba_map_storage_service.dart';
import '../services/suba_map_history_service.dart';

/// Таблица карты как в Nissan V6: градиент значений, подсветка правок,
/// зачёркнутое старое + новое значение, Сохранить/Сброс, зум, полноэкранный.
class SubaMapTableView extends StatefulWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final bool isFullscreen;
  final VoidCallback? onFullscreenTap;
  final VoidCallback? onSaved;

  const SubaMapTableView({
    super.key,
    required this.originalMap,
    this.updatedMap,
    required this.changes,
    this.isFullscreen = false,
    this.onFullscreenTap,
    this.onSaved,
  });

  @override
  State<SubaMapTableView> createState() => _SubaMapTableViewState();
}

class _SubaMapTableViewState extends State<SubaMapTableView> {
  int? _selRow, _selCol;
  MapCell? _selChange;
  double _cellSize = 52;
  Map<String, MapCell> _changesMap = {};
  bool _hasSavedEdits = false;
  DateTime? _savedAt;

  @override
  void initState() {
    super.initState();
    _rebuild();
    _cellSize = widget.isFullscreen ? 62 : 52;
    _checkSaved();
  }

  @override
  void didUpdateWidget(SubaMapTableView old) {
    super.didUpdateWidget(old);
    _rebuild();
    _checkSaved();
  }

  void _rebuild() {
    _changesMap = {};
    for (final c in widget.changes) {
      _changesMap['${c.rpmIndex}_${c.loadIndex}'] = c;
    }
  }

  Future<void> _checkSaved() async {
    final has = await SubaMapStorageService.hasEdits(widget.originalMap.address);
    final at  = await SubaMapStorageService.getUpdatedAt(widget.originalMap.address);
    if (mounted) setState(() { _hasSavedEdits = has; _savedAt = at; });
  }

  Future<void> _save() async {
    if (widget.updatedMap == null) return;
    final ok = await _confirm('Сохранить правки?',
        'Правок: ${widget.changes.length}');
    if (ok != true) return;
    await SubaMapHistoryService.saveRevision(widget.updatedMap!);
    await SubaMapStorageService.saveMap(widget.updatedMap!);
    _snack('Сохранено в дефолт', Colors.green);
    await _checkSaved();
    widget.onSaved?.call();
  }

  Future<void> _reset({bool all = false}) async {
    final ok = await _confirm(
      all ? 'Сбросить ВСЕ карты?' : 'Сбросить эту карту?', '');
    if (ok != true) return;
    if (all) {
      await SubaMapStorageService.resetAll();
    } else {
      await SubaMapStorageService.resetMap(widget.originalMap.address);
    }
    _snack('Сброшено', Colors.green);
    await _checkSaved();
    widget.onSaved?.call();
  }

  Future<bool?> _confirm(String title, String body) => showDialog<bool>(
    context: context,
    builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: Text(title),
      content: body.isNotEmpty
          ? Text(body, style: const TextStyle(color: Colors.white70))
          : null,
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('ДА')),
      ],
    ),
  );

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(
      SnackBar(content: Text(m), backgroundColor: c));
  }

  // Градиент: зелёный→жёлтый→красный
  Color _cellColor(int r, int l) {
    final change = _changesMap['${r}_$l'];
    if (change == null) {
      final val   = widget.originalMap.data[r][l];
      final range = widget.originalMap.maxValue - widget.originalMap.minValue;
      final norm  = range > 0
          ? ((val - widget.originalMap.minValue) / range).clamp(0.0, 1.0)
          : 0.5;
      if (norm < 0.5) {
        return Color.lerp(
          const Color(0xFF1B5E20), const Color(0xFF827717), norm * 2)!
            .withOpacity(0.85);
      } else {
        return Color.lerp(
          const Color(0xFF827717), const Color(0xFFB71C1C), (norm - 0.5) * 2)!
            .withOpacity(0.85);
      }
    }
    final pct = change.deltaPercent.abs();
    if (pct < 3)  return change.delta > 0 ? Colors.yellow.shade800 : Colors.lightBlue.shade800;
    if (pct < 8)  return change.delta > 0 ? Colors.orange.shade800 : Colors.blue.shade800;
    return change.delta > 0 ? Colors.red.shade800 : Colors.blueAccent.shade700;
  }

  @override
  Widget build(BuildContext context) {
    final map = widget.originalMap;
    return Column(children: [
      Container(
        padding: const EdgeInsets.all(6),
        color: const Color(0xFF16213E),
        child: Column(children: [
          Row(children: [
            const Icon(Icons.grid_on, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Expanded(child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Text(map.name, style: const TextStyle(
                  color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12),
                  overflow: TextOverflow.ellipsis),
                if (_hasSavedEdits && _savedAt != null)
                  Text('С правками • ${DateFormat("dd.MM HH:mm").format(_savedAt!)}',
                    style: const TextStyle(color: Colors.orange, fontSize: 9)),
              ],
            )),
            Text('${map.rows}x${map.cols}',
              style: const TextStyle(color: Colors.white54, fontSize: 10)),
            if (!widget.isFullscreen && widget.onFullscreenTap != null) ...[
              const SizedBox(width: 6),
              GestureDetector(
                onTap: widget.onFullscreenTap,
                child: Container(
                  padding: const EdgeInsets.all(4),
                  decoration: BoxDecoration(
                    color: Colors.cyan.withOpacity(0.3),
                    borderRadius: BorderRadius.circular(3)),
                  child: const Icon(Icons.fullscreen, color: Colors.cyan, size: 18)),
              ),
            ],
            if (widget.isFullscreen) ...[
              const SizedBox(width: 6),
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize - 8).clamp(32, 90)),
                child: const Icon(Icons.remove_circle_outline, color: Colors.white70, size: 20)),
              Text('${_cellSize.toInt()}', style: const TextStyle(color: Colors.white54, fontSize: 10)),
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize + 8).clamp(32, 90)),
                child: const Icon(Icons.add_circle_outline, color: Colors.white70, size: 20)),
            ],
          ]),
          if (widget.changes.isNotEmpty || _hasSavedEdits)
            Padding(
              padding: const EdgeInsets.only(top: 6),
              child: Row(children: [
                if (widget.changes.isNotEmpty && widget.updatedMap != null)
                  Expanded(child: ElevatedButton.icon(
                    onPressed: _save,
                    icon: const Icon(Icons.save, size: 14),
                    label: const Text('Сохранить', style: TextStyle(fontSize: 11)),
                    style: ElevatedButton.styleFrom(
                      backgroundColor: Colors.green, padding: const EdgeInsets.symmetric(vertical: 8)),
                  )),
                if (widget.changes.isNotEmpty && _hasSavedEdits) const SizedBox(width: 6),
                if (_hasSavedEdits)
                  Expanded(child: PopupMenuButton<String>(
                    onSelected: (v) => v == 'one' ? _reset() : _reset(all: true),
                    itemBuilder: (_) => const [
                      PopupMenuItem(value: 'one', child: Text('Сбросить эту')),
                      PopupMenuItem(value: 'all', child: Text('Сбросить ВСЕ')),
                    ],
                    child: Container(
                      padding: const EdgeInsets.symmetric(vertical: 8, horizontal: 12),
                      decoration: BoxDecoration(
                        color: Colors.orange, borderRadius: BorderRadius.circular(4)),
                      child: const Row(
                        mainAxisAlignment: MainAxisAlignment.center,
                        children: [
                          Icon(Icons.restore, color: Colors.white, size: 16),
                          SizedBox(width: 4),
                          Text('Сброс', style: TextStyle(
                            color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
                        ]),
                    ),
                  )),
              ]),
            ),
        ]),
      ),
      if (_selChange != null)
        Container(
          padding: const EdgeInsets.all(6),
          margin: const EdgeInsets.all(4),
          decoration: BoxDecoration(
            color: Colors.cyan.withOpacity(0.15),
            borderRadius: BorderRadius.circular(6),
            border: Border.all(color: Colors.cyan.withOpacity(0.5))),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: [
              Text('RPM ${_selChange!.rpm.toStringAsFixed(0)} | Load ${_selChange!.load.toStringAsFixed(0)}%',
                style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12)),
              const Spacer(),
              GestureDetector(
                onTap: () => setState(() { _selChange = null; _selRow = null; _selCol = null; }),
                child: const Icon(Icons.close, size: 14, color: Colors.white54)),
            ]),
            Text(
              '${_selChange!.currentValue.toStringAsFixed(2)} → ${_selChange!.suggestedValue.toStringAsFixed(2)}',
              style: TextStyle(
                color: _selChange!.delta > 0 ? Colors.green : Colors.orange, fontSize: 13)),
            Text(_selChange!.reason, style: const TextStyle(color: Colors.white70, fontSize: 10)),
          ]),
        ),
      Expanded(
        child: InteractiveViewer(
          constrained: false,
          minScale: 0.5,
          maxScale: 3.0,
          boundaryMargin: const EdgeInsets.all(20),
          child: _buildTable(map),
        ),
      ),
    ]);
  }

  Widget _buildTable(TuningMap map) {
    const headerH = 28.0;
    return Padding(
      padding: const EdgeInsets.all(4),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          Container(
            width: _cellSize, height: headerH,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: const Center(child: Text('RPM/%',
              style: TextStyle(fontSize: 9, color: Colors.white70))),
          ),
          ...List.generate(map.cols, (j) => Container(
            width: _cellSize, height: headerH,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.loadAxis[j].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
          )),
        ]),
        ...List.generate(map.rows, (i) => Row(children: [
          Container(
            width: _cellSize, height: _cellSize,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.rpmAxis[i].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
          ),
          ...List.generate(map.cols, (j) => _cell(i, j, map)),
        ])),
      ]),
    );
  }

  Widget _cell(int i, int j, TuningMap map) {
    final change = _changesMap['${i}_$j'];
    final hasC   = change != null;
    final isSel  = _selRow == i && _selCol == j;
    final orig   = map.data[i][j];
    final upd    = widget.updatedMap?.data[i][j] ?? orig;

    return GestureDetector(
      onTap: () => setState(() { _selRow = i; _selCol = j; _selChange = change; }),
      child: Container(
        width: _cellSize, height: _cellSize,
        decoration: BoxDecoration(
          color: _cellColor(i, j),
          border: Border.all(
            color: isSel ? Colors.cyan : hasC ? Colors.white54 : Colors.white12,
            width: isSel ? 2 : (hasC ? 1 : 0.5)),
        ),
        child: hasC
          ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
              Text(orig.toStringAsFixed(1),
                style: TextStyle(fontSize: 9, color: Colors.white.withOpacity(0.7),
                  decoration: TextDecoration.lineThrough)),
              Text(upd.toStringAsFixed(1),
                style: TextStyle(fontSize: 11, fontWeight: FontWeight.bold,
                  color: change!.delta > 0 ? Colors.greenAccent : Colors.yellowAccent)),
            ])
          : Center(child: Text(orig.toStringAsFixed(1),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
      ),
    );
  }
}

/// Полноэкранный просмотр карты
class SubaMapFullscreenView extends StatelessWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final VoidCallback? onSaved;

  const SubaMapFullscreenView({
    super.key,
    required this.originalMap,
    this.updatedMap,
    required this.changes,
    this.onSaved,
  });

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      backgroundColor: const Color(0xFF1A1A2E),
      appBar: AppBar(
        title: Text(originalMap.name),
        backgroundColor: const Color(0xFF16213E),
        leading: IconButton(
          icon: const Icon(Icons.close),
          onPressed: () => Navigator.pop(context)),
      ),
      body: SubaMapTableView(
        originalMap: originalMap,
        updatedMap:  updatedMap,
        changes:     changes,
        isFullscreen: true,
        onSaved:     onSaved,
      ),
    );
  }
}
'''

write('lib/widgets/suba_map_table_view.dart', MAPVIEW, 'widgets/suba_map_table_view.dart (как V6)')

# ── Экран анализатора V6-стиля на Subaru-сигналах (заменяет HeatGrid-экран) ──
SCREEN = r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import '../generated/subaru_rom.g.dart';
import '../models/analysis_result.dart';
import '../models/rom_table.dart';
import '../models/tuning_map.dart';
import '../services/connection_service.dart';
import '../services/suba_analyzer_service.dart';
import '../services/suba_export_service.dart';
import '../services/suba_map_storage_service.dart';
import '../services/suba_rom_bridge.dart';
import '../services/suba_tuning_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/suba_map_table_view.dart';

/// Анализатор Subaru в UX Nissan V6: Онлайн/Из лога, 4 карты, паттерны,
/// REC+автоанализ, сплиттер логов, карта с правками, экспорт и «В прошивку».
/// Сигналы: rpm/load(TPS), KCA, FBKC/FKL, IAM, AFR, триммы, AVCS, boost/tboost.
class AnalyzerScreen extends StatefulWidget {
  const AnalyzerScreen({super.key});
  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen>
    with SingleTickerProviderStateMixin {
  final _analyzer = SubaAnalyzerService();
  final _tuning   = SubaTuningService();
  final _export   = SubaExportService();
  late final TabController _tab;

  String _map = 'timing';
  TuningPattern _pattern = TuningPattern.stability;
  TuningMap? _orig, _upd;
  final Map<String, TuningMap> _romBaseline = {};
  final Map<String, RomTableDef> _romDefs = {};

  List<SubaSample>? _log;
  String? _logName;
  AnalysisResult? _logResult;
  bool _busy = false;

  bool _recording = false;
  final List<SubaSample> _buf = [];
  AnalysisResult? _onlineResult;
  StreamSubscription? _dataSub;
  Timer? _autoTimer;

  static const _mapNames = {
    'timing': 'Зажигание',
    'fuel':   'Топливо',
    'avcs':   'AVCS',
    'boost':  'Target Boost',
  };

  @override
  void initState() { super.initState(); _tab = TabController(length: 2, vsync: this); }
  @override
  void dispose() { _tab.dispose(); _dataSub?.cancel(); _autoTimer?.cancel(); super.dispose(); }

  bool get _connected {
    final s = ConnectionService.I.elm.state;
    return s == SsmState.ecuReady || s == SsmState.polling;
  }

  Future<TuningMap> _baseMap() async {
    final ov = _romBaseline[_map];
    if (ov != null) {
      final saved = await SubaMapStorageService.loadMapData(ov.address);
      if (saved != null && saved.length == ov.rows &&
          (saved.isEmpty || saved[0].length == ov.cols)) {
        return TuningMap(
          name: ov.name, address: ov.address, rows: ov.rows, cols: ov.cols,
          rpmAxis: ov.rpmAxis, loadAxis: ov.loadAxis, data: saved,
          units: ov.units, minValue: ov.minValue, maxValue: ov.maxValue);
      }
      return ov;
    }
    switch (_map) {
      case 'fuel': return _tuning.getFuelMap();
      case 'avcs': return _tuning.getAvcsMap();
      case 'boost': return _tuning.getBoostMap();
      default: return _tuning.getTimingMap();
    }
  }

  Future<(AnalysisResult, TuningMap, TuningMap)?> _doAnalysis(List<SubaSample> data) async {
    final m = await _baseMap();
    late AnalysisResult r;
    switch (_map) {
      case 'fuel':
        r = await _analyzer.analyzeFuelMap(data, m, pattern: _pattern);
      case 'avcs':
        r = await _analyzer.analyzeAvcsMap(data, m, pattern: _pattern);
      case 'boost':
        r = await _analyzer.analyzeBoostMap(data, m, pattern: _pattern);
      default:
        r = await _analyzer.analyzeTimingMap(data, m, pattern: _pattern);
    }
    final u = m.copy();
    for (final c in r.changes) {
      if (c.rpmIndex < u.rows && c.loadIndex < u.cols) {
        u.data[c.rpmIndex][c.loadIndex] = c.suggestedValue;
      }
    }
    return (r, m, u);
  }

  void _openMap() {
    if (_orig == null) return;
    Navigator.push(context, MaterialPageRoute(builder: (_) => SubaMapFullscreenView(
      originalMap: _orig!, updatedMap: _upd,
      changes: (_logResult?.changes ?? _onlineResult?.changes) ?? [],
      onSaved: () async {
        final r = _log ?? _buf;
        if (r.isNotEmpty) {
          final a = await _doAnalysis(List<SubaSample>.from(r));
          if (a != null && mounted) setState(() {
            if (_logResult != null) { _logResult = a.$1; } else { _onlineResult = a.$1; }
            _orig = a.$2; _upd = a.$3;
          });
        } else if (mounted) {
          setState(() {});
        }
      })));
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  // ── Онлайн ───────────────────────────────────────────────────
  void _startRec() {
    final svc = ConnectionService.I;
    if (!_connected || svc.poller == null) {
      _snack('Нет опроса: подключись в Настройках', Colors.red); return;
    }
    setState(() { _recording = true; _buf.clear(); _onlineResult = null; });
    _dataSub = svc.poller!.snapshots.listen((s) {
      if (!_recording) return;
      _buf.add(SubaSample.fromCanons(s.c, s.ts));
      if (_buf.length > 5000) _buf.removeRange(0, 1000);
    });
    _autoTimer = Timer.periodic(const Duration(seconds: 5), (_) async {
      if (_buf.length >= 20 && mounted) {
        final r = await _doAnalysis(List<SubaSample>.from(_buf));
        if (r != null && mounted) setState(() {
          _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
        });
      }
      if (mounted) setState(() {});
    });
  }

  void _stopRec() {
    setState(() => _recording = false);
    _dataSub?.cancel(); _autoTimer?.cancel();
  }

  // ── Из лога + сплиттер ───────────────────────────────────────
  Future<void> _loadLog() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null || r.files.single.path == null) return;
      setState(() => _busy = true);
      _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _logName = r.files.single.name;
      setState(() => _busy = false);
      _snack('Загружено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  Future<void> _analyzeLog() async {
    if (_log == null || _log!.isEmpty) return;
    setState(() => _busy = true);
    final r = await _doAnalysis(_log!);
    if (r != null) setState(() { _logResult = r.$1; _orig = r.$2; _upd = r.$3; _busy = false; });
    else setState(() => _busy = false);
  }

  Future<void> _mergeAndAnalyze() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv'], allowMultiple: true);
      if (r == null || r.files.length < 2) {
        _snack('Выберите 2+ файла', Colors.orange); return;
      }
      setState(() => _busy = true);
      final logs = <List<SubaSample>>[];
      for (final f in r.files) {
        if (f.path != null) logs.add(await _analyzer.loadLogFromCSV(f.path!));
      }
      _log = await _analyzer.mergeLogs(logs);
      _logName = '${r.files.length} логов → ${_log!.length} записей';
      setState(() => _busy = false);
      _snack('Объединено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  // ── База из ROM + запись в прошивку ──────────────────────────
  Future<RomTableDef?> _pickDef() async {
    final defs = SubaruRom.tables;
    final query = TextEditingController();
    List<RomTableDef> filtered = defs;
    return showDialog<RomTableDef>(
      context: context,
      builder: (c) => StatefulBuilder(builder: (c, setD) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Text('Карта ROM A2TB100B', style: TextStyle(fontSize: 14)),
        content: SizedBox(
          width: double.maxFinite, height: 380,
          child: Column(children: [
            TextField(
              controller: query, style: const TextStyle(fontSize: 12),
              decoration: const InputDecoration(
                hintText: 'поиск...', isDense: true,
                prefixIcon: Icon(Icons.search, size: 16)),
              onChanged: (v) => setD(() {
                filtered = defs.where((d) =>
                  d.name.toLowerCase().contains(v.toLowerCase())).toList();
              }),
            ),
            Expanded(child: ListView.builder(
              itemCount: filtered.length,
              itemBuilder: (_, i) {
                final d = filtered[i];
                return ListTile(
                  dense: true,
                  leading: Icon(
                    d.editable ? Icons.grid_on : Icons.grid_off,
                    size: 18,
                    color: d.editable ? Colors.greenAccent : Colors.white38),
                  title: Text(d.name, style: const TextStyle(fontSize: 12)),
                  subtitle: Text(
                    '${d.rows}x${d.cols} ${d.units} ${d.addrHex}',
                    style: const TextStyle(fontSize: 10, color: Colors.white54)),
                  onTap: () => Navigator.pop(c, d),
                );
              },
            )),
          ]),
        ),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c), child: const Text('Отмена')),
        ],
      )),
    );
  }

  Future<void> _loadBaseFromRom() async {
    final rom = ConnectionService.I.rom;
    if (!rom.loaded) { _snack('Сначала загрузи ROM на вкладке ROM', Colors.orange); return; }
    final def = await _pickDef();
    if (def == null || !mounted) return;
    try {
      final table = rom.readTable(def);
      final tm = SubaRomBridge.toTuningMap(table);
      setState(() {
        _romBaseline[_map] = tm;
        _romDefs[_map] = def;
        _logResult = null; _onlineResult = null; _orig = null; _upd = null;
      });
      _snack('База: ${def.name} (${tm.rows}x${tm.cols})', Colors.green);
    } catch (e) {
      _snack('Ошибка чтения ROM: $e', Colors.red);
    }
  }

  Future<void> _writeToRom() async {
    if (_upd == null || _orig == null) return;
    final rom = ConnectionService.I.rom;
    if (!rom.loaded) { _snack('Сначала загрузи ROM на вкладке ROM', Colors.orange); return; }
    var def = _romDefs[_map];
    if (def == null) {
      def = await _pickDef();
      if (def == null || !mounted) return;
      _romDefs[_map] = def;
    }
    if ((_upd!.rows != def.rows) || (_upd!.cols != def.cols)) {
      _snack('Размер ${def.name} (${def.rows}x${def.cols}) != карты анализа (${_upd!.rows}x${_upd!.cols})', Colors.red);
      return;
    }
    final ok = await showDialog<bool>(
      context: context,
      builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Text('Записать в прошивку?'),
        content: Text(
          '${def!.name} (${def.addrHex})\n'
          'Правок: ${(_logResult?.changes ?? _onlineResult?.changes ?? []).length}\n'
          'Будет создан НОВЫЙ файл V8MOD_*.bin, оригинал не меняется.',
          style: const TextStyle(color: Colors.white70, fontSize: 12)),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c, false), child: const Text('Отмена')),
          TextButton(
            onPressed: () => Navigator.pop(c, true),
            style: TextButton.styleFrom(foregroundColor: Colors.green),
            child: const Text('ЗАПИСАТЬ')),
        ],
      ));
    if (ok != true) return;
    try {
      final table = rom.readTable(def);
      SubaRomBridge.applyToRomTable(table, _upd!);
      final path = await SubaRomBridge.writeTableToMod(rom, table);
      if (!mounted) return;
      _snack(path == null ? 'Не записано (read-only?)' : '→ ${path.split('/').last}',
        path == null ? Colors.red : Colors.green);
    } catch (e) {
      _snack('Ошибка записи: $e', Colors.red);
    }
  }

  // ── UI ───────────────────────────────────────────────────────
  @override
  Widget build(BuildContext context) {
    return Column(children: [
      Container(
        color: const Color(0xFF16213E),
        child: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.wifi, size: 18), text: 'Онлайн'),
          Tab(icon: Icon(Icons.folder_open, size: 18), text: 'Из лога'),
        ]),
      ),
      Expanded(child: TabBarView(controller: _tab, children: [
        _onlineTab(), _logTab(),
      ])),
    ]);
  }

  Widget _mapSelector() {
    final hasRomBase = _romBaseline.containsKey(_map);
    return Column(children: [
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Row(children: [
          const Text('Карта:', style: TextStyle(color: Colors.white70, fontSize: 12)),
          const SizedBox(width: 8),
          Expanded(child: DropdownButtonFormField<String>(
            value: _map, dropdownColor: const Color(0xFF16213E), isDense: true,
            items: const [
              DropdownMenuItem(value: 'timing', child: Text('Зажигание')),
              DropdownMenuItem(value: 'fuel',   child: Text('Топливо')),
              DropdownMenuItem(value: 'avcs',   child: Text('AVCS')),
              DropdownMenuItem(value: 'boost',  child: Text('Буст')),
            ],
            onChanged: (v) => setState(() {
              _map = v!; _logResult = null; _onlineResult = null;
              _orig = null; _upd = null;
            }))),
        ]))),
      const SizedBox(height: 4),
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Row(children: [
          Expanded(child: Text(
            hasRomBase
              ? 'База: ROM ${_romDefs[_map]?.addrHex ?? ''}'
              : 'База: дефолт EJ20X',
            style: TextStyle(
              color: hasRomBase ? Colors.greenAccent : Colors.white54, fontSize: 11))),
          TextButton.icon(
            onPressed: _busy ? null : _loadBaseFromRom,
            icon: const Icon(Icons.sd_card, size: 14),
            label: const Text('Из ROM', style: TextStyle(fontSize: 11))),
          if (hasRomBase)
            TextButton(
              onPressed: () => setState(() {
                _romBaseline.remove(_map); _romDefs.remove(_map);
                _logResult = null; _onlineResult = null; _orig = null; _upd = null;
              }),
              child: const Text('Дефолт', style: TextStyle(fontSize: 11))),
        ]))),
      const SizedBox(height: 4),
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('Паттерн:', style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold)),
          const SizedBox(height: 4),
          Wrap(spacing: 6, runSpacing: 6, children: TuningPattern.all.map((p) {
            final sel = _pattern.type == p.type;
            return GestureDetector(
              onTap: () => setState(() => _pattern = p),
              child: Container(
                padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 6),
                decoration: BoxDecoration(
                  color: sel ? Colors.orange.withOpacity(0.3) : const Color(0xFF0F3460),
                  borderRadius: BorderRadius.circular(6),
                  border: Border.all(color: sel ? Colors.orange : Colors.white24)),
                child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                  Text(p.name, style: TextStyle(
                    color: sel ? Colors.orange : Colors.white70,
                    fontSize: 11, fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
                  Text(p.description, style: const TextStyle(color: Colors.white38, fontSize: 8)),
                ])));
          }).toList()),
        ]))),
    ]);
  }

  Widget _onlineTab() {
    final svc = ConnectionService.I;
    final hz = svc.elm.stats.hz;
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: _recording ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
            Row(children: [
              Icon(_connected ? Icons.check_circle : Icons.error,
                color: _connected ? Colors.green : Colors.red, size: 18),
              const SizedBox(width: 6),
              Expanded(child: Text(
                _connected ? 'ЭБУ готов · ${hz.toStringAsFixed(1)} Гц' : 'Нет подключения',
                style: const TextStyle(fontSize: 12))),
              if (_recording)
                Container(
                  padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                  decoration: BoxDecoration(color: Colors.red, borderRadius: BorderRadius.circular(4)),
                  child: const Text('REC', style: TextStyle(
                    color: Colors.white, fontWeight: FontWeight.bold, fontSize: 10))),
            ]),
            const SizedBox(height: 4),
            Row(children: [
              _stat('Записей', _buf.length.toString(), Colors.blue),
              const SizedBox(width: 4),
              _stat('Правок', (_onlineResult?.changes.length ?? 0).toString(), Colors.orange),
            ]),
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: _recording ? _stopRec : (_connected ? _startRec : null),
            icon: Icon(_recording ? Icons.stop : Icons.play_arrow, size: 18),
            label: Text(_recording ? 'СТОП' : 'ЗАПИСЬ', style: const TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _recording ? Colors.red : Colors.green,
              minimumSize: const Size.fromHeight(40)))),
          const SizedBox(width: 4),
          Expanded(child: ElevatedButton.icon(
            onPressed: _buf.length >= 20 ? () async {
              final r = await _doAnalysis(List<SubaSample>.from(_buf));
              if (r != null && mounted) setState(() {
                _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
              });
            } : null,
            icon: const Icon(Icons.refresh, size: 18),
            label: const Text('АНАЛИЗ', style: TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(40)))),
        ]),
        if (_onlineResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_onlineResult!),
        ],
        if (_onlineResult == null)
          const Padding(padding: EdgeInsets.all(20),
            child: Text('Нажми ЗАПИСЬ → покатайся → правки автоматически',
              style: TextStyle(color: Colors.white54, fontSize: 12),
              textAlign: TextAlign.center)),
      ]),
    );
  }

  Widget _logTab() {
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
          child: Column(children: [
            Row(children: [
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _loadLog,
                icon: const Icon(Icons.folder_open, size: 18),
                label: const Text('CSV'),
                style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(38)))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _mergeAndAnalyze,
                icon: const Icon(Icons.merge_type, size: 18),
                label: const Text('Объединить'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.purple,
                  minimumSize: const Size.fromHeight(38)))),
            ]),
            if (_logName != null) ...[
              const SizedBox(height: 4),
              Text(_logName!, style: const TextStyle(color: Colors.white70, fontSize: 11)),
              Text('Записей: ${_log?.length ?? 0}',
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ],
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        SizedBox(width: double.infinity, child: ElevatedButton.icon(
          onPressed: _busy || _log == null ? null : _analyzeLog,
          icon: const Icon(Icons.analytics, size: 18),
          label: const Text('АНАЛИЗИРОВАТЬ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, minimumSize: const Size.fromHeight(40)))),
        if (_busy) ...[
          const SizedBox(height: 8),
          const LinearProgressIndicator(minHeight: 3),
        ],
        if (_logResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_logResult!),
        ],
      ]),
    );
  }

  Widget _stat(String l, String v, Color c) => Expanded(
    child: Container(
      padding: const EdgeInsets.all(3),
      decoration: BoxDecoration(color: const Color(0xFF0F3460), borderRadius: BorderRadius.circular(4)),
      child: Column(children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 9)),
        Text(v, style: TextStyle(color: c, fontSize: 13, fontWeight: FontWeight.bold)),
      ])));

  Widget _resultCard(AnalysisResult r) {
    return Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(6),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Icon(Icons.assessment, color: Colors.green, size: 18),
          const SizedBox(width: 4),
          Expanded(child: Text(r.mapName, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
          Text('${r.changes.length} правок', style: const TextStyle(color: Colors.orange, fontSize: 12)),
        ]),
        if (r.patternName.isNotEmpty)
          Text('Паттерн: ${r.patternName}', style: const TextStyle(color: Colors.white54, fontSize: 10)),
        Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),
        const SizedBox(height: 4),
        if (_orig != null) SizedBox(width: double.infinity,
          child: ElevatedButton.icon(onPressed: _openMap,
            icon: const Icon(Icons.grid_on, size: 20),
            label: const Text('ОТКРЫТЬ КАРТУ', style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan.shade700, minimumSize: const Size.fromHeight(42)))),
        const SizedBox(height: 6),
        SizedBox(height: 250,
          child: r.changes.isEmpty
            ? const Center(child: Text('Правок нет', style: TextStyle(color: Colors.green, fontSize: 14)))
            : ListView.builder(
                itemCount: r.changes.length,
                itemBuilder: (_, i) {
                  final c = r.changes[i];
                  final dc = c.delta > 0 ? Colors.green : Colors.orange;
                  return Card(color: const Color(0xFF0F3460), margin: const EdgeInsets.symmetric(vertical: 2),
                    child: ListTile(dense: true,
                      title: Text('RPM ${c.rpm.toStringAsFixed(0)} | ${c.load.toStringAsFixed(0)}%',
                        style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                      subtitle: Text(
                        '${c.currentValue.toStringAsFixed(2)} → ${c.suggestedValue.toStringAsFixed(2)} (${c.reason})',
                        style: TextStyle(color: dc, fontSize: 10)),
                      trailing: Text('${(c.confidence * 100).toInt()}%',
                        style: TextStyle(
                          color: c.confidence > 0.8 ? Colors.green : Colors.orange,
                          fontSize: 11, fontWeight: FontWeight.bold))));
                })),
        const SizedBox(height: 4),
        Wrap(spacing: 4, runSpacing: 4, children: [
          _expBtn('WinOLS', Colors.blue, () async {
            if (_upd != null) { final p = await _export.exportToWinOLS(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('JSON', Colors.green, () async {
            if (_upd != null && _orig != null) {
              final p = await _export.exportToJson(r, _orig!, _upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('HEX', Colors.orange, () async {
            if (_upd != null) { final p = await _export.exportHexPatch(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('В прошивку', Colors.red.shade700, _writeToRom),
        ]),
      ])));
  }

  Widget _expBtn(String l, Color c, VoidCallback onTap) => ElevatedButton.icon(
    onPressed: onTap, icon: const Icon(Icons.download, size: 12),
    label: Text(l, style: const TextStyle(fontSize: 10)),
    style: ElevatedButton.styleFrom(
      backgroundColor: c, padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4)));
}
'''

# Бэкап старого HeatGrid-экрана (один раз) и замена на V6-стиль
import shutil
_old = 'lib/screens/analyzer_screen.dart'
_bak = 'lib/screens/analyzer_screen.heatgrid.bak'
if os.path.exists(_old) and not os.path.exists(_bak):
    shutil.copy(_old, _bak)
    print('   · бэкап старого экрана: analyzer_screen.heatgrid.bak')
write('lib/screens/analyzer_screen.dart', SCREEN, 'screens/analyzer_screen.dart (V6-стиль, Subaru)')

# ── Кнопки "В дефолт" в ROM и ЭБУ Картах + проверка ──
edit('lib/screens/ecu_maps_screen.dart',
     "import '../widgets/heat_map.dart';",
     "import '../widgets/heat_map.dart';\n"
     "import '../services/suba_map_storage_service.dart';\n"
     "import '../services/suba_rom_bridge.dart';",
     'ecu_maps: imports моста')

edit('lib/screens/ecu_maps_screen.dart',
     "        const Text('Прочитано живьём из ECU. Read-only: правка карт — на экране ROM (в .bin-файл).',\n"
     "            style: TextStyle(fontSize: 11, color: HeatColors.dim)),",
     "        FilledButton.tonalIcon(\n"
     "          onPressed: () async {\n"
     "            try {\n"
     "              final tm = SubaRomBridge.toTuningMap(table);\n"
     "              await SubaMapStorageService.saveMap(tm);\n"
     "              if (context.mounted) {\n"
     "                ScaffoldMessenger.of(context).showSnackBar(SnackBar(\n"
     "                  content: Text('Сохранено в дефолт анализатора: ${tm.name}'),\n"
     "                  backgroundColor: Colors.green));\n"
     "              }\n"
     "            } catch (e) {\n"
     "              if (context.mounted) {\n"
     "                ScaffoldMessenger.of(context).showSnackBar(SnackBar(\n"
     "                  content: Text('Ошибка: $e'), backgroundColor: Colors.red));\n"
     "              }\n"
     "            }\n"
     "          },\n"
     "          icon: const Icon(Icons.save_alt, size: 18),\n"
     "          label: const Text('В дефолт (анализатор)'),\n"
     "        ),\n"
     "        const SizedBox(height: 8),\n"
     "        const Text('Прочитано живьём из ECU. Read-only: правка карт — на экране ROM (в .bin-файл).',\n"
     "            style: TextStyle(fontSize: 11, color: HeatColors.dim)),",
     'ecu_maps: кнопка "В дефолт"')

edit('lib/screens/rom_screen.dart',
     "import '../widgets/heat_map.dart';",
     "import '../widgets/heat_map.dart';\n"
     "import '../services/suba_map_storage_service.dart';\n"
     "import '../services/suba_rom_bridge.dart';",
     'rom_screen: imports моста')

edit('lib/screens/rom_screen.dart',
     "  String _ax(double v)",
     "  Future<void> _saveAsDefault() async {\n"
     "    final t = _table;\n"
     "    if (t == null) return;\n"
     "    try {\n"
     "      final tm = SubaRomBridge.toTuningMap(t);\n"
     "      await SubaMapStorageService.saveMap(tm);\n"
     "      if (!mounted) return;\n"
     "      ScaffoldMessenger.of(context).showSnackBar(SnackBar(\n"
     "        content: Text('Сохранено в дефолт анализатора: ${tm.name}'),\n"
     "        backgroundColor: Colors.green));\n"
     "    } catch (e) {\n"
     "      if (!mounted) return;\n"
     "      ScaffoldMessenger.of(context).showSnackBar(\n"
     "        SnackBar(content: Text('Ошибка: $e'), backgroundColor: Colors.red));\n"
     "    }\n"
     "  }\n"
     "\n"
     "  String _ax(double v)",
     'rom_screen: метод _saveAsDefault')

edit('lib/screens/rom_screen.dart',
     "            label: const Text('Править ячейку'),\n"
     "          )",
     "            label: const Text('Править ячейку'),\n"
     "          ),\n"
     "          const SizedBox(height: 8),\n"
     "          FilledButton.tonalIcon(\n"
     "            onPressed: _saveAsDefault,\n"
     "            icon: const Icon(Icons.save_alt, size: 18),\n"
     "            label: const Text('В дефолт (анализатор)'),\n"
     "          )",
     'rom_screen: кнопка "В дефолт"')

# ── Проверка ──
print()
print('=' * 64)
print('  Проверка анализатора V6-стиля')
print('=' * 64)
required = [
    'lib/models/tuning_map.dart',
    'lib/models/analysis_result.dart',
    'lib/services/suba_analyzer_service.dart',
    'lib/services/suba_tuning_service.dart',
    'lib/services/suba_map_storage_service.dart',
    'lib/services/suba_map_history_service.dart',
    'lib/services/suba_rom_bridge.dart',
    'lib/services/suba_export_service.dart',
    'lib/widgets/suba_map_table_view.dart',
    'lib/screens/analyzer_screen.dart',
]
missing = [p for p in required if not os.path.exists(p)]
assert not missing, f'НЕТ ФАЙЛОВ: {missing}'
for p in required:
    print(f'  ✓ {p} ({os.path.getsize(p)} B)')
s = open('lib/screens/rom_screen.dart', encoding='utf-8').read()
assert 'В дефолт (анализатор)' in s, 'патч rom_screen не применился'
s = open('lib/screens/ecu_maps_screen.dart', encoding='utf-8').read()
assert 'В дефолт (анализатор)' in s, 'патч ecu_maps не применился'
s = open('lib/screens/analyzer_screen.dart', encoding='utf-8').read()
assert 'SubaAnalyzerService' in s and 'SubaMapFullscreenView' in s, \
    'analyzer_screen не заменён'
print()
print('  Все файлы на месте. Патчи ROM/ЭБУ применены.')
print('  Дальше: ячейка 10/10 (сборка APK).')
print('=' * 64)
print()
print('ЧТО ИЗМЕНИЛОСЬ:')
print('  · Анализатор = UX V6: Онлайн/Из лога, 4 карты, паттерны,');
print('    REC+автоанализ, сплиттер CSV, карта с правками, экспорт, «В прошивку».');
print('  · Сигналы Subaru: KCA, FBKC/FKL+IAM, AFR, триммы, AVCS, boost/tboost.');
print('  · Torque V6 заменён на Target Boost; MAF-таблица Hitachi НЕ портирована.');
print('  · Дефолты EJ20X — стартовые шаблоны; эталон — ROM/ECU через «В дефолт».');
print('  · Запись MOD: пакетная, все типы хранения, проверка границ,');
print('    подтверждение, новый файл V8MOD_*.bin (оригинал не трогается).')
print('  · Старый HeatGrid-экран сохранён в analyzer_screen.heatgrid.bak.')


OK  models/tuning_map.dart (как V6) (2607 B)
OK  models/analysis_result.dart (как V6) (1107 B)
OK  services/suba_map_storage_service.dart (2392 B)
OK  services/suba_map_history_service.dart (2838 B)
OK  services/suba_tuning_service.dart (Subaru-дефолты) (4516 B)
OK  services/suba_analyzer_service.dart (Subaru) (20148 B)
OK  services/suba_rom_bridge.dart (ROM↔анализ + MOD) (4854 B)
OK  services/suba_export_service.dart (WinOLS/JSON/HEX) (2735 B)
OK  widgets/suba_map_table_view.dart (как V6) (14198 B)
   · бэкап старого экрана: analyzer_screen.heatgrid.bak
OK  screens/analyzer_screen.dart (V6-стиль, Subaru) (25020 B)
OK  ecu_maps: imports моста
OK  ecu_maps: кнопка "В дефолт"
OK  rom_screen: imports моста
OK  rom_screen: метод _saveAsDefault
OK  rom_screen: кнопка "В дефолт"

  Проверка анализатора V6-стиля
  ✓ lib/models/tuning_map.dart (2607 B)
  ✓ lib/models/analysis_result.dart (1107 B)
  ✓ lib/services/suba_analyzer_service.dart (21857 B)
  ✓ lib/services/suba_tuning_service.dart (4

In [ ]:
# @title ⚡ ЭКСПРЕСС-ФИКС: Ошибка синтаксиса в rom_screen.dart (1 секунда)
# Исправляет: lib/screens/rom_screen.dart:222:9: Error: Expected ']' before this. else
import os, re
os.chdir('/content/suba_run_v8')

p = 'lib/screens/rom_screen.dart'
s = open(p, encoding='utf-8').read()

# 1. Добавляем импорты моста, если их нет
if "import '../services/suba_map_storage_service.dart';" not in s:
    s = s.replace(
        "import '../widgets/heat_map.dart';",
        "import '../widgets/heat_map.dart';\n"
        "import '../services/suba_map_storage_service.dart';\n"
        "import '../services/suba_rom_bridge.dart';",
        1,
    )

# 2. Добавляем метод _saveAsDefault, если его нет
if "_saveAsDefault" not in s:
    save_method = """  Future<void> _saveAsDefault() async {
    final t = _table;
    if (t == null) return;
    try {
      final tm = SubaRomBridge.toTuningMap(t);
      await SubaMapStorageService.saveMap(tm);
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('Сохранено в дефолт анализатора: ${tm.name}'),
        backgroundColor: Colors.green));
    } catch (e) {
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(content: Text('Ошибка: $e'), backgroundColor: Colors.red));
    }
  }

  String _ax(double v)"""
    s = s.replace("  String _ax(double v)", save_method, 1)

# 3. Чиним блок кнопок: в Dart внутри списка if (cond) требует spread ...[ ] для нескольких элементов
pattern = r'const SizedBox\(height: 10\),\s*if\s*\(d\.editable\).*?(?=if\s*\(_modified\))'
replacement = '''const SizedBox(height: 10),
        if (d.editable) ...[
          const Text('Долгий тап по ячейке — правка значения',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
          const SizedBox(height: 6),
          FilledButton.icon(
            onPressed: _editCell,
            icon: const Icon(Icons.edit, size: 18),
            label: const Text('Править ячейку'),
          ),
          const SizedBox(height: 8),
          FilledButton.tonalIcon(
            onPressed: _saveAsDefault,
            icon: const Icon(Icons.save_alt, size: 18),
            label: const Text('В дефолт (анализатор)'),
          ),
        ] else ...[
          const Text('Карта read-only (нет обратной формулы frexpr)',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
        ],
        '''

s, count = re.subn(pattern, replacement, s, flags=re.S)
assert count == 1, 'Не удалось найти блок кнопок в rom_screen.dart'

open(p, 'w', encoding='utf-8').write(s)
print('=' * 64)
print('✅ rom_screen.dart исправлен! Ошибка Expected \']\' устранена.')
print('   Теперь сразу запускай ячейку 10/10 (Сборка APK)!')
print('=' * 64)

✅ rom_screen.dart исправлен! Ошибка Expected ']' устранена.
   Теперь сразу запускай ячейку 10/10 (Сборка APK)!


In [ ]:
# @title 🔨 Ячейка 10/10: Сборка SUBA RUN V8 APK (~10-15 мин)
# ============================================================================
import os, glob

os.chdir('/content/suba_run_v8')

# окружение (на случай перезапуска сессии)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

print('=' * 64)
print('  ЭТАП 1: flutter clean + pub get')
print('=' * 64)
!flutter clean 2>&1 | tail -2
!rm -rf build
!rm -f pubspec.lock
pub_result = !flutter pub get 2>&1
for line in pub_result[-6:]:
    print(line)
if any('version solving failed' in l for l in pub_result):
    raise SystemExit('pub get провалился — проверь констрейнт SDK в pubspec (должно быть <4.0.0)')

print()
print('=' * 64)
print('  ЭТАП 2: Патч flutter_bluetooth_serial (namespace/SDK36 — фикс из V6)')
print('=' * 64)
plugin_dirs = glob.glob('/content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*') or \
              glob.glob('/root/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')
for plugin_dir in plugin_dirs:
    with open(f'{plugin_dir}/android/build.gradle', 'w') as f:
        f.write('''group 'io.github.edufolly.flutterbluetoothserial'
version '1.0-SNAPSHOT'
buildscript {
    repositories { google(); mavenCentral() }
    dependencies { classpath 'com.android.tools.build:gradle:8.13.0' }
}
allprojects { repositories { google(); mavenCentral() } }
apply plugin: 'com.android.library'
android {
    namespace 'io.github.edufolly.flutterbluetoothserial'
    compileSdk 36
    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }
    defaultConfig { minSdk 21 }
    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}
dependencies { implementation 'androidx.core:core:1.13.1' }
''')
    with open(f'{plugin_dir}/android/src/main/AndroidManifest.xml', 'w') as f:
        f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
</manifest>
''')
    print('  OK ' + plugin_dir.split('/')[-1])
if not plugin_dirs:
    print('  плагин не найден в кэше (проверь pub get)')

print()
print('=' * 64)
print('  ЭТАП 2.5: Версии Gradle/AGP/Kotlin под ТЕКУЩИЙ Flutter stable')
print('=' * 64)
# Flutter stable постоянно двигает минимумы (сейчас Gradle >= 8.14).
# Принудительно выставляем совместимую связку перед сборкой.
import re as _re
_wp = 'android/gradle/wrapper/gradle-wrapper.properties'
with open(_wp, 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip
''')
_sg = 'android/settings.gradle.kts'
_s = open(_sg).read()
_s = _re.sub(r'"com\.android\.application"\) version "[^"]+"',
             '"com.android.application") version "8.13.0"', _s)
_s = _re.sub(r'"org\.jetbrains\.kotlin\.android"\) version "[^"]+"',
             '"org.jetbrains.kotlin.android") version "2.1.20"', _s)
open(_sg, 'w').write(_s)
print('  Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.20')

print()
print('=' * 64)
print('  ЭТАП 3: Проверка структуры')
print('=' * 64)
required = [
    'lib/main.dart', 'lib/constants.dart',
    'lib/models/live_snapshot.dart', 'lib/models/rom_table.dart',
    'lib/ssm/ssm_elm.dart',
    'lib/generated/subaru_pids.g.dart', 'lib/generated/subaru_rom.g.dart',
    'lib/services/settings_service.dart', 'lib/services/logger_service.dart',
    'lib/services/alert_service.dart', 'lib/services/rom_service.dart',
    'lib/services/analyzer_service.dart', 'lib/services/expr_eval.dart',
    'lib/services/dtc_service.dart', 'lib/services/custom_pid_service.dart',
    'lib/services/profile_service.dart', 'lib/services/export_service.dart',
    'lib/services/connection_service.dart',
    'lib/widgets/heat_colors.dart', 'lib/widgets/heat_map.dart',
    'lib/screens/dashboard_screen.dart', 'lib/screens/graphs_screen.dart',
    'lib/screens/log_graph_screen.dart', 'lib/screens/logging_screen.dart',
    'lib/screens/events_screen.dart', 'lib/screens/dtc_screen.dart',
    'lib/screens/analyzer_screen.dart', 'lib/screens/ecu_maps_screen.dart',
    'lib/screens/rom_screen.dart', 'lib/screens/rom_diff_screen.dart',
    'lib/screens/service_screen.dart', 'lib/screens/perf_screen.dart',
    'lib/screens/export_screen.dart', 'lib/screens/custom_pid_screen.dart',
    'lib/screens/profile_screen.dart', 'lib/screens/terminal_screen.dart',
    'lib/screens/settings_screen.dart',
]
all_ok = True
for rf in required:
    ok = os.path.exists(rf) and os.path.getsize(rf) > 50
    all_ok &= ok
    if not ok:
        print('  НЕТ ' + rf)
import subprocess
total_lines = int(subprocess.check_output("find lib -name '*.dart' | xargs wc -l | tail -1 | awk '{print $1}'", shell=True))
print(f'  файлов: {sum(1 for rf in required if os.path.exists(rf))}/{len(required)} · строк Dart: {total_lines}')
if not all_ok:
    raise SystemExit('Не хватает файлов — прогони ячейки 0-9 по порядку!')

print()
print('=' * 64)
print('  ЭТАП 4: СБОРКА APK')
print('=' * 64)
result = !flutter build apk --release --no-tree-shake-icons --android-skip-build-dependency-validation 2>&1
tail = result[-70:]
keep = [l for l in result if any(k in l.lower() for k in
        ['error', 'failed', 'built', 'app-release.apk', 'exception'])]
for l in keep[-25:]:
    print(l)

apk = '/content/suba_run_v8/build/app/outputs/flutter-apk/app-release.apk'
print()
if os.path.exists(apk):
    size_mb = os.path.getsize(apk) / (1024 * 1024)
    print('=' * 64)
    print(f'  APK СОБРАН: {size_mb:.1f} MB · {total_lines} строк Dart')
    print('=' * 64)
    !cp {apk} /content/SubaRunV8.apk
    from google.colab import files
    files.download('/content/SubaRunV8.apk')
    print()
    print('УСТАНОВКА:')
    print(' 1. Установи SubaRunV8.apk, разреши Bluetooth')
    print(' 2. Настройки -> выбери ELM327 -> CONNECT + INIT ECU (жди ECU ID 5204584007)')
    print(' 3. Смотри статистику протокола: снапш/с и мс/кадр')
    print(' 4. Анализатор: Ось Y = Буст, Метрика = FBKC / KCA — цветная карта как в V6')
    print(' 5. ROM: загрузи свой .bin A2TB100B — все карты из дефинишна (через include A2TB100K)')
    print(' 6. ЭБУ Карты: чтение карты живьём из ECU блоками')
    print(' 7. 17 экранов как в V6 — но с параметрами Subaru V7')
else:
    print('APK НЕ СОБРАН. Хвост лога:')
    for l in tail:
        print(l)
    print('Скинь этот вывод — найдём причину.')


  ЭТАП 1: flutter clean + pub get
Deleting ephemeral...                                                0ms
Deleting .flutter-plugins-dependencies...                            0ms
+ xdg_directories 1.1.0
+ xml 6.6.1 (7.0.1 available)
+ yaml 3.1.4
Changed 87 dependencies!
17 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

  ЭТАП 2: Патч flutter_bluetooth_serial (namespace/SDK36 — фикс из V6)
  OK flutter_bluetooth_serial-0.4.0

  ЭТАП 2.5: Версии Gradle/AGP/Kotlin под ТЕКУЩИЙ Flutter stable
  Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.20

  ЭТАП 3: Проверка структуры
  файлов: 37/37 · строк Dart: 9865

  ЭТАП 4: СБОРКА APK
✓ Built build/app/outputs/flutter-apk/app-release.apk (59.6MB)

  APK СОБРАН: 56.9 MB · 9865 строк Dart


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


УСТАНОВКА:
 1. Установи SubaRunV8.apk, разреши Bluetooth
 2. Настройки -> выбери ELM327 -> CONNECT + INIT ECU (жди ECU ID 5204584007)
 3. Смотри статистику протокола: снапш/с и мс/кадр
 4. Анализатор: Ось Y = Буст, Метрика = FBKC / KCA — цветная карта как в V6
 5. ROM: загрузи свой .bin A2TB100B — все карты из дефинишна (через include A2TB100K)
 6. ЭБУ Карты: чтение карты живьём из ECU блоками
 7. 17 экранов как в V6 — но с параметрами Subaru V7
